# LayerNorm forward that saves its statistics

LayerNorm normalises each row of a `(M, N)` activation tensor and applies a learned scale and shift:

$$\mu_i = \frac1N\sum_j x_{ij}, \qquad \sigma_i^2 = \frac1N\sum_j (x_{ij} - \mu_i)^2, \qquad r_i = \frac{1}{\sqrt{\sigma_i^2 + \varepsilon}}, \qquad y_{ij} = (x_{ij} - \mu_i)\, r_i\, w_j + b_j$$

A training kernel also **saves** $\mu$ and $r$ ("rstd") so that the backward pass doesn't recompute them.

Write `layer_norm_fwd(x, w, b, eps)`.

## Inputs
- `x`: contiguous `float32` CUDA tensor `(M, N)` with $1 \le N \le 8192$.
- `w`, `b`: `float32` `(N,)`. `eps`: Python float (e.g. `1e-5`).

## Output
A tuple `(y, mean, rstd)`:
- `y`: `float32` `(M, N)`.
- `mean`, `rstd`: `float32` `(M,)`, the $\mu_i$ and $r_i$ above (population variance, $\varepsilon$ inside the square root).

## Requirements
- One program per row, the whole row in one block: `BLOCK = triton.next_power_of_2(N)`. Read `x` once.
- Compute the variance from centred values, with padding lanes forced to 0 *after* centring.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def layer_norm_fwd(x, w, b, eps):
    """x: (M, N), w, b: (N,), float32 CUDA. Return (y (M, N), mean (M,), rstd (M,))."""
    raise NotImplementedError


@triton.jit
def _layer_norm_fwd_kernel(x_ptr, w_ptr, b_ptr, y_ptr, mean_ptr, rstd_ptr, N, stride_row, eps,
                           BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0, 4.0]], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0, 1.0], "dtype": "float32"}, {"__nd__": [0.0, 0.0, 0.0, 0.0], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[-1.3416354656219482, -0.4472118020057678, 0.4472118020057678, 1.3416354656219482]], "dtype": "float32"}, {"__nd__": [2.5], "dtype": "float32"}, {"__nd__": [0.8944236040115356], "dtype": "float32"}]}}, {"a": [{"__nd__": [[2.0, 2.0, 2.0], [0.0, 3.0, 6.0]], "dtype": "float32"}, {"__nd__": [1.0, 2.0, 3.0], "dtype": "float32"}, {"__nd__": [0.5, 0.0, -0.5], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[0.5, 0.0, -0.5], [-0.7247438430786133, 0.0, 3.17423152923584]], "dtype": "float32"}, {"__nd__": [2.0, 3.0], "dtype": "float32"}, {"__nd__": [316.2277526855469, 0.4082479476928711], "dtype": "float32"}]}}, {"a": [{"__nd__": [[5.0], [-1.0]], "dtype": "float32"}, {"__nd__": [2.0], "dtype": "float32"}, {"__nd__": [1.0], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[1.0], [1.0]], "dtype": "float32"}, {"__nd__": [5.0, -1.0], "dtype": "float32"}, {"__nd__": [316.2277526855469, 316.2277526855469], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.6664369106292725, 0.09745096415281296, -1.8996468782424927, -1.9400697946548462, -1.4105191230773926, -0.8005887269973755, -0.7341026067733765, -0.45682114362716675, -0.19085001945495605, -0.8749718070030212, 0.004578832071274519, 0.5929142832756042, 0.27219006419181824, 0.8811423778533936, 1.8304601907730103, -0.6295367479324341, 1.602476954460144, -1.0598102807998657, -0.06221362575888634, 1.0090560913085938, 1.3654922246932983, 0.6893417835235596, 0.7105263471603394, 1.326422095298767, 0.09636664390563965, 0.694972813129425, -0.6910847425460815, -1.5508253574371338, -2.1468052864074707, 1.5177860260009766, 0.660645067691803, -0.5316464900970459, -0.6482747793197632, -0.5815600752830505, -1.2175205945968628, -0.8994107246398926, -0.4625582993030548, 1.0252244472503662, 0.2964872717857361, -1.5846532583236694, 1.1330275535583496, 0.849331259727478, 0.14462172985076904, 0.035854510962963104, 0.4705255925655365, 0.13624516129493713, -0.7631070613861084, 1.8389570713043213, 0.8864462971687317, 0.3841524124145508, -0.821735143661499, -1.1125961542129517, 0.9218247532844543, 0.302937775850296, 0.5157114267349243, -1.0367920398712158, 0.04134040325880051, -1.2178770303726196, -0.42521458864212036, -0.055233433842659, 1.3714048862457275, -0.7375121116638184, -1.124310851097107, 0.5543838739395142, 0.7334564924240112, -1.5033477544784546, 0.5872058272361755, 1.1054304838180542, -0.3264319598674774, 0.17977800965309143, 0.8309114575386047, 0.6859360933303833, -0.09821706265211105, -0.27890002727508545, 0.2563827335834503, 0.5321318507194519, -0.25337982177734375, -0.7651000618934631, 0.5872423052787781, 0.76915442943573, 0.9444971084594727, -0.4199290871620178, 1.528764247894287, 2.309852361679077, -0.776409387588501, 0.274394154548645, 0.566169023513794, -2.0024759769439697, -0.4219236969947815, -1.2400115728378296, -0.5635338425636292, -1.3787423372268677, -0.597500741481781, -0.23792493343353271, -0.4234524667263031, 1.059053659439087, -0.22741635143756866, 0.5658458471298218, 0.37958064675331116, -0.18089665472507477, -0.2189166396856308, 0.7672070264816284, 0.6283630728721619, 1.123435139656067, -0.8312219977378845, 0.36668646335601807, -0.13746817409992218, 0.0814082995057106, 0.7284713983535767, 1.583181381225586, -1.477735996246338, -0.8188950419425964, 0.9723223447799683, 0.4045029878616333, 1.6087360382080078, 0.8835115432739258, 0.19378238916397095, 0.5478714108467102, 2.0887134075164795, -0.39458516240119934, 0.04800829663872719, -1.4032115936279297, -0.3721541166305542, 0.22642433643341064, -0.19614171981811523, 0.23648767173290253, -1.3228704929351807, -0.31321027874946594, -0.019289398565888405, -0.009657721035182476, 1.482064127922058, -1.7362898588180542, 0.9542151689529419, 0.6727385520935059, -1.8752543926239014, -1.013688564300537, 0.48294007778167725, 1.646759271621704, -0.811723530292511, -0.24941426515579224, 0.10802905261516571, -1.1243010759353638, 0.3670794665813446, 1.4076290130615234, -0.4951663613319397, -1.296591877937317, 0.270722895860672, -0.011358404532074928, -1.8272128105163574, 1.0529600381851196, 0.9632642865180969, 0.19505752623081207, 0.16353434324264526, 0.08790697902441025, -0.0688595324754715, 0.2895575165748596, -1.118957281112671, -0.16825233399868011, -0.267320454120636, 0.7318381667137146, 0.3239074647426605, -0.09606265276670456, -0.09504620730876923, 1.5710115432739258, 0.41256868839263916, -0.0031824209727346897, -2.3597025871276855, 0.4921945631504059, -1.1765246391296387, 0.483458548784256, 0.796193540096283, -2.7431936264038086, 0.5441086292266846, 1.9693487882614136, -0.1935047060251236, 1.6774976253509521, -1.3027997016906738, -0.9670006036758423, 0.5078331232070923, -0.8298941850662231, 1.57459557056427, 1.1890997886657715, -0.7050508856773376, 0.4206835627555847, 1.1600546836853027, 0.8120075464248657, 1.0316351652145386, -0.13724391162395477, -0.5546457767486572, 0.7248934507369995, -1.3719638586044312, 0.1562597006559372, 0.6537139415740967, -0.8056143522262573, -0.5023577213287354, 1.4343708753585815, -0.7534964084625244, 0.062144309282302856, 0.035919349640607834, 0.4327668845653534, -0.6719274520874023, 0.44022467732429504, -1.270142912864685, -1.6397671699523926, 2.333156108856201, 1.8491404056549072, -1.5111572742462158, -0.171781986951828, -0.7787221074104309, -0.029550544917583466, 0.17051446437835693, -0.42059698700904846, 1.4482614994049072, -0.9951601028442383, -1.3538174629211426, 0.17974333465099335, 1.2721463441848755, -2.820021152496338, 0.6343653202056885, 0.06157997250556946, 0.7938839197158813, 1.081926703453064, -0.5161436200141907, 0.914392352104187, 0.14393064379692078, -0.06909192353487015, -0.582145094871521, -0.9120836853981018, 1.508118748664856, -0.7542171478271484, -0.1848614513874054, 0.573733925819397, -1.3926104307174683, -0.5341432690620422, -0.8017592430114746, 0.18641014397144318, -0.49669981002807617, -0.5005370378494263, -1.2133954763412476, -0.2926948368549347, -0.04060789570212364, 0.20228494703769684, -0.8435244560241699, -0.41350141167640686, 0.36753568053245544, 0.0397958979010582, -1.5812326669692993, -0.8365502953529358, 0.12678228318691254, 0.07907849550247192, 1.4830073118209839, 0.013588910922408104, -1.1394749879837036, -0.7795869708061218, 0.10577531903982162, -0.7969285249710083, -0.26568174362182617, -0.6146412491798401, 0.03864860162138939, 0.3104894757270813, -0.030257858335971832, 1.2502501010894775, -1.8404347896575928, 0.16118519008159637, 0.02282528206706047, 0.024472909048199654, -1.0442951917648315, -0.7045685052871704, 1.5588966608047485, 0.7733958959579468, 0.4796876311302185, 0.920204758644104, -1.364351749420166, 1.1156662702560425, -1.3695870637893677, -1.1532862186431885, -0.3414125144481659, 0.9136188626289368, -1.0553842782974243, 0.8319299221038818, 1.6375080347061157, 1.0857617855072021, 0.32747286558151245, -1.2249706983566284, -0.5803651213645935, -0.9739200472831726, -0.2753080427646637, -0.22169065475463867, -0.6206457018852234, 1.5845060348510742, 1.563896656036377, 0.290560245513916, 0.04100065678358078, 0.7142095565795898, 0.11050824075937271, -2.369741439819336, -0.2878890931606293, 0.04851808771491051, 0.8489458560943604, 0.8498520255088806, -1.3896700143814087, -1.3116588592529297, -0.8946686387062073, 0.0055274274200201035, -2.4649384021759033, 0.6985729932785034, -0.7362072467803955, -1.331173062324524, -0.8730186820030212, 0.9428954720497131, -0.8643147349357605, -0.7583707571029663, -0.9587639570236206, 0.11281324177980423, 0.31346631050109863, -0.4508599638938904, 0.7565551996231079, 1.0966023206710815, 0.6878781318664551, 0.39838728308677673, -0.23751524090766907, -0.48932939767837524, 1.4308993816375732, -0.32397958636283875, -0.08766931295394897, 2.1327221393585205, -0.17788361012935638, 0.24743147194385529, -0.18272937834262848, 1.0996531248092651, -0.7318033576011658, -0.013239964842796326, 0.8428524136543274, -1.6461900472640991, 0.6811447739601135, -1.3209364414215088, 0.1891210377216339, 1.5098371505737305, -0.48156753182411194, 1.2130540609359741, 1.755834698677063, -0.19619250297546387, 1.1820896863937378, 0.0734281837940216, 0.3419162333011627, -0.6556059122085571, -0.8934639096260071, -0.10031826049089432, 0.7139267325401306, -1.2345240116119385, -1.0359339714050293, 0.6776043772697449, 0.6066852807998657, 0.20933684706687927, -0.8725513219833374, 0.6254077553749084, 0.2773534953594208, -1.5800155401229858, -1.2887343168258667, 0.7723346948623657, -0.5638163089752197, 1.2228727340698242, 1.7631734609603882, -0.9550921320915222, 1.9628636837005615, -0.5454713702201843, 0.008268055506050587, 0.9108070731163025, 1.5523098707199097, 0.14140282571315765, 0.3187463879585266, -0.7806751132011414, 1.1564339399337769, 0.9344583749771118, 0.9479121565818787, -1.4226094484329224, 0.7882772088050842, 0.7183764576911926, 1.239113450050354, -2.2338027954101562, 0.8151543736457825, -0.7810491919517517, 0.0384051539003849, -0.13305702805519104, -1.2292033433914185, 0.6793984770774841, 2.2960000038146973, 1.5955841541290283, 1.0937141180038452, 0.7778294682502747, 0.08755683153867722, -0.5299493074417114, -0.3219834566116333, 0.23550167679786682, -0.9716704487800598, -1.186239242553711, -1.5796679258346558, -0.8060291409492493, 0.44566893577575684, 1.6486107110977173, -0.9080609083175659, -1.5216789245605469, 0.029244793578982353, 0.013335887342691422, 1.3624234199523926, -0.23109766840934753, 1.0712521076202393, 0.7917563319206238, 0.6333984136581421, 0.34151342511177063, 1.237514853477478, -0.7208716869354248, 1.4324594736099243, -0.06036628410220146, 0.07779738306999207, -0.04031125828623772, -0.8604190349578857, 0.34835347533226013, -1.499324083328247, -0.5862991809844971, -0.9327945113182068, 0.3398352861404419, 0.6732407212257385, -0.9637230634689331, 2.3372833728790283, -1.4222334623336792, 0.08299577236175537, -0.5698367357254028, 0.5616409778594971, 0.3981066048145294, -0.24102017283439636, -1.4938292503356934, -0.39527347683906555, 0.3106262683868408, -0.3638739585876465, -0.13476309180259705, -2.410281181335449, 0.22157561779022217, 0.9800869226455688, -1.0131428241729736, -0.7070747017860413, 0.15211327373981476, -0.8247848153114319, 0.6466959118843079, 1.013898253440857, 0.16867084801197052, 0.36044037342071533, 0.7756431102752686, 0.18255934119224548, -1.2561266422271729, 0.3641361892223358, -1.1320691108703613, -0.40853357315063477, 0.721622884273529, 1.1686469316482544, 1.0352199077606201, 0.09575198590755463, -0.2317226231098175, -0.07428724318742752, 0.25742971897125244, 1.156644582748413, 0.5341574549674988, 0.7930722236633301, 0.477994829416275, 0.40647047758102417, -0.5511596202850342, 0.8848145008087158, -0.35289710760116577, 0.17759199440479279, -0.3293488621711731, -2.107595682144165, -0.0872669443488121, 0.5437832474708557, -1.35910964012146, 0.7355346083641052, 1.6163408756256104, 0.27552518248558044, 1.730930209159851, 1.728423833847046, -0.3650466203689575, -1.0048340559005737, -0.8177555203437805, 0.45259371399879456, -1.8317581415176392, 0.5471630692481995, 0.7674961090087891, -2.103851318359375, -1.054192066192627, 0.8762537837028503, 0.7484927773475647, -1.4786407947540283, -0.4097738564014435, 0.4562062919139862, 0.02397821471095085, -0.7517199516296387, -1.1471543312072754, 0.5335880517959595, -0.47768861055374146, 0.10169516503810883, 0.4958469867706299, 0.7972583174705505, -1.5000698566436768, 0.9618111252784729, 1.1942976713180542, -0.38335907459259033, 0.9196569919586182, -0.7501534819602966, 0.31042203307151794, 0.28237858414649963, 0.5913611054420471, -1.6642869710922241, 0.10144039988517761, -0.8611063957214355, -1.1637953519821167, -1.6903825998306274, -0.05851138383150101, -1.4208680391311646, 0.6406483054161072, -0.2859947383403778, 0.010912120342254639, 0.007131379563361406, -1.2817833423614502, 0.7702886462211609, -1.01692795753479, -0.8001323342323303, 0.11619091033935547, 0.18560950458049774, -1.4764081239700317, 0.7251982092857361, 0.3403889834880829, 0.20454765856266022, -1.4198147058486938, -0.5224799513816833, 0.23153451085090637, 0.082083560526371, -0.6755774021148682, -0.2100559026002884, 0.25239673256874084, -1.8894168138504028, -0.646405816078186, -0.6782200336456299, -1.5397372245788574, 1.0314699411392212, -2.1118335723876953, -0.6998741030693054, 0.312006413936615, 0.5726600289344788, 0.6440542936325073, 0.28575050830841064, -0.06342202425003052, -0.32847386598587036, 0.6831619739532471, 1.6823816299438477, -1.0832022428512573, -0.7572153210639954, -0.569341242313385, -0.11791062355041504, 1.270039677619934, 0.4195995330810547, 2.4598302841186523, -0.011684290133416653, -0.03785523772239685, 1.376857876777649, -1.1978967189788818, 1.1925098896026611, -0.1530984491109848, -0.9458968639373779, -0.5297178626060486, -0.32359224557876587, 0.1779966801404953, -1.5158957242965698, 0.9026845097541809, 0.010264032520353794, -1.0039159059524536, 0.19455227255821228, -1.5846765041351318, 0.8455109596252441, -0.6008337736129761, -0.851019561290741, 0.04783753305673599, -0.14570865035057068, 0.1600869745016098, 1.4601837396621704, -0.25389453768730164, 0.4951513409614563, -0.6599344611167908, -1.1152843236923218, 0.5078067183494568, -1.5281916856765747, -0.3721211850643158, -0.09176456183195114, 0.09730885922908783, 0.8237769603729248, 0.02767268568277359, -0.5530027151107788, -0.9073894023895264, 0.20725469291210175, 1.2020536661148071, 1.8252472877502441, -1.2192373275756836, 1.4053831100463867, -0.2522899806499481, 0.19130240380764008, -1.7963265180587769, -0.2890174984931946, -0.7484326958656311, 0.1657319813966751, -0.9563183784484863, 0.17836159467697144, 0.5903900861740112, -0.8892029523849487, 0.2576816976070404, 2.3632993698120117, -0.8478465676307678, 1.3468992710113525, -0.4019058346748352, 0.2956036329269409, 0.7469778060913086, 1.6068978309631348, -0.5005243420600891, 0.3396919071674347, 0.49111974239349365, 0.06895285844802856, -1.1132192611694336, -1.8171943426132202, -1.8331952095031738, 1.258976936340332, -0.607987105846405, -0.07593528181314468, -1.5136092901229858, 0.8776839375495911, 2.7680490016937256, -0.006897838786244392, 0.14136753976345062, -0.8268433213233948, 0.3131392002105713, 0.6072252988815308, 0.6191595792770386, -0.9044663310050964, 0.0613529197871685, -0.8773091435432434, 0.5272790789604187, 0.7717328071594238, -0.45600664615631104, -0.14766032993793488, 0.5699074864387512, -0.4515575170516968, 0.45070111751556396, -0.8175614476203918, -1.6397579908370972, 0.03917427733540535, -0.21872617304325104, 0.9572261571884155, 0.608443558216095, 2.5918617248535156, 0.8439708948135376, -0.6168424487113953, -0.6874656081199646, -1.9007114171981812, 0.49229663610458374, -0.4140772223472595, 0.11511602252721786, 0.5292184352874756, -0.6198113560676575, -0.544804036617279, -0.21539488434791565, 0.5676620602607727, 0.27455076575279236, -0.2622440457344055, -0.16969387233257294, 0.8474162817001343, 0.11545294523239136, -0.5286154747009277, -1.773543119430542, 0.39738035202026367, 0.061973053961992264, -1.7129098176956177, 1.237870693206787, 0.6939187049865723, 0.33438533544540405, -0.6721007227897644, 0.062070075422525406, 1.2321133613586426, 0.2802800238132477, 0.521921694278717, 0.2737044394016266, 1.7279586791992188, -2.5107498168945312, 1.2933261394500732, -0.8303359150886536, 0.023115815594792366, -0.7570978999137878, -1.9652776718139648, 1.2842305898666382, -2.2417266368865967, -0.14112283289432526, -1.5247365236282349, -0.002878612605854869, -0.032248515635728836, 1.8087769746780396, -0.6212125420570374, -0.26674285531044006, 0.36613738536834717, -1.441658854484558, -1.0516033172607422, -0.393963098526001, 1.811525821685791, -0.49610647559165955, -0.20126105844974518, -1.4533697366714478, -0.09907514601945877, 0.3738645911216736, 1.8782374858856201, -2.6378748416900635, -0.9366917610168457, -1.877910852432251, -2.4098193645477295, -1.1049529314041138, 0.8978478312492371, -0.7577036023139954, -2.287304401397705, -0.7214193344116211, 1.0065563917160034, -0.0491383895277977, 0.4610980749130249, 0.11397057771682739, 1.2388041019439697, 0.6139585375785828, -0.7300158739089966, -0.2034495621919632, 0.8469656109809875, -1.555885672569275, -0.9125674366950989, -0.24840278923511505, 1.0883448123931885, -0.5425571799278259, -0.16998609900474548, -0.12038867175579071, 0.78410404920578, -0.8968467116355896, -0.35484039783477783, -0.6727759838104248, 0.388239324092865, 0.2159082442522049, -0.2205125242471695, 0.44104427099227905, -0.8887530565261841, 0.6181566119194031, 1.161939024925232, -0.4850600063800812, -0.7374064922332764, 1.4436249732971191, -1.0165435075759888, -1.6962045431137085, -0.14960618317127228, 0.44924041628837585, 0.29077282547950745, -0.3184223473072052, -0.0958399772644043, 0.10926010459661484, 0.5730992555618286, 0.17393286526203156, 1.4468222856521606, 0.20428809523582458, -0.15509377419948578, 0.9289107918739319, -1.0404549837112427, 0.07728496193885803], [-1.1118049621582031, -0.7353610396385193, -1.4227625131607056, 0.607446014881134, 0.3193722665309906, -0.43797507882118225, -0.17586077749729156, 0.5086708068847656, 1.0009901523590088, -1.199338436126709, -2.05600643157959, 0.08186870068311691, -0.9317778944969177, -0.052908118814229965, -0.7639892101287842, 1.3029751777648926, -1.789670705795288, -0.21980543434619904, 0.07671292126178741, -0.5663526058197021, -0.33696505427360535, 0.9636523127555847, 0.7223472595214844, -0.033754002302885056, 0.7666631937026978, -1.4431178569793701, -0.0012283872347325087, 0.3072841763496399, -1.0706536769866943, -0.956121027469635, -0.03424064442515373, 0.33060961961746216, 0.11337918788194656, 0.40489381551742554, 0.3950735926628113, 0.22747637331485748, -0.0895509198307991, -0.6673805117607117, -0.5181664824485779, 0.044590871781110764, 0.06016075983643532, -0.43620026111602783, -1.1943693161010742, -0.899815022945404, -0.6466013193130493, -0.29332348704338074, -0.5221445560455322, 0.3368062973022461, -0.13947747647762299, -1.5343009233474731, -0.2191113829612732, 0.21270953118801117, -0.009266442619264126, 0.6047654151916504, 0.5833562016487122, 0.9883265495300293, -1.1763890981674194, 1.820202350616455, -1.0579845905303955, 0.6050256490707397, -1.0413613319396973, 0.9805266857147217, 2.3726141452789307, 0.5323537588119507, 2.281269073486328, -1.1745572090148926, -1.2928539514541626, 1.5448294878005981, -0.8516146540641785, 0.32806116342544556, 1.2689552307128906, 2.666262626647949, 0.4032028317451477, -0.5256436467170715, 0.7146652340888977, 0.6559657454490662, 0.8275923728942871, 0.5246255993843079, -1.458176612854004, -2.082214832305908, 0.2870144844055176, 0.20951369404792786, 0.38234636187553406, 1.3719993829727173, 1.1365104913711548, -0.6500821113586426, 0.8035921454429626, 0.12342183291912079, -0.7884109020233154, 0.8836864233016968, 1.3398656845092773, -0.968506932258606, -0.4839029610157013, -1.5306119918823242, -0.6008164882659912, -0.6203649044036865, -0.4922502040863037, 0.27717384696006775, -0.06637079268693924, -1.5522210597991943, -1.8898701667785645, -0.5001703500747681, -1.5943808555603027, 0.5318495035171509, 0.8021987080574036, 0.09140002727508545, -0.45370522141456604, 0.5036238431930542, -0.34679511189460754, 0.305451363325119, -0.7369917035102844, 0.6266635060310364, 1.0653363466262817, -1.7364046573638916, 1.7014127969741821, 0.5459204316139221, 0.018949158489704132, 0.2361670881509781, 0.30753299593925476, 1.315625786781311, 1.0425866842269897, 0.5058766603469849, -1.2011805772781372, 0.6745821237564087, -1.0095367431640625, 0.06862244009971619, -1.221683144569397, -1.4619967937469482, -1.9325270652770996, -0.1490873396396637, -0.7462036609649658, 0.2057655155658722, 0.2812109887599945, -0.8201246857643127, 1.4869461059570312, -0.17802923917770386, 0.02058751881122589, 0.3570212423801422, -2.4521045684814453, -0.720305323600769, 0.7771812677383423, -1.3425401449203491, 0.006472283508628607, 0.25228193402290344, 0.4226287007331848, -1.7267510890960693, -0.7680688500404358, -0.3073561191558838, -0.5286037921905518, 1.4762941598892212, 0.42015206813812256, -0.17524950206279755, 0.5217678546905518, -0.3267988860607147, -1.0583661794662476, 2.0244834423065186, 0.6948733329772949, 0.2253386676311493, -0.043800510466098785, 0.9201340079307556, -1.0598338842391968, -0.7011087536811829, -0.46580740809440613, 0.318215548992157, -0.9095081686973572, -0.5874388813972473, 0.8137738704681396, 1.2331855297088623, -0.05938373878598213, 0.2568044364452362, -0.374153196811676, -2.446465253829956, -0.7744159698486328, 1.1136773824691772, 1.3817565441131592, -0.8604642748832703, 0.3009670078754425, 0.6196174621582031, 0.27398237586021423, -0.9400174617767334, -1.089654803276062, 0.12507542967796326, 0.13531582057476044, 2.1161065101623535, -0.49002066254615784, 1.101209282875061, -0.6180583238601685, -1.2047683000564575, 0.3269118070602417, -0.2590157091617584, 0.5354284048080444, 0.9578336477279663, 0.9458491802215576, 1.4571729898452759, 0.044639479368925095, -0.072685107588768, 0.2038792371749878, 2.2122535705566406, 0.4434986412525177, 0.8160019516944885, -1.0958466529846191, 0.8943811655044556, -0.13645149767398834, 0.7426347136497498, 0.9132848978042603, 0.9633949995040894, -0.061236537992954254, 0.7131638526916504, -1.0904797315597534, 0.3879014849662781, 1.1539781093597412, -0.2513623833656311, -0.21798701584339142, 0.17997170984745026, -1.097103476524353, 0.3119088113307953, -1.4830344915390015, -0.41868144273757935, 0.4271315634250641, -1.311414122581482, 0.8934561014175415, 0.18410827219486237, -1.7165486812591553, -0.02264132723212242, -0.5732383728027344, 0.4768439531326294, 0.7368028163909912, 0.422146737575531, -0.7047054767608643, -0.5756914019584656, -0.5541507005691528, -0.18629533052444458, 0.906420111656189, 1.2345740795135498, -0.6194753050804138, -0.1126282662153244, -2.492628335952759, 0.3217274844646454, -1.4974021911621094, -0.5537286996841431, -0.8933160305023193, -0.5704723000526428, -0.41296327114105225, -0.6723472476005554, 0.22721630334854126, 0.6200355887413025, -0.36687979102134705, 0.28165295720100403, -1.7980796098709106, -1.1593292951583862, 0.9327244162559509, -0.06515969336032867, -0.33707284927368164, -0.041402850300073624, 0.9866658449172974, -0.46339449286460876, -2.11272931098938, 0.3629370331764221, 0.9623817801475525, -1.2087846994400024, 0.06514333188533783, -1.7623491287231445, 0.5447107553482056, -0.5463951230049133, -0.10559063404798508, -0.41759374737739563, 0.35708746314048767, -1.7260617017745972, 0.01902882382273674, 0.49039754271507263, -1.050180435180664, 1.419862985610962, 0.16190752387046814, -0.1129663959145546, 0.2090030014514923, -0.6152137517929077, -0.53374844789505, -0.6200640201568604, 1.4509526491165161, 0.7315167784690857, 0.12310224771499634, 0.06158000975847244, 0.7883384823799133, -0.2785910964012146, 0.45407238602638245, -0.3216842710971832, 0.3844458758831024, -0.6319654583930969, 0.9299474358558655, -0.23949038982391357, -0.4048349857330322, 2.813096284866333, 0.5516615509986877, 0.5043401718139648, -0.2435779720544815, 0.9445319771766663, -0.7930615544319153, -0.4524032175540924, -1.4130990505218506, -0.40052348375320435, 0.7716001272201538, 0.8888729214668274, -0.6322725415229797, -2.0018808841705322, -1.0527549982070923, -0.31043604016304016, 0.005440654698759317, 0.005441522225737572, -0.5635161995887756, 0.30680274963378906, 2.3109798431396484, -0.944059431552887, 0.4957260191440582, 0.17934828996658325, 0.4093295931816101, -0.27419036626815796, 0.8211297392845154, -1.125878930091858, -0.8282729983329773, -0.5909066796302795, 1.4196232557296753, -1.2761693000793457, 0.131223663687706, 0.5733364820480347, -1.579911708831787, -1.3206610679626465, -0.7936452627182007, -1.5600117444992065, 0.23035521805286407, 1.2473559379577637, -0.25303950905799866, -0.2739129662513733, -0.5701056718826294, -1.8170256614685059, -0.1605031043291092, -0.602621853351593, -0.5493544936180115, -0.631395161151886, -1.3644589185714722, -1.2506924867630005, 0.3231441378593445, -0.6248276829719543, 0.6629050374031067, -1.3460791110992432, -0.742128849029541, -0.4629664719104767, 0.21252861618995667, -0.757063627243042, -1.4298683404922485, -0.1270674467086792, -0.46849241852760315, 0.4789575934410095, 1.8281043767929077, 1.302603840827942, 1.954330325126648, -1.6323425769805908, 1.0270684957504272, -0.02803005650639534, 1.1986149549484253, -0.7706345915794373, -0.49054503440856934, -0.7817252278327942, 0.2103053331375122, 1.116276741027832, -1.6167303323745728, -0.012856753543019295, -0.7506751418113708, 1.646872878074646, 0.7558472156524658, 3.614293336868286, -1.7292273044586182, -0.07102078944444656, 0.2342555671930313, -0.37825146317481995, 1.5495023727416992, -2.021946668624878, -0.2562631666660309, 1.3537863492965698, 0.8639322519302368, -0.20922653377056122, 1.032239556312561, -1.3786654472351074, 1.4843639135360718, 1.1649612188339233, 1.0813097953796387, 0.5732459425926208, -0.5146844983100891, 1.0006319284439087, -0.609889030456543, -0.020389143377542496, -0.1804826855659485, 1.04720938205719, -0.20058219134807587, -0.6194688677787781, 1.3501485586166382, 0.9164758920669556, 1.433171272277832, -0.7851384282112122, -0.9308531284332275, 0.5447943210601807, 0.4447125494480133, -0.04614429175853729, -1.045591115951538, 1.170058250427246, 0.8103673458099365, 0.8249891400337219, -1.1069929599761963, 1.1932809352874756, -1.2045762538909912, 0.5225210785865784, 0.993575394153595, -0.10223715752363205, 0.17314396798610687, -1.3462756872177124, 1.5260615348815918, -0.5190685391426086, -2.2770047187805176, 0.469401091337204, 0.6747241020202637, 0.4426451623439789, 0.06104603409767151, -0.9286196827888489, -1.749895691871643, -0.5155923366546631, 0.22656556963920593, 0.9621156454086304, -0.33708876371383667, 1.0309518575668335, 0.12199153751134872, 1.3847326040267944, -0.06740080565214157, -0.15433719754219055, 0.7345091700553894, -1.1067737340927124, -1.0673388242721558, 0.8737757205963135, 1.7019585371017456, -1.4375888109207153, 1.1860647201538086, -1.4158450365066528, -1.147131085395813, 0.45696914196014404, 2.522446393966675, 1.4282467365264893, -0.3474709391593933, 0.7241929173469543, 1.8774174451828003, 1.3430840969085693, -0.8062641620635986, -1.1140891313552856, -0.2130971997976303, 0.8555880188941956, -1.0739840269088745, -0.30331575870513916, 0.16108417510986328, 0.19340714812278748, 0.7326359748840332, -1.119232416152954, 0.4236223101615906, 0.37321579456329346, 0.3778170049190521, -0.45368510484695435, 0.7650325298309326, -1.1632866859436035, 0.3568277955055237, -0.45632562041282654, 0.4016529619693756, -1.129633903503418, 1.5543359518051147, 1.2838960886001587, -0.3504226505756378, 1.0141874551773071, 0.05964086204767227, -1.4021865129470825, -0.7276180386543274, -0.022532127797603607, 0.36460575461387634, -1.3953100442886353, -0.29889559745788574, -0.9512739777565002, 0.31222957372665405, 0.35717499256134033, 0.1076798215508461, -0.2942638695240021, 0.28239360451698303, 0.6995838284492493, 0.2986863851547241, 0.06381547451019287, -1.076703429222107, 1.2099673748016357, -0.6684353351593018, 1.1629207134246826, 0.6773950457572937, -1.2896995544433594, 0.7534627318382263, 0.25185835361480713, 0.6371204853057861, -0.24960069358348846, -0.7092557549476624, -0.023528756573796272, -0.12295902520418167, 0.5034772753715515, 0.4210355877876282, -0.1856977790594101, 0.2345181405544281, -0.031482186168432236, 0.3568945825099945, 0.26373037695884705, 0.8357738852500916, -1.5058504343032837, -0.060947950929403305, 0.35724419355392456, -0.42684659361839294, 3.1646361351013184, -1.3636853694915771, -1.304450511932373, 2.0760464668273926, 0.6523794531822205, 0.06561900675296783, -0.6002504229545593, 2.038609743118286, -0.690007209777832, -1.0718388557434082, -0.007313535548746586, 0.7594105005264282, 0.09639258682727814, 0.7949671745300293, -0.9258104562759399, -0.3336068391799927, 0.23739124834537506, 0.13981518149375916, -1.3181976079940796, -1.0420535802841187, -0.287326455116272, -0.8674833178520203, -0.6786569952964783, 1.722635269165039, 0.44384685158729553, -0.8778411149978638, -1.1229467391967773, 2.592381715774536, 1.6445997953414917, -0.386604368686676, -0.09011279791593552, 0.2689739465713501, 0.39582404494285583, 0.8911583423614502, -0.05665643513202667, 0.27909398078918457, 1.6244146823883057, -0.1295173168182373, -0.8763993978500366, -0.05635710805654526, 0.6913721561431885, 0.5705646276473999, -0.7594121098518372, -1.2381174564361572, -0.2543475925922394, 0.8768439292907715, -0.16424722969532013, 1.201606273651123, -0.7900735139846802, 1.362606406211853, 0.2236800193786621, -1.1613551378250122, -0.025222044438123703, 0.8938621282577515, 0.17116960883140564, -1.1300880908966064, -0.8098739385604858, -0.9875648021697998, 0.22187799215316772, 1.8040988445281982, -0.4704081416130066, 0.5783897042274475, 0.5033027529716492, -1.5268961191177368, -0.1025276780128479, -0.685275673866272, -1.9698989391326904, 0.7430716156959534, 0.8940654993057251, -0.3670564293861389, 0.14391572773456573, -0.40414100885391235, -0.19377990067005157, -0.6986350417137146, 2.618293046951294, 0.6819595098495483, -0.5994781851768494, 0.3999495506286621, 0.4484604299068451, 0.9434779286384583, 0.3818090856075287, -0.9608853459358215, -0.06369253993034363, 0.2367803454399109, -0.4218403995037079, -0.2655199468135834, 0.16949990391731262, 1.0686136484146118, 2.040675163269043, 0.3441047668457031, 0.08140844851732254, -2.8705708980560303, -1.5582853555679321, 2.5900685787200928, 0.5600128173828125, -0.17696039378643036, 3.4521939754486084, 0.308409720659256, 0.7662204504013062, -1.610026240348816, 1.3613371849060059, -0.7363054156303406, -0.8993391990661621, 1.0849299430847168, 1.668470025062561, -0.36959272623062134, 0.211943119764328, -0.6577189564704895, -0.13489286601543427, 0.7056273221969604, 1.18716561794281, -1.0957552194595337, 1.5439578294754028, 0.13821928203105927, 0.8892773985862732, -0.09456778317689896, -0.9019352793693542, 1.0235179662704468, 1.7196083068847656, 0.31944262981414795, -1.426358938217163, -1.4168131351470947, -0.3442464768886566, 0.8420328497886658, -1.035106897354126, -0.5868498086929321, 1.3519796133041382, -0.6191974878311157, 1.3080374002456665, -0.05975113809108734, -2.2921643257141113, -1.7083510160446167, 0.3836773931980133, 1.030516505241394, -0.03260388225317001, 1.2827337980270386, 1.037124752998352, 0.8110095262527466, 0.5488037467002869, 0.755439817905426, 0.315051406621933, 0.11456255614757538, 0.5420417785644531, -1.2767575979232788, 0.18713581562042236, 1.13313627243042, -1.3951849937438965, -1.5533512830734253, -0.8641473650932312, -0.5506129264831543, 0.29173406958580017, -0.1274801641702652, 0.17358329892158508, 0.5235456228256226, 2.8914177417755127, 2.1195170879364014, -0.7621533870697021, -0.750410795211792, 1.8043731451034546, 0.863416850566864, -0.875966489315033, -1.3174470663070679, -0.6972124576568604, 1.4296172857284546, -0.7356042861938477, -0.7669306397438049, 0.4753669202327728, -1.3063551187515259, -2.271704912185669, -0.17871719598770142, 0.7521144151687622, 0.6523144841194153, -0.3640909194946289, -0.2934283912181854, 0.49388331174850464, 0.7259252667427063, 0.40915167331695557, 0.5838856101036072, -0.6550851464271545, 0.6984615325927734, -0.5616801381111145, -0.2702152729034424, 0.8718583583831787, -0.014953108504414558, -0.01108534261584282, -1.2796145677566528, 0.5509651899337769, 0.7895538210868835, -0.7810702323913574, 0.8872363567352295, 0.824390709400177, 0.002096022479236126, -0.6118545532226562, 0.11612039804458618, -1.0317459106445312, 0.9267825484275818, 0.05754483491182327, -0.1899932622909546, -0.6236147284507751, -1.9398025274276733, 1.0850379467010498, -0.6497369408607483, -0.180532306432724, 1.556840419769287, 1.3005993366241455, -1.340086579322815, -0.35548821091651917, -0.07257260382175446, -0.22350344061851501, 0.4390174150466919, 0.16345015168190002, -0.5346090197563171, -1.0825949907302856, 1.0174628496170044, 0.6985533833503723, 0.0011145025491714478, -1.1118508577346802, -0.8776597380638123, -0.36375531554222107, -1.293449878692627, 0.36331936717033386, -0.489709734916687, -0.4895017743110657, 0.2769411504268646, -0.6766194701194763, 0.2808154225349426, 1.6903166770935059, 1.0747416019439697, 0.0304854828864336, -0.0766569972038269, 0.3235243856906891, -0.7495428919792175, 0.3846800625324249, -0.6171046495437622, 1.5699093341827393, 2.1701109409332275, -0.13895359635353088, -0.5192111730575562, -1.9612553119659424, -1.3036316633224487, -0.0466964989900589, -0.309026300907135, 0.5527421236038208, 2.1220028400421143, 0.13296334445476532, 1.1182612180709839, 1.2624454498291016, 0.5628970861434937, 1.3465070724487305, -0.5886648297309875, -0.21253247559070587, 0.2087392657995224, 1.030206322669983, 1.9542657136917114, -0.5442360043525696], [0.4031440019607544, -0.707862377166748, -0.8298385143280029, -1.1205354928970337, -0.6547960042953491, 0.13440746068954468, 0.8651196956634521, 0.030249174684286118, -0.4554361402988434, -0.8430349826812744, -0.5624987483024597, -1.8753700256347656, 1.489085078239441, 0.9718501567840576, -1.4170411825180054, -0.4115992784500122, -1.505219578742981, 0.08955730497837067, 0.14213064312934875, 1.398799180984497, -0.5442899465560913, -0.45184093713760376, 1.4521323442459106, -0.0223222728818655, -0.3263937532901764, -1.0055941343307495, -0.0043561263009905815, -2.0906646251678467, -0.7893864512443542, 0.3950846195220947, -0.5764362812042236, 0.7125585079193115, 1.1737515926361084, 1.3146601915359497, 0.9495786428451538, 0.3752846419811249, 1.4610345363616943, -0.662314772605896, -1.0117970705032349, 1.2256115674972534, 0.28660115599632263, 0.8291991949081421, 0.9900211691856384, 0.9289596080780029, -1.1801767349243164, -0.18362665176391602, -0.1652175784111023, 0.3225873112678528, 1.9054502248764038, -0.2635168135166168, 1.236761212348938, 0.9517610669136047, -0.7145252227783203, -0.24120986461639404, 0.2191789746284485, -0.0724855437874794, -1.2954109907150269, 0.43976837396621704, -0.31663256883621216, -0.9482461214065552, -1.220993161201477, -0.4788668155670166, 0.8836521506309509, 0.4531140923500061, -0.3931826651096344, 1.3422062397003174, -0.38859090209007263, -0.8207137584686279, 0.2479083389043808, 0.3121921718120575, -0.37305086851119995, -0.7097551822662354, -0.4603015184402466, 0.8926680684089661, 1.6047834157943726, -0.26093724370002747, -0.41237422823905945, -0.43146923184394836, 1.310072898864746, -0.1369360089302063, 0.36295250058174133, -0.5504385828971863, -0.020180558785796165, -0.2921435236930847, -1.0513628721237183, 0.6253662109375, 0.4048021733760834, 0.6092973947525024, 0.5089482069015503, 0.15300478041172028, 1.0712248086929321, 2.2949273586273193, -1.0243840217590332, -0.7946131229400635, 1.0690172910690308, 1.607144832611084, 0.1522797793149948, -0.839371383190155, -0.08830424398183823, -1.7386001348495483, 0.6213862895965576, 0.0006934241391718388, -0.10597635060548782, -1.4412496089935303, 1.714490294456482, 0.3830436170101166, -0.6304632425308228, -0.25156262516975403, -2.019368886947632, -0.1822487860918045, 0.4310045838356018, 0.7382609248161316, -0.8360212445259094, -0.49780383706092834, 0.30212122201919556, 0.5734325647354126, 2.8524575233459473, -0.2622378468513489, -0.22929896414279938, 0.5920355916023254, -0.6065828204154968, -0.8094111084938049, 0.613411009311676, -0.16356272995471954, -0.4885919392108917, -0.1559949666261673, 0.3110346794128418, -1.1619658470153809, 1.4783326387405396, 0.8109374046325684, 0.9327874779701233, 0.6804944276809692, 1.7551133632659912, -0.6534587740898132, -0.40841421484947205, -0.22160863876342773, 0.3871324360370636, -0.08512987196445465, 0.11623433232307434, 1.0677634477615356, -1.157839059829712, 0.8750774264335632, 0.022353772073984146, 1.740760087966919, -0.19199040532112122, 1.0820316076278687, -1.4716286659240723, -0.1934199184179306, 1.489234447479248, 1.0116713047027588, -0.6947486996650696, -0.2394818365573883, -0.22348712384700775, -1.227367877960205, 0.43257102370262146, 0.352222740650177, 0.6189720034599304, 0.11016949266195297, -0.7980598211288452, 0.7944583892822266, 0.28449758887290955, 1.1794742345809937, 0.8294121623039246, 0.6884897947311401, -1.0314390659332275, -1.0147231817245483, -1.2665597200393677, -0.4220482409000397, 0.06561350077390671, -2.063697576522827, 0.7110735774040222, -1.843684196472168, -0.7630359530448914, -0.27519819140434265, -0.7875118255615234, -1.4412426948547363, -0.003158841049298644, -1.9846761226654053, 0.7571138739585876, -0.10354629904031754, -0.1852659434080124, -0.27294689416885376, 1.6642611026763916, -0.007416278123855591, 0.6099370718002319, -0.9236070513725281, -0.1268165111541748, -0.7181277275085449, -0.3822128474712372, 0.6853033900260925, 0.290160208940506, -0.30382585525512695, -0.3715100884437561, -1.3039768934249878, -1.2874547243118286, 1.204906940460205, -0.4648889899253845, -0.5589531660079956, 0.4128452241420746, -1.3263323307037354, 1.1829856634140015, 0.006501778494566679, 2.1656055450439453, -0.5853173136711121, -0.46254852414131165, 0.5803070664405823, -0.09291341155767441, -0.058833882212638855, -1.1339190006256104, -1.2997394800186157, -0.6216516494750977, 1.1424431800842285, -1.4039409160614014, 0.631240725517273, 0.15632116794586182, 0.21256157755851746, 0.5102090239524841, -2.58520770072937, 0.5105905532836914, -0.36324650049209595, 1.140376091003418, 0.9142332077026367, 0.6948572993278503, 0.48860618472099304, -0.40312156081199646, 0.9180940985679626, 0.07353323698043823, -0.8625620603561401, -1.0953034162521362, 0.42016661167144775, -0.6156162619590759, -1.1399250030517578, -0.2735993564128876, -0.7399489879608154, 0.7496187686920166, -1.8693689107894897, -0.19028103351593018, 0.7714825868606567, 1.1693161725997925, -1.7990024089813232, -1.4728220701217651, -0.5577833652496338, 0.15687426924705505, -1.1296457052230835, 0.059099797159433365, 1.955579400062561, -0.5759612917900085, 1.4600774049758911, -0.27849072217941284, 0.03052065707743168, -0.020354395732283592, 0.5310627818107605, 1.319838285446167, 1.3072630167007446, 0.8105338215827942, -1.8182648420333862, 1.1469155550003052, 0.6001983284950256, 0.7692679166793823, 0.41771024465560913, -1.114949345588684, 0.3999342620372772, -0.40040117502212524, -0.5856550931930542, -1.0269720554351807, 1.693831205368042, 0.09601747989654541, 0.9734287858009338, -0.20882971584796906, -1.3404654264450073, 0.8567607998847961, 0.5980794429779053, 0.4496498703956604, 0.47277477383613586, -0.40076717734336853, 3.0029685497283936, -1.2971265316009521, -1.0962462425231934, 0.4558817744255066, -1.6667133569717407, -0.1308428794145584, -0.10859359800815582, -0.9538829326629639, 1.7098394632339478, -0.24032723903656006, 1.8991700410842896, 1.537750244140625, 0.6841764450073242, -0.6834844946861267, -1.260324239730835, -0.7489315867424011, -0.47142714262008667, -1.2381941080093384, -0.7074543237686157, 0.0484643317759037, 0.05590956285595894, -0.46369123458862305, 0.325928270816803, -0.6969454884529114, -1.6384203433990479, -1.1001567840576172, -0.2141086757183075, -1.0190941095352173, -1.896899700164795, -0.9380028247833252, -0.45938652753829956, -0.508266806602478, 1.1095361709594727, 0.37254664301872253, 1.972588062286377, 0.3642459213733673, -0.2077157199382782, 1.2650471925735474, 2.0943214893341064, -1.693266749382019, -1.6124725341796875, 0.5657249689102173, -0.4826992452144623, -1.2016383409500122, 0.6060646772384644, -0.8541797399520874, 1.667012095451355, -0.9127965569496155, -0.3347601592540741, 0.6517671942710876, -0.21158809959888458, -0.5268287062644958, -0.04698505625128746, 0.6098983287811279, 0.10478033125400543, -0.6564635038375854, -0.5224271416664124, 0.606768786907196, -0.7504668235778809, -0.5758365392684937, 0.007490001153200865, 0.8283060789108276, 1.5216306447982788, 0.11363541334867477, -0.24971817433834076, 0.6822963356971741, 1.6795613765716553, -0.9439765214920044, -0.544854998588562, 0.7017839550971985, -0.34159189462661743, -2.12953519821167, -1.1768453121185303, 0.3124004006385803, 1.0444284677505493, -0.2131626307964325, -1.9669086933135986, -0.46737682819366455, 0.15891636908054352, -0.5030697584152222, -0.5628652572631836, 0.4778842329978943, 0.8658263683319092, -0.48468026518821716, 0.4028983414173126, 0.593903660774231, 0.8611577153205872, 0.062138307839632034, 0.4707735478878021, 0.8395698666572571, 1.1947740316390991, -0.38336604833602905, 1.8552802801132202, 0.2977396845817566, 1.8094862699508667, -1.7750593423843384, 1.3049581050872803, -0.6449785828590393, -0.17611382901668549, -0.0812964141368866, -0.15201129019260406, 2.574730396270752, 0.757869303226471, -0.34690147638320923, 2.0110273361206055, 1.1632875204086304, -0.8558839559555054, -1.937896490097046, 1.205411434173584, -0.8590965270996094, 0.37839287519454956, 1.213863730430603, 0.505931556224823, -0.09077224880456924, 1.2743656635284424, -1.1508961915969849, 2.128662586212158, 0.6748195290565491, -1.0310031175613403, 1.3220207691192627, 1.050255298614502, -0.7253952622413635, 0.9871940612792969, -0.2623586356639862, 0.6188302040100098, -0.2777787148952484, -1.5015912055969238, -0.5640962719917297, -0.14734649658203125, -1.5167025327682495, 0.47062280774116516, 0.49155083298683167, 1.2234631776809692, 1.3494031429290771, 0.16848206520080566, -0.7729305028915405, -0.26558834314346313, -1.8192824125289917, 0.8623709082603455, -1.1272029876708984, 0.6923019289970398, -0.16995780169963837, 0.607793927192688, 1.2764517068862915, -1.135056972503662, -0.07013319432735443, -0.07375284284353256, -1.0412946939468384, 1.1987143754959106, 0.10636831820011139, 0.15402273833751678, 0.36973121762275696, -1.7318084239959717, 3.5729434490203857, -0.9942511916160583, -0.6351552605628967, -1.6645277738571167, 1.4795764684677124, -1.9572901725769043, 0.008400187827646732, -1.110998272895813, -0.9838542938232422, 1.5113776922225952, -0.4143884778022766, -0.9178705215454102, 0.6224117875099182, 0.36660441756248474, 1.1255747079849243, -0.06529072672128677, 0.33086714148521423, -1.0793492794036865, -0.629979133605957, -1.670985460281372, 0.26121819019317627, 0.038181502372026443, 0.831343412399292, 0.27731820940971375, -0.3299287259578705, -0.14823435246944427, -1.0183472633361816, -1.0286041498184204, -0.977216899394989, -0.7097839713096619, -2.155625581741333, 1.6913477182388306, 1.4989500045776367, 1.793041467666626, -0.09300047159194946, 1.4349095821380615, -1.246995449066162, -0.7894993424415588, -1.4727343320846558, -0.9039158225059509, 0.20629890263080597, -1.6483049392700195, 1.448606252670288, -0.9371486306190491, 0.4065588116645813, 0.530750572681427, 1.2335907220840454, 0.846089780330658, -2.395156145095825, 0.23043490946292877, 0.11589327454566956, -2.589313507080078, -0.07410187274217606, -0.4171511232852936, -0.37215331196784973, 0.26612260937690735, -1.4491119384765625, -0.07190058380365372, 0.679121732711792, -0.9057595133781433, -0.7622585892677307, 1.7541640996932983, -1.0576238632202148, -0.5087571144104004, -0.6608886122703552, -0.13001061975955963, 0.4971928596496582, 0.6516899466514587, -1.9401788711547852, 1.2789669036865234, -1.6825586557388306, -1.26776921749115, -1.9603155851364136, -0.16054761409759521, -2.4374067783355713, 0.6516562700271606, -1.7066073417663574, 0.8480719923973083, 0.31799277663230896, -0.3548746109008789, 0.5960139036178589, -0.7678563594818115, 2.095127820968628, -0.189533069729805, 0.1874852329492569, 0.8239834904670715, 1.9702847003936768, 2.399564743041992, -0.07548707723617554, 0.13456566631793976, 0.7122303247451782, -0.44353151321411133, -0.19031940400600433, -0.9005047082901001, -0.06887679547071457, 0.6141993999481201, -1.4404053688049316, 0.7209489941596985, 0.23583745956420898, -0.3454717695713043, -0.9640341997146606, 1.1942241191864014, -0.6712746024131775, 1.3893522024154663, -1.183454155921936, -0.40388575196266174, -0.08279253542423248, 2.1272876262664795, -0.7008686065673828, -0.03747273609042168, -0.8482251167297363, -0.633432149887085, 0.776980996131897, -2.022000312805176, 1.8423789739608765, 0.15839864313602448, -0.3003595471382141, 0.8314390182495117, 0.8690986037254333, 1.2128719091415405, -0.821382999420166, -0.8177534341812134, -0.29623228311538696, -0.38167646527290344, 1.0033961534500122, -0.606447696685791, 0.40213730931282043, -0.47146257758140564, 0.6775426268577576, -0.2093953639268875, 0.41440948843955994, 1.2409919500350952, -0.04593497887253761, -0.07835282385349274, 1.797147274017334, -1.937063455581665, 1.8133779764175415, 0.9623357057571411, 0.8314774036407471, 1.4103612899780273, -0.7765142917633057, 0.0386522077023983, 2.1514317989349365, 0.3251650333404541, -0.6717984080314636, 0.14002388715744019, 0.41880714893341064, -0.5547805428504944, 0.8252148628234863, -0.07278573513031006, 0.7568846940994263, 0.7589089870452881, -0.7299049496650696, -0.7367501854896545, -0.5068906545639038, 2.66607928276062, -0.22933369874954224, 1.2415268421173096, 0.11543762683868408, 1.3056650161743164, 0.04071991518139839, -0.03172193095088005, 0.6569221019744873, -0.17194746434688568, -1.2501389980316162, 1.0546483993530273, 0.8947708010673523, -0.2280394583940506, -0.5228016972541809, 0.3492542505264282, 1.5023328065872192, 0.58153235912323, -1.317583680152893, -0.08085278421640396, -1.0654102563858032, -0.03770429268479347, 0.07156509906053543, -0.06449019908905029, 0.6475527882575989, 0.9608264565467834, 0.524067759513855, -0.21397864818572998, 0.1823568046092987, 0.08443338423967361, 0.0764668732881546, 0.7261767983436584, 0.8000129461288452, -0.19273006916046143, 1.0960530042648315, 1.0845911502838135, -0.7881680130958557, 0.1743943840265274, 2.998883008956909, 0.08905918151140213, -0.8939439058303833, -0.334039568901062, -1.3864879608154297, -1.7643860578536987, 1.500712513923645, 1.2832647562026978, -0.35230323672294617, -1.155503749847412, 2.007375478744507, -1.0760115385055542, -1.716174840927124, -0.1284884810447693, -0.8060537576675415, 0.7185952067375183, 0.10743119567632675, 0.8240987062454224, 0.12273862957954407, -1.0729345083236694, 1.2321439981460571, -2.136958360671997, -0.46697306632995605, 0.12844125926494598, -1.6173100471496582, -1.3133020401000977, -1.51344895362854, -0.3697076439857483, -1.1593207120895386, 0.15792977809906006, 0.6644602417945862, -0.2408376783132553, -0.4087587893009186, 2.4786288738250732, 1.233076810836792, -0.39114630222320557, -0.1464088261127472, -1.1451101303100586, 0.16194520890712738, -0.2915206551551819, 0.294994056224823, -0.7552273869514465, 1.489066243171692, -0.8399326205253601, -1.1710596084594727, -1.4059760570526123, -1.1827669143676758, -0.41575056314468384, -1.4062565565109253, 0.5414227843284607, -0.06459973752498627, 0.8645175695419312, -2.4320573806762695, 0.6327763199806213, -3.055844783782959, 0.26586443185806274, -0.8054555058479309, 0.677257776260376, -0.022539446130394936, 1.5119578838348389, -2.20172119140625, -0.8626549243927002, -0.07764196395874023, -0.09380043298006058, 0.00617347564548254, 1.4745752811431885, 1.0892044305801392, -0.3344314396381378, -0.9068212509155273, 1.26632821559906, -0.41687747836112976, -0.5000651478767395, -0.08518567681312561, -0.38341644406318665, 0.7689140439033508, -0.4816039502620697, 0.7279418706893921, 0.4179867208003998, 0.3150297999382019, 0.8391689658164978, 0.5881949663162231, 0.3792237341403961, 0.21339601278305054, 1.1013954877853394, -1.1233081817626953, 2.257779598236084, -0.057154010981321335, -0.28811219334602356, 1.7570233345031738, 1.5447804927825928, 0.24863971769809723, 1.102623701095581, 1.3372843265533447, 0.5496292114257812, 2.2426605224609375, 1.0392817258834839, 0.36557406187057495, 1.4267617464065552, -0.9801437854766846, 0.9253385066986084, 1.0696529150009155, -1.698138952255249, 0.1109447330236435, 0.3302401602268219, -0.8163146376609802, 0.9100637435913086, -0.3099638521671295, 0.6237466931343079, 0.4506239891052246, 0.6342830657958984, 0.4184451997280121, -0.20052021741867065, 0.624748706817627, 1.4427729845046997, -1.5367273092269897, -0.41251030564308167, -0.35913360118865967, -2.7038867473602295, 1.1857770681381226, -0.3058924078941345, 0.03897726535797119, -1.0059471130371094, -0.777526319026947, 0.7594493627548218, 0.7463433742523193, 0.43762272596359253, -0.6782824993133545, -0.1692737638950348, -0.8015658855438232, -0.2683233618736267, 1.3524889945983887, 0.507909893989563, -0.5219687819480896, 0.4393409788608551, -1.415230393409729, -0.6818138957023621, 0.42173272371292114, 0.5010182857513428, -0.2812132239341736, 0.9772287011146545, 1.2222874164581299, 0.34540697932243347, 0.4301384687423706, 0.16361774504184723, -1.0352598428726196, -0.868037223815918, -0.44701525568962097, 1.1626991033554077], [-0.25870123505592346, 0.5937132835388184, -0.873320996761322, 0.08190758526325226, -0.2133139967918396, -0.4782675802707672, 1.5456053018569946, -0.9125567674636841, 1.2059013843536377, -0.12612728774547577, -1.0215587615966797, -0.4893253743648529, 0.08885306119918823, -1.3273876905441284, -0.8166413307189941, 0.055371999740600586, -1.596055507659912, -1.606147050857544, 1.442839503288269, -0.9549368619918823, -0.7809525728225708, 2.410271406173706, -0.07432255148887634, 0.345103919506073, 0.20253747701644897, 0.1183856725692749, 1.596042275428772, 0.24039511382579803, -1.6573561429977417, -0.042533401399850845, -0.8165196776390076, -0.5098713636398315, 0.7609427571296692, -1.5133048295974731, 1.7427858114242554, -0.5781856179237366, -0.6567172408103943, 1.2968531847000122, -0.22239728271961212, 0.5639740824699402, 2.018057107925415, 0.9715737700462341, 0.04906996339559555, 0.6848325729370117, 0.7743726968765259, -0.8783493041992188, -0.010364391840994358, -0.6934624910354614, 0.6320441365242004, -0.5669820308685303, 1.2418856620788574, -1.3876880407333374, 0.4722394645214081, 1.2452505826950073, -0.915067732334137, -2.159640073776245, 0.5544635653495789, 1.3075783252716064, 0.4085690975189209, -0.31704139709472656, -0.025255350396037102, 1.1659917831420898, -3.258673667907715, 1.3305305242538452, 0.15165461599826813, 0.24214966595172882, 0.7410734295845032, 0.8390903472900391, -1.1600075960159302, 1.1275711059570312, -1.8064037561416626, 0.3042362332344055, -0.576093316078186, -1.0893688201904297, 0.03173661231994629, -2.882242202758789, 2.841658592224121, -0.21175327897071838, -0.8011131286621094, -0.6844363808631897, 0.4222376048564911, 0.037149541079998016, 0.3682962954044342, 0.3438107967376709, 0.3421165347099304, -1.2387752532958984, -0.4104441702365875, 0.006259786896407604, -1.5831668376922607, -0.9531621932983398, -0.9722024202346802, 1.9313416481018066, 1.3251265287399292, 0.511792778968811, -0.5615286827087402, 1.3973743915557861, -0.06652718782424927, 0.011117242276668549, -0.8368919491767883, -0.09537317603826523, 0.07557235658168793, -1.2481882572174072, -0.3463020920753479, 1.6758837699890137, 0.31557363271713257, 0.18351104855537415, -2.370980739593506, 0.6781191825866699, -0.4566473066806793, -1.5140635967254639, -1.4677690267562866, 1.8990992307662964, -0.08077745139598846, 0.5887555480003357, 0.3845207095146179, 0.6661741137504578, -0.4059476852416992, 0.6203743815422058, -1.3208680152893066, -1.8764269351959229, 1.62021005153656, -0.7069939374923706, -0.9545174837112427, -0.7008717656135559, 0.7362011075019836, 1.0326787233352661, -2.202242851257324, -1.5371341705322266, -0.3112952411174774, 0.10545442998409271, 0.7574728727340698, 1.890376329421997, -0.7734553813934326, 1.660508632659912, -0.40527138113975525, 1.5676051378250122, -0.7209383845329285, -2.227057695388794, 1.433327555656433, -0.41659635305404663, 0.2961597740650177, 0.3172915279865265, 1.4015096426010132, -0.5597562193870544, 1.7967960834503174, 0.07811546325683594, -1.5867503881454468, -0.9903452396392822, 0.48849084973335266, 1.9931206703186035, -0.5604302287101746, -1.5993437767028809, -0.28068313002586365, 1.9739456176757812, 1.4220229387283325, 0.2837703824043274, -0.15809962153434753, -0.6375107765197754, -0.19540748000144958, -0.8987412452697754, -0.7719702124595642, 1.3694860935211182, -2.0242559909820557, 1.049901008605957, -0.26262742280960083, -0.1817859560251236, 0.30973106622695923, 0.8549445271492004, 0.3459095358848572, -0.2755248546600342, -0.5365145802497864, 1.3101192712783813, 1.159070372581482, 0.3035542964935303, 0.9491690397262573, 0.4375591278076172, -0.5719471573829651, 1.4786750078201294, 1.3774958848953247, -0.12806850671768188, -0.41905930638313293, 2.243910789489746, -1.7120996713638306, -1.9392924308776855, -1.582857608795166, 0.053342554718256, 1.0545361042022705, 1.053573489189148, -0.05247711390256882, -0.08652151376008987, -0.17752605676651, -0.4498941898345947, -0.5763172507286072, -0.08924002945423126, 1.2154837846755981, -1.2118843793869019, 0.9052709341049194, 0.3406527042388916, 1.5432378053665161, -0.8583043813705444, 1.684341311454773, -0.09368958324193954, -1.4684005975723267, -0.06061510741710663, -0.3364771604537964, -0.6783344149589539, 0.49634337425231934, -0.33960291743278503, 2.0121476650238037, 0.4472784399986267, -1.115885853767395, -0.2678423821926117, 3.423760175704956, 0.6426331400871277, 0.7255212664604187, 1.359531044960022, 0.4395625591278076, -0.31607386469841003, 0.09788594394922256, 0.23638592660427094, -1.1755142211914062, 0.01767808012664318, 0.6578450798988342, -0.4252937138080597, -1.2503677606582642, 0.5529675483703613, -0.8698291182518005, -2.336186170578003, 0.5222419500350952, 0.45375847816467285, -1.4161399602890015, -1.4996169805526733, 0.12967759370803833, 0.5187206268310547, 0.5155349969863892, 0.7571266293525696, -2.617431879043579, -0.38168489933013916, 1.1396533250808716, -1.4580998420715332, 1.5554307699203491, -0.015964804217219353, 0.4432933032512665, -0.6091143488883972, -0.03367951884865761, 0.8314130306243896, 0.824661910533905, 0.594443678855896, -0.28093287348747253, -0.3300477862358093, 1.9657522439956665, -0.2887618839740753, -0.9304395318031311, -0.027949245646595955, 0.030155198648571968, 0.31033438444137573, 0.0917411521077156, -1.0183846950531006, -0.5180274248123169, -1.0214194059371948, 0.10616297274827957, -1.2253636121749878, 0.8209788203239441, 1.5209622383117676, 0.9495744705200195, 1.4721260070800781, 0.5916654467582703, -0.6141781210899353, 1.7135759592056274, -0.3950743079185486, 0.6636720895767212, 0.4597664177417755, 0.21405428647994995, -0.13792096078395844, 0.836903989315033, -1.2101640701293945, -0.6922736167907715, -2.262162446975708, -0.3699013292789459, 0.8341326117515564, -0.40144214034080505, -2.1904590129852295, -0.7395092844963074, 1.1050618886947632, -0.42032888531684875, 1.4971026182174683, -0.3777318596839905, 0.5641431212425232, -0.6182277202606201, -3.3690505027770996, -1.0641765594482422, 0.7563896179199219, -1.6854077577590942, -1.1556240320205688, 0.7119712829589844, 0.057233888655900955, -1.3513376712799072, -0.07874299585819244, -0.062481991946697235, 1.7461185455322266, -0.18762585520744324, 1.6998779773712158, -0.0954853892326355, -1.1048046350479126, 0.9704805612564087, -0.8919624090194702, -0.5375480651855469, 1.052635669708252, -1.3479562997817993, -0.7629420161247253, -1.380428671836853, 0.6736414432525635, -1.6477340459823608, -0.23268163204193115, 0.568246603012085, -0.09709805250167847, 0.0021693790331482887, 0.10921847075223923, 0.38059940934181213, -0.9095215797424316, -0.8740195631980896, -0.5043206810951233, -0.8678144812583923, -0.8171334266662598, -0.3651125133037567, -1.1298835277557373, 1.3569356203079224, 0.6443467140197754, -0.6408736705780029, -0.7227927446365356, -0.8602641224861145, -0.592067539691925, -1.6611948013305664, -1.5591120719909668, -1.4900119304656982, -0.6916226744651794, -1.3889349699020386, -1.002926230430603, 0.8530969619750977, -0.12906989455223083, 0.7454469799995422, -1.2147389650344849, 1.0283938646316528, 0.6941471695899963, 0.044034913182258606, 0.0848814994096756, 1.049865484237671, 0.7328077554702759, -2.6061360836029053, -2.0682947635650635, -1.1691060066223145, -0.771391749382019, -0.08233099430799484, -1.4358772039413452, 3.1097164154052734, 1.0173648595809937, 1.6474320888519287, 0.05246003344655037, 0.6459296941757202, 0.4604215919971466, -0.5024461150169373, 1.3363118171691895, 0.9105977416038513, 0.5419209599494934, 2.3546555042266846, 0.8758307695388794, -0.3881210684776306, 0.086414635181427, 1.3918941020965576, -1.4728697538375854, 0.9409118890762329, -1.733573079109192, 1.2464344501495361, 2.0904619693756104, 0.2049132138490677, -0.3048647344112396, -1.3298709392547607, 1.7016706466674805, 1.511968731880188, -0.0012956570135429502, -1.1580923795700073, -1.7555845975875854, 0.5957924723625183, -0.6380584239959717, 0.42333218455314636, 1.506081461906433, 0.20433883368968964, 1.7147700786590576, 2.153157949447632, -0.5035732984542847, -0.6129745841026306, -1.4188873767852783, 0.754630446434021, -0.6827892661094666, -0.5998035073280334, -0.2633449137210846, 1.1009947061538696, 1.354844570159912, 0.7211103439331055, 0.9055929780006409, -1.6110715866088867, -0.22874359786510468, -0.6449229121208191, 0.5002416968345642, 0.9360300898551941, -1.7084121704101562, -0.29114437103271484, -1.127689242362976, -0.40417855978012085, -0.08252429962158203, 0.21355277299880981, -0.5292931795120239, -0.16312064230442047, 0.0637899711728096, 0.32226234674453735, -0.36793336272239685, 0.375544935464859, 0.5791478753089905, 0.3922126591205597, -0.05612018331885338, 1.200588345527649, 1.888170599937439, -1.1148849725723267, 0.1889285296201706, -0.39806658029556274, -0.3530212938785553, -0.3825890123844147, -0.4466113746166229, 0.5568805932998657, 0.5190579295158386, -0.1713515669107437, 0.9842765927314758, 0.1465529203414917, -0.8557980060577393, 0.8060261011123657, 0.0990464836359024, 0.4471052289009094, -0.7507067322731018, 1.4753676652908325, 1.0172979831695557, -1.5350421667099, -0.044750992208719254, -1.0746848583221436, -2.5884554386138916, -0.5421274900436401, -0.27518463134765625, -1.2288072109222412, 1.4687734842300415, -0.4818039536476135, 0.849947452545166, -0.1858615279197693, -0.5688363909721375, -0.6165891289710999, 0.32505902647972107, -0.5752682685852051, 0.07966268807649612, -0.28297993540763855, 2.487448215484619, -0.20930324494838715, -0.915683388710022, 0.5294076204299927, -0.8415471911430359, 0.12666882574558258, -1.1195635795593262, -2.147306203842163, -1.1343116760253906, -0.6124444603919983, -0.4502372145652771, -0.306327223777771, -0.34451359510421753, 1.3385790586471558, 1.2708271741867065, -0.9346713423728943, 0.7938745617866516, -1.4434504508972168, 0.16392956674098969, 0.7220248579978943, 0.10992381721735, 0.3180813491344452, -0.028767280280590057, -0.4257197678089142, -0.7946216464042664, -0.07543114572763443, -0.13362625241279602, -1.1613013744354248, 0.535971462726593, 0.15756019949913025, -1.059444785118103, -0.39530107378959656, -0.6019068360328674, -0.8379858136177063, -1.2960201501846313, -1.4289472103118896, -0.814934253692627, 1.9481518268585205, 2.8910961151123047, -1.0821410417556763, -0.1961241513490677, -0.3099910616874695, 0.9045627117156982, -1.3208270072937012, -0.18295840919017792, -0.9056520462036133, 0.00695306621491909, -0.28166666626930237, 0.6480841636657715, -0.5380723476409912, 0.45613762736320496, -1.0479258298873901, -2.3925533294677734, -0.5510473847389221, -0.48605650663375854, -1.145526647567749, -0.38773512840270996, -0.01038823276758194, -0.5671594738960266, -0.7434185147285461, -1.4310551881790161, 0.39363333582878113, -1.385802984237671, 0.9031692743301392, -0.869725227355957, 1.8341810703277588, 0.4005454480648041, -0.9722712635993958, -1.2234392166137695, 0.8988863825798035, -1.5089452266693115, 2.398181676864624, -0.7694394588470459, 0.31040459871292114, 0.5723942518234253, -1.327015995979309, -0.035692691802978516, -0.019174233078956604, -0.38278624415397644, -0.9269840121269226, -0.10609991103410721, -1.137082576751709, -0.2871444523334503, 0.3501172363758087, 0.20935183763504028, -0.9663482308387756, -1.1453014612197876, 0.2545483410358429, -0.7926793098449707, -0.6138224005699158, -0.02648378163576126, 0.5255911350250244, 0.22886434197425842, -0.24059432744979858, 1.5048706531524658, -0.38491374254226685, -1.3844282627105713, -0.048897501081228256, 0.19334524869918823, 0.4665389657020569, -1.2410551309585571, 0.39117974042892456, 1.7971656322479248, 1.1523476839065552, 0.8790443539619446, 0.6269077658653259, -0.6973565816879272, -0.26350677013397217, -0.6311144828796387, 0.34536212682724, -0.40534311532974243, -0.15637899935245514, -0.19812388718128204, 0.8324515223503113, 0.5107769966125488, -0.8202090263366699, -0.6297902464866638, 0.19647465646266937, -1.0249831676483154, -1.061331033706665, -1.177026391029358, -0.6932167410850525, 0.5099368691444397, -0.3649374842643738, 1.1003979444503784, 1.1702719926834106, -0.21052393317222595, 0.27474871277809143, 0.2712399363517761, -0.8608877658843994, -0.19073446094989777, 0.3936315178871155, -0.3211377263069153, -0.7548539638519287, 0.6205253005027771, -0.30815669894218445, -0.10962370038032532, 0.8157958388328552, -0.16563470661640167, -0.05943454056978226, 1.132871389389038, -0.8075392246246338, -1.8099114894866943, 1.9839110374450684, 0.9223659038543701, 0.48471203446388245, -2.018791675567627, -0.7094323635101318, -1.4379565715789795, -0.48143136501312256, -1.3301963806152344, -0.3132348954677582, 0.1814950406551361, -0.30809807777404785, 0.27341708540916443, -1.90227472782135, 2.4533324241638184, 1.5125513076782227, -0.4473913908004761, -0.6667222380638123, -0.40800341963768005, -0.4634552299976349, 1.4283901453018188, 1.5143437385559082, -0.35422199964523315, 0.12577643990516663, -0.05003656819462776, 0.4240053594112396, -0.3763476610183716, 1.6002116203308105, -1.6696853637695312, 0.9078635573387146, -0.16731037199497223, -0.1293908953666687, -0.16691428422927856, 0.7553223371505737, -0.6825742721557617, 0.2433335781097412, -0.7974683046340942, -1.0286353826522827, 0.10790394246578217, -0.5861929059028625, 0.5171560645103455, -0.4087543487548828, -1.1360634565353394, 0.6648339033126831, -0.9517565965652466, -0.3554559648036957, 1.400223731994629, 0.5830283164978027, 1.2760783433914185, 0.20651143789291382, -0.8438435196876526, -0.2448045313358307, -0.15493738651275635, -1.0331653356552124, -0.7573505640029907, -0.05725361779332161, -1.3634240627288818, 0.49772554636001587, -0.15427619218826294, 0.8914034366607666, -3.0638184547424316, -0.4500575065612793, 0.3577820658683777, -0.9728692173957825, 0.05406703054904938, -0.19381853938102722, 0.19951650500297546, 1.7186956405639648, -0.24521951377391815, 0.058907829225063324, -0.6842135787010193, -1.3540586233139038, -0.22314225137233734, 0.08120246231555939, -0.5132534503936768, -1.0088541507720947, 1.2847217321395874, -1.2853552103042603, -1.0611094236373901, -1.1719368696212769, -1.6621222496032715, 0.595343828201294, -1.0754735469818115, 0.10038677603006363, -0.770973265171051, -1.6674147844314575, -0.16957135498523712, 0.4561986029148102, 0.26825109124183655, -0.5574204921722412, -1.604817271232605, 0.3750595152378082, 0.9097856879234314, -0.3536999523639679, 1.7004996538162231, -1.2884721755981445, 0.35161978006362915, 0.42312368750572205, -1.3929088115692139, -0.4049800634384155, -0.2797093987464905, -0.6478711366653442, 0.13027386367321014, -1.407161831855774, -0.4338800311088562, -0.0424814410507679, -2.280287981033325, -1.5185472965240479, 0.9794642925262451, 1.5184497833251953, -0.3650665581226349, 0.25928807258605957, -0.930034339427948, 0.2876104712486267, 1.7714003324508667, -2.575362205505371, 1.6408727169036865, 0.5657072067260742, -1.197635293006897, 0.7699812650680542, -0.6970468759536743, -0.1110931858420372, -2.008667230606079, 1.2694222927093506, -0.8172304034233093, -1.4498659372329712, -0.24482594430446625, 0.7718872427940369, -1.03616201877594, -0.8849138021469116, 0.05238940194249153, -0.3228030800819397, -0.542555570602417, -0.4217956066131592, 1.194137692451477, 1.5927928686141968, -0.6429977416992188, -1.2710932493209839, 0.6697946786880493, -1.158648133277893, -0.08662394434213638, -1.1239080429077148, -0.8659319281578064, -0.8406195640563965, -0.2644495368003845, -1.658445119857788, -0.41337332129478455, -0.2858026325702667, 0.3552038073539734, 0.7560580372810364, -0.12679249048233032, 1.1338458061218262, 1.172686219215393, 0.3527073562145233, -0.344023197889328, 1.3410841226577759, -1.1365710496902466, 1.979853868484497, -0.163519024848938, 0.9469960331916809, -1.4791779518127441, -1.14570152759552, 2.485856056213379, -0.02225825935602188, 0.7335453629493713, -1.1778678894042969, 0.34256845712661743, 0.594737708568573, 0.3838481307029724]], "dtype": "float32"}, {"__nd__": [-0.471729040145874, 0.025249119848012924, -0.2371494472026825, -0.9022148251533508, 0.37070488929748535, 0.26748257875442505, 0.7800310254096985, 0.05796950310468674, -1.651745319366455, -0.34160006046295166, -1.473437786102295, 0.08086219429969788, -0.6619201302528381, -0.038441386073827744, 0.07010111212730408, -0.6964787244796753, -0.09172811359167099, 1.094887375831604, 0.47781649231910706, -0.7004719376564026, 0.2953709661960602, -0.028581125661730766, 0.38338446617126465, 0.7283919453620911, -1.348819613456726, 0.29169589281082153, -1.5211848020553589, -0.849908709526062, 0.3885752856731415, 0.7440000176429749, -0.2824547588825226, -2.2793641090393066, -0.6687821745872498, 0.308702677488327, 0.6879746913909912, 0.5672483444213867, 0.6615834832191467, 0.10324428975582123, -1.4310417175292969, 1.5711644887924194, 0.8174782395362854, 0.44288793206214905, 0.1647108942270279, -1.9019898176193237, 0.6197249293327332, 1.7637189626693726, 0.07010415941476822, 0.7300142049789429, 0.42711570858955383, 1.3428630828857422, -1.153414249420166, -0.49436622858047485, -0.9733947515487671, -0.9646241068840027, 0.8137302994728088, -0.2985811233520508, -0.34513798356056213, -0.5953818559646606, 0.11080179363489151, -0.7855169773101807, -0.03650535270571709, 1.3992671966552734, 0.4187626838684082, -0.42411041259765625, 0.4520426094532013, 0.5939469337463379, -0.23468151688575745, -0.18113310635089874, -1.2187936305999756, 2.0146963596343994, 0.7808119058609009, -1.7607005834579468, -1.8399142026901245, 2.4194765090942383, 0.8005656003952026, -0.6678617596626282, 0.9399711489677429, -0.4444388747215271, 1.3322502374649048, 1.3524763584136963, -1.0454570055007935, -2.09633469581604, -0.5649570822715759, 0.7523432970046997, 1.2130504846572876, 1.4869859218597412, 0.895034670829773, 0.0287869181483984, -0.782281219959259, 0.3702555000782013, -0.3355197310447693, 0.2075035572052002, 0.25214850902557373, -0.8299177289009094, -0.9423630833625793, -0.8070825934410095, -1.7458616495132446, 0.1186242401599884, -0.8415425419807434, 1.8646525144577026, 0.7226560115814209, -1.5884110927581787, -2.3860201835632324, -1.6487771272659302, -1.3278955221176147, 0.22310498356819153, -0.3258756697177887, 0.8744275569915771, -0.6829918026924133, 0.6858683824539185, -0.3213798701763153, -0.9587864279747009, 0.7570763826370239, 0.062039367854595184, 1.5274314880371094, -0.9791252613067627, 0.9002101421356201, -0.05692771449685097, 1.2243317365646362, 0.15818043053150177, -0.06552793830633163, -1.2871158123016357, 0.8863213062286377, -1.3228617906570435, 0.5857462882995605, 0.3708454668521881, -1.1141282320022583, -0.7452681660652161, -0.7956457138061523, 0.36865338683128357, 1.5384621620178223, -0.37490496039390564, -0.2042420655488968, 1.300483226776123, -0.966523289680481, 0.21356213092803955, 0.48336052894592285, -1.0320457220077515, 1.1036529541015625, 0.6498321890830994, 0.31961923837661743, 0.6719926595687866, 1.9465370178222656, 2.310345411300659, 0.9075124859809875, -0.7448791265487671, 0.9745438098907471, -0.9772104620933533, 0.3534333407878876, -0.7229222655296326, 1.3055870532989502, 1.041795015335083, 0.8237727880477905, 0.8047603964805603, 0.48449987173080444, 1.8297522068023682, 0.9130763411521912, -0.23794858157634735, -1.5375661849975586, -0.9557377696037292, -0.4406379163265228, -1.224124550819397, -0.15246990323066711, 1.5467877388000488, -1.2837615013122559, -0.10989667475223541, -0.29620665311813354, -1.687469720840454, -0.9033857583999634, 0.14465558528900146, -0.22538131475448608, -2.5825486183166504, -0.25557565689086914, -0.8479751348495483, 0.24531289935112, -0.2641902267932892, 1.605790615081787, 0.052316583693027496, 0.41825881600379944, -0.7490391731262207, 0.7514071464538574, 0.9053683876991272, -0.006818652153015137, -1.287317156791687, -0.6785793900489807, 0.6497639417648315, 1.4302152395248413, 0.7386678457260132, -1.2812515497207642, -1.4010167121887207, 1.2142372131347656, 1.105821967124939, 0.02598567306995392, -0.4327370524406433, -1.828963279724121, 0.5678789615631104, -0.38514143228530884, -0.6447581648826599, -2.9687085151672363, 0.31276243925094604, -0.046031732112169266, -1.2894854545593262, 0.6170507073402405, -0.929968535900116, 0.49619197845458984, 1.443401575088501, -0.7751806378364563, -0.724054753780365, 0.20341040194034576, 0.5931476354598999, -0.6372950077056885, -0.684037983417511, 1.2534706592559814, -0.3972828984260559, 0.10716228187084198, 0.010770830325782299, -0.8825167417526245, -0.28566280007362366, 0.18126744031906128, -0.5161563158035278, 0.2645360827445984, -0.6579145193099976, -1.3314601182937622, 1.4581646919250488, -1.7380629777908325, 1.239080548286438, 1.7391785383224487, 1.5378360748291016, -0.09763722866773605, 1.2956171035766602, -0.09614889323711395, 0.11211125552654266, 2.4612877368927, -1.801456093788147, 1.2332007884979248, -1.710411548614502, 0.7734826803207397, -0.9623323082923889, 1.192274570465088, -0.0257602296769619, 0.2385483980178833, -1.470948338508606, 0.2539868652820587, -0.2959514856338501, 1.5044654607772827, 2.022054433822632, -0.17890231311321259, 0.7975058555603027, 1.6337696313858032, 0.27668505907058716, -0.7149483561515808, -1.5665429830551147, -0.5030964612960815, -0.7289902567863464, 0.8284896016120911, -1.0340194702148438, 1.1532098054885864, 0.6879922747612, -0.31592971086502075, 0.5617128610610962, 0.877913236618042, -0.7087542414665222, -0.3184644877910614, 0.2243083268404007, 0.7576251029968262, 1.3089903593063354, 0.5595348477363586, -1.6123634576797485, -0.4244781732559204, -1.1192371845245361, 0.5074332356452942, 1.8420555591583252, -0.16455872356891632, 1.1202799081802368, -0.0568678081035614, 1.0801737308502197, -0.014938294887542725, -0.1305365264415741, 1.2231433391571045, 1.4661831855773926, -0.47116681933403015, 0.955617368221283, 0.828713595867157, -1.518004059791565, -0.43564650416374207, -0.264054536819458, -0.7729189991950989, -0.8860017657279968, -0.7570404410362244, -0.5940070152282715, 1.392283320426941, -0.1522299200296402, -0.5591866374015808, -1.0490211248397827, -1.4567314386367798, 0.558975100517273, 1.3009644746780396, 0.7756621241569519, 1.0293642282485962, -0.6318325996398926, -0.3447960615158081, 0.43920615315437317, 0.29595696926116943, -1.4517207145690918, -0.4162479639053345, 1.7310739755630493, -1.0827670097351074, -0.06323500722646713, 0.1058906689286232, 0.6210066676139832, 0.36808058619499207, -0.4316839575767517, -0.48504889011383057, 1.9581308364868164, -0.7151296734809875, -1.7590008974075317, 0.25078991055488586, -0.5667927265167236, -1.716289758682251, -0.5692325830459595, 0.47212082147598267, 0.8099950551986694, 1.0237083435058594, 0.7755378484725952, 0.47623196244239807, 0.1171315461397171, 0.30530378222465515, 0.16139821708202362, 2.575758934020996, 0.9681357145309448, 0.2323017716407776, 1.1386069059371948, -0.7564461827278137, 0.3972708284854889, 0.4239743649959564, 2.594850540161133, -1.6525768041610718, 1.0871565341949463, -0.21860955655574799, -0.7637662291526794, 1.4785975217819214, -0.22454963624477386, -1.1351391077041626, -0.8805090188980103, 1.0462841987609863, -0.05092674121260643, -1.3157451152801514, -0.4579785466194153, -2.0665879249572754, -0.7811164259910583, -0.0948050245642662, 1.390013337135315, -0.1692880094051361, 1.0200917720794678, 0.045392896980047226, -0.23716092109680176, -0.3506891429424286, -1.9792205095291138, -1.1132124662399292, 1.009728193283081, -2.237825870513916, -0.3812597393989563, -0.19430163502693176, -0.20480801165103912, -0.6854464411735535, 0.13959155976772308, 0.2141074389219284, -0.8001348376274109, 1.3970894813537598, -0.8204456567764282, 0.6060006022453308, 0.22673043608665466, 0.6365978717803955, 0.5118862986564636, -0.07142843306064606, 0.9883491396903992, 1.198359727859497, -1.4433907270431519, -0.6131957173347473, -1.2816176414489746, -1.4999381303787231, -1.8217133283615112, -1.3929803371429443, 0.7961215376853943, 0.6497178077697754, -0.6096041798591614, -0.20672607421875, -0.32660138607025146, 0.10459502041339874, 1.374578595161438, -0.4040258228778839, 0.001957569969817996, -0.33310258388519287, -0.8079567551612854, 0.5331071019172668, -0.03879388049244881, -1.0777339935302734, 0.8951800465583801, 0.5848331451416016, 0.15367093682289124, -1.1500811576843262, 0.0899939313530922, -0.001426511793397367, 0.9707115292549133, -0.3081597089767456, 0.5829471945762634, -0.08752667158842087, -0.061981238424777985, 0.49565935134887695, 0.004734151065349579, 0.5553621649742126, 0.9921878576278687, 1.3777562379837036, 3.521148681640625, 1.1786787509918213, -0.44060200452804565, 1.1860990524291992, 0.6165435910224915, -0.35266926884651184, -1.8596410751342773, 0.2423309087753296, -1.0239944458007812, -1.7073183059692383, 0.44619274139404297, 1.4867674112319946, 0.335663765668869, 1.285383939743042, -0.08064067363739014, -3.573934316635132, 0.5093120336532593, -0.6803979873657227, -0.16562232375144958, 1.1476718187332153, -0.27533406019210815, 0.5004322528839111, -1.4082260131835938, -0.3388877809047699, 0.814386785030365, 1.2363187074661255, 0.03492843359708786, 0.46094685792922974, 0.5971617102622986, 0.9856906533241272, 0.47024258971214294, -0.15191960334777832, -1.436313271522522, -0.9492221474647522, -0.29915758967399597, -0.7925804257392883, 1.734197735786438, 0.4234935939311981, 0.37100154161453247, -0.6009954810142517, -0.806702733039856, 0.256043940782547, -0.3716295659542084, -0.9403207898139954, 0.8636696934700012, -0.062463197857141495, 0.255473256111145, 3.046560049057007, -0.6211023926734924, -0.135144904255867, 0.8461413979530334, 0.7576002478599548, 1.0550274848937988, 1.4696303606033325, 0.03245314583182335, 1.9284454584121704, -0.09048081189393997, -0.20236173272132874, -0.9428125023841858, 0.3631598949432373, 0.32759615778923035, 0.12944725155830383, 0.5121287703514099, 1.8341034650802612, -0.7918320894241333, -1.1917710304260254, -1.9289615154266357, -0.4476841986179352, -0.11085664480924606, 1.2981034517288208, 0.7381054162979126, 1.0566223859786987, 0.18460382521152496, -1.1671689748764038, -1.3928202390670776, 1.511134147644043, -0.6694251894950867, 1.7454657554626465, -0.2904197871685028, -0.9462634921073914, 0.6419535875320435, 0.3517232835292816, 0.8290873169898987, 1.5363070964813232, -0.7939435243606567, 1.0801588296890259, 1.7035521268844604, -0.022692393511533737, -0.5455631613731384, -1.719913125038147, -0.9787624478340149, 2.12614107131958, 0.7250282168388367, -0.9508615136146545, -0.8521852493286133, 0.37168094515800476, 1.8890912532806396, 0.6295484304428101, 0.08052466809749603, 0.617139995098114, 1.0712153911590576, 0.7449394464492798, -0.029693540185689926, 1.6462135314941406, 0.1253194808959961, 0.05042661726474762, 0.09941855818033218, -0.7606338262557983, 0.808049201965332, -1.3335394859313965, 1.7408881187438965, -1.137278437614441, -0.203614741563797, -1.1955968141555786, -0.24698695540428162, -1.5646425485610962, -0.635129451751709, 0.5378215312957764, -0.6890847682952881, -2.4939935207366943, -0.19115759432315826, -0.46104565262794495, -1.3185616731643677, -2.0606207847595215, -1.0114494562149048, -0.5867132544517517, 2.1395390033721924, -0.4179251790046692, 0.2781210243701935, -1.5709803104400635, 0.8612430691719055, 1.4498275518417358, 1.2135305404663086, 0.010750406421720982, 0.5109797716140747, -0.18226554989814758, 0.21025924384593964, 1.6239330768585205, -0.11946877092123032, -2.0283141136169434, -0.422939270734787, 0.6446925401687622, -0.14599211513996124, -0.008902658708393574, 0.3479706645011902, -3.074052333831787, 0.6005359292030334, 2.277923345565796, -0.07724153995513916, 1.160000205039978, -0.1510104387998581, -0.6651332378387451, -0.4831351637840271, -1.5163902044296265, 0.15344487130641937, -0.10395191609859467, -1.5616576671600342, 0.2482585906982422, -0.03369140625, 0.20879581570625305, -1.3096359968185425, 0.8561911582946777, -0.09701096266508102, 2.235687255859375, -0.2593117654323578, -0.9157798290252686, 2.0031425952911377, 1.2734483480453491, -0.47449731826782227, -0.8307318091392517, -0.5853196978569031, 2.406237840652466, 0.017243141308426857, -0.042316701263189316, 0.24349768459796906, 0.6833325028419495, 0.06223691254854202, -0.9056762456893921, 0.03606696054339409, -0.7141614556312561, 1.1510921716690063, 0.2677403390407562, 1.173761010169983, 0.46734297275543213, -0.5218635201454163, -1.753366470336914, -0.07753194123506546, -0.04969298094511032, -1.6946638822555542, -0.8242945075035095, 0.9643057584762573, 0.4681546688079834, -0.07379927486181259, 0.2188744693994522, 0.5073877573013306, 0.888612687587738, -0.15758265554904938, -2.803804874420166, -1.137412667274475, 0.669076144695282, -0.9438906908035278, 1.3167308568954468, -0.9541250467300415, 0.25775110721588135, 0.18538141250610352, 2.2355053424835205, -0.5801641345024109, -0.8082218170166016, 0.8516268730163574, 0.08216724544763565, -1.412725567817688, -0.2826439142227173, 0.000628271431196481, 0.5910409092903137, 0.3980028033256531, 0.7219738364219666, 1.0540564060211182, -0.41888871788978577, -0.6902557015419006, -0.6781851053237915, -1.5480287075042725, -1.6112127304077148, -1.7729607820510864, 0.8111223578453064, -0.5296801924705505, 1.6029620170593262, 0.7499611377716064, 1.1111156940460205, 0.15970094501972198, 0.9874318242073059, -0.35215362906455994, 0.15016107261180878, 0.1452113687992096, 0.19396939873695374, -1.119903564453125, -0.09899555146694183, 0.24781537055969238, 1.193035364151001, -0.7108547687530518, -0.2513503432273865, 0.36575987935066223, -0.31962844729423523, -1.14504075050354, -1.136364459991455, 0.26431629061698914, 1.1282551288604736, -0.4497119188308716, 0.08059228211641312, 0.21251854300498962, 0.020149271935224533, -0.8751373887062073, -0.21293911337852478, -0.5281715989112854, -0.8468348383903503, -0.5904145240783691, -0.41208598017692566, 0.2432163655757904, -0.9230834245681763, -0.45339635014533997, -0.5589194297790527, 0.0330260768532753, 0.2074892669916153, 0.2963033616542816, 0.16455243527889252, -0.20508554577827454, 0.5206462740898132, -0.1539708971977234, -1.2145726680755615, 0.46413108706474304, -0.4486689865589142, -0.49563372135162354, -1.363005518913269, 0.37518903613090515, 0.8696622252464294, -0.5435877442359924, 0.5033255219459534, 1.2951486110687256, 0.5322743058204651, -0.06975125521421432, -1.5983935594558716, 0.19420462846755981, 0.47157931327819824, 0.7134035229682922, 0.9589130282402039, -0.6077523231506348, -1.2985624074935913, 0.20976579189300537, 1.7089247703552246, 0.005530456081032753, 0.4865000545978546, 0.04668094962835312, -0.03742825984954834, 0.9536791443824768, 0.9574000835418701, -0.018222689628601074, -1.293615460395813, 1.7303227186203003, 0.7570580840110779, 0.8291904926300049, 0.7946118712425232, -1.4130007028579712, -0.08408238738775253, 0.5400002002716064, 1.3139399290084839, -0.7918195128440857, 2.571376323699951, -0.2946591377258301, 0.6263272166252136, 0.6590508222579956, 1.0159369707107544, 1.408970832824707, 2.4723165035247803, 1.8839361667633057, 1.1315604448318481, -1.5227257013320923, 0.36160603165626526, -0.9816455245018005, -1.3425772190093994, 0.6225912570953369, -0.15278080105781555, 1.2910277843475342, -1.6811453104019165, -1.414523720741272, -1.3976515531539917, -0.1903284192085266, -1.0388256311416626, -0.4120982587337494, 0.5559903383255005, 2.2952020168304443, -0.9915465712547302, -1.324222207069397, 1.5690943002700806, -1.3968063592910767, 0.5846996307373047, 0.7552120089530945, -0.02416003867983818, -0.3484332263469696, -1.501860499382019, 0.5554491281509399, -1.1328153610229492, 0.9753450155258179, -0.006610484793782234, -0.9307470917701721, 0.06347720324993134, 0.7536510825157166, 0.4194875657558441, 0.2640688717365265, -0.2737899720668793, -0.575749397277832, 1.8389524221420288, -1.8278950452804565, 1.162983775138855, -0.9803946018218994, 0.4476763904094696, 2.456758499145508], "dtype": "float32"}, {"__nd__": [0.7224065661430359, -0.3185434639453888, -1.0773307085037231, 0.7566339373588562, 0.5993292331695557, -1.371812343597412, -0.3232767581939697, -1.5769845247268677, -0.19766634702682495, -0.2346307784318924, -1.0687960386276245, -0.4609455168247223, -0.6059760451316833, -0.6826083660125732, 1.2334567308425903, -1.7175008058547974, 0.980526328086853, -0.7766686677932739, -0.6869392395019531, -0.7005655765533447, 0.6525402069091797, -0.8732960224151611, 0.5110387802124023, 1.4024382829666138, -0.2943210303783417, -0.8375810980796814, -0.7058596611022949, 0.4524564743041992, 1.8850780725479126, -1.2601063251495361, 0.6154503226280212, -3.3897159099578857, 1.1710423231124878, -0.14318867027759552, 1.2283121347427368, -0.02327934093773365, 0.3402496576309204, -0.7454332113265991, -1.2532882690429688, 1.506920337677002, -0.44704702496528625, -0.3712112009525299, -0.02745276689529419, 0.4041864573955536, 0.9296955466270447, -1.3569748401641846, -1.3396600484848022, -0.03169641271233559, -0.2874625325202942, 1.1960797309875488, 0.3443548381328583, 1.62350594997406, 0.8406296372413635, -0.6054822206497192, -0.3870125114917755, -0.3838254511356354, -1.3644993305206299, -0.3453689217567444, 0.09994705766439438, 0.19835595786571503, 0.8392284512519836, 0.08744291216135025, -0.4951077699661255, -1.32848060131073, -0.2098150998353958, -1.1469430923461914, 1.537018895149231, 1.1492494344711304, -0.3224223554134369, 1.434546709060669, 1.9249674081802368, -1.3104045391082764, -0.1882704496383667, -0.4902576804161072, -0.70060795545578, 0.18134072422981262, 1.5953381061553955, -0.1968250870704651, -1.814712643623352, 0.34500113129615784, -0.5313435196876526, -0.30038633942604065, -0.7150633335113525, 0.6390014886856079, 0.6296166181564331, -0.63289874792099, 0.022345678880810738, -1.2781630754470825, -1.7951982021331787, 1.272502064704895, 1.3038233518600464, -0.5001243948936462, 0.16339215636253357, -0.24886095523834229, 1.7054778337478638, -0.43839403986930847, 0.15097777545452118, -0.13888335227966309, 0.2804691195487976, 1.2542791366577148, -0.838005542755127, -0.953114926815033, 0.533119261264801, -0.11497397720813751, 0.24329732358455658, -0.8344405293464661, -0.26867035031318665, 0.5577666163444519, 0.7549623847007751, -0.7215986251831055, 1.0165799856185913, 1.2590495347976685, 0.6437606811523438, -1.390668511390686, 1.598241925239563, -1.2679451704025269, -0.9414604306221008, -0.299966961145401, 0.78183913230896, 0.2468252032995224, 1.784527063369751, -1.1610523462295532, 1.2458913326263428, -0.9707600474357605, 0.0829702764749527, -1.9561885595321655, 0.2848645746707916, -0.09562281519174576, 0.641582190990448, 0.5389129519462585, 0.15936507284641266, -2.2993149757385254, -1.9792141914367676, -0.07854211330413818, -0.5755456686019897, 1.8033781051635742, -0.5323631167411804, 0.11391586810350418, -0.0941048115491867, 0.13929453492164612, -0.08392943441867828, -0.19417676329612732, 0.6706797480583191, 0.2531471848487854, 0.9502940773963928, 0.28724202513694763, 0.6546667218208313, 0.7376655340194702, -1.7534129619598389, 3.0874826908111572, -1.706539273262024, -0.864786684513092, -0.38219302892684937, -0.3267185389995575, -1.2021764516830444, 0.33933281898498535, 0.38561099767684937, 1.1275677680969238, 1.414475440979004, 1.5967708826065063, -0.15302373468875885, 1.3767014741897583, -0.20905326306819916, -0.7925959229469299, -1.6517279148101807, -1.2195723056793213, -0.6198312044143677, -0.4864892065525055, 2.8389523029327393, 1.4628150463104248, 0.5055631995201111, 1.2065229415893555, 1.1049408912658691, 0.8228274583816528, 1.9138015508651733, -1.6594040393829346, 0.09383198618888855, -0.2819111943244934, -1.4157640933990479, 1.6683595180511475, -0.22528895735740662, 0.26848655939102173, -0.13329200446605682, -0.5074527263641357, 0.09745407104492188, -0.21775774657726288, -1.6785718202590942, 1.1898932456970215, 1.3796463012695312, -1.0119603872299194, 0.5038976073265076, -0.6807728409767151, -3.532557487487793, -0.03503933921456337, -1.509570837020874, -0.1572646051645279, 0.2894001305103302, -1.6133649349212646, -1.296526312828064, -0.45261895656585693, -0.9842169880867004, -0.13173173367977142, -1.5241053104400635, -0.30809280276298523, 0.6548023223876953, 0.8620024919509888, -0.6074571013450623, -0.7801511883735657, 0.5463595390319824, 0.9227914810180664, 2.3733203411102295, 0.37826666235923767, -0.33250731229782104, -1.0675342082977295, -2.179671287536621, 1.0678118467330933, 0.19089703261852264, -1.5364269018173218, -0.8993419408798218, 0.8556156158447266, -0.07375401258468628, 1.494255542755127, -2.508214235305786, 0.9292861819267273, 2.650282859802246, 1.4629987478256226, -0.5313365459442139, -0.19052574038505554, -0.7728397250175476, 0.49313104152679443, -0.4036542475223541, 0.6463989615440369, -0.9545993208885193, -0.5057106614112854, -0.320758193731308, 0.7303597927093506, -0.6999818682670593, 0.44801992177963257, -0.70925372838974, 1.083225131034851, -0.47766199707984924, 0.43588170409202576, 0.32647058367729187, 0.876715898513794, -0.7683303952217102, 0.8511166572570801, -0.9978392124176025, 0.6034963726997375, -0.8670993447303772, 0.45313653349876404, 0.9461447596549988, 0.47160229086875916, 0.9588867425918579, -0.2372462898492813, 1.4649739265441895, -0.6003016233444214, -0.542646586894989, -0.35664188861846924, -0.6959313750267029, 0.0509001649916172, -1.1127032041549683, 1.9779106378555298, -0.13326339423656464, 0.9181216955184937, 1.761697769165039, 0.3593420386314392, -2.3246676921844482, -1.7638198137283325, -0.23548145592212677, -0.20425021648406982, -0.7869409322738647, -0.9830197095870972, 0.8958513736724854, -0.5917766094207764, -0.9259575009346008, -0.2186766415834427, 0.18134431540966034, 0.32791805267333984, -0.2987349331378937, 0.8878852725028992, -1.9733805656433105, -2.288409471511841, -1.8928894996643066, 1.0271918773651123, -0.050927385687828064, 0.5653238296508789, 0.674994707107544, -1.1275928020477295, -0.14455263316631317, -0.431108683347702, -0.4398239254951477, 2.1311051845550537, 0.38774874806404114, -0.003119910368695855, 0.3383539915084839, 0.5957463383674622, -0.9512141942977905, 0.7949086427688599, 1.523451805114746, 1.173933744430542, 0.9482601881027222, 1.8845946788787842, -0.727455198764801, -0.4794560670852661, -0.4556659460067749, 0.21753187477588654, -0.3674093782901764, 0.18313872814178467, -0.08832282572984695, -1.093981385231018, -0.819660484790802, -0.8588060736656189, 0.2930087149143219, 1.3885222673416138, -1.5899206399917603, -0.947371244430542, -0.08419773727655411, 0.42167115211486816, 1.3852646350860596, -0.8664558529853821, 3.0066568851470947, 0.9207918047904968, 0.47152572870254517, 1.220859169960022, 0.8353801369667053, -0.1633179932832718, 1.122237205505371, -2.232234001159668, -1.6281158924102783, 0.8215406537055969, -0.48531588912010193, 1.0638437271118164, 0.6587415933609009, -0.22270122170448303, 0.30601808428764343, -0.9662783145904541, -0.9101883769035339, 0.08253884315490723, 0.16869723796844482, 1.5595594644546509, -1.0234079360961914, 0.0664316788315773, -1.6291561126708984, 1.296891450881958, -0.35130494832992554, 0.20176991820335388, 2.602501392364502, -1.575838327407837, -0.7736201286315918, 1.65202796459198, -0.42639780044555664, 0.6933231353759766, -0.7783587574958801, 0.8690526485443115, -1.477614164352417, 1.1301331520080566, 0.18723724782466888, 0.03849169984459877, 1.8355014324188232, -0.2966056168079376, 0.35057955980300903, -0.8275076150894165, -1.7109849452972412, -0.036621756851673126, -0.6191122531890869, 0.11428520083427429, -0.1742544025182724, 1.796319603919983, 1.4825077056884766, 0.43897607922554016, 0.7521399259567261, -0.1814769208431244, -1.3569729328155518, -1.1290388107299805, 1.2249641418457031, 0.16309095919132233, -1.0560246706008911, 0.8884973526000977, 1.6275198459625244, -1.4461109638214111, -0.43332716822624207, 0.1789858341217041, 0.5084440112113953, 0.02875986509025097, 0.32103314995765686, 1.8317815065383911, 2.3340647220611572, 0.9365736842155457, 0.10059155523777008, -0.4101378321647644, 0.9213647246360779, -0.8497439622879028, -0.21782813966274261, -0.8700709342956543, -0.8939486742019653, -0.16383403539657593, -1.1747450828552246, 1.9153242111206055, 0.00023146296734921634, -0.9914471507072449, 0.888740599155426, 1.2268016338348389, 0.8076795935630798, 0.030183393508195877, 0.9545401930809021, 1.450222134590149, -0.6083005666732788, -0.16093093156814575, -0.816665768623352, 1.747786521911621, 1.467952013015747, -0.5011246204376221, -1.1676726341247559, 0.7124063372612, 0.10541455447673798, -1.4030743837356567, 0.6698348522186279, -0.039336446672677994, -0.28155019879341125, 0.5622634291648865, -1.318315029144287, 0.4505438208580017, -0.6121566891670227, 0.6047871708869934, -0.3286384344100952, -1.5918124914169312, -0.43804699182510376, -0.49024760723114014, 0.82354736328125, 0.40612438321113586, -0.37984251976013184, -0.0193620678037405, 0.22189642488956451, -0.6268829703330994, 1.2537121772766113, -0.6585767865180969, 0.8751273155212402, -0.5036580562591553, 0.15872620046138763, -0.8369162678718567, 1.0569040775299072, -0.8204352259635925, -1.2379542589187622, -0.5896616578102112, 0.3798675537109375, -0.2619436979293823, 0.7564403414726257, -1.3473355770111084, -0.2314683347940445, 0.013464692048728466, 1.4306647777557373, -1.7052818536758423, -1.3976447582244873, -0.8665175437927246, -0.7118526101112366, 0.9332501888275146, -0.9155077934265137, -0.4043586850166321, -0.6565291285514832, -0.8199310898780823, -0.18905749917030334, -0.0026270183734595776, 0.48295387625694275, 2.3029701709747314, -1.2608006000518799, -0.1860899031162262, 1.3021199703216553, -1.0589386224746704, 0.6471219658851624, 0.6531859040260315, 1.1507271528244019, 0.7631750106811523, -0.8903071284294128, -0.2415524125099182, -0.542108416557312, -0.41956624388694763, 0.6596624255180359, -0.6401057839393616, -0.20164892077445984, -0.9846832156181335, 0.06655841320753098, -1.4333537817001343, 0.30985772609710693, -0.5915626287460327, -0.3074935972690582, -0.5342404842376709, -0.9176005125045776, 0.9456586241722107, 0.019473029300570488, -0.3045555055141449, 1.0153617858886719, 0.4066372215747833, 1.2445554733276367, -1.5166106224060059, 1.0256052017211914, -1.1391068696975708, -0.24334044754505157, 1.653316855430603, 0.177960604429245, -0.4700140655040741, -1.0393041372299194, 0.6775354743003845, 0.5565072298049927, 0.015423388220369816, 0.5839868783950806, 0.45814499258995056, 0.39896613359451294, -1.1950899362564087, -0.7768170237541199, 0.5156179666519165, -0.15143881738185883, 0.02396472729742527, 0.845542311668396, 0.21102052927017212, -0.24503515660762787, -0.4679587185382843, -0.9215635061264038, -0.12633256614208221, -0.5569912791252136, -1.9768671989440918, -0.6899425983428955, -1.1766407489776611, -0.7857353091239929, 1.8031803369522095, 2.214418888092041, -0.7437330484390259, 0.5094056129455566, -0.16655008494853973, -1.6224005222320557, -1.9112663269042969, 1.2476136684417725, -1.1461471319198608, 0.07099545001983643, -0.3988029658794403, 1.3834222555160522, -1.6386065483093262, -1.0838510990142822, -1.154098391532898, 0.3102022111415863, 0.245045006275177, -0.039260029792785645, 1.5192724466323853, 0.40250086784362793, 1.0261017084121704, 1.1586754322052002, 0.38355517387390137, 0.9865173101425171, -1.2968299388885498, -1.0249985456466675, -1.1175297498703003, -2.281947374343872, 0.7947337627410889, -1.8980462551116943, -1.1402037143707275, -0.524318516254425, -0.8794280886650085, -0.5634749531745911, -0.01169730257242918, -1.7347804307937622, -1.37956702709198, 2.871389150619507, 0.7240031361579895, -0.31827786564826965, -0.25621816515922546, 2.20611572265625, -0.8582639098167419, -0.9547664523124695, -0.7766866683959961, 0.4816961884498596, 0.3825377821922302, -0.647407054901123, 0.6922587752342224, 0.8115817308425903, 1.5116513967514038, 1.331778883934021, 0.6734862923622131, 0.803613543510437, -0.3202977180480957, 0.9893507957458496, -1.7308259010314941, -0.05396999046206474, -0.7492548823356628, 0.7300494909286499, -0.6070077419281006, 0.23939314484596252, -0.4005579650402069, -0.19792528450489044, -2.019183397293091, -0.9764387011528015, -0.7817586660385132, -0.12000018358230591, 0.36515888571739197, 1.2409738302230835, 0.09946766495704651, 0.6670919060707092, -0.32453107833862305, 1.7225133180618286, 1.115082859992981, -1.5684242248535156, 0.9861491322517395, 1.3756155967712402, -0.3709883689880371, 1.2579971551895142, -0.40678662061691284, -0.1250128298997879, 1.4959173202514648, 0.7415908575057983, -2.0338826179504395, -1.008994698524475, -0.13200613856315613, -0.4710889458656311, 1.1420339345932007, -1.5766630172729492, -0.5472908020019531, 1.2568389177322388, -0.19570615887641907, -0.025810111314058304, 1.1882883310317993, -0.3294706642627716, -1.0517776012420654, 0.16609275341033936, -0.9234798550605774, -1.134204387664795, 1.2100067138671875, 0.167506605386734, -0.49733349680900574, 0.04880030080676079, -1.5094908475875854, 1.0742683410644531, -0.11935883015394211, 0.8046090006828308, 0.0598653107881546, 0.43555963039398193, 0.6264694929122925, -0.6639689207077026, -0.29085224866867065, 0.5310088992118835, 0.47342467308044434, 0.4439810812473297, -0.6366090178489685, -0.05869079381227493, -0.7937825322151184, -0.1432211548089981, 1.337483525276184, -0.6354005336761475, -1.2093019485473633, -0.1601833552122116, 0.5318969488143921, -1.3475502729415894, -0.1641138195991516, 0.7553540468215942, 1.7025474309921265, 1.2542728185653687, -1.9625957012176514, 0.5243273973464966, 0.37824884057044983, 0.9181689620018005, 0.7316194176673889, -1.453055739402771, -1.6945515871047974, 0.961209237575531, -0.34506505727767944, -0.08467676490545273, 0.7438939809799194, -0.08083603531122208, 0.37025487422943115, -1.241824984550476, -0.4722220301628113, -0.9838034510612488, -1.3883626461029053, -0.8380546569824219, 0.07601792365312576, -0.32710692286491394, 0.4374493658542633, 1.0203758478164673, 0.16732774674892426, 0.2736981213092804, 2.0681002140045166, -0.9643469452857971, 0.30822470784187317, -0.43675369024276733, -0.1879919022321701, -0.3489729166030884, -0.5447000861167908, 0.09911546111106873, 0.8995206356048584, 1.5434741973876953, -0.09845848381519318, 0.5216915607452393, -0.7049986124038696, 0.7995439171791077, 0.336858332157135, -0.549351692199707, -0.039295539259910583, 0.04410402849316597, -0.8247629404067993, 2.4313199520111084, -0.4993875324726105, -0.6471688151359558, -1.1286636590957642, -0.2830185890197754, -2.2194113731384277, -1.8111227750778198, -0.12139125913381577, -1.013816237449646, -0.6674590706825256, -0.11646778136491776, 0.5569525361061096, 0.30519795417785645, 2.171349287033081, 1.2838188409805298, -0.18676604330539703, 0.8006134629249573, -0.30663633346557617, -0.833674430847168, 0.8141158819198608, 0.4861305356025696, 1.011879801750183, -0.7090663313865662, -0.7606720924377441, 1.0937567949295044, 1.2312719821929932, 0.3956305980682373, 0.9915672540664673, -0.4269126355648041, -0.09324248880147934, -1.3083338737487793, -0.18085892498493195, 0.5346252918243408, 0.7459222078323364, -0.6973771452903748, -0.24024638533592224, 0.05465541034936905, -0.31949251890182495, -0.3137378990650177, 0.12435673922300339, -0.696463406085968, 0.9621135592460632, -0.49124106764793396, -0.17302677035331726, -0.34223228693008423, 1.439823031425476, -0.7497026324272156, 0.8793322443962097, 0.4123384654521942, -0.5457924604415894, -0.15889334678649902, -0.9442898035049438, -0.05206039547920227, 0.2664605677127838, -0.5801018476486206, 1.8163022994995117, 0.7203795909881592, 0.3665424883365631, 0.8624489307403564, 0.6543347835540771, -0.19253408908843994, -1.1158127784729004, 1.6489266157150269, 0.503397524356842, -0.5438928008079529, 0.8895925879478455, -0.009709074161946774, -1.3337781429290771, 0.4074220359325409, 2.8967509269714355, -0.10756280273199081, 1.1406981945037842], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[1.0313529968261719, -0.3150884807109833, -0.6188902854919434, 2.538532257080078, 0.0706467553973198, -1.5841854810714722, -0.8888445496559143, -1.6023554801940918, 0.06989029794931412, 0.0629257932305336, -1.128580927848816, -0.40835461020469666, -0.8164339065551758, -0.719093918800354, 1.3689675331115723, -1.2879984378814697, 0.8248843550682068, -1.9401503801345825, -0.7006307244300842, -1.4582675695419312, 1.0811657905578613, -0.8947410583496094, 0.8071189522743225, 2.4299426078796387, -0.4773719608783722, -0.617012619972229, 0.32926198840141296, 1.7881560325622559, 1.0343681573867798, -0.06301494687795639, 0.41191938519477844, -2.215353012084961, 1.5964537858963013, -0.31820759177207947, 0.3847765624523163, -0.5317594408988953, 0.04676717147231102, -0.6320234537124634, -1.7443276643753052, -1.0173789262771606, 0.5422625541687012, 0.03454035893082619, 0.003138607135042548, 0.26535651087760925, 1.2541356086730957, -1.0447163581848145, -1.3925971984863281, 1.3859058618545532, 0.12027012556791306, 1.7788796424865723, 1.2854117155075073, 2.175891637802124, -0.12428492307662964, -0.9429271221160889, 0.0771043449640274, -0.07366190105676651, -1.391654133796692, 0.3848572075366974, 0.05508342385292053, 0.21518130600452423, 0.7860302329063416, -0.932051956653595, -0.9681017398834229, -1.5873748064041138, 0.15003201365470886, -2.051150321960449, 1.3857760429382324, 0.9352238774299622, 0.04627859964966774, 1.8821451663970947, 2.6254007816314697, -2.6252827644348145, -0.066888727247715, -1.1029824018478394, -0.4591842293739319, -0.21094568073749542, 1.3821568489074707, 0.13969804346561432, -0.9560801982879639, 1.4716778993606567, -1.5922603607177734, 0.5369348526000977, -1.6305042505264282, 2.467165470123291, -0.30310821533203125, -0.15671375393867493, 0.5796441435813904, -1.336880087852478, -1.4811209440231323, 0.8098952770233154, 1.4877773523330688, -0.7892221808433533, 0.016270529478788376, -0.07393363118171692, 2.0853192806243896, -1.3532414436340332, 0.4999488294124603, -0.06506102532148361, -0.08077093213796616, 0.9714716076850891, -0.9760869741439819, -2.273130416870117, -1.1063605546951294, -2.093925952911377, 1.3397685289382935, -0.7416523098945618, -0.23391419649124146, 0.6628796458244324, 0.21479788422584534, 0.4284484386444092, 1.4973076581954956, 2.038489580154419, 1.4338668584823608, -1.362434983253479, 4.19985818862915, -2.199658155441284, -0.7283968329429626, -0.3343336880207062, 3.476289987564087, 0.18779972195625305, 1.7789186239242554, 0.6648249626159668, 0.935764491558075, -1.3286139965057373, -0.015124054625630379, -1.8520011901855469, 1.772568941116333, 0.1196172907948494, 0.6289823055267334, 0.5484312772750854, 2.5777790546417236, -1.638053297996521, -2.1885337829589844, 0.8748598098754883, 1.2684322595596313, 1.5866456031799316, -0.2730933427810669, -1.68460214138031, -0.9831095337867737, -0.005413503386080265, -0.03668975085020065, -0.9531881213188171, 1.4810272455215454, 3.7066919803619385, 0.5170440673828125, 1.261596441268921, 0.9630414247512817, 0.7141573429107666, -2.4101104736328125, 2.2725989818573, -0.35624954104423523, -0.6168355941772461, -0.21304747462272644, -0.22455933690071106, -1.219396948814392, 0.9540411233901978, -0.640645444393158, 1.1605384349822998, 1.7854052782058716, 0.8375619053840637, -0.31674492359161377, 1.4547251462936401, -0.19949570298194885, 1.7815083265304565, -2.2466886043548584, -1.2231472730636597, 0.09401803463697433, -1.4078176021575928, 3.908220052719116, 1.5404845476150513, 0.3114934265613556, 8.456908226013184, 0.9516490697860718, -0.9384443759918213, 1.873389720916748, -2.128218173980713, -2.0169875621795654, -0.3324727416038513, -1.1806225776672363, 2.2858259677886963, 1.0279650688171387, 1.416778802871704, -0.12855340540409088, -1.1148887872695923, -0.7427697777748108, 0.3523867130279541, -0.09803378582000732, 1.1112825870513916, 2.0703094005584717, -2.1148006916046143, -1.1792689561843872, -0.46205198764801025, -3.51401948928833, 0.31079503893852234, -0.6227839589118958, 0.70735102891922, 0.5763919353485107, -1.6779961585998535, -1.5134176015853882, -0.30112114548683167, -0.9538077712059021, -0.7663083076477051, -2.314335346221924, 1.239158272743225, 1.8725138902664185, 3.68015456199646, 0.5789294838905334, -0.6771758198738098, 0.3894679546356201, 0.9258761405944824, 2.237853527069092, 0.6519597172737122, 1.5939924716949463, -0.6719832420349121, -2.3262033462524414, 1.0702043771743774, -1.0043761730194092, -0.7116943597793579, -0.7736619114875793, 0.8041775226593018, 0.1533975899219513, 0.7328961491584778, -1.8436204195022583, 2.3635141849517822, 2.328721523284912, 1.418660044670105, -1.5184180736541748, -1.5892385244369507, -0.9289590716362, -0.4732782244682312, -0.3886764347553253, 0.7170847058296204, -4.419089317321777, 0.4270886480808258, -1.301378846168518, 0.33860573172569275, -1.0704748630523682, 0.9127981066703796, -2.1660208702087402, 1.09011709690094, -0.47915539145469666, 0.07477149367332458, 0.11350961774587631, 0.9929535984992981, -0.14130686223506927, 1.0069708824157715, -0.7110414505004883, -0.059425368905067444, -0.5938720703125, 0.48572811484336853, -0.1784304529428482, 0.39340993762016296, 1.5350433588027954, 0.3256814479827881, 1.585489273071289, 0.21675606071949005, -0.8188936710357666, -0.7702890038490295, -0.7199066281318665, 0.25179529190063477, -1.1087812185287476, 1.0340651273727417, 0.4628245234489441, 0.9636328220367432, 1.8067668676376343, 0.4394457936286926, -2.910259246826172, -0.6441197991371155, -0.9365514516830444, -1.1415481567382812, -0.516469419002533, 0.8398948907852173, 1.122663140296936, 0.7438234686851501, -0.8472678661346436, -1.4711772203445435, 0.18609529733657837, 0.199629008769989, -1.592896580696106, 2.204683780670166, -2.789952278137207, -1.178740382194519, -1.5819144248962402, 2.900160312652588, 0.1955227553844452, 0.8224141001701355, 0.8678563833236694, -0.9557526707649231, 0.31532034277915955, -1.4279404878616333, 1.866892695426941, 2.0798051357269287, 0.34394997358322144, -0.8172627687454224, 0.11930597573518753, -0.7571832537651062, -1.2928004264831543, 0.8617067933082581, 2.466092109680176, 0.594739556312561, 1.4325417280197144, 1.3032231330871582, -0.9912956953048706, -0.5397871732711792, 0.5928817391395569, 1.532958745956421, 0.42002159357070923, 0.2681210935115814, -0.18034633994102478, -0.4648224413394928, -1.1362175941467285, -0.5349517464637756, 0.7576450705528259, 1.6876436471939087, -1.8478913307189941, -0.18839584290981293, 0.12144707888364792, -0.24286077916622162, 0.10009735822677612, -1.121900200843811, 2.9073452949523926, 0.5389974117279053, 2.0264742374420166, 0.9882201552391052, 0.8091688752174377, 0.0998024046421051, 1.0768859386444092, -2.185059070587158, -2.023668050765991, 1.9596855640411377, -0.6531945466995239, 1.0890140533447266, -0.029196487739682198, -0.8863120675086975, 0.6205335259437561, -4.426001071929932, -1.2933393716812134, 1.8228099346160889, 0.2699809968471527, 0.5719003677368164, 1.7204668521881104, 0.10404859483242035, -3.060622215270996, 1.198330283164978, 0.05697685852646828, 0.2345513254404068, 3.77382230758667, -1.5446274280548096, -2.3768885135650635, 2.623532295227051, -0.327999472618103, 1.7193710803985596, -0.890876054763794, 1.1269359588623047, -1.517040491104126, 0.9679018259048462, 0.07385861873626709, 3.2088770866394043, 3.282595634460449, 0.5478740930557251, 1.5781583786010742, -1.324411153793335, -2.0730338096618652, 0.15878775715827942, -2.0381994247436523, 0.040365271270275116, -0.16474825143814087, 1.0122928619384766, 3.7804086208343506, 0.28933390974998474, 0.9740607142448425, -0.35681432485580444, -0.5711200833320618, -0.6149091124534607, 1.1522265672683716, -1.2588331699371338, -0.033982228487730026, -0.23794879019260406, 0.8180074095726013, 1.4753068685531616, -1.7543611526489258, 1.5884774923324585, 0.40308502316474915, -0.052510254085063934, -0.48346254229545593, 1.380664348602295, 1.8346967697143555, 0.384738028049469, 0.2229100912809372, 0.7473107576370239, 0.8702228665351868, -0.8507490754127502, -0.1185961440205574, -1.096237063407898, -1.4117523431777954, -0.1175261065363884, 0.5512189865112305, 1.1995302438735962, 0.29133743047714233, -0.7233541011810303, 1.92998206615448, 1.0880876779556274, 0.8075852394104004, 0.07838087528944016, 0.5083364844322205, 1.3314756155014038, -0.7086207866668701, -0.21401624381542206, -0.47350195050239563, 1.7496318817138672, 2.200195074081421, -1.2069119215011597, 0.927280068397522, 0.6182522773742676, 0.2426895797252655, -1.4004515409469604, -0.3454461991786957, 0.20536530017852783, 0.25387218594551086, 1.6257212162017822, -1.5439250469207764, 0.05316856503486633, -1.8647035360336304, 0.17507831752300262, 3.326417922973633, -2.074596405029297, -0.2814188003540039, -0.44550827145576477, -1.3850098848342896, 0.6345311403274536, -0.23424763977527618, 0.23114243149757385, -0.20717942714691162, -0.7253944873809814, 1.0829041004180908, -0.5123313069343567, 1.7096027135849, -0.2874470353126526, 1.458932876586914, -0.8723434805870056, 0.7356038093566895, -0.7048888802528381, -2.045283079147339, -0.2576136291027069, 0.21477334201335907, -0.5645092129707336, 0.36780908703804016, -1.5985599756240845, -0.40983816981315613, -2.182253122329712, 1.605674147605896, -2.1273128986358643, -1.164692759513855, -1.4987962245941162, -0.3925361633300781, 0.5211796760559082, -1.0425214767456055, -0.5808479189872742, -0.6539576053619385, -0.7426117062568665, 3.572455883026123, -0.36875206232070923, 0.36702147126197815, 2.752495050430298, -0.9144785404205322, -0.7509940266609192, 2.702570915222168, -1.0696463584899902, 1.0711889266967773, 0.6808310747146606, 1.585534691810608, 0.8146730661392212, -0.672609806060791, -0.6912991404533386, -0.43878352642059326, 0.45676007866859436, 1.2491567134857178, -2.0890910625457764, -2.379392385482788, -0.32394328713417053, 0.516779899597168, -1.3433644771575928, 0.9653174877166748, -1.9664781093597412, 0.32960352301597595, -0.38077405095100403, 1.5857247114181519, 2.4176278114318848, 1.4460657835006714, -0.8478822708129883, -1.597184658050537, 0.5195802450180054, 0.763209342956543, -1.4776554107666016, 0.7641628980636597, -2.0951943397521973, 0.661369264125824, 2.0179660320281982, 0.3305167257785797, 0.46654418110847473, -1.0588690042495728, 1.5062310695648193, -1.2197043895721436, -1.2312288284301758, -0.18464970588684082, 1.1752277612686157, 1.1042150259017944, -1.499812126159668, -0.6547158360481262, 1.741197943687439, -1.2148603200912476, 0.035316355526447296, 0.3168410658836365, -1.0427607297897339, -1.5235220193862915, -0.46722182631492615, -3.2869765758514404, -0.038627419620752335, -0.5701324939727783, -1.9721806049346924, -0.7228178381919861, -2.2212235927581787, -1.8982045650482178, 0.030603330582380295, 3.1168458461761475, -0.7755497694015503, 0.23655731976032257, 0.2025589942932129, -2.854536533355713, -2.1581015586853027, 1.380907416343689, -0.15676657855510712, 1.3322430849075317, -0.4515264630317688, 1.3276783227920532, -0.7625574469566345, -0.7090430855751038, -1.4549388885498047, 1.4381734132766724, -1.111768364906311, 0.23955313861370087, 1.0853793621063232, -1.333328366279602, -0.8282040953636169, 0.15890230238437653, 0.8194791674613953, 0.9932834506034851, -0.937415599822998, -1.0855118036270142, -1.1238179206848145, -2.7766451835632324, 0.7058588862419128, -5.507623672485352, -0.6805130839347839, -1.0072022676467896, -0.7985067963600159, -0.562705934047699, 0.45883142948150635, -3.1818549633026123, 0.1730644404888153, 2.9254183769226074, 0.7242662906646729, 1.378713846206665, -0.07413364946842194, 1.360166311264038, -0.7989082336425781, 0.4775853455066681, -0.8554370403289795, 0.5128370523452759, 0.03847278282046318, -1.0285773277282715, 0.6595292687416077, 0.8212839961051941, 2.827463388442993, 1.535107135772705, 0.8293505311012268, 2.842984676361084, -0.16810069978237152, 1.764321208000183, -1.5597331523895264, -0.20066587626934052, -0.8449881076812744, -0.5569947957992554, -0.4739476144313812, 1.560530424118042, -0.4117347002029419, -0.1505243331193924, -1.8822977542877197, -2.0343198776245117, -0.8035334348678589, -0.06630855053663254, 0.3700888156890869, 0.6056109070777893, 0.17372646927833557, 0.523221492767334, -1.3863945007324219, 1.8396508693695068, 0.446188747882843, -4.9483418464660645, 1.0813500881195068, 1.3014492988586426, 0.011439451016485691, 1.0650203227996826, -2.1676461696624756, -0.2484808713197708, 1.5505222082138062, 0.7870309948921204, -2.51863169670105, -0.8128791451454163, -0.2340819537639618, 2.012570381164551, 0.7974970936775208, 0.08622992783784866, 0.2483638972043991, 3.142228841781616, 0.16756805777549744, 0.0623970553278923, 1.3384586572647095, 3.473921775817871, -0.7715830206871033, -0.14742882549762726, -0.4594554901123047, -1.1253880262374878, 2.7894442081451416, 0.6897386908531189, -0.49850475788116455, 0.841233491897583, -1.7460402250289917, 1.0433125495910645, -1.7352337837219238, 0.40853434801101685, -1.9452407360076904, 0.41610947251319885, 0.34417906403541565, 0.6591309309005737, -0.9298162460327148, 1.0705755949020386, 0.11452412605285645, -1.0013099908828735, -0.5620473623275757, -1.0292389392852783, -0.7007811665534973, 0.681995153427124, 1.491309642791748, -0.6530022025108337, -1.1183226108551025, -0.24401770532131195, -0.031386323273181915, -1.2672092914581299, -0.5764185190200806, 0.8465411067008972, 1.8382335901260376, 0.99588942527771, -1.7188267707824707, -0.34578388929367065, -0.6644178628921509, 1.6039870977401733, 0.5527514219284058, -3.6353631019592285, -1.940133810043335, 0.9295077919960022, -0.3120937645435333, -0.07290239632129669, 1.274749517440796, 0.03177739307284355, 0.4692472219467163, -1.7704219818115234, -0.6613892912864685, -0.8865581154823303, -1.4224265813827515, -1.681904911994934, 0.0055172196589410305, -0.04089910164475441, 0.377922385931015, 1.1132705211639404, 0.19697688519954681, -0.012553743086755276, 1.7976200580596924, -0.5712243914604187, 0.24934391677379608, 0.36582863330841064, -0.14150266349315643, -0.9380295276641846, -0.7064430713653564, -0.6870579123497009, 1.0194010734558105, 3.1322033405303955, 1.2966750860214233, 1.2144407033920288, -1.773240566253662, 0.8313677906990051, 0.3890945315361023, 2.649280309677124, 0.2261660099029541, -1.034723162651062, -0.9035531282424927, 0.9502410292625427, -0.5193495154380798, -0.6502906680107117, -0.7278847098350525, -1.3221287727355957, -2.2207422256469727, -1.6090668439865112, -0.18947218358516693, -0.9743614792823792, -1.0227123498916626, 1.7154704332351685, 0.5656698942184448, 0.5287024974822998, -0.3732090890407562, 1.2332013845443726, 0.164259135723114, 2.3760085105895996, 3.506052255630493, -0.7550541162490845, -0.21760836243629456, -2.748678684234619, 1.8903555870056152, 1.7760050296783447, -0.5398191213607788, -0.36867377161979675, 0.7620857357978821, 1.4919395446777344, 0.9702889323234558, 0.8432427048683167, 0.19680491089820862, 0.18513670563697815, -1.2044181823730469, 0.2739716172218323, 0.9177528023719788, -1.9240866899490356, -1.221963882446289, 0.1936912089586258, -0.6056318879127502, -2.2703957557678223, 0.8691372275352478, -0.5002903938293457, 0.9790436625480652, -1.372728705406189, 0.195281982421875, -0.5267977118492126, -0.07843907177448273, -1.1842321157455444, 0.5355440378189087, 0.10992851853370667, -1.2343699932098389, -0.6765578389167786, -0.4333585798740387, -0.0820227861404419, 0.42915409803390503, 0.5139772891998291, 2.667320728302002, 1.8733638525009155, -1.3132587671279907, 0.8632371425628662, 0.18760159611701965, -0.17112885415554047, -1.337545394897461, 1.6222859621047974, 0.5427638292312622, -0.7163363695144653, 0.7651683688163757, 2.8138935565948486, -1.7863121032714844, 0.2621382176876068, 1.9176969528198242, -0.5743048787117004, 1.4255201816558838], [1.2666213512420654, -0.337955117225647, -0.7283499240875244, 0.21169331669807434, 0.7140598893165588, -1.4961315393447876, -0.47679033875465393, -1.5478246212005615, -1.8598836660385132, 0.19002896547317505, 2.0533549785614014, -0.4555533528327942, 0.035828642547130585, -0.6798750162124634, 1.1775109767913818, -2.633420467376709, 1.1499179601669312, -1.0413367748260498, -0.6575953960418701, -0.28306975960731506, 0.5457611680030823, -0.9009673595428467, 0.7876397967338562, 1.3649095296859741, -1.3285657167434692, -1.2729007005691528, -0.6780670285224915, 0.19991880655288696, 1.4531420469284058, -2.0000123977661133, 0.6301437020301819, -4.121349811553955, 1.1049011945724487, -0.020656703040003777, 1.494479775428772, 0.09898772835731506, 0.26842403411865234, -0.8176325559616089, -0.4708546996116638, 1.5518125295639038, -0.41067713499069214, -0.5762508511543274, -0.23137618601322174, 2.1862244606018066, 0.5094833970069885, -1.9158824682235718, -1.3782751560211182, 0.2072494775056839, -0.3556337356567383, -0.9331575632095337, 0.6223523020744324, 1.5244115591049194, 0.8664129376411438, -1.1854747533798218, 0.08444260060787201, -0.680433988571167, -0.9435390830039978, -1.4431668519973755, -0.02178388275206089, -0.2741550803184509, 0.8787142038345337, 1.4663078784942627, 0.5135278701782227, -1.5520856380462646, 0.8367644548416138, -1.8702600002288818, 1.8511996269226074, 0.866259753704071, 0.7594473958015442, 2.0759782791137695, 2.9246346950531006, -6.079822540283203, -0.9153985381126404, -1.8316209316253662, -0.1293099969625473, -0.2551783621311188, 2.374638319015503, -0.42763659358024597, -3.823439836502075, -2.5570759773254395, -0.8203209042549133, -0.7137419581413269, -0.9262862801551819, 1.6814773082733154, 2.0184237957000732, -1.6464595794677734, 0.7424294352531433, -1.2750205993652344, -1.1513491868972778, 1.6007022857666016, 0.8499367237091064, -0.7091137170791626, 0.034360453486442566, 1.0639227628707886, 2.300348997116089, 0.0872102752327919, 1.0592845678329468, -0.10728754848241806, 0.35188934206962585, -1.736466407775879, -2.2465405464172363, -0.1138622984290123, 4.4629364013671875, -0.9834140539169312, -0.8231449723243713, -0.8173891305923462, -0.11197089403867722, 0.9931103587150574, 1.0087333917617798, -0.5190882682800293, 1.2641942501068115, 0.6611023545265198, 1.4554402828216553, -1.5018563270568848, 4.229107856750488, -1.7977527379989624, -0.9393377304077148, -0.3127431869506836, 1.1459424495697021, 0.45688965916633606, 1.7157970666885376, -1.8048220872879028, 0.14239275455474854, -1.860568881034851, -0.531540036201477, -1.9364814758300781, 1.695343017578125, 1.0309809446334839, 2.2270803451538086, 0.47645094990730286, -1.04046630859375, -2.3718020915985107, -2.0344574451446533, -1.1910569667816162, -2.0283782482147217, 1.760874629020691, -0.5304137468338013, -0.24521885812282562, -2.879619598388672, -0.35029757022857666, 0.16458433866500854, -1.1279442310333252, 0.6504403352737427, 0.8097184896469116, 1.326963186264038, 1.6148737668991089, -0.12715500593185425, 1.0613545179367065, -1.9504268169403076, 2.0086922645568848, -1.1679524183273315, -1.069165587425232, 0.04321211948990822, -0.6092818975448608, -1.7346549034118652, 4.095265865325928, 1.0187233686447144, 1.0767993927001953, 1.509487271308899, 0.713977575302124, 0.3319104313850403, 2.274949073791504, -0.13385066390037537, -0.3157047927379608, -0.4362070858478546, -1.1517025232315063, -0.8612220883369446, -2.5852344036102295, 2.909168004989624, 1.4983320236206055, 0.595609188079834, 7.709740161895752, 1.3116332292556763, -0.1282147318124771, 2.2561635971069336, -1.4225037097930908, 0.5605979561805725, -0.24966081976890564, -1.3057247400283813, 2.4009454250335693, -1.0751415491104126, 0.3688514828681946, -0.13411927223205566, -3.270512342453003, 0.44894692301750183, 0.5026993751525879, -2.606612205505371, 0.26751863956451416, 0.9732317924499512, -0.6171296834945679, 1.1479005813598633, 0.38327035307884216, -3.5078718662261963, -0.6723376512527466, -1.561919927597046, -0.209125354886055, 0.2156764268875122, -3.060631513595581, -2.5920495986938477, -0.19702363014221191, -0.931863009929657, -1.2888503074645996, -1.6206828355789185, -0.9983268976211548, 1.1096491813659668, 2.259078025817871, -0.5457378029823303, -1.2957382202148438, 0.31612783670425415, 1.1479227542877197, 1.6323081254959106, 0.5656882524490356, -0.6331796646118164, -1.1338717937469482, -2.301689386367798, 1.0710631608963013, 1.543956995010376, -1.4092925786972046, -0.8232712149620056, 1.5564157962799072, 0.16337703168392181, 1.3816158771514893, -0.14898033440113068, 0.8707237839698792, 3.698439836502075, 2.0459651947021484, 0.749133825302124, 0.44700556993484497, -0.7008357048034668, -0.29145348072052, -0.34754693508148193, 0.6231390237808228, 1.2843283414840698, -2.7487804889678955, -1.1227461099624634, 0.9564064145088196, -2.684223175048828, 0.14786745607852936, -2.5547423362731934, 1.0982463359832764, -0.6995810270309448, 1.318792700767517, 0.21491844952106476, 1.0851792097091675, -0.4444516599178314, 2.0984694957733154, -0.9276931881904602, 0.8195655941963196, -3.898178815841675, 0.12049306184053421, 0.27656057476997375, 0.6026123762130737, 1.1408156156539917, -0.19398623704910278, 2.286583423614502, -0.09284382313489914, -3.0531232357025146, -0.11307107657194138, -1.0013940334320068, -0.6528171896934509, -1.0691725015640259, 3.26694917678833, -0.30519163608551025, 0.7890059351921082, 1.667021632194519, -0.22176890075206757, -2.1299209594726562, 1.1088320016860962, -0.23651693761348724, -0.7463411092758179, -1.3403772115707397, 1.6595405340194702, 0.8714126944541931, -0.7402191758155823, -0.9371410012245178, -0.9164403080940247, 0.18974991142749786, 0.41288846731185913, 1.494828224182129, 1.9594393968582153, -2.0246615409851074, -2.244507074356079, -1.2390865087509155, 1.4853709936141968, -0.24574974179267883, 0.6566565632820129, 0.3843611478805542, -0.5400854349136353, -0.8514088988304138, -0.2755645513534546, -1.0397510528564453, 1.6958900690078735, 0.08188910782337189, -0.5261552333831787, 0.7258949875831604, 1.1260014772415161, -2.0281460285186768, 0.4229588806629181, 0.01884511113166809, 1.4434019327163696, 0.6821380257606506, 2.276242733001709, -0.923797070980072, 2.5163466930389404, -0.000586011738050729, -0.3613157868385315, -0.35500916838645935, 0.1838628649711609, -0.15112879872322083, -0.9097641706466675, 0.04370853304862976, -0.43482059240341187, 0.05543801933526993, 1.714239239692688, -1.8770169019699097, -0.4243658483028412, 0.12206720560789108, 1.0837123394012451, 2.867783784866333, -0.5128921866416931, 3.6838321685791016, -0.14977571368217468, 0.5914438962936401, 1.6622422933578491, 0.058062393218278885, -0.32345816493034363, 0.869326114654541, -2.4923887252807617, -1.0653451681137085, 2.039658784866333, -0.5493632555007935, 0.7256235480308533, 1.112501621246338, -0.9674397110939026, 0.22923476994037628, -2.609079122543335, 0.04606340080499649, -0.6377202868461609, 0.4773648679256439, 2.549133539199829, -0.5600903630256653, 0.213691845536232, -2.3791439533233643, 2.5235886573791504, -1.1629329919815063, 0.22674055397510529, 2.3390071392059326, -1.21358060836792, 2.282510757446289, 1.7667903900146484, -0.37937694787979126, 1.3503046035766602, -1.0918689966201782, 2.2101595401763916, -1.3876917362213135, 1.5299454927444458, -0.17502498626708984, 0.12888440489768982, 0.49031755328178406, -1.1093024015426636, 1.5109357833862305, -0.5163214206695557, -1.7494546175003052, -0.26686739921569824, 0.5254916548728943, 0.11007532477378845, -0.3422134518623352, 0.46277230978012085, 2.538316249847412, -2.5786478519439697, -0.32949644327163696, -0.2017969787120819, -1.2153459787368774, -1.3356959819793701, 1.1130281686782837, -1.8967630863189697, -1.3903708457946777, -1.0846444368362427, 1.0963574647903442, -1.150165319442749, -1.990694284439087, 2.777627944946289, -1.5817416906356812, 0.9633862972259521, 1.0282212495803833, 1.4848929643630981, 2.4463579654693604, 0.6080212593078613, 0.03359531983733177, -0.4621783196926117, 1.0027879476547241, -0.84768146276474, -0.1438533216714859, -0.3446374535560608, -0.16716408729553223, -0.1995219737291336, -2.735494613647461, 1.1815497875213623, -0.566275954246521, -0.9084722399711609, 0.3854261040687561, 1.221024990081787, 0.8092287182807922, 1.1748323440551758, 0.7044814825057983, 1.9319735765457153, -0.5077551603317261, -0.2354898303747177, -1.4354983568191528, 1.7502349615097046, 2.022625207901001, -0.6217110753059387, -0.9472346305847168, -4.193853378295898, 1.9242839813232422, -1.1617653369903564, -2.1114401817321777, 0.2460455596446991, -0.518820583820343, -0.24764801561832428, -1.3073142766952515, 1.4401133060455322, 2.4712719917297363, 0.3620019257068634, -0.009558808989822865, -1.2673609256744385, -0.9028863310813904, -0.5738696455955505, 0.4386255145072937, 1.1184767484664917, -0.3213818669319153, 0.009588712826371193, 1.064178705215454, -0.3106573522090912, 0.6991330981254578, -1.8925817012786865, 0.29123398661613464, -1.7144250869750977, 1.6368064880371094, -0.8880687355995178, 0.508480966091156, -0.5516143441200256, 1.2871772050857544, 0.08896538615226746, 0.43641942739486694, -1.3009130954742432, -1.0493111610412598, -1.7530163526535034, 0.4353236258029938, -1.9912590980529785, 1.3311976194381714, -1.387077808380127, -0.727538526058197, -0.602639377117157, -0.6740429401397705, 0.8660916090011597, -1.6038134098052979, -1.4072978496551514, -0.6825183629989624, -0.7268009185791016, 0.9358657598495483, 0.29602137207984924, 0.37955307960510254, 1.2822768688201904, -0.9973181486129761, -0.6962331533432007, 1.880584478378296, -1.0969700813293457, 3.678732395172119, 0.5359619855880737, 1.2266666889190674, -0.19833548367023468, -0.8743430376052856, -0.7167401909828186, -0.6406034231185913, -0.4400770366191864, 1.3121215105056763, 0.5029030442237854, 0.18280228972434998, 0.9241054654121399, -0.06872769445180893, -1.471947431564331, 0.43067362904548645, -0.8261715769767761, -0.0204215869307518, -0.4053501784801483, -1.254148006439209, 0.8784874677658081, -1.669633388519287, -1.12124502658844, -0.2071349173784256, 0.06629762798547745, 0.6053402423858643, -2.373958110809326, 1.2905521392822266, -0.9397355318069458, 0.7312015891075134, 1.8694218397140503, -0.6236432790756226, -0.5399772524833679, -1.0360654592514038, 0.406002402305603, -0.1545528918504715, 0.2178913950920105, 1.057570457458496, 0.4224735200405121, 0.06820528209209442, -1.4103589057922363, -0.46555936336517334, -2.424769639968872, -0.2013772577047348, 0.05200432613492012, 0.5657323598861694, 3.658541679382324, -1.2962696552276611, -0.4278543293476105, 2.5443999767303467, -0.044882357120513916, -0.5544663667678833, -2.0395681858062744, -2.2622809410095215, -1.760406255722046, 0.6982283592224121, 1.7605433464050293, 1.3508118391036987, -0.7603340744972229, -0.44194748997688293, 0.07142424583435059, -1.0621412992477417, -2.05460262298584, 1.3153384923934937, -0.20577818155288696, 2.770369291305542, -0.33939874172210693, 1.8001519441604614, -0.7013277411460876, -4.677796840667725, -1.595847725868225, 0.8467330932617188, -2.2476258277893066, -1.1397769451141357, 1.9821574687957764, 1.0501477718353271, 0.9321050643920898, 1.5326858758926392, 0.8539840579032898, 0.9961287379264832, -1.3351211547851562, -1.0739030838012695, -0.771928071975708, -2.524604558944702, 0.9038081169128418, -1.7466715574264526, -1.4319491386413574, -0.1592305600643158, -0.7635988593101501, -0.5520546436309814, -0.1081007793545723, -4.438163757324219, -1.490625023841858, 5.630952835083008, 0.7877073287963867, 1.2779237031936646, -0.28818148374557495, 3.0071470737457275, -0.8375856876373291, -2.3146932125091553, -0.7524455189704895, 0.6035643815994263, 1.702111840248108, -0.9022802114486694, 0.6851896047592163, 1.1931334733963013, 2.163761854171753, 1.8234875202178955, 0.6252201199531555, -2.7243568897247314, -0.2887050211429596, 1.646515130996704, -5.7990593910217285, 0.8917678594589233, -1.174891471862793, 1.0559221506118774, -0.6831675171852112, -0.7957342267036438, -0.4042673707008362, -0.1669808328151703, -1.3715338706970215, -0.5116485953330994, -0.8209608793258667, -0.4749082326889038, 0.3810811936855316, 0.5642752647399902, 0.5291991829872131, 0.39952072501182556, -0.42093220353126526, 1.827691674232483, 1.349023699760437, -1.062636137008667, 0.9740330576896667, 1.3221720457077026, -3.877678155899048, 0.9820412397384644, -0.34293729066848755, -1.5068745613098145, 1.614742636680603, 1.3174325227737427, -1.7520235776901245, -1.184876561164856, -0.6854851841926575, -1.3074239492416382, 0.27040618658065796, -2.689344882965088, -1.844893217086792, 0.24325808882713318, 0.6977813839912415, 0.25569409132003784, 1.5013480186462402, -1.2121930122375488, -1.1676154136657715, 0.7233038544654846, -1.0554149150848389, -1.0763275623321533, -0.48056671023368835, 0.48894378542900085, -0.49635249376296997, 0.12226247042417526, -1.1544201374053955, 0.9921833872795105, -1.1092243194580078, 0.37341704964637756, -1.1418793201446533, 0.2256171554327011, 2.9101858139038086, 1.697235107421875, 0.3632856607437134, 1.2154592275619507, 1.0429623126983643, -0.5450078845024109, 0.3872164487838745, -0.7809674739837646, -0.5829377770423889, -0.22034023702144623, 2.16870379447937, -0.900214672088623, -1.154813528060913, 0.04087097570300102, 0.5882806181907654, -1.4756879806518555, 0.09442747384309769, 1.7242356538772583, 1.3158063888549805, 1.0644270181655884, -1.8510111570358276, 0.4923301637172699, -0.23679567873477936, 2.420781135559082, 0.7776903510093689, -0.16522184014320374, -1.0454503297805786, 0.831852912902832, -0.5364313721656799, -0.0963619202375412, 0.4977725148200989, -0.049460966140031815, 0.285511314868927, -1.6806787252426147, -2.2074527740478516, -1.8697295188903809, -1.5820106267929077, -0.11418074369430542, -0.752642035484314, -0.8109595775604248, 0.407311350107193, 0.7373819947242737, -0.04891572520136833, 0.5114012956619263, 2.2258219718933105, -1.3814263343811035, 0.23601649701595306, 1.2060245275497437, -1.2738147974014282, -0.2593626379966736, -0.9173689484596252, -0.7866665720939636, 0.7534820437431335, 1.2677923440933228, -0.36367669701576233, 0.886667788028717, -0.18528331816196442, 1.1082185506820679, 0.38475900888442993, -1.6635154485702515, -0.15411783754825592, -0.09419464319944382, -0.20101776719093323, 2.4003453254699707, -0.48215922713279724, 1.0737102031707764, -1.0140769481658936, 1.0673390626907349, -2.223921775817871, -1.3781160116195679, -0.08284236490726471, -1.0132596492767334, -1.2802354097366333, -0.01909996196627617, 0.5764838457107544, -0.8984781503677368, 2.243704319000244, 1.1238882541656494, -0.7295228242874146, -0.7887347340583801, -1.8500083684921265, -0.7763918042182922, 0.7052624225616455, 2.555072546005249, -0.02749663218855858, -4.275672435760498, -0.6485704183578491, 1.036630392074585, 1.0694674253463745, 0.8343238830566406, 1.2030360698699951, -1.8202309608459473, -2.210404634475708, -0.15054304897785187, -1.2424211502075195, 0.528884768486023, 1.878453254699707, 0.5301186442375183, -0.4823702573776245, 0.2592843770980835, 0.13807637989521027, 0.5565330386161804, 0.8563061356544495, -1.0683982372283936, 1.0970090627670288, -0.7718018293380737, -0.8781547546386719, 0.25920426845550537, 1.4723048210144043, -0.6551247239112854, 0.4638732671737671, -0.8167392611503601, -1.0713549852371216, -0.5377241373062134, 0.25496429204940796, -0.10525069385766983, 0.3218867778778076, 0.24265649914741516, 0.6931348443031311, 2.249422311782837, 0.3033846616744995, 0.8646498322486877, 0.14421363174915314, -0.055905845016241074, -1.0261889696121216, 2.121366500854492, 0.8397253751754761, -0.696793258190155, 0.10681721568107605, -1.1477106809616089, -0.9055106043815613, 0.635818600654602, 1.8808573484420776, 0.7792455554008484, -0.2680317461490631], [0.5409233570098877, -0.33670997619628906, -0.8779542446136475, 1.7758114337921143, 0.35216209292411804, -1.3403493165969849, 0.3349670469760895, -1.5761668682098389, 0.5763559937477112, 0.05703983083367348, -0.22154508531093597, -0.61295485496521, -1.5750384330749512, -0.7191256284713745, 1.1336095333099365, -1.4214694499969482, 1.1192165613174438, -0.696686327457428, -0.6270676255226135, -1.6632126569747925, 0.4880425035953522, -0.8600047826766968, 1.0582396984100342, 1.3746541738510132, 0.16475751996040344, -1.1337695121765137, -0.6749975681304932, 2.2320241928100586, 1.5740172863006592, -0.979834794998169, 0.7817791104316711, -4.967584609985352, 0.4015330970287323, 0.2552413046360016, 1.8666226863861084, 0.17924544215202332, 1.290374994277954, -0.8150427937507629, 0.20861563086509705, 3.3957040309906006, -0.22723594307899475, -0.01328335516154766, 0.13198839128017426, -1.3215234279632568, 0.19289474189281464, -1.7070060968399048, -1.352290391921997, 0.19070622324943542, 0.5145903825759888, 0.822947084903717, -1.0550097227096558, 1.163755178451538, 1.547424077987671, -0.3588345944881439, -0.22273705899715424, -0.3575500547885895, -0.9146308898925781, -0.5960958003997803, 0.06330996751785278, 0.9511986374855042, 0.8841111660003662, -0.6008511781692505, -0.1340138018131256, -1.512707233428955, -0.39367714524269104, -0.3640992045402527, 1.6314013004302979, 1.299889087677002, -0.6032727360725403, 2.027519464492798, 1.623005986213684, -0.04028434678912163, 0.682826578617096, 1.6177078485488892, 0.5634912848472595, 0.3652028739452362, 1.1950887441635132, 0.0008562035509385169, -0.10130560398101807, 0.1393475979566574, -0.8917893767356873, 0.8799130916595459, -0.6947159767150879, 0.40854740142822266, -0.6572965383529663, 0.26759302616119385, 0.36815768480300903, -1.2611899375915527, -2.1784191131591797, 1.3228975534439087, 0.9519594311714172, -0.030143415555357933, -0.09734861552715302, 0.41981279850006104, 0.7192769646644592, -1.7146778106689453, -0.08539360761642456, -0.2397371232509613, 0.3677562177181244, -1.9975316524505615, -0.40323665738105774, -0.9288606643676758, 0.8225119113922119, 2.2731010913848877, -1.9982479810714722, -0.7530648112297058, -0.05927399545907974, 0.32518425583839417, 2.1366372108459473, -0.8567782044410706, 0.8840399384498596, 0.5708460211753845, 0.002618661615997553, -1.4223531484603882, 2.032512664794922, -1.8103471994400024, 1.5962762832641602, -0.28422120213508606, 0.4832797646522522, 0.3373764157295227, 1.825077772140503, -0.10507892072200775, 1.7721002101898193, -0.734601616859436, -0.21081764996051788, -2.0196030139923096, -0.04176803678274155, 0.7769493460655212, -0.5147539377212524, 0.8301557302474976, 1.561092734336853, -2.5468921661376953, -2.3322298526763916, -0.9439097046852112, -0.1677941381931305, 1.7529321908950806, -0.3540970981121063, 0.21770644187927246, 0.01577993482351303, 0.81854647397995, -0.45683392882347107, 0.3795474171638489, 0.6828618049621582, 4.2134318351745605, 0.7626435160636902, -0.5019229650497437, -0.786279022693634, 0.9411162734031677, -1.2359280586242676, 2.37213397026062, -2.6288809776306152, -1.1293771266937256, -0.5783157348632812, -1.321256160736084, -1.0016099214553833, 0.9506701827049255, 0.9327508211135864, 1.1053107976913452, 2.658576250076294, 0.8573741912841797, -0.2705852687358856, -0.03875406086444855, -0.332306832075119, 0.24115225672721863, -0.31521913409233093, -1.1069859266281128, -0.2422357201576233, 0.24827750027179718, 2.79445743560791, 1.1638073921203613, 0.3498776853084564, 5.980013847351074, 1.302839994430542, 1.0682936906814575, 1.717881679534912, -1.2767549753189087, 0.06316424906253815, -0.38594239950180054, -1.1077065467834473, 1.7573989629745483, -0.3756389915943146, 0.00843227468430996, -0.14446178078651428, -0.4774201214313507, -0.30307549238204956, -0.8245825171470642, -1.8816615343093872, 0.6508921980857849, 1.8868085145950317, -1.943850040435791, 0.8346880078315735, -1.0323423147201538, -3.542567014694214, 0.5326938033103943, 0.8599198460578918, 0.5137283205986023, 0.47349920868873596, -1.2448904514312744, -2.4672691822052, -0.8698995113372803, -1.0376040935516357, -0.11948588490486145, -0.20583876967430115, 0.24774512648582458, 0.41877490282058716, 1.6714597940444946, -0.5235021710395813, -0.726257860660553, 0.31387344002723694, 0.14710432291030884, 2.777242660522461, -0.3875109553337097, -2.1015443801879883, -1.3104411363601685, -2.1647322177886963, 1.0699154138565063, -0.24253270030021667, -0.7978881001472473, -0.8102474808692932, 1.0501978397369385, 0.22184953093528748, 0.9069471955299377, -3.4064810276031494, 1.6141256093978882, 3.374384641647339, 2.5738580226898193, -0.4319864809513092, -1.533432126045227, -0.664993405342102, 1.0134996175765991, -0.34329110383987427, 0.5175932049751282, -1.66316556930542, 0.8478695154190063, 0.5783368945121765, 3.935486316680908, -0.858604371547699, -0.2745034396648407, 0.6573365330696106, 1.1296954154968262, -0.8306589722633362, 1.2748075723648071, 0.36201754212379456, 1.2137141227722168, -0.7039700150489807, 4.748945236206055, -0.8925737142562866, 1.7480663061141968, -1.3453785181045532, 0.45711395144462585, 0.9720177054405212, -0.3302411437034607, 0.3069714903831482, -1.172764778137207, 2.1191632747650146, 1.2848198413848877, 0.7534928917884827, 0.04278440773487091, -0.9324374198913574, 0.2751338481903076, -2.0995545387268066, 1.7075003385543823, -0.001447606598958373, 0.7839783430099487, 0.9763069152832031, 2.542097568511963, -2.280200719833374, -3.2980082035064697, -0.14060594141483307, 1.3047322034835815, -0.36294952034950256, 0.08253956586122513, 0.8249363303184509, -0.08325473964214325, -0.9023986458778381, 2.9879698753356934, 0.2008410394191742, 0.47222602367401123, 0.23594294488430023, -1.5642712116241455, -1.9045896530151367, -2.406799077987671, -2.691774606704712, -1.5282480716705322, 0.06008221209049225, 0.07112109661102295, -0.493956059217453, -1.7159266471862793, 0.38178887963294983, 0.32243263721466064, -1.4983892440795898, 2.2048609256744385, 1.084818720817566, 0.7512157559394836, 0.2914339005947113, 0.6178867220878601, -1.5715314149856567, 1.0337938070297241, 0.7940024137496948, 2.2128915786743164, 1.3307719230651855, 1.7841229438781738, -1.031941294670105, 2.2806344032287598, -0.06096987798810005, -0.6004611253738403, 0.1968381106853485, 0.11441555619239807, -0.05080481618642807, 0.11360478401184082, -0.6922829151153564, -0.7627979516983032, -0.30910658836364746, 5.433141708374023, -0.37501034140586853, 1.899689793586731, 0.0528101772069931, 0.7026332020759583, 3.462397813796997, -1.2002530097961426, 2.598313331604004, 2.2498767375946045, -0.473532497882843, 0.9504516124725342, 1.1362736225128174, -0.18981929123401642, 0.9575061202049255, -2.2423465251922607, -0.10788066685199738, 0.9069114923477173, -0.6405878663063049, 0.45447275042533875, 0.21463336050510406, -0.525355875492096, 0.056605566293001175, -0.9883722066879272, -2.244281053543091, 1.709320306777954, 0.14749613404273987, 1.761307716369629, -0.04433421418070793, -0.30482256412506104, -0.5460511445999146, 1.7877581119537354, 0.3617714047431946, 0.219872385263443, 5.4082841873168945, -1.0328567028045654, -1.3822935819625854, 0.853663980960846, -0.40479958057403564, -2.046164035797119, -0.6970198750495911, 1.0138908624649048, -1.5010346174240112, 1.26659095287323, 0.026270180940628052, -1.6331007480621338, 2.3895182609558105, 0.09160959720611572, -0.9346303343772888, -1.1477395296096802, -1.719883918762207, -0.1291816532611847, -1.1801332235336304, 0.27781760692596436, -0.259250670671463, 0.3336954116821289, 1.8736363649368286, -1.023434042930603, -0.32663771510124207, 0.10896851122379303, -1.7752124071121216, -1.226806640625, 1.2318754196166992, -0.0020029626321047544, 1.9914294481277466, -0.17567773163318634, 1.8487229347229004, -3.9872593879699707, -2.143575429916382, 1.757695198059082, 3.2136123180389404, 0.9698382616043091, -0.24409128725528717, 1.6122511625289917, 2.0879616737365723, 0.7775586843490601, 0.08948610723018646, 1.3089255094528198, 1.389959692955017, -0.8456336855888367, -0.435921311378479, -0.02926686219871044, -0.2019890993833542, -0.2037091702222824, -0.38054516911506653, 2.779348611831665, -0.16159911453723907, -0.8993850350379944, 1.2246081829071045, 1.0910580158233643, 0.8085021376609802, -0.12746387720108032, 1.423985481262207, 1.7135889530181885, -0.6496643424034119, -0.2353096604347229, -0.15982311964035034, 1.7485036849975586, 1.0324586629867554, -0.7788601517677307, -3.680854082107544, 3.6741740703582764, -1.233878254890442, -1.6992065906524658, 0.45055264234542847, 0.32326340675354004, -0.7233335375785828, 2.6898269653320312, -1.3390737771987915, 0.5419459342956543, 1.1820358037948608, 1.1292521953582764, -0.1951877921819687, -1.545785665512085, 0.013779263012111187, -0.3501608073711395, -11.81076717376709, -0.10528993606567383, 0.060530439019203186, 0.25727754831314087, 1.8912615776062012, -0.08687704801559448, 1.249903917312622, 0.918862521648407, 1.2119120359420776, 0.7066618800163269, -0.3701860308647156, -0.8693373203277588, 1.3346916437149048, -0.6123822331428528, -0.1510031521320343, -0.627680778503418, 0.3323342800140381, 1.3017780780792236, 1.3659206628799438, -0.8457314372062683, -0.42458969354629517, 0.05159836634993553, 1.773821234703064, -1.608946442604065, -1.1909809112548828, -0.7347940802574158, -0.9750853180885315, 1.319102168083191, 0.012774640694260597, -1.027410626411438, -0.5217087268829346, -0.39455726742744446, 4.301028728485107, -1.0995646715164185, 0.497602254152298, 3.49617600440979, -2.2118358612060547, -1.0307748317718506, -0.8724682331085205, -1.0886119604110718, 1.0117489099502563, 0.8028574585914612, 0.8626073598861694, 1.6563714742660522, -0.7493600249290466, -0.07397221028804779, -0.38546615839004517, 0.0029170685447752476, -3.7356982231140137, -0.8088186979293823, -0.3199031352996826, 4.010232448577881, 0.10667417198419571, -1.385623574256897, -0.19099655747413635, -0.4081169068813324, -1.8461533784866333, -0.5503784418106079, -1.6867748498916626, 2.221729040145874, -1.1494712829589844, -1.4609705209732056, -0.8472468852996826, 0.5581212043762207, 1.881205677986145, -1.6098055839538574, 1.1937966346740723, -0.6153354048728943, -3.2303342819213867, 0.6567714214324951, -1.64559006690979, -2.6436996459960938, -0.9947298765182495, 0.773294985294342, 4.7504377365112305, -0.6028717756271362, -3.056234836578369, 1.0576891899108887, 0.11362324655056, -0.8809212446212769, -0.5625767111778259, -0.9562066793441772, 1.1494301557540894, 0.007510934490710497, 0.9506903886795044, 1.071189522743225, 1.2018406391143799, -0.5383005738258362, -1.0713425874710083, -0.11157196760177612, -0.5221003890037537, -2.0222792625427246, -0.5339260697364807, -1.9127370119094849, -0.6731656789779663, 2.8381094932556152, 3.860689163208008, -0.8863814473152161, 0.24824416637420654, -0.07780344784259796, -0.09828917682170868, -2.654977560043335, 0.8802133202552795, -2.086674928665161, 3.0442678928375244, -0.31901881098747253, 1.4287176132202148, -4.405364990234375, 0.38442331552505493, -1.1002870798110962, 0.8141859173774719, -1.136061429977417, -0.3553239703178406, 0.9559133648872375, -2.449063777923584, 1.1479425430297852, 0.7027324438095093, 1.3669931888580322, 0.995631754398346, -0.6890227794647217, -0.8732958436012268, -1.2917735576629639, -2.7859818935394287, 0.8419598340988159, -3.8884294033050537, -0.8785326480865479, -0.27693799138069153, -0.8086894750595093, -0.5693278908729553, -0.08966817706823349, -2.95184588432312, -0.6484495997428894, 2.7310402393341064, 0.7312508821487427, 1.7351484298706055, 0.036919135600328445, 1.0179708003997803, -1.3126477003097534, -2.1836979389190674, -0.5640466213226318, 0.5635814666748047, 0.3474675416946411, -0.12052486836910248, 0.6819081902503967, 0.6688389778137207, 1.3502929210662842, 1.674500584602356, 0.7285250425338745, 2.6015710830688477, -0.29740071296691895, 0.3150658905506134, -0.2518901824951172, -0.9981027841567993, -0.3942350745201111, 1.1618211269378662, -2.1486294269561768, -0.3474629521369934, -0.379556268453598, -0.20210501551628113, -1.707087755203247, -0.9596887826919556, -0.7847141027450562, -0.6968659162521362, 0.35841962695121765, 2.1397104263305664, 1.2876688241958618, 0.9009198546409607, -0.6092892289161682, 1.472222089767456, 0.9422630667686462, -4.158468723297119, 0.9425749182701111, 1.4414827823638916, -0.20776362717151642, 2.143991231918335, -0.4583117365837097, -0.09918538480997086, 1.5018253326416016, 0.8789636492729187, -1.557450532913208, -0.5603313446044922, -0.0959782749414444, -0.9345089793205261, 1.0647374391555786, -1.5364913940429688, -1.2134660482406616, 2.282783269882202, 0.0022841100580990314, 0.2508571743965149, 1.3851629495620728, -2.1163218021392822, -1.1430773735046387, -2.2299447059631348, -0.8616895079612732, -1.2085192203521729, 1.7015713453292847, 0.5615023374557495, -0.4984452426433563, 0.9209245443344116, -1.0082238912582397, 0.8099488019943237, -1.3466955423355103, -0.02442913129925728, 0.8090605139732361, 1.6031469106674194, 0.8488610982894897, 0.6525232195854187, -1.5288054943084717, 0.6046712398529053, 0.04803920164704323, 0.6139416694641113, -1.4483141899108887, 1.2842568159103394, -1.1355177164077759, -0.617262601852417, 1.2981491088867188, -0.8791682720184326, -1.4011590480804443, -0.45504581928253174, 0.9612732529640198, -1.2319048643112183, -0.12917062640190125, 1.5241899490356445, 1.884045124053955, 1.360397219657898, -1.0673956871032715, 0.13771283626556396, 0.8416607975959778, 1.101659893989563, 0.42658156156539917, -1.2894634008407593, -1.5570764541625977, 0.9835519194602966, -0.5079745650291443, -0.055178262293338776, 1.4884198904037476, 0.17040114104747772, 1.116737723350525, -0.23282957077026367, -0.2188362032175064, -0.4012734889984131, -1.2613667249679565, -0.7640572786331177, -0.3063177466392517, 1.0328212976455688, 0.45769259333610535, 0.386888325214386, 0.24089375138282776, 0.13934320211410522, 1.9333269596099854, -0.9843190908432007, 0.07930848002433777, 2.2404210567474365, -0.5933344960212708, -0.30719029903411865, -0.4905841648578644, 0.1125042513012886, 1.4433932304382324, 2.471040964126587, 0.09089720994234085, 0.06002498418092728, 0.9043827652931213, 0.5705137252807617, 0.3726382553577423, -0.3885160982608795, -0.11640053242444992, 0.39696401357650757, -1.1776241064071655, 3.1097798347473145, -0.7421669960021973, -1.0330287218093872, -0.9570591449737549, 0.6887410879135132, -2.2174150943756104, -1.7157045602798462, -0.07103655487298965, -0.971432626247406, 1.4573469161987305, -0.18613116443157196, 0.5624614357948303, -1.9331659078598022, 4.800353050231934, 1.4588203430175781, 0.7086932063102722, 1.844054102897644, -1.0559622049331665, -1.019747257232666, 1.3632774353027344, 0.9425656795501709, -0.09831055998802185, -3.255002975463867, -1.0269612073898315, 1.7496157884597778, 0.1084408238530159, 0.49144092202186584, 1.4315342903137207, -2.4722096920013428, 1.5807061195373535, -1.6749893426895142, -1.1005430221557617, 0.6908060908317566, 0.14275579154491425, -1.2343096733093262, -0.3742605745792389, -0.03777211531996727, 1.5111697912216187, 2.2807488441467285, 0.7268667817115784, -0.17528563737869263, 1.4766292572021484, -1.6989428997039795, -0.04116363078355789, -0.32956674695014954, -0.8915331363677979, 0.03235851600766182, -0.0990624949336052, 1.551216959953308, -1.1310360431671143, -0.5623898506164551, -1.0833975076675415, -0.03242743760347366, 0.36494195461273193, -2.574958086013794, 2.0878303050994873, 1.3261349201202393, 0.7768649458885193, 0.8718525767326355, 1.2999025583267212, -0.16694045066833496, -0.7525569796562195, 1.5249882936477661, 0.7556604146957397, -0.8721263408660889, 0.7011294960975647, 0.7471089363098145, -1.601853847503662, -0.8077628016471863, 3.758209705352783, -0.3136011064052582, 3.940488338470459], [0.8048102855682373, -0.30202412605285645, -0.8941618800163269, 0.6153978109359741, 0.5509347319602966, -1.4756500720977783, 0.9091178774833679, -1.623970627784729, -2.26165771484375, -0.21899859607219696, 0.28165730834007263, -0.49320605397224426, -0.7140662670135498, -0.6359428763389587, 1.1831761598587036, -1.8085577487945557, 1.1158442497253418, -2.4025990962982178, 0.020226644352078438, -0.10394194722175598, 0.4509339928627014, -0.9424847364425659, 0.512808620929718, 1.7028937339782715, -0.6636972427368164, -0.7815704345703125, -3.1838364601135254, 0.18841832876205444, 1.2886852025985718, -1.233672022819519, 0.8180099725723267, -2.4348056316375732, 0.6247289776802063, -0.5737472176551819, 2.44718337059021, -0.2986046373844147, -0.031387437134981155, -0.6072893738746643, -1.053829312324524, 2.4894227981567383, 1.2200939655303955, 0.08129093050956726, -0.006928050424903631, -1.0087329149246216, 1.4440287351608276, -2.7278552055358887, -1.3349761962890625, -0.4678594768047333, 0.007900459691882133, 0.5589248538017273, -1.1372926235198975, 2.2526257038116455, 0.3187677264213562, -1.8477710485458374, -1.0485541820526123, 0.22028480470180511, -1.5771337747573853, -1.148216724395752, 0.15249024331569672, 0.3801386058330536, 0.8373180627822876, 1.7816358804702759, -1.789936900138855, -1.9098421335220337, -0.10839031636714935, -0.9614102840423584, 1.3498473167419434, 0.9875205159187317, 0.9587346911430359, 3.798609495162964, 0.6133891344070435, -1.9667046070098877, 0.7010250091552734, -2.8673324584960938, -0.6143433451652527, 2.0019125938415527, 4.265128135681152, -0.139479398727417, -2.7501626014709473, -0.45119428634643555, -1.0410047769546509, -0.537310779094696, -0.9608451724052429, 0.9483906626701355, 1.1264656782150269, -2.3098716735839844, -0.2660776674747467, -1.275774359703064, -0.6509763598442078, 0.9577776789665222, 1.5952340364456177, -0.09444598108530045, 0.507706880569458, -0.7257230877876282, 2.1476080417633057, -1.5971884727478027, 0.1296834498643875, -0.12847964465618134, 0.9006450176239014, 1.2247159481048584, -0.72933030128479, 0.8527818322181702, 1.153180718421936, -2.9288113117218018, -0.26631516218185425, -0.7774708867073059, 0.4576381742954254, 1.2016384601593018, 1.0057425498962402, -1.678709626197815, 1.450588583946228, -0.585357129573822, 0.6425033807754517, -1.350378155708313, 2.2868428230285645, -1.9775360822677612, -1.227615237236023, -0.33868810534477234, -0.6966618895530701, -0.029651541262865067, 1.6762434244155884, -0.375099778175354, 0.49133285880088806, -0.17085561156272888, 0.5473602414131165, -1.5552563667297363, 2.585204601287842, 0.9610992074012756, 0.8212628364562988, 0.6050649285316467, 1.4109095335006714, -3.0173351764678955, -1.8412972688674927, 2.1214451789855957, -0.26894718408584595, 2.145360231399536, -0.8340725302696228, 2.2696850299835205, 1.529086947441101, -0.0740005224943161, 0.0326981358230114, 0.06483952701091766, 3.4733097553253174, -0.8268194794654846, 2.605781078338623, 0.1733826845884323, -0.7741686701774597, 1.6036484241485596, -1.558342695236206, 1.630707025527954, -2.3176891803741455, -2.4049813747406006, -0.5437026619911194, 1.279964566230774, -0.4949270486831665, 0.9849556684494019, 0.3154379427433014, 1.2567883729934692, 1.588426113128662, 2.3585867881774902, 0.14388585090637207, -0.3476756513118744, 0.07936102896928787, 0.9055883288383484, -1.4225733280181885, -1.2085950374603271, -0.7318248152732849, -2.019202470779419, 2.4656054973602295, 1.435179352760315, 0.605823278427124, -2.282318592071533, 0.7972173690795898, 0.5073075890541077, 2.15909481048584, -1.7921333312988281, -0.6758288145065308, -0.20265977084636688, -0.8233211040496826, 1.7040507793426514, -0.47372376918792725, 2.3137173652648926, -0.12246360629796982, 1.8212924003601074, 1.0897899866104126, -0.1340905874967575, -0.10192333161830902, 2.0034985542297363, 1.3465129137039185, -1.001807689666748, 0.38764065504074097, -1.079545259475708, -3.5451228618621826, -0.030759448185563087, -3.812053680419922, -0.7828493118286133, -0.07926925271749496, -1.8765312433242798, -5.980043888092041, -0.6896222233772278, -1.0631543397903442, -0.11339874565601349, -2.3577826023101807, -0.32478228211402893, 0.5305963754653931, 0.020844927057623863, -1.0412216186523438, -0.5967061519622803, 0.9600197672843933, 1.2263953685760498, 3.0158803462982178, 0.5038381218910217, 3.9372644424438477, -1.3463579416275024, -2.0958240032196045, 1.0828800201416016, -0.25419914722442627, -1.470588207244873, -0.8681490421295166, 0.6972755193710327, -0.35581502318382263, 1.432356834411621, -3.462364673614502, 0.43833842873573303, 4.630005836486816, 2.2245752811431885, -1.8687324523925781, -3.566005229949951, -0.829933226108551, 1.1644598245620728, -0.27863699197769165, 0.4915257394313812, -0.4549623429775238, -1.5529452562332153, 0.39231404662132263, -0.6604901552200317, -2.609288215637207, 0.7312160730361938, 0.7037805914878845, 1.1177709102630615, -0.09849286079406738, 0.34561416506767273, 0.45548999309539795, 1.0292631387710571, -0.7019231915473938, 2.641458511352539, -1.1550661325454712, 1.1258342266082764, -1.187814474105835, 0.38560712337493896, -0.4755343794822693, 0.7910472750663757, 1.3754116296768188, -0.2734862267971039, 1.5529733896255493, -0.9918636679649353, -0.3510908782482147, -0.9850854277610779, -0.5610709190368652, -0.4638516902923584, -0.9545636773109436, 2.7679762840270996, -0.4120027720928192, 1.2671377658843994, 2.519561290740967, 2.33392071723938, -1.959704041481018, -0.9247922897338867, -0.9754636287689209, 0.1396932452917099, -0.42042896151542664, -0.017789693549275398, 0.8489437103271484, -0.6558908224105835, -0.9766123294830322, -1.406806468963623, 0.19025225937366486, 0.6050430536270142, -0.6446667313575745, 2.189932346343994, -1.8256725072860718, -4.250521659851074, -2.425131320953369, -0.7208197712898254, 0.0936465784907341, 0.16059070825576782, 0.8994788527488708, -1.6817853450775146, 0.252370685338974, 1.4693410396575928, -1.7735984325408936, 2.0074260234832764, 1.2612533569335938, 1.0951058864593506, -0.7822441458702087, 0.6698383092880249, -2.560805082321167, 0.7951550483703613, 1.5400563478469849, 0.05248064175248146, 0.9846590757369995, 2.6444013118743896, -0.7321797609329224, 0.9686143398284912, -0.8805072903633118, -1.1508909463882446, 0.11534402519464493, 0.11354623734951019, -0.2189856320619583, -1.506974458694458, -1.285473346710205, -1.1747905015945435, 1.0329325199127197, 1.0960139036178589, -2.0400867462158203, -0.9165326356887817, -0.06438492238521576, 0.3178899586200714, 0.6180669069290161, -0.40675461292266846, 2.641680955886841, 0.5858269929885864, -0.31368136405944824, 0.6642269492149353, 0.7029090523719788, -0.28301164507865906, 1.5485807657241821, -2.1186914443969727, -3.035334348678589, 0.21549300849437714, -0.6617908477783203, 0.4958273768424988, 1.8225736618041992, -0.7944855690002441, -0.2757103443145752, -2.511986494064331, 1.194848656654358, -0.8941766619682312, -0.029471034184098244, 1.5966962575912476, 0.16214467585086823, 0.3144225478172302, -2.851659059524536, 0.6348174214363098, -0.22604990005493164, 0.19365036487579346, 1.1580191850662231, -1.9374204874038696, 4.304955959320068, 3.163051128387451, -0.3259029686450958, -0.24251039326190948, -0.7778218388557434, -0.47689738869667053, -1.3368524312973022, 0.8772631883621216, -0.4015532433986664, -0.2146649807691574, 1.0506505966186523, 0.23313139379024506, 1.271929144859314, -1.3522738218307495, -1.8979830741882324, -0.16030287742614746, -2.2413573265075684, 0.24391017854213715, -0.23860223591327667, 1.6675565242767334, 3.480977773666382, 1.551019549369812, 1.3532261848449707, -0.5462719798088074, -0.5363990664482117, -0.04907071590423584, 1.2052383422851562, -0.05392731726169586, -2.513653516769409, -1.6110187768936157, 0.6787711381912231, -1.5430423021316528, 1.1405664682388306, 3.1490025520324707, -0.40573549270629883, -0.4040068984031677, 0.6384627223014832, 0.892079770565033, 2.277090311050415, 0.3668384253978729, 0.32764124870300293, -0.9775810837745667, 1.1311352252960205, -0.8522945046424866, -0.48788753151893616, -0.3957253694534302, -1.163910150527954, -0.1568821668624878, -2.4115145206451416, 3.163583993911743, 0.455314964056015, -0.844300389289856, 2.602142572402954, 1.2137027978897095, 0.8084645867347717, 0.5770402550697327, 0.6503424644470215, 0.5265601873397827, -0.5902495980262756, -0.09772618860006332, -0.9733710289001465, 1.7477706670761108, 1.6259877681732178, -0.9355283379554749, -1.280285120010376, 1.2015825510025024, 0.5654281377792358, -1.2793056964874268, 1.1942023038864136, 0.35530680418014526, -0.44317954778671265, 0.5207610130310059, -1.0167547464370728, -1.5084201097488403, 1.10760498046875, 0.7210727334022522, -0.7898508310317993, -1.6812355518341064, -0.8174412250518799, -0.4614250659942627, -1.3867013454437256, 0.7023681402206421, -0.31878331303596497, -0.1906256377696991, 0.4737056791782379, -0.4189130365848541, 1.6844440698623657, -0.9024965763092041, 0.7017239928245544, -1.0355664491653442, 2.0275790691375732, -0.7996765375137329, 0.40426138043403625, -0.8005058765411377, -2.1922953128814697, -1.7371832132339478, 0.44827768206596375, 0.011981121264398098, 1.8177399635314941, -1.797631859779358, 0.0789400190114975, 1.580193281173706, 1.3866846561431885, -1.8819818496704102, -1.0834949016571045, -1.1835507154464722, -0.835402250289917, 0.8758851885795593, -0.729047417640686, 1.7512176036834717, -0.6486183404922485, -1.0277771949768066, 1.613651156425476, 0.4579069912433624, 0.4559151232242584, 1.4468121528625488, -2.784543752670288, -1.2687379121780396, 0.5398443341255188, -1.0706524848937988, 0.22093793749809265, 0.6765420436859131, 0.8717501163482666, -0.4744735658168793, -1.1924699544906616, 0.03654485568404198, -0.7138596773147583, -0.298546701669693, 2.088487386703491, -0.7856353521347046, -0.6619266271591187, -1.0790421962738037, 0.21747416257858276, -1.3562147617340088, 0.31445080041885376, -0.6307221055030823, -1.419510841369629, -0.42382848262786865, -1.1861811876296997, 2.273538589477539, -0.4452342689037323, 0.03580562397837639, -0.27281826734542847, 0.7503300905227661, 2.4867165088653564, -1.9759916067123413, 1.7189891338348389, 1.2556002140045166, -1.7419252395629883, 1.7436920404434204, -0.06460167467594147, 1.159505844116211, -1.0119017362594604, 0.7326524257659912, 1.9390039443969727, -0.06645360589027405, 0.16510018706321716, 0.970832347869873, 0.8233940601348877, -1.6386244297027588, -1.127004861831665, -3.734400510787964, -0.44038841128349304, -0.007905177772045135, 0.2055150419473648, -0.11052042990922928, -0.1952807456254959, -0.4538647532463074, -1.9851034879684448, -0.2910964787006378, -0.5338109731674194, -2.1032028198242188, -1.4164904356002808, -1.7979339361190796, -3.2668635845184326, 2.6151413917541504, 3.202261447906494, -0.5171397924423218, -0.627633810043335, 0.1768842488527298, -5.391665458679199, -1.484868049621582, 1.4513124227523804, -1.5826976299285889, 3.0976486206054688, -0.4068664312362671, 1.3565682172775269, -1.24916672706604, 0.6152589321136475, -1.1275116205215454, 0.9138565063476562, -0.18787847459316254, -0.21368812024593353, 1.5972793102264404, 1.7580091953277588, 0.13310715556144714, 1.6290428638458252, -0.4585820436477661, 0.9809268116950989, -1.2706996202468872, -1.1321721076965332, -1.0545669794082642, -2.5370285511016846, 0.6107138991355896, -1.294783115386963, -0.6033211946487427, -0.5054024457931519, -0.9181032180786133, -0.568198561668396, -0.40489843487739563, -3.1405396461486816, -0.2838517129421234, 5.599209785461426, 0.652035117149353, 0.47810205817222595, -0.16542179882526398, 2.3254125118255615, -0.5988970994949341, -1.5806453227996826, -0.8253728151321411, 0.48951131105422974, 0.563339352607727, -0.4273463487625122, 0.6729333400726318, 0.6610967516899109, 2.21303129196167, 1.5612001419067383, 0.7627230286598206, -1.3319355249404907, -0.04342581331729889, 1.5362848043441772, -0.5834561586380005, -0.4079820513725281, -1.293495774269104, -0.27923473715782166, -0.5321840047836304, 1.0673168897628784, -0.3946838974952698, -0.16575247049331665, -2.0456247329711914, -0.6623225808143616, -0.7964093089103699, 0.4751855731010437, 0.38969624042510986, 1.4000731706619263, 0.06526575237512589, 0.9000849723815918, -0.42333945631980896, 1.7314369678497314, 0.5000336170196533, -0.3263271152973175, 1.11664879322052, 1.275923252105713, -2.021345853805542, 0.806073784828186, -2.225757360458374, -0.411992609500885, 1.593440294265747, 0.655949592590332, -2.6512060165405273, -1.21134614944458, -0.1719357818365097, 0.15337678790092468, 0.7521527409553528, -2.762930154800415, -2.8717706203460693, 3.2948572635650635, 0.14604035019874573, -0.17310668528079987, 1.1289894580841064, -1.1651010513305664, -1.9022653102874756, -1.0862709283828735, -1.1513534784317017, -1.117836356163025, 1.1701205968856812, 0.029231557622551918, -0.49751511216163635, 1.0139894485473633, -2.1251251697540283, 1.7671844959259033, -0.20980794727802277, 0.8251074552536011, 0.11883063614368439, -0.11472807079553604, 1.5349823236465454, -1.1691228151321411, 0.9477620124816895, -0.2179928719997406, 0.37711599469184875, -0.3465304374694824, -0.2017776221036911, -0.41492000222206116, -0.9579364657402039, 0.5711027383804321, 1.636339545249939, -0.6757599711418152, -1.0004076957702637, -0.0352935716509819, -0.9439414143562317, -1.3750429153442383, -0.3484169542789459, 0.5630720257759094, 1.7549930810928345, 1.4874805212020874, -2.2038512229919434, 0.5175464153289795, 1.8083845376968384, 0.2807709276676178, 0.7122885584831238, -0.38826993107795715, -0.3892417252063751, 0.9321339130401611, -0.2547828257083893, -0.10219018161296844, 0.6305899620056152, -0.05707445368170738, 0.22716595232486725, -2.722306966781616, -0.3768264055252075, -1.0390961170196533, -1.5314900875091553, 0.30645331740379333, 0.139540895819664, -0.41421905159950256, 0.42350488901138306, 0.8327686786651611, 0.5602946281433105, 0.080667644739151, 2.263955593109131, -1.5176736116409302, 0.5452558398246765, -1.233318567276001, -0.6377174258232117, -0.4272719621658325, -0.2112138718366623, 2.2044150829315186, 0.8665005564689636, 1.9961564540863037, -0.28205811977386475, 0.2875562310218811, -2.6266462802886963, 1.034608006477356, 0.26978403329849243, -0.12247385084629059, 0.29678696393966675, -0.5105178952217102, -0.5259696245193481, 2.8996171951293945, 0.27711033821105957, -0.23560944199562073, -1.1695919036865234, -1.228286862373352, -2.21828556060791, -2.4394452571868896, -0.13749799132347107, -1.0151479244232178, -2.708900213241577, -1.4566683769226074, 0.5381944179534912, -1.704463005065918, 1.6901112794876099, 1.5329205989837646, -0.8729448318481445, 1.083957314491272, -2.8493406772613525, -0.6295608282089233, 1.717306137084961, 1.3099961280822754, 1.873189091682434, 1.414035439491272, -0.5835944414138794, 1.0742520093917847, -0.005407310556620359, 1.7278831005096436, -0.019836103543639183, -3.722618341445923, -0.3969165086746216, -0.3719438910484314, 1.2363927364349365, 0.25125250220298767, 0.620430052280426, -0.3791583478450775, -0.5208614468574524, 0.10557527095079422, 1.2789835929870605, -3.0469746589660645, 0.9000774621963501, 0.9236865043640137, 0.8235087990760803, 0.5993660092353821, -0.1699993908405304, -0.9071533679962158, -0.3164461553096771, -0.015386739745736122, 1.1180555820465088, -1.997599482536316, -0.09169530123472214, -0.2764410078525543, -0.6253537535667419, -0.0716814324259758, 0.2826308310031891, -2.351562738418579, 2.4924402236938477, 0.2447267770767212, 0.11523846536874771, 0.8533195853233337, 1.611493706703186, -0.06543880701065063, -1.1777052879333496, 2.0674943923950195, 0.1438549906015396, -0.2599024772644043, -0.5464910268783569, 0.09188709408044815, -2.778244733810425, -0.8352686762809753, 2.4947638511657715, 0.18577726185321808, 2.2466540336608887]], "dtype": "float32"}, {"__nd__": [-0.03456798195838928, 0.01664271391928196, 0.016056926921010017, -0.07906965166330338], "dtype": "float32"}, {"__nd__": [1.0364863872528076, 1.0223422050476074, 0.9938826560974121, 0.9724593162536621], "dtype": "float32"}]}}, {"a": [{"__nd__": [[104.48136901855469, 91.39807891845703, 101.28623962402344, 101.17301940917969, 99.1356430053711, 96.90272521972656, 101.48721313476562, 108.73787689208984, 103.48473358154297, 100.00696563720703, 103.02987670898438, 97.78213500976562, 96.55097961425781, 97.94253540039062, 100.84375762939453, 108.32707214355469, 104.54288482666016, 101.6120376586914, 98.44883728027344, 107.59896087646484, 93.67352294921875, 99.15731811523438, 108.7193603515625, 107.03274536132812, 103.02630615234375, 103.78710174560547, 96.56837463378906, 104.49083709716797, 102.13951873779297, 107.36325073242188, 97.00758361816406, 104.68157958984375, 99.71369171142578, 105.38230895996094, 96.27376556396484, 95.58049774169922, 97.45459747314453, 100.05921173095703, 104.3024673461914, 89.77689361572266, 102.90122985839844, 96.435546875, 103.46772766113281, 88.96263122558594, 101.2272720336914, 95.66893005371094, 105.13597869873047, 95.95410919189453, 92.00432586669922, 94.66085052490234, 89.7648696899414, 105.3917465209961, 101.35770416259766, 102.9993667602539, 104.63369750976562, 93.15756225585938, 94.37335205078125, 101.50640106201172, 93.1355209350586, 100.87501525878906, 100.4454116821289, 101.09429931640625, 93.36256408691406, 103.24413299560547, 103.16542053222656, 92.15460205078125, 99.73734283447266, 103.73882293701172, 94.81375885009766, 102.57134246826172, 94.43280792236328, 103.03408813476562, 101.03644561767578, 105.78610229492188, 86.65870666503906, 89.73018646240234, 103.77693939208984, 103.07373046875, 108.0724105834961, 99.88524627685547, 103.39706420898438, 101.180908203125, 99.38276672363281, 95.10297393798828, 93.8043212890625, 92.74195861816406, 104.01583862304688, 91.39332580566406, 101.86299896240234, 96.31359100341797, 98.97174072265625, 108.68644714355469, 90.69257354736328, 96.38508605957031, 109.12662506103516, 103.30574798583984, 96.4837875366211, 107.62722778320312, 92.13716125488281, 107.35356140136719, 112.12312316894531, 93.99634552001953, 97.73236083984375, 102.42121124267578, 102.94868469238281, 101.24207305908203, 99.22590637207031, 96.94854736328125, 107.96051025390625, 98.04241180419922, 102.30722045898438, 104.48198699951172, 103.82563018798828, 90.27691650390625, 101.7204818725586, 98.00110626220703, 99.04724884033203, 99.23527526855469, 102.60649871826172, 102.44986724853516, 97.21893310546875, 103.35435485839844, 95.66032409667969, 101.23218536376953, 89.21531677246094, 96.37202453613281, 110.11771392822266, 101.33277130126953, 96.46527099609375, 105.26972961425781, 98.79100036621094, 96.74113464355469, 95.51811981201172, 105.06053924560547, 99.83279418945312, 100.26378631591797, 96.13227081298828, 101.1802978515625, 104.50531768798828, 99.3277816772461, 96.17236328125, 97.38402557373047, 92.55233764648438, 103.7927017211914, 98.98463439941406, 96.62602233886719, 105.03326416015625, 95.68684387207031, 100.26156616210938, 100.92878723144531, 98.20948028564453, 100.12852478027344, 100.9136962890625, 104.21017456054688, 100.63664245605469, 103.81361389160156, 104.63285064697266, 104.30387878417969, 96.47179412841797, 102.34979248046875, 96.85667419433594, 100.07280731201172, 92.73410034179688, 106.353515625, 99.99693298339844, 97.864013671875, 103.35568237304688, 107.5301742553711, 93.57244873046875, 106.51007843017578, 105.19401550292969, 101.06339263916016, 102.06045532226562, 95.58761596679688, 95.75464630126953, 94.52130889892578, 97.87060546875, 104.10511779785156, 100.2938232421875, 94.23434448242188, 95.8206787109375, 104.21725463867188, 94.1170654296875, 86.62367248535156, 110.16065979003906, 105.46437072753906, 95.1585464477539, 99.26998901367188, 95.2130126953125, 108.2750244140625, 104.65729522705078, 96.72459411621094, 100.65824890136719, 104.56787109375, 100.66806030273438, 107.05125427246094, 95.78536224365234, 94.55020141601562, 106.44146728515625, 102.84415435791016, 100.22531127929688, 103.91234588623047, 99.13816833496094, 97.51791381835938, 103.29023742675781, 89.46295166015625, 101.1351547241211, 100.92755126953125, 99.64804077148438, 100.94862365722656, 101.33667755126953, 101.26984405517578, 98.64285278320312, 96.69159698486328, 102.13506317138672, 102.99070739746094, 95.09199523925781, 109.95577239990234, 101.8994140625, 101.99593353271484, 102.36127471923828, 104.1161880493164, 91.72329711914062, 99.30904388427734, 100.66002655029297, 93.35435485839844, 93.7491226196289, 99.75379180908203, 99.13712310791016, 98.87189483642578, 106.32919311523438, 92.26567077636719, 97.7447280883789, 102.831787109375, 96.61695098876953, 105.52265930175781, 101.1619873046875, 109.66978454589844, 104.40424346923828, 97.89380645751953, 106.27975463867188, 100.14534759521484, 99.19174194335938, 102.58101654052734, 93.49226379394531, 95.02733612060547, 96.84235382080078, 102.87939453125, 98.94683837890625, 99.32754516601562, 100.04411315917969, 103.13603210449219, 96.26663970947266, 100.10816192626953, 95.42801666259766, 106.63592529296875, 107.96854400634766, 96.15425109863281, 95.46572875976562, 104.00949096679688, 96.22966766357422, 99.62381744384766, 98.39130401611328, 102.60762786865234, 96.43384552001953, 105.15889739990234, 101.7580795288086, 101.88053894042969, 105.93997955322266, 104.60421752929688, 97.32386779785156, 91.8226318359375, 92.1334228515625, 100.27815246582031, 108.80066680908203, 97.16895294189453, 100.09558868408203, 101.1106948852539, 98.3510513305664, 94.93075561523438, 103.90583038330078, 109.6513442993164, 96.25054168701172, 103.58961486816406, 106.70235443115234, 99.45726013183594, 92.7372055053711, 106.93338012695312, 99.49564361572266, 100.82086181640625, 100.92174530029297, 95.41890716552734, 106.59691619873047, 94.18280029296875, 106.06663513183594, 99.64737701416016, 94.46749114990234, 102.07472229003906, 87.74497985839844, 106.5008773803711, 99.8841552734375, 92.5718994140625, 102.19316101074219, 99.25360870361328, 104.36444091796875, 90.67208099365234, 99.10771179199219, 96.79385375976562, 94.55818176269531, 101.91011047363281, 97.65692138671875, 101.50997161865234, 98.58885192871094, 97.09088134765625, 97.00919342041016, 104.36553955078125, 114.54734802246094, 105.17723083496094, 102.66847229003906, 98.76798248291016, 103.43301391601562, 107.05062866210938, 100.00093078613281, 103.18067932128906, 96.89707946777344, 100.85154724121094, 97.40877532958984, 103.26116180419922, 96.99992370605469, 93.02764129638672, 106.4004898071289, 95.21392059326172, 103.62574005126953, 94.74237060546875, 101.2988510131836, 95.64424133300781, 111.57161712646484, 109.62879943847656, 107.96637725830078, 93.03472137451172, 93.70592498779297, 100.5741195678711, 92.17656707763672, 104.03304290771484, 90.86609649658203, 104.177978515625, 101.41519927978516, 95.83802795410156, 108.47599029541016, 110.83636474609375, 99.26763153076172, 105.58989715576172, 90.98929595947266, 93.35552978515625, 99.35601043701172, 92.82496643066406, 98.92416381835938, 98.90951538085938, 103.66991424560547, 109.51850128173828, 85.52352142333984, 95.18685913085938, 99.35839080810547, 109.09508514404297, 107.34585571289062, 97.72757720947266, 96.37062072753906, 99.65019989013672, 103.19734191894531, 95.097900390625, 105.65251159667969, 101.37108612060547, 93.92469787597656, 104.04119110107422, 105.11650085449219, 99.12884521484375, 105.12543487548828, 103.75823211669922, 104.38729858398438, 97.60323333740234, 95.66291046142578, 93.88194274902344, 95.75038146972656, 104.69123840332031, 94.25706481933594, 97.68568420410156, 104.43733215332031, 96.51122283935547, 94.22337341308594, 103.90629577636719, 101.7093505859375, 93.89735412597656, 89.77325439453125, 106.12802124023438, 105.53410339355469, 101.07952880859375, 90.17584991455078, 100.75297546386719, 96.8757553100586, 94.15464782714844, 112.65147399902344, 92.95060729980469, 105.15599822998047, 102.60456085205078, 95.06586456298828, 97.65435028076172, 94.67780303955078, 100.54739379882812, 96.86497497558594, 93.07787322998047, 101.04125213623047, 97.11898040771484, 98.36652374267578, 101.08208465576172, 104.08306121826172, 106.06028747558594, 100.7281265258789, 97.42463684082031, 104.14308166503906, 93.704345703125, 103.45772552490234, 102.77375030517578, 91.87222290039062, 95.33917999267578, 101.27690124511719, 93.8345947265625, 100.39995574951172, 96.31118774414062, 105.36532592773438, 94.62815856933594, 98.78831481933594, 102.29612731933594, 100.64334869384766, 107.34483337402344, 103.79955291748047, 92.72661590576172, 105.36865234375, 99.50440979003906, 110.88591003417969, 92.22062683105469, 103.16053009033203, 91.19002532958984, 97.57745361328125, 109.16897583007812, 96.03296661376953, 98.60187530517578, 98.75204467773438, 107.7605972290039, 95.43146514892578, 103.88692474365234, 98.58036041259766, 100.42039489746094, 102.37098693847656, 105.56304168701172, 100.48688507080078, 100.24443817138672, 98.94190979003906, 104.85832977294922, 92.04033660888672, 112.28547668457031, 108.28711700439453, 90.54885864257812, 97.56047821044922, 105.42374420166016, 90.47402954101562, 97.91719818115234, 100.63788604736328, 93.66465759277344, 102.1177749633789, 101.2042007446289, 101.0163803100586, 94.8139419555664, 94.18057250976562, 92.84232330322266, 98.2144546508789, 103.59910583496094, 105.00944519042969, 97.59662628173828, 96.4634017944336, 96.80586242675781, 90.04241943359375, 97.0109634399414, 94.9533920288086, 106.88912200927734, 97.89041137695312, 96.68904876708984, 95.72613525390625, 100.25236511230469, 105.16822814941406, 102.934814453125, 97.18901824951172, 109.78475952148438, 97.29173278808594, 106.42318725585938, 88.34270477294922, 104.05472564697266, 101.34766387939453, 102.83840942382812, 103.25897216796875, 97.9513931274414, 94.00183868408203, 95.762451171875, 104.57173156738281, 109.64590454101562, 107.05580139160156, 108.03244018554688, 98.6581039428711, 94.283447265625, 95.35375213623047, 108.99137878417969, 109.45925903320312, 98.13526916503906, 102.7118148803711, 102.05506896972656, 108.08746337890625, 99.34496307373047, 93.34979248046875, 93.23607635498047, 99.37291717529297, 99.96930694580078, 101.50727081298828, 101.8958969116211, 98.75234985351562, 104.08990478515625, 103.6040267944336, 95.61629486083984, 94.78516387939453, 107.48005676269531, 90.80912780761719, 101.97846221923828, 98.49304962158203, 93.12987518310547, 99.46691131591797, 97.4643325805664, 96.07128143310547, 96.68749237060547, 104.20820617675781, 101.5662841796875, 98.00528717041016, 95.134521484375, 95.02783966064453, 109.89002227783203, 91.95298767089844, 112.8968276977539, 106.79549407958984, 97.4072265625, 107.81025695800781, 89.55148315429688, 107.95726013183594, 104.74559020996094, 92.40142059326172, 97.44416809082031, 102.51403045654297, 91.83223724365234, 96.03984832763672, 104.2127456665039, 103.10721588134766, 100.9043197631836, 105.04121398925781, 101.85736083984375, 101.03852081298828, 97.74701690673828, 101.49801635742188, 106.50381469726562, 98.9437484741211, 101.35344696044922, 92.28364562988281, 97.700927734375, 109.27664184570312, 100.40613555908203, 107.07987213134766, 90.24734497070312, 96.29684448242188, 105.42481994628906, 106.98961639404297, 99.28266906738281, 92.75736236572266, 103.53153991699219, 102.9825668334961, 101.11448669433594, 97.28850555419922, 101.6492919921875, 104.38833618164062, 105.6742172241211, 93.9729232788086, 100.40752410888672, 99.24298095703125, 100.9841537475586, 98.99922943115234, 94.23482513427734, 103.92461395263672, 102.98978424072266, 96.7916488647461, 93.77560424804688, 102.94210815429688, 102.25535583496094, 99.73173522949219, 97.54174041748047, 97.0066909790039, 97.04859161376953, 103.26007080078125, 99.7553939819336, 103.59222412109375, 104.20762634277344, 99.3410873413086, 95.40756225585938, 96.00879669189453, 99.92383575439453, 105.86700439453125, 95.68022155761719, 97.27688598632812, 95.91867065429688, 104.32169342041016, 100.40943908691406, 95.94503784179688, 107.78177642822266, 99.56006622314453, 105.60420227050781, 100.10893249511719, 95.60104370117188, 107.57632446289062, 97.7548599243164, 95.4286117553711, 95.90594482421875, 105.65612030029297, 96.51028442382812, 96.0030517578125, 97.88581085205078, 91.29664611816406, 113.93627166748047, 106.46199035644531, 92.79790496826172, 110.32379913330078, 98.36808776855469, 91.70063781738281, 95.15367126464844, 93.45228576660156, 103.71588134765625, 100.13805389404297, 92.67980194091797, 100.59867095947266, 96.74628448486328, 96.9698715209961, 100.4261245727539, 101.50398254394531, 90.42216491699219, 106.53376007080078, 100.345947265625, 104.13707733154297, 104.49995422363281, 106.75338745117188, 102.78834533691406, 103.08352661132812, 100.3372573852539, 108.73765563964844, 102.24266052246094, 105.20419311523438, 99.5228271484375, 100.36422729492188, 89.65184020996094, 104.33086395263672, 94.31312561035156, 96.28272247314453, 107.1453857421875, 102.18785095214844, 102.31752014160156, 106.03636169433594, 95.06725311279297, 91.92560577392578, 100.01081848144531, 99.34400177001953, 100.92981719970703, 89.9251480102539, 103.54594421386719, 97.99164581298828, 94.54281616210938, 97.79149627685547, 101.07855987548828, 99.7857437133789, 97.11766052246094, 97.285888671875, 94.03060913085938, 103.68953704833984, 104.71659088134766, 102.011474609375, 89.96791076660156, 101.54821014404297, 93.70515441894531, 95.16776275634766, 104.26920318603516, 98.62100219726562, 100.97081756591797, 97.11041259765625, 100.12373352050781, 105.99171447753906, 95.68083190917969, 96.75166320800781, 105.0002670288086, 99.92082214355469, 97.53448486328125, 103.04550170898438, 97.52413177490234, 100.35100555419922, 101.05685424804688, 95.97916412353516, 101.71085357666016, 99.18769073486328, 102.16001892089844, 94.48783874511719, 97.60049438476562, 98.2446517944336, 100.1301040649414, 101.65398406982422, 101.15805053710938, 109.30912780761719, 100.11843872070312, 103.50001525878906, 97.66993713378906, 101.89395141601562, 105.01923370361328, 97.9875259399414, 96.3248519897461, 110.01377868652344, 98.29580688476562, 90.52604675292969, 116.75520324707031, 94.89212036132812, 101.3287353515625, 93.9478530883789, 111.31417846679688, 98.6667709350586, 106.87999725341797, 94.46971130371094, 112.43140411376953, 90.79044342041016, 98.99606323242188, 104.90806579589844, 99.14249420166016, 93.81185913085938, 109.14524841308594, 100.44374084472656, 95.125, 101.12405395507812, 95.42288208007812, 106.25051879882812, 103.69575500488281, 97.49684143066406, 107.02208709716797, 97.64473724365234, 98.26458740234375, 91.60205078125, 100.30726623535156, 91.2010498046875, 95.7883071899414, 96.13928985595703, 103.5683822631836, 96.49174499511719, 96.86078643798828, 93.4758071899414, 91.92247009277344, 95.52662658691406, 98.58914184570312, 102.59867095947266, 106.05767059326172, 97.25115203857422, 101.52755737304688, 99.68672943115234, 96.4440689086914, 106.35066223144531, 98.73307800292969, 98.76288604736328, 97.03131103515625, 99.23180389404297, 98.15837097167969, 109.41744995117188, 93.82144165039062, 93.04975128173828, 103.61991882324219, 99.6853256225586, 89.1806411743164, 106.95458984375, 92.82508850097656, 104.99629211425781, 100.90690612792969, 106.92992401123047, 99.9227523803711, 98.19930267333984, 97.72721099853516, 97.76806640625, 98.25082397460938, 97.39348602294922, 97.87084197998047, 91.31041717529297, 96.17227172851562, 98.85432434082031, 91.69084167480469, 97.13723754882812, 98.3414535522461, 105.2060317993164, 100.73607635498047, 95.84664154052734, 106.99955749511719, 100.66239929199219, 94.53495025634766, 103.95056915283203, 97.65950775146484, 95.43921661376953, 88.87805938720703, 100.32213592529297, 95.1607894897461, 98.67733764648438, 106.53588104248047, 100.7544174194336, 105.39920806884766, 110.21855163574219, 102.92276000976562, 93.12361145019531, 99.29995727539062, 103.97368621826172, 108.8648681640625, 103.0274658203125, 98.90935516357422, 109.10590362548828, 94.47645568847656, 95.35469818115234, 110.42872619628906, 97.6590576171875, 94.1015853881836, 97.54632568359375, 108.00080871582031, 96.96481323242188, 100.17350006103516, 109.21543884277344, 100.45977020263672, 97.415771484375, 108.93155670166016, 102.55073547363281, 102.53533172607422, 98.78997039794922, 99.15357971191406, 103.5955810546875, 94.8210220336914, 90.52718353271484, 95.27664184570312, 104.55220031738281, 104.05033111572266, 90.36454010009766, 106.3738021850586, 92.57292938232422, 93.87382507324219, 101.36936950683594, 97.615478515625, 101.43628692626953, 100.29195404052734, 92.06754302978516, 101.9138412475586, 105.2416000366211, 105.64274597167969, 98.59474182128906, 99.10636901855469, 101.06764221191406, 110.0047378540039, 103.35843658447266, 95.78169250488281, 101.98240661621094, 95.7706527709961, 101.53235626220703, 111.53800201416016, 107.3165512084961, 95.66522216796875, 94.80098724365234, 100.0024642944336, 116.92201232910156, 96.8361587524414, 105.55583190917969, 98.81464385986328, 97.63787078857422, 96.82821655273438, 111.60476684570312, 94.2632827758789, 92.3752670288086, 97.43868255615234, 97.11861419677734, 93.2230224609375, 101.03189086914062, 102.56013488769531, 93.08177185058594, 102.53681182861328, 96.45584869384766, 107.83019256591797, 101.40595245361328, 98.38249206542969, 104.67635345458984, 104.12947845458984, 101.48583221435547, 94.93343353271484, 98.79366302490234, 100.8913803100586, 95.10309600830078, 108.82796478271484, 100.52735900878906, 93.72945404052734, 102.23828887939453, 109.41154479980469, 107.89965057373047, 95.99609375, 95.47623443603516, 103.28610229492188, 96.45874786376953, 87.91222381591797, 98.13444519042969, 100.47421264648438, 105.35823059082031, 95.9169692993164, 95.81084442138672, 102.2704849243164, 104.6347885131836, 100.96573638916016, 94.23846435546875, 101.0939712524414, 105.04981994628906, 100.63670349121094, 96.70539855957031, 105.01551818847656, 104.16351318359375, 90.64976501464844, 98.49366760253906, 100.63372802734375, 104.32681274414062, 97.27561950683594, 100.53382873535156, 96.82917022705078, 105.13169860839844, 96.8264389038086, 100.17328643798828, 101.33671569824219, 91.61531829833984, 113.7581787109375, 100.71756744384766, 101.721435546875, 97.87678527832031, 101.29564666748047, 103.14204406738281, 100.37801361083984, 100.89461517333984, 92.68838500976562, 103.36933135986328, 99.67819213867188, 98.04940032958984, 99.47897338867188, 94.57945251464844, 105.76715850830078, 95.00485229492188, 106.002685546875, 97.31220245361328, 99.8746109008789, 97.18138885498047, 109.18852233886719, 95.80593872070312, 97.29167938232422, 103.40928649902344, 97.49729919433594, 97.81411743164062, 101.80626678466797, 94.44235229492188, 104.93152618408203, 106.58465576171875, 96.37773132324219, 103.51126861572266, 89.67220306396484, 103.15875244140625, 98.07080078125, 104.91960906982422, 107.73404693603516, 95.82162475585938, 103.16146087646484, 109.91105651855469, 103.57672119140625, 103.04569244384766, 100.5045394897461, 92.32854461669922, 96.23308563232422, 90.36048889160156, 99.86942291259766, 96.19673919677734, 95.46790313720703, 95.69831848144531, 108.48986053466797, 93.68201446533203, 100.9187240600586, 106.412109375, 111.41663360595703, 95.85862731933594, 98.7420883178711, 103.15638732910156, 99.17669677734375, 91.58197021484375, 101.16537475585938, 99.07489776611328, 104.9000473022461], [105.40367126464844, 95.74678802490234, 102.73885345458984, 110.0527572631836, 102.0424575805664, 95.77116394042969, 107.81108856201172, 99.86859130859375, 100.3730239868164, 97.68033599853516, 91.63874816894531, 88.13423156738281, 93.26471710205078, 84.24851989746094, 96.89852142333984, 102.46639251708984, 102.87137603759766, 105.75392150878906, 104.7435073852539, 103.9986343383789, 105.7242202758789, 97.24530029296875, 100.81626892089844, 95.67193603515625, 91.3668212890625, 100.83551788330078, 104.24214172363281, 106.44103240966797, 86.72515106201172, 105.83558654785156, 98.62867736816406, 100.99783325195312, 97.07723999023438, 100.45919036865234, 97.43135070800781, 101.91289520263672, 107.90962982177734, 93.02711486816406, 97.6385726928711, 98.66775512695312, 89.38008117675781, 94.19013214111328, 101.52938842773438, 91.28409576416016, 91.96021270751953, 94.1783218383789, 106.97004699707031, 101.53829193115234, 93.23472595214844, 104.80683898925781, 95.74256896972656, 92.53811645507812, 101.62232208251953, 104.37326049804688, 99.98196411132812, 104.27556610107422, 106.91835021972656, 105.77685546875, 101.94561767578125, 102.66189575195312, 100.44244384765625, 104.25328826904297, 114.52149963378906, 94.3836441040039, 103.50342559814453, 100.23114013671875, 92.75759887695312, 102.28120422363281, 95.8622817993164, 84.98705291748047, 106.83625793457031, 101.65579986572266, 102.86488342285156, 111.51612091064453, 95.87857055664062, 103.3636245727539, 97.36019897460938, 100.69482421875, 98.99823760986328, 99.73235321044922, 96.30438232421875, 94.72823333740234, 90.3713150024414, 98.86074829101562, 95.11843872070312, 97.8624038696289, 95.88260650634766, 97.51256561279297, 101.4112319946289, 98.83460998535156, 92.37879943847656, 102.90374755859375, 102.67857360839844, 101.78298950195312, 97.26012420654297, 95.02328491210938, 102.08220672607422, 98.0409164428711, 102.45921325683594, 109.18787384033203, 101.46189880371094, 94.9180679321289, 93.01048278808594, 97.07537841796875, 92.85266876220703, 96.27339172363281, 96.62367248535156, 101.3656005859375, 104.88844299316406, 99.7918701171875, 97.75428009033203, 105.1025390625, 98.05519104003906, 100.92388153076172, 97.87753295898438, 104.39433288574219, 105.47101593017578, 97.60554504394531, 103.92536163330078, 101.49063110351562, 96.77482604980469, 103.52842712402344, 100.18437957763672, 98.546875, 101.30162811279297, 101.47183990478516, 99.81216430664062, 104.58161163330078, 102.27862548828125, 93.36880493164062, 96.53724670410156, 97.18865966796875, 98.62933349609375, 99.71837615966797, 99.54151153564453, 108.37740325927734, 98.19808959960938, 99.769287109375, 98.42469024658203, 96.31277465820312, 107.34640502929688, 102.37937927246094, 103.15898132324219, 99.15001678466797, 96.42581939697266, 104.23633575439453, 96.35233306884766, 103.37979888916016, 103.98394012451172, 101.90863037109375, 102.94586181640625, 97.85501861572266, 96.75460052490234, 105.55773162841797, 107.78160858154297, 95.78897857666016, 91.39286041259766, 102.38948059082031, 102.5321273803711, 98.1418685913086, 105.54466247558594, 96.1692886352539, 95.47129821777344, 94.55865478515625, 99.87061309814453, 100.96083068847656, 105.77515411376953, 99.1620101928711, 99.01631164550781, 108.13764953613281, 103.87382507324219, 101.42225646972656, 100.39624786376953, 97.74581909179688, 98.0106430053711, 103.06932830810547, 108.42650604248047, 101.01128387451172, 98.05531311035156, 104.77152252197266, 100.95282745361328, 97.41886901855469, 98.23961639404297, 97.269775390625, 104.27210998535156, 100.81646728515625, 106.0121078491211, 104.00387573242188, 98.90023803710938, 99.65750122070312, 96.18635559082031, 100.54216003417969, 95.5188980102539, 97.19329071044922, 96.5790786743164, 97.25117492675781, 98.28274536132812, 108.673095703125, 95.1642837524414, 94.37205505371094, 91.87026977539062, 88.230712890625, 101.61051177978516, 103.46683502197266, 94.34187316894531, 100.50132751464844, 101.3644027709961, 105.30661010742188, 102.52047729492188, 97.77127838134766, 104.3155517578125, 89.6104736328125, 104.26576232910156, 106.18455505371094, 109.957763671875, 96.66974639892578, 95.46747589111328, 105.04534912109375, 103.82398223876953, 101.84033203125, 106.91423797607422, 100.51219940185547, 101.03313446044922, 105.01058959960938, 109.19155883789062, 98.09130859375, 104.51805877685547, 93.64109802246094, 100.20195770263672, 97.13194274902344, 100.7563705444336, 94.64266204833984, 101.44969940185547, 102.77803802490234, 99.83750915527344, 102.13642883300781, 93.26426696777344, 112.38580322265625, 101.20724487304688, 100.11853790283203, 105.32981872558594, 92.68285369873047, 104.37122344970703, 98.75553131103516, 107.17303466796875, 99.30016326904297, 93.7304916381836, 96.24803161621094, 111.54100799560547, 103.0166015625, 96.88446044921875, 89.95584869384766, 105.66878509521484, 93.41847229003906, 99.76312255859375, 95.0925521850586, 91.24825286865234, 102.81320190429688, 103.1609878540039, 97.38514709472656, 100.00687408447266, 110.89479064941406, 96.84307098388672, 102.22382354736328, 112.29359436035156, 97.93218231201172, 99.9539794921875, 99.80603790283203, 94.92010498046875, 105.21914672851562, 105.54679870605469, 99.88597106933594, 100.73961639404297, 88.6719741821289, 106.02064514160156, 105.1741714477539, 107.49380493164062, 99.92784118652344, 101.68307495117188, 96.59039306640625, 96.10769653320312, 105.58975219726562, 100.37798309326172, 106.3698501586914, 102.1330795288086, 96.8670425415039, 95.35513305664062, 100.31874084472656, 99.2360610961914, 97.73609924316406, 109.55945587158203, 106.7037582397461, 103.25990295410156, 103.16433715820312, 102.22225189208984, 110.25826263427734, 96.84465789794922, 95.22415924072266, 97.37631225585938, 98.70140838623047, 93.37906646728516, 104.63236999511719, 105.31397247314453, 92.36043548583984, 89.27777862548828, 98.60186004638672, 98.2542953491211, 99.23799133300781, 101.85340881347656, 97.75027465820312, 95.2841796875, 100.28209686279297, 94.06858825683594, 96.85953521728516, 103.82952117919922, 94.46662139892578, 105.08678436279297, 100.68944549560547, 103.23570251464844, 98.87249755859375, 105.03681945800781, 99.67243957519531, 99.36376190185547, 104.26729583740234, 101.66519927978516, 106.39054107666016, 99.72791290283203, 97.6015396118164, 93.8022232055664, 101.81446838378906, 111.23960876464844, 102.92212677001953, 105.11042022705078, 99.08430480957031, 102.59049987792969, 95.86653900146484, 90.20146179199219, 101.4852523803711, 97.62487030029297, 99.06143951416016, 95.94856262207031, 102.5615005493164, 98.77416229248047, 104.68927764892578, 108.77387237548828, 93.16100311279297, 105.27670288085938, 91.70450592041016, 111.22799682617188, 104.4023666381836, 105.8056640625, 95.79286193847656, 93.32884979248047, 91.3158187866211, 99.47496032714844, 102.088623046875, 97.15989685058594, 98.17549896240234, 93.40374755859375, 104.42943572998047, 104.4131088256836, 100.01141357421875, 99.6176528930664, 94.47696685791016, 105.29183197021484, 101.67842102050781, 99.67328643798828, 101.44371795654297, 102.27371978759766, 96.89511108398438, 101.37511444091797, 97.22574615478516, 103.48394012451172, 100.69129180908203, 99.57919311523438, 104.4999771118164, 94.71292877197266, 101.31742095947266, 85.82762145996094, 103.00005340576172, 103.49430847167969, 99.766357421875, 93.759521484375, 94.91352844238281, 103.42266082763672, 105.51217651367188, 99.85466766357422, 99.128173828125, 100.02703857421875, 98.21209716796875, 99.51040649414062, 108.00335693359375, 96.9643783569336, 101.8890380859375, 100.854736328125, 96.79592895507812, 99.70526123046875, 88.66017150878906, 102.67182159423828, 102.09965515136719, 95.49571228027344, 97.26348114013672, 109.29900360107422, 96.59843444824219, 103.4711685180664, 103.53536987304688, 103.42716217041016, 105.82672119140625, 102.70624542236328, 101.6011734008789, 103.6066665649414, 99.96125030517578, 93.38147735595703, 94.36971282958984, 95.20945739746094, 110.59471130371094, 104.52947998046875, 96.60638427734375, 96.17366027832031, 103.31786346435547, 93.02006530761719, 95.54248809814453, 105.10558319091797, 103.34529876708984, 100.65556335449219, 95.68827056884766, 98.20869445800781, 94.10151672363281, 104.5678939819336, 88.22147369384766, 97.54508972167969, 99.80327606201172, 95.93250274658203, 101.34688568115234, 109.21769714355469, 97.22834014892578, 100.87372589111328, 100.0326156616211, 96.27285766601562, 96.09168243408203, 106.05708312988281, 103.92069244384766, 101.04283905029297, 105.19017028808594, 90.36669921875, 97.6937026977539, 99.82283782958984, 101.125244140625, 95.8605728149414, 96.54413604736328, 99.47919464111328, 95.9674301147461, 110.54795837402344, 95.11319732666016, 99.4183120727539, 105.8005599975586, 99.696533203125, 96.3321762084961, 104.3633804321289, 104.82949829101562, 104.78588104248047, 101.02342224121094, 87.56591796875, 98.59857177734375, 96.10003662109375, 101.92044067382812, 103.99989318847656, 92.78953552246094, 105.8624267578125, 102.44914245605469, 97.80223083496094, 89.36676025390625, 109.06983947753906, 103.1890869140625, 88.92935180664062, 103.02361297607422, 97.70602416992188, 102.3790512084961, 99.88538360595703, 105.02471923828125, 93.0525131225586, 100.69867706298828, 109.65785217285156, 96.13148498535156, 97.56172943115234, 104.73597717285156, 94.60103607177734, 103.0645751953125, 105.61674499511719, 98.66265869140625, 106.53711700439453, 88.19640350341797, 92.01408386230469, 102.05008697509766, 94.87126159667969, 98.97440338134766, 105.22798919677734, 96.18594360351562, 96.40680694580078, 93.01425170898438, 93.2964859008789, 93.10303497314453, 93.9057388305664, 100.17491149902344, 105.23704528808594, 98.75845336914062, 89.37869262695312, 89.72357940673828, 91.20866394042969, 101.54448699951172, 101.19182586669922, 104.7577133178711, 93.55919647216797, 98.28685760498047, 99.37689208984375, 110.34868621826172, 109.96299743652344, 100.77191925048828, 109.43061065673828, 98.4088363647461, 98.41374206542969, 104.57550048828125, 101.2792739868164, 98.74390411376953, 102.90879821777344, 96.77204895019531, 96.87855529785156, 97.21239471435547, 102.34986877441406, 97.63624572753906, 97.05376434326172, 99.994873046875, 96.30327606201172, 102.80486297607422, 92.59151458740234, 102.00050354003906, 103.593505859375, 92.0771713256836, 106.49302673339844, 97.55835723876953, 104.46430206298828, 95.08956146240234, 102.77210998535156, 106.07617950439453, 99.46729278564453, 105.02510833740234, 104.79584503173828, 96.98860931396484, 94.74043273925781, 108.10029602050781, 97.61295318603516, 105.68717956542969, 94.24896240234375, 100.31581115722656, 109.37493133544922, 107.94377136230469, 98.4963150024414, 103.85598754882812, 98.68807220458984, 92.84165954589844, 111.23713684082031, 97.176025390625, 101.59282684326172, 96.75553131103516, 93.68482971191406, 100.87226104736328, 105.24888610839844, 95.48401641845703, 97.83047485351562, 100.83551025390625, 102.94905853271484, 94.04142761230469, 107.55349731445312, 108.55205535888672, 100.16251373291016, 94.59705352783203, 97.77267456054688, 107.1822738647461, 101.03285217285156, 91.90409088134766, 98.77691650390625, 106.53187561035156, 99.33047485351562, 102.48050689697266, 109.8247299194336, 98.61537170410156, 100.64257049560547, 101.84210968017578, 104.95992279052734, 97.49406433105469, 108.5066909790039, 93.65782928466797, 94.3360824584961, 98.66145324707031, 99.25119018554688, 91.94863891601562, 100.8731460571289, 98.6570053100586, 105.41444396972656, 100.77169036865234, 106.47164916992188, 107.64849853515625, 97.86259460449219, 99.14946746826172, 100.37137603759766, 96.80132293701172, 99.3827133178711, 96.75714874267578, 100.73064422607422, 96.98973846435547, 106.40042114257812, 111.24482727050781, 92.1896743774414, 99.21623229980469, 101.55178833007812, 94.93777465820312, 99.64090728759766, 100.29007720947266, 92.83350372314453, 106.959716796875, 103.0760498046875, 103.88237762451172, 98.33221435546875, 100.33186340332031, 102.13188171386719, 98.74321746826172, 103.05712127685547, 98.90404510498047, 102.65438079833984, 95.65533447265625, 100.03842163085938, 101.39568328857422, 99.53787994384766, 110.68500518798828, 98.46349334716797, 104.3675308227539, 99.94642639160156, 87.32037353515625, 105.08488464355469, 107.61174011230469, 109.06932830810547, 98.38236999511719, 96.4063949584961, 91.71417236328125, 102.77232360839844, 101.88813018798828, 104.88837432861328, 109.58255767822266, 100.58916473388672, 104.8829345703125, 110.18833923339844, 106.35618591308594, 93.45411682128906, 92.65650939941406, 101.61831665039062, 97.49098205566406, 99.16353607177734, 104.00109100341797, 99.9946060180664, 104.5677490234375, 105.01143646240234, 101.40884399414062, 91.3906021118164, 97.4963150024414, 93.50897979736328, 98.91344451904297, 99.31038665771484, 98.23968505859375, 91.43827056884766, 94.19033813476562, 97.6361312866211, 99.40612030029297, 99.1937255859375, 99.13422393798828, 109.45147705078125, 91.83048248291016, 100.48954772949219, 99.13553619384766, 92.80480194091797, 90.9131088256836, 92.96492004394531, 101.7186279296875, 97.87298583984375, 96.82128143310547, 96.60099792480469, 97.33491516113281, 95.87031555175781, 92.7610855102539, 90.4397201538086, 100.67341613769531, 105.04031372070312, 98.75686645507812, 102.52131652832031, 99.5259017944336, 96.99430847167969, 102.69759368896484, 100.18104553222656, 101.4179916381836, 106.62178039550781, 98.25751495361328, 94.80223846435547, 101.94691467285156, 109.30745697021484, 99.94697570800781, 105.11904907226562, 96.41851043701172, 110.88896942138672, 99.84602355957031, 96.96492767333984, 106.5897216796875, 95.13996887207031, 97.2563705444336, 97.31550598144531, 95.81836700439453, 115.73284912109375, 103.9183120727539, 101.57360076904297, 106.09861755371094, 102.38748931884766, 109.48471069335938, 103.48646545410156, 94.79946899414062, 104.71861267089844, 102.04361724853516, 99.3959732055664, 102.78424072265625, 100.06867218017578, 96.55975341796875, 107.72779846191406, 100.05824279785156, 102.81572723388672, 97.34088897705078, 91.3453598022461, 98.59423828125, 99.91840362548828, 108.22368621826172, 97.53971862792969, 102.64107513427734, 95.38522338867188, 90.36935424804688, 97.26162719726562, 93.85255432128906, 102.59059143066406, 95.26583862304688, 94.72511291503906, 101.60399627685547, 90.54598999023438, 103.50415802001953, 88.57701110839844, 101.5892333984375, 102.029541015625, 89.37556457519531, 93.5433578491211, 98.12728118896484, 90.677978515625, 104.97366333007812, 103.5368423461914, 98.49825286865234, 97.99420166015625, 103.37545013427734, 97.27739715576172, 98.11949920654297, 99.6930923461914, 106.91304016113281, 104.0692138671875, 98.22271728515625, 93.6544418334961, 92.07101440429688, 96.05187225341797, 96.05522155761719, 95.590087890625, 104.32633972167969, 103.87481689453125, 100.44690704345703, 94.81867218017578, 106.29878234863281, 98.262451171875, 105.94620513916016, 99.66912841796875, 96.60934448242188, 105.1146469116211, 98.13213348388672, 104.04412078857422, 97.40642547607422, 107.58689880371094, 86.8786849975586, 99.0611343383789, 105.8905258178711, 100.52588653564453, 100.43347930908203, 97.65369415283203, 91.62401580810547, 96.88050842285156, 97.44780731201172, 91.8355712890625, 102.32489776611328, 100.62450408935547, 98.49065399169922, 94.61164855957031, 92.92147064208984, 106.9343490600586, 100.22367858886719, 104.37893676757812, 99.1640853881836, 91.46810913085938, 102.2788314819336, 97.01382446289062, 92.01084899902344, 106.17671203613281, 105.44336700439453, 107.64988708496094, 97.55596160888672, 103.29667663574219, 106.18888854980469, 102.3386459350586, 98.036865234375, 97.08942413330078, 93.63150787353516, 100.05441284179688, 105.92324829101562, 98.96441650390625, 101.7375717163086, 109.40825653076172, 96.82244873046875, 104.81581115722656, 107.95094299316406, 98.08055114746094, 103.42764282226562, 100.04419708251953, 92.93846130371094, 100.99739074707031, 101.81119537353516, 101.1807861328125, 101.7420654296875, 102.943115234375, 97.93350982666016, 99.31025695800781, 94.24894714355469, 104.1230697631836, 99.1671371459961, 98.40836334228516, 110.96308135986328, 97.68499755859375, 103.7861328125, 94.1565933227539, 104.00547790527344, 92.45893096923828, 104.14456176757812, 98.27743530273438, 105.73127746582031, 95.12045288085938, 109.70932006835938, 100.05267333984375, 95.08014678955078, 100.73216247558594, 103.06002807617188, 98.03195190429688, 95.12127685546875, 109.84952545166016, 97.45174407958984, 92.64788055419922, 109.12315368652344, 109.02239227294922, 101.48594665527344, 98.1660385131836, 108.09982299804688, 104.7943115234375, 103.8004150390625, 98.48171997070312, 103.6897964477539, 106.28204345703125, 110.03135681152344, 107.03722381591797, 101.21944427490234, 103.30427551269531, 99.40726470947266, 100.89320373535156, 105.87923431396484, 97.50967407226562, 108.65292358398438, 103.4465103149414, 102.04325866699219, 99.82567596435547, 107.08300018310547, 101.30130767822266, 94.86488342285156, 94.80854034423828, 97.87507629394531, 103.61832427978516, 99.7106704711914, 97.52705383300781, 100.96629333496094, 97.32550048828125, 102.87361907958984, 105.5987777709961, 103.49820709228516, 101.76913452148438, 92.1997299194336, 105.75035095214844, 100.91452026367188, 100.28691101074219, 98.6703872680664, 105.81212615966797, 99.33965301513672, 98.1003189086914, 94.95011901855469, 105.16133117675781, 86.24362182617188, 104.36812591552734, 94.2710189819336, 106.60759735107422, 97.72077178955078, 105.41294860839844, 100.90058135986328, 109.87823486328125, 100.86180114746094, 103.2798080444336, 99.10553741455078, 97.15991973876953, 99.2658920288086, 99.20738983154297, 108.97827911376953, 100.74488067626953, 97.89521026611328, 86.78321838378906, 89.65630340576172, 98.05104064941406, 98.7578125, 107.45539855957031, 97.49263000488281, 90.78549194335938, 103.51628112792969, 95.77989196777344, 98.09596252441406, 105.68411254882812, 105.43997192382812, 95.51554870605469, 110.06001281738281, 98.4460220336914, 98.17179870605469, 97.81140899658203, 95.33609008789062, 111.11262512207031, 95.63982391357422, 91.18941497802734, 92.0676040649414, 96.25543975830078, 99.39169311523438, 99.9588851928711, 103.35430908203125, 96.790283203125, 103.7663345336914, 93.23464965820312, 99.93255615234375, 110.32984924316406, 89.06572723388672, 104.07081604003906, 98.7735824584961, 97.15086364746094, 109.65357971191406, 90.87054443359375, 93.581787109375, 95.70639038085938, 106.4747085571289, 104.6787338256836, 103.402587890625, 95.92301940917969, 100.75396728515625, 99.98725128173828, 91.49050903320312, 95.72073364257812, 96.96002960205078, 99.19483184814453, 104.8973388671875, 97.6578140258789, 103.89341735839844, 100.24284362792969, 99.97647857666016, 107.09589385986328, 93.7537612915039, 100.08197784423828, 105.60140228271484, 91.11746978759766, 98.39154052734375, 100.32645416259766, 95.6357650756836, 104.31224060058594, 103.15394592285156, 105.03363800048828, 94.22515106201172, 101.9351577758789, 94.48686218261719, 95.69369506835938, 100.83953094482422, 94.96631622314453, 103.87349700927734, 105.54305267333984, 96.89671325683594, 105.11044311523438, 110.07481384277344], [97.1406478881836, 97.98900604248047, 102.7231674194336, 108.80962371826172, 103.05494689941406, 108.8191146850586, 90.64522552490234, 108.41561126708984, 100.13328552246094, 101.22907257080078, 98.76616668701172, 98.66937255859375, 98.2229995727539, 107.81424713134766, 95.80908966064453, 102.9296646118164, 97.0086669921875, 101.08061981201172, 95.4004135131836, 97.77645111083984, 100.09264373779297, 101.0425796508789, 99.64103698730469, 96.66210174560547, 104.54866027832031, 103.19598388671875, 94.07625579833984, 102.16362762451172, 96.05489349365234, 109.27542877197266, 99.39778137207031, 100.17454528808594, 94.71186828613281, 99.53533172607422, 109.7744369506836, 105.54798889160156, 95.09014892578125, 98.48280334472656, 91.96756744384766, 100.7756576538086, 89.5385971069336, 110.65400695800781, 102.91030883789062, 101.53702545166016, 103.79441833496094, 104.1249771118164, 92.17340087890625, 95.47953033447266, 92.99835205078125, 91.786376953125, 97.23020935058594, 105.66748809814453, 102.4193115234375, 98.61119079589844, 93.26141357421875, 101.18585968017578, 105.28599548339844, 92.09888458251953, 108.8895263671875, 97.97040557861328, 102.6497802734375, 103.56400299072266, 98.55061340332031, 101.7256088256836, 95.87308502197266, 103.66058349609375, 100.54766845703125, 104.32460021972656, 95.4751968383789, 104.59432983398438, 89.13607788085938, 97.98919677734375, 99.9568099975586, 98.56629180908203, 107.7745590209961, 94.96021270751953, 111.55853271484375, 100.1335220336914, 104.41863250732422, 91.1268081665039, 109.27228546142578, 94.08917236328125, 100.14752960205078, 99.8939437866211, 100.697998046875, 105.7601318359375, 100.5060806274414, 106.98200225830078, 91.67497253417969, 99.64608001708984, 98.44412231445312, 100.62738800048828, 94.9931869506836, 96.90106964111328, 97.5040054321289, 109.1146011352539, 96.8268814086914, 109.11824798583984, 92.85612487792969, 92.94365692138672, 93.89637756347656, 106.50201416015625, 93.85970306396484, 103.59443664550781, 93.86550903320312, 95.63736724853516, 107.9725570678711, 106.95083618164062, 95.51396179199219, 99.06439971923828, 92.3338851928711, 92.68339538574219, 97.54975891113281, 102.62547302246094, 101.42020416259766, 100.91546630859375, 98.23563385009766, 106.08092498779297, 115.01566314697266, 93.85326385498047, 110.16529846191406, 106.11649322509766, 93.64083099365234, 98.03723907470703, 109.81475067138672, 97.4637680053711, 98.63375854492188, 105.20320129394531, 107.66442108154297, 93.06307220458984, 98.43905639648438, 101.09188842773438, 107.74048614501953, 96.10242462158203, 102.30732727050781, 97.90248107910156, 103.6851806640625, 104.1989517211914, 100.47797393798828, 96.25643920898438, 103.28020477294922, 102.47101593017578, 109.85905456542969, 97.145263671875, 104.52901458740234, 98.72258758544922, 102.26516723632812, 98.60228729248047, 93.48087310791016, 101.78544616699219, 100.08016967773438, 93.62591552734375, 100.91631317138672, 103.37459564208984, 105.72132110595703, 97.69895935058594, 105.85305786132812, 89.0294418334961, 100.24845886230469, 107.91822814941406, 105.29399871826172, 103.64704132080078, 107.81153869628906, 103.79847717285156, 95.00442504882812, 103.06399536132812, 100.5451431274414, 93.41426086425781, 99.27645111083984, 100.86312103271484, 97.62046813964844, 100.34407043457031, 105.49214172363281, 106.94212341308594, 101.19308471679688, 103.02818298339844, 103.07293701171875, 89.03892517089844, 99.33187103271484, 93.50141143798828, 106.31983184814453, 97.04312896728516, 93.10258483886719, 106.27607727050781, 99.8562240600586, 104.33367919921875, 92.96573638916016, 97.1571273803711, 95.41717529296875, 97.55498504638672, 95.4548568725586, 103.39410400390625, 91.68966674804688, 95.68167877197266, 99.56634521484375, 94.12592315673828, 102.20696258544922, 101.11461639404297, 101.92999267578125, 100.20818328857422, 101.86116027832031, 98.27716827392578, 101.99850463867188, 102.44791412353516, 101.30303955078125, 95.1528549194336, 97.1457748413086, 98.65784454345703, 94.74297332763672, 100.34740447998047, 110.1200180053711, 91.9774398803711, 93.36737060546875, 98.26078796386719, 100.7177963256836, 102.94749450683594, 96.09182739257812, 99.40814208984375, 100.69930267333984, 104.96778106689453, 91.52892303466797, 91.25768280029297, 104.07262420654297, 104.11275482177734, 97.9022216796875, 96.00796508789062, 107.19715881347656, 97.68421936035156, 98.37958526611328, 106.46781921386719, 103.03857421875, 98.74099731445312, 106.81027221679688, 95.5799560546875, 84.19032287597656, 103.6060791015625, 114.40176391601562, 91.96050262451172, 96.27484893798828, 95.14952850341797, 104.92448425292969, 100.4006118774414, 87.01399993896484, 101.74008178710938, 98.46452331542969, 99.55299377441406, 100.26085662841797, 96.64745330810547, 99.28740692138672, 102.36438751220703, 108.9244613647461, 104.47491455078125, 100.57636260986328, 105.59346771240234, 97.0330810546875, 96.19898986816406, 99.7004165649414, 96.20030975341797, 107.74817657470703, 100.57926177978516, 94.95585632324219, 98.06178283691406, 101.7446060180664, 97.9107666015625, 100.05248260498047, 96.78180694580078, 91.61477661132812, 108.0733871459961, 94.93687438964844, 93.22933197021484, 95.87097930908203, 96.34326171875, 101.84593963623047, 101.03260803222656, 98.90391540527344, 100.34173583984375, 99.47037506103516, 99.17853546142578, 105.47354888916016, 99.7989273071289, 102.88016510009766, 101.01079559326172, 101.95115661621094, 100.6592788696289, 107.23324584960938, 96.4001693725586, 102.02794647216797, 93.8993911743164, 108.73210144042969, 104.6669921875, 106.3441390991211, 98.06136322021484, 93.96018981933594, 102.4554443359375, 97.48223876953125, 110.45484924316406, 98.45146179199219, 104.14720153808594, 88.45442962646484, 98.11677551269531, 98.36132049560547, 99.902587890625, 100.86540985107422, 100.34667205810547, 93.38471221923828, 100.22510528564453, 94.35223388671875, 103.08329010009766, 99.53865814208984, 102.22844696044922, 101.80046844482422, 101.3881607055664, 106.7132797241211, 95.02274322509766, 109.0302505493164, 97.65042114257812, 94.25447845458984, 97.62229919433594, 99.01691436767578, 96.58079528808594, 96.36651611328125, 96.11787414550781, 102.7083969116211, 91.9228744506836, 101.873046875, 98.37450408935547, 88.7335205078125, 98.75729370117188, 97.1502914428711, 102.61585235595703, 105.10568237304688, 93.08648681640625, 111.29057312011719, 97.30907440185547, 102.69627380371094, 101.4245834350586, 92.92430114746094, 105.08349609375, 106.66436004638672, 106.34089660644531, 104.1031494140625, 107.27590942382812, 97.54861450195312, 106.37921142578125, 99.862548828125, 100.92124938964844, 104.9429702758789, 102.3609619140625, 93.39842987060547, 99.06544494628906, 94.15764617919922, 102.2977294921875, 104.9205093383789, 102.61808013916016, 88.81729888916016, 99.33209991455078, 110.51752471923828, 108.04698181152344, 102.35881805419922, 92.6166000366211, 96.0359115600586, 102.9894790649414, 105.09745025634766, 100.23909759521484, 108.27194213867188, 101.88483428955078, 97.57936096191406, 101.68025970458984, 92.97285461425781, 106.77498626708984, 102.99090576171875, 100.03263092041016, 100.51876068115234, 104.30490112304688, 101.57029724121094, 103.81151580810547, 101.9277572631836, 105.24203491210938, 104.28889465332031, 109.5806884765625, 95.10523986816406, 101.58556365966797, 96.92266845703125, 96.96543884277344, 101.60140991210938, 100.68670654296875, 87.17504119873047, 93.03361511230469, 98.3249282836914, 101.0265121459961, 97.46049499511719, 97.6043472290039, 95.07417297363281, 95.48072052001953, 97.57389831542969, 100.37198638916016, 102.8665771484375, 95.02780151367188, 103.38420104980469, 102.68423461914062, 100.72161865234375, 109.34186553955078, 97.44798278808594, 104.87506103515625, 101.4679946899414, 105.22740173339844, 89.36035919189453, 87.71177673339844, 109.37516021728516, 104.06927490234375, 91.27306365966797, 106.27098083496094, 98.53531646728516, 106.18504333496094, 100.84619140625, 95.00474548339844, 102.68404388427734, 91.37284851074219, 96.89271545410156, 103.39131927490234, 101.8709716796875, 94.64907836914062, 102.11827087402344, 105.61091613769531, 96.11724090576172, 101.315673828125, 94.80821228027344, 101.02571105957031, 91.96050262451172, 100.59170532226562, 100.68721008300781, 107.73316955566406, 102.51758575439453, 98.72388458251953, 96.84735107421875, 101.63170623779297, 101.18598937988281, 98.32661437988281, 96.44068145751953, 97.535888671875, 106.57430267333984, 103.3966064453125, 93.3302993774414, 89.37383270263672, 103.28257751464844, 100.29618835449219, 94.83097839355469, 90.34007263183594, 99.4427490234375, 100.8384017944336, 96.9690170288086, 110.9668197631836, 102.92066955566406, 97.59406280517578, 104.83723449707031, 101.31455993652344, 102.62418365478516, 91.22822570800781, 99.32540893554688, 100.35133361816406, 106.66510009765625, 104.03687286376953, 96.10365295410156, 99.16568756103516, 93.9535903930664, 105.38119506835938, 94.79660034179688, 91.99298858642578, 94.53165435791016, 104.18328094482422, 92.80311584472656, 91.18426513671875, 107.74602508544922, 103.59960174560547, 98.57722473144531, 105.74251556396484, 101.37238311767578, 113.61422729492188, 103.43339538574219, 95.95799255371094, 104.89892578125, 102.45499420166016, 90.68649291992188, 102.03681945800781, 94.31266784667969, 94.8203125, 86.24736022949219, 105.12277221679688, 95.6270523071289, 93.14812469482422, 91.38159942626953, 85.59815979003906, 102.1315689086914, 101.6024398803711, 94.9530258178711, 107.79365539550781, 91.66484832763672, 96.40750122070312, 92.88874053955078, 102.06099700927734, 97.76256561279297, 92.58889770507812, 99.66448974609375, 102.76778411865234, 105.69844055175781, 89.17411041259766, 91.93700408935547, 97.19017028808594, 94.53295135498047, 98.96044921875, 90.62306213378906, 105.10944366455078, 100.3947525024414, 100.93535614013672, 110.15310668945312, 99.56893157958984, 100.80586242675781, 93.79598236083984, 100.90631866455078, 101.86544799804688, 94.90315246582031, 106.9500961303711, 103.45114135742188, 99.93575286865234, 94.6102523803711, 92.21932983398438, 99.55567169189453, 101.49481964111328, 92.34077453613281, 90.60550689697266, 102.0811767578125, 104.6015396118164, 100.18057250976562, 103.42922973632812, 110.57102966308594, 94.73787689208984, 96.69856262207031, 108.83248901367188, 96.4151611328125, 96.24653625488281, 103.81759643554688, 101.57150268554688, 104.3736801147461, 101.3306884765625, 96.13931274414062, 106.88676452636719, 90.69890594482422, 108.41850280761719, 99.5416488647461, 94.33208465576172, 110.17160034179688, 100.66175079345703, 98.89279174804688, 97.81375122070312, 105.02233123779297, 102.0521011352539, 93.30835723876953, 98.36534881591797, 113.008544921875, 96.14675903320312, 105.72003936767578, 93.27561950683594, 110.65138244628906, 98.2071304321289, 95.11759948730469, 106.130126953125, 100.50926971435547, 94.30735778808594, 96.31726837158203, 94.46563720703125, 101.84796905517578, 108.25270080566406, 99.61468505859375, 105.80022430419922, 101.93791961669922, 96.82596588134766, 94.9681396484375, 102.94951629638672, 100.10760498046875, 102.82904815673828, 102.88801574707031, 99.69454193115234, 104.72666931152344, 105.98998260498047, 101.96344757080078, 99.83719635009766, 94.24574279785156, 95.6698226928711, 105.7936019897461, 99.18155670166016, 92.03974151611328, 88.86509704589844, 95.87345886230469, 100.80609893798828, 91.0426025390625, 98.8077621459961, 107.73137664794922, 101.13451385498047, 104.87540435791016, 101.06058502197266, 100.86412048339844, 100.92646789550781, 106.6517333984375, 102.27234649658203, 99.57334899902344, 110.55030822753906, 95.92837524414062, 103.56561279296875, 104.7245101928711, 99.07645416259766, 101.19772338867188, 102.66272735595703, 103.28694915771484, 100.18157196044922, 99.406982421875, 91.76129913330078, 94.66808319091797, 101.61294555664062, 95.05081176757812, 98.2525405883789, 99.47238159179688, 101.70992279052734, 91.27777099609375, 108.31549835205078, 103.2085952758789, 102.21077728271484, 103.39653015136719, 88.67439270019531, 94.8925552368164, 102.37567138671875, 89.54224395751953, 102.25503540039062, 95.66827392578125, 102.24625396728516, 96.03134155273438, 108.04710388183594, 106.99515533447266, 98.41612243652344, 96.30167388916016, 100.12051391601562, 94.84996032714844, 97.60901641845703, 103.10250854492188, 96.79222106933594, 101.04395294189453, 104.21682739257812, 93.7403335571289, 94.90680694580078, 94.25244140625, 93.47993469238281, 107.64553833007812, 91.12730407714844, 95.13704681396484, 100.55071258544922, 97.60057067871094, 107.7306900024414, 93.29497528076172, 96.19466400146484, 99.49729919433594, 93.07048034667969, 98.2931137084961, 102.66716003417969, 100.61737823486328, 96.39054107666016, 96.33353424072266, 110.5758056640625, 94.90826416015625, 97.39107513427734, 105.8618392944336, 100.3572769165039, 108.66680145263672, 108.21117401123047, 95.3882064819336, 98.63117218017578, 105.1325912475586, 101.91353607177734, 103.85791778564453, 105.8473129272461, 103.9740982055664, 93.83352661132812, 101.56594848632812, 99.26428985595703, 102.3046646118164, 102.9184799194336, 99.34141540527344, 109.37994384765625, 98.10807037353516, 88.47394561767578, 98.09203338623047, 90.3122787475586, 97.73558807373047, 103.8117446899414, 102.0533218383789, 94.18426513671875, 98.53904724121094, 102.77737426757812, 93.60220336914062, 100.95415496826172, 102.96997833251953, 97.3426513671875, 95.50476837158203, 92.89141845703125, 95.0879898071289, 104.0827865600586, 88.65046691894531, 110.36421203613281, 99.45651245117188, 101.66309356689453, 98.08361053466797, 93.66561889648438, 102.99200439453125, 101.58928680419922, 98.80706787109375, 103.17965698242188, 96.88510131835938, 104.93254852294922, 98.64661407470703, 107.39460754394531, 100.17638397216797, 91.84295654296875, 97.25885772705078, 97.00020599365234, 109.23782348632812, 99.69866180419922, 93.78915405273438, 101.3919677734375, 101.88763427734375, 98.7803726196289, 97.15021514892578, 105.6927261352539, 98.70675659179688, 99.4436264038086, 88.88630676269531, 94.74282836914062, 98.17231750488281, 94.36079406738281, 97.19670104980469, 96.1666259765625, 107.20081329345703, 99.10907745361328, 101.53883361816406, 93.9874267578125, 98.92239379882812, 94.56256866455078, 96.47420501708984, 97.77313995361328, 100.6866455078125, 102.01434326171875, 101.90376281738281, 90.19451904296875, 103.3140640258789, 102.17170715332031, 107.45024871826172, 101.15666198730469, 100.12857055664062, 98.8628158569336, 90.29034423828125, 100.85999298095703, 102.41078186035156, 102.9171142578125, 95.34587097167969, 102.68819427490234, 107.78358459472656, 105.92504119873047, 86.46287536621094, 98.41876983642578, 105.04865264892578, 93.88713073730469, 106.35218811035156, 100.96795654296875, 98.30889129638672, 98.61455535888672, 88.29928588867188, 103.28013610839844, 98.85016632080078, 98.39678192138672, 105.27532196044922, 100.02157592773438, 92.88507843017578, 107.96995544433594, 100.4640884399414, 107.98475646972656, 102.81841278076172, 99.9614028930664, 95.86833953857422, 96.319091796875, 104.94800567626953, 103.09772491455078, 87.5240707397461, 89.5296630859375, 101.47080993652344, 98.00528717041016, 101.51359558105469, 99.53471374511719, 105.0424575805664, 93.75883483886719, 97.22898864746094, 98.45063781738281, 97.59636688232422, 100.85114288330078, 92.99462127685547, 96.49999237060547, 97.21533203125, 96.87165069580078, 97.94670867919922, 96.54244995117188, 103.19141387939453, 94.14488220214844, 101.6310043334961, 105.53741455078125, 97.99819946289062, 100.15966796875, 96.14604187011719, 92.55989837646484, 107.1677474975586, 94.53840637207031, 101.5605697631836, 96.68899536132812, 90.7863540649414, 105.0560531616211, 100.05866241455078, 98.74301147460938, 100.12079620361328, 99.8714370727539, 93.12893676757812, 101.67687225341797, 97.48137664794922, 104.49099731445312, 100.20406341552734, 107.53421020507812, 98.96741485595703, 107.51014709472656, 107.39906311035156, 100.44979095458984, 104.74799346923828, 99.57949829101562, 100.96379089355469, 102.88935089111328, 97.384521484375, 102.3978271484375, 98.3188705444336, 101.2112808227539, 105.2601318359375, 101.43301391601562, 95.08421325683594, 104.7752456665039, 96.3049087524414, 95.59109497070312, 100.6841049194336, 100.93023681640625, 99.14618682861328, 98.17379760742188, 99.20264434814453, 101.21183776855469, 100.39669799804688, 101.30558776855469, 97.13221740722656, 97.49815368652344, 99.4502182006836, 104.32817077636719, 102.74107360839844, 103.35489654541016, 96.13566589355469, 88.96220397949219, 92.54881286621094, 105.56793212890625, 100.677978515625, 102.02005004882812, 95.62261962890625, 99.1567153930664, 100.92445373535156, 95.0599136352539, 105.27686309814453, 105.08395385742188, 101.06890106201172, 99.89955139160156, 98.24065399169922, 97.98857116699219, 104.0810317993164, 100.7433853149414, 96.14503479003906, 101.3550796508789, 98.78012084960938, 109.10466003417969, 95.66248321533203, 96.84290313720703, 92.85533905029297, 99.33778381347656, 100.14849853515625, 102.50209045410156, 88.7484130859375, 89.64180755615234, 100.3617172241211, 98.4472885131836, 104.29781341552734, 107.04576873779297, 98.81270599365234, 104.31761932373047, 102.36929321289062, 94.80282592773438, 98.4179458618164, 92.90126037597656, 103.3256607055664, 101.60665130615234, 104.63997650146484, 98.18138122558594, 100.18911743164062, 104.16807556152344, 96.33704376220703, 98.08403015136719, 103.7513427734375, 96.79087829589844, 96.7156753540039, 106.02853393554688, 105.39246368408203, 94.49246215820312, 99.2994613647461, 95.24668884277344, 105.46836853027344, 90.64228820800781, 94.19775390625, 98.24310302734375, 97.15203857421875, 107.49855041503906, 95.82608032226562, 101.92784881591797, 93.80274200439453, 90.16708374023438, 96.23975372314453, 93.82795715332031, 98.01300048828125, 98.22067260742188, 106.75688934326172, 101.788330078125, 98.72488403320312, 103.43115997314453, 99.41651153564453, 95.95892333984375, 97.8698959350586, 88.47886657714844, 95.58179473876953, 107.4767837524414, 99.04058837890625, 92.43719482421875, 102.70073699951172, 102.74414825439453, 95.71377563476562, 96.19654083251953, 108.79474639892578, 95.19672393798828, 107.07728576660156, 107.88166046142578, 93.79021453857422, 90.32695770263672, 101.49606323242188, 102.23023223876953, 99.48431396484375, 95.2286148071289, 99.29690551757812, 100.29972839355469, 95.49765014648438, 98.85072326660156, 105.2177505493164, 100.12909698486328, 93.16817474365234, 101.84625244140625, 90.15825653076172, 94.26582336425781, 102.30634307861328, 102.95341491699219, 95.28784942626953, 91.27021026611328, 108.04704284667969, 104.02525329589844, 101.81565856933594, 99.81078338623047, 102.49993133544922, 91.97657012939453, 92.39860534667969, 95.50160217285156, 91.8659896850586, 100.64405822753906, 91.22254943847656, 108.6881103515625, 97.32642364501953, 98.7234878540039, 100.4198989868164, 101.70266723632812, 94.58048248291016, 106.83170318603516, 99.44176483154297, 105.11187744140625, 98.89143371582031, 96.86457824707031, 99.41380310058594, 96.75106811523438, 106.58251953125, 110.0894775390625, 104.26112365722656, 98.9024887084961]], "dtype": "float32"}, {"__nd__": [1.3549457788467407, 2.254474401473999, 0.1640487164258957, 0.31352323293685913, 1.5677095651626587, 0.7204802632331848, 0.5887486338615417, 0.5352562665939331, -1.5778590440750122, -0.7889564633369446, 1.548192024230957, 0.4756806492805481, -1.2405940294265747, -0.0015754246851429343, 0.16581180691719055, 0.5417919754981995, 0.6465843319892883, -2.2288873195648193, -1.536957025527954, 1.064084768295288, -0.9029706120491028, 0.6057139039039612, -0.30708080530166626, -0.7667565941810608, -1.9800124168395996, 1.4413444995880127, -0.9977886080741882, -0.21216867864131927, -0.32451221346855164, 0.2635829448699951, 0.45180433988571167, 1.6549004316329956, -0.9893851280212402, 0.24530759453773499, 0.6447998881340027, 1.5041530132293701, -0.2173628956079483, -0.6763923764228821, -0.05539079010486603, -0.1182091161608696, -0.5637862682342529, -0.843321681022644, -0.991997241973877, -0.8973010778427124, 2.16110897064209, -1.0951131582260132, -0.24405573308467865, 0.42638716101646423, 0.826280415058136, -0.7800310254096985, 0.209403395652771, 0.5800429582595825, 1.1242778301239014, -0.310763955116272, 0.7459372878074646, -0.7529876828193665, -1.1962270736694336, 2.4700326919555664, -1.337942123413086, -0.29696357250213623, -0.09877312183380127, -0.29910972714424133, 2.718353509902954, -0.22372739017009735, 1.547953724861145, 1.8970518112182617, 0.46964284777641296, 0.3826983571052551, -0.8722043037414551, 0.26341405510902405, 0.6618251800537109, -0.5110541582107544, 0.026826512068510056, 0.9547046422958374, -0.9760154485702515, 1.0832701921463013, 1.3180630207061768, -1.2181177139282227, 1.2025319337844849, -0.2396366000175476, 1.7453999519348145, 0.7334040999412537, 0.6419274806976318, 0.43678924441337585, 2.7236785888671875, -0.5513979196548462, 1.1406627893447876, -0.2504808306694031, 0.06429494172334671, -1.305156946182251, -0.7317361235618591, 0.3776809573173523, 0.9589440226554871, -0.2895386517047882, -1.4258155822753906, -0.6953167915344238, -0.5759866237640381, 1.5922648906707764, 0.35478344559669495, -0.515645444393158, -0.11773519217967987, 0.5844106674194336, -1.2059112787246704, -0.7496514916419983, 2.200784683227539, -0.43712276220321655, -0.809873640537262, -0.0413745753467083, 0.5744428038597107, -0.8435286283493042, 0.9489805698394775, -0.3169354200363159, 0.3396359384059906, 0.7967603802680969, 1.1267539262771606, 0.7271752953529358, 1.4080586433410645, -0.002936979755759239, 0.6602380871772766, 0.44934573769569397, -0.9923325777053833, -0.22598251700401306, -0.010349188931286335, 0.2147589772939682, 1.2097760438919067, -1.8094146251678467, -0.4072255492210388, 1.135904312133789, -0.4370879828929901, -1.2070741653442383, 0.5452213883399963, -0.3859078586101532, -1.3585647344589233, 0.9182666540145874, 1.7935985326766968, 1.0974830389022827, -0.050409093499183655, 0.7030196189880371, 0.1726396232843399, -0.10391674935817719, 0.037058357149362564, 0.7472151517868042, 0.8616482019424438, 0.5774425268173218, 0.1411544382572174, -0.3059038817882538, -0.6354800462722778, -0.5441249012947083, 1.5531320571899414, -1.3135735988616943, -0.12253240495920181, -0.7931858897209167, 0.5437363982200623, 0.3025805950164795, -0.7755828499794006, 0.2639382779598236, -0.4020399749279022, -0.5312399864196777, 1.7809759378433228, -0.2004576176404953, 1.7385709285736084, 0.30894988775253296, 0.7384709119796753, 0.7699940800666809, -0.6288779973983765, -0.31842732429504395, 0.7549446225166321, 1.3569962978363037, -1.243309736251831, -0.48221656680107117, -2.3079752922058105, 0.030521472916007042, -1.1793628931045532, 0.043059591203927994, 1.5326457023620605, -0.9996669888496399, 0.4546085000038147, 0.22278010845184326, -1.043247103691101, 1.4084633588790894, 1.132348895072937, -1.5122792720794678, 0.579502284526825, -0.7116193771362305, -2.643434524536133, -0.42221546173095703, 0.3715485632419586, -0.6007144451141357, 0.43472805619239807, 0.1092643141746521, 0.6142086982727051, 0.0011174525134265423, 1.536757469177246, 1.7418372631072998, -0.8114296793937683, 0.27641427516937256, -0.0424080528318882, -0.8524971008300781, 0.7681889533996582, 1.771376609802246, 0.230880469083786, 0.8494817018508911, -0.19965144991874695, 0.7886435985565186, -1.3908252716064453, -0.024199286475777626, -1.5488840341567993, -0.35198789834976196, 0.6765750646591187, -0.5009797811508179, 0.9569767713546753, 1.4244147539138794, -1.3869550228118896, 0.23294973373413086, 0.03828025981783867, -0.8174428343772888, -0.690159261226654, -0.6343002915382385, -0.35935115814208984, 1.6825835704803467, 0.8011777997016907, 0.4761921763420105, -0.1185469850897789, 0.554268479347229, -0.629006564617157, -0.16495823860168457, -0.6067411303520203, -1.0455940961837769, 1.0221765041351318, -1.116356611251831, 0.8085615038871765, 0.6848360300064087, -1.1000651121139526, 1.6239503622055054, -1.025122046470642, -0.44011807441711426, -0.12779735028743744, 0.7700372934341431, -0.1574687510728836, -0.3276572823524475, 1.0384800434112549, -1.5369089841842651, 1.619733214378357, -0.723808228969574, 0.4855197072029114, 0.8941130042076111, 1.773531436920166, -0.4246419668197632, -0.2631622850894928, -1.5821142196655273, 0.5219836831092834, -0.4629482924938202, 0.20000338554382324, 1.6932072639465332, 1.9533730745315552, -0.3187616169452667, 0.1670013815164566, -0.655511736869812, -0.9515519738197327, -1.1046786308288574, -0.899824857711792, 2.425142526626587, 0.21564605832099915, -0.7556003332138062, 1.596979022026062, -1.7619051933288574, -0.6511378884315491, 0.9837552309036255, -0.30806565284729004, -0.1731778085231781, -0.6556663513183594, -0.15482529997825623, 1.868199110031128, -0.14230258762836456, 0.5879307389259338, 0.24606093764305115, -1.1565848588943481, 0.7022593021392822, 3.2234153747558594, 0.5334697365760803, -1.808274745941162, -0.09672783315181732, 1.0715835094451904, 0.49657514691352844, -2.3958418369293213, -1.0666894912719727, 0.4877918064594269, -1.5760819911956787, 0.22003494203090668, -0.46072688698768616, -0.39849016070365906, 1.1542853116989136, -0.03737662360072136, -0.4728076756000519, -1.5707285404205322, -0.7800079584121704, -0.9844523668289185, 1.2988128662109375, -0.9477558732032776, 1.2314908504486084, 0.13900674879550934, -0.3209267854690552, -1.072670340538025, 1.859953761100769, 0.3167496621608734, 1.1452162265777588, -0.8041462302207947, 2.870028257369995, 1.1135748624801636, -1.1971925497055054, -1.0244309902191162, -2.205010175704956, 1.4481475353240967, -0.5589110255241394, -0.7258488535881042, -0.8600937128067017, 1.070111870765686, -0.14001689851284027, 0.6545016765594482, 0.98542720079422, -0.37692877650260925, 0.260270357131958, -0.6583082675933838, 1.4398497343063354, 0.3749541938304901, -0.9459764957427979, -1.2808640003204346, -1.039373517036438, 0.25168851017951965, -0.6151542067527771, -0.43496862053871155, -0.14358513057231903, 1.1014572381973267, -0.2724473476409912, 1.0159507989883423, 2.2154316902160645, -0.21551212668418884, 0.3041840195655823, -1.0299354791641235, 1.129299521446228, -1.6752413511276245, 0.15117286145687103, 1.1720588207244873, -0.4435434341430664, 0.214149609208107, -1.4698344469070435, -0.95076984167099, -1.1595638990402222, -0.7217252850532532, -0.2632577121257782, -0.48705822229385376, -1.1848969459533691, 1.438955307006836, 1.1608951091766357, -0.25791049003601074, 1.5771942138671875, -0.9219408631324768, -0.4670076370239258, 0.7739143967628479, 0.8552200794219971, 0.49458616971969604, 0.557313859462738, 0.09779813885688782, 0.05852543190121651, 0.908842146396637, 1.154796838760376, -0.24993668496608734, -0.8416865468025208, 1.3665388822555542, 1.73572838306427, -0.3867042660713196, -0.17474916577339172, -1.5529536008834839, -0.7882176637649536, -1.2149945497512817, 0.0041620926931500435, 0.2601860761642456, 0.2329898327589035, -0.1370556652545929, -2.7073793411254883, -0.45533695816993713, 1.7060186862945557, -1.0092109441757202, 0.26563647389411926, -0.9696242213249207, 1.3545130491256714, -0.36329662799835205, 0.9629800915718079, 1.0686087608337402, -0.43397051095962524, 1.0759284496307373, 1.7432358264923096, 0.2029295265674591, 0.7753245830535889, 1.2144744396209717, 0.5941399931907654, 0.14129547774791718, 3.940347909927368, 1.012769103050232, -0.5451552867889404, -0.7092717885971069, -0.39974290132522583, 0.929150402545929, 0.2691071927547455, 0.2422245740890503, -0.27154645323753357, 0.995429277420044, 0.22529301047325134, 0.2926112413406372, 0.20866115391254425, -0.5216392874717712, -1.3975437879562378, 0.7426352500915527, -2.698695182800293, -0.5637127757072449, -0.11813715845346451, -0.5826502442359924, -0.0817352831363678, 0.3481501042842865, 1.2709726095199585, 0.9221388697624207, -0.5618408918380737, 0.17184337973594666, 0.23809851706027985, 0.9390193223953247, 0.688795268535614, 0.6180480718612671, 0.7875561714172363, -0.482125461101532, 0.6106991171836853, -0.9440826773643494, -0.8414669036865234, 1.143602967262268, 0.6087020039558411, -1.59927499294281, 0.3696335554122925, 0.8636007905006409, 0.6791250705718994, -0.13580186665058136, 1.1796435117721558, -0.4506981372833252, -0.09279932081699371, 0.006802667863667011, 0.013801672495901585, -1.3231468200683594, -0.14562135934829712, 0.9316383004188538, -1.4765191078186035, -0.02193642035126686, -0.4029851257801056, 0.3147057890892029, -2.356055974960327, 0.8407593965530396, 1.1513293981552124, 0.6830876469612122, -0.7433845400810242, 0.27246853709220886, -1.003791332244873, -1.3203020095825195, 0.7769734263420105, 0.5555534362792969, 1.2942287921905518, -1.536699652671814, -3.4641501903533936, 0.08249411731958389, 0.17772486805915833, -0.07895121723413467, -2.1513471603393555, -1.8460438251495361, 1.7353302240371704, 0.6070077419281006, -0.3061603605747223, 0.011601865291595459, 0.7352491021156311, 0.23314741253852844, -0.6768824458122253, -0.5437946915626526, -2.198402166366577, -0.10411261767148972, 1.0972639322280884, -0.355000764131546, -1.5059436559677124, -0.31724870204925537, 1.4482020139694214, 0.03508877381682396, -0.06653125584125519, -0.19237260520458221, -0.3816736042499542, 0.21173202991485596, -0.14429418742656708, -2.1748459339141846, 1.2452788352966309, 1.3798608779907227, -0.685741662979126, -0.09426849335432053, -0.6888917684555054, -0.7030941247940063, -0.7146508097648621, 0.36446380615234375, -0.32085251808166504, 0.19800259172916412, 0.9475281834602356, -0.16999752819538116, -0.12130449712276459, 0.24663576483726501, -0.39965370297431946, 1.2281427383422852, 0.051971036940813065, 0.3850557208061218, -0.46998733282089233, 0.2855566143989563, -0.13851198554039001, -0.5951152443885803, 0.15973293781280518, 2.309054374694824, 1.1091549396514893, -0.08435501158237457, 0.7187088131904602, 0.3743472397327423, -0.9288447499275208, 0.8161840438842773, 0.6806471943855286, 0.4809471368789673, -0.15555378794670105, -0.9779945611953735, 2.0229275226593018, 0.17818689346313477, 0.48511844873428345, 0.7264437675476074, 1.271194338798523, -0.7457906007766724, -0.4935374855995178, -0.07876890897750854, -0.49936702847480774, 0.4252767860889435, -0.13063594698905945, -0.08348142355680466, 0.9680198431015015, 1.0549060106277466, -1.4047367572784424, -1.6255171298980713, -1.7764105796813965, 0.5464508533477783, -0.7000289559364319, 0.9518277049064636, -0.9347237348556519, 0.3307451605796814, -2.2817394733428955, 0.4056949019432068, -1.8204519748687744, 1.3689601421356201, 0.6540514826774597, 0.24220632016658783, -2.1791574954986572, 1.1042193174362183, 0.06320010125637054, -3.274007558822632, 0.39174067974090576, -0.1465654969215393, 0.23433327674865723, 0.6854856014251709, -1.6442052125930786, -0.291914165019989, 0.1675059050321579, 0.09768272191286087, -0.00042553621460683644, -0.244087815284729, -0.8344780802726746, 0.35274508595466614, 1.1983492374420166, 1.3094173669815063, 0.5349320769309998, -1.2481498718261719, -0.7786087393760681, -0.4618596136569977, 0.1546248495578766, -0.1474227011203766, 0.16959898173809052, 0.520390510559082, -1.1003516912460327, -1.2681293487548828, 0.5264962315559387, 0.4016878306865692, 0.7079569697380066, -0.5035843849182129, -0.9625496864318848, -0.5808736681938171, -1.2278485298156738, -1.1833935976028442, 0.18327215313911438, 0.8253080248832703, 1.1546494960784912, 0.5131943225860596, -0.4005208909511566, 1.0671443939208984, -0.13454195857048035, 0.21477310359477997, -0.8290606737136841, -0.29262998700141907, 0.8126768469810486, 1.1866661310195923, 0.48765829205513, -1.8781471252441406, 0.4595324695110321, 1.1594698429107666, -0.1251600980758667, -1.074830412864685, -0.910449743270874, 0.737589955329895, 0.23196184635162354, 1.6156436204910278, 0.3178313672542572, 0.733932375907898, -0.09221027791500092, 0.5906102061271667, 0.023090150207281113, 0.5798932909965515, 0.3315431475639343, 0.31287747621536255, -0.39904287457466125, 0.9346115589141846, -1.0147712230682373, -0.1602906435728073, 1.0850757360458374, -0.7396562695503235, 0.9293280839920044, -0.41001108288764954, 0.21779851615428925, -0.7283425331115723, -0.43985652923583984, -0.6585632562637329, -1.1887390613555908, 1.0627281665802002, 0.5975268483161926, -0.12142172455787659, 0.5942996740341187, 0.24169176816940308, -1.1175265312194824, 0.8392676711082458, -0.6008999943733215, 0.4274549186229706, 0.8778753280639648, -0.11951430886983871, -0.7755584716796875, -1.014426827430725, -0.08160951733589172, -1.0140196084976196, 0.5311876535415649, 1.8833816051483154, -0.09562493860721588, -0.15096251666545868, 1.6761823892593384, -2.2224645614624023, 2.2472431659698486, 0.5442873239517212, -1.3779126405715942, 0.8068851232528687, -1.1615339517593384, -0.7338857650756836, 0.13572958111763, 0.6685168147087097, -1.103595495223999, -1.1668479442596436, -2.072155714035034, -0.5655651092529297, 0.19646768271923065, 0.12628480792045593, -1.3147627115249634, 0.43619081377983093, 0.6035304665565491, -0.8157081604003906, -0.9128212332725525, -0.12991411983966827, -0.5916678309440613, -0.23668356239795685, 0.7646790146827698, 0.53148353099823, 1.8025833368301392, 0.07044041901826859, 1.3950393199920654, 0.3740677535533905, 0.38132375478744507, -0.5407549142837524, 0.018615402281284332, -1.6466599702835083, -2.1263933181762695, -0.7899996042251587, -1.2983664274215698, 1.4149738550186157, 1.2671966552734375, 0.8301305770874023, 0.9671801924705505, 1.0604463815689087, -1.993483543395996, 0.7301434278488159, 0.275423139333725, -0.5569289326667786, -1.334575891494751, -0.33208954334259033, 0.21283937990665436, -0.4512752890586853, 1.4286593198776245, -0.10200367122888565, -1.10198974609375, 0.9041648507118225, 0.5358026027679443, -0.019618479534983635, -0.45502665638923645, 0.9345349669456482, -0.45583027601242065, -1.7584137916564941, -0.7314437627792358, 1.026691198348999, -0.7088545560836792, 2.001058578491211, 0.3167155385017395, -0.4085560142993927, 0.407158225774765, -0.745574414730072, -0.06719724833965302, -0.8482852578163147, -0.3775477409362793, 0.4386261999607086, 1.7331502437591553, 0.011027815751731396, 0.11018890887498856, 0.5555190443992615, 0.007779715582728386, -0.20092712342739105, 0.05261692777276039, -0.9553802013397217, 0.20030860602855682, 0.1787136197090149, -1.5166441202163696, 0.3512447774410248, 0.3393523395061493, -0.19698543846607208, 0.08939959853887558, -1.2447682619094849, 0.2916088402271271, 1.3817040920257568, -0.5325501561164856, -0.4855484366416931, 0.918204128742218, 0.15337872505187988, 1.0465220212936401, -0.1061210110783577, -0.866130530834198, 1.1600209474563599, -0.08388438820838928, 0.13691608607769012, 1.0233080387115479, -0.3889603018760681, -1.888355016708374, -2.438863754272461, -0.9838711023330688, -0.19947026669979095, -0.7465090155601501, 0.5952033400535583, -1.63003671169281, -0.7962582111358643, -1.2673419713974, 0.6789860129356384, 0.786161482334137, 1.9957854747772217, 1.7940025329589844, 0.33800268173217773, -1.0526281595230103, -0.6273284554481506, 0.513444721698761, 0.8128803968429565, -0.2444094568490982, -1.511133074760437, -1.6074436902999878, 0.36684882640838623, -1.1748502254486084, 0.8489120006561279, 1.3211066722869873, 0.9084151387214661, 0.19750171899795532, -0.049508269876241684, -0.5547329187393188, 1.6576106548309326, 0.48589083552360535, 0.4285343587398529, -0.4188770651817322, -0.548774242401123, 0.19775667786598206, -0.19624851644039154, -0.6504977345466614, 0.12401784211397171, -0.8266720175743103, 1.2639631032943726, -1.432450771331787, -0.2247830480337143, -0.27030640840530396, -0.008633699268102646, -2.132105588912964, 0.8945361971855164, 0.4499530494213104, 0.38654547929763794, -0.8651161789894104, -1.9989460706710815, -0.36256343126296997, -0.45960935950279236, 0.3905656635761261, 0.13179130852222443, -0.7190724015235901, 1.104243278503418, 1.9398465156555176, -0.7544521689414978, 1.135758399963379, -0.5222062468528748, -0.26809394359588623, 0.3038170337677002, 1.0690056085586548, 0.8116688132286072, -0.9762474298477173, -1.2437052726745605, -0.10071305185556412, 2.555936574935913, -0.8485958576202393, -0.402088463306427, -0.29352524876594543, 0.9537133574485779, 0.3148907721042633, -0.5735777020454407, 1.3263813257217407, 0.9535549283027649, -0.0004266120959073305, -1.2895411252975464, -1.3357247114181519, 1.5268365144729614, 0.7718225121498108, 1.206842064857483, 2.2552411556243896, -1.1660877466201782, 0.6681228280067444, -0.9883134365081787, -0.46588292717933655, -0.6457360982894897, -2.045938730239868, -0.7333539724349976, -0.28456512093544006, -0.23020172119140625, 0.40048840641975403, -0.4225662052631378, 0.6273874044418335, 0.7565691471099854, -1.1910536289215088, 1.5242880582809448, -1.6365948915481567, -1.3297914266586304, 1.4845731258392334, 1.2042827606201172, 1.3127870559692383, 0.05212286114692688, 2.0082523822784424, -0.3697996735572815, -2.0567853450775146, 1.6760121583938599, 0.31945356726646423, 0.4456772208213806, -0.3532828688621521, 0.7685907483100891, 1.1490439176559448, 0.8004547953605652, 0.6707897782325745, 0.43260762095451355, 0.35607072710990906, 2.603275775909424, 0.38809898495674133, -1.6859289407730103, 0.7802978754043579, -2.4068048000335693, 1.2522830963134766, -1.6707180738449097, 0.0821739137172699, -1.4409347772598267, -0.3213954269886017, 1.6684956550598145, -1.1444705724716187, -1.1508853435516357, 0.17689181864261627, 0.9587417244911194, 0.5586330890655518, -0.6404390335083008, 0.577399730682373, 1.8737095594406128, 0.758581817150116, 1.2882018089294434, -0.6965116858482361, -0.4364526569843292, -1.089626431465149, 0.4571017622947693, 1.119619369506836, 0.6853086352348328, 1.2630149126052856, 1.2513666152954102, -1.3543617725372314, -0.800430417060852, -0.3787861466407776, 1.3446210622787476, -0.2287440299987793, 0.2713092267513275, 0.8270491361618042, -1.3106303215026855, -0.6575443148612976, -1.0203553438186646, -1.063014030456543, -1.002731204032898, 0.7194691896438599, -1.248597264289856, -0.14579638838768005, -0.3134765326976776, 0.9790476560592651, -0.8787275552749634, -0.11984635144472122, -1.2963262796401978, -0.3843716084957123, 2.1563851833343506, 1.96311616897583, -0.18033097684383392, -0.26473569869995117, 0.18174980580806732, 0.582565188407898, 1.6649740934371948, -1.5834063291549683, -0.48264145851135254, -0.612244188785553, -0.07411552220582962, 1.3823274374008179, -0.9980162382125854, -0.42094817757606506, -0.1784835159778595, 0.9973959922790527, -0.9408079385757446, 0.8974564671516418, -1.0208288431167603, 0.7706937193870544, -0.33109748363494873, -0.055231403559446335, -0.8519315123558044, 0.3847792148590088, 0.313101202249527, -1.3025211095809937, -0.5000728368759155, -0.4202136695384979, 1.296773910522461, -1.712085247039795, 1.343281626701355, -0.8293371200561523, -1.0310972929000854, 1.0939879417419434, 0.027845187112689018, 1.3921400308609009, -1.0468840599060059, -1.3790504932403564, 1.4993221759796143, 0.8121216297149658, 0.5111612677574158, -2.1998507976531982, -1.9557782411575317, 0.5363941192626953, -0.2390931397676468, -1.3459633588790894, 1.5436135530471802, -0.015263172797858715, 0.8932323455810547, -0.6257132291793823, 0.07744953036308289, 0.3439065217971802, 1.636385440826416, -0.5020518898963928, -0.01691894233226776, -0.7300314903259277, 0.41625505685806274, -0.7465835213661194, 0.6189526915550232, -0.8048104047775269, 1.602612018585205, 0.24737149477005005, -0.4265965521335602, 0.09811731427907944, 1.0158329010009766, 1.6620293855667114, 0.4186939001083374, -0.6758039593696594, 1.5249886512756348, -0.6382291913032532, -0.44085368514060974, -1.182865023612976, -0.8263596296310425, -0.753206729888916, 0.13666145503520966, -1.0345838069915771, 0.7595328092575073, -0.905724287033081, 1.0281531810760498, 0.2818559408187866], "dtype": "float32"}, {"__nd__": [0.36727771162986755, -0.29767659306526184, -2.8492887020111084, -0.2013087272644043, 0.06523488461971283, 2.01932692527771, 1.6061923503875732, -0.13792455196380615, -1.318600058555603, 0.2928273677825928, 0.6937121152877808, 1.118469476699829, 0.02545592561364174, -1.7472643852233887, -0.27208828926086426, 0.8956280946731567, 1.7377064228057861, 0.9507508873939514, -0.3105061948299408, 0.4336870610713959, 2.156886100769043, 0.34007543325424194, -0.7243291735649109, 0.8108477592468262, -0.7925955057144165, -0.09372884035110474, -0.02083008736371994, 0.5447556972503662, 0.7595691680908203, 0.9298920035362244, 1.09650456905365, -0.3610122799873352, -0.8345437049865723, -0.6310745477676392, 0.41537731885910034, -1.1001862287521362, 1.690662145614624, 0.6922370195388794, -0.7495214343070984, 0.8116564750671387, 1.5105111598968506, -0.42680758237838745, -0.5880931615829468, 1.36919367313385, -0.43530458211898804, 0.3917507529258728, -0.24837420880794525, 1.4516974687576294, -0.39259666204452515, 0.22028255462646484, -0.0893155187368393, -1.0437579154968262, 0.029683712869882584, -0.7225894927978516, 2.397521734237671, 0.1055082306265831, 0.8430556058883667, -0.42539718747138977, 0.5012709498405457, 0.017757758498191833, 0.9483307003974915, 1.6903270483016968, -1.5546164512634277, -0.03011021763086319, 0.5369920134544373, 1.165209174156189, -0.4822511374950409, 0.07864441722631454, 0.16034221649169922, 2.404334783554077, 1.101291537284851, -0.21845228970050812, 0.08712036907672882, -0.6101548075675964, 1.421941876411438, -1.6065099239349365, 1.9899994134902954, -0.1835578829050064, -0.8776994943618774, 1.6831982135772705, -0.008492638356983662, -0.48665088415145874, 0.24585287272930145, 1.2721872329711914, -0.06532778590917587, 0.27928397059440613, -0.2618667185306549, -1.0188658237457275, 0.31154823303222656, -1.313656210899353, 0.11390350759029388, -0.25710198283195496, 2.7364580631256104, 0.38649222254753113, 0.41046595573425293, -0.10806387662887573, -0.18485665321350098, 1.0457314252853394, -1.5021756887435913, -2.3014893531799316, 0.12185867130756378, -1.5059995651245117, 0.6323903799057007, 0.5249811410903931, -0.15720872581005096, -0.7613802552223206, -0.3490124046802521, -0.9609842896461487, -0.5548940300941467, -1.0584701299667358, 1.319644808769226, 0.745730996131897, -3.3757195472717285, 0.3350221812725067, 0.09426223486661911, -0.2757907509803772, 0.9376717805862427, 1.1432979106903076, 0.48906561732292175, -0.010448350571095943, 1.0361404418945312, 0.8858643770217896, -1.0241214036941528, -1.4368319511413574, -0.2927955687046051, -1.8645254373550415, 0.3814115524291992, 0.33254557847976685, 0.5712548494338989, -0.5923303961753845, -0.5642575621604919, 0.861840546131134, 0.18640440702438354, 0.0027101514860987663, -0.4260580837726593, 0.2048468440771103, -0.14524303376674652, -0.9729963541030884, 0.3737064003944397, 0.339591383934021, -0.722752571105957, 0.35132983326911926, -0.5143672823905945, -0.6573595404624939, 0.6914791464805603, -0.08930434286594391, 1.2870107889175415, -0.023975307121872902, 0.18749909102916718, 0.22685568034648895, -0.17931117117404938, -0.8656759262084961, -0.948249101638794, -0.004583953879773617, -0.7331506013870239, 0.9773755073547363, -0.9737434387207031, -1.9333465099334717, 1.220294713973999, 2.2266745567321777, 1.61448335647583, -1.452187180519104, -1.534628987312317, -1.9865427017211914, 1.0555098056793213, 1.9985501766204834, 0.5306827425956726, -0.33794382214546204, -0.1063193753361702, 1.351989984512329, 1.0970232486724854, 1.3016481399536133, 0.7912545204162598, -0.4877345561981201, -0.76320481300354, 0.9454045295715332, -1.3719147443771362, -0.5495279431343079, 0.3994118273258209, 1.8045567274093628, -0.4561580419540405, -0.8294330835342407, 1.4703450202941895, 0.8055076599121094, -0.07269486784934998, -0.54543536901474, -0.21122972667217255, -1.0722229480743408, 0.33524268865585327, 0.3228498101234436, -1.1819463968276978, -0.9848331809043884, 0.2169782668352127, 0.6970963478088379, -1.1639739274978638, 0.21667362749576569, 1.4828767776489258, 0.3394363224506378, -0.6211510896682739, 0.35934290289878845, -2.288559675216675, -2.0413126945495605, 2.1081206798553467, -1.5078999996185303, 0.9996179342269897, -0.36923491954803467, -0.326851487159729, -0.7284495234489441, -0.2181273251771927, 0.343117356300354, 1.0768027305603027, 2.00749135017395, 1.9760977029800415, 0.872217059135437, 1.4942467212677002, 0.7757824063301086, -0.2307041883468628, 0.18415242433547974, 0.1623356193304062, 0.2202337235212326, 0.7416179180145264, -0.3840005695819855, 1.2748820781707764, -0.14348763227462769, 0.9612458348274231, -0.0640849769115448, 0.35984623432159424, -0.2579987049102783, 0.27084434032440186, 0.33922088146209717, -0.5346770882606506, -0.3159646987915039, -0.4235990345478058, -0.6214484572410583, 0.11042912304401398, 2.1687228679656982, 1.1069080829620361, -0.4844905436038971, 1.9343352317810059, 1.1556898355484009, -0.9406468868255615, -0.1039615347981453, -0.6891313791275024, 0.1456206738948822, -0.761121392250061, 0.434243768453598, -0.1924966424703598, -1.523310661315918, 1.7406262159347534, -0.9002743363380432, 1.2224212884902954, -0.7942206263542175, -0.4544711410999298, 0.33628910779953003, 0.499830424785614, 0.9149288535118103, 0.4454292058944702, 0.06570129841566086, -0.5588758587837219, 0.9479033946990967, 1.2522239685058594, 0.3247572183609009, 0.017609013244509697, 0.9452522397041321, -0.06329062581062317, 0.2531276345252991, 0.16111354529857635, 0.44180479645729065, 1.7784607410430908, 0.7395608425140381, -1.1492938995361328, 0.03946789354085922, 0.5719696283340454, -0.5158019661903381, -0.2652989625930786, 1.6102961301803589, -0.3390234112739563, -2.181267023086548, -0.46582716703414917, -0.20072653889656067, 0.07736606895923615, 0.424835741519928, -1.9118390083312988, -0.8586122393608093, 0.5675725340843201, -0.921642541885376, -0.17452289164066315, -1.565674066543579, -1.139016032218933, 2.1378536224365234, 2.302074670791626, -0.9287430644035339, -0.5598131418228149, -1.8862041234970093, -0.6921071410179138, 0.20519709587097168, 0.3080345392227173, -0.504209578037262, 0.8373968005180359, 0.20732244849205017, 3.448716640472412, 0.1732555627822876, -0.7167881727218628, 1.6138404607772827, -0.3200537860393524, 0.6687532663345337, -0.38142403960227966, -0.6239685416221619, 1.3907568454742432, -1.0108375549316406, 0.23460155725479126, 0.0435975007712841, -0.4125903844833374, 0.4624851644039154, -1.1027421951293945, 2.308335304260254, -0.09307880699634552, 1.3655308485031128, 0.9232555031776428, 1.2145888805389404, -0.0015952559188008308, 0.6606426239013672, -0.33672237396240234, 0.5379296541213989, -1.168495774269104, 0.30507317185401917, -1.5705231428146362, -0.5060782432556152, 0.3259294927120209, -0.28194499015808105, -0.2397185117006302, 0.8658427596092224, 0.12770475447177887, -0.5895209908485413, 0.29435357451438904, -1.383147120475769, 0.7372515201568604, -0.9656832218170166, 1.1796492338180542, -0.20544670522212982, -1.2182270288467407, -0.7823191285133362, -3.008523464202881, 1.086408257484436, -0.29086288809776306, -0.4503064751625061, -1.2131465673446655, 0.7206408381462097, -0.12513746321201324, -0.15486393868923187, 0.9535566568374634, -0.41259947419166565, -0.31341317296028137, 0.34614846110343933, -0.26273638010025024, -0.08952317386865616, -1.6609995365142822, 0.5822679996490479, 0.7700230479240417, 1.1053611040115356, 1.4647430181503296, 1.6239368915557861, 1.0700980424880981, -1.7029192447662354, -0.5478549599647522, 0.7132061719894409, -1.8453229665756226, 0.7051661610603333, -1.4304618835449219, 0.3723199963569641, 1.308777093887329, 0.9094814658164978, 0.2463715523481369, 0.0495264045894146, -0.3291062116622925, 1.1732819080352783, 1.0997618436813354, 0.12063717842102051, -0.5157222747802734, 0.2034604847431183, -0.22582894563674927, -1.2789764404296875, -0.6868200302124023, -0.9185442328453064, -0.4413972496986389, 3.3020591735839844, 1.6975324153900146, -0.6735321879386902, -2.4864134788513184, 1.5010982751846313, 0.37491437792778015, -1.0286518335342407, 2.1073830127716064, 0.7644051909446716, -0.28609025478363037, 0.9100728631019592, -1.1964389085769653, 0.40814143419265747, -0.27081099152565, -2.113036870956421, 0.011815948411822319, -0.1471257209777832, -0.8207794427871704, -0.273861825466156, -0.6477497220039368, 0.7675692439079285, 2.1525747776031494, 0.8378525376319885, -0.8486781716346741, 0.31834185123443604, -0.44746077060699463, -0.14724937081336975, -0.1457124799489975, -0.49638631939888, -1.016842007637024, 0.2929811477661133, 0.05751407518982887, -1.5936875343322754, 0.3503972887992859, -0.5010589361190796, 0.49547308683395386, -1.2213850021362305, 0.8300923705101013, -0.30843132734298706, -0.3264923393726349, 0.3618182837963104, 1.6353256702423096, -1.6028625965118408, 1.8662075996398926, 1.074619174003601, 0.758963406085968, -1.0797853469848633, -0.44828981161117554, -0.28064554929733276, 0.7162195444107056, -0.6757580041885376, -0.5501517057418823, 2.044999122619629, 0.6556773781776428, -0.945310115814209, -1.6148319244384766, -0.461466521024704, 0.023366499692201614, 0.13957025110721588, 0.4049408435821533, -0.3861294090747833, -0.45491769909858704, 0.6954067945480347, -0.6197870373725891, -0.1357785165309906, 0.4832688570022583, -1.5193437337875366, -0.3854023516178131, 0.41325318813323975, -1.023851752281189, -1.2363280057907104, 0.20041769742965698, 1.049937129020691, -0.40004971623420715, -0.09301957488059998, 0.338459312915802, 0.4984298348426819, 1.268389344215393, 2.3244473934173584, 0.3066999018192291, -0.5838955640792847, 0.5129422545433044, -0.5853179693222046, 0.38033348321914673, -0.12736481428146362, 0.10183598101139069, 1.2077786922454834, 1.0265283584594727, -1.1645292043685913, -0.08764553815126419, 1.0917201042175293, -1.1663938760757446, -0.5314270853996277, -0.033192798495292664, -2.0752909183502197, -0.5330464243888855, -1.0851755142211914, 1.1923032999038696, -0.14017972350120544, 0.8488841652870178, -2.8906373977661133, -0.2773299813270569, -1.3015893697738647, -0.16221578419208527, 0.32753509283065796, 1.4816551208496094, 1.1540838479995728, 2.3836851119995117, -0.8663408756256104, -0.16575869917869568, 0.30282050371170044, 0.3433540463447571, -0.4931546449661255, -1.0021039247512817, 2.125243902206421, -0.7764472961425781, -0.48759037256240845, -0.5912545323371887, 0.07996746152639389, 0.4352589547634125, -0.0061385901644825935, -1.6433535814285278, -0.10416553169488907, -1.57552170753479, -0.13736554980278015, 0.6173902153968811, 0.6862581968307495, -0.541854202747345, 0.25818440318107605, 1.5383635759353638, 0.06953884661197662, 0.06626737117767334, 0.07485004514455795, 1.001417875289917, -0.9202920198440552, -0.8476320505142212, -0.6222934126853943, -2.2732136249542236, -0.08162587136030197, -0.5136752128601074, -0.9570178389549255, -0.10437989979982376, -1.0733243227005005, 0.346347451210022, 0.5189645886421204, 0.25676608085632324, -0.3970352113246918, 0.26985278725624084, -2.262482166290283, 0.6595216393470764, 1.085853934288025, 0.4831591844558716, -0.4487384259700775, -0.3402464687824249, -1.0073546171188354, -1.73685884475708, -1.4161040782928467, -1.3860180377960205, 0.7805432677268982, -0.5772499442100525, 0.4994567036628723, 2.1583197116851807, 0.7871970534324646, -0.8839773535728455, 1.5182245969772339, 1.9279253482818604, 1.3265480995178223, -1.288212537765503, -0.5755071043968201, -0.14827264845371246, -0.5122190713882446, -1.9224437475204468, -1.1094176769256592, -0.3127495050430298, -0.41624465584754944, -1.2600853443145752, 0.22252683341503143, 0.46433326601982117, -0.5440700054168701, -1.474749207496643, 0.5963460206985474, 1.1132680177688599, 0.01985604129731655, -1.2476394176483154, 0.6744250655174255, -0.6652292013168335, -1.533136248588562, 0.585959255695343, 0.6522770524024963, -0.4015025496482849, -0.5057985186576843, 0.18571171164512634, -1.5343093872070312, -2.1861865520477295, 2.5471785068511963, 1.9504038095474243, 0.23562011122703552, -0.16777516901493073, -0.19284741580486298, 1.762387752532959, 0.34197258949279785, -1.425819993019104, 0.8879282474517822, -0.3851664364337921, 1.30166494846344, -1.2135910987854004, -0.844142496585846, -0.38176462054252625, 0.2152215987443924, -0.4284273684024811, 0.8109439015388489, -0.7769185304641724, -0.8991401195526123, 0.43591612577438354, -0.8044221997261047, -0.3095279633998871, -0.47830891609191895, 1.2181499004364014, -1.4983851909637451, 0.9316478371620178, 0.16024525463581085, 0.4887852370738983, 0.1133519634604454, -0.48282119631767273, -0.682677149772644, 2.313830852508545, 0.5574156641960144, 1.8143551349639893, -0.47764459252357483, 0.7197860479354858, -0.5345192551612854, 1.984629511833191, -0.8932390809059143, 0.7137729525566101, -0.0799839049577713, -0.7076296806335449, -1.149562120437622, 1.052841305732727, 0.5206415057182312, 0.3394586443901062, -1.7129026651382446, 1.2471237182617188, -1.3728634119033813, -0.07539716362953186, 0.38116031885147095, -0.5527113676071167, -0.3774353265762329, 0.39417964220046997, 0.4598199427127838, 0.411529541015625, -0.9119381308555603, 0.4838345944881439, -0.5351724028587341, -0.4419355094432831, 1.0614866018295288, -1.1691036224365234, 0.11792387813329697, 0.6650087237358093, -1.23301362991333, 0.1156764030456543, 0.20446114242076874, 0.6702035069465637, -1.0641582012176514, 0.32852014899253845, -0.5434412956237793, 0.44435861706733704, -0.5632609724998474, -1.1012835502624512, -1.6459288597106934, -0.357857882976532, -1.1235119104385376, -0.5148912668228149, -0.7282727360725403, -0.32768386602401733, -1.5608962774276733, 1.7513606548309326, 0.037916213274002075, -0.22066785395145416, 0.21110789477825165, -1.001468300819397, -2.127969264984131, -0.5315389037132263, 0.0038334724958986044, 1.3118603229522705, 0.2681829631328583, 0.4317314028739929, 0.30642691254615784, -1.310497522354126, -0.30082955956459045, 0.16168157756328583, -1.432583212852478, -1.3719631433486938, 0.8088291883468628, -0.1839418113231659, 1.6480424404144287, -1.9896374940872192, -0.45618778467178345, 1.5820655822753906, -1.518660068511963, 2.0185298919677734, -2.063615083694458, 0.7407494187355042, -0.5888592004776001, -0.07480252534151077, 0.6984665393829346, 1.5017423629760742, 0.6012908816337585, -1.6662274599075317, 0.9049509763717651, 0.7776637077331543, -1.9801965951919556, -0.5997837781906128, -0.30523982644081116, -0.7952449321746826, -0.916428804397583, -0.7080882787704468, -0.3509313464164734, 1.1175851821899414, 1.0228735208511353, 0.20492757856845856, -1.2133289575576782, 0.5576204061508179, 1.399056077003479, -1.0159950256347656, 0.6266438961029053, 0.04233215004205704, 0.2532992362976074, 0.07512091100215912, -0.2728841304779053, 0.08878093212842941, 1.2246540784835815, -2.082913398742676, -0.37669995427131653, -1.8867144584655762, 0.5302185416221619, -1.40848970413208, -1.3856655359268188, -1.8712645769119263, 0.3707906901836395, -0.9521830081939697, 1.0733721256256104, 0.12258081883192062, -0.827267587184906, 0.7044947147369385, -0.4078027307987213, -1.6366429328918457, -1.1477689743041992, 0.7517736554145813, 1.5913385152816772, 1.4023656845092773, 1.380194067955017, 0.6902856826782227, -0.2198229879140854, 0.6863625645637512, 0.2836875021457672, -1.2333110570907593, 0.8280522227287292, 0.6645877361297607, -0.569318950176239, -0.16952428221702576, 0.2746182680130005, 0.4517695903778076, 0.7491005659103394, -1.7064545154571533, -1.18833327293396, -1.9790352582931519, -0.5410916805267334, -0.24737760424613953, 0.9158933758735657, -1.2569552659988403, -0.5740251541137695, 1.0047128200531006, -0.37294507026672363, -2.0947139263153076, 0.6305652856826782, 0.2760225832462311, -0.8333868384361267, 0.9272864460945129, -1.3495124578475952, -0.7322567105293274, -1.729121446609497, 1.1845762729644775, -0.7984795570373535, 0.5513758063316345, 0.7005895376205444, -1.5814769268035889, -0.9372328519821167, 0.9508477449417114, 0.6767721772193909, 0.055987000465393066, 0.33081167936325073, -0.33164530992507935, -0.9921431541442871, -0.5637383460998535, -1.6654409170150757, 0.9403198957443237, 0.333991140127182, -1.6255193948745728, 0.0775260403752327, 0.6650009155273438, -0.2234899252653122, -0.7753109931945801, 0.5854605436325073, -0.7561152577400208, -0.3842206597328186, -0.15761485695838928, 0.7405943870544434, 0.5768728256225586, -0.8713657855987549, 0.4039919376373291, 0.6278235912322998, 0.1235550120472908, 1.301979660987854, -0.8904434442520142, -0.5406908392906189, -2.1386051177978516, 0.7383795380592346, 1.3214253187179565, 0.8252679109573364, 1.347906470298767, -1.2886693477630615, -1.4644535779953003, 1.6749191284179688, -0.08080471307039261, 0.016047105193138123, 1.470406174659729, 0.6566489934921265, -0.39873582124710083, 0.7987450957298279, 0.16355997323989868, 0.4304186999797821, -0.6408706307411194, -0.22526314854621887, -1.5425539016723633, -0.03574991598725319, -0.9914643168449402, -0.645009458065033, 0.6118648052215576, -0.14891111850738525, -1.4494683742523193, 0.7129093408584595, 1.1861460208892822, -0.22989794611930847, -0.4039767384529114, 0.9134353995323181, -2.6049880981445312, 0.35592877864837646, 0.44236406683921814, -0.4415293335914612, -1.6240134239196777, -1.14061439037323, -0.6558944582939148, 0.6674396395683289, -0.5526235103607178, -0.5029727220535278, -0.8099289536476135, 0.22712628543376923, 0.2871827483177185, 1.4468966722488403, 0.7564820051193237, 1.1485012769699097, 0.11103421449661255, -0.5038869380950928, -0.3583259582519531, 1.2295794486999512, -0.2706024944782257, 0.16178381443023682, -2.089155673980713, -0.15439645946025848, -0.24683430790901184, -0.0520038977265358, -0.39709046483039856, 0.377713680267334, 0.626543402671814, -0.9681905508041382, 0.30832257866859436, -1.0101743936538696, 0.70819491147995, -0.8278593420982361, 0.5186185240745544, 1.0499902963638306, -1.6932365894317627, 0.05860590562224388, 0.8793731331825256, 1.4684263467788696, -0.3265099823474884, -1.1294934749603271, 0.9877879619598389, -0.7190772294998169, 0.4698193669319153, 1.6605210304260254, -0.09689641743898392, 0.0545320026576519, 1.8936959505081177, -0.5821921229362488, 0.8936070799827576, -0.7213621139526367, 0.6937404274940491, 0.030308397486805916, 0.9231522679328918, -1.1093721389770508, -0.10585028678178787, -2.2628116607666016, -0.08885639905929565, -1.389518141746521, 0.99936842918396, -0.6773983240127563, -0.721038818359375, -0.37219610810279846, -0.7428561449050903, -1.5388119220733643, -1.301413655281067, 0.7580915093421936, -0.9642396569252014, -0.6848717331886292, -0.21455930173397064, -1.1417652368545532, 1.2018015384674072, -0.7548782825469971, -0.30243048071861267, 0.2742123603820801, -1.0399034023284912, 0.03631305694580078, -1.067765235900879, -0.1416793018579483, -0.9454846382141113, 1.1338205337524414, -0.5673004984855652, 0.399139940738678, -0.9203020334243774, -0.5628319978713989, -0.22866487503051758, 0.26143136620521545, -0.5843280553817749, 0.6509676575660706, 0.5768477916717529, -1.6434860229492188, -0.26574385166168213, 0.11267825961112976, -0.04873977601528168, 0.23844388127326965, -0.4318089187145233, 0.38278067111968994, 0.1609736979007721, -0.5518737435340881, 0.2511972486972809, -1.0831481218338013, -0.9040471911430359, -0.45490649342536926, 0.4506140649318695, -0.017483720555901527, -0.8754922747612, 0.01515339594334364, 0.26052820682525635, -0.7689399123191833, 0.27479270100593567, -0.28818681836128235, -0.2049221396446228, 0.021716821938753128, -0.08118942379951477, 1.8426662683486938, 1.1464645862579346, 0.4363633096218109, 0.6091923117637634, -0.5505589842796326, 0.8355196714401245, -2.749537229537964, 1.8085120916366577, -0.09447002410888672, -0.29434120655059814, -0.21112887561321259, 2.0942742824554443, 0.985967755317688, 0.3609136641025543, -0.08093886822462082, 0.25015169382095337, -2.584667682647705, -1.1180070638656616, 0.511635422706604, 1.0178974866867065, -0.267069935798645, -1.9728487730026245, -0.7261404395103455, -0.26030433177948, 1.3063547611236572, -0.48662176728248596, 1.049074649810791, 1.1335841417312622, 0.6221051216125488, -1.8343017101287842, -1.0416240692138672, -0.9878902435302734, -0.7034555077552795, -0.7113264203071594, -0.7565749883651733, -0.0635308250784874, 0.8750304579734802, -0.3768450617790222, -0.11744064092636108, -0.27183768153190613, -0.22171087563037872, -0.4768438935279846, 0.135537788271904, 0.16598212718963623, 0.27197617292404175, 0.7590461373329163, -1.4657855033874512, -0.7483106851577759, 0.9833104014396667, 0.21554817259311676, 0.8063656687736511, 1.1933622360229492, -0.3519095182418823, 0.4621976912021637, 0.22921279072761536], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[1.5585259199142456, -4.0755414962768555, -2.8074169158935547, -0.12821677327156067, -0.19301050901412964, 1.5864826440811157, 1.7795709371566772, 0.7775754928588867, -2.398742914199829, 0.288548082113266, 1.6160286664962769, 0.914383053779602, 0.8559848666191101, -1.7466378211975098, -0.24409383535385132, 1.7788431644439697, 2.31394100189209, 0.24004265666007996, 0.14880910515785217, 2.0170326232910156, 3.2687771320343018, 0.2428613156080246, -1.2484484910964966, -0.2452963888645172, -1.9707832336425781, 0.9780674576759338, 0.6437608003616333, 0.3578278720378876, 0.6226676106452942, 1.3099677562713623, 0.8343247175216675, 1.1586534976959229, -0.7832475304603577, -0.3722454309463501, -0.051196690648794174, -2.3922181129455566, 1.7978225946426392, 0.6816673278808594, -0.796285092830658, 1.0471649169921875, 1.1888058185577393, 0.15677304565906525, -1.2638827562332153, 3.2995688915252686, 0.09141243249177933, 1.3135154247283936, -0.4941425621509552, 1.1165497303009033, -1.6793887615203857, 1.0303955078125, -0.5070023536682129, -0.43067359924316406, 0.3323349356651306, -0.9058718085289001, 3.0755279064178467, 1.1085834503173828, 2.1525745391845703, 0.31124964356422424, 2.2893385887145996, -0.03419205918908119, 0.9393380284309387, 1.6251933574676514, -5.066985607147217, -0.17275384068489075, 1.5001393556594849, -1.7334575653076172, -0.5044313669204712, 0.35961422324180603, 1.040139079093933, 2.5376737117767334, 0.3844705820083618, -0.5233268737792969, 0.09265898913145065, 0.47245484590530396, 3.960775852203369, -3.7745935916900635, 2.9675068855285645, -0.9196683764457703, 1.022834300994873, 1.6875944137573242, 1.1564610004425049, -0.31454190611839294, 0.17108787596225739, 0.8562646508216858, -3.3496155738830566, 1.058566927909851, 0.637290894985199, -0.5988978743553162, 0.3352004289627075, -0.37940144538879395, 0.2578613758087158, 0.3850887715816498, 0.9974228143692017, 0.5897067785263062, -2.1364831924438477, -0.5597485899925232, 0.20830143988132477, 3.4237921237945557, -2.045487880706787, -3.044053077697754, -0.15734638273715973, -2.188783884048462, 1.161496877670288, 0.16748906672000885, 1.1189881563186646, -0.8691815733909607, -0.22987902164459229, -0.9364978075027466, 0.3404276967048645, -0.7394358515739441, 1.751068115234375, 0.4670478403568268, -3.120607852935791, -1.1745673418045044, 0.47740286588668823, -0.5566840767860413, 0.6814199686050415, 1.1437245607376099, 0.8278077840805054, 0.20634911954402924, 1.5710303783416748, 0.7369188666343689, -1.0153930187225342, -1.3842836618423462, -2.8357067108154297, -0.589961051940918, -0.4248340427875519, 0.6327959895133972, 0.8711836934089661, -1.8394023180007935, -0.6907650828361511, 1.1058605909347534, 1.369928240776062, 0.9138928055763245, -0.47733384370803833, 0.26584047079086304, -0.107374407351017, -0.8081015348434448, 0.5262959599494934, 0.3528103232383728, -0.7503016591072083, -0.027346519753336906, -1.764028787612915, -0.22733677923679352, 0.6640645861625671, 0.11100346595048904, 0.6598178744316101, 0.4321151375770569, 0.2731424570083618, -0.016729792580008507, -0.13696534931659698, -0.8888068795204163, -0.8490225076675415, 0.24541610479354858, -0.8327255249023438, 1.175008773803711, -1.3391039371490479, -2.3819918632507324, 0.00046146451495587826, 2.1338765621185303, 0.5543642044067383, -1.446539044380188, -2.579432964324951, -1.0280711650848389, 1.053330898284912, 2.1300771236419678, 1.0284639596939087, 1.663022756576538, 1.449195384979248, 0.7369943857192993, -1.2533050775527954, 1.308110237121582, 0.31192687153816223, -0.5246618986129761, -2.0275893211364746, 2.0108642578125, -1.5591062307357788, -0.37003159523010254, 0.3353131413459778, 0.22446779906749725, -1.3757089376449585, -2.0810089111328125, 0.8069567084312439, 2.661456823348999, -5.328468322753906, -0.9976896047592163, -0.5609962344169617, -0.9890283346176147, -0.06937558203935623, 0.499859094619751, -0.6208421587944031, -0.9855433702468872, 0.4207623600959778, 2.2579171657562256, -1.2731294631958008, 0.5984106063842773, 1.5176076889038086, 1.2432303428649902, 0.3482672870159149, 1.3503749370574951, -2.277462959289551, -1.3888543844223022, 2.1409103870391846, -1.8869519233703613, 0.10033663362264633, -0.31953901052474976, -0.6764914393424988, -0.7936362624168396, -0.2618792951107025, 0.24827642738819122, 1.3304877281188965, 2.366499185562134, 2.3380372524261475, 0.7226631045341492, 1.5103626251220703, 0.2950533628463745, 0.42796385288238525, -1.0516080856323242, 0.027585696429014206, 0.8828848004341125, 1.1143044233322144, 0.0007020674529485404, 1.466004729270935, -0.21602259576320648, 0.8776171207427979, 0.1493210643529892, 1.09801185131073, -0.2119760662317276, 0.10275886952877045, 0.5806134939193726, 0.4679619073867798, -1.3475301265716553, 0.056409530341625214, 0.2831811010837555, 0.7835015058517456, 1.6922825574874878, 1.0773898363113403, 0.9727115631103516, 1.7982627153396606, 1.2891229391098022, 0.3370717465877533, -0.15382994711399078, -0.9382023215293884, -0.2221350520849228, -1.3761614561080933, -0.43036144971847534, -1.2788931131362915, -1.7638074159622192, 1.793679118156433, -0.6989452838897705, 1.2290390729904175, -1.0796133279800415, -0.5994709134101868, 0.37893354892730713, -1.2362353801727295, 0.5005620718002319, 0.7059781551361084, 0.5553252100944519, 0.2798110246658325, 0.07848060131072998, 1.9110815525054932, 0.15645921230316162, -0.0492590107023716, 0.5574168562889099, -1.1689345836639404, -1.5290268659591675, -0.06507963687181473, 0.8070684671401978, 1.419865369796753, 0.5831509828567505, -0.8093104362487793, 0.28607550263404846, -2.290337324142456, -0.5241097807884216, 0.7475042343139648, 1.4752620458602905, -0.36531293392181396, -2.0260956287384033, -1.4906890392303467, -0.7266519069671631, -1.3092081546783447, 0.24213838577270508, -2.692091226577759, -0.5085051655769348, -2.57792592048645, -0.8129229545593262, -0.8643650412559509, -3.706015110015869, -1.1597932577133179, 2.0621278285980225, 2.2287278175354004, -1.9566707611083984, -0.6081153154373169, -1.3510242700576782, -2.5593197345733643, 0.25573885440826416, 1.3676244020462036, 0.02728460356593132, 3.1016762256622314, 1.7756932973861694, 3.446136951446533, 0.6374745965003967, -1.1805508136749268, 1.3503007888793945, -0.04880475252866745, -1.4126676321029663, -0.24457263946533203, -2.4092187881469727, 0.21191075444221497, -1.4622628688812256, 0.6991733312606812, -0.6155481338500977, -0.8057699203491211, 0.7777270674705505, -0.6817638278007507, 1.571608304977417, 2.951249122619629, 1.2234034538269043, 1.2669754028320312, 0.9815111756324768, -0.25581955909729004, 1.0200525522232056, -0.3395172357559204, 1.4381043910980225, -1.394171118736267, 0.14392244815826416, -0.927592933177948, -1.1722160577774048, 0.17949940264225006, 0.5531256198883057, -0.7851478457450867, 0.9994575381278992, 0.9120502471923828, -0.3109038770198822, 0.5561674237251282, -3.258573293685913, 0.24938198924064636, -0.39248645305633545, -0.426779568195343, -1.7369071245193481, 0.8340153098106384, -0.7647562026977539, -4.794384479522705, 0.7352832555770874, -0.6719644665718079, -1.655481219291687, -1.4797650575637817, 1.6583638191223145, -1.322663426399231, -0.713019847869873, 1.0212348699569702, -1.7108418941497803, -2.8395652770996094, -1.155432105064392, -0.2313500940799713, -2.2929837703704834, -1.4710562229156494, 0.6798193454742432, 1.3278019428253174, 2.698495864868164, 0.06857474893331528, 1.102376937866211, 1.0582419633865356, -1.5987350940704346, 0.7595707774162292, 0.2054479867219925, -1.6691973209381104, 0.7592405080795288, -0.571673572063446, -1.2822073698043823, 0.8803529143333435, 0.8619830012321472, 2.082460880279541, -0.575709342956543, -1.5480053424835205, 1.172590732574463, 1.3612380027770996, 0.29257693886756897, -0.6337018609046936, 1.4596213102340698, 0.157966747879982, -3.310281753540039, 0.14658784866333008, -0.674113929271698, 0.6420766115188599, 2.6954057216644287, 1.381251573562622, -1.3256858587265015, -3.6875247955322266, 1.1682928800582886, 0.7384335994720459, -3.0990242958068848, 1.7029415369033813, 1.6953710317611694, 1.0313267707824707, 1.0377380847930908, -1.466934323310852, 1.0035467147827148, -0.8845881223678589, -1.4929684400558472, -1.7433782815933228, 0.40153786540031433, 0.11852351576089859, -0.13589535653591156, -0.8801578879356384, 0.8908498287200928, 1.1220521926879883, 0.8628507852554321, -1.0266278982162476, 0.03713157773017883, -0.5556484460830688, 0.6333369016647339, -0.3795841932296753, -1.0776128768920898, -1.468603491783142, 0.15269134938716888, -0.027699828147888184, -1.5529136657714844, 0.6334866285324097, -2.0584466457366943, 1.1218711137771606, -1.5279936790466309, 0.5580422878265381, -0.5241726040840149, -0.08852894604206085, -0.4646782875061035, 1.6861088275909424, -2.1669790744781494, 1.3591060638427734, 0.436469703912735, 0.9785136580467224, -1.460508108139038, -0.2999681234359741, 0.59488844871521, -0.4769038259983063, -1.1992634534835815, 0.3587482273578644, 1.9820339679718018, 0.3664379417896271, -2.7325782775878906, -1.8948289155960083, -0.3021906316280365, 0.020175974816083908, 0.16433846950531006, 1.4245818853378296, -0.3469628095626831, -0.6781725883483887, -1.5482417345046997, -0.6003057360649109, -0.4432973265647888, 0.3973027765750885, -1.7223378419876099, 0.007291059475392103, 1.6686789989471436, -0.9561286568641663, -1.2748335599899292, 0.14522646367549896, 0.09352672100067139, 1.6468127965927124, 1.774180293083191, 1.2397732734680176, -1.884963035583496, 1.9942123889923096, -1.3586790561676025, 0.1535770297050476, -0.6554592847824097, 0.5027867555618286, 2.0675113201141357, -0.3906162977218628, 0.287760853767395, 0.22478079795837402, 1.5165932178497314, 1.0133908987045288, -2.1892361640930176, -0.16799220442771912, 0.6132340431213379, -1.7005677223205566, 0.49141669273376465, 0.03828706219792366, -2.755251407623291, 0.1558154821395874, -0.21227771043777466, 1.5036630630493164, 1.8139821290969849, 0.8345715999603271, -2.847891092300415, -0.11755755543708801, -1.3219499588012695, 0.052335452288389206, 0.24425221979618073, 2.666651725769043, 3.5385823249816895, 1.6595251560211182, -1.7292667627334595, 0.04845438897609711, -0.24544797837734222, 0.1554623693227768, -0.8921785950660706, -0.7686737775802612, 2.2522976398468018, -1.0075671672821045, -1.2678272724151611, -0.7437136173248291, -0.14902089536190033, 0.7760899066925049, -0.6346492171287537, -1.9601918458938599, -0.1619710624217987, -1.9233253002166748, -0.9644960761070251, 1.1460318565368652, 0.7361336946487427, -0.8594236373901367, 0.3229365348815918, 5.194480895996094, -0.06783201545476913, 0.17547237873077393, -0.8715419769287109, 0.9570978283882141, -0.9184993505477905, -0.6040799021720886, -0.3675310015678406, -2.3884377479553223, -0.20649512112140656, -1.2059563398361206, -2.680523633956909, -0.2851129174232483, -0.3627380132675171, -0.954516589641571, 1.0152614116668701, 0.47320452332496643, 0.26308783888816833, 0.27773264050483704, -2.017242431640625, 0.33497756719589233, 1.1698271036148071, 0.4142167270183563, -0.1487223505973816, -0.7468743920326233, 0.32161983847618103, -0.1651473492383957, -4.854145526885986, -2.2425012588500977, -0.9853185415267944, 0.6897158622741699, 0.9689244031906128, 2.664111852645874, 5.433553218841553, -0.2519221603870392, -0.17621980607509613, -0.09784311801195145, 1.0027672052383423, -1.168319582939148, 2.8913938999176025, -0.9977203011512756, -0.45996975898742676, -3.9223406314849854, -1.0386461019515991, -0.4576312005519867, -0.33029845356941223, -1.1182817220687866, 0.939233660697937, 0.3777526021003723, -0.33064600825309753, -1.494500756263733, 0.5962318181991577, 1.4800796508789062, 0.3911144435405731, -0.6071934103965759, 0.7743360996246338, 1.1504360437393188, -2.549739360809326, 1.4834891557693481, -0.17571276426315308, -1.0337862968444824, -0.5268300771713257, 0.39361870288848877, -1.4166581630706787, -1.880977988243103, 2.303229808807373, 2.616725444793701, 0.4073302149772644, 0.17808517813682556, 0.5944888591766357, 2.353043556213379, 0.2614600956439972, -1.342309594154358, 0.6469639539718628, -0.15870435535907745, 1.0960776805877686, -0.5777225494384766, -0.16531413793563843, -0.7012090086936951, 0.7004269361495972, 0.18902069330215454, 0.7511414289474487, -0.7872970104217529, -0.5045196413993835, 0.6057788729667664, -1.2695066928863525, 0.45075732469558716, -0.4996210038661957, -0.10698246955871582, -1.1189360618591309, 0.7871678471565247, 0.2719815671443939, 1.322142243385315, 0.12319356203079224, 0.36523905396461487, -0.8774096369743347, 1.461246371269226, 0.30539509654045105, 2.436732530593872, -0.4853920042514801, 0.254509836435318, -0.4993370473384857, 1.9371672868728638, -0.5290548801422119, 0.7217000126838684, 0.2611834406852722, 0.6789299845695496, -0.7087789177894592, 1.1952813863754272, -0.3424527645111084, -0.48051726818084717, -2.3424365520477295, 1.5654809474945068, -1.4618983268737793, 1.1595269441604614, -0.8176853060722351, -1.386426568031311, 1.2896058559417725, 2.5409910678863525, 0.27182847261428833, 0.6078241467475891, -1.471962809562683, 0.17578068375587463, -1.3506324291229248, -0.41589900851249695, 1.9180238246917725, -1.1173934936523438, -0.43629777431488037, 0.7352424263954163, -1.3007023334503174, -0.18638069927692413, 0.3567686080932617, -0.6277171969413757, -1.0261143445968628, 1.8577353954315186, -0.6278603672981262, 0.24465562403202057, 0.35624513030052185, -2.4485740661621094, -1.4887940883636475, 0.5730653405189514, -1.7325634956359863, 0.30840444564819336, -0.6247585415840149, -0.38286474347114563, -1.8346253633499146, 2.319462537765503, 1.2590100765228271, 0.6216166019439697, -2.6886961460113525, -1.2454005479812622, -2.0382564067840576, -0.3821636736392975, 1.2649574279785156, 0.6258615255355835, 0.2719106078147888, 0.532911479473114, 0.13697226345539093, -1.055430293083191, -0.7129353880882263, 0.25354498624801636, -2.244377613067627, -1.5990196466445923, 1.1958165168762207, -0.18660278618335724, 0.8684954643249512, -2.18637752532959, -0.8991481065750122, 1.190258264541626, -1.5014386177062988, 1.365029215812683, 2.0934879779815674, 0.49869504570961, 1.0018905401229858, -1.4042768478393555, 1.7600648403167725, 1.281569480895996, 0.7885804176330566, -2.260305166244507, 0.8486821055412292, 1.6349436044692993, -2.2113821506500244, -0.24876749515533447, -1.6138122081756592, -0.7914597392082214, -1.0180387496948242, -0.9783070087432861, -1.0358643531799316, 1.1101789474487305, 0.7909640073776245, -0.501338005065918, -1.0321424007415771, 0.5606527328491211, 1.205272912979126, -2.018144369125366, 0.8383823037147522, 0.6379430294036865, 0.23174326121807098, 0.4109039902687073, -0.43606796860694885, 3.7346086502075195, 1.2332663536071777, -2.36381459236145, -0.5603082776069641, -2.1654953956604004, 0.46408170461654663, -1.0785644054412842, -1.116240382194519, -1.011752963066101, -0.19894947111606598, -0.9725404381752014, 1.434352993965149, -0.42927324771881104, -0.8252173066139221, 0.9411464929580688, -0.2913353741168976, -1.3917893171310425, -0.8778352737426758, 0.5594973564147949, -2.096635580062866, 0.7720998525619507, 1.315043568611145, 0.5006851553916931, -0.2344299852848053, 2.185506582260132, 0.8056519031524658, -1.107966423034668, 1.332870364189148, 0.5560342073440552, -1.3862961530685425, 0.0183132141828537, 1.0341531038284302, 0.503212034702301, -0.4421202838420868, -2.2352752685546875, -1.16024649143219, -2.203014612197876, -0.4755317270755768, 0.41937604546546936, 2.4613142013549805, 0.5718400478363037, -1.2636176347732544, 1.1405576467514038, 0.08165029436349869, -2.8506102561950684, 3.1951277256011963, 0.9683661460876465, -0.4893687069416046, 1.2746095657348633, -0.4163346588611603, -1.7954736948013306, -1.1866778135299683, 1.1652723550796509, -0.07181009650230408, -0.2291591912508011, 0.5756475329399109, -1.7745521068572998, -0.7965357303619385, 1.1713969707489014, 1.248329997062683, 0.7321270704269409, 1.7435511350631714, -1.4803763628005981, -0.05288752540946007, -0.6158685088157654, -2.081920623779297, 0.8728819489479065, 1.1089820861816406, -0.0014930712059140205, 0.16555212438106537, 1.2466672658920288, -0.21887341141700745, -0.584570050239563, 0.49849382042884827, -0.6713774800300598, -0.1646677702665329, -0.2202359437942505, 1.0809502601623535, -1.5628111362457275, 0.19353926181793213, 0.453368604183197, 1.065327525138855, 0.1283465176820755, 1.9838634729385376, 0.02260720357298851, -0.47418567538261414, -2.4505488872528076, -0.4476400911808014, 1.054731845855713, 1.2107274532318115, 0.9914653301239014, -1.4655907154083252, -1.5812952518463135, 3.2337443828582764, -0.006853072438389063, -1.809226155281067, 1.6622072458267212, 2.1108627319335938, -0.4777907133102417, 0.514988899230957, 0.7710548043251038, 1.0449045896530151, -1.7274978160858154, -0.09577315300703049, -2.512699842453003, -0.21050666272640228, 0.5300005078315735, -0.4677231013774872, -0.1047629714012146, 0.16650336980819702, -2.3107388019561768, 1.3554706573486328, 1.4460198879241943, -1.7522895336151123, -0.8570007681846619, 0.9127671122550964, -1.8459031581878662, 0.30524495244026184, 3.196248769760132, -0.3690956234931946, -2.2281394004821777, 2.8020377159118652, -1.2414700984954834, 1.0009418725967407, -0.3231067359447479, -0.42786067724227905, -1.2659533023834229, 2.287973165512085, 1.6407980918884277, 1.708216667175293, 0.5509079098701477, 1.4668943881988525, 0.9044226408004761, 0.27955642342567444, -1.4525474309921265, 2.6496193408966064, 0.1432022750377655, 0.917212188243866, -2.4675369262695312, -0.06372391432523727, -2.1074342727661133, 0.4439651072025299, -0.34352678060531616, 2.5987982749938965, 0.7265205979347229, -0.6176229119300842, 0.6645637154579163, -0.38475194573402405, 1.002296805381775, -0.5382779240608215, 0.8192819952964783, 0.10565672814846039, -1.4504568576812744, 1.5727152824401855, 1.4992303848266602, 1.1684612035751343, -2.958940029144287, -1.1277295351028442, -4.590264797210693, -1.1980013847351074, -2.1512060165405273, 1.375736117362976, 0.666977047920227, 0.003968593664467335, -1.3775744438171387, -0.2234499156475067, -1.5839314460754395, -0.15357865393161774, 1.3366400003433228, -0.20307275652885437, 1.1202421188354492, -0.8278173208236694, 0.7567745447158813, -1.9744280576705933, -1.3780394792556763, -0.22650359570980072, 1.3582847118377686, -0.46022382378578186, -1.121380090713501, -1.2553035020828247, -0.6083690524101257, -2.642012119293213, -1.460069179534912, 0.9830754399299622, -2.155795097351074, -3.0251927375793457, -0.30024248361587524, -0.6794764399528503, 1.794990062713623, -1.176213264465332, 0.11720538139343262, -0.3690820038318634, 0.11258216947317123, -0.3883116841316223, -0.3662997782230377, 2.3632454872131348, -0.5842590928077698, 1.2033700942993164, -1.8788503408432007, 0.5147958993911743, -0.6651339530944824, -0.1247631162405014, -1.0275555849075317, 0.23834264278411865, 0.8689166307449341, 0.5672920942306519, 2.712083578109741, -1.3914235830307007, -0.1504572182893753, -0.1476876586675644, 0.09977059066295624, -0.8229022026062012, -0.9148058295249939, 0.18039266765117645, -0.24879056215286255, -0.22863753139972687, 0.24316981434822083, -1.9334670305252075, -1.9082326889038086, -0.19574172794818878, 0.44384899735450745, 0.24692344665527344, 0.6611258387565613, 2.4299933910369873, 0.11333462595939636, -0.5067301988601685, 0.41072726249694824, -0.3023855686187744, -0.7311107516288757, 0.05168427899479866, -0.025218158960342407, 3.6971280574798584, 0.8154031038284302, 0.4610629975795746, 0.12050407379865646, -0.38331857323646545, -0.5809175968170166, -3.686915397644043, 2.810107946395874, 1.1923526525497437, -0.3088432252407074, -0.23955948650836945, 2.6662440299987793, -1.494113564491272, -0.8609623312950134, -0.5071540474891663, 0.592542290687561, -1.5184803009033203, -0.29111331701278687, 0.7030162215232849, 1.2764129638671875, -1.5687401294708252, 0.018281275406479836, -0.7154059410095215, 0.35579735040664673, 2.5657575130462646, -0.4385329782962799, 0.9209109544754028, 2.712311029434204, -0.13818614184856415, -1.8205654621124268, -1.4952901601791382, -0.1805659979581833, -1.2279495000839233, -0.340681254863739, -0.8391406536102295, -2.4578611850738525, 0.6940685510635376, 0.42444801330566406, -0.11954382061958313, -1.0221692323684692, -1.6858986616134644, -0.8268586993217468, -0.9876234531402588, -1.7093147039413452, 0.1548789143562317, 0.20523691177368164, -4.107718467712402, -0.08336948603391647, 1.1652709245681763, 0.3003388047218323, 0.9684962034225464, -0.05212613195180893, -0.5617097020149231, 0.2806359529495239, 0.5000607371330261], [1.8002625703811646, -2.1976258754730225, -2.7618186473846436, 0.4171070456504822, 0.6862910985946655, 1.4156010150909424, 2.507768154144287, -0.15485569834709167, -1.425316333770752, 0.6575227975845337, -1.8625879287719727, 0.004999796859920025, 1.6769142150878906, -1.7423720359802246, -0.3742450773715973, 1.1554604768753052, 2.0993247032165527, -1.5601357221603394, -1.73631751537323, 1.264847755432129, 1.144949197769165, 0.008229351602494717, -0.7718831300735474, 1.4683231115341187, 2.5826590061187744, 0.1349347084760666, -0.848019540309906, 0.2770552337169647, 1.6091681718826294, 1.2310594320297241, 0.9719737768173218, -0.04560904949903488, -0.25977882742881775, -0.6103240251541138, 0.08572512865066528, -0.5426583290100098, 1.3535901308059692, 1.6242653131484985, -0.7234618067741394, 0.8433294296264648, 2.6919965744018555, 0.5422318577766418, -0.8809210062026978, 2.9133973121643066, -3.8669121265411377, 1.6526625156402588, -0.5817145705223083, 1.5783098936080933, -1.4974040985107422, -0.5130615830421448, -0.26596325635910034, -1.898838996887207, 0.38212037086486816, -0.9882383346557617, 2.390568256378174, -0.5236884355545044, -0.7786260843276978, 2.3682920932769775, -0.003264850936830044, -0.13608482480049133, 0.9403010010719299, 1.4417022466659546, 6.197744369506836, 0.21844957768917084, 1.5952560901641846, 1.2405463457107544, -1.1542993783950806, 0.24823229014873505, 0.8755658268928528, 1.6245992183685303, 1.9878123998641968, -0.3820234537124634, 0.10208948701620102, 1.5478994846343994, 2.2191641330718994, -0.895730197429657, 1.2976881265640259, -0.34308189153671265, -1.1216998100280762, 1.6972029209136963, -1.2879042625427246, -1.2517260313034058, -0.9741721749305725, 1.171742558479309, -2.697481393814087, 0.5144121646881104, -1.1926690340042114, -0.8948113322257996, 0.3290325105190277, -1.0068070888519287, 1.2155468463897705, -0.043468575924634933, 3.2363882064819336, 0.28657376766204834, 1.1874531507492065, 0.5769076347351074, -0.41754212975502014, 0.4226891100406647, -1.3325300216674805, -3.2308242321014404, 0.08866804093122482, -2.093815565109253, 2.298011064529419, 0.9607518315315247, -3.2653086185455322, -0.4382950961589813, 0.19375458359718323, -0.9718641042709351, -0.005608417093753815, -1.0190523862838745, 0.8947869539260864, 0.42932260036468506, -3.507662773132324, 0.47527968883514404, -0.3828558027744293, 0.348832368850708, 2.445488929748535, 1.1446987390518188, 0.9952601194381714, 0.11876697838306427, 1.6716722249984741, 0.7302588224411011, -1.0244371891021729, -1.4994829893112183, 0.010096163488924503, -2.3781557083129883, 0.39881470799446106, 1.3501179218292236, 0.3777867257595062, 0.9897822141647339, -0.9389308094978333, 1.0775651931762695, 0.5606899857521057, -0.05348053574562073, -0.5982375144958496, 2.0077757835388184, -0.12707743048667908, -1.008972406387329, 0.31919169425964355, 0.4155927300453186, -0.6693922877311707, 0.6968844532966614, 0.016297420486807823, -0.7572787404060364, 0.5913833975791931, -0.3425561189651489, 1.7468334436416626, -0.38273072242736816, 1.396165370941162, -0.2589298188686371, -0.24963651597499847, -0.5262913107872009, -1.29864501953125, 0.3245973289012909, -1.9163341522216797, 0.7571339011192322, -0.2904607653617859, -2.1800780296325684, 2.0974528789520264, 2.3011300563812256, 3.5014255046844482, -1.6868666410446167, -2.197006940841675, -2.8154830932617188, 1.0751521587371826, 1.9401805400848389, 1.384297490119934, -0.5695523023605347, 0.14153288304805756, 0.5825628042221069, -0.6490594744682312, 1.3100143671035767, 0.7061001658439636, -0.5070840120315552, -1.37204909324646, 0.3473745584487915, -0.6206977963447571, -0.5064792037010193, 0.8046716451644897, 3.118931293487549, -0.25037524104118347, -0.0525699220597744, 1.266249179840088, 1.19194757938385, -2.2797513008117676, -0.6108353137969971, 0.22620469331741333, -1.5420628786087036, 0.23865026235580444, 0.31485477089881897, -1.646438717842102, -0.9847204089164734, -1.1470348834991455, -0.27501294016838074, -0.6130434274673462, 0.06555700302124023, 1.4974526166915894, -1.1106387376785278, -1.3565958738327026, -1.6126856803894043, -2.65925931930542, -4.013644695281982, 2.0459983348846436, -0.9744196534156799, 2.556248903274536, -0.37148261070251465, -0.7337793707847595, -1.0939873456954956, 0.11354522407054901, 0.5657297968864441, 1.88398277759552, -0.9129695892333984, 0.8198336362838745, 1.154380440711975, 1.569037914276123, 1.3162106275558472, 0.38885927200317383, -0.4419545531272888, -0.10600435733795166, 0.8198722004890442, 1.8270972967147827, -0.3387521207332611, 1.2514649629592896, 0.3998299837112427, -0.17285296320915222, -0.0011737573659047484, -0.1760999858379364, 1.056435227394104, 0.30556774139404297, 0.9757301807403564, -0.4189954698085785, -1.0419065952301025, -0.7310765385627747, 0.2569580674171448, 0.1491258293390274, 1.9862290620803833, 1.2770410776138306, 1.387919306755066, 1.8978345394134521, 1.1499381065368652, 0.14255280792713165, 2.117926597595215, 0.6948079466819763, 0.32705533504486084, -0.07858645915985107, 0.30594688653945923, -2.390834093093872, -1.2073310613632202, 1.1444743871688843, -1.8303253650665283, 0.8993824124336243, 0.12349254637956619, -0.23251408338546753, -1.866449236869812, 0.39749976992607117, 1.2246028184890747, 0.15685150027275085, -0.2934057414531708, -1.1452854871749878, 1.5227113962173462, 1.256200909614563, 5.510130405426025, -0.11760379374027252, 0.6189496517181396, 3.790919065475464, 0.9802491664886475, 0.17076902091503143, 0.39857712388038635, 2.088197946548462, 0.5626971125602722, -1.8611910343170166, 0.04383578523993492, 0.833095133304596, -0.19775965809822083, 0.42787593603134155, 1.8594164848327637, -2.0379371643066406, -2.1952927112579346, 0.5831804275512695, -0.5617448091506958, 1.4728554487228394, 0.31899479031562805, -1.8383182287216187, -0.23902228474617004, -0.4242779314517975, -0.2578466236591339, -0.6232027411460876, -1.6554341316223145, -1.1733646392822266, 2.345769166946411, 1.554747462272644, 0.5873356461524963, -0.583574652671814, -2.177891254425049, -1.3699333667755127, -1.364890694618225, 0.9249911904335022, -1.7323569059371948, 1.3321993350982666, -0.11448753625154495, 3.266801118850708, -0.11744450032711029, -1.8323055505752563, -1.1930783987045288, -0.9902194738388062, 0.3470538556575775, -0.10053327679634094, -1.0709043741226196, 1.7904778718948364, -0.47391173243522644, 1.1911957263946533, -0.0660804957151413, -2.1112568378448486, 0.8111186027526855, -1.6455498933792114, 3.249850034713745, 0.9719358682632446, 1.3473424911499023, 1.336225986480713, 0.9902569651603699, -0.37302252650260925, 0.6423633694648743, -0.25050023198127747, 1.7387250661849976, -1.0477920770645142, -0.8790984749794006, -1.4945482015609741, -0.009510847739875317, 0.01750796101987362, -0.4980425238609314, -1.1992714405059814, 0.7841051816940308, 1.2290385961532593, -0.5388543009757996, 0.8063951134681702, -3.197986125946045, 1.1540498733520508, -0.8785330057144165, 1.6669789552688599, -0.4205423891544342, 0.12705808877944946, -0.7069903612136841, -3.298022508621216, 0.6796731948852539, 0.07764682918787003, 1.536311388015747, -2.1949212551116943, 2.6202518939971924, -1.7156344652175903, -0.3814111053943634, 0.399916410446167, 0.5752246379852295, -2.210766553878784, -1.6444382667541504, -0.23460014164447784, 0.5496187806129456, -1.140411138534546, 0.7526366710662842, -0.2390252649784088, 1.8458787202835083, 1.8914060592651367, 1.621971607208252, 1.0621751546859741, -1.766865849494934, 0.393330454826355, 1.0879582166671753, -1.8278111219406128, 0.4708983600139618, -0.8269103765487671, -0.6982284188270569, 1.206365704536438, 1.0058919191360474, -0.8093559145927429, -0.05315021425485611, -0.22148025035858154, 1.1769435405731201, 0.8275563716888428, 0.17969495058059692, -0.1326925903558731, -1.3792648315429688, -0.5363047122955322, -1.3672630786895752, 0.558357298374176, -1.1859657764434814, -1.0888736248016357, 4.763508319854736, 1.710019588470459, -0.8443030714988708, -2.48689603805542, 1.6562892198562622, 0.2650430500507355, 1.7067956924438477, 1.9849878549575806, 1.0481462478637695, -0.08882568776607513, 0.5320277810096741, -1.2054497003555298, -8.407578468322754, 0.25583434104919434, -2.335139036178589, 0.6445958018302917, 0.07044702768325806, 0.87412428855896, -0.45555055141448975, -0.4836896061897278, 0.5802185535430908, 2.818164587020874, 1.0948772430419922, -0.694536566734314, 0.3828841745853424, -0.8146791458129883, -0.12852531671524048, -1.1172327995300293, 2.509246349334717, -0.48216870427131653, 0.04735969379544258, -0.458461731672287, -1.538631558418274, 0.08624067157506943, 0.3214350640773773, -0.7764585018157959, -0.7253078818321228, 1.0017529726028442, -0.15306338667869568, -0.210773766040802, -0.2265929877758026, 1.413893461227417, -2.521556854248047, 1.4356086254119873, -0.34441620111465454, 1.2204923629760742, -1.0423526763916016, -1.3702625036239624, -0.12282363325357437, -2.175499200820923, -0.8794988989830017, -0.40665164589881897, 2.045437812805176, 0.7560651898384094, -1.8593870401382446, -2.149440288543701, -0.5325290560722351, 0.024723250418901443, 0.1535869836807251, 2.9208695888519287, -0.3191989064216614, -0.4927752912044525, 0.3769804537296295, -0.6017913818359375, 0.14060436189174652, 0.4491988718509674, 0.36391475796699524, 1.354902982711792, -0.7005751132965088, -1.1059865951538086, -2.0805883407592773, 0.1825735867023468, 1.7802451848983765, -1.526110053062439, 0.6409144997596741, 0.8584702014923096, 0.7516107559204102, 5.0373430252075195, 3.2997934818267822, 0.24291284382343292, -0.5177563428878784, 0.45125359296798706, 2.4796905517578125, -1.7386895418167114, 0.6989736557006836, -0.16419315338134766, 1.8501789569854736, 1.0471686124801636, -0.7073542475700378, -0.5969149470329285, 0.6928786039352417, -0.9177734851837158, -1.5479509830474854, -0.030243562534451485, -0.9966549873352051, -0.04565086215734482, -1.2835347652435303, 0.5911931395530701, -1.251012921333313, 0.8318454027175903, -2.952258825302124, -0.07183481752872467, -1.5295606851577759, 0.07058903574943542, 0.3663419783115387, -1.303544521331787, -1.7456200122833252, 0.20724311470985413, -1.1390304565429688, -0.07007263600826263, 0.4458315074443817, -0.3759280741214752, 0.04735453799366951, -1.2619168758392334, 2.568171739578247, -1.0387872457504272, -1.779071569442749, -0.38640084862709045, 0.07649236172437668, 0.6880127787590027, 0.09381149709224701, -4.2174177169799805, -0.20956438779830933, -2.243896245956421, -0.27749761939048767, 0.6827148795127869, 0.5573755502700806, 0.21586674451828003, 0.203412726521492, 1.2419006824493408, 2.3219072818756104, -0.09863051772117615, 0.179875448346138, 1.6939760446548462, -0.6240912675857544, -1.1071183681488037, -0.013370679691433907, -2.154914617538452, -0.04227779060602188, -1.067845106124878, -2.2536916732788086, -0.21486108005046844, -1.3422423601150513, 0.6780775785446167, -0.07967483997344971, 0.6934645771980286, -0.3936883807182312, 0.3276090621948242, -2.535229444503784, 0.03706081956624985, 1.0351803302764893, 0.42460697889328003, -1.9635696411132812, 1.0015580654144287, -0.3242955505847931, -3.1555140018463135, 0.310707688331604, -1.0910762548446655, -0.05244648456573486, -0.6825239062309265, -0.41947001218795776, 2.468553304672241, 2.1525275707244873, -1.3062176704406738, -1.373116374015808, 1.2769705057144165, 2.054759979248047, -1.5637216567993164, -0.6983568072319031, 1.8824806213378906, -0.4137879014015198, -0.9347510933876038, -0.8144240379333496, -0.2740647494792938, -0.7476939558982849, 0.2517816424369812, 1.1457345485687256, 0.37451887130737305, -0.6519838571548462, -1.5967072248458862, 0.5962754487991333, 0.8625565767288208, 0.7662598490715027, -1.4002740383148193, 0.8645365834236145, 0.08711191266775131, -2.163466691970825, -1.2621163129806519, -0.6535700559616089, -0.4136069715023041, -0.67109215259552, 0.2511790990829468, -1.2955818176269531, -2.083420515060425, 4.306570529937744, 2.2629449367523193, 0.909330427646637, -0.2230176031589508, 0.1486407369375229, 0.7916788458824158, 0.6098003387451172, -1.4959179162979126, 0.4499185383319855, -1.5333802700042725, 1.2102293968200684, 0.16321033239364624, -2.2918701171875, -0.956723690032959, 0.32303380966186523, -0.5918369293212891, 1.0248883962631226, -0.7412552833557129, -0.6752479076385498, 0.12581153213977814, -0.6857017874717712, 1.1948788166046143, 0.25286003947257996, 2.0189626216888428, -1.577951192855835, 1.0096908807754517, 0.2397504597902298, 0.6255530714988708, 0.699609637260437, -0.3810276389122009, -0.8214252591133118, 4.3394317626953125, 1.2588869333267212, 0.6820904612541199, -0.46289029717445374, 0.8967317938804626, -0.5576543807983398, 1.9403042793273926, -0.8762273192405701, 0.270725816488266, -0.6242002248764038, -0.14728157222270966, -1.9189881086349487, 1.1063735485076904, 0.5852401256561279, 0.033423956483602524, -1.948106288909912, 1.0028280019760132, -1.4210929870605469, -0.45163941383361816, 0.7597636580467224, -0.5538891553878784, -0.6970621347427368, 0.29140183329582214, 1.7127678394317627, 0.4489439129829407, -0.40458592772483826, 0.4798913896083832, 2.2597038745880127, 0.3930208086967468, 0.16487570106983185, -0.4086841344833374, -0.16659530997276306, 0.7502157688140869, 0.03603994846343994, -0.4318936765193939, 0.17460955679416656, -0.2993943393230438, -0.0655556172132492, 0.5360068082809448, -0.6347747445106506, 0.14256151020526886, 1.523648977279663, 1.774390459060669, -4.9063873291015625, -0.18766465783119202, -0.4352271258831024, -0.6523659229278564, -1.636112928390503, -0.32266858220100403, -1.4396765232086182, 2.40678071975708, -0.2616754174232483, 1.7629544734954834, 1.2440012693405151, -0.2757827043533325, -2.171111822128296, -0.5494055151939392, 0.4668642282485962, 0.5744348168373108, -0.42529404163360596, 0.8158885836601257, 0.4183751046657562, -1.2891349792480469, -0.19661007821559906, -0.27716484665870667, -2.666332483291626, -1.323830008506775, 0.4917781949043274, -0.284086138010025, -0.8545850515365601, -2.5096588134765625, -0.32942435145378113, 1.811529278755188, -1.530411958694458, 3.1294479370117188, -0.9361492991447449, 1.387314796447754, 1.2681828737258911, -2.7450053691864014, 0.8590794205665588, 2.3203275203704834, 0.35910508036613464, -1.1461973190307617, 1.1024422645568848, 0.3415846824645996, -1.8355783224105835, -0.6164107918739319, -0.669937789440155, -1.2260656356811523, -0.9906393885612488, -0.24389730393886566, 0.1881784051656723, 0.9313461780548096, 1.0407333374023438, 1.110526442527771, -1.5940489768981934, 0.5156952142715454, 1.4154702425003052, -1.5795503854751587, 0.03816966712474823, 1.734207034111023, 0.6524341702461243, -0.4731772840023041, 0.31451550126075745, 6.272523880004883, 1.4670356512069702, -2.2070696353912354, 0.10959013551473618, -2.2327003479003906, 0.40518540143966675, -1.9855917692184448, -0.9971064329147339, -1.4665066003799438, 1.057782530784607, -0.9535574913024902, 1.1331086158752441, 0.12688127160072327, -0.8325793147087097, 0.4000997543334961, -0.4075033962726593, -2.1605007648468018, -1.253741979598999, 0.4463723599910736, 2.0196495056152344, 1.394698143005371, 1.9274119138717651, 0.7867931723594666, -0.173875629901886, 1.823952555656433, -0.2706466317176819, -1.9858514070510864, 1.4753696918487549, 0.41986143589019775, -1.4300345182418823, -0.3296206295490265, 0.5989060997962952, 0.6498117446899414, 0.15684251487255096, -4.320746898651123, -1.214082956314087, -1.9251432418823242, -2.6864726543426514, 0.24907219409942627, 1.6227003335952759, 3.231085777282715, -1.5313082933425903, 0.867032527923584, -0.14802518486976624, -2.3330845832824707, -0.442764550447464, 0.707230806350708, -0.3570828139781952, 0.8823595643043518, -0.2845633327960968, 0.8543785810470581, -2.366920232772827, 0.7605549693107605, 0.8500285148620605, 1.0423918962478638, 0.2990497946739197, -2.291594982147217, -1.1439034938812256, -0.19268445670604706, 0.544684112071991, -0.32017481327056885, -1.1186496019363403, -0.6268104314804077, 0.5460992455482483, -0.6281298995018005, -1.7983609437942505, 0.8907759785652161, 0.5410967469215393, -0.31591594219207764, -0.17326758801937103, 1.3023278713226318, 0.8605080842971802, -0.6707539558410645, 0.813553512096405, -0.7752916216850281, -0.4359549582004547, -0.21559235453605652, 2.107952117919922, -0.20633618533611298, -0.14366811513900757, 0.7664365768432617, 0.5057166218757629, 0.1225438192486763, 1.9475619792938232, -1.844131588935852, -1.170053482055664, -1.613362431526184, 0.7052936553955078, -0.38955679535865784, 0.8870012164115906, 2.122224807739258, -1.1157774925231934, -1.5426599979400635, 2.809563159942627, 1.255020022392273, 2.0827739238739014, 0.3390145003795624, 0.10384634137153625, -0.7344978451728821, 0.4737842082977295, 0.30162686109542847, 0.011271918192505836, -1.1104483604431152, 1.0038363933563232, -1.5486918687820435, -0.15256108343601227, -1.52708899974823, -0.930271565914154, -0.13024698197841644, 0.03632409870624542, -0.5511533617973328, 1.2037811279296875, 1.40610933303833, 0.6571069955825806, -0.40118762850761414, 0.9140306711196899, -2.8506462574005127, -0.11243763566017151, 0.7883300185203552, -0.18139265477657318, -0.9320196509361267, -2.070741891860962, -0.49088793992996216, -0.09255043417215347, -1.3487999439239502, -0.4239271283149719, -0.6039490699768066, -4.174938201904297, 0.6255031824111938, 1.2365214824676514, 1.0225203037261963, 1.46186363697052, 0.7405526638031006, 0.004184137098491192, -0.6191527843475342, -0.10685914009809494, -1.7430649995803833, -2.955746650695801, -2.0952632427215576, -1.6002695560455322, -0.08027347177267075, 0.7309409976005554, -0.41757774353027344, -1.561932921409607, -0.08808659762144089, 0.07508120685815811, -2.126199722290039, -0.4384998679161072, 1.4969145059585571, -0.9291248321533203, 0.2367987185716629, 2.8748574256896973, -0.9426644444465637, 0.5563969612121582, 0.7476229071617126, 1.724914312362671, 2.8766837120056152, -0.36561352014541626, -1.3372031450271606, -0.536333441734314, -1.0812785625457764, 1.50722336769104, -0.3809147775173187, 0.1491289734840393, 2.608160972595215, -1.1276013851165771, 2.015592575073242, -1.174930453300476, 0.7398644685745239, 0.2758456766605377, 1.1631320714950562, -1.6771068572998047, 0.5521241426467896, -2.5075876712799072, 1.2344772815704346, -1.4370876550674438, 0.36503952741622925, -0.8058217167854309, -0.48881232738494873, -0.9820777773857117, -0.24187716841697693, -0.7745299339294434, -1.0667853355407715, -1.1879092454910278, 0.4445708394050598, -0.920791506767273, -0.2551313042640686, -1.0404695272445679, 2.731945514678955, -0.7238332033157349, -0.4054204225540161, -0.5524394512176514, -2.3635175228118896, 1.8201245069503784, -1.9389595985412598, 1.0628825426101685, -2.243532180786133, 0.8069705963134766, -1.8900911808013916, 0.37414321303367615, -1.5278598070144653, -0.4024461805820465, -0.7907418012619019, 0.28321826457977295, 0.14765705168247223, 0.7087136507034302, 0.22805851697921753, 1.8136193752288818, -0.291136234998703, 0.2238582819700241, -0.5224984884262085, -0.9507287740707397, -1.0799846649169922, 0.7789772152900696, -0.5443331599235535, -0.24624857306480408, 0.38601788878440857, -0.13461805880069733, -0.06947442889213562, -0.29475221037864685, 0.25200068950653076, 1.044482946395874, -0.0398189015686512, 1.7866424322128296, 0.5785918235778809, -1.0506571531295776, 0.4193025231361389, -0.23717698454856873, -2.063018321990967, -0.31065359711647034, -0.6258522272109985, 3.8834028244018555, 1.5178438425064087, 0.48909124732017517, 0.5912147760391235, -1.670792579650879, -0.020688774064183235, -3.359423875808716, 3.187192678451538, -0.11530441790819168, -0.2378990203142166, -3.2146527767181396, 1.2616788148880005, 1.3267511129379272, -0.4883672893047333, 1.4571539163589478, -0.6711263656616211, 0.20647761225700378, 0.5457651615142822, 1.1919769048690796, 0.799142062664032, -1.1605340242385864, -3.2201905250549316, -0.7283169627189636, -0.2677013576030731, 2.357755184173584, -0.5522891283035278, 0.8413562178611755, 0.8648594617843628, 0.141161248087883, -1.8264058828353882, -1.5967389345169067, -0.9704009294509888, -0.6956902146339417, 0.14939036965370178, 0.2373226433992386, -0.04692826792597771, 1.1462749242782593, 0.37129104137420654, -0.149063378572464, -0.21244268119335175, -1.6586912870407104, -0.12396146357059479, -0.28000107407569885, 1.6677578687667847, 1.0009509325027466, 0.5937083959579468, -0.1756557673215866, -0.04326559603214264, 0.8632223606109619, 0.07938797026872635, 0.023725247010588646, 2.0174736976623535, 0.20643018186092377, 1.4902403354644775, 0.7863891124725342], [-0.3168352246284485, -1.062035322189331, -2.753070831298828, 0.35565415024757385, 1.0864182710647583, 3.3005714416503906, 0.5612810850143433, 0.7717072367668152, -1.4451137781143188, 0.0605471096932888, 0.40404459834098816, 1.0204676389694214, 0.3893139660358429, -1.7497564554214478, -0.39897236227989197, 1.2352731227874756, 1.3945616483688354, 0.35922423005104065, 0.9884198307991028, 0.028699902817606926, 2.0916600227355957, 0.4963217079639435, -0.7193982601165771, 1.2697206735610962, -2.6605725288391113, 0.884885311126709, 1.08073890209198, 0.44352439045906067, 0.9923005700111389, 1.422141671180725, 1.067762851715088, -0.21497194468975067, 0.13480018079280853, -0.6400830745697021, 1.6824684143066406, 0.6127318143844604, 1.8875466585159302, 0.8562621474266052, -0.6655338406562805, 0.7873327136039734, 2.63309645652771, -2.229031562805176, -1.206214427947998, 1.050991415977478, 1.2848458290100098, -0.5506858825683594, 0.1118597611784935, 1.0979409217834473, -1.4789471626281738, 1.4306546449661255, -0.19137679040431976, -0.3696579039096832, 0.6223068237304688, -0.6550295948982239, 1.455165982246399, -0.10982073843479156, -0.4579261243343353, -4.107227802276611, -1.896436333656311, 0.11952042579650879, 0.8918154239654541, 1.465722680091858, -2.177778959274292, -0.11769743263721466, -0.6281804442405701, 2.6255438327789307, -0.4065468907356262, 0.4229242503643036, 0.884714663028717, 2.655195951461792, -0.2685867249965668, -0.045203160494565964, 0.08834576606750488, -0.8260874152183533, -0.11440616846084595, -2.615241050720215, 5.039858818054199, -0.2812836468219757, 0.22621935606002808, 2.0859429836273193, 3.250025987625122, -1.2944844961166382, 0.29911068081855774, 1.2867705821990967, 0.45376673340797424, -0.3715132474899292, -0.08727157115936279, -1.3743361234664917, 0.21038156747817993, -1.2939857244491577, 0.2968831956386566, -0.19033513963222504, 1.8496803045272827, 0.5462422370910645, 1.0290727615356445, -1.3847278356552124, 0.14129237830638885, 3.970407724380493, -1.9784917831420898, -1.618031620979309, 0.255979984998703, -0.7314737439155579, 2.014784336090088, -0.042399708181619644, -2.6775755882263184, -0.4122055768966675, -1.655186414718628, -1.01944899559021, -1.0276196002960205, -0.9498288035392761, -0.051305465400218964, 1.1819368600845337, -3.520036458969116, 0.7871202826499939, 0.4680989980697632, -0.10628446936607361, 0.5281754732131958, 1.1396472454071045, 2.4630374908447266, -0.5261206030845642, -0.9897146224975586, 0.6034000515937805, -1.011814832687378, -1.50761878490448, 2.0940606594085693, -1.0652556419372559, 0.46814537048339844, 1.5495353937149048, -0.10735460370779037, 0.9793944954872131, -0.7011369466781616, 0.7585738897323608, -1.9430652856826782, -0.6473128199577332, 0.48010513186454773, -0.1858099102973938, -0.18428994715213776, -0.35782161355018616, 0.3991827666759491, 0.4100229740142822, -0.6969812512397766, 0.7527509927749634, 1.1931053400039673, -0.9483895301818848, 0.8241044282913208, -0.02946326695382595, 0.9711910486221313, 0.0952642485499382, -1.7079615592956543, -0.30276328325271606, -0.18786345422267914, 0.07984525710344315, -0.8214128613471985, 0.21142199635505676, -1.6426610946655273, 0.8729227781295776, -1.4555623531341553, -0.8226845860481262, 1.4031970500946045, 1.905503273010254, 3.5080227851867676, -1.21517813205719, -0.36686134338378906, -1.373049259185791, 1.6356792449951172, 1.7905678749084473, 0.6520035862922668, -2.0117104053497314, 0.002266803989186883, 1.2445191144943237, 2.0458145141601562, 1.3053531646728516, -0.5389180779457092, -0.4269621670246124, -0.3227551281452179, 0.29946669936180115, -1.074190378189087, -1.014880657196045, 0.4792217016220093, 0.09130700677633286, 1.004224419593811, -0.03704981505870819, 0.7202548384666443, -0.10617545247077942, -0.14145909249782562, -0.9260146617889404, -0.702091634273529, -0.7708572745323181, -0.03073371760547161, 0.27653321623802185, -1.6944938898086548, -0.9840312004089355, -2.1966605186462402, -0.679197371006012, -1.139095425605774, -0.08580652624368668, 1.4622834920883179, 0.10752508044242859, -0.28971654176712036, 0.5273113250732422, -2.1920535564422607, -2.281463861465454, 2.0193068981170654, -1.0877844095230103, 0.5700259208679199, -0.3476121723651886, 0.4536284804344177, -0.6551381349563599, -0.8768830895423889, 0.2819766104221344, 3.0220069885253906, -0.1495615839958191, 3.699531078338623, 0.8056154251098633, 1.5016906261444092, 0.2604845464229584, 0.2592759132385254, 0.2232189178466797, 0.09375672787427902, 1.945483922958374, -0.5418940782546997, -1.172127604484558, 1.1740758419036865, 0.3321824073791504, 1.185176968574524, 0.05573209002614021, -0.5267338752746582, 0.15880505740642548, 0.0023391905706375837, -1.1328355073928833, -0.010580001398921013, -0.4474678039550781, -1.947824478149414, -1.9368914365768433, 3.2234973907470703, 1.834613561630249, 0.7401595711708069, -1.6531412601470947, 2.0404958724975586, 1.4486738443374634, 0.11537615209817886, -0.3075171113014221, -4.713730812072754, -0.13979165256023407, -0.8805950880050659, 0.40449628233909607, -0.0060600643046200275, -1.2679634094238281, 1.763046145439148, -1.717240810394287, 2.161426067352295, -1.2242990732192993, -0.42110952734947205, 2.2795557975769043, -0.5275083780288696, 1.1345560550689697, 0.4446863532066345, 0.5171805620193481, -2.0518078804016113, 0.7630110383033752, 2.0908989906311035, -0.4629591107368469, 0.10283339023590088, 1.2129894495010376, 0.03952785208821297, 1.2663226127624512, 1.19332754611969, 2.0478098392486572, 2.0667343139648438, 0.9594256281852722, -0.6554921865463257, 0.1417756974697113, 1.3473026752471924, -0.552232027053833, -0.35946765542030334, 1.6400532722473145, -0.2818616032600403, -2.2560431957244873, 3.1580617427825928, -0.19282515347003937, -1.0387272834777832, 0.40048548579216003, -1.4450712203979492, -0.7677310705184937, -2.950174331665039, -0.2286471724510193, 0.04527604207396507, 0.22883491218090057, -0.7514666318893433, 1.6925373077392578, 1.7862509489059448, -1.30376398563385, -0.5177008509635925, -2.138767957687378, -0.003943882882595062, -1.4313511848449707, 0.5527964234352112, 0.619171142578125, 2.925412893295288, -0.17944073677062988, 3.41170597076416, 0.1619981974363327, -0.9563326239585876, 1.8405672311782837, -0.7125737071037292, 0.7811356782913208, 0.4629735052585602, 1.261430025100708, 1.3505867719650269, -1.5972213745117188, -0.1814468801021576, -0.6741716265678406, 1.566469430923462, 0.9761051535606384, -2.4234957695007324, 2.656876564025879, -1.2372066974639893, 1.4230406284332275, 0.8328834772109985, 0.6091861128807068, 0.24576376378536224, 0.47718846797943115, -0.7209325432777405, -1.6578572988510132, -1.0108965635299683, 0.5545018315315247, 1.181479573249817, -0.3098078966140747, 0.19932633638381958, -0.6298390030860901, -0.6974440217018127, 1.0521469116210938, 2.6186580657958984, -0.4609334468841553, 0.8848876357078552, -0.6462097764015198, 1.0237157344818115, -0.6469041705131531, -0.21802537143230438, 1.255653738975525, -2.652763605117798, -0.5590953826904297, -3.5068130493164062, 0.5092242360115051, -0.2850273549556732, -0.794590175151825, -2.1834139823913574, 0.12264596670866013, 0.7672975063323975, -0.12101174890995026, 1.4835237264633179, -1.009010672569275, 1.14873206615448, 1.0031861066818237, 0.2871733009815216, -0.21011017262935638, -3.6066367626190186, -0.17772281169891357, 1.168811559677124, -0.08286721259355545, 1.108203649520874, 1.9798297882080078, 1.1728553771972656, -1.6970158815383911, 0.9711329340934753, 1.201246976852417, -1.7405672073364258, 0.38311606645584106, -3.2339282035827637, 2.765331983566284, 1.0617252588272095, 0.898908793926239, 0.004819431342184544, -0.656527578830719, -0.76787269115448, 1.176608681678772, 1.2119052410125732, 0.3720276951789856, -0.6380623579025269, -5.014235973358154, 0.18526604771614075, -0.6577962636947632, -0.13425979018211365, -1.0617635250091553, -0.7974520325660706, 3.557219982147217, 2.5887882709503174, -1.932969331741333, -2.778534412384033, 1.3905171155929565, -0.1010422632098198, -1.7507771253585815, 1.9229382276535034, 0.1213294044137001, -0.7964089512825012, 0.9854385256767273, -1.1096045970916748, -3.209005832672119, 0.4540855586528778, -2.4286320209503174, -0.12663660943508148, -0.8988507986068726, -1.2340790033340454, -0.0028088875114917755, -0.5651207566261292, 0.4753539264202118, 0.1358344852924347, 0.308794766664505, -0.29651176929473877, 0.4956399202346802, 0.4143153429031372, -1.9363043308258057, -0.31817713379859924, -3.9057583808898926, -1.1406099796295166, 0.4019608199596405, -0.2797655165195465, -1.460250973701477, 0.15774038434028625, 0.4104173481464386, 0.8826886415481567, -0.6640278100967407, 0.9105595946311951, -0.03435729071497917, -0.9884856939315796, 0.5762717127799988, 1.0414384603500366, -1.4023066759109497, 2.597907543182373, 1.1783188581466675, 0.5810258388519287, -2.397531032562256, 0.17649221420288086, -0.39956578612327576, 1.6153993606567383, -0.5378358364105225, -0.30316922068595886, 1.859573483467102, 0.7428280711174011, -1.4497591257095337, -2.2185168266296387, -0.5281133651733398, 0.014864185824990273, 0.11164440959692001, -0.5158223509788513, -0.4024432599544525, -1.3459895849227905, 3.404021739959717, -0.618584394454956, -0.22364367544651031, 0.3138137459754944, -6.698451995849609, 0.1401844173669815, -0.0659932866692543, -0.3408767879009247, -1.4676157236099243, 0.35495322942733765, 2.7170536518096924, -0.29737669229507446, 0.0024009505286812782, 1.0924538373947144, 1.5899291038513184, 2.3558199405670166, 2.702010154724121, 0.21364717185497284, -0.3872993290424347, 0.588986337184906, 2.666020393371582, 2.2540252208709717, 1.3858129978179932, -0.7193956971168518, 1.718887209892273, 1.0447261333465576, -0.6073064804077148, -0.13987986743450165, 0.2951487898826599, -1.341731071472168, -6.501842021942139, -0.10871347039937973, -2.8830065727233887, -0.8922703266143799, -1.8894859552383423, 1.7527968883514404, 0.5148921608924866, 0.811767578125, -2.82686448097229, 0.22949744760990143, -1.704506754875183, -0.3317748010158539, 0.513020396232605, 5.028468132019043, -2.284799098968506, 3.03340482711792, -1.1182894706726074, -0.0778442993760109, -0.784137487411499, 1.451048731803894, -0.029892385005950928, -1.4890923500061035, 1.9785945415496826, -0.8523439764976501, -1.8092018365859985, -0.5893042683601379, 0.007761780172586441, 0.7233805060386658, 0.8181126117706299, -3.512892246246338, -0.12990255653858185, -1.9662467241287231, -0.06728248298168182, 0.1093468889594078, 0.5403976440429688, -0.6199923753738403, 0.2960396707057953, 6.2468132972717285, 0.03609289973974228, 0.04841160401701927, -0.7579938769340515, 1.0880095958709717, -1.309320330619812, -1.6167601346969604, 0.33940255641937256, -1.9226781129837036, -0.0880909413099289, 0.46393826603889465, -3.924757957458496, -0.1102149561047554, -0.905293881893158, -0.7021352648735046, -1.7470227479934692, -0.08704844117164612, -0.8677490949630737, 0.26280882954597473, -2.6243038177490234, 1.5614620447158813, 1.2131794691085815, 0.5325243473052979, 1.275241732597351, -1.0224945545196533, -0.05254719406366348, -3.038067102432251, -2.0580313205718994, -0.8891811370849609, 0.5605373382568359, -1.2441656589508057, -0.8096540570259094, 1.5747841596603394, -3.091754674911499, -0.8983747959136963, 3.4369702339172363, 4.724355220794678, 1.4465659856796265, -1.3275333642959595, 0.23798170685768127, 0.9957236647605896, -0.48344266414642334, 2.183619499206543, -1.2134103775024414, -0.6934361457824707, -0.5800934433937073, -0.4564018249511719, 2.2951138019561768, -0.15935330092906952, -0.5937180519104004, -1.5627046823501587, 0.595812976360321, 1.0757546424865723, 0.9034250378608704, -1.4825242757797241, -0.5573384761810303, -0.12128011882305145, -0.6410930752754211, 0.6124316453933716, -0.272792786359787, -0.6014872789382935, -0.5933815836906433, 0.3227618932723999, -1.4273308515548706, -2.147073984146118, 1.8790221214294434, 1.1657495498657227, 0.23267343640327454, 0.22516340017318726, 0.674545168876648, 1.541822910308838, 0.3205143213272095, -0.8037772178649902, 1.860945463180542, -1.7896314859390259, 1.2822585105895996, -2.453338146209717, -3.295264482498169, -0.7680176496505737, 0.13042320311069489, -2.2394893169403076, 0.8350226283073425, -0.4406569302082062, -1.1279008388519287, 0.14115066826343536, -0.5919283032417297, -0.044825587421655655, -0.3635857105255127, -1.3259544372558594, -1.2693625688552856, 0.8976860046386719, -0.10469117015600204, 1.2862107753753662, -0.5705999732017517, 0.23839178681373596, -0.7120057344436646, 2.779597520828247, 0.7400745749473572, 2.3257176876068115, -0.485908567905426, 0.6832765340805054, -0.5704613924026489, 1.4115186929702759, -0.7707456946372986, 0.42796626687049866, 0.03474808111786842, -0.7534542679786682, -1.543723702430725, 1.317502498626709, 2.343416213989258, -0.16456152498722076, -1.260928988456726, 0.9526671171188354, -1.8433325290679932, 0.6124627590179443, 0.15305842459201813, 0.7581198215484619, -0.965857982635498, -0.4483085870742798, 0.7545682787895203, 0.4991690218448639, 0.05521718040108681, 0.8274534344673157, -0.2496035099029541, -1.0033433437347412, 1.0148065090179443, -1.576359510421753, -0.2449295073747635, 0.5860475301742554, -0.7886027693748474, -0.14627130329608917, 0.1327638179063797, 1.8562874794006348, -1.564341425895691, -1.6858786344528198, -0.4267219603061676, 0.2105361372232437, -3.3801703453063965, 0.8914190530776978, -1.282346248626709, -0.5837275981903076, -3.2806692123413086, -1.5289490222930908, 0.07300794124603271, -0.2952761650085449, -1.7374324798583984, 1.5644530057907104, -0.5972808599472046, -0.42466098070144653, 1.561224341392517, -0.6266700625419617, -1.7111109495162964, -0.6504167318344116, 0.603286862373352, 1.8353557586669922, 0.34300413727760315, -0.9945683479309082, -1.2083665132522583, -1.2003933191299438, -0.17451269924640656, -0.0886300578713417, -1.1051232814788818, -0.9423267841339111, 2.9670844078063965, -0.12539976835250854, 0.041702691465616226, -1.8548698425292969, -0.4903978705406189, 1.309146523475647, -1.507030963897705, 2.1414289474487305, -6.0781731605529785, 0.9902019500732422, 2.26663875579834, -0.5260351300239563, -1.6330406665802002, 1.1791654825210571, 1.3744012117385864, -1.1831295490264893, 3.0636818408966064, 0.6086326241493225, -1.8157364130020142, 0.06668798625469208, -0.6264272332191467, -1.0060466527938843, -1.0154858827590942, -0.3359096646308899, -2.2591261863708496, 1.2100218534469604, 0.08360879868268967, -1.75239896774292, -0.09864674508571625, 0.5586431622505188, 1.2264786958694458, -1.315555453300476, 1.166480302810669, -1.0814346075057983, -0.013559904880821705, -0.10876362770795822, -0.7519047856330872, -1.0215299129486084, 1.5472192764282227, -1.9969230890274048, 0.2339637130498886, -1.95277738571167, 0.633745014667511, -0.9997945427894592, -1.1846749782562256, -1.0553406476974487, 0.36248666048049927, -0.9649768471717834, 1.1093225479125977, 0.35765862464904785, -0.8287015557289124, 0.8055670857429504, -0.34639406204223633, -1.446794033050537, -1.1587163209915161, 0.3731364905834198, 3.0680809020996094, 1.2958675622940063, 1.0244234800338745, 0.7875850200653076, -0.2819850742816925, -1.133402705192566, 0.24867720901966095, -0.7428407073020935, 1.4252417087554932, 0.7406037449836731, -1.4957252740859985, -0.2669496536254883, -0.1243620365858078, 0.43177998065948486, 0.36112573742866516, -1.2119131088256836, -1.032063603401184, -1.8829140663146973, -0.05122788995504379, -0.8349788784980774, 0.3866671919822693, -1.4502567052841187, -0.4085331857204437, 1.3725723028182983, -0.538861870765686, -1.781966209411621, -0.3872920870780945, 0.9574538469314575, -1.5680432319641113, 1.9972779750823975, -0.396284818649292, -5.906287670135498, -2.1866261959075928, 1.5364941358566284, 0.4025540053844452, -0.26165372133255005, 0.8255437612533569, -1.806239128112793, -0.8842591643333435, 4.3258843421936035, -0.4410627782344818, -0.006626022048294544, 0.6354712247848511, 0.5898352265357971, -0.9150689244270325, -1.7781928777694702, -1.347008228302002, 0.9331483840942383, -0.5620119571685791, -0.6224304437637329, 0.10015708953142166, 0.3420378565788269, 0.05528134107589722, -1.3358792066574097, 0.7159302830696106, -0.2880610525608063, 0.9121565818786621, -0.11524095386266708, 1.0182385444641113, 1.0193114280700684, -0.8185881972312927, 0.17022711038589478, 0.9430193305015564, 0.12776505947113037, 1.832423210144043, -1.2623955011367798, -0.4414641857147217, -2.647096872329712, 1.2835352420806885, 2.3015072345733643, 1.0273939371109009, 1.5075334310531616, -1.531543493270874, -1.3750903606414795, 2.459134817123413, 0.33107247948646545, 2.2211191654205322, 1.7248401641845703, 0.7535730004310608, -0.03352935239672661, 1.1742022037506104, 0.6057554483413696, -0.653186023235321, -0.34929928183555603, 0.35384857654571533, 0.6304588317871094, -0.14075490832328796, -0.8238170742988586, -0.48239514231681824, 0.5806068778038025, -0.15741966664791107, -2.6790125370025635, 0.8331857919692993, 1.4375369548797607, 1.0064795017242432, -0.314325213432312, 0.9127839207649231, -2.414451837539673, -1.677588939666748, 2.733672618865967, -0.331884503364563, -0.43842652440071106, -1.20396089553833, -0.9387292265892029, 1.081014633178711, -0.10074521601200104, -0.7465894818305969, -0.6326421499252319, -0.3681112229824066, -0.5066834688186646, 1.3517704010009766, 0.9652629494667053, 1.5440700054168701, 0.3934324085712433, -1.0107210874557495, -0.21618960797786713, 0.9485026001930237, -0.4425482153892517, 0.6575292348861694, -1.953827977180481, 0.2776811718940735, -0.08825485408306122, 0.35413801670074463, -0.42349323630332947, -0.4958876967430115, 0.6462773084640503, -2.819934129714966, 1.2972052097320557, -0.7833535075187683, 0.3956051468849182, -0.08460777252912521, -0.5594344735145569, 2.362993001937866, -1.5438141822814941, 0.3598281443119049, 0.5325578451156616, 1.4289931058883667, 0.2848939597606659, -1.4833227396011353, -0.8427691459655762, 0.09872984141111374, -0.16341076791286469, 1.7037047147750854, 0.38734760880470276, 0.02666470967233181, 0.666030764579773, -0.6462974548339844, -0.27358874678611755, -1.0865051746368408, 0.9059318900108337, 0.3547538220882416, 0.162014901638031, -1.4239449501037598, 0.7540723085403442, -2.3063161373138428, 0.0669517070055008, -0.9773819446563721, -1.7646492719650269, 0.6954091191291809, -0.7755258083343506, -0.10039600729942322, -0.3340359926223755, 0.06405606120824814, -1.423399806022644, 1.8925905227661133, -0.3168632686138153, 0.6179713010787964, -0.010305413044989109, -0.6365667581558228, 2.148834228515625, -0.8391095399856567, -0.041629135608673096, 0.024914870038628578, -1.1592967510223389, -0.535099446773529, -0.3922790288925171, 0.1989772468805313, -1.7351713180541992, 0.7213611006736755, 0.1668567955493927, 0.21941056847572327, -1.2677550315856934, -1.56404447555542, -0.15587309002876282, 0.36631882190704346, -2.040395498275757, 1.3333712816238403, -1.7526016235351562, -2.2115392684936523, -0.17509621381759644, -0.28975722193717957, -0.18721634149551392, 0.4895470142364502, -2.35899019241333, 3.3410277366638184, 0.4896678030490875, 0.15377101302146912, 0.27597776055336, -1.4892045259475708, -2.2764599323272705, -0.624865710735321, 0.485448956489563, 0.7055662870407104, -0.8190874457359314, -0.645317018032074, 0.630402684211731, -2.4631855487823486, 0.5428714752197266, -0.37191128730773926, -0.09123258292675018, -0.5263850092887878, 0.10107776522636414, 1.0733683109283447, 1.5384536981582642, 0.7260922193527222, 2.909093141555786, 0.9645527601242065, 2.7668638229370117, -4.072364330291748, 3.004516363143921, -2.104151964187622, -0.28468969464302063, 0.4712259769439697, 2.1431615352630615, 2.1977617740631104, 0.23596404492855072, 0.010604588314890862, -0.17212781310081482, -2.2094411849975586, -3.2189598083496094, 0.5542045831680298, 1.3243064880371094, -0.8257489800453186, -4.859415531158447, -0.7098554372787476, 0.19080331921577454, 0.9111947417259216, -0.5537809729576111, 0.4807307720184326, 3.7965962886810303, 0.19983574748039246, -1.8412232398986816, -1.0541290044784546, -0.7619152665138245, 0.42725518345832825, -1.5976682901382446, -0.09232848137617111, -2.525355339050293, 0.9195672869682312, 0.3321283757686615, 0.05453033745288849, -0.7478368282318115, -0.5465453863143921, -0.41981127858161926, -0.12600210309028625, -1.3672881126403809, -0.6150139570236206, 0.7833003401756287, -2.7119688987731934, -0.6139364242553711, 1.404258131980896, 0.20728245377540588, 1.4075273275375366, 2.2119338512420654, -2.1875269412994385, 1.374376654624939, 0.18398939073085785]], "dtype": "float32"}, {"__nd__": [99.97918701171875, 100.02933502197266, 99.7231674194336], "dtype": "float32"}, {"__nd__": [0.19527992606163025, 0.19678623974323273, 0.19550688564777374], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.039397113025188446, 0.624671220779419, -0.5724942684173584, 1.3033456802368164, -1.8043731451034546, -0.7916365265846252, -0.0017493967898190022, 0.49297720193862915, -0.2701129615306854, 0.9329560399055481, -0.23208263516426086, -0.9596045613288879, 0.6491449475288391, -0.5862677693367004, 0.6013981103897095, -0.9272598028182983, 0.6402002573013306, -0.569881021976471, 0.33652785420417786, -0.982483983039856, 2.026575803756714, 0.6754035353660583, 1.2724123001098633, 0.6685903668403625, -0.46784695982933044, 0.011263448745012283, -0.2874762713909149, 1.0574644804000854, 0.7815889716148376, -0.16894109547138214, 0.6727017760276794, -0.3634073734283447, 0.6463963985443115, 0.18554428219795227, 1.5160789489746094, -1.2154643535614014, -0.6883718967437744], [-1.4878779649734497, 0.3725886940956116, -0.5850383639335632, 0.9838339686393738, 1.9048742055892944, 0.17374469339847565, -0.7008556723594666, -0.9686408042907715, -0.30535775423049927, -0.6388628482818604, 0.4002062976360321, 0.17449794709682465, -1.1202143430709839, -0.3953096568584442, 1.3158408403396606, 1.3002116680145264, -0.9163544178009033, -1.2532031536102295, 1.1443593502044678, -0.14296522736549377, 0.49020904302597046, 0.7234731912612915, 0.027489174157381058, -0.16820847988128662, -1.506655216217041, -2.968933343887329, -1.5279991626739502, 0.5576167106628418, -1.234277367591858, 0.5963054895401001, -1.026366114616394, 0.4466853737831116, 0.4148483872413635, 1.538994312286377, -0.06852270662784576, -0.7104881405830383, 0.9439892172813416], [1.0809614658355713, -1.4837243556976318, -1.7903943061828613, 0.7599524855613708, -0.673947811126709, -1.6786748170852661, 0.7960150241851807, -1.636887788772583, -1.366701364517212, -0.5500472784042358, -0.7132323980331421, -0.6042237877845764, 0.7045072913169861, -0.9130715727806091, 2.1468710899353027, 1.062445044517517, -0.8563186526298523, 0.7577031254768372, -0.24285012483596802, 1.58634614944458, -1.5435298681259155, -1.2105159759521484, 0.2117936909198761, -0.4553791880607605, 0.4140651524066925, 0.5051944851875305, -0.03340720757842064, -0.6301872730255127, 1.9020410776138306, 0.5659253597259521, 0.17647378146648407, 0.3429180681705475, 0.6987420320510864, 1.308359146118164, 0.51209956407547, 0.617055356502533, -0.5587434768676758], [-0.2205285131931305, -0.6385735273361206, 0.13656437397003174, -0.23458175361156464, -0.5917116403579712, -0.21999771893024445, 0.4896233379840851, -0.5725350379943848, -2.746504783630371, 0.3236391842365265, 0.7011784911155701, -0.3562028706073761, 1.992323637008667, -1.2888463735580444, -0.14096124470233917, -0.9058916568756104, -0.15190854668617249, -0.17120812833309174, 0.5044415593147278, 0.963392436504364, 0.8072493076324463, 0.09169755131006241, 1.7347811460494995, 1.6353557109832764, 1.5360462665557861, -0.2843043804168701, 0.5144317150115967, 0.04811782389879227, -1.3352463245391846, 0.5986931920051575, -0.2577504515647888, -1.2485411167144775, 0.6872135400772095, 1.2087455987930298, 0.18251006305217743, 0.27142438292503357, -1.3435901403427124], [-0.16738756000995636, 0.6365430951118469, 1.871585726737976, -1.0531755685806274, -0.6319634914398193, 0.9692972898483276, 0.9714828133583069, 0.5407354831695557, 0.8915616273880005, 1.6251400709152222, 1.468104600906372, 0.030359826982021332, -0.35481691360473633, -0.04608231410384178, 0.03922651335597038, -0.2840888500213623, 1.8615360260009766, -1.1373395919799805, 0.2128807157278061, -0.7030988335609436, -0.5843300819396973, -0.4346577823162079, -1.9094055891036987, 0.5233993530273438, 1.2844599485397339, -0.9764290452003479, -0.0699055939912796, 1.286428689956665, -0.7440987229347229, -0.27584710717201233, 1.0228283405303955, 0.5138874053955078, 0.3564397692680359, -1.2293556928634644, 0.14720119535923004, -0.4156029522418976, 0.2519854009151459], [0.1599123179912567, -0.3063049614429474, 1.0326188802719116, 0.9051975607872009, -1.5774258375167847, 0.9189897775650024, 0.4612535536289215, -0.38106605410575867, -0.35773584246635437, -1.2648817300796509, -0.9936415553092957, -0.6008539795875549, 1.4149610996246338, 0.024107947945594788, 0.6684690117835999, 0.3563823699951172, 0.8454887270927429, 0.6823973059654236, 0.016771556809544563, 0.11634071171283722, -0.43163105845451355, -1.2786035537719727, -0.00426451675593853, 1.7385424375534058, 0.5457109808921814, -1.6193552017211914, -0.3287770748138428, -1.8958024978637695, 0.4318656921386719, -0.970483660697937, 0.38587358593940735, -0.05509006977081299, -0.9393086433410645, -1.1410205364227295, 0.9128280282020569, 1.4204720258712769, 0.3061227798461914], [-1.7001961469650269, 0.5132413506507874, -0.9088915586471558, -0.47197479009628296, 1.665042757987976, 0.035073716193437576, 2.462757110595703, 0.5831893682479858, -1.636506199836731, 2.1690428256988525, 0.9481106400489807, 1.7361271381378174, -0.1074235737323761, 1.6062471866607666, -1.624863624572754, -1.101699948310852, -0.17263078689575195, 1.5179500579833984, -0.9627034664154053, -1.3941532373428345, 0.7113365530967712, 1.5227932929992676, 1.6188210248947144, 0.15369343757629395, -1.9374775886535645, -0.4496695101261139, 1.1394903659820557, 0.05455509573221207, 1.785252332687378, -0.9977176785469055, -0.11232580989599228, -0.8842718005180359, -2.372786283493042, -1.0419937372207642, 1.0221858024597168, 1.2343076467514038, 2.142359733581543], [-0.1439656764268875, -0.40428850054740906, 0.9879752993583679, -0.16683156788349152, -1.7253278493881226, -1.2422679662704468, 0.7698559165000916, -0.9378936886787415, 0.2102079689502716, 0.7404927611351013, -0.22183391451835632, -0.9140152931213379, -0.22374753654003143, -1.6968302726745605, -2.317692756652832, -0.15736758708953857, 1.4561902284622192, 0.5606374144554138, 0.24474738538265228, -0.016121024265885353, -0.5835840106010437, 0.7500216364860535, 0.798778772354126, -0.512610673904419, 1.7512445449829102, 0.7552556395530701, 0.4378821849822998, 0.4113694131374359, 1.1870399713516235, -1.392753005027771, -0.04670233651995659, -0.41305461525917053, -0.3813204765319824, -0.005100506357848644, 1.5141969919204712, 0.766366183757782, -0.9145316481590271], [1.7245428562164307, -0.5304793119430542, -0.4453296959400177, -1.2469747066497803, -0.7809170484542847, -1.117283821105957, -0.8567011952400208, -0.7971055507659912, 0.43284597992897034, 0.4748366177082062, 0.03632790222764015, 1.5096673965454102, 0.23753973841667175, -0.008761463686823845, 1.1012500524520874, -0.894612193107605, -0.28490522503852844, -0.49418580532073975, 2.066258668899536, -1.8659415245056152, 0.30403444170951843, 1.2755026817321777, -0.9401960968971252, 0.28160879015922546, 0.5648475885391235, 0.706737220287323, 0.5922232866287231, 0.24538397789001465, -1.4746767282485962, -0.9110764861106873, 0.8186895847320557, 0.21872259676456451, 0.19527195394039154, 0.42101234197616577, 0.6239817142486572, 0.9819127917289734, 0.5531080961227417], [-0.5313342809677124, 0.5522186160087585, 0.2458828240633011, 0.023130765184760094, -1.6684849262237549, -0.0626862570643425, -1.0942736864089966, 0.048111990094184875, -0.20345839858055115, 0.054922837764024734, 1.805958867073059, 0.10360591858625412, -0.18091443181037903, -1.5577442646026611, -0.3487389087677002, -1.9995896816253662, -1.6036677360534668, -1.2419166564941406, -0.15182383358478546, 0.3368053436279297, -0.3684585392475128, 1.3837246894836426, -1.8734251260757446, -0.5475342869758606, -0.23256663978099823, 0.7601022720336914, -0.09232800453901291, -1.7625623941421509, -0.17714855074882507, 0.6484934687614441, -0.29221946001052856, 0.7276643514633179, -1.1175161600112915, 1.7500022649765015, 0.619010329246521, -0.5708553194999695, -0.06802400946617126], [0.5574967265129089, 0.4746326506137848, -1.2734278440475464, 0.8438655734062195, -0.3438272178173065, -0.5544512271881104, 0.985706627368927, -1.358012080192566, -0.0639464259147644, -0.6070559620857239, 0.3955606520175934, 1.1449013948440552, -0.5873100757598877, 1.2466137409210205, -0.3728983402252197, -0.2081788331270218, -0.10622639209032059, -0.4618135094642639, 0.44798484444618225, -1.3985399007797241, 1.420324683189392, -0.31050506234169006, 0.2032991498708725, -0.6365026235580444, -0.23960167169570923, -1.627248764038086, 0.6094983220100403, 0.5680209398269653, 0.22079601883888245, 0.3566843867301941, -1.713985562324524, 2.4312171936035156, 0.5587208867073059, 0.6256989240646362, 0.2831110656261444, -0.5577482581138611, -0.33723947405815125], [-0.3948683440685272, -0.6603465676307678, -0.8066596388816833, -0.39254119992256165, 1.3705447912216187, -0.09239940345287323, -0.020881325006484985, 2.4464359283447266, -0.04949686676263809, -0.23526477813720703, -0.4764035642147064, -1.9507051706314087, 2.23006272315979, -0.24399434030056, 1.1594158411026, -0.6131616830825806, 0.9860709309577942, 0.0012654263991862535, -0.45967617630958557, 1.633873701095581, -0.11196289956569672, 1.0148146152496338, -0.04074779152870178, 0.4315943121910095, 0.42451128363609314, -1.6768852472305298, -0.8455479145050049, 2.2255630493164062, -0.36173319816589355, 1.6645090579986572, -0.7498019933700562, -0.45749232172966003, 1.0561941862106323, 0.4919835925102234, 1.2678450345993042, 1.4966639280319214, 1.7318166494369507], [0.8219319581985474, -1.2689379453659058, -1.2456462383270264, -1.0370112657546997, 1.210329532623291, -0.4039931893348694, -0.7301310300827026, -1.090395450592041, 1.7522412538528442, 0.4572816491127014, 2.089857578277588, -0.23823292553424835, 1.3073146343231201, 0.9292913675308228, -1.0922985076904297, 0.7561500072479248, 0.6993263363838196, 1.3989311456680298, -1.1464958190917969, -1.611549735069275, 0.4794802963733673, -0.47461602091789246, -0.6770089268684387, -0.6756075024604797, 1.5079282522201538, 1.002374291419983, -0.23673713207244873, -0.43253186345100403, 0.1337590515613556, -0.048880599439144135, 0.9355314373970032, 0.9624543190002441, 0.9384495615959167, 1.6522495746612549, 0.4626029431819916, -0.15245194733142853, -0.8584677577018738], [0.02199772372841835, 0.6854940056800842, -0.002617123071104288, -1.520517349243164, -0.22702839970588684, 0.26996248960494995, 0.4514092206954956, -1.9801392555236816, 1.2902157306671143, 1.2554596662521362, -1.8486266136169434, 0.5739654302597046, 0.6338523030281067, -1.4146567583084106, -0.6164612770080566, -0.5551325082778931, -0.09381712228059769, -1.0623652935028076, 1.1032956838607788, -0.3500782251358032, 0.9437578320503235, -0.49273884296417236, 1.256680965423584, 0.3048122823238373, 0.5976778864860535, 0.8586507439613342, -0.43803930282592773, 0.6988499760627747, 0.4588490426540375, -0.05623636767268181, -0.6012396812438965, 1.2370010614395142, 0.7140598297119141, -0.5715346336364746, 0.6262467503547668, 0.459755003452301, 0.9433396458625793], [-0.24831639230251312, 1.6204347610473633, -2.526949167251587, 0.21201221644878387, 0.22966623306274414, -0.826877236366272, 0.6371060013771057, -0.4837758243083954, -0.04209517315030098, 1.045666217803955, -0.17391811311244965, -1.0240837335586548, -0.36437657475471497, -0.12558183073997498, 0.8780151009559631, 0.09086538106203079, 0.8910771608352661, 1.3381065130233765, 1.0670937299728394, 0.2624010741710663, -1.4537711143493652, 0.14762789011001587, 0.013230839744210243, -1.3835704326629639, -0.06720104813575745, -0.21524348855018616, 1.0265017747879028, -0.1989966481924057, 1.2774261236190796, -0.09377854317426682, -1.4254004955291748, -0.05366993322968483, -1.9690390825271606, 0.10957561433315277, -1.3484524488449097, -0.8263503313064575, 1.7743871212005615], [-0.13124361634254456, -0.2974223792552948, 0.1435641050338745, -1.271263837814331, 1.1113048791885376, -1.1560903787612915, -0.09688582271337509, -1.228333830833435, -0.07200455665588379, -1.2128440141677856, 0.5645060539245605, 1.404592752456665, -0.9606682062149048, 0.8474855422973633, 0.29657912254333496, 1.0321693420410156, 1.4060735702514648, 0.35459867119789124, 1.0804107189178467, -0.09123178571462631, 0.4239046275615692, -2.1872332096099854, 0.34479108452796936, 0.7109801173210144, -0.08234552294015884, -0.821334183216095, 0.11310090869665146, 0.42622581124305725, 1.2292511463165283, -0.5816866159439087, 0.3311585485935211, -0.5224239230155945, 1.0790935754776, 1.3111870288848877, 0.1644488424062729, 1.2186343669891357, -0.048525501042604446]], "dtype": "float32"}, {"__nd__": [0.1911548376083374, -0.30064335465431213, -1.2802377939224243, 1.9075045585632324, -1.7502257823944092, -1.1488844156265259, 0.1293339878320694, 0.9491105079650879, -0.8979775905609131, -0.385206401348114, 0.82436603307724, 0.19724982976913452, 0.06758682429790497, -0.6562841534614563, -0.9918254017829895, -0.5240035653114319, 0.7291731238365173, 0.8144723773002625, -0.9584784507751465, -0.1589798927307129, -1.665956974029541, 0.2886291444301605, 0.058879245072603226, 1.2760920524597168, -0.7490410804748535, 1.0577133893966675, -0.6737192273139954, 1.4817554950714111, 0.7090903520584106, 1.2492377758026123, -0.32549014687538147, -0.648423433303833, 1.3865119218826294, -0.02993099018931389, 1.5770699977874756, 0.2774741053581238, 0.6953363418579102], "dtype": "float32"}, {"__nd__": [0.2586596608161926, 2.444979429244995, -0.8817906975746155, 0.3314993381500244, 1.3654124736785889, -1.5742197036743164, 0.05621945485472679, -0.13376247882843018, 0.39910465478897095, 1.4185798168182373, 0.40477871894836426, 1.326008677482605, -0.6372172832489014, 0.8937156200408936, 0.06798543781042099, 0.6154683232307434, -1.6234204769134521, -1.580383539199829, -0.1531500518321991, -0.37693190574645996, -0.023482350632548332, -1.9948264360427856, 1.1588404178619385, 0.6089174151420593, -1.410178780555725, -1.0643374919891357, 1.12445068359375, -0.28492212295532227, -0.06400883942842484, 0.30544978380203247, 1.8994983434677124, 0.0912473201751709, -2.023569345474243, -0.6894194483757019, -0.8951876759529114, 0.26648396253585815, -2.188000202178955], "dtype": "float32"}, 0.001], "e": {"__tup__": [{"__nd__": [[0.24140916764736176, 2.2613372802734375, 0.1721024066209793, 3.0473742485046387, 5.388855934143066, -0.326873779296875, 0.038173336535692215, 0.29626035690307617, 0.8130650520324707, 1.0410348176956177, 0.062306489795446396, 1.0721672773361206, -0.5939518213272095, 1.4447972774505615, -0.5102015733718872, 1.2695080041885376, -1.164455771446228, -2.2483084201812744, -0.40779489278793335, -0.16798311471939087, -3.8387062549591064, -1.8009830713272095, 1.2404900789260864, 1.4555267095565796, -0.8874614238739014, -1.195434331893921, 1.4490423202514648, 1.388356328010559, 0.502409040927887, -0.11904452741146088, 1.6819524765014648, 0.46262869238853455, -1.1405638456344604, -0.6919581890106201, 1.7520946264266968, -0.17563949525356293, -2.8569178581237793], [0.00340342428535223, 2.3028111457824707, -0.2956300675868988, 2.366729974746704, -2.0687649250030518, -1.8954716920852661, -0.017554938793182373, -0.922171950340271, 0.5661525130271912, 1.6150989532470703, 0.8167330026626587, 1.3813081979751587, -0.7033171653747559, 1.0731781721115112, -1.3102976083755493, -0.10474969446659088, -2.1920766830444336, -2.482210636138916, -1.3253471851348877, -0.37244948744773865, -1.0017284154891968, -1.7599080801010132, 1.1669347286224365, 0.541630208492279, -0.39628612995147705, -3.9992802143096924, 2.0503644943237305, 0.6822378635406494, -0.8361082673072815, 1.1678167581558228, 2.188138723373413, -0.262076199054718, -1.3109668493270874, -0.7375044822692871, -0.8255488276481628, 0.10561012476682663, -1.4730316400527954], [0.46153080463409424, 2.8682281970977783, 1.2985191345214844, 1.7664138078689575, 2.4649274349212646, 0.25881946086883545, 0.15800084173679352, -1.609881043434143, 1.5621188879013062, 1.6146230697631836, -0.14427678287029266, 1.215334177017212, -0.5899829864501953, 1.4570863246917725, -2.0024266242980957, 0.06868848949670792, -2.2095205783843994, -0.9694629311561584, 0.0511803925037384, -0.623007595539093, 2.4177887439727783, -2.325244426727295, 1.1720598936080933, 0.07577842473983765, -1.7242146730422974, -0.5280942320823669, 1.1322287321090698, -1.1533546447753906, 1.2490639686584473, 1.011832594871521, 1.8374879360198975, -0.13619045913219452, -1.062276005744934, -0.727737545967102, -0.08515520393848419, 0.4370404779911041, -2.547699451446533], [0.20549170672893524, 2.659539222717285, -1.001987099647522, -0.22698310017585754, 2.5290448665618896, -1.2553035020828247, 0.11593425273895264, -0.7458138465881348, 3.0119991302490234, 1.3073382377624512, 0.9670891165733337, 1.2432645559310913, -0.5002016425132751, 1.8066962957382202, 0.26163530349731445, 1.1353669166564941, -1.7741048336029053, -1.7650715112686157, -0.6104860901832581, -0.5288041830062866, -1.3439520597457886, -1.9812196493148804, 1.262405514717102, 2.721306085586548, -2.5726118087768555, -1.428808331489563, 0.7959749102592468, -0.2823430895805359, -1.0847278833389282, 1.0241878032684326, 2.002652406692505, 0.9660648107528687, -1.0979844331741333, -0.7256630659103394, -0.6716330647468567, 0.3315199911594391, -3.194965124130249], [0.1921856701374054, 2.2832934856414795, -3.311976194381714, -2.1930196285247803, 2.869716167449951, -2.6131982803344727, 0.17349231243133545, 0.2765052318572998, -0.3360777795314789, 0.7919399738311768, 1.6032299995422363, 1.300380825996399, -0.674674391746521, 1.0342448949813843, 0.18716217577457428, 0.8650506138801575, -0.24735428392887115, -2.7338192462921143, -0.2213222086429596, -0.2278328388929367, 1.3209813833236694, -2.180170774459839, 1.0253832340240479, 1.1361595392227173, -2.3476009368896484, -2.3747687339782715, 1.2863930463790894, 1.5727027654647827, -0.7610524296760559, -0.2782194912433624, 1.585952877998352, -0.1698673963546753, -1.7056987285614014, -0.6439981460571289, -0.897115170955658, 0.09412680566310883, -2.1085927486419678], [0.2969300448894501, 2.5393264293670654, -2.3699393272399902, 2.2808029651641846, 4.367537021636963, -2.7657501697540283, 0.12508293986320496, -0.5098418593406677, 0.7318246960639954, 1.9465768337249756, -0.4786390960216522, 1.2000510692596436, -0.5301632881164551, 0.8605886697769165, -0.6867047548294067, 0.39705193042755127, -0.9262714385986328, -0.9481356739997864, -0.1937778741121292, -0.40112337470054626, 0.7295201420783997, -2.394813060760498, 1.1599706411361694, 3.0854406356811523, -1.8787524700164795, -2.9275074005126953, 1.3525675535202026, -3.3466756343841553, 0.2905687391757965, -1.0013784170150757, 1.7532434463500977, 0.11513684689998627, -3.4263434410095215, -0.6524809002876282, 0.7297083735466003, 0.7076743841171265, -1.9366992712020874], [-0.01343472395092249, 2.369741439819336, 0.17451147735118866, -0.612162709236145, -0.5969151258468628, -1.446339726448059, 0.2792395353317261, 0.15395763516426086, 1.6340619325637817, 0.8398908972740173, 0.8821530938148499, 1.5577645301818848, -0.6520226001739502, 0.18707647919654846, 1.4232767820358276, 1.1242097616195679, -1.8191033601760864, -0.7577974200248718, 0.6766741275787354, -0.1874261051416397, -0.6899400949478149, -1.7022649049758911, 1.222797155380249, 0.581335723400116, -0.2095833420753479, -1.569759726524353, 0.6368184685707092, -0.42802590131759644, 0.7954663038253784, -0.8091804385185242, 1.9720056056976318, 0.6141791939735413, -4.702310085296631, -0.6617115139961243, 0.10639715194702148, 0.48721104860305786, -1.157437801361084], [0.23528644442558289, 2.565030097961426, -2.267458200454712, 0.051844146102666855, 4.524882793426514, -0.09089449793100357, 0.16618262231349945, -1.0517246723175049, 0.1704416424036026, 1.1031043529510498, 0.23566658794879913, 1.140244722366333, -0.6512198448181152, 2.0585224628448486, 2.483656167984009, 0.6870137453079224, -0.47086623311042786, -1.069242238998413, -0.43245017528533936, -0.37912267446517944, 0.9596337080001831, -1.7555186748504639, 1.2107133865356445, -0.04774709790945053, -2.829336166381836, -0.1814761608839035, 0.7896553874015808, 0.4096073508262634, 0.8536950349807739, -1.5075047016143799, 1.9056061506271362, 0.356220006942749, -2.543330430984497, -0.6901829242706299, 1.6949342489242554, 0.501369297504425, -2.843229055404663], [0.6067258715629578, 2.645209789276123, -0.1493641436100006, -2.446143627166748, 3.014460802078247, -0.06557226181030273, -0.07644714415073395, -1.0449509620666504, 0.04318144917488098, 1.2480610609054565, 0.37104231119155884, 1.638431191444397, -0.6249857544898987, 0.95320725440979, -1.0562338829040527, 1.1748820543289185, -1.9115779399871826, -2.0902278423309326, -2.259606122970581, -0.03691047430038452, -0.44714513421058655, -1.6122043132781982, 1.0930224657058716, 0.9018758535385132, -1.816109538078308, -0.32561829686164856, 0.7389996647834778, -0.003943476360291243, -1.2746235132217407, -1.0508891344070435, 1.6319864988327026, -0.012645038776099682, -1.837275743484497, -0.7008923292160034, 0.06232678145170212, 0.5444793105125427, -1.8201760053634644], [0.19815853238105774, 2.1924326419830322, -1.538623571395874, 0.8566394448280334, 4.043669700622559, -1.7852764129638672, -0.06242556497454643, 0.1528363674879074, 0.3690642714500427, 1.2994604110717773, 2.2004082202911377, 1.3972547054290771, -0.6333299875259399, 1.8204128742218018, 0.18860235810279846, 1.602501392364502, -2.6887810230255127, -2.455892324447632, -0.23803794384002686, -0.47392570972442627, 0.21418088674545288, -1.496212363243103, 1.0558619499206543, 0.1829652041196823, -1.4119651317596436, 0.058853525668382645, 1.021999716758728, -2.701144218444824, -0.020374933257699013, 1.4832059144973755, 1.9194461107254028, -0.5748667120933533, -3.3298873901367188, -0.7528273463249207, 0.5420137643814087, 0.16695792973041534, -2.0642244815826416], [0.37618574500083923, 2.288456678390503, 0.9956575036048889, 2.1252241134643555, 2.0825841426849365, -0.8283793926239014, 0.1986924260854721, -1.6168760061264038, 0.4813636243343353, 1.691685438156128, 0.7598665356636047, 1.578992247581482, -0.6836182475090027, -0.02388511225581169, 0.5071718096733093, 0.7493832111358643, -1.7252615690231323, -2.0233592987060547, -0.623124361038208, -0.12118011713027954, -2.681753396987915, -2.1021618843078613, 1.1713337898254395, -0.3385280668735504, -1.1919974088668823, -3.0408761501312256, 0.6704081892967224, 0.6438198089599609, 0.10055442899465561, 0.7883399128913879, 2.5398313999176025, -1.6885292530059814, -1.1691826581954956, -0.710142195224762, -0.417472243309021, 0.08531156182289124, -2.4677138328552246], [0.1352483630180359, 2.7135448455810547, 0.4366154968738556, -0.8958613276481628, -0.3874996602535248, -1.1567113399505615, 0.017849119380116463, 1.769535779953003, 0.6894880533218384, 1.6099109649658203, -0.19015148282051086, 0.9123330116271973, -0.5153263807296753, 1.2250357866287231, -0.729987621307373, 1.0604932308197021, -1.1546950340270996, -1.805188775062561, 0.5236079096794128, -0.575215220451355, 0.6123403906822205, -1.801550030708313, 1.1402809619903564, 0.7690502405166626, -1.4992235898971558, -3.012371301651001, 1.842700481414795, 2.3811612129211426, -0.4998822510242462, 1.8992345333099365, 2.217425584793091, 0.5477619171142578, -1.0415815114974976, -0.6948618292808533, 0.5331885814666748, 0.5770343542098999, -1.2572200298309326], [0.38438400626182556, 2.875856399536133, 0.9232048392295837, -1.9598973989486694, -0.46551838517189026, -0.9213905334472656, -0.059452999383211136, -1.3245530128479004, -1.026909589767456, 1.3056930303573608, 1.9922181367874146, 1.2466222047805786, -0.5599589347839355, 0.3916122615337372, 1.3142552375793457, 0.30529654026031494, -1.2332381010055542, -0.5747423768043518, 1.1031653881072998, -0.09461569041013718, -0.5486822724342346, -2.17921781539917, 1.1093083620071411, -0.4628033936023712, -2.416675090789795, -0.17781102657318115, 1.394592523574829, -1.169185996055603, -0.0856153666973114, 0.03922174870967865, 1.648444414138794, -0.42634591460227966, -0.9500924944877625, -0.733957827091217, -0.42462682723999023, 0.1786121279001236, -2.8991262912750244], [0.24247217178344727, 2.242349147796631, -0.7373436689376831, -3.194453477859497, 2.0119988918304443, -1.802677035331726, 0.1087711974978447, -2.3869659900665283, -0.8270432353019714, 0.9079065918922424, -1.4283133745193481, 1.4337981939315796, -0.5956555008888245, 2.027392625808716, 0.8760506510734558, 1.005640983581543, -1.7817316055297852, -2.659227132797241, -1.25705087184906, -0.2958313524723053, -1.6382933855056763, -2.189147472381592, 1.236979603767395, 0.9135209918022156, -1.8398107290267944, -0.14202824234962463, 1.5358974933624268, 0.7363937497138977, 0.2301451414823532, 0.08790884166955948, 2.1590182781219482, -0.7546903491020203, -1.0437874794006348, -0.6665714979171753, 0.060899846255779266, 0.38187676668167114, -1.5143436193466187], [0.2220657765865326, 1.9305574893951416, 2.3331775665283203, 0.8602743148803711, 0.8487797975540161, -0.6775761246681213, 0.14804399013519287, -0.5429698824882507, 0.38248273730278015, 0.9848682284355164, 0.30940479040145874, 1.132464051246643, -0.6581416726112366, 0.9373481869697571, -0.8794463872909546, 0.5348385572433472, -0.9171881675720215, -0.4208659827709198, -1.2532289028167725, -0.42915788292884827, 2.339951753616333, -1.9337351322174072, 1.1632466316223145, -1.1102285385131836, -1.404898762702942, -1.2312082052230835, 0.3790414333343506, -0.49418333172798157, 0.9016773104667664, 0.26284259557724, 2.351858615875244, 0.08688566833734512, -4.717894554138184, -0.6945951581001282, -2.963425874710083, 0.050078924745321274, -0.8892484307289124], [0.20129697024822235, 2.5924532413482666, -0.9007983207702637, -2.7330162525177, -0.6016414165496826, 0.11988817900419235, 0.02250070869922638, -1.6118686199188232, 0.607611894607544, 2.0116465091705322, 0.8146946430206299, 1.613992691040039, -0.7217422127723694, 0.35454756021499634, -0.12066314369440079, 0.0740700215101242, -0.5575931072235107, -1.3713130950927734, -1.1964354515075684, -0.33651426434516907, -0.5834429264068604, -2.761502265930176, 1.173292636871338, 1.4576594829559326, -1.2273772954940796, -2.218233346939087, 1.1379683017730713, 0.21706639230251312, 0.8287736177444458, -0.71429842710495, 1.824690341949463, 0.5765149593353271, -0.5164705514907837, -0.729914665222168, -0.8340272903442383, 0.6124626398086548, -2.330745220184326]], "dtype": "float32"}, {"__nd__": [0.1147347241640091, -0.11395573616027832, -0.02141534723341465, 0.04644716531038284, 0.14831076562404633, -0.02167406491935253, 0.18227815628051758, -0.0290696918964386, 0.07343674451112747, -0.23480099439620972, 0.01663823053240776, 0.2970971465110779, 0.16422949731349945, 0.09605687856674194, -0.06027713045477867, 0.13060873746871948], "dtype": "float32"}, {"__nd__": [1.1978555917739868, 0.9719164371490479, 0.962730884552002, 1.0418205261230469, 1.1015245914459229, 1.1025398969650269, 0.7561452388763428, 1.0642122030258179, 1.1028121709823608, 1.0673449039459229, 1.1367510557174683, 0.9330076575279236, 1.0000107288360596, 1.1434448957443237, 1.018062949180603, 1.1460081338882446], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.6369377374649048, 1.2008917331695557, 0.9031282663345337, -1.577446699142456, -0.9514182209968567, 1.568644404411316, -0.016637520864605904, -0.300552636384964, -1.0127700567245483, 0.38520732522010803, 0.8029825091362, -0.2202073484659195, -0.18793852627277374, -0.065167136490345, 0.5176045298576355, 1.2449196577072144, -0.7977603673934937, -1.0585274696350098, -0.2366773784160614, -0.8052099347114563, 1.8849272727966309, -1.8051831722259521, 0.04124735668301582, -0.14240853488445282, 0.48622986674308777, 0.28880318999290466, 0.5689650177955627, 0.22758448123931885, 0.3923911154270172, -0.24560004472732544, 1.2665972709655762, 0.4904140830039978, -0.2862415015697479, -0.15079329907894135, -0.24185849726200104, 0.35030853748321533, 0.4466116428375244, -0.8246879577636719, -1.0089545249938965, 0.7501510381698608, -1.1644350290298462, -0.6433604955673218, 0.40138623118400574, 0.45031699538230896, -0.19241155683994293, -0.9640030264854431, -0.45023033022880554, -1.455927848815918, -0.8941437005996704, 0.7844529151916504, -0.5434344410896301, 0.1238352358341217, 0.1049768477678299, 0.2524242699146271, -0.7261725068092346, -0.897870659828186, 0.16808731853961945, -0.17107078433036804, -0.2535225450992584, -1.1024805307388306, 0.0671158954501152, -0.4693267047405243, 0.8511101007461548, -0.5681319832801819, 0.9011591076850891, -0.8200994729995728, -1.0429798364639282, 0.8171758651733398, -0.5050714612007141, -1.1951841115951538, -0.18252189457416534, -0.872999370098114, -0.6533207893371582, 1.2520160675048828, 1.7443068027496338, 2.304080009460449, 0.9457533359527588, -1.859780192375183, -1.5053030252456665, -0.5456998348236084, -0.007715314626693726, 0.13462309539318085, -0.7962612509727478, 0.35829025506973267, 1.2179100513458252, -1.6241087913513184, 1.0317293405532837, 0.9987166523933411, -0.8548310995101929, 0.08140219748020172, 0.21082447469234467, -0.57526695728302, -0.21806606650352478, 0.34326133131980896, -0.10460781306028366, -1.2061817646026611, 1.7102018594741821, -0.08897875994443893, -0.37356600165367126, -1.4950153827667236, -0.5409054160118103, -0.36395400762557983, 0.7706712484359741, 0.222971573472023, -0.8116633892059326, 0.6889857649803162, 0.10730992257595062, -1.1728768348693848, 0.7965933084487915, 3.063260793685913, -0.3378813564777374, -0.07290612906217575, 0.3397352695465088, 0.15878121554851532, -0.45421528816223145, 0.12287090718746185, -0.11137328296899796, -0.5863035917282104, -0.6959311962127686, -0.8557760119438171, 1.0390775203704834, -0.3428056240081787, -1.2444573640823364, 0.3017302453517914, -2.135101795196533, 0.5311833024024963, -0.19540292024612427, 1.3629018068313599, 0.5943830609321594, 0.2457973062992096, -1.0360283851623535, 0.3109247386455536, 0.7237465381622314, -0.19999471306800842, 0.5439332723617554, 1.5034582614898682, -0.6872455477714539, 0.8985446095466614, -0.5854886770248413, 0.4289894998073578, 0.6623515486717224, 0.43500009179115295, 1.2177188396453857, 1.0637184381484985, -1.3767035007476807, -0.7054986953735352, 0.04379407316446304, -0.47870907187461853, -0.4287288188934326, 1.114896535873413, 0.15467549860477448, 0.41723355650901794, -0.38554030656814575, 0.19129492342472076, 0.243768572807312, -0.7456900477409363, -1.208396553993225, 0.08951331675052643, 2.0788931846618652, -0.4022236168384552, 0.0308193601667881, 0.37692898511886597, -1.0786941051483154, -0.6113921999931335, -0.9709240198135376, -1.0092887878417969, -1.3100042343139648, 2.4424078464508057, -1.054307460784912, -0.5329465270042419, -0.5587352514266968, 0.4007509648799896, 0.12748171389102936, -0.19628754258155823, -1.4826035499572754, -2.2160398960113525, -1.2115994691848755, -2.2827727794647217, -1.4345675706863403, -0.5397589802742004, 0.11986009031534195, 0.17801766097545624, 0.055231090635061264, 0.41253986954689026, -0.4332534372806549, 0.05720619857311249, 1.2852457761764526, 0.5108131766319275, 0.6345673203468323, -0.3097624182701111, -0.016900187358260155, -0.47065654397010803, 1.0638258457183838, 0.4991195797920227, 0.9482572078704834, 0.613571286201477, 2.042959213256836, -1.7497633695602417, -0.8587310910224915, -0.36687231063842773, 0.43650999665260315, 0.6444451212882996, -0.3941546380519867, -1.1870754957199097, 1.505739688873291, 0.2179281860589981, -0.4802896976470947, 2.3135578632354736, 1.009702205657959, 1.1390206813812256, 0.519431471824646, 1.4932799339294434, 0.6142337322235107, -2.1270382404327393, -1.6591590642929077, 0.7431097030639648, -0.12342987954616547, -0.7848386168479919, 0.0706302747130394, -0.3300468623638153, -0.6251822113990784, 0.2062331587076187, 2.06404447555542, -0.4779309928417206, 0.27513647079467773, -0.30790024995803833, 1.3424508571624756, -0.46767351031303406, 1.8421430587768555, -0.3669739067554474, 1.8102903366088867, 0.33687281608581543, -0.1466202735900879, 1.4217581748962402, -0.45281460881233215, 0.6372304558753967, 0.3138764798641205, -1.090104103088379, -0.6229797005653381, 0.05481891706585884, -0.9345971941947937, -0.7221638560295105, -0.6719546318054199, 1.3181660175323486, -1.237466812133789, 0.3808003067970276, -1.245503544807434, 0.009338549338281155, -0.6932859420776367, 0.6920879483222961, 0.2831505835056305, -0.577939510345459, -0.6025912165641785, -1.0884065628051758, -1.1682151556015015, -0.4391918480396271, 0.5107643604278564, -1.197200894355774, 0.975687563419342, -0.7610347867012024, 0.23925578594207764, 0.8813082575798035, -0.48758795857429504, -0.060186367481946945, -0.23322272300720215, 0.9467137455940247, -0.8287791013717651, 1.932645559310913, -0.11358889192342758, 0.7167014479637146, -0.3465096354484558, 0.5497658848762512, 0.40143874287605286, -0.13718968629837036, -0.9939143657684326, 2.186974048614502, 0.6348745226860046, 0.01957898959517479, 1.0238953828811646, 0.6257266402244568, -0.609182596206665, -0.9590198397636414, 0.255083292722702, 0.2077236920595169, -1.075356364250183, -0.24531838297843933, -0.640240490436554, 0.7665724754333496, 0.657745897769928, 1.1091325283050537, -0.6953722238540649, -0.4357195496559143, 0.14414410293102264, -0.5692102313041687, 1.4825133085250854, 0.14577117562294006, 0.510053277015686, 1.9182900190353394, 0.523319661617279, -0.3336048722267151, 0.41211754083633423, -0.06696907430887222, 1.690550684928894, -0.09274978935718536, -0.17697666585445404, 0.29698997735977173, -2.72310471534729, 0.038207583129405975, 0.672798752784729, -0.6132103204727173, -0.504153311252594, -2.989360809326172, -2.7968671321868896, 1.294715404510498, -1.7864598035812378, -0.013263493776321411, 0.0071611893363296986, -0.2062588781118393, -0.13378998637199402, 0.5155723690986633, -0.1359298974275589, 1.292962908744812, -0.8282879590988159, -0.36569926142692566, 0.5510348677635193, 0.3626752495765686, 0.7207581400871277, 0.7459659576416016, 0.7456874251365662, 0.8185802698135376, 0.4581793546676636, -0.4595945477485657, 0.23775571584701538, 0.7788098454475403, 0.3950871229171753, 1.5586992502212524, -1.6226450204849243, -0.4541073143482208, 1.3448604345321655, 0.28583061695098877, 0.11687341332435608, 0.9941174983978271, -0.1095854714512825, -1.4214504957199097, 0.6994573473930359, -0.07708714157342911, 0.46353277564048767, 0.19156843423843384, 1.1446219682693481, 0.24574825167655945, -0.05788392573595047, 0.6313132643699646, 1.025646448135376, 0.047238122671842575, -2.8544952869415283, -0.7387692928314209, -0.8199537992477417, 0.0961560606956482, -0.33949360251426697, -0.7025647163391113, 0.7536102533340454, 0.675713062286377, -1.308279275894165, -0.16206687688827515, 0.2905290722846985, 0.005481170490384102, -0.43115460872650146, 0.6430748105049133, -0.2337329238653183, 0.06632427126169205, 0.2309579849243164, 1.0341514348983765, 0.13981398940086365, 0.811199426651001, -0.041650645434856415, 1.456315279006958, 1.0722482204437256, -0.6272650957107544, 0.47928303480148315, -0.5625591278076172, -1.684709072113037, 0.6512537002563477, -0.37393975257873535, 0.3489663004875183, 0.5445590019226074, 0.562375009059906, 1.1670185327529907, 0.6063064336776733, -1.7609829902648926, -0.44271060824394226, 0.30044636130332947, 1.534940481185913, -0.035785190761089325, -0.35822632908821106, -0.7387562394142151, 1.14324152469635, 2.1920669078826904, -0.7071967124938965, -1.2438855171203613, -0.46294504404067993, -0.10455448180437088, 0.5895018577575684, 0.4478285312652588, 0.9781799912452698, 0.8975736498832703, -0.13536600768566132, -0.8854725360870361, -0.8185879588127136, -1.0014066696166992, 0.7766211628913879, -0.17052122950553894, 0.13050438463687897, -0.4335983395576477, -0.5239241123199463, -0.6915290951728821, -1.5871331691741943, 0.4335111975669861, -0.2181607484817505, -1.5919623374938965, -0.49049636721611023, 1.0972788333892822, 2.2322001457214355, -0.8440428376197815, 0.5876013040542603, -0.5295724272727966, 1.1615687608718872, 0.7961394190788269, -1.317845344543457, -1.2611128091812134, -0.4107569456100464, -1.3447422981262207, -0.12421126663684845, 0.5062633156776428, -1.8114988803863525, 1.1177716255187988, -2.3869552612304688, -0.3599896728992462, 0.49069249629974365, 0.11402475088834763, 1.2021677494049072, 0.21086646616458893, 0.3616395592689514, 1.3983701467514038, -0.6668121218681335, 0.23465856909751892, -0.6370493769645691, -0.00896927248686552, 0.16130447387695312, -0.36984187364578247, -0.14459826052188873, -0.07860791683197021, 1.625352144241333, 0.07899869233369827, 1.0975053310394287, -1.4304174184799194, 0.8736228942871094, 0.5437179803848267, -1.3544602394104004, -0.9057214260101318, 1.2035685777664185, -0.5535044074058533, 0.26163768768310547, 0.2486594170331955, 0.2856005132198334, -2.1379852294921875, 0.6230419874191284, -0.1539398431777954, 0.27523669600486755, 0.08464042097330093, -0.025042880326509476, 0.24216309189796448, -0.6783386468887329, 1.0224312543869019, 0.9974050521850586, -0.5239923000335693, -0.40588775277137756, -0.06450319290161133, 1.6062991619110107, 1.0941805839538574, 0.3169509470462799, 0.11674769967794418, 0.6416986584663391, 2.265359878540039, -1.2662668228149414, 1.2160022258758545, 1.839900255203247, 0.36730486154556274, 0.10618746280670166, -0.811926007270813, 0.759808361530304, 0.6921700835227966, -0.09183409810066223, 1.5174269676208496, -0.7869306802749634, 0.22345712780952454, 0.7158486843109131, 1.0764480829238892, 0.09079215675592422, 0.268362432718277, -0.9177654385566711, -1.2714465856552124, -0.14947527647018433, -0.25024574995040894, -0.13118243217468262, -0.5339202880859375, -0.32211336493492126, -1.2169525623321533, -0.25576770305633545, -1.2458956241607666, -0.9622005820274353, -0.39520737528800964, -0.9315573573112488, 0.8436744809150696, -1.0422706604003906, -1.56950044631958, 0.3443743884563446, 1.0851086378097534, 1.2107138633728027, -1.3332396745681763, 0.4376562833786011, -0.480075478553772, 1.2708251476287842, -1.695196270942688, -0.8683580160140991, 0.5132502913475037, 1.2123042345046997, 1.2215447425842285, 0.8374542593955994, 1.8547598123550415, 0.210868239402771, -0.04685498774051666, -1.5292422771453857, 0.1724124550819397, -2.067664623260498, 0.420249342918396, 0.09960394352674484, -0.7920533418655396, 1.862904667854309, 0.7530531287193298, -0.5638495087623596, 0.447998046875, 0.8124850988388062, -0.6105476021766663, 0.7907059192657471, -0.17263755202293396, 0.04720137268304825, 0.45031407475471497, 2.0641324520111084, 1.429672360420227, -0.8583344221115112, -1.3016972541809082, -0.14724349975585938, -0.2791569232940674, 0.16213476657867432, 1.9930453300476074, 0.7737792730331421, -0.7549452185630798, 0.6270191669464111, -1.033217430114746, -1.3181301355361938, 1.1944676637649536, -0.09463373571634293, -0.13662613928318024, 0.8426346182823181, 0.6605558395385742, 0.10377894341945648, -1.7935395240783691, -0.14602981507778168, -0.006344458553940058, -1.213989496231079, -0.29649895429611206, -0.5040531754493713, -1.7633484601974487, -0.47293800115585327, 0.27582037448883057, -2.2694616317749023, 0.3051365911960602, 1.8857316970825195, -1.4688102006912231, -1.9322500228881836, 0.8484823107719421, 0.9222697019577026, 0.5099873542785645, 1.5643244981765747, 2.1863996982574463, 0.14556726813316345, -0.16644425690174103, -0.3377724885940552, -0.5451219081878662, -0.33243051171302795, 0.18495561182498932, 0.8617907166481018, -1.3508563041687012, 0.5165318846702576, 1.491745114326477, 1.5271897315979004, -0.7534762024879456, -0.3256644904613495, 0.8089457750320435, 0.3185996413230896, 1.0797184705734253, 1.0663673877716064, 0.33146560192108154, -1.3061399459838867, 0.7167451977729797, -0.634458065032959, 1.015236258506775, -1.4377795457839966, 0.4460393786430359, -1.1404166221618652, -0.786238431930542, -0.7649400234222412, 0.5420755743980408, 0.7023894786834717, 1.1253474950790405, 0.2540214955806732, 1.4649178981781006, -0.3320406973361969, -1.1834784746170044, 0.7033734321594238, 1.2817009687423706, 0.021221455186605453, -1.1952587366104126, 0.39570853114128113, -0.06513185799121857, 0.15758004784584045, 0.7333214282989502, 1.143332600593567, 1.3561707735061646, -0.332777202129364, 0.45294129848480225, -1.403425693511963, 1.5228655338287354, 0.7042182087898254, -1.2513145208358765, -1.1009503602981567, 0.12392888218164444, -0.10313323885202408, -1.3441321849822998, -0.04069351404905319, -1.1904382705688477, 1.791387677192688, 0.4765044152736664, -0.9732173085212708, 2.0847997665405273, -1.2960928678512573, 0.6440669298171997, 1.222196340560913, 0.31159666180610657, -0.38938581943511963, 0.9108607172966003, -1.1549060344696045, 1.9112838506698608, -1.477170705795288, -0.7797466516494751, -0.9708166122436523, -0.606789231300354, 0.2771068513393402, 1.1719285249710083, 1.022603154182434, -0.05382711440324783, -0.3818413317203522, 0.038017891347408295, -0.35394808650016785, -0.484112024307251, -1.2193385362625122, -0.5207952857017517, 0.1650976687669754, -1.4197046756744385, -0.5462684631347656, 0.8606700897216797, -0.8822888731956482, 1.7747176885604858, -0.7781266570091248, 0.3090227544307709, -0.4539166986942291, -0.4996034801006317, -0.703080952167511, -0.5746704936027527, 1.396538257598877, 0.6214730143547058, -0.7598959803581238, -1.174330234527588, -0.1910885125398636, -0.6300627589225769, -0.3068809509277344, -1.320976972579956, -0.20420795679092407, 0.07493378967046738, -0.29936930537223816, -0.028579790145158768, 0.04438198357820511, -0.05373489484190941, -0.023731784895062447, -1.2914329767227173, 0.26858440041542053, -0.3736798167228699, 1.1195251941680908, -0.43416595458984375, -1.0689564943313599, -0.3436122536659241, 0.8020573854446411, 2.3810698986053467, -0.7249927520751953, 0.6557362079620361, 0.1057814359664917, 1.0040329694747925, 0.2365187257528305, 2.6609416007995605, 1.8962963819503784, -0.694557785987854, -1.481552004814148, -3.1098804473876953, -0.34298911690711975, 0.2803384065628052, -2.3990747928619385, -1.4052584171295166, 0.6528236269950867, -1.4778773784637451, -1.232401967048645, 0.6199060082435608, -0.19745886325836182, 1.247256875038147, -0.7678158283233643, -0.4745137393474579, 0.12194656580686569, -0.9468886256217957, -0.9361812472343445, -1.4832626581192017, 1.1819803714752197, -1.0124863386154175, 2.882769823074341, -0.1638544499874115, -1.316615343093872, -1.3124076128005981, 0.4803345203399658, -0.5654039978981018, 0.45989084243774414, 0.605274498462677, 0.16174571216106415, -0.10865528136491776, -1.0694316625595093, -0.1827811896800995, -0.4584917426109314, -0.7284421920776367, 0.5539408326148987, 1.2274824380874634, 0.09995432198047638, 1.4738390445709229, -0.7956847548484802, -0.10212220996618271, -1.3715885877609253, 2.354614019393921, 0.3493632674217224, -0.34827446937561035, -1.391398549079895, -0.11291299760341644, 1.409194827079773, -1.361596703529358, 0.2892983555793762, -0.27444127202033997, 1.035951852798462, -0.45062586665153503, -0.7571342587471008, 0.5233553647994995, -1.3747873306274414, 0.975238561630249, 0.7225548028945923, 1.1534216403961182, 1.1781038045883179, -0.20804822444915771, -0.030676374211907387, -0.8968945741653442, -0.43084651231765747, -0.768161952495575, 0.7964774370193481, 0.2577069103717804, -1.7652263641357422, -0.26557353138923645, 1.731000542640686, 0.8778337836265564, -0.6903477311134338, -0.34581708908081055, -1.479857325553894, 0.9208086133003235, -0.6934078931808472, 0.07047411799430847, 1.379253625869751, 0.8237109780311584, -1.343767762184143, -0.7526090145111084, 1.1652436256408691, 2.5580811500549316, 0.1869390308856964, -1.0082956552505493, 0.2327152043581009, -1.0584208965301514, 1.955704689025879, -0.8812699913978577, 0.12312459200620651, -0.3692912757396698, -0.08428570628166199, -1.4179280996322632, -1.065244436264038, 2.370309352874756, 1.693743348121643, 1.4691903591156006, -2.018179416656494, 1.0270777940750122, -0.6007934808731079, -2.1611931324005127, -0.6221739649772644, -0.6965292096138, -1.2484575510025024, 0.5014587044715881, 0.5164044499397278, -0.4655987024307251, -2.6170175075531006, 1.8566277027130127, 0.04169600456953049, -0.5287814736366272, -1.0993268489837646, -0.7023271322250366, 0.2819870710372925, -0.018529826775193214, -2.229959726333618, 0.9893848299980164, 0.3964318037033081, 2.1454241275787354, 0.7804512977600098, 0.730255663394928, 1.6633437871932983, 1.7252522706985474, 1.4979474544525146, -1.5704785585403442, 1.7172142267227173, 0.9626286625862122, -0.1910703033208847, -1.5966273546218872, 2.047746181488037, 1.4659062623977661, -0.711868941783905, -0.30618762969970703, 0.48809394240379333, 0.10404421389102936, 0.056027837097644806, 1.4249815940856934, -1.8490757942199707, 0.9894763827323914, 1.5270839929580688, 0.2353457510471344, 0.626460611820221, 0.2428268939256668, 0.26215842366218567, -0.05941587686538696, 0.3883267045021057, -0.5694198608398438, -0.8080945611000061, 0.2568867802619934, 0.6627138257026672, 0.6510918140411377, 0.7307960391044617, 0.17047184705734253, 0.4308469891548157, -1.1718498468399048, 1.5421903133392334, -0.26573827862739563, 0.2492709755897522, 0.547262966632843, -0.7910476922988892, 0.17735840380191803, 1.601228952407837, 0.42903900146484375, -0.6727016568183899, -1.342604398727417, 1.7102108001708984, 0.14140407741069794, 0.18389734625816345, 0.5929914712905884, 1.7898846864700317, -0.840070366859436, -0.4395204782485962, -1.7332422733306885, -1.261216402053833, 1.5818313360214233, 0.5981761813163757, -0.19312520325183868, 0.6578525900840759, -0.9235526323318481, 0.4207402169704437, -0.9359283447265625, 0.9732410907745361, -0.1698458194732666, -0.4756489098072052, 0.19518481194972992, -0.11570717394351959, -1.0862783193588257, -0.7280581593513489, -0.8470348119735718, 1.0087332725524902, -1.025460124015808, -1.0797669887542725, -0.40206530690193176, 0.11691905558109283, 0.12086567282676697, -0.0945887640118599, 0.5302171111106873, 0.8498538136482239, -0.041978392750024796, -0.6916983723640442, 0.21886123716831207, 0.33040666580200195, -1.0250612497329712, -1.928983211517334, 0.24483609199523926, 1.3566216230392456, 0.6040801405906677, 1.6631444692611694, 0.7438623309135437, 0.7484642267227173, 2.1710104942321777, -0.09392791986465454, -1.4003292322158813, 1.6556782722473145, 0.2972971796989441, -0.3630671203136444, 1.6673076152801514, -0.9070195555686951, -1.3563497066497803, 0.2947615385055542, -0.936713457107544, -0.6683164834976196, -0.5695319771766663, -0.5687214136123657, 1.1201894283294678, -0.024163737893104553, -0.70188969373703, 0.8193432092666626, -0.17711901664733887, 1.0939394235610962, -0.7359398007392883, 0.0298642348498106, 1.9207165241241455, 0.2161143571138382, -2.0015251636505127, 1.555498480796814, -0.13732002675533295, 1.4750117063522339, -0.5408148169517517, 0.6302429437637329, -0.07212007790803909, -0.16717202961444855, -0.17569467425346375, -0.17646366357803345, -1.179444670677185, 0.011177255772054195, -0.0028476843144744635, 1.1499823331832886, 0.6091049313545227, 0.7212242484092712, 0.2687496542930603, -0.2426687330007553, 0.59518963098526, -0.41019436717033386, -0.3321891725063324, 2.5691471099853516, 1.685968041419983, 0.17298366129398346, -0.5557829141616821, -0.9368608593940735, -1.0745255947113037, -1.9364227056503296, 0.26464012265205383, -0.29682105779647827, 0.13554006814956665, -1.2162822484970093, -1.7469522953033447, -0.9466441869735718, -1.236106514930725, 0.23001790046691895, -0.18328656256198883, 0.4020574688911438, 0.5232714414596558, -0.2997870445251465, -0.2611554265022278, -0.4854874014854431, -0.6533368229866028, -0.05615369230508804, -0.39709362387657166, 0.07616205513477325, -1.028874397277832, 0.7083160877227783, 0.6046169400215149, 0.39625853300094604, 0.26249751448631287, -0.403714120388031, -0.07164280116558075, -1.41419517993927, -0.046219274401664734, -1.1727081537246704, -0.2580016553401947, 0.5742600560188293, -1.0884274244308472, -0.5799095034599304, 0.009301434271037579, -0.7242822647094727, 1.3071731328964233, -0.8945713043212891, 2.546226978302002, 0.13348157703876495, 0.48099616169929504, 0.6305789351463318, -1.1363563537597656, -0.04472732171416283, 0.5736358761787415, -2.1302568912506104, 1.0252013206481934, 0.5146886110305786, -0.4927627742290497, -0.24022230505943298, 0.46634966135025024, 0.5544410347938538, -0.8533048033714294, -0.9158825278282166, -0.8991901278495789, -0.3114422857761383, -1.1501792669296265, -1.4494317770004272, -0.39565786719322205, 1.4142431020736694, -0.9691381454467773, 0.5687422752380371, 1.6940138339996338, 1.2182278633117676, 3.6282877922058105, -0.6644192337989807, 0.29737937450408936, -0.3405638337135315, 1.1142133474349976, 0.7194666266441345, -0.30449378490448, -1.1337698698043823, -0.2799590826034546, 0.19833363592624664, 0.5315709710121155, 0.8001095652580261, -0.3310745358467102, 2.0682969093322754, 1.401982069015503, -0.7917925715446472, -0.49438023567199707, 0.030599208548665047, -0.14394541084766388, 1.0876953601837158, 0.9997479319572449, 1.7474513053894043, -0.27884578704833984, 1.07443106174469, -0.796786904335022, -1.861986756324768, 0.8023771047592163, -0.8461903929710388, -1.1188945770263672, 0.7525453567504883, 1.8459455966949463, -0.46904537081718445, -1.4885543584823608, -0.9087952375411987, -1.1497511863708496, 0.8968800902366638, 1.3748974800109863, -1.4143433570861816, -0.3999171257019043, 0.2388761341571808, 2.423581600189209, 0.030684834346175194, -0.43241918087005615, 1.25558340549469, 2.3637940883636475, 0.38668522238731384, 0.0229702889919281, -0.6718264222145081, -1.6926547288894653, 0.3938760757446289, 0.3709113597869873, 2.4120025634765625, 1.4347014427185059, 1.3871467113494873, -0.021617362275719643, -0.44709011912345886, 0.798550546169281, -1.076222538948059, 2.197030782699585, -0.1732957363128662, 0.6697879433631897, 1.6693058013916016, 0.07208188623189926, -0.7868697047233582, -0.7464056611061096, -0.6260271072387695, -1.7083451747894287, -0.6233818531036377, 0.19636160135269165, 1.1010704040527344, 0.6587285995483398, -1.3120708465576172, 0.7785423398017883, -0.3204531967639923, -1.0584650039672852, 0.3158433139324188, -0.38567543029785156, 1.6099692583084106, 0.7120752930641174, 0.19968143105506897, -0.1759072095155716, 0.19310973584651947, 0.5042331218719482, 1.2163456678390503, 0.4703630805015564, -0.35560673475265503, 0.4389268159866333, -1.4914770126342773, 1.1719839572906494, -0.27378201484680176, -0.30154624581336975, 0.6117507815361023, 0.46408551931381226, -1.0735492706298828, -0.5908896923065186, 0.5210155248641968, -1.3806180953979492, 1.4116507768630981, -0.5874679684638977, -0.6871639490127563, 2.077279567718506, 0.8604628443717957, 2.012364625930786, 1.1625516414642334, -0.7520922422409058, 0.9264176487922668, 0.81572026014328, -0.1265576183795929, -1.381852149963379, 1.3727893829345703, 1.9545189142227173, 0.5533037185668945, 0.5287293791770935, 1.2955718040466309, -0.5156236886978149, -2.4803786277770996, 2.239877462387085, 0.3182165026664734, 1.2269409894943237, -0.7489805817604065, -0.054894253611564636, -0.936576247215271, 0.16487295925617218, -0.26084771752357483, 1.3462554216384888, 0.43664470314979553, 0.5858250856399536, 0.4711885452270508, 0.421930730342865, 1.530349612236023, 0.9067435264587402, 0.08827956765890121, 0.5663931369781494, -1.7432328462600708, 0.7143409848213196, -1.2253438234329224, 1.0970234870910645, 1.0488609075546265, 0.39743441343307495, 0.7509044408798218, 2.1383697986602783, 0.04276273772120476, 1.331987738609314, 1.5810470581054688, 0.9291089773178101, -0.06332943588495255, -0.2695756256580353, -1.431006669998169, -0.6578571796417236, 1.574715495109558, 0.43468788266181946, -0.682368814945221, -0.19026632606983185, 1.537203311920166, -0.022431600838899612, -1.7860360145568848, -0.8693684935569763, -0.32999715209007263, 0.27081483602523804, -1.4720323085784912, -1.4397333860397339, -0.9879745244979858, -0.7085061073303223, 0.19526705145835876, -0.17551200091838837, -1.695974588394165, -0.46424010396003723, 1.7071096897125244, 1.6432483196258545, 1.3027976751327515, -1.723031997680664, -0.6455795168876648, -1.5777307748794556, 0.4340893626213074, -1.2074296474456787, -0.39906200766563416, -1.1413336992263794, -1.4237561225891113, -0.6245747804641724, -2.4309990406036377, -0.08070226013660431, 1.2448196411132812, 0.723296046257019, 0.7996281981468201, 0.33135274052619934, 1.6927976608276367, 0.24507516622543335, 0.540619969367981, 0.2623722553253174, 1.2191996574401855, 3.1411798000335693, 1.5756796598434448, 1.0319881439208984, 1.3956916332244873, -0.5368829965591431, 0.33249780535697937, 0.13092990219593048, -2.52698016166687, -0.11289681494235992, 0.8312215805053711, 1.7938958406448364, -1.1722090244293213, 0.29538458585739136, 0.029767705127596855, -1.6135859489440918, -0.8684678077697754, 1.2714378833770752, 0.1565740406513214, -0.8391764163970947, 0.7517647743225098, -1.553045630455017, 2.1354403495788574, -1.7733906507492065, -1.6253547668457031, 0.9635005593299866, 1.2559349536895752, -0.6728944182395935, 1.2430375814437866, 1.455102801322937, -0.44859394431114197, -0.12440560013055801, 1.819705605506897, 1.0961596965789795, -0.12400002032518387, 0.6746299862861633, 1.148521900177002, 0.9403181672096252, -0.44722825288772583, 0.6436370611190796, -2.644338846206665, 0.44966721534729004, -2.1268045902252197, 0.29203328490257263, -1.2037749290466309, 1.3662300109863281, -0.6426218748092651, -0.3652465343475342, -0.13103917241096497, 0.14327633380889893, 1.0250715017318726, 1.2023673057556152, -0.7196059226989746, 1.9716815948486328, -2.5369961261749268, -0.05127352476119995, -0.024742336943745613, -0.7241564393043518, -1.05683171749115, -1.1052021980285645, 1.6958132982254028, -0.34459188580513, -0.0250233244150877, -0.21428507566452026, -1.3509109020233154, 0.34165310859680176, -1.33259916305542, -0.4511495530605316, -0.656388521194458, -0.5838367938995361, 2.179724931716919, -0.11894246190786362, 0.9962378144264221, -0.7592772245407104, -0.21061304211616516, -0.2937256097793579, -1.8439722061157227, 1.4311937093734741, -1.4096403121948242, -1.0150402784347534, 0.931032657623291, -0.23088598251342773, 0.37818875908851624, 1.3383539915084839, -0.8256304860115051, 0.9161171913146973, 0.5832004547119141, 0.256060391664505, 0.10270020365715027, -0.30729690194129944, -0.49744629859924316, -0.6550679802894592, 0.5865590572357178, 0.6573070883750916, 1.4586213827133179, 0.5730116963386536, 2.025639533996582, -0.8086093664169312, 0.7628703713417053, -0.19966840744018555, 0.12313470244407654, -1.1284010410308838, -0.5444810390472412, 0.0889502689242363, -0.9285518527030945, -0.9776943325996399, 0.4738653898239136, -1.0954880714416504, 1.1919188499450684, -0.12499355524778366, 0.26667043566703796, 1.5195571184158325, 1.6504019498825073, -0.24557970464229584, 0.09745997190475464, -0.6273441314697266, -0.7055547833442688, 0.03782854974269867, 0.8379676938056946, -0.7408170104026794, 0.9649276733398438, 0.8072922229766846, -1.0098211765289307, -0.7826253771781921, -0.453047513961792, -1.4879882335662842, 1.7124323844909668, 0.5901594758033752, 0.7931517958641052, -0.2046758532524109, 0.581095814704895, 0.24373289942741394, 0.5842924118041992, -0.4346661865711212, -1.641431212425232, -0.4251006245613098, 0.7894250154495239, 0.7298871874809265, -0.2608771026134491, -1.0547438859939575, 2.579284429550171, 1.504157304763794, -0.6637396216392517, 1.1509171724319458, -0.41335001587867737, 0.13457979261875153, -0.8757703900337219, -0.30073562264442444, -1.5513598918914795, -0.8767307996749878, 0.9122986793518066, 0.46108293533325195, 1.403337836265564, 2.393333911895752, 0.13994364440441132, -0.1061859056353569, 0.5445466637611389, -0.0012227102415636182, -0.32405221462249756, -1.5380775928497314, 0.2160678207874298, -1.0178396701812744, 0.49058327078819275, -1.043763279914856, -0.9654569625854492, -0.1816125363111496, -0.8394172787666321, 0.09699027985334396, -1.8184518814086914, 0.38273894786834717, -0.6744340658187866, 0.4169130325317383, 0.6304218173027039, -0.7470414638519287, 0.5885933041572571, 0.30263057351112366, 0.5625940561294556, 0.4310297667980194, -0.5729561448097229, -1.0799591541290283, -0.7468481659889221, 0.393271267414093, 0.526016116142273, 0.8039595484733582, -0.21834181249141693, 0.2965557873249054, -0.14309164881706238, -0.5490131378173828, 0.5602738857269287, 0.08921424299478531, -1.3261892795562744, 0.012460357509553432, 0.47229644656181335, -0.8024311661720276, -1.3153886795043945, 0.5091753602027893, -0.1831662803888321, 0.017622804269194603, -0.8919708132743835, -0.4640295207500458, -0.8652534484863281, -0.11812685430049896, -1.8524880409240723, -0.05890501290559769, -1.7538940906524658, -0.20705537497997284, 0.16681183874607086, 0.4594375491142273, -0.6236356496810913, -2.963325262069702, -0.6414172649383545, -0.2596627175807953, 0.5284818410873413, 0.5567244291305542, -1.3473896980285645, 2.4716436862945557, 0.4206402003765106, -0.656828761100769, -0.3929522633552551, 0.6838197708129883, -1.088158130645752, 1.913928508758545, -2.371777057647705, 0.6712661385536194, 2.6985573768615723, 2.3861632347106934, 1.8106184005737305, -1.0419456958770752, -1.146565318107605, 0.6692424416542053, 0.33650314807891846, -0.18301308155059814, 0.5329617857933044, -0.5857495069503784, -1.296748161315918, 0.029941454529762268, -0.21836742758750916, -0.1924978494644165, -0.12909449636936188, 0.723671019077301, 0.09420555830001831, 1.359959363937378, -1.556910514831543, 0.6938773393630981, 0.03415780887007713, 0.04888070374727249, -0.19283241033554077, 2.0640742778778076, 0.08842725306749344, -0.7832372784614563, -0.05021923780441284, -0.4337266683578491, -1.5795464515686035, -0.5841533541679382, -0.19968296587467194, -0.42957329750061035, 0.0139670604839921, 0.8586198687553406, -2.1296474933624268, 0.3215146064758301, -0.27319979667663574, -0.23072342574596405, 1.0752934217453003, 1.4647239446640015, 1.3538520336151123, 0.15606635808944702, -1.739179015159607, 0.22109459340572357, -0.2415715605020523, 1.9804604053497314, 0.1807720959186554, -0.9131194949150085, -0.7652429342269897, 0.46002450585365295, 1.3528478145599365, 2.3334414958953857, -0.6067235469818115, -0.5745665431022644, -1.6738433837890625, -0.24813631176948547, 1.2063844203948975, 0.19394531846046448, 0.5475954413414001, -0.5913664698600769, 0.9833461046218872, 0.3716261684894562, -0.23024487495422363, -0.8286714553833008, 0.2489262819290161, -0.495712548494339, 0.2470443993806839, -0.4009167551994324, 0.28490516543388367, -1.5302934646606445, -0.32447025179862976, -0.22947774827480316, 0.36586254835128784, 1.038977861404419, 0.8154563307762146, 1.0676618814468384, 0.26009148359298706, -0.39008814096450806, -0.3457944691181183, 0.08553972840309143, -0.5759910941123962, 1.1476860046386719, -0.40239182114601135, 1.9372313022613525, 0.13391877710819244, 0.6120815277099609, -0.60227370262146, -0.2408711314201355, -1.2353057861328125, -0.17626953125, 0.9426848888397217, -0.23919875919818878, 2.662780523300171, 0.2868514955043793, -0.8331940770149231, -0.35645678639411926, 2.0466744899749756, -0.1630202829837799, -1.1285760402679443, 1.024397850036621, -1.4418922662734985, 0.2659260928630829, 2.3374979496002197, 0.5638649463653564, 0.1630646288394928, -0.5974726676940918, 0.029572362080216408, -1.0288112163543701, 0.41127675771713257, -1.9777218103408813, 0.05100058764219284, -1.4564862251281738, -1.728301763534546, -0.7352917790412903, -0.45466718077659607, 0.5421484708786011, -0.8294732570648193, 0.40250423550605774, -2.389676570892334, -1.2858526706695557, -0.6621679067611694, -0.34758585691452026, -0.75325608253479, -0.005932093132287264, 0.7424334287643433, 1.275225281715393, 0.1712440699338913, 0.7818110585212708, -0.3794257938861847, 0.8985660672187805, -2.3709402084350586, -0.021221788600087166, 0.7907149791717529, -2.3754124641418457, 1.6341712474822998, 0.20872607827186584, 1.075844168663025, -0.6469795107841492, -0.5877233147621155, -0.7642488479614258, 0.09338968992233276, 1.018196940422058, -0.23268184065818787, -0.8561652302742004, 0.3088011145591736, 0.7381386756896973, 0.39079204201698303, -0.2646547853946686, 1.473747968673706, -2.2497012615203857, 0.2082311362028122, 0.2495240569114685, -1.2688080072402954, -0.2391049563884735, -2.37014102935791, 1.9312477111816406, -0.057772502303123474, -0.26767781376838684, -0.9698829054832458, -1.2149044275283813, 0.28626903891563416, -0.41456127166748047, 0.8686480522155762, 1.3559470176696777, -1.329380989074707, 0.0883767157793045, -1.4730759859085083, 1.1149101257324219, 0.44495296478271484, -1.0721429586410522, 1.1764159202575684, 0.0764671117067337, 1.0136743783950806, 0.5645272731781006, 0.8899354934692383, -0.2217242419719696, -0.8233786225318909, 0.6332885026931763, 0.0034003641922026873, -0.785814642906189, -0.47316521406173706, 0.17614592611789703, -0.0480898916721344, 1.4427831172943115, -1.2917448282241821, -0.10274501144886017, -1.2147806882858276, 0.9710740447044373, 0.1788710653781891, -0.06025725230574608, 1.5858205556869507, 1.6172947883605957, 0.18634648621082306, 0.8845071792602539, -0.9365406036376953, 0.8322022557258606, 0.918319582939148, 1.484855055809021, 0.31384041905403137, -0.47776997089385986, 0.1697363257408142, 0.4984162151813507, -0.18634410202503204, 1.1581343412399292, 1.6349319219589233, 0.674760103225708, 0.17262503504753113, 0.6899327635765076, -0.1857510358095169, 0.34705814719200134, 1.4327260255813599, -1.900682806968689, -0.15940923988819122, 0.5215682983398438, -0.514905571937561, 0.8333770632743835, 0.787413477897644, 0.06341707706451416, 1.5765150785446167, -1.8488376140594482, -0.09145653992891312, -2.142636775970459, -2.0005974769592285, 2.4347479343414307, 1.737721562385559, 0.0828799158334732, 2.2838363647460938, -0.5688346028327942, -3.035841226577759, -0.7391296625137329, -0.09960687160491943, 0.15853776037693024, 0.8307912349700928, -0.0006066359928809106, -1.339414119720459, 0.2617712914943695, 0.598979651927948, 0.3089471459388733, -0.37129878997802734, -0.6536067128181458, -1.0391888618469238, 0.406642884016037, 0.029203006997704506, -1.576974868774414, 0.46457627415657043, -0.12091831862926483, 0.7741330862045288, 1.7878203392028809, -1.3925236463546753, 0.17166486382484436, 0.09162603318691254, 0.2771814167499542, 0.07616417109966278, -0.3150678277015686, -0.231371209025383, 1.5802652835845947, 1.409330129623413, -0.07425133138895035, -0.7368630766868591, -0.9808093905448914, 1.038169264793396, -0.7431752681732178, -0.30934327840805054, 0.29774847626686096, -0.7135605812072754, -0.12179377675056458, 1.7450007200241089, -1.4685630798339844, 0.10234002023935318, -0.011882014572620392, -0.7310032844543457, -0.021495331078767776, -0.844574511051178, 0.06770069897174835, 0.26432403922080994, 0.9836207032203674, 0.6532570123672485, 0.11343898624181747, -0.924254298210144, 0.4042219817638397, -0.667772650718689, -1.7164825201034546, -0.5664599537849426, -0.13622702658176422, 1.530393123626709, -1.3893240690231323, 0.004938052501529455, 1.5666167736053467, 0.9286696910858154, -0.6182101964950562, 0.5445944666862488, 0.17460514605045319, -2.785015106201172, 0.9505500197410583, 0.3566942512989044, 2.5679917335510254, -0.2859390676021576, 1.5792180299758911, -0.4446294605731964, -1.3978307247161865, 0.5899001955986023, -0.524584174156189, 0.6036129593849182, -0.572560727596283, 0.08008679002523422, 0.33914080262184143, -0.8191154599189758, -0.1754644215106964, -1.5688318014144897, -0.5826495289802551, -1.1500564813613892, -0.0740794837474823, -0.07191722095012665, 0.24429352581501007, 0.7391995191574097, -1.0118426084518433, -1.9737013578414917, -0.4120825231075287, 0.5747989416122437, -0.9442527890205383, 0.06564968824386597, 1.9491605758666992, -2.143237829208374, 0.8129788041114807, 0.18622836470603943, 2.0020620822906494, 2.006894826889038, 0.12525425851345062, -1.6228532791137695, -1.0619195699691772, -1.7350229024887085, -0.14096881449222565, 0.7934900522232056, -1.9562805891036987, 0.22986134886741638, -0.1834874004125595, 0.3792808949947357, 0.029927976429462433, 0.28444957733154297, 0.7656733393669128, 1.5038387775421143, -0.468963086605072, -0.22328661382198334, -0.7178621888160706, -0.6166436672210693, 0.16253896057605743, 1.1037991046905518, -1.760172724723816, 2.0575907230377197, 1.0327249765396118, 0.9181694388389587, -1.149466633796692, -3.2494072914123535, -0.05879748612642288, 0.26954275369644165, 0.15993207693099976, -0.14339008927345276, -0.5531888008117676, -0.12497545778751373, -0.772392749786377, -0.4327419400215149, 0.1549273282289505, -0.09774240106344223, -0.7288464307785034, 0.4008428156375885, -0.6498560309410095, -0.2204405963420868, -0.5352771282196045, -1.8977688550949097, 0.18007926642894745, -0.6556957364082336, -0.959153413772583, 0.963610053062439, -0.340954065322876, -0.7529328465461731, -0.4687241315841675, -0.7414798140525818, 0.2213667929172516, 1.7468149662017822, -0.35234904289245605, 0.3970279097557068, 0.3301912546157837, 0.37607231736183167, -0.018520470708608627, 0.33999964594841003, 0.23240423202514648, -0.18111498653888702, 1.6517895460128784, 0.3935370147228241, 1.8630790710449219, -1.061231255531311, -1.2439113855361938, -1.9391196966171265, -0.3852117955684662, 0.7411635518074036, 0.6571813821792603, -1.2144747972488403, -0.6041097640991211, 1.8325343132019043, -0.492311954498291, -1.7598044872283936, 0.025062721222639084, -0.05089113116264343, -0.516399085521698, -0.6862857937812805, -1.5989274978637695, 0.2947709858417511, -0.23710446059703827, 0.0031090322881937027, 1.113874077796936, -0.6970046162605286, 0.3013993799686432, 0.9681863188743591, 0.45237183570861816, 0.012009147554636002, 0.13686725497245789, 0.9577381610870361, -0.2502020299434662, -0.5209578275680542, 0.7818681001663208, -0.7312028408050537, -1.0373995304107666, 1.9369850158691406, -0.13336460292339325, -0.7701277732849121, -0.31989824771881104, -0.5514385104179382, -0.4439601004123688, 0.12441544979810715, 0.041374046355485916, 1.1285918951034546, -0.5003721117973328, 0.5474130511283875, -1.4032853841781616, -0.8561069965362549, -0.4586966633796692, 0.5935057401657104, -1.7184762954711914, -1.018310308456421, 0.45701250433921814, -0.10179735720157623, 0.8436566591262817, 1.01180899143219, -1.369904637336731, 0.5852770805358887, 0.7001991868019104, 2.723362922668457, 0.8831400871276855, -1.280063509941101, 1.5204920768737793, -1.3796497583389282, 0.7470172643661499, 0.44748809933662415, -0.9132682681083679, 0.7114615440368652, -1.225942850112915, 1.8151969909667969, -1.161405086517334, 0.7899780869483948, -1.4286353588104248, -0.6937590837478638, 0.8330516219139099, 1.6572072505950928, -0.7731272578239441, -0.7661756277084351, -0.7617485523223877, 0.1954752653837204, -0.7607282400131226, -1.0012965202331543, 0.7644417881965637, 0.30489382147789, 0.31904730200767517, 0.585644543170929, 1.4168822765350342, 0.7678791880607605, -0.16243287920951843, 1.0985246896743774, -0.15492001175880432, -1.0989636182785034, -0.003122285706922412, 0.6465703248977661, 0.5733794569969177, 0.11260979622602463, 0.030574316158890724, -0.525424599647522, 1.192459225654602, -0.4255668818950653, -1.324546456336975, 0.4142161011695862, -0.07491028308868408, -2.3962254524230957, -0.5628560185432434, -2.63250994682312, 0.4658830463886261, -1.4923560619354248, -2.4705586433410645, 0.9404280781745911, 1.4522125720977783, -0.3375212848186493, -0.03081212006509304, 0.6481285095214844, 0.44259053468704224, 0.37561190128326416, -0.21357080340385437, 0.8994858264923096, 0.39875608682632446, 0.23014289140701294, -0.49704477190971375, 0.08763016015291214, 1.254669189453125, -2.2113327980041504, 0.5828016400337219, -0.3526920974254608, -1.0664417743682861, -0.8157774209976196, 0.1183919683098793, 0.16801869869232178, 0.2526899576187134, -0.970352292060852, 0.12061161547899246, 1.2609257698059082, -1.1819260120391846, -0.4269263744354248, -1.3156580924987793, 1.1517435312271118, -1.1583102941513062, -1.2865211963653564, -0.5247215032577515, -0.04925612732768059, -0.17219841480255127, -2.3635566234588623, 0.1095554456114769, 1.4675815105438232, -0.267932653427124, 1.3064438104629517, -0.5637697577476501, 0.6766629219055176, 1.682034969329834, 1.2478095293045044, 1.8535351753234863, -0.43611568212509155, -0.5513636469841003, 1.1733890771865845, 0.5942509770393372, 0.36492377519607544, 1.7850998640060425, 0.9068421125411987, -0.729241132736206, -0.9371416568756104, 0.7688467502593994, -0.0796077772974968, 0.7138862013816833, 2.9107918739318848, -0.5480113625526428, 1.0142446756362915, -0.20014260709285736, 0.595834493637085, -0.41430601477622986, -1.351016879081726, -0.4573671519756317, 0.021555539220571518, 0.9021198749542236, -0.798494279384613, 1.181386113166809, -0.09167028218507767, 0.7565574645996094, -1.141211986541748, -0.9018226265907288, 0.16260525584220886, 1.5042353868484497, 0.2432677298784256, 0.9052488207817078, 0.23388107120990753, -0.47376951575279236, 0.6512406468391418, 0.3202108144760132, -0.23784565925598145, 0.000325436849379912, -1.782692790031433, 1.9290492534637451, 0.9167464375495911, -0.003970969934016466, 0.3846230208873749, 0.12758636474609375, 0.9888551235198975, 0.5231887698173523, -0.7001231908798218, 0.8204972147941589, -1.690019130706787, 0.15242834389209747, 0.6728619933128357, -0.781061589717865, -0.11693032830953598, -0.553499698638916, -0.0634317547082901, 1.2444894313812256, 0.6854956150054932, 0.5480302572250366, 1.1079789400100708, -0.41294005513191223, -1.0386475324630737, 1.606763482093811, 0.3378181755542755, 0.3012841045856476, -0.9531718492507935, -1.5028401613235474, 0.6845787763595581, 0.5482251644134521, -1.076587200164795, -1.1570638418197632, -0.9388041496276855, -0.3007437586784363, -0.5767609477043152, 0.4894999563694, -0.24919921159744263, -0.16501496732234955, 0.7857937216758728, -1.3795677423477173, -0.03896878659725189, 1.4614843130111694, -1.0914227962493896, 0.4570541977882385, -0.6530608534812927, -0.21060174703598022, -0.17026375234127045, 0.7676213979721069, -1.9762142896652222, -1.3594518899917603, -1.0977706909179688, 0.416191041469574, -0.846185564994812, -0.6627853512763977, 0.018961824476718903, -0.6031431555747986, 0.18110616505146027, 1.6279350519180298, -0.7364009618759155, -2.1029295921325684, -0.5758081078529358, 0.11157165467739105, 0.9100534915924072, -0.2665588855743408, -0.4510990381240845, -0.6460066437721252, 0.3849033713340759, 0.5937716960906982, 0.46151188015937805, 1.1653704643249512, 0.19811977446079254, -0.6647701263427734, 1.0615417957305908, -0.0193767286837101, -0.24978744983673096, 0.22509528696537018, 0.36382701992988586, 0.24983343482017517, -0.060133229941129684, 0.6286330223083496, 0.5287817716598511, 0.6292487382888794, 1.4054315090179443, 0.950039267539978, -0.8025496006011963, -0.9755847454071045, -0.5626087188720703, 0.8995797634124756, 0.26263415813446045, 0.5810258984565735, -0.21785254776477814, -0.6771411895751953, -0.7802140116691589, 1.1938183307647705, 0.28866878151893616, -0.31536996364593506, 0.4412821829319, -0.2987002730369568, -1.266675353050232, -0.3739895224571228, -0.4809384346008301, -0.01266664918512106, 0.417388379573822, 0.3143511414527893, -1.0483338832855225, 0.4244981110095978, -0.6869203448295593, 0.02911841869354248, -0.15590722858905792, -0.019122498109936714, -0.9190529584884644, 1.1371411085128784, -0.1746329814195633, 0.28938740491867065, -2.1945464611053467, -0.09662532061338425, 0.47020846605300903, -0.47448039054870605, 0.47933557629585266, -0.522427499294281, 0.24573400616645813, -0.5595825910568237, 0.09517033398151398, -0.7895780801773071, -1.1227152347564697, -0.615713894367218, 0.669506847858429, -2.2284176349639893, -0.3171105682849884, 0.22929388284683228, 0.9261288046836853, 0.14088352024555206, 0.3421637713909149, -0.25179895758628845, -1.2427387237548828, 0.1459212303161621, 1.004112958908081, 0.9046658277511597, -0.08543451130390167, -2.3763999938964844, -0.34634390473365784, -0.785526692867279, -0.13062697649002075, -1.4381506443023682, -0.21292340755462646, -0.05215630307793617, -0.17122119665145874, 1.3324720859527588, 0.33482885360717773, -1.6897376775741577, 0.3649146854877472, -1.8774374723434448, -0.9072301387786865, 0.4594476521015167, 0.701066255569458, -1.9460521936416626, 0.8744785785675049, 0.7936697602272034, -0.3329978883266449, -0.6315682530403137, -0.26159265637397766, -0.34237751364707947, 0.48710915446281433, 2.232165575027466, 0.5892630219459534, 0.08281468600034714, -1.4743874073028564, -1.6140198707580566, 0.5520848035812378, 1.0104143619537354, -0.7580298781394958, 0.8191595673561096, 0.839734673500061, 0.3516407608985901, -1.247013807296753, -1.4170737266540527, 2.0028724670410156, 0.19196657836437225, -0.8013191223144531, 0.35516297817230225, 0.1842924803495407, 0.714101254940033, 0.18806211650371552, -1.353537678718567, 0.9888126254081726, 0.38897472620010376, -0.23329688608646393, -0.9427207112312317, 0.4638056755065918, 0.33477333188056946, 1.4339686632156372, 1.1502118110656738, 0.3611631989479065, -0.3485717177391052, -1.5840349197387695, 1.4086159467697144, 0.5788040161132812, 0.6923450827598572, 1.0224947929382324, -0.4679862856864929, 0.5512080788612366, -0.7028889656066895, 0.9218918085098267, -0.7665977478027344, 0.7635779976844788, 1.1344077587127686, 0.34695905447006226, 2.3760483264923096, 2.014193058013916, 0.26388898491859436, -0.3904121220111847, -1.094486117362976, -0.9073835611343384, -0.3214135468006134, -0.5517856478691101, -1.446952223777771, -1.3618707656860352, -0.09040320664644241, -0.1995219588279724, -0.2211640328168869, -1.2798244953155518, -0.6774054765701294, 0.6021679043769836, 0.6117214560508728, 0.2348897010087967, -1.3417329788208008, -0.1302412897348404, -1.0967745780944824, 0.44132179021835327, -1.4108730554580688, -0.2698957324028015, -0.2964222729206085, -0.20824746787548065, 0.07667584717273712, -0.0947781577706337, 1.2218921184539795, 0.10117389261722565, 0.6556726098060608, -1.8804478645324707, 0.02709304168820381, 1.0556994676589966, -0.7631692886352539, 0.26003125309944153, -0.34917211532592773, 1.9828310012817383, 0.16195964813232422, -1.5799927711486816, -0.7274214029312134, 1.1483310461044312, -0.011208332143723965, 0.7201072573661804, -0.8251707553863525, -0.723888099193573, -0.4574258029460907, -1.0788354873657227, -0.553374707698822, 0.23039007186889648, -0.39479607343673706, 1.4895013570785522, 2.1748766899108887, -0.06864111870527267, 0.011272190138697624, 0.8383949398994446, -0.35551995038986206, -2.189669609069824, -0.4275627136230469, 0.2951117157936096, 0.7651837468147278, -0.8732216954231262, -0.9847422242164612, 2.99141526222229, -1.3720571994781494, 0.7063950896263123, 0.40938600897789, -1.4049296379089355, -1.0422403812408447, -0.033121973276138306, -0.1784835159778595, 1.0966832637786865, -1.1025364398956299, 0.565775454044342, -0.7807446718215942, -0.1403256207704544, -0.6738017201423645, -2.311913251876831, -0.03136012703180313, 0.7451344132423401, -0.49170008301734924, -0.2222529947757721, -0.8838133811950684, 1.3573262691497803, -0.7442035675048828, 1.8127355575561523, -0.6146275401115417, -0.9979702830314636, -0.24551257491111755, 0.05755831301212311, -0.5195554494857788, 0.02411344274878502, 0.2850934863090515, -1.067457675933838, 0.5530588626861572, -0.861392617225647, 1.1443672180175781, -1.264551043510437, -0.7164291143417358, -0.7865820527076721, 0.22050201892852783, 0.20772209763526917, -0.04093248024582863, -1.9974110126495361, 1.6620689630508423, -0.671704113483429, 1.3864625692367554, -0.791439414024353, -1.4489495754241943, 0.2378469556570053, -1.6520936489105225, 1.5622996091842651, -0.49236854910850525, -0.978744626045227, 1.8616600036621094, 1.2215423583984375, 0.18222467601299286, 2.973719358444214, -1.0828473567962646, -0.48272788524627686, 0.09405705332756042, 0.3874959647655487, -0.9293912053108215, -1.1025604009628296, -0.7692615389823914, 1.4449076652526855, -1.013420820236206, -0.5768575668334961, -0.172601118683815, 1.0633301734924316, -0.7367687225341797, -0.9363442063331604, -1.1314575672149658, -0.258293479681015, 0.017316704615950584, -0.35949498414993286, 1.850545883178711, 0.8796143531799316, -0.6425577402114868, -0.37663689255714417, -1.471815586090088, -0.5801689624786377, -1.6932411193847656, 0.8913574814796448, -1.8648971319198608, 0.6099178791046143, 1.464645504951477, 1.1755974292755127, 0.4189572334289551, 0.9114553332328796, -0.8529674410820007, -0.7936775088310242, 1.5447633266448975, -2.495480537414551, -0.22002620995044708, -0.2786389887332916, -1.1755833625793457, 1.8942140340805054, 0.3690958023071289, -0.6696709394454956, -0.1356315314769745, -0.2907969653606415, -0.29175814986228943, -0.2133411467075348, 0.7769401669502258, -0.4229593276977539, -0.4598398208618164, -0.0663331151008606, 0.49303221702575684, -0.5092114806175232, 0.6024105548858643, 0.9552678465843201, -0.3699265122413635, 1.2928404808044434, -0.4819813370704651, -0.35488176345825195, -0.7651288509368896, 0.07684279978275299, -0.04495525360107422, 0.917159378528595, -0.8453361988067627, -1.2554078102111816, 1.358611822128296, 0.2965792417526245, 0.0246884822845459, 0.005003064870834351, 0.1251998096704483, -0.4308585524559021, 1.5964139699935913, 0.2694508135318756, 0.08555315434932709, -0.5955447554588318, -0.4891480803489685, 0.036951709538698196, -0.10546348243951797, -1.107814073562622, 0.09131158143281937, -1.4103411436080933, -1.444182276725769, -0.7685914635658264, 0.05406932532787323, -0.033165667206048965, 1.5880911350250244, -0.9115713834762573, -0.6549336910247803, -0.17807215452194214, -0.3433752954006195, -1.0555334091186523, -1.0320545434951782, -1.1918126344680786, 0.7441543936729431, -0.2515922486782074, 0.9963705539703369, -0.5175154209136963, 0.5349757671356201, 0.17550253868103027, -0.20117062330245972, -0.5589704513549805, 0.9605915546417236, -0.16206589341163635, 1.341579556465149, -0.38686707615852356, 1.278400182723999, -0.6026902198791504, -0.8764503598213196, 1.2657935619354248, -1.445470929145813, 1.8241748809814453, -0.5485360622406006, -0.1412011682987213, -0.7837560772895813, 1.196429967880249, -0.5518760085105896, -0.9272716045379639, -1.7734416723251343, 0.7258908152580261, 1.0109996795654297, 0.8034696578979492, 0.5637671947479248, -1.7639838457107544, -0.43647274374961853, 0.4335942566394806, -0.7739510536193848, -1.3440072536468506, -0.4707098603248596, 0.45860493183135986, 0.24594120681285858, -1.7257986068725586, -0.0333445742726326, -0.3249432146549225, -0.4319634437561035, 1.162879228591919, -1.7932296991348267, -0.03517867624759674, 0.9573215842247009, -0.6682486534118652, 0.5931817293167114, -0.09409494698047638, -0.657837986946106, 0.07976195216178894, -0.15113262832164764, 0.4954357147216797, 1.5688031911849976, 0.8024213910102844, -0.38674110174179077, 1.249667763710022, -0.15942226350307465, 2.0546188354492188, -0.5486004948616028, 0.7273000478744507, -1.1552801132202148, -0.22783774137496948, -0.7881777286529541, 0.8531097769737244, -1.5439550876617432, 1.0122554302215576, 0.21329720318317413, -0.8056253790855408, -0.1826084405183792, 1.0051027536392212, -2.664778709411621, 0.6840790510177612, -1.369778037071228, 0.09292100369930267, -0.3979294002056122, -1.0893394947052002, -1.703011393547058, 0.7796986699104309, -0.5217834711074829, 0.6621214151382446, -1.146674633026123, -1.0078809261322021, 1.0152500867843628, -0.17279474437236786, 0.10447390377521515, 0.3265632390975952, 0.2321011871099472, -1.6514893770217896, -1.744554042816162, -0.34685009717941284, -0.5696914792060852, -0.3707250952720642, -0.1811974048614502, -0.7439920902252197, 0.14424356818199158, -0.39901918172836304, -0.3080114722251892, -0.8355510830879211, 1.0357604026794434, -0.7060827016830444, 0.5956608653068542, -0.2757035195827484, 0.23725175857543945, 0.7305553555488586, -0.5145041942596436, -0.4174298346042633, -0.44211849570274353, 1.0766772031784058, -2.4208462238311768, -0.7566994428634644, -0.09227985143661499, 1.4038499593734741, -2.164738655090332, -0.10041666030883789, -0.8266683220863342, 1.1197909116744995, -1.523041844367981, 1.0347278118133545, -0.8137195706367493, -1.4461830854415894, -1.246850609779358, 0.6784278154373169, -0.9388551115989685, 1.637532353401184, 0.7357922196388245, 1.2958776950836182, -0.574380099773407, 0.4968429207801819, -1.6850746870040894, -0.08187498152256012, -1.9541667699813843, -0.3258974552154541, -0.45005348324775696, -0.7578790187835693, 0.6780298948287964, -0.22710587084293365, -0.5100646615028381, 1.4610649347305298, -0.27682095766067505, -0.4866860508918762, -0.047287795692682266, -0.4524626135826111, 2.2696890830993652, -0.6769306063652039, 0.39615002274513245, -0.3394833207130432, -0.07880731672048569, 0.18558356165885925, -0.5936065316200256, -0.394449383020401, -0.17482149600982666, 0.7647644877433777, -1.791564702987671, -1.1241681575775146, -1.7049920558929443, -0.20101162791252136, -0.19238314032554626, -2.018630266189575, 0.2481495589017868, -3.809117317199707, 0.15271854400634766, -0.4222782254219055, 0.19052866101264954, -0.35711419582366943, 1.5512312650680542, 1.9062511920928955, -1.029457688331604, 0.8622057437896729, 0.2748768627643585, -1.454322099685669, -1.3462473154067993, 0.49249520897865295, 1.0766842365264893, 0.7165787816047668, 0.07885375618934631, -0.6382157206535339, 0.33475667238235474, 0.3832450807094574, -1.3069839477539062, 0.404471218585968, 1.1833614110946655, 0.7505167722702026, 0.11396247893571854, -1.1496700048446655, -1.6639761924743652, -0.5446133613586426, 1.3556667566299438, -0.7327044010162354, 1.7408860921859741, -1.678847312927246, 1.1810678243637085, 0.1301787942647934, 0.4173625111579895, -2.292407274246216, 1.420276403427124, 0.4598633944988251, -0.28367578983306885, 0.10663367807865143, 1.1504242420196533, 1.3610973358154297, 0.8565553426742554, -0.77271568775177, -0.2482287436723709, 0.09139437228441238, 0.31730061769485474, -1.3237428665161133, 0.36794373393058777, -0.8715896010398865, 0.15689054131507874, -0.177633136510849, -0.06662964075803757, -0.45427563786506653, -0.7195484638214111, 1.1856030225753784, -0.08885759860277176, -0.09607429802417755, -0.7963846921920776, 0.03312671557068825, -0.05600885674357414, -1.4797438383102417, -0.08211544901132584, 0.031936176121234894, -0.9587777853012085, -0.3894081711769104, -1.343251347541809, -0.6099766492843628, 0.5172646045684814, -0.8787668347358704, 0.8263413310050964, -2.2495856285095215, -0.6576688885688782, 1.1908888816833496, 1.0575331449508667, 0.43022337555885315, -0.9395313262939453, 0.6312812566757202, -0.24485595524311066, -1.6565991640090942, 0.7868642210960388, -0.8926425576210022, -0.12687155604362488, 0.8162994384765625, 0.25796085596084595, -1.4755258560180664, -1.207516074180603, -0.02054310403764248, -1.0646235942840576, -0.004325147718191147, -0.4845488667488098, -1.239853858947754, -0.711391806602478, -0.44741034507751465, 0.4323626160621643, -1.4019018411636353, -0.5246163010597229, -1.5874706506729126, 0.7133932113647461, -1.3680764436721802, 0.5425982475280762, -1.1945796012878418, -0.4315468370914459, 0.28129255771636963, 1.22930908203125, 0.634201169013977, 1.7921186685562134, -0.8907065391540527, 0.1617072969675064, -1.6246308088302612, 0.18490839004516602, -0.00717924302443862, -0.13466419279575348, -0.16948199272155762, 0.15198162198066711, 0.39057573676109314, 0.48161929845809937, -1.5009583234786987, -0.27936190366744995, 0.3061217665672302, -1.393025279045105, 1.4310972690582275, -1.036462664604187, 0.5869665145874023, -0.7610134482383728, -0.7238035798072815, -0.12796960771083832, 0.10317671298980713, -2.391676425933838, -0.5269520878791809, -0.1389535516500473, 1.801391363143921, 1.3357149362564087, -0.4924755096435547, 0.8382512331008911, -0.2356957495212555, 3.400486707687378, 0.37789294123649597, 0.9246858954429626, -0.04344688355922699, 0.02846503257751465, 0.9331222176551819, 0.5747090578079224, -0.5862610936164856, 0.06933874636888504, 0.01387182530015707, 0.4669593274593353, -2.34216570854187, -1.6723577976226807, -2.872746229171753, 0.47856512665748596, -1.3469038009643555, 0.7451106905937195, 0.46406495571136475, 0.8928242921829224, -0.0620056688785553, -0.8865812420845032, -0.5797040462493896, -2.242324113845825, 1.6719541549682617, -0.2834058701992035, 0.16290107369422913, 0.8247731328010559, 0.47243499755859375, -0.7287535667419434, -0.2735415995121002, 1.38186514377594, 1.2136805057525635, 0.4022444784641266, 0.5080950856208801, -1.6389127969741821, -1.3098630905151367, 0.29983198642730713, 1.703364372253418, -1.8076056241989136, -1.0374641418457031, 0.7051450610160828, 1.5059826374053955, 1.3846830129623413, 1.0860601663589478, 0.44968053698539734, 0.65486741065979, 0.09096678346395493, 0.998586893081665, 0.43659794330596924, -0.6210398077964783, -1.1023128032684326, -0.7113890051841736, -1.1546566486358643, -1.591813325881958, -0.5874257683753967, -0.0004576125356834382, -0.04411561042070389, -0.1771448850631714, 1.0140725374221802, 2.031836986541748, 0.18001216650009155, -0.21116960048675537, 0.865705132484436, -1.5238258838653564, 1.7680236101150513, 1.035024881362915, 1.036784291267395, 0.34211769700050354, -0.8580604195594788, -0.49595674872398376, 1.1182565689086914, -1.3145487308502197, -0.22306375205516815, -1.158706784248352, 0.20651376247406006, 1.1919738054275513, -0.014499506913125515, 0.8242668509483337, -0.4994003176689148, -1.350422978401184, 0.549964427947998, 0.653171718120575, 0.7371259331703186, 0.42201223969459534, -1.0255846977233887, 0.4134409427642822, 0.7402939200401306, -1.1065665483474731, 0.8980203866958618, 0.8289914727210999, -1.3915014266967773, 0.1445278823375702, 0.011008537374436855, -0.028621064499020576, 0.7052134275436401, -0.5089665651321411, 0.04490673542022705, 1.2231826782226562, 0.479824036359787, 0.16366052627563477, -0.189695805311203, -0.35701122879981995, -1.3605926036834717, 1.3843328952789307, 1.803776741027832, 2.077179193496704, -1.487169623374939, -1.3811508417129517, 1.325475811958313, -1.1749569177627563, 1.2235751152038574, -0.40977993607521057, -0.10540466010570526, 0.9174689650535583, 0.2353219985961914, 0.5106361508369446, 0.8680224418640137, 0.5045236945152283, 1.3730340003967285, -0.3330589532852173, 1.2277207374572754, -0.8598037362098694, -0.647697925567627, -0.43395093083381653, 1.2928272485733032, 1.330506443977356, -1.3189314603805542, -1.516236662864685, 1.5737556219100952, -1.224233627319336, -0.6716104745864868, 0.921168327331543, -1.0873067378997803, -0.5055004358291626, 0.8644704818725586, -0.786962628364563, 0.056460071355104446, -0.5844568610191345, -0.9682497382164001, -0.3961603045463562, 0.358628511428833, 0.5181295871734619, -1.3987884521484375, 0.09913824498653412, 0.7308669090270996, 0.1291475147008896, -0.5026664137840271, 1.0886380672454834, 0.9097041487693787, 0.5905003547668457, -1.7742315530776978, -0.8643094897270203, 0.034189749509096146, 0.3433375656604767, -1.226148009300232, 1.4696452617645264, -0.6679414510726929, -1.280621886253357, -0.6600788235664368, -0.9810835123062134, 0.14963430166244507, -1.645331859588623, 0.13014230132102966, -0.16265536844730377, 0.3450281620025635, 0.055906906723976135, 0.6999649405479431, -0.2589961886405945, -0.5679763555526733, 0.6727419495582581, 1.7267364263534546, 1.2609000205993652, 0.3843464255332947, -0.13771066069602966, -1.7625590562820435, -0.6592172384262085, 2.181029796600342, -0.17414626479148865, -0.9242054224014282, -1.7138376235961914, 0.043082304298877716, -0.36552464962005615, 0.07516901195049286, 0.147819384932518, -2.6633098125457764, -0.3299063742160797, 0.7581389546394348, 0.48293134570121765, -0.6079745292663574, 0.7215160727500916, -0.8757839798927307, -0.3003609776496887, -1.031408429145813, -2.325796127319336, -0.503933846950531, -0.22516503930091858, 0.017571303993463516, 1.2629106044769287, 1.5450021028518677, 0.4248906970024109, -0.5327705144882202, -0.5103147029876709, -0.987498939037323, -0.5836338400840759, 1.6889432668685913, -0.7365595698356628, -1.1901358366012573, -1.5632420778274536, 0.34635645151138306, 1.3768138885498047, -0.44615480303764343, -1.194254755973816, 0.5972875356674194, -0.7695639133453369, 0.43823114037513733, 0.4105350971221924, 0.547455370426178, 0.6908671259880066, -0.13708525896072388, -1.7360297441482544, -0.054831162095069885, -0.8086007237434387, 1.8991708755493164, -0.05029325559735298, 0.4344256520271301, -2.036261558532715, 1.5943775177001953, -1.3784195184707642, -0.8395110368728638, -1.4975721836090088, -0.22253617644309998, 1.2445660829544067, 0.29997122287750244, 0.015720374882221222, 0.23859643936157227, -0.35468024015426636, -0.9263056516647339, 0.23626603186130524, -2.1178319454193115, 0.6362844109535217, -1.2127892971038818, 0.46021145582199097, -1.1217305660247803, -0.19295687973499298, 0.8650444746017456, -0.8877149820327759, -0.16047731041908264, -0.8809875845909119, 0.46415162086486816, 1.1464511156082153, -0.9526111483573914, 0.25499820709228516, -1.4902945756912231, 1.1166647672653198, 0.9475147128105164, -0.24358494579792023, 0.9309899210929871, 0.5519481301307678, -0.3734052777290344, 0.14210863411426544, -0.6332988739013672, -0.6621935367584229, -1.1967755556106567, 1.4593234062194824, 1.428113579750061, 0.27491751313209534, -0.6636807322502136, -0.3038772940635681, -2.099912166595459, 0.022901834920048714, 0.08937613666057587, 0.5840211510658264, -1.1283574104309082, 0.8820125460624695, -0.5738637447357178, -0.3382219672203064, -0.7954604625701904, -0.33115673065185547, -0.9874895215034485, 1.1229681968688965, -0.5938582420349121, 1.0903353691101074, 0.7481397390365601, -0.8178576231002808, 1.0373561382293701, -1.3587309122085571, 0.2368605136871338, 2.3225975036621094, 0.4310978949069977, -0.23884932696819305, 1.8228812217712402, -0.1939612776041031, -0.40032854676246643, -0.24096553027629852, -0.7527142763137817, 1.9458109140396118, 0.33277493715286255, -0.5928913950920105, 0.9445247054100037, 0.8319065570831299, -0.6601167917251587, -0.11795146018266678, -1.5367923974990845, -0.45514270663261414, -0.8466752171516418, 0.3740283250808716, 0.6610342264175415, 0.8758078217506409, -1.6369460821151733, -1.0178661346435547, 2.6703684329986572, -0.019373642280697823, 0.6087024807929993, 1.1991513967514038, 0.6261922717094421, -1.0882564783096313, -0.2541015148162842, 1.2718722820281982, 0.9577218890190125, -0.6636751294136047, 0.5621768832206726, -0.12894800305366516, -0.9683475494384766, -0.7417566180229187, 1.5438734292984009, -0.052005454897880554, 1.0063320398330688, 0.2761659026145935, 1.0389902591705322, 0.7668874859809875, 0.24194903671741486, 0.24575529992580414, 1.9190839529037476, 1.2268766164779663, 1.032820224761963, 0.7067254781723022, -2.2287895679473877, 0.17130596935749054, 1.01729154586792, -2.341944694519043, 0.3614867925643921, -1.24153733253479, 1.331173062324524, -0.9395949244499207, 0.8286852836608887, -0.16448454558849335, 0.35629627108573914, -0.2779773771762848, 0.2137337177991867, 0.34362682700157166, 0.11256644129753113, 2.604094982147217, -0.1492180973291397, -0.15311579406261444, -0.3932672142982483, -0.1984211504459381, -1.4250524044036865, -0.9602113962173462, 0.9488764405250549, -0.3789609968662262, 0.5947715640068054, 0.3212582468986511, -0.7835723161697388, 1.806039571762085, -0.4736016094684601, -0.3971288502216339, 0.594897985458374, -0.9461533427238464, 0.7541803121566772, 1.2745004892349243, 0.9381309151649475, -0.7463558316230774, 1.2906813621520996, -1.0136425495147705, 0.8916951417922974, -1.0789556503295898, 0.903035581111908, 1.512343406677246, 1.038976788520813, 0.28333622217178345, 0.2095136046409607, 1.014339804649353, 1.2657171487808228, -0.6683713793754578, 0.9751946330070496, -0.8030964732170105, 1.4287185668945312, -0.4366994798183441, 0.08416809886693954, 0.12482795864343643, -1.325566053390503, -0.5454687476158142, 1.4093444347381592, -1.4211149215698242, 0.10524599254131317, -1.5739705562591553, 0.40478581190109253, 0.10969194769859314, 1.3418989181518555, 1.0478200912475586, 1.2947006225585938, 0.09723632782697678, -2.3192903995513916, -1.3302617073059082, 0.19567471742630005, -0.3630955219268799, 0.5148137807846069, -1.2635979652404785, 0.6830624341964722, 0.344197541475296, 0.5235785245895386, -0.8432947397232056, 0.511587381362915, 2.0855021476745605, -0.35640251636505127, -1.16322922706604, 0.5426862239837646, -0.22072243690490723, 1.5263196229934692, -0.19090624153614044, -0.6171966791152954, 1.4894030094146729, 0.45605939626693726, -0.3302406072616577, 1.9406640529632568, -0.8890125155448914, -1.1608424186706543, -1.3275105953216553, -0.8990734219551086, 0.8619444966316223, -1.0523934364318848, -0.17261402308940887, -1.00554621219635, 0.9896649122238159, -0.06871621310710907, -1.0733942985534668, 2.092400312423706, -0.694130539894104, -1.0378525257110596, -1.76430344581604, 0.9129244089126587, -1.186490535736084, 0.36671897768974304, -0.5503106117248535, 0.4325801134109497, -1.1851786375045776, 1.0432897806167603, 1.720361590385437, -0.6013774275779724, 0.4052431583404541, -1.0182251930236816, -0.0659482479095459, 0.6914497017860413, 0.4281163811683655, 0.09383188933134079, 0.4464952349662781, 0.10995455086231232, -0.9274221658706665, -0.8784857988357544, 0.05229197442531586, -0.8884113430976868, 0.6768172383308411, -0.5891937017440796, 0.7248321771621704, 0.4093934893608093, -0.3206786811351776, 0.37010204792022705, -0.7080976963043213, 0.11347728967666626, -0.39810284972190857, -0.08254837989807129, -1.432356357574463, 1.336087703704834, -0.31531020998954773, 0.5154381394386292, 1.3795814514160156, 0.6142758131027222, -3.313021659851074, -1.1853121519088745, -0.7625851631164551, -1.0104472637176514, -1.135384202003479, -2.1170122623443604, -2.2862770557403564, -0.7417058348655701, 0.06796308606863022, 0.34955891966819763, 0.08537056297063828, 0.2638549506664276, 2.2299153804779053, -0.40118762850761414, -0.6859414577484131, -1.0584396123886108, 0.704802393913269, 0.16479763388633728, -1.1847678422927856, -0.340285062789917, 0.5241720080375671, 0.43495145440101624, -0.018633846193552017, -0.46894386410713196, 0.44614267349243164, -0.10808620601892471, -1.6443185806274414, -1.494685411453247, -0.8110698461532593, 1.1227777004241943, 1.831242322921753, -0.6707407236099243, -0.8844414949417114, 0.40240412950515747, 1.1851831674575806, -0.05392790585756302, 0.6531047821044922, -1.088395357131958, 0.35955744981765747, -1.585461974143982, 1.7296655178070068, 1.2908121347427368, -0.7474086284637451, 0.3567984700202942, 0.3700371980667114, -0.26529020071029663, -0.8307801485061646, 0.2558799684047699, 2.002408742904663, -0.08009612560272217, 0.26693546772003174, -1.052765130996704, -2.2913639545440674, -0.9522417783737183, 1.476887583732605, -0.3159011900424957, 0.4630014896392822, -0.28536662459373474, 0.5675001740455627, -0.2765454947948456, 0.49922341108322144, -0.9068674445152283, 1.5033442974090576, -1.6082546710968018, -0.5269213914871216, 1.48269522190094, -1.4384748935699463, -0.4386153221130371, -0.29773643612861633, 1.2442618608474731, -0.21020227670669556, -0.31944432854652405, 1.0168211460113525, 1.053748369216919, -1.4571335315704346, 0.4805772006511688, 1.4949817657470703, -1.8573219776153564, 0.696325957775116, -2.493682622909546, -0.9419079422950745, 0.04762493073940277, 1.0204631090164185, 0.029638255015015602, -0.3490374684333801, -0.3215996325016022, -0.7047531604766846, -1.1211618185043335, 0.17249450087547302, -1.6185716390609741, 0.7365900278091431, 0.08053752034902573, -0.41958820819854736, 0.9996461272239685, -0.8716509342193604, -0.31189635396003723, 0.4271847903728485, -1.0328640937805176, -0.21083147823810577, -1.4413505792617798, -0.5518928170204163, -1.950844645500183, 0.4839615821838379, 1.0655604600906372, 0.36791640520095825, -1.6958082914352417, -0.8163912296295166, 0.7436467409133911, 0.15013986825942993, -0.634486973285675, -0.18718981742858887, -1.0292601585388184, 0.4131343364715576, 1.6634926795959473, -0.8224222660064697, 0.5419327616691589, 1.4006246328353882, -0.5655835866928101, -0.12941761314868927, -0.6631492376327515, -0.6470106840133667, 0.5238497257232666, 0.08917499333620071, -0.14218153059482574, 2.232619047164917, 0.875766932964325, 0.7423262000083923, 0.8924974799156189, 1.8929986953735352, 0.7471731305122375, 0.604722261428833, -0.9321962594985962, 0.4789436459541321, 1.0652915239334106, 0.8401144742965698, -0.33930471539497375, -1.0454933643341064, -1.465837001800537, -0.06682828068733215, 0.6281542181968689, -0.3703373372554779, 0.7385886907577515, 0.4776242971420288, 1.2751977443695068, 0.7771610021591187, 0.4138912260532379, 1.199659824371338, 0.18532714247703552, -0.6401711702346802, 0.5085151195526123, 0.5751602053642273, -0.8119139671325684, -0.6519144177436829, 1.3833125829696655, -1.876417636871338, 1.1742407083511353, 0.0645105242729187, -0.8171104788780212, 1.9659178256988525, 0.9101942181587219, 0.892232358455658, -2.572327136993408, 0.6431794762611389, -0.10978922247886658, 0.6324506402015686, 0.8346311450004578, -1.409623384475708, -0.1704712063074112, 0.6350814700126648, 1.034769058227539, -0.2891460061073303, -1.299887776374817, -1.6703506708145142, 0.02245170623064041, -0.2175157368183136, -1.1526033878326416, -0.5351434350013733, 0.1292576640844345, 0.5912965536117554, -0.8182837963104248, -1.1822543144226074, 0.7470630407333374, 0.2871741056442261, -0.9330024719238281, -0.22962462902069092, 0.6403096318244934, -0.19492731988430023, -0.4516073167324066, 0.21590012311935425, -0.454753041267395, 2.2213637828826904, 0.5082467794418335, 0.418098509311676, -0.21923229098320007, 0.19522225856781006, 0.4541206359863281, -0.3005109429359436, -1.2256098985671997, 0.309408038854599, -0.005370514001697302, -0.1484561562538147, 0.2975535988807678, -0.5075845718383789, -0.739845335483551, 0.2210530787706375, 1.4405509233474731, -0.9015283584594727, -0.9504377245903015, -0.49850431084632874, -0.8208052515983582, -0.43906930088996887, -1.0039199590682983, 1.4630374908447266, 1.7120310068130493, -0.020431431010365486, 1.1478246450424194, -0.37136322259902954, 1.0094704627990723, -0.45444175601005554, 0.41978976130485535, -0.8327195644378662, -0.33670902252197266, 0.04671413451433182, -0.9403664469718933, -1.7067465782165527, -0.15028558671474457, 0.21699534356594086, -0.3928467035293579, 0.4676947295665741, 0.2334459125995636, -0.47941118478775024, 0.6611179113388062, 0.14063473045825958, -0.8803531527519226, -0.24085308611392975, -1.0701059103012085, -0.39695289731025696, 0.9776099920272827, -0.3737213611602783, -0.5587900876998901, 0.3016863167285919, -0.02360096573829651, 1.2394875288009644, -1.182800054550171, 0.8478527069091797, 0.9649490714073181, 0.6052069664001465, -1.810019612312317, -1.219808578491211, -0.16858124732971191, 1.636675477027893, 0.6867020726203918, -1.8403453826904297, -2.167271614074707, 1.9961521625518799, 0.04807150363922119, -1.4255555868148804, -0.010222847573459148, -0.403854638338089, 1.6254914999008179, -0.39118319749832153, 0.7822645902633667, 1.896155595779419, 0.9660003185272217, 0.8072847723960876, 0.3673281967639923, 2.094883918762207, 1.593620777130127, 0.7514418363571167, 0.746224045753479, -0.5608136057853699, -0.19674010574817657, 1.2544755935668945, -0.41235119104385376, 0.7303610444068909, -0.8952244520187378, 0.012099935673177242, 0.992683470249176, -0.9264669418334961, -0.2534232437610626, 0.8312582969665527, 0.5424161553382874, -1.0824350118637085, 0.002151530934497714, 1.9029719829559326, 0.3240002989768982, -1.951466679573059, 0.6416613459587097, -0.25021079182624817, 0.6905475854873657, -0.5305766463279724, 0.26038098335266113, 1.4593182802200317, -0.2634003758430481, 0.456080824136734, 0.43459177017211914, 0.07265441119670868, 0.9687241315841675, -1.190859079360962, 0.32769936323165894, -1.8123533725738525, 1.4372683763504028, 0.186203271150589, -0.7119708061218262, -0.18669898808002472, -2.5298943519592285, -0.030122654512524605, -1.8851581811904907, -0.14376665651798248, -1.3704763650894165, 0.9989019632339478, 0.5752736330032349, 1.000261664390564, 1.1736438274383545, 0.5709714889526367, -0.4813753366470337, 0.6974393129348755, -0.9412084221839905, -0.30222058296203613, -0.13871967792510986, -0.41237980127334595, 0.4256400167942047, -0.4653366208076477, -0.5194478034973145, 0.875837504863739, -1.0069879293441772, 0.312506765127182, 0.32301604747772217, 0.07841389626264572, -0.22607479989528656, 0.2132580429315567, 0.07027056813240051, -1.1462368965148926, -0.7262541651725769, -0.29445087909698486, -0.5762459635734558, 0.6169731616973877, -0.34685301780700684, 0.6681429147720337, 1.74471116065979, -3.062772750854492, -0.13931870460510254, 0.6404218077659607, -0.4249270260334015, -0.48040270805358887, 0.5117174386978149, 0.031531479209661484, 0.8250983357429504, -0.6460134387016296, 0.6478019952774048, 1.190600872039795, 1.9496588706970215, -0.6026361584663391, 1.927914023399353, 1.0962576866149902, 0.3904927372932434, -0.9974298477172852, 0.5551510453224182, 2.7683377265930176, 1.7623591423034668, 1.0522364377975464, 0.3292088210582733, -0.12609881162643433, 1.3121649026870728, 1.6139166355133057, -0.012797441333532333, -0.23744899034500122, 1.2441785335540771, -0.3690774738788605, 1.1983821392059326, -1.136836051940918, -1.2928283214569092, 0.6353534460067749, -0.19147561490535736, 0.5517373085021973, 1.5201513767242432, -1.0918809175491333, -0.5495142936706543, -1.2553282976150513, 0.20951537787914276, -0.3995642364025116, 0.06629843264818192, -1.6080251932144165, -0.783801257610321, -0.3357933759689331, -0.5023289918899536, -1.0472276210784912, 2.486818790435791, 0.7717161774635315, -0.5230225920677185, 0.6347784996032715, -0.43091025948524475, -1.8645817041397095, -0.7057151794433594, -0.783830463886261, -1.0425617694854736, -0.5305999517440796, 0.318145215511322, -0.6788529753684998, 1.4098551273345947, 1.6266776323318481, -0.23291775584220886, 0.8317117094993591, 0.5285998582839966, 1.4213136434555054, -0.9349840879440308, 1.3158351182937622, 0.9181143045425415, -0.12292975932359695, 0.2126174122095108, -0.7850203514099121, -0.3684026002883911, -0.5726766586303711, -0.20265567302703857, -0.5161880850791931, 2.5229005813598633, 0.24647314846515656, 0.5258211493492126, 2.2278032302856445, 1.4776839017868042, 0.3424258828163147, 0.9790397882461548, 0.1567934900522232, -1.5252952575683594, -0.8436150550842285, -1.2091283798217773, -2.300025224685669, -0.028486084192991257, -0.2493310570716858, 0.620637834072113, 0.45130082964897156, 1.2489126920700073, -0.7236931324005127, 1.3959524631500244, 0.9671940207481384, -0.19351312518119812, -2.6917452812194824, 1.7348178625106812, -0.31285563111305237, -0.11433586478233337, -0.7626615762710571, -0.3691171705722809, -0.4311976134777069, -0.3471507728099823, -0.5312430262565613, -0.8550276756286621, 0.38529103994369507, 0.20726190507411957, -0.27684977650642395, -0.3782208561897278, 0.3342263400554657, 0.44552016258239746, -0.5070405006408691, -1.4300527572631836, 0.28732553124427795, -0.06770039349794388, 0.7604098916053772, -0.23581890761852264, 1.0417617559432983, -0.598074197769165, 0.7343490719795227, -0.1489340215921402, -1.8501977920532227, -1.335405945777893, -0.8418019413948059, -0.6852820515632629, 1.4556571245193481, 0.12229693681001663, 0.393971711397171, -0.6698294281959534, 0.9720155596733093, 0.20824000239372253, -0.09288383275270462, -0.9808993935585022, 0.4289141893386841, -0.16583871841430664, -1.0993002653121948, 0.5114818215370178, -0.6735108494758606, 0.94771409034729, -0.3990446627140045, -0.3877336084842682, 0.8892289996147156, 0.23979038000106812, -1.5093785524368286, 1.0700716972351074, 0.7656331658363342, 2.232213258743286, -0.28016960620880127, -1.1953740119934082, 0.582470715045929, -0.32808804512023926, 0.3231891989707947, 0.5886074900627136, 0.6221043467521667, -1.6885815858840942, -1.1668812036514282, -0.19466474652290344, -0.2484358251094818, 1.5090656280517578, 2.0762782096862793, 0.797828733921051, 0.3721987307071686, 0.37231090664863586, 0.0306989848613739, -0.898709237575531, 1.5457711219787598, 0.11870735138654709, 0.010200310498476028, 0.462631493806839, 0.9656102061271667, 1.060639500617981, 0.23386713862419128, 1.915419101715088, -0.6726053953170776, 0.795940637588501, 0.4109993278980255, -0.013412444852292538, 2.0900967121124268, -0.5171285271644592, 0.5025956630706787, 0.6237219572067261, 0.3180732727050781, 0.6076026558876038, -1.9763153791427612, 0.28134140372276306, 0.7072145938873291, -0.1829787939786911, 1.2621256113052368, -0.9582265615463257, -0.9112985134124756, -0.04281414672732353, -0.24780021607875824, -0.7724046111106873, 1.8445768356323242, -1.688663125038147, 0.5903334617614746, 0.7524674534797668, 0.5118560791015625, -1.528784155845642, -0.46335312724113464, -0.7681577801704407, -0.7575640082359314, -1.6587687730789185, -1.3040715456008911, -0.43799638748168945, 0.1999497413635254, 1.4469128847122192, -0.10156388580799103, 0.6247029304504395, -1.628779411315918, -0.07264510542154312, 0.3255719840526581, -0.9918811321258545, -1.6652507781982422, 0.2185298204421997, 0.78420490026474, -0.7162536978721619, -1.5079309940338135, -0.8668317794799805, -0.030323099344968796, 1.73930823802948, -0.6563314199447632, -0.2565918266773224, 1.6393028497695923, 0.06881488114595413, -0.15946164727210999, -0.18011608719825745, 0.09577136486768723, 0.10361063480377197, -0.9967343211174011, -0.2817416787147522, 0.3045676648616791, -0.4214126169681549, 0.7177413105964661, -0.20617415010929108, 0.20540596544742584, -0.2577389180660248, -0.31710517406463623, -1.6369372606277466, 0.8911246061325073, 0.5870236754417419, -0.3759077191352844, 1.3770374059677124, -1.2328910827636719, -0.23252446949481964, -0.4241047203540802, -0.17546357214450836, 0.27916690707206726, -1.7572393417358398, -0.8919432759284973, -0.4997915029525757, 0.13158023357391357, -0.9446253180503845, -1.2943787574768066, 1.240293264389038, -1.405432939529419, -1.0724296569824219, -0.6470343470573425, -1.3466538190841675, -0.3778272569179535, -0.1503136307001114, 1.4318263530731201, -1.113416075706482, -0.32370391488075256, 0.7006704211235046, 1.3533896207809448, 0.9259213209152222, 0.7247433066368103, 1.4393247365951538, -0.8956848382949829, 0.2578720152378082, -0.09625796973705292, -0.555203378200531, 1.1236472129821777, -1.2278629541397095, 0.614323616027832, -0.31922468543052673, 0.279173880815506, 0.4757665991783142, 0.9136064648628235, 1.2316169738769531, -0.9224412441253662, 0.988037109375, 0.4134732782840729, 0.5361922979354858, -0.25439631938934326, -1.1934356689453125, -0.39046645164489746, -0.48647722601890564, -1.2864975929260254, 1.0786195993423462, 1.4189319610595703, -0.7945113778114319, -1.3676091432571411, -0.9569045901298523, 3.114138603210449, -1.2959339618682861, 0.8192855715751648, 1.0165601968765259, 0.2719145119190216, -0.5617008209228516, 0.3119162917137146, 0.33580514788627625, 1.2366105318069458, 0.35884377360343933, -0.02484217658638954, -0.44129395484924316, -0.22804385423660278, 0.3175392150878906, -0.03428664430975914, -0.9040247797966003, 0.28207799792289734, 1.916709065437317, 0.9014037251472473, -0.9383812546730042, -0.2614380121231079, -0.540681004524231, 1.678230881690979, 0.16796846687793732, -1.7243982553482056, 1.8790919780731201, 0.9719703793525696, -0.3177301287651062, 0.19355431199073792, 0.5485783815383911, 1.1315512657165527, 0.27064278721809387, -0.27559715509414673, 0.16038277745246887, 0.20492230355739594, -0.17465639114379883, -0.7947238087654114, 0.3047272861003876, 1.6422443389892578, -0.01948513835668564, 1.1893889904022217, 0.18881063163280487, 1.0018255710601807, -0.32525384426116943, -0.36070194840431213, 0.8211538791656494, 0.84611976146698, 1.0173591375350952, -0.9493862986564636, 1.3054447174072266, -0.6786969900131226, 1.3738025426864624, 1.353083848953247, 0.12511925399303436, -0.5057693123817444, 0.1001010611653328, -0.5324180722236633, 0.36432430148124695, -0.9669500589370728, 0.8907234072685242, 1.4437767267227173, 2.8458008766174316, 0.36539164185523987, -0.5135583877563477, -0.570565938949585, 2.0727574825286865, -1.1110986471176147, 1.049832820892334, 0.09801588207483292, -0.07953982055187225, 0.5588926672935486, 0.134688600897789, -0.06452804058790207, 0.5438465476036072, -0.8477840423583984, -0.8294959664344788, -0.8905866742134094, 0.8027496337890625, 0.9478382468223572, 2.5694618225097656, -0.26379483938217163, 1.8735215663909912, -0.707007110118866, -0.0064141228795051575, -0.7895178198814392, -0.30857914686203003, 0.15906240046024323, 0.07380887866020203, 0.11109603941440582, -0.7124094367027283, 0.8425837755203247, -0.472865492105484, -0.3985636234283447, 0.7492926120758057, 1.2103773355484009, -0.9968438148498535, 1.8402025699615479, 0.9647889733314514, -1.447058916091919, 2.1951539516448975, 1.2716224193572998, -1.735708475112915, 0.9080698490142822, 2.6346118450164795, -0.10872934758663177, -0.9186254739761353, -0.9886565804481506, 0.9661169052124023, -0.928843080997467, 0.9788206219673157, -0.18062086403369904, -0.44322746992111206, -1.528556227684021, 1.027725338935852, -0.23387208580970764, 0.27898961305618286, 0.28145381808280945, 0.16448014974594116, -0.5372087359428406, -1.6706656217575073, 0.903577446937561, 1.1473751068115234, -0.21014754474163055, -0.3675553500652313, -0.8300986289978027, -0.7007383108139038, 0.055452752858400345, 0.9402013421058655, 0.16824424266815186, 0.4505709111690521, 0.019761400297284126, 0.48885664343833923, 0.11740146577358246, 1.3879948854446411, -1.475210428237915, 0.2625713646411896, 0.2459617555141449, -0.5449819564819336, -0.546333372592926, -1.2584130764007568, -0.9998798966407776, 0.3015563189983368, 0.10383708029985428, 1.2485731840133667, 0.3894954025745392, -1.2249233722686768, -1.121698260307312, 1.8255131244659424, -0.9748709797859192, -0.6160969734191895, -1.6027206182479858, 0.31781044602394104, 1.3503085374832153, 2.3545315265655518, -0.4177277386188507, 0.36046847701072693, 0.36070412397384644, -0.4511856138706207, 1.4387322664260864, 1.0673754215240479, 0.6201325058937073, -0.09058943390846252, 0.48064282536506653, 0.16827392578125, 0.5197319388389587, 1.4139301776885986, -1.0850344896316528, 0.4284885823726654, -1.6998367309570312, -1.0779271125793457, 0.08379281312227249, -0.5460827350616455, 0.7842618823051453, -0.5537664890289307, 1.2030256986618042, 0.9373198747634888, -0.7255991697311401, -2.24652099609375, 0.5621898174285889, 0.06999529153108597, -0.9146708250045776, 0.9907180070877075, 0.21225759387016296, 0.36947718262672424, -0.14324739575386047, 1.3838675022125244, 2.0277748107910156, 0.6550751328468323, -1.274832010269165, -0.780849277973175, 0.8213781714439392, -1.2471929788589478, -0.9721011519432068, -0.9508316516876221, 1.3219619989395142, 0.7084934711456299, 0.36694565415382385, -1.894961953163147, -1.0089919567108154, 0.8319621086120605, -0.3015744090080261, -1.610413908958435, -1.008072853088379, -0.26981911063194275, -1.6961199045181274, -0.35176217555999756, 0.6233055591583252, 0.5465588569641113, 0.37685930728912354, -1.2025848627090454, -0.8021082282066345, -0.46970418095588684, -1.0240046977996826, 0.5571536421775818, 0.7444874048233032, 1.036505103111267, 0.2265186607837677, -1.5875811576843262, -0.5230945348739624, 0.3333243131637573, -1.5582956075668335, -1.4838048219680786, -1.154421091079712, 0.42780423164367676, -0.5570715069770813, -1.0553113222122192, 1.4080579280853271, 2.3004865646362305, -1.27898108959198, -0.7238847613334656, 1.451926589012146, 0.48032066226005554, -0.8648383617401123, 0.19966855645179749, 1.9859733581542969, 0.5552172064781189, -0.5854976177215576, -0.7490299940109253, 0.3810807764530182, -0.6726468801498413, -0.8537359833717346, -0.8813251852989197, -0.3215353786945343, 0.6129181385040283, -0.2804129719734192, 2.629457950592041, 0.13132165372371674, 0.6704089641571045], [1.1394460201263428, -0.10821672528982162, 0.6929487586021423, 0.7570101022720337, -1.0296754837036133, -1.063462734222412, -1.546337604522705, -0.15328361093997955, 0.7510078549385071, 0.7719486355781555, 0.8337613344192505, -0.6031129956245422, 0.5751333832740784, -0.35653156042099, 0.5198183059692383, 0.6173290610313416, -0.37631329894065857, 0.17749251425266266, -0.33741703629493713, -0.2426801472902298, 0.9337937831878662, -1.427364468574524, 1.9708819389343262, 0.5961292386054993, 0.11485611647367477, -1.204451560974121, -0.3927222192287445, -0.6024690866470337, 1.022172451019287, -0.1884678155183792, -0.4746916592121124, 0.4494107663631439, -1.8415193557739258, -0.10239669680595398, -0.46190589666366577, 0.6465132832527161, 1.987326979637146, 0.29478394985198975, -1.0287597179412842, 0.09504514187574387, -0.44865182042121887, -0.766468346118927, 0.666131317615509, -0.8647577166557312, -0.9415493011474609, 0.9947391748428345, -1.1717697381973267, -0.2658281922340393, -0.664000391960144, 0.6557215452194214, -0.3971754312515259, 0.17451457679271698, -0.18244647979736328, -0.3311871588230133, 0.9639434814453125, 0.5030342936515808, -2.5816094875335693, 0.4788649380207062, 2.2953569889068604, -0.05554041266441345, 1.5884126424789429, -0.3885943591594696, 0.6707319021224976, 1.2647359371185303, -0.5836604833602905, 0.7450616955757141, -0.029951218515634537, -0.4480544924736023, -0.7857255339622498, -0.07525740563869476, 0.13327303528785706, 0.8936538100242615, -0.5316113829612732, 0.5526309013366699, 0.044955212622880936, -0.05500679463148117, -1.500918984413147, -0.6912915706634521, -0.06114102527499199, -0.8226897120475769, 0.629409909248352, 0.8693742752075195, 0.29064658284187317, 1.3881057500839233, 0.29577216506004333, -1.0436497926712036, 0.5012067556381226, 1.638077735900879, -2.242990493774414, 0.10477712005376816, 1.6265345811843872, -1.3589900732040405, -1.0057302713394165, -1.0810071229934692, -1.5128395557403564, 1.1031670570373535, -0.5389454960823059, -0.31543055176734924, 0.05719338357448578, -1.4460601806640625, -1.987497329711914, 0.7801472544670105, 1.217658519744873, 0.1097870022058487, 0.3916492760181427, 0.6882947087287903, -1.4041956663131714, -2.5225791931152344, 1.1880375146865845, -0.2229927033185959, 0.3347708284854889, 2.806028127670288, 0.08547689020633698, -0.7900124788284302, 0.44855332374572754, -0.28752371668815613, 0.7861573696136475, -0.8084988594055176, -1.9020265340805054, 0.6017224788665771, 0.9825136065483093, -1.6009209156036377, -1.2004107236862183, 1.5336496829986572, -0.865525484085083, -0.960483193397522, 1.2069343328475952, 0.7459540963172913, -0.49570462107658386, 0.08216250687837601, 1.2448567152023315, -0.16461476683616638, -1.9107109308242798, 0.13413824141025543, 0.30201464891433716, 1.3948942422866821, -0.18131284415721893, 1.8123193979263306, 0.522269070148468, -0.840826153755188, -0.008550385944545269, 2.3120405673980713, 0.4404253661632538, 1.0187554359436035, 0.3808360695838928, -0.6454780697822571, -0.14343257248401642, -0.5091542601585388, 0.4955160915851593, -1.0027542114257812, 0.6500203609466553, 0.7327825427055359, 0.07677672058343887, 0.6826671957969666, 0.18157809972763062, 0.6144508123397827, 0.28610843420028687, -0.4558330774307251, 0.8259643912315369, -2.011399745941162, -0.8672751188278198, 0.535824716091156, -0.001343115116469562, -0.49338939785957336, -0.4366247057914734, -0.571577787399292, -1.202813744544983, 0.548753023147583, 0.9592086672782898, 0.7101865410804749, 0.17101304233074188, 0.13878704607486725, -0.4937742352485657, 0.9248692989349365, -1.6202272176742554, 2.108675241470337, 0.763079822063446, 1.7936837673187256, -0.12056748569011688, -1.409591555595398, -0.9107545018196106, 2.3089356422424316, -1.0601311922073364, 0.45912399888038635, -0.20793232321739197, -0.6162289977073669, 0.28688815236091614, -1.4798595905303955, 1.9893757104873657, -1.2703945636749268, 0.4285111725330353, 1.7107343673706055, -0.5555971264839172, -0.6119723320007324, -0.8820422291755676, -1.5344918966293335, -0.6903720498085022, -0.8473025560379028, 0.5594539046287537, 0.7689881324768066, 0.1535121649503708, 0.3264174461364746, 0.37384384870529175, 1.2086548805236816, -0.7703487277030945, -1.4809916019439697, -0.5301149487495422, 0.27075421810150146, 0.6287165284156799, -0.16611838340759277, 0.8147266507148743, -0.27974411845207214, -0.22425371408462524, 0.34883394837379456, 0.693077802658081, -1.0067473649978638, 0.3221954107284546, 0.687168300151825, -0.23106171190738678, 0.29731500148773193, 0.7755866646766663, -1.7171403169631958, -0.31673529744148254, -0.8519550561904907, -0.9962076544761658, -0.8047215938568115, 1.411208987236023, 0.5986036658287048, -0.4712112247943878, -0.3162815272808075, -0.9829043745994568, -1.0661791563034058, 0.7224587202072144, 0.6235765814781189, -0.15749527513980865, 0.4673798978328705, 0.25438621640205383, -0.4718940854072571, 0.7491825819015503, -0.7590013146400452, -0.47609132528305054, -1.8195770978927612, -0.19429388642311096, -1.0294249057769775, 0.2334538847208023, 1.3769237995147705, 2.864703893661499, 0.018534667789936066, 1.1270712614059448, 1.7687809467315674, -0.23423133790493011, -0.022737715393304825, 1.900668740272522, 1.7306362390518188, 1.389233112335205, 0.1038898304104805, -0.7648016214370728, -1.311052918434143, -1.1134546995162964, -0.7561022639274597, 2.0348353385925293, 0.9872627258300781, 0.4893929064273834, 0.6001131534576416, -0.38728880882263184, -1.5086373090744019, 0.8496423363685608, -1.3719345331192017, -0.9542644023895264, 0.8419657349586487, -0.505355179309845, 0.2993484139442444, -0.8032959699630737, -0.8409674763679504, -0.09319107979536057, -0.9646089673042297, -0.09697261452674866, 0.8924283981323242, 0.7208818793296814, 0.2911210060119629, 0.27126753330230713, 0.41685751080513, -1.279971718788147, 0.6299455165863037, -1.2194291353225708, 0.4336390197277069, 0.05458270013332367, -0.3779178559780121, -0.8026808500289917, 0.5922533869743347, 0.4217991232872009, 0.09296735376119614, 2.2052900791168213, -1.3835150003433228, -0.5096740126609802, 0.8205063939094543, -0.1805197298526764, 0.4803949296474457, -0.6529021263122559, 2.0359859466552734, 1.1566047668457031, -1.5743988752365112, 1.4136234521865845, -0.5510644316673279, -0.2059316635131836, 1.3287594318389893, 0.7983535528182983, 0.21729254722595215, -0.19320674240589142, -0.42982080578804016, -0.04207305610179901, -0.3116138279438019, -1.6935337781906128, 1.347165822982788, -2.1858978271484375, 0.6747479438781738, -1.4249255657196045, 1.1547662019729614, 1.4378825426101685, 0.7786539196968079, -1.258033275604248, 1.6479136943817139, -0.4222065508365631, 0.44117388129234314, 1.7191301584243774, 2.1920905113220215, 0.15717081725597382, 1.2224105596542358, -1.3557066917419434, 0.42283424735069275, 0.34360820055007935, 0.2706714868545532, -1.3647533655166626, 1.7067689895629883, 0.8280465006828308, 2.1349282264709473, 2.0385305881500244, 0.6538974642753601, -0.37453892827033997, -1.1305447816848755, -0.07488639652729034, -1.1839874982833862, 1.2918187379837036, 0.4626443684101105, 1.9816738367080688, 0.723342776298523, -0.8269872069358826, -0.33047980070114136, -0.7687812447547913, 0.20610660314559937, 2.569042921066284, -1.2746907472610474, 0.8212498426437378, 1.5907973051071167, 0.12390762567520142, 0.8412542939186096, -0.9934331774711609, 0.6422141790390015, -0.05401508882641792, 0.6145111322402954, 1.1946054697036743, -0.030478524044156075, -0.005801972467452288, 0.6836407780647278, -0.017279164865612984, -0.2683611512184143, -1.3091894388198853, 0.026051538065075874, -1.0058006048202515, -0.7679576873779297, -0.6757093667984009, 0.1713704764842987, -0.9548499584197998, -0.5397365093231201, 0.6356006264686584, 0.5863631963729858, 2.7437422275543213, 0.9987011551856995, -1.6904171705245972, 0.47674015164375305, 0.4926088750362396, -0.30177178978919983, 0.12954048812389374, 0.15845640003681183, -0.8651108145713806, -0.9998754858970642, 0.4542196989059448, 1.340491771697998, -0.07423675060272217, -0.05755159631371498, -0.5589446425437927, -1.0866211652755737, -0.7640129923820496, -0.253207802772522, 1.1138511896133423, 0.5974181294441223, -1.2974928617477417, 1.3471578359603882, 0.22230219841003418, 0.4864928126335144, -0.11484615504741669, 0.9856760501861572, -0.4807263910770416, -0.4235129654407501, -0.50487220287323, -0.8473517298698425, 0.24011342227458954, -0.08867643773555756, -2.25784969329834, -0.2927880883216858, 0.13917355239391327, 0.7753288149833679, -0.20718537271022797, -0.2266940325498581, -0.4493848979473114, 0.9193058013916016, -0.18114720284938812, -0.24330322444438934, -1.020275354385376, 1.8546292781829834, -1.6193757057189941, -1.1575630903244019, -0.2640865743160248, 0.5877402424812317, -0.7960620522499084, -0.4642527401447296, 0.017888428643345833, -0.34331604838371277, 0.5756635069847107, 0.669014573097229, -0.9237415790557861, -1.1324812173843384, 0.45733776688575745, 1.0137674808502197, 0.07709435373544693, -0.4294915795326233, -0.6471347808837891, -0.40961945056915283, 0.31819742918014526, 0.5834786295890808, 0.9950719475746155, 1.2593239545822144, -0.867457926273346, -0.5733853578567505, 0.7892639636993408, -1.3343981504440308, -1.5147809982299805, 0.9136096239089966, 0.5035342574119568, -0.4571067988872528, 0.06640198081731796, 1.1400209665298462, 0.4534638226032257, 0.8655641674995422, 1.6122145652770996, -1.0252920389175415, 1.6898287534713745, -1.694883942604065, 1.134524941444397, -1.561545968055725, -0.427275151014328, 0.48436951637268066, -0.1410447061061859, 1.3530303239822388, -0.8151494860649109, -0.4933396577835083, 0.4246333837509155, -0.2732445001602173, 0.7243461012840271, 1.1636532545089722, 0.5199717879295349, 0.6273786425590515, -1.8196394443511963, -0.5353077054023743, 0.45072227716445923, 0.11567200720310211, -0.042362235486507416, 0.906836211681366, -0.84617018699646, 0.09995696693658829, -1.4355783462524414, -0.05436485633254051, -0.03717339038848877, 0.3247263431549072, -1.3770829439163208, -0.6478778719902039, -0.2903329133987427, -0.8916302919387817, 0.5687721371650696, 0.21408459544181824, -0.6403028964996338, -0.5588631629943848, 0.22846989333629608, 0.2040839046239853, -1.295816421508789, 1.5656516551971436, 0.8910203576087952, -0.184125617146492, -0.04781661927700043, -0.28625237941741943, 2.1291065216064453, -0.8946731090545654, -0.7207402586936951, 1.6308799982070923, 0.36713144183158875, -0.6752980351448059, -0.8331540822982788, -0.27253469824790955, -1.3823049068450928, 1.4102410078048706, -0.11019972711801529, 0.08236511051654816, -1.3819974660873413, -0.5106223225593567, 0.374273419380188, 0.5189889073371887, 1.3649678230285645, 0.6950892210006714, 0.7338956594467163, 1.951104998588562, -0.3945775330066681, 1.0134544372558594, 1.3481861352920532, 0.45567235350608826, -1.2021660804748535, 0.030095426365733147, 2.094919443130493, -1.1669230461120605, 1.3783032894134521, 0.6373569965362549, -0.19690072536468506, -0.4713250696659088, 0.6794289350509644, -1.4768434762954712, -0.11032082140445709, -0.6086466908454895, -1.036757230758667, -0.13359689712524414, 0.1751694530248642, 0.24531972408294678, 0.09857092797756195, -0.6605344414710999, 2.0765225887298584, 0.0607982836663723, 0.528123676776886, 0.2825518250465393, 1.7140920162200928, 1.356054663658142, -1.6652946472167969, -0.09307233989238739, -0.877875566482544, 0.6876981258392334, 1.555192232131958, 0.16558243334293365, 0.7851308584213257, 1.9368231296539307, 1.1894360780715942, 0.8371967673301697, 1.9979934692382812, 0.5713059306144714, 0.7057353854179382, -1.4009792804718018, -1.4405975341796875, 0.3834724724292755, -0.4056185185909271, 0.7566285729408264, 1.5522857904434204, 0.8793790340423584, -1.804855465888977, -0.47546708583831787, -1.5803828239440918, -0.8051774501800537, -0.5456202626228333, -1.2143279314041138, -1.071771264076233, -0.09113282710313797, 1.0239800214767456, -0.506574273109436, 1.3935506343841553, -1.127899169921875, -0.9870036840438843, -1.0004853010177612, 1.165845513343811, -1.1090432405471802, -1.8784070014953613, 0.9811217188835144, -1.3996272087097168, -0.9363850355148315, -0.43663352727890015, 0.12488251179456711, 0.11133702844381332, -0.18573088943958282, 2.449383497238159, -0.034309271723032, -0.6938503980636597, -1.651633858680725, 2.3291420936584473, -0.1087871789932251, -0.8820580840110779, 1.2632371187210083, 0.6011345982551575, -1.0147652626037598, 0.10809910297393799, -0.45705047249794006, 0.8297982811927795, -0.01938977837562561, 1.517012119293213, 0.7320971488952637, 0.2387172132730484, 0.6433234214782715, -0.08100321888923645, 0.466309517621994, 0.5983296632766724, -0.8358055353164673, 1.8537638187408447, 2.2722649574279785, 0.009101219475269318, 2.3924899101257324, -1.748174786567688, 0.3981216847896576, -0.00224998127669096, 1.7120907306671143, 1.1685478687286377, -0.37718889117240906, 0.7233037352561951, 0.15894974768161774, -0.060710709542036057, -1.5189645290374756, -0.7304803133010864, -0.5499151945114136, -0.18575672805309296, 0.6815814971923828, -0.03612732142210007, 0.16185824573040009, 0.6454988718032837, -0.9848507642745972, -0.0008777194889262319, -0.5405141711235046, 0.9836341738700867, 1.039826512336731, -0.10236894339323044, -1.0948610305786133, 0.6196403503417969, 0.18601974844932556, 0.3799746632575989, -1.5275038480758667, 0.8190017342567444, 0.0775831788778305, -0.0348246693611145, 0.3734188973903656, 1.257606863975525, 0.06475145369768143, 0.7166990041732788, 0.4239880442619324, -1.1715586185455322, -1.1145203113555908, -1.138528823852539, 1.7247182130813599, -0.933311402797699, 0.4836302399635315, 0.08763901889324188, 1.3391108512878418, -0.5087565183639526, 0.03832359239459038, 1.340988278388977, -0.4953329563140869, -0.02405363880097866, 1.1332361698150635, -0.1321573704481125, 0.3352608382701874, -0.5857759714126587, 0.13179978728294373, 0.12735411524772644, -2.1853513717651367, 0.19230090081691742, -0.6759694814682007, -0.4081501364707947, -0.900570809841156, -0.45741772651672363, -0.46847471594810486, -1.133489727973938, -2.1410861015319824, -1.7725279331207275, 1.6926583051681519, -1.152011752128601, -0.5371050834655762, -1.9095710515975952, 0.9630138278007507, 1.9779767990112305, 1.345172643661499, -1.4007171392440796, -0.0389280766248703, -1.6324702501296997, -0.46818676590919495, -1.2068233489990234, -0.501275897026062, -0.4373132884502411, -0.7945066690444946, 0.25527462363243103, -0.8068956136703491, -0.09995471686124802, 0.6401122808456421, -0.49955230951309204, -0.230868861079216, 1.5217736959457397, 0.7721641659736633, -0.23829597234725952, 0.953909695148468, 1.7759711742401123, 0.5255785584449768, -0.6812450289726257, -1.337432861328125, 0.04473639279603958, 0.8218866586685181, -2.3934433460235596, -0.3177037239074707, -0.6234300136566162, 0.2472028136253357, 0.04509495198726654, 1.9325071573257446, -0.6408928632736206, -0.5855502486228943, 1.197619915008545, 0.5715173482894897, 1.4258531332015991, 0.5889952182769775, -1.7864010334014893, -0.45044729113578796, 2.0205297470092773, 0.3136049509048462, 0.6790900826454163, -1.409565806388855, -1.40384042263031, -0.9708766937255859, 0.14290912449359894, 1.410577654838562, -0.5624521374702454, -0.4461401104927063, 1.1985567808151245, 1.3241908550262451, -2.4951977729797363, 1.119985818862915, 1.3663201332092285, 1.0893765687942505, 0.14142680168151855, 0.13318750262260437, -1.0476232767105103, -0.7868056893348694, -2.063610792160034, -1.3219257593154907, -0.029411381110548973, 1.2836823463439941, 0.3836062550544739, 1.950378656387329, -1.1168699264526367, -1.1098443269729614, 0.5976607203483582, -1.0134854316711426, -0.9543192982673645, -0.6909155249595642, 0.6471285820007324, 0.4890711009502411, -0.5017374753952026, 0.18073798716068268, 0.6905156373977661, 0.8745865821838379, 1.8573148250579834, -0.32760488986968994, 1.101616382598877, -0.31061896681785583, -1.2683100700378418, -0.6123183965682983, 0.004690607078373432, 0.9258161187171936, 0.34489721059799194, -0.49989059567451477, 1.1102250814437866, -1.1715600490570068, -0.06685847043991089, -0.5511009097099304, -0.2269379049539566, -1.1142003536224365, -0.9679699540138245, -1.3754115104675293, 0.623369038105011, -2.1143712997436523, 0.13710175454616547, 0.4328576624393463, 0.8282271027565002, -0.7480437755584717, 0.1738634556531906, -0.18315298855304718, -0.35903602838516235, 1.1485689878463745, 0.8598043322563171, -0.9207119345664978, 1.0079553127288818, -0.5345807671546936, 0.7784454822540283, 0.0012201013742014766, -0.744444727897644, 0.7469464540481567, -0.8924839496612549, 0.1684626042842865, 0.7030706405639648, 0.43989449739456177, -0.8627411127090454, 0.31564152240753174, 0.43953582644462585, -0.6692149639129639, -1.645206332206726, -0.22342050075531006, -0.27965810894966125, -0.09543805569410324, 0.631608247756958, -1.5165863037109375, 0.6968073844909668, -0.20149922370910645, 0.3127290606498718, 1.4156385660171509, 1.2946888208389282, 0.11181029677391052, 0.4422483444213867, -0.5666500926017761, 0.2738129794597626, -1.0318211317062378, 0.6715855598449707, -0.6735349297523499, 0.39766359329223633, 1.0882339477539062, -0.16546432673931122, -0.2444034069776535, -1.6718950271606445, 0.010485882870852947, -0.055272698402404785, -1.406481146812439, -0.23279444873332977, -0.6187724471092224, -0.9728392958641052, -0.7624120116233826, 0.14268919825553894, -0.6554834842681885, 0.49726811051368713, -0.09541583806276321, 0.6629723310470581, 0.7840033769607544, 0.21993310749530792, 0.05472010746598244, -1.5268694162368774, -0.3637968599796295, -0.04810964688658714, 0.7184495329856873, -0.019320309162139893, 1.3857190608978271, -1.3852362632751465, -0.8454073667526245, 0.8302082419395447, -0.025595806539058685, -0.9329794645309448, 0.2602192163467407, -0.9499207139015198, 0.14908558130264282, 0.6795241832733154, -0.50653475522995, -0.7000631093978882, -1.443554162979126, 0.7292399406433105, 0.9874166250228882, -0.24873432517051697, 0.28680139780044556, -0.233722522854805, -0.22216194868087769, 1.6592999696731567, 1.8480215072631836, -0.3804176449775696, -0.14360466599464417, 0.05355076491832733, -0.49800601601600647, -1.2598990201950073, 0.21615812182426453, -0.3273501694202423, 2.2457644939422607, 0.05606682971119881, -0.13412708044052124, 0.12075736373662949, -1.0262274742126465, -0.47303879261016846, -0.04532405734062195, -0.19122640788555145, 0.6992167234420776, 0.11071810871362686, 0.20453095436096191, 1.0919997692108154, 0.4327540397644043, 1.315994381904602, 0.8726187348365784, 0.4983757436275482, 2.9107825756073, 1.005683183670044, 1.959113597869873, -1.653517484664917, -1.4603482484817505, -1.2167413234710693, 0.5212433934211731, 0.7226472496986389, -1.3907654285430908, -0.25025424361228943, -0.3523106575012207, -0.49863401055336, 0.03863018751144409, -0.28468629717826843, -1.0540430545806885, 1.5575796365737915, -1.3334197998046875, -1.1889145374298096, 0.5696052312850952, -0.060325395315885544, -1.8287558555603027, 3.0972695350646973, -0.8054129481315613, -0.9400126338005066, 1.5526257753372192, -0.4670070707798004, -0.5771504044532776, -0.25247839093208313, 0.1475270539522171, -1.1459527015686035, 1.8976850509643555, 0.4560700058937073, 1.5652377605438232, 0.13781233131885529, -0.935751736164093, -0.03688185662031174, 0.7717215418815613, -0.943780243396759, 0.29307234287261963, -1.258604645729065, 1.0535383224487305, -0.8752785325050354, 0.21447336673736572, 0.11919804662466049, -2.0683000087738037, 0.8754043579101562, -0.6932395100593567, 1.2330065965652466, -1.4814984798431396, 0.7829603552818298, -1.4409739971160889, -0.3323303163051605, 0.3577865958213806, 0.36873117089271545, 0.47485294938087463, 1.1358479261398315, -0.6456308364868164, -0.5801849365234375, 1.7096049785614014, -0.20253896713256836, -0.15460577607154846, -1.8046374320983887, 0.12406543642282486, -0.027565136551856995, 1.2289133071899414, -0.2808529734611511, 0.08260418474674225, -0.30221807956695557, 0.47938966751098633, 0.837912380695343, -0.6316053867340088, 0.1325606107711792, -0.8903289437294006, 0.961713433265686, -0.869125247001648, -1.001800537109375, -0.6326931715011597, -0.8990762233734131, 0.6987177729606628, -1.293973445892334, -0.37845543026924133, -0.38453996181488037, 0.2157241255044937, -0.8367440104484558, 0.1840754598379135, -1.3648775815963745, -1.8236534595489502, -0.6724024415016174, -0.6758114695549011, -0.824840784072876, -0.17483127117156982, 0.5023694634437561, 1.1922290325164795, -0.3918496072292328, -0.4498545825481415, 2.4122869968414307, 0.11327770352363586, -1.2676888704299927, 0.25239187479019165, -1.4783751964569092, 1.6616971492767334, 0.6351022124290466, 0.7262360453605652, 0.25173723697662354, -0.2089303731918335, -0.5663889646530151, 0.8322933912277222, 0.6648645401000977, -0.2610451579093933, -1.693681001663208, 0.9166898727416992, -1.2963359355926514, 0.754327118396759, 1.612608551979065, -1.851817011833191, 0.20382143557071686, 0.2851393222808838, 1.5819462537765503, 1.6220252513885498, -0.18802927434444427, 0.3287881016731262, 1.226048231124878, -0.7186442613601685, 2.1026394367218018, -0.23661138117313385, 1.3937656879425049, 0.5777601599693298, 0.9965543150901794, 0.1170085072517395, 0.5758837461471558, -0.02942575328052044, 1.5979722738265991, 1.8007615804672241, -0.418151319026947, 0.1666606068611145, -1.2794729471206665, -0.8372961282730103, -0.09518425911664963, 1.6243867874145508, -0.6855950355529785, 0.18063613772392273, -1.4061315059661865, 1.2553300857543945, -0.5486718416213989, -0.3266850411891937, 0.9849569201469421, 0.2541571855545044, 0.5094062089920044, -0.6395809054374695, 0.8050772547721863, -1.3234930038452148, 0.9215129613876343, -1.6607919931411743, -0.7334463596343994, -0.06946984678506851, -1.3400897979736328, 1.2554562091827393, -0.7881941795349121, -0.10572578012943268, 0.0874423161149025, 0.4583672285079956, 0.6428970098495483, 0.6827969551086426, 0.18865694105625153, -0.4040258526802063, -1.0642993450164795, -1.7223929166793823, 0.25367289781570435, -1.7664469480514526, -2.094369411468506, 0.32463133335113525, -0.4220770001411438, 0.30952420830726624, 0.4178383946418762, -0.14139558374881744, 0.5530443787574768, -1.1094534397125244, 0.12526452541351318, 1.897520661354065, 1.2802541255950928, 0.7903846502304077, 1.5647878646850586, 0.021885281428694725, 0.8737240433692932, -0.33087632060050964, 1.9379938840866089, 0.5197656750679016, 1.7057955265045166, -0.4115291237831116, 0.2419889271259308, -0.49350616335868835, -1.925944447517395, -0.3958747386932373, -1.5427747964859009, 0.42319124937057495, -1.1608444452285767, -0.13853630423545837, 0.3852117657661438, 1.0643560886383057, -0.06954753398895264, 0.6235255002975464, 0.03860386461019516, 1.7008413076400757, -0.833188533782959, -0.4266607165336609, -0.609386146068573, 0.7886492609977722, 0.12359499931335449, 0.5875241756439209, -0.5273051857948303, -1.5640678405761719, -0.5420952439308167, -0.6743307709693909, -0.5781545042991638, -0.34876060485839844, -1.0456104278564453, -0.14993765950202942, 0.5066438913345337, -1.1862070560455322, -0.4813278913497925, 1.25912344455719, 1.994694471359253, 0.8207011818885803, -0.6296737194061279, -2.284914493560791, -1.692561388015747, 0.1424974501132965, -2.177452802658081, 1.3101874589920044, 0.566870927810669, 0.33474332094192505, -0.008749358355998993, 1.7910020351409912, 0.4539203345775604, 0.16428573429584503, 0.054530467838048935, -1.09857177734375, 0.8746853470802307, 0.8263269066810608, 1.6027179956436157, -0.08580633252859116, -0.13412225246429443, 0.322246253490448, -0.3369438648223877, 0.42879918217658997, -0.3666125237941742, 0.6391932964324951, -0.23011280596256256, 0.40508803725242615, 0.6099017262458801, 1.5047023296356201, 0.4728091359138489, 1.4051648378372192, 0.23324431478977203, 1.1829349994659424, 1.1032816171646118, -0.5847375392913818, 0.09839648008346558, 0.5876886248588562, 1.1260744333267212, 0.5195674896240234, -1.250784993171692, 1.3488929271697998, -2.4375016689300537, -0.8293445110321045, -1.010001301765442, 0.8979687094688416, -0.09086374938488007, -0.38207972049713135, 0.2790282070636749, -0.2879743278026581, 0.2250981330871582, 1.0059713125228882, 0.9639720916748047, -0.8196488618850708, 0.7109970450401306, 0.8336535692214966, 2.578803777694702, 0.8884792923927307, 0.35825416445732117, 1.0700894594192505, 0.3767673969268799, -0.03026282787322998, 0.03661501035094261, -1.249392032623291, -0.34933769702911377, 1.6418684720993042, -0.22286860644817352, -1.222448468208313, 0.410787433385849, 1.2455157041549683, -0.6802433729171753, -1.0135008096694946, -0.7243836522102356, 1.5703036785125732, 1.4760525226593018, -1.3291409015655518, -0.18341079354286194, -0.46535900235176086, -0.008575689978897572, 1.1116329431533813, 0.8960780501365662, 0.06465153396129608, 1.288813591003418, 1.7808952331542969, -1.5084537267684937, -0.4761243462562561, -0.25549179315567017, 0.8996567726135254, -0.0008038839441724122, -0.5651834607124329, 0.11215350031852722, 1.0188320875167847, -0.08275380730628967, 1.0857484340667725, -1.579305648803711, 0.4324356019496918, 1.1097408533096313, 1.0982455015182495, -0.16063353419303894, 0.6486702561378479, 0.5069267749786377, -0.17563967406749725, 0.2367655634880066, 0.9538850784301758, -0.6716400980949402, -0.6186319589614868, 0.5069918632507324, -0.23578040301799774, -2.3285367488861084, 0.7108325958251953, 0.33707815408706665, -0.5397087931632996, 0.7722958922386169, 0.3755636215209961, 1.2394559383392334, 0.4306461215019226, -2.237462282180786, -0.23969881236553192, 0.0004310099466238171, -0.06758717447519302, 0.48807093501091003, -1.8111083507537842, 0.32332557439804077, 0.37138843536376953, 1.1298835277557373, -0.25664252042770386, 0.5234155654907227, -2.157177209854126, 1.4312208890914917, 0.8909333944320679, -1.4496445655822754, -0.39741051197052, 0.28022003173828125, -0.762334942817688, -2.077317476272583, 0.9163851737976074, -1.682194709777832, 0.50765460729599, 0.8664751648902893, -0.006895855534821749, 0.29413819313049316, 1.2459540367126465, -1.0461554527282715, -0.33151793479919434, -0.3710293173789978, 0.6243773102760315, 1.5730069875717163, -0.7633824944496155, -1.1763964891433716, -0.42263156175613403, -0.6860037446022034, -0.7646980881690979, 0.023435186594724655, -0.7960846424102783, 1.5848151445388794, -0.2927092909812927, -1.3903454542160034, 2.8207814693450928, 0.7231966853141785, -0.04073312133550644, 0.0255899578332901, 0.3049464821815491, -0.06986162811517715, -0.902391254901886, 0.5197467803955078, 1.2089452743530273, 0.7230356931686401, 1.4645969867706299, -0.9477957487106323, 1.4608286619186401, 0.5807462334632874, -1.8995510339736938, -0.4792163670063019, -0.8697288632392883, 2.221412420272827, -0.4707847535610199, 0.37533676624298096, 0.015698915347456932, 0.253003865480423, 0.1231100931763649, -0.5735467076301575, -0.4582713842391968, 0.5046181082725525, 2.0504746437072754, 0.433562308549881, 0.27142852544784546, 0.5065680146217346, 0.41244372725486755, -1.1362236738204956, 1.4363579750061035, 1.6525745391845703, 0.5554129481315613, -1.3263545036315918, -0.37003153562545776, -0.3829706907272339, -0.6609015464782715, 1.886361002922058, -0.2306404560804367, -0.1940215528011322, 0.919518768787384, -0.9817540645599365, -0.5370628237724304, 1.4339358806610107, -0.3346959948539734, -1.64829683303833, 2.211697578430176, 1.0624396800994873, -1.0719718933105469, -0.14937862753868103, 1.3950703144073486, 0.14980921149253845, -2.017603874206543, 0.7353686690330505, -0.4526563584804535, 0.07437495142221451, -1.9620294570922852, 2.471172332763672, -0.6434545516967773, -0.6477328538894653, -0.5740192532539368, -1.2276391983032227, 1.5855873823165894, -1.6031701564788818, -1.5926144123077393, -0.2127261906862259, 0.6031306385993958, -0.46958017349243164, -0.8840510845184326, -0.7013071179389954, -0.12890741229057312, -1.0639210939407349, -0.5405740141868591, 0.23380863666534424, -0.6842237710952759, 0.03951947018504143, -1.5934650897979736, -1.7757364511489868, 1.9917998313903809, -0.5367386341094971, -1.8247020244598389, 1.2042648792266846, -0.12895344197750092, -0.059184130281209946, -0.5992594361305237, 0.49550536274909973, -0.5467147827148438, -0.8488996624946594, -0.8597093224525452, 0.955560564994812, -0.2549133896827698, -1.9354922771453857, 1.1861752271652222, -0.082753986120224, 0.11392835527658463, 0.45047125220298767, -1.1713502407073975, 0.9833297729492188, 1.4670723676681519, 0.39962854981422424, -1.375455379486084, -0.37796908617019653, 0.13435682654380798, -0.3455284535884857, -0.30239397287368774, -0.10133778303861618, 0.9890618324279785, -0.27406373620033264, -1.7107247114181519, 0.681763231754303, 0.14386002719402313, 0.17200300097465515, 0.4362989068031311, 0.34549522399902344, 0.83113694190979, 0.18645887076854706, 0.32373467087745667, -0.20941387116909027, 0.1544795036315918, 2.286362648010254, 0.4984295666217804, 0.19024111330509186, 0.04198438674211502, -0.0036161940079182386, 0.476857453584671, -0.4134075343608856, 2.048211097717285, 0.5672853589057922, -1.2385400533676147, 1.315950632095337, 0.052016664296388626, -1.8937584161758423, 1.2582359313964844, -1.4770538806915283, -0.7475300431251526, 0.10993407666683197, 0.3110464811325073, 0.6766164898872375, -1.7471622228622437, -1.2085171937942505, -1.0286006927490234, -1.4232980012893677, 1.5230467319488525, -0.8521735072135925, 0.1534762680530548, 0.06996939331293106, -0.34659215807914734, -1.3229008913040161, 0.11455405503511429, -0.05236722528934479, -0.47370660305023193, -0.2774284780025482, 0.2717701196670532, -0.7027313709259033, -1.3382432460784912, -1.9369667768478394, 0.6497467756271362, -1.254905104637146, -1.0884649753570557, 0.30433574318885803, -0.14236435294151306, -0.569581151008606, 0.09040651470422745, 0.4652135372161865, -0.8484101295471191, 0.09880977869033813, -1.1815974712371826, -1.3509827852249146, -0.5535910725593567, -0.47074028849601746, -0.9793156385421753, -0.7391744256019592, -0.25200679898262024, 0.12871861457824707, -0.07522811740636826, -1.791588544845581, 2.976011276245117, 1.0216304063796997, 0.10994116216897964, -1.8829052448272705, -1.061174750328064, -0.09625399112701416, 1.1399978399276733, 1.0289955139160156, 0.29013386368751526, 1.9161301851272583, 1.980438470840454, 0.2853742241859436, -0.47573864459991455, 0.7816653847694397, -0.5166563987731934, 0.40023866295814514, 0.4688471853733063, -0.26712146401405334, -1.900330901145935, -0.050788093358278275, 0.07355246692895889, -0.34178122878074646, -1.3754305839538574, -1.7520484924316406, -0.5271040201187134, -0.4120425283908844, 1.6406941413879395, -0.6859986782073975, 0.9868731498718262, 0.48215264081954956, -0.49062398076057434, -0.46525079011917114, -1.7493655681610107, 0.28844836354255676, -0.7099059224128723, -0.8013098835945129, -0.07197204232215881, -2.055553913116455, -1.5673118829727173, 2.1305627822875977, 0.3276601731777191, -0.5278086066246033, 0.048311203718185425, 0.6464205384254456, -1.376731514930725, 0.8254798650741577, -0.24257881939411163, 0.24724076688289642, -1.4092856645584106, 1.8947652578353882, -0.5892581343650818, -0.9323741793632507, 0.2948039770126343, 0.1113513633608818, 0.8805403113365173, 1.1706064939498901, 0.7301470637321472, 0.1572599709033966, 0.7803440690040588, 0.3124973773956299, 0.5245321393013, 0.18683084845542908, -0.19350214302539825, -1.4424189329147339, 0.03624734282493591, 1.790460228919983, -0.4273088276386261, -1.7429606914520264, -0.5020753741264343, 0.8731003403663635, -0.35394009947776794, -0.10272310674190521, -1.0421183109283447, 1.443814754486084, -1.304738998413086, 0.2416936457157135, -1.0236032009124756, -0.4160492420196533, 1.0438952445983887, -1.1400436162948608, 1.054305911064148, -0.409863144159317, 0.2099309265613556, -0.4287135601043701, -1.0098892450332642, -0.7839744687080383, 0.19380000233650208, 0.5246690511703491, 0.6804664134979248, 0.6043158173561096, 1.4410101175308228, 1.1126717329025269, 1.242075800895691, -1.2686864137649536, 1.046379804611206, -1.127482295036316, -1.1087826490402222, -0.3726669251918793, -1.4814263582229614, 0.22782675921916962, 0.22606025636196136, -0.39060965180397034, 1.2425938844680786, 0.6640313863754272, -1.0432270765304565, 0.23823872208595276, 0.02758275903761387, 0.5360377430915833, -0.5491349697113037, -0.09042193740606308, 0.8076984286308289, 1.1845834255218506, -1.4744820594787598, 2.240675926208496, -0.041514359414577484, -1.4699347019195557, -1.944398283958435, -0.3626677393913269, -1.7371826171875, 0.28207480907440186, 1.1860311031341553, 0.7701683044433594, 1.445739507675171, -0.4524271488189697, -0.9123327136039734, -0.32496777176856995, -0.0219894889742136, -0.1124148964881897, 1.3884588479995728, -1.0040340423583984, -1.6996859312057495, -0.6313786506652832, 0.01429872028529644, 1.263631820678711, 0.011529604904353619, 0.2651107609272003, 0.7025279998779297, 0.5708325505256653, -1.1613991260528564, -1.0149272680282593, -1.4159750938415527, 0.13923141360282898, 0.46841463446617126, 0.7488277554512024, -0.0440451055765152, -0.1191256046295166, -0.7548933029174805, 0.6667020320892334, -0.0013842021580785513, 2.193566083908081, -1.1527105569839478, -0.2638362646102905, 0.9944018721580505, -0.3509885370731354, -1.0960571765899658, 2.4444549083709717, 0.18752293288707733, 0.9406977295875549, 0.8440380692481995, 2.0991644859313965, 0.16925036907196045, 0.1491580307483673, 0.445746511220932, 0.007510773371905088, -2.1053977012634277, 0.6430444717407227, -0.009135553613305092, -1.7917900085449219, -1.23633873462677, 0.8867433667182922, -1.1599712371826172, 1.0920816659927368, -0.302627295255661, 1.2762986421585083, 0.7275998592376709, -0.05575425922870636, -0.029447009786963463, -0.005105639807879925, -0.2586974799633026, 0.6524155139923096, -1.743027925491333, 0.7859103679656982, 1.1815146207809448, -3.2146153450012207, -0.6012689471244812, -0.5921007990837097, -1.4332008361816406, 0.345600962638855, 1.1957634687423706, 2.6612250804901123, -1.429771065711975, -0.31923311948776245, -0.3100185990333557, -0.05212802812457085, -0.3452202081680298, -1.9977341890335083, 0.0950651690363884, -0.8460609912872314, -0.6840306520462036, 1.4889835119247437, -0.4366898238658905, 0.46118828654289246, 0.03859023377299309, 0.6494923830032349, 2.028491258621216, 0.12926959991455078, -0.23942984640598297, 2.0336203575134277, -0.2681220769882202, -0.9245471954345703, 1.337938666343689, -0.894466757774353, -0.8476330637931824, -1.1859060525894165, 1.3549352884292603, -1.11758553981781, -0.3531138598918915, -1.0153638124465942, 0.9793271422386169, 0.5945640802383423, 0.05883823335170746, -1.3814246654510498, 1.6127277612686157, -1.166542649269104, 0.7407540678977966, -0.11442463845014572, -0.4773806035518646, -0.1815994679927826, 0.69325852394104, -1.4701571464538574, -0.27247366309165955, 1.7593507766723633, 1.1905765533447266, -0.31164082884788513, -1.98701810836792, -0.2014358937740326, 1.6812889575958252, -2.1674551963806152, -1.697933316230774, -0.48927634954452515, -0.9154899716377258, 0.1385730355978012, -1.287928581237793, 0.07970687001943588, -0.3266339600086212, 0.5138896107673645, 1.98273766040802, -0.755094587802887, 0.0178277175873518, -0.314092755317688, -0.35505491495132446, 0.01653384603559971, -0.9408391714096069, 1.0903071165084839, 0.776591956615448, -1.5006885528564453, -0.016831448301672935, 0.6455620527267456, -1.5365445613861084, -1.034916639328003, 0.044585999101400375, -1.249345064163208, 0.1818464994430542, 1.0047554969787598, 0.032612964510917664, -0.709877610206604, 0.29250192642211914, 0.05409315228462219, -0.09115204215049744, -0.6253824830055237, -0.4845872223377228, 0.46573054790496826, 0.48378899693489075, -0.5140328407287598, -0.22888165712356567, 1.9594920873641968, 0.6187648177146912, -0.12904313206672668, -0.44208019971847534, 0.9305114150047302, -0.39754730463027954, -1.0598654747009277, 1.0815857648849487, 1.2540764808654785, -0.2690967321395874, -0.4819539189338684, 1.3339403867721558, 1.114384651184082, 0.6093880534172058, -1.1557880640029907, -0.5575458407402039, -1.2814949750900269, 0.8650952577590942, 0.610535204410553, 0.795088529586792, -1.3820974826812744, 0.3663148581981659, -0.7328958511352539, -0.9569233059883118, 0.6117189526557922, 0.08812925964593887, -1.0772820711135864, 0.20917770266532898, 1.3624347448349, 0.8265240788459778, 1.1150397062301636, -0.8249359130859375, -3.2035794258117676, 0.9831935167312622, 0.04179219901561737, 0.007080776616930962, -0.9634760618209839, 1.0306143760681152, 0.07655324041843414, 0.40556490421295166, -2.074035167694092, -0.051786307245492935, -1.126974105834961, 0.009433978237211704, 0.12516024708747864, -0.36256542801856995, -0.03041658364236355, -0.6112262010574341, 0.7268526554107666, 0.38942599296569824, 0.3069703280925751, 0.09241694957017899, -1.34988534450531, -1.4392894506454468, 0.5448161363601685, -0.23457421362400055, -1.342361569404602, 1.283997654914856, 0.15215753018856049, 0.43566805124282837, 1.1965211629867554, -0.8353994488716125, -0.6998729109764099, 0.32425737380981445, -0.12823006510734558, -0.12391287833452225, 0.5355860590934753, -1.8449429273605347, -0.9055057168006897, 0.17307622730731964, -0.711759090423584, 0.208961620926857, 0.18937228620052338, -0.3379700481891632, -1.3383129835128784, 0.05901971086859703, -0.8806131482124329, -0.8234415650367737, -0.38622355461120605, 1.1051503419876099, -1.5204027891159058, -0.7430832982063293, 0.40375280380249023, -1.174658179283142, -0.9001476168632507, -0.9587838649749756, -1.0180189609527588, -1.5455973148345947, 0.038064438849687576, -1.0914220809936523, -0.40622466802597046, -1.0691614151000977, 1.681345820426941, 0.3763115108013153, -0.7305166125297546, -0.3910714387893677, -0.21466487646102905, 1.3755335807800293, -1.1613640785217285, 1.244871735572815, 1.1543786525726318, -2.318638563156128, 0.8031267523765564, 0.2142641693353653, -0.17437388002872467, -0.161343514919281, -0.5751501321792603, -0.011581698432564735, 0.5243083834648132, 0.6865211129188538, 0.23998036980628967, -0.0687749832868576, -1.8867578506469727, 0.2544831931591034, -1.3376322984695435, -0.3772166967391968, 0.4239078462123871, 0.6123040914535522, -1.2795515060424805, -0.49099549651145935, 0.28231266140937805, 0.04009723663330078, -0.6076240539550781, 0.5573052167892456, 1.2032426595687866, -0.872432291507721, 0.7049265503883362, -0.21110942959785461, 1.146553874015808, -0.9193727970123291, 0.21708495914936066, 0.4637189209461212, 1.1266261339187622, 1.6166929006576538, -1.1284334659576416, -0.9970539808273315, 0.9123339653015137, -0.9833629131317139, 0.6541666984558105, 0.9240661859512329, 0.12402810156345367, -2.137118101119995, -0.7492896318435669, -1.1110855340957642, -0.14131738245487213, -2.103181838989258, 2.1325862407684326, -1.6927809715270996, -0.8037968873977661, -1.1457518339157104, -0.9719645977020264, -1.0676308870315552, -0.2830578684806824, 0.27734407782554626, 0.6774065494537354, 2.0415916442871094, -0.5162751078605652, -1.746222734451294, -0.56731778383255, 1.8739597797393799, 0.49988219141960144, -0.4063257575035095, 1.4508970975875854, -2.1119301319122314, 1.8757127523422241, 1.4822090864181519, 0.49573543667793274, -1.507381796836853, -0.5659382343292236, 0.5757305026054382, 0.4957328140735626, -0.9267403483390808, -0.44694945216178894, -0.5085301399230957, 0.039633072912693024, 0.30812394618988037, 1.223422884941101, -2.177947759628296, -0.06109333410859108, -0.6256054043769836, 1.092389464378357, -0.988196611404419, -1.8012298345565796, 0.3001019060611725, 0.22420856356620789, -0.5023160576820374, 0.4662063717842102, -0.10903016477823257, -0.07256835699081421, 0.5337608456611633, 0.9984337687492371, 1.0913761854171753, 0.5669671893119812, -0.4243207573890686, 0.8250559568405151, -1.9142985343933105, -0.3878549337387085, -1.2398477792739868, -0.6694236397743225, -0.2822183072566986, 0.2862716019153595, -0.10359333455562592, -0.36705711483955383, 0.40446239709854126, 0.2701777517795563, 1.7192827463150024, -0.7822006344795227, -0.12312570214271545, -1.0630978345870972, 2.082765579223633, 0.4523835778236389, -1.635606288909912, 1.3828555345535278, 0.13225427269935608, -1.580717921257019, 0.37321650981903076, 0.09672406315803528, 0.3637131154537201, 2.370697021484375, -0.20173683762550354, 0.4919382333755493, -0.8418593406677246, -1.1683002710342407, -0.21779265999794006, 0.23904091119766235, 0.48595693707466125, -1.0804489850997925, -0.8419842720031738, -0.44939470291137695, -0.9975191354751587, -0.04210401326417923, 0.38434937596321106, 2.4214885234832764, 1.0145052671432495, -1.0582950115203857, 0.3029915988445282, 0.9074814915657043, 1.3355478048324585, 0.1517847627401352, -0.18962523341178894, 0.7248372435569763, 0.8431436419487, -1.1023987531661987, 0.8220888376235962, -0.8911628127098083, -0.9796738028526306, 0.6099364161491394, -0.8082523941993713, 0.03645014017820358, -0.409138947725296, 1.214614987373352, -0.9200888872146606, 0.3060137927532196, 0.13913936913013458, 0.10639532655477524, 0.44500797986984253, -3.203082799911499, 1.4517673254013062, -0.7256460785865784, -0.1220734715461731, 0.8523116111755371, -0.11678017675876617, -0.5131821036338806, -1.6742833852767944, 0.570378839969635, -0.5946875214576721, -1.0077931880950928, 0.5531360507011414, 0.4950561225414276, 0.28604719042778015, -0.49666130542755127, 0.13582366704940796, -0.21182923018932343, 1.1042848825454712, 0.2544480860233307, -1.138965129852295, 0.5087699294090271, -1.1620334386825562, 0.3527761995792389, -0.8908843994140625, 0.6343719959259033, 0.7017423510551453, 1.016243815422058, -0.07842791080474854, -0.06941735744476318, -1.880908727645874, 0.9584680199623108, 0.5357762575149536, -0.9492800235748291, -0.15346196293830872, -1.46713125705719, -1.9066394567489624, 1.1593010425567627, -2.566462278366089, -0.2226918637752533, -1.1006532907485962, 0.37330469489097595, -0.01930602267384529, 1.1442606449127197, -0.15764808654785156, -2.0136191844940186, 1.410086750984192, 0.6828337907791138, 0.11398646235466003, -2.299961805343628, 1.5733635425567627, -0.684349536895752, -0.39387261867523193, -0.8963472843170166, -1.1181737184524536, -0.9847855567932129, -0.060494739562273026, 0.36087867617607117, -0.6349561810493469, -1.2082853317260742, -0.7497269511222839, -1.1140104532241821, -0.7895422577857971, -0.29930436611175537, 0.6991630792617798, -0.44567424058914185, -0.37488117814064026, 0.2655991017818451, 1.7204136848449707, -1.6809580326080322, 0.9482569694519043, 1.2306418418884277, -0.6561703681945801, -0.15520013868808746, -0.22879555821418762, -1.123473882675171, -0.11591646075248718, -0.20804479718208313, -0.31121712923049927, -0.468718558549881, -0.37789323925971985, -0.327399343252182, 0.5399198532104492, -1.1841323375701904, -0.5952889919281006, -1.5883517265319824, -2.454103708267212, 0.9916375279426575, 0.6210103631019592, 0.1802121102809906, -0.12847302854061127, 0.938897430896759, -0.013287273235619068, 1.5096797943115234, -0.8945862054824829, -0.10472966730594635, 0.816412091255188, -0.2460654377937317, 0.16388842463493347, 0.9091240763664246, 0.14150036871433258, 0.23225925862789154, -1.3583236932754517, -0.04782184213399887, -1.354437232017517, -0.11817513406276703, -0.1708226501941681, 0.5997289419174194, -0.4670644700527191, 0.1994340419769287, 0.5415805578231812, -1.1625542640686035, 1.9879460334777832, -0.5677573680877686, 0.03253963589668274, 0.41547951102256775, -1.5124266147613525, -0.21129412949085236, -1.246012806892395, 1.7113341093063354, -0.7441405057907104, 1.1641346216201782, 1.766432285308838, -0.8216919898986816, -0.5701112151145935, 1.2628321647644043, -0.6154837012290955, -0.016993876546621323, 0.1922127604484558, 1.1408172845840454, -2.119206666946411, -1.7232544422149658, -0.2428646683692932, 1.0227712392807007, 0.276294082403183, -2.4015331268310547, 1.3893489837646484, -0.43555811047554016, 0.0019652030896395445, 0.03547794744372368, -0.6375299096107483, -0.19454774260520935, -1.3131481409072876, -0.8689770698547363, -2.0914292335510254, 0.02387162297964096, 1.329392671585083, -1.9275168180465698, 0.8587230443954468, 0.4686393737792969, 0.12928619980812073, -0.5177517533302307, 1.146851658821106, -0.004767841659486294, 2.497545003890991, -0.6294209957122803, -0.427844762802124, -0.8099836707115173, 0.5655243396759033, -1.4584065675735474, -0.5219356417655945, 0.4177710711956024, 0.7909135222434998, -0.04868296906352043, -0.9156410098075867, 1.0516276359558105, 1.3854714632034302, 0.3189082443714142, 0.04684307798743248, -1.6195954084396362, 0.2821349799633026, 0.9203072786331177, -0.6164823770523071, -0.6566616892814636, 0.8040162324905396, 0.6517367362976074, -0.30331137776374817, 1.0807862281799316, 0.7127394676208496, 2.030946731567383, -1.0790846347808838, 0.24022680521011353, -1.6360208988189697, -0.8118186593055725, -0.8660256266593933, -1.1254801750183105, 0.8634752035140991, -1.580708384513855, 0.2793188691139221, 0.7233896255493164, 0.656808078289032, -1.5467678308486938, 0.23712055385112762, 1.4247804880142212, -0.3275781273841858, 1.7750376462936401, -0.015047537162899971, 0.8674896955490112, 0.2925567030906677, 2.444518566131592, 1.680747628211975, 0.8742551207542419, -0.6652087569236755, 0.16159190237522125, -0.7670936584472656, 0.6478346586227417, 0.706834614276886, -0.37175220251083374, -1.0016112327575684, 0.3604520559310913, 0.6965556144714355, 0.2850187122821808, 1.8458173274993896, 0.9783127307891846, -2.4639394283294678, 1.0148786306381226, 0.1046748012304306, -0.5288130044937134, -0.6512414216995239, 0.8457295894622803, 0.7672117352485657, 0.031989410519599915, -0.3203143775463104, 0.5769633650779724, -0.04158563166856766, 2.418036699295044, 0.5133448243141174, -1.4305084943771362, 0.6272096633911133, -0.4280339479446411, 1.0118449926376343, 0.23476454615592957, 0.09508205205202103, 0.6515514850616455, 0.598992645740509, 1.1214667558670044, 0.8783706426620483, 0.5318827629089355, 0.33432304859161377, -0.6760551929473877, -0.806361973285675, 0.7715712785720825, 0.33430591225624084, -0.12313418835401535, -1.4081017971038818, -1.2031223773956299, -1.4165593385696411, 0.8694505095481873, 0.5526388883590698, -0.3458001911640167, 0.9594976902008057, -0.14142677187919617, 0.11852381378412247, 0.02182265929877758, -0.6040094494819641, 0.2509385645389557, 2.167222261428833, -0.14237447082996368, -0.5446720719337463, -1.759462833404541, 0.1778249740600586, 0.03995511308312416, -0.6889740824699402, -1.6676462888717651, 2.3980648517608643, 0.19575217366218567, -1.340204119682312, -1.2365719079971313, 0.4661998748779297, 0.33292725682258606, -0.6088696718215942, 1.5502444505691528, -0.4178273677825928, 0.8807932138442993, -0.40075212717056274, 1.2355221509933472, 0.015664726495742798, 0.09428612142801285, -1.8509138822555542, 0.8598519563674927, -1.0213075876235962, 0.7689033150672913, 0.9308406710624695, 2.576183557510376, -0.46990254521369934, -0.7756111025810242, 0.00829008687287569, 0.574577271938324, 1.1630690097808838, 2.0548765659332275, 0.3515290915966034, -0.004263365641236305, 0.4028945565223694, 1.4692364931106567, -0.21068653464317322, -0.6861279010772705, -0.043540649116039276, 0.4406880736351013, 1.1762722730636597, -1.2271063327789307, -0.6267465949058533, 0.344110906124115, 1.3288300037384033, -0.8081284165382385, -0.39648962020874023, -1.436735987663269, 0.41864579916000366, 1.273290753364563, -0.1211184710264206, 1.1214160919189453, 0.8905428051948547, -1.406156301498413, -0.40871337056159973, 1.1081644296646118, 0.9327327609062195, -0.7782814502716064, -0.9108805656433105, 0.42662128806114197, -1.112223744392395, -0.8120625019073486, 0.13670121133327484, 1.4072552919387817, 0.6230705976486206, 1.2730578184127808, -2.5548245906829834, -0.004522710107266903, -0.02590368129312992, 0.9698587656021118, -0.8163802027702332, 1.4448431730270386, -0.21937081217765808, 0.3379487693309784, -1.5869359970092773, -0.7766532301902771, 1.0419894456863403, -1.0276082754135132, 0.1478426605463028, 0.3966038227081299, -0.402538001537323, -0.3420170247554779, 0.09722079336643219, 1.8964617252349854, 3.3444552421569824, -0.7483017444610596, 2.2364437580108643, 1.3398321866989136, 1.7845752239227295, -0.6603100299835205, 0.4868094027042389, 1.323607087135315, -0.12386403977870941, -0.9459167122840881, -1.231278419494629, -0.7179662585258484, -0.5547631978988647, 0.23420952260494232, 0.15071062743663788, 0.35021039843559265, 1.2287297248840332, -0.3970717787742615, -0.28530481457710266, -0.3019736409187317, 1.5371094942092896, 0.11250803619623184, 1.4318288564682007, 1.278382658958435, -1.883428931236267, -1.07811439037323, -0.819861888885498, -0.3576356768608093, 0.40920528769493103, 1.6354719400405884, -1.01010262966156, 1.0671294927597046, -0.6314973831176758, 1.4522956609725952, -0.9134172201156616, 0.9491735100746155, -0.402226060628891, -0.6384910941123962, -0.38430675864219666, -0.15348544716835022, 0.5079593062400818, 2.7628207206726074, 0.8139608502388, 1.3452389240264893, -0.774458646774292, -0.3129848539829254, 0.7751654982566833, -0.7240068912506104, 1.05713951587677, -1.2685528993606567, -0.16355107724666595, -1.567428708076477, -0.12260370701551437, 0.12281932681798935, -0.021496731787919998, 0.9691195487976074, 0.999079167842865, -1.7075871229171753, 0.5913239121437073, 1.059393286705017, 0.7216893434524536, -0.5928049683570862, -0.07183697074651718, -0.137559175491333, 1.0058101415634155, 0.3185703158378601, -1.0699236392974854, -1.2216876745224, -0.20293200016021729, -0.32477307319641113, -0.6118754148483276, 0.6058660745620728, -0.8867148160934448, 1.6599324941635132, 0.09938207268714905, -0.5281431078910828, -1.1161446571350098, -0.7591534852981567, -0.5206742882728577, 0.6647231578826904, 1.170001745223999, -0.5181229114532471, 0.26025304198265076, 1.07493257522583, 1.055009126663208, 0.1852971464395523, 0.37392598390579224, 2.2859504222869873, 0.06417424976825714, 0.6362459659576416, 1.4491653442382812, -0.3426150977611542, -0.21377012133598328, -0.5907989144325256, 0.4935898184776306, 0.7775622010231018, -0.2595128118991852, 1.0971883535385132, 0.4215371012687683, -0.2677910327911377, -0.07219504565000534, -0.9294326901435852, 0.724390983581543, -0.3816373646259308, -1.9662662744522095, 0.3893186151981354, 1.2467684745788574, -0.15385451912879944, 0.2632564604282379, -0.846156895160675, 0.21111129224300385, -1.8608959913253784, -0.5489915609359741, 0.045826416462659836, 2.079566717147827, -1.5880353450775146, -0.3261550962924957, 0.1047460287809372, 0.7642428278923035, -0.9388701319694519, -0.5648359060287476, -0.5977325439453125, 1.0291610956192017, -0.6145197749137878, 0.059254009276628494, 0.4724206328392029, -0.8051067590713501, 1.3789445161819458, 0.41282233595848083, -0.1534854769706726, -0.4764280617237091, -0.8780924081802368, -0.48465847969055176, -0.4697609543800354, 0.1929253190755844, -0.35030877590179443, -1.7556957006454468, 1.1511205434799194, -0.9602183699607849, -0.6232109069824219, 0.009150292724370956, 0.6393671631813049, 0.13743029534816742, 0.4823996424674988, 0.14417140185832977, -0.11273748427629471, -1.4830782413482666, -0.8847295641899109, -1.1465659141540527, -1.6541370153427124, 0.1936609148979187, 0.45625004172325134, -1.0248485803604126, -1.3654237985610962, -1.2893351316452026, -0.19136331975460052, 0.31029292941093445, 0.2079179435968399, -1.6671079397201538, -0.09670911729335785, 0.2266029268503189, 0.9627143144607544, 0.7738050222396851, 0.9711207151412964, 2.206944227218628, 0.824409008026123, -0.05843810364603996, 0.267820805311203, -1.12250554561615, -0.10532300174236298, -0.6429513096809387, 0.2611487805843353, 0.7845023274421692, 1.9977636337280273, 0.8242723941802979, -1.1526647806167603, -0.029171783477067947, 0.38472139835357666, -0.011338826268911362, -1.3897651433944702, -0.33877089619636536, -1.4131520986557007, -0.5297800898551941, 1.6036005020141602, 1.1282861232757568, -1.4714072942733765, -0.6471872329711914, 0.3332087993621826, 0.7299787998199463, -0.10733180493116379, 1.9600982666015625, -0.012071357108652592, -0.8101524114608765, 0.2872474491596222, -1.0524392127990723, -0.10879820585250854, 0.3244929909706116, -0.5763409733772278, -1.197799801826477, -0.46234455704689026, 1.1138628721237183, -0.20610001683235168, 1.0355318784713745, 0.4004140794277191, -0.05234348401427269, 0.17072953283786774, -0.14029668271541595, 1.4656527042388916, -0.6796729564666748, 1.8948893547058105, 0.9598857164382935, -1.5775326490402222, -0.5180568695068359, 0.32240504026412964, 0.624565601348877, 1.319674015045166, 0.7676274180412292, -0.34072887897491455, -0.8101325035095215, 0.9022361040115356, -0.3612217605113983, 1.7217392921447754, 1.5942801237106323, 0.217799574136734, 2.406083345413208, -1.5610318183898926, 0.18877297639846802, 0.29538920521736145, 0.35171541571617126, -0.27070945501327515, 0.2900466322898865, -1.071478009223938, 1.3169032335281372, 0.22243180871009827, -0.47900307178497314, -1.264075517654419, -1.1040458679199219, -0.18916241824626923, -0.0240673515945673, -0.2684450149536133, 0.9370107054710388, 0.3253338634967804, 0.7172061204910278, 0.7215079069137573, 0.9290335774421692, 0.00457735313102603, -1.1454106569290161, -0.02692125365138054, -1.2878916263580322, -0.7257997393608093, 0.816853404045105, -0.16463053226470947, -0.2595512866973877, 2.7428791522979736, 1.6151926517486572, 0.6735759973526001, 1.116239070892334, 1.085422396659851, -0.3649634122848511, 1.3694124221801758, -1.1162688732147217, 0.4000365734100342, 0.4502962827682495, 0.2659071683883667, 0.250870943069458, -0.7190874218940735, -0.6853755712509155, 1.5969197750091553, 0.8278754949569702, -0.46982431411743164, -0.6904231309890747, 0.013972758315503597, -0.2673388421535492, 0.46438562870025635, -2.3837039470672607, -0.25568559765815735, -0.6696827411651611, 0.47834500670433044, -0.33163201808929443, 1.8455169200897217, -1.1427953243255615, 0.7445534467697144, -1.018255591392517, 0.5581691265106201, 0.6986381411552429, 0.3180556893348694, -0.6909104585647583, -0.5834816098213196, -0.346834272146225, -1.4209141731262207, 0.16470380127429962, 1.6393187046051025, 0.11428658664226532, -1.0490307807922363, -0.8367331624031067, 1.483808159828186, -0.8332757353782654, 1.8338184356689453, -0.22274518013000488, -2.4569125175476074, 0.673571765422821, -0.387967050075531, -0.1252100169658661, 1.7272135019302368, 0.5978400707244873, 0.21120217442512512, -0.4521109163761139, 1.2113453149795532, -0.9827516078948975, -0.4653650224208832, 0.7187369465827942, -0.060173481702804565, 0.08734405040740967, 1.3905736207962036, 1.4019739627838135, 2.605156183242798, 0.6556360721588135, 0.240810364484787, 0.33905482292175293, 1.5191478729248047, -0.3601841628551483, -0.7121778130531311, 1.0001044273376465, 1.1564123630523682, 0.3023459315299988, -0.618619978427887, 0.6625850200653076, -1.1946879625320435, -1.5312328338623047, -0.43844690918922424, 1.0496015548706055, -0.26215729117393494, 0.6581824421882629, -0.39555996656417847, 0.9399080872535706, 1.2138715982437134, -1.4576643705368042, -0.15157410502433777, 0.5558156967163086, 2.1659319400787354, -0.3644491136074066, 2.1469669342041016, 0.32177072763442993, 0.008564471267163754, -2.826347827911377, -1.1816980838775635, 1.2944236993789673, -1.0867537260055542, 0.8355098962783813, -0.12605354189872742, -1.198241949081421, -2.097337007522583, -0.12779860198497772, 0.12518265843391418, -0.3735286593437195, -0.014826045371592045, 1.14253568649292, -1.8207602500915527, 0.2050814926624298, 0.3597607910633087, -0.7662248611450195, 1.944365382194519, 1.46615731716156, -0.5097607970237732, -1.3234708309173584, 0.3881814777851105, -2.6573801040649414, -1.2910354137420654, -0.011349269188940525, 2.4876627922058105, -0.2550719082355499, 1.3426618576049805, 1.7177820205688477, 0.4341757893562317, 0.14012640714645386, -1.0533077716827393, -1.5528069734573364, 0.12272676080465317, 0.8985345959663391, -0.218898743391037, -0.3487106263637543, 1.6024250984191895, -0.5975432991981506, -1.45899498462677, -0.37280166149139404, -1.7741782665252686, -0.3238808810710907, 0.7250666618347168, 0.1093381941318512, -1.8898212909698486, 0.4209129512310028, -0.30207914113998413, 0.9628042578697205, 0.5763722658157349, 1.6505986452102661, 1.2611618041992188, -1.2627142667770386, -0.1864190399646759, 1.4227575063705444, 2.281820297241211, 0.08886835724115372, -0.7445703148841858, -0.067134790122509, 1.963754415512085, -0.06525369733572006, 0.32139772176742554, -1.207210659980774, -0.12486029416322708, 0.27720898389816284, -2.2470812797546387, -0.28135091066360474, -2.040316343307495, 0.1135968491435051, -1.850221872329712, 0.8235236406326294, 2.2223000526428223, -0.37382158637046814, -0.04572249576449394, -1.1805723905563354, -0.35910195112228394, -0.16800203919410706, -0.2170497477054596, -1.6936843395233154, -0.47547000646591187, -0.11455245316028595, -0.5056387186050415, 1.1852023601531982, 0.5587724447250366, -0.07975392043590546, -1.4274334907531738, 0.48689404129981995, 0.4085441827774048, -0.44793519377708435, -1.0661624670028687, 0.06410501152276993, 0.5270939469337463, -0.08725164830684662, -0.12415757775306702, -0.818548321723938, -1.5167760848999023, 0.4567563533782959, 0.42420151829719543, 0.30390000343322754, 0.5291468501091003, -0.33344075083732605, -0.850364625453949, -1.293038249015808, 0.8471203446388245, -0.008475800976157188, 0.36603567004203796, 0.7429283261299133, -0.7747461795806885, -0.3702850639820099, -1.3255177736282349, 1.7290563583374023, 1.4804259538650513, 0.7033079266548157, 1.2392463684082031, 0.522980809211731, -1.2814505100250244, 0.11038369685411453, 1.1748147010803223, -1.1851388216018677, -0.9725620150566101, 0.3180272877216339, -0.8313809633255005, -0.8856648206710815, -0.3029040992259979, 0.2801581621170044, -0.9984162449836731, 0.6719560623168945, -0.7061339020729065, -0.5474186539649963, 0.030148781836032867, 1.6531256437301636, -0.5811824798583984, 0.9696993231773376, -0.5086974501609802, 0.4860523045063019, -0.4274837374687195, 1.5988003015518188, -1.511915683746338, 0.5815023183822632, -0.5956908464431763, -0.46758168935775757, 0.39284443855285645, -1.509130835533142, 0.18588250875473022, 0.13656260073184967, 0.8716983199119568, -1.1352241039276123, 0.0140846511349082, 1.1095070838928223, -1.808609127998352, -0.42882785201072693, -0.7142760753631592, -1.1887083053588867, -0.6587338447570801, -1.2453349828720093, 0.48803088068962097, -0.28649985790252686, -0.852383017539978, -0.19898715615272522, 1.0790517330169678, -0.9294533729553223, 1.1541823148727417, 0.7273261547088623, 0.12640593945980072, -0.6680108904838562, -0.18402758240699768, -0.09874013066291809, -1.4963310956954956, 0.36325833201408386, 1.3561779260635376, 0.05860058590769768, -0.3743647336959839, -0.9601685404777527, 0.012951213866472244, 0.7587406039237976, -0.11036384105682373, 0.09698937833309174, 1.846602201461792, 1.1666678190231323, 0.36601269245147705, -0.5321223735809326, -2.2677571773529053, 0.3076916038990021, 0.841796338558197, -1.3100007772445679, 0.18253973126411438, -0.7038148641586304, 1.1721495389938354, 0.08831077069044113, 0.32954657077789307, 1.453469157218933, 0.4646378755569458, -0.6205803751945496, 1.1031476259231567, -0.3669421672821045, 1.6782180070877075, -0.23120048642158508, 0.3958864212036133, 1.3985097408294678, -0.6575828194618225, -1.1947003602981567, 0.06349921971559525, -0.5571338534355164, 0.062496282160282135, 1.3075826168060303, -0.6379991173744202, 1.013651967048645, -0.5580053329467773, 1.5572521686553955, -0.8642040491104126, 0.44899237155914307, 0.06803380697965622, -0.003931927494704723, -0.08969022333621979, -0.35728245973587036, -1.192811369895935, 0.29430335760116577, -0.595757007598877, -0.569036066532135, 0.8300830721855164, 1.0204041004180908, -0.33434373140335083, -1.033434271812439, 0.31690242886543274, -0.13398635387420654, -1.3109135627746582, 0.09978130459785461, -0.6345042586326599, 0.36438027024269104, -0.6807299256324768, -0.49088284373283386, -0.514798104763031, -1.052395224571228, 0.5373397469520569, 0.9794329404830933, -0.1968528926372528, 0.5211384296417236, -0.6382368206977844, -1.5538018941879272, 1.311743140220642, -1.429614782333374, 0.3493933379650116, -0.21848350763320923, -0.6895962357521057, -0.31876587867736816, 0.3230830132961273, -1.239217758178711, -0.6673682928085327, -1.2013922929763794, -1.8400133848190308, 0.20080944895744324, -0.9175202250480652, -0.12702837586402893, 0.3618021607398987, -0.22291231155395508, 0.42229437828063965, -0.8174274563789368, -0.47974035143852234, -0.45267465710639954, -1.7444008588790894, 2.638714075088501, 1.1310797929763794, -1.1885147094726562, 0.849372386932373, -0.1742049902677536, 0.12867863476276398, -0.9228097200393677, -1.9458544254302979, -0.7164071798324585, -0.7407848834991455, 2.076306104660034, -0.258635550737381, 0.5145148634910583, -0.4377042353153229, -0.3379611670970917, 0.6511198282241821, -0.8139669895172119, 0.16334068775177002, -0.6123086214065552, 0.10690762102603912, 0.17414630949497223, -0.8145079612731934, 0.3499794602394104, 0.45228707790374756, 0.9280651211738586, -1.1473840475082397, -1.1888258457183838, -0.5380429029464722, 0.6421610116958618, 1.2704148292541504, -0.224731907248497, -1.7469558715820312, 0.5624826550483704, -0.16326439380645752, -1.1641590595245361, 0.6275311708450317, 0.9393172860145569, -0.026910878717899323, -0.48308876156806946, -1.1200292110443115, -0.4487360119819641, 0.5152908563613892, -0.8104248046875, -0.9164648056030273, -0.29618334770202637, -0.43402034044265747, 0.18717075884342194, 0.7636762857437134, -1.579512119293213, 1.391281247138977, -1.0128482580184937, -1.0695840120315552, -2.012615442276001, -0.07596617937088013, 0.010150330141186714, -0.26721784472465515, -0.7938060164451599, 0.5853992700576782, -0.20784522593021393, -1.1710047721862793, -0.41088053584098816, -1.1816209554672241, -0.20355920493602753, 0.4084969758987427, -0.5486438274383545, 0.827020525932312, -0.27751627564430237, 0.3158760070800781, 0.32149508595466614, -0.6295222640037537, 1.1654998064041138, -1.3669495582580566, -0.48097217082977295, 0.3662920594215393, 0.3399270474910736, 1.0146199464797974, -0.4615485370159149, 0.3575235605239868, 0.32419878244400024, 1.122606873512268, 2.084001064300537, -0.46397754549980164, -0.026170674711465836, 0.1404145359992981, 0.31569162011146545, 0.6076194047927856, -1.9914706945419312, 1.264796257019043, 0.36720189452171326, 1.4447160959243774, 1.0592824220657349, 1.0576295852661133, -1.471858263015747, -0.3785912096500397, -1.2160104513168335, 0.11329247057437897, -0.47294655442237854, -1.1189188957214355, -0.07532647997140884, 0.39353466033935547, 0.5322502851486206, -1.403963327407837, -0.27188003063201904, -0.2057611048221588, -0.39103007316589355, 0.6638065576553345, -0.7581067085266113, -0.035188913345336914, 0.10398975759744644, 0.22397562861442566, 0.3779880404472351, 1.5740610361099243, 0.42975351214408875, 0.8673452734947205, 0.2549538016319275, 0.5973795056343079, 0.7410573363304138, -0.664880633354187, 0.7156314849853516, -0.8043702244758606, -0.27355945110321045, 1.9792938232421875, 0.15820369124412537, -0.6501301527023315, 0.6030388474464417, 0.9448968172073364, -1.951359510421753, 0.5439221262931824, 0.8923115730285645, -0.38117021322250366, 0.3484594523906708, -1.2483490705490112, -0.2180732637643814, 0.06209937855601311, 1.640882968902588, -0.36146092414855957, -0.3500233590602875, 0.02111193537712097, 0.4043559730052948, -0.6920906901359558, 0.8524792790412903, -1.735849380493164, 1.9470629692077637, -1.3874655961990356, 0.8694530129432678, -0.7485734224319458, -0.49220576882362366, 0.11894628405570984, 1.077414631843567, 0.6396403312683105, 0.3616795241832733, -0.9317037463188171, 1.1551599502563477, -0.16382671892642975, -0.5009229183197021, -1.36617112159729, 0.3677245080471039, -0.4057966470718384, 1.264066457748413, 1.897409200668335, 0.06455125659704208, -2.1503443717956543, 0.14090310037136078, 1.1294653415679932, 0.543283224105835, -2.1377463340759277, 0.5615034699440002, 0.4440698027610779, 0.7632085680961609, -0.24100138247013092, 0.1840084046125412, -0.3968845307826996, 0.4375648498535156, 2.1371500492095947, 1.0370451211929321, 0.1335168182849884, -0.9618202447891235, 1.627387285232544, 1.2674428224563599, -1.1400583982467651, 0.7140840888023376, -1.0668846368789673, 1.8260533809661865, 0.6482726335525513, 0.9173588156700134, -1.0190250873565674, -1.3098095655441284, 1.2314990758895874, 1.6060709953308105, 0.15217038989067078, 0.3922354578971863, 0.01163938082754612, -0.8665156960487366, -1.7102864980697632, -0.007526862435042858, -0.6080150008201599, -0.6390305757522583, 0.822708010673523, -0.212367445230484, 0.7707567811012268, 1.1626129150390625, -0.4845044016838074, -0.3706105053424835, 0.23712624609470367, -0.16200074553489685, -1.0963565111160278, 0.49597522616386414, -0.3061038553714752, 1.396453857421875, 0.5704533457756042, -0.8803266882896423, 2.1380157470703125, -0.7028799057006836, -0.1639699786901474, -0.9813948273658752, -0.43873974680900574, 3.176600694656372, -0.17850197851657867, 0.5299438238143921, 0.8138202428817749, 1.3846032619476318, 1.015987515449524, -1.870841145515442, 0.47973230481147766, -1.394114375114441, 0.0822618156671524, -1.752724051475525, 3.144345760345459, 1.4893947839736938, 0.5371492505073547, -0.7217385172843933, -0.8429660797119141, 0.8993360996246338, -1.4328457117080688, 0.45286843180656433, 0.3080751895904541, -2.289513111114502, 0.14861200749874115, 0.9428974986076355, -1.2421168088912964, 0.6643730401992798, 1.0878183841705322, 0.3703709542751312, -0.9414041638374329, -0.4961472153663635, 1.0697096586227417, 0.37718483805656433, 1.2204631567001343, -1.879648208618164, -0.5546815991401672, 0.27157020568847656, 1.0003544092178345, -0.7982860803604126, -1.2460533380508423, 0.2787255346775055, 0.47502392530441284, -0.6739305853843689, -0.3358963131904602, -0.0008726405212655663, -0.3930317759513855, 1.3762705326080322, 1.0178722143173218, -0.45045724511146545, 1.054478645324707, 0.8348461389541626, -1.1448379755020142, 0.004672336392104626, -1.681544542312622, 2.3231427669525146, -0.20065349340438843, 0.8365398049354553, 0.8724905252456665, 1.3147343397140503, -1.4672901630401611, -0.7837239503860474, -1.4874815940856934, 0.015375501476228237, -1.4576724767684937, 1.0063588619232178, 0.5015835762023926, -0.5121801495552063, -2.2966790199279785, 0.38696980476379395, 0.3051695227622986, 0.08540485799312592, -1.056839942932129, 0.15113486349582672, -0.4947803020477295, -0.6762261390686035, -0.47728756070137024, -1.4203832149505615, -0.9415287375450134, 1.3176755905151367, -1.359532356262207, 0.27121463418006897, -0.3850051462650299, 0.1808965802192688, 0.35605016350746155, 1.1344279050827026, 1.4723308086395264, 0.14990323781967163, -0.0051281326450407505, -1.6665558815002441, 1.1890883445739746, -1.9271034002304077, 0.8061094284057617, -0.4617708623409271, 0.7991672158241272, -0.9873459935188293, 0.628206193447113, -0.9233678579330444, -0.08255738019943237, 0.10000810772180557, -0.6210971474647522, 1.5503555536270142, -0.9758661389350891, 1.153963327407837, 0.21509353816509247, 0.3082019090652466, -0.6418778300285339, -0.2595829367637634, 1.151336908340454, -1.1203341484069824, 1.5237908363342285, 0.26623332500457764, -0.5109491944313049, 0.3349935710430145, 2.216007709503174, 1.181624412536621, 1.8119266033172607, -1.023209810256958, 0.10391966998577118, -1.8790966272354126, 2.634564161300659, -0.10617965459823608, -1.3920958042144775, 0.04071876034140587, -0.5676181316375732, -1.415254831314087, 1.795035481452942, -0.7161480188369751, -0.5387866497039795, -0.2769041955471039, -0.27726060152053833, -2.1833484172821045, 0.5854549407958984, 1.4549111127853394, 0.12398575246334076, 0.41763585805892944, 2.1373941898345947, -2.22916316986084, -1.9430521726608276, -0.002851054770871997, -0.467095285654068, -1.084436058998108, 0.6052959561347961, 0.04333991929888725, 0.6829099059104919, 1.4362571239471436, -0.1677158623933792, -0.8805029988288879, 0.03521539643406868, 0.4894430935382843, -0.30820152163505554, -0.6669467091560364, -1.0459845066070557, -1.0010175704956055, -0.938788890838623, 0.41737544536590576, -1.846453309059143, 0.05606571212410927, -0.2664774954319, -0.02852047048509121, 1.2409155368804932, 0.4539417624473572, -0.4860858619213104, 0.47947362065315247, 0.7629753947257996, -0.17264054715633392, -1.8788769245147705, -0.8928012847900391, 1.0907715559005737, 1.1275150775909424, 0.6027831435203552, 0.10799986124038696, -0.19296646118164062, -1.0091887712478638, 1.1315244436264038, -0.6180170774459839, -0.6875830292701721, -0.5608851909637451, 1.4288166761398315, -0.24490207433700562, -1.082808256149292, 0.02373848296701908, 1.493153691291809, 1.7890855073928833, -0.44216257333755493, 0.49778419733047485, -1.4100806713104248, -0.7678461670875549, 0.48688048124313354, 0.18415285646915436, 0.07389455288648605, 0.26713523268699646, -0.32921984791755676, 1.1932841539382935, 0.4843473732471466, -0.8476508855819702, -0.8927335143089294, 1.0351608991622925, -2.3188886642456055, -0.5460962057113647, -0.8814705014228821, -0.25440964102745056, 1.6841456890106201, 0.6311766505241394, -0.29846200346946716, -0.4351060390472412, 0.6245079040527344, -0.7477034330368042, -2.2592973709106445, -1.258130669593811, -0.5231677889823914, -1.208827018737793, 0.02645627222955227, 1.4955127239227295, -1.64454984664917, 0.06228623911738396, 1.039467215538025, -0.4608570337295532, 0.45357558131217957, -0.4998703598976135, 0.32484737038612366, 1.174324631690979, 0.7130368947982788, -0.4994353652000427, -1.107321858406067, 0.6750911474227905, -1.2511786222457886, -0.8790145516395569, 2.415374517440796, -1.1047532558441162, -0.14017072319984436, -0.5787460803985596, -0.44074806571006775, 0.8664129972457886, -2.3174550533294678, 1.0475523471832275, -0.49457746744155884, -0.8366247415542603, 0.13306334614753723, -0.9837217926979065, -0.2681278884410858, -0.7711582779884338, 1.5622872114181519, -0.5780207514762878, -1.128220558166504, 0.3830677568912506, 0.030809590592980385, 1.0673905611038208, 1.4730916023254395, -0.13323542475700378, -3.0351152420043945, 0.8978908658027649, 0.3562190532684326, 0.10049888491630554, -1.508289098739624, 0.9866666197776794, 0.1281082034111023, 0.8565880656242371, 0.8153287768363953, 0.8371855616569519, 0.7545833587646484, 0.42106571793556213, 0.366300106048584, -0.8502461314201355, 0.551947295665741, -0.43700751662254333, 0.735600471496582, 0.45807626843452454, -1.0977572202682495, 0.2858961224555969, 0.8014999628067017, -0.2937461733818054, -1.1591945886611938, 0.19517073035240173, 1.2153759002685547, -1.0573796033859253, -0.11017793416976929, 1.116023302078247, -0.693463146686554, 2.823009967803955, 0.868057370185852, -1.8370273113250732, 1.0409374237060547, 1.1148585081100464, -0.08368081599473953, -0.19173721969127655, -1.1548926830291748, 1.6216776371002197, -0.16215278208255768, 1.894215703010559, -0.1807955801486969, 0.1862727403640747, 0.15621179342269897, -0.2723240256309509, -0.22145168483257294, 0.6652965545654297, -1.877119541168213, -1.8943842649459839, -0.7796894311904907, 1.159303069114685, -0.9664576053619385, 0.44690924882888794, -0.935904860496521, -0.41615864634513855, 0.9646828770637512, -1.9920504093170166, 1.9398378133773804, -0.10027526319026947, -1.285376787185669, 0.9465423822402954, 0.06015099212527275, 0.266937255859375, -0.6243382096290588, -0.9141787886619568, -0.4540095627307892, 0.19313903152942657, -2.6157822608947754, -2.1778674125671387, -1.4464244842529297, 0.09864036738872528, 0.5633237957954407, -0.8763555288314819, 0.17061515152454376, 1.7482603788375854, -1.0583628416061401, 0.23719428479671478, -1.5796997547149658, 0.13953657448291779, -0.36839866638183594, 0.18188780546188354, -0.3121799826622009, -0.5613601207733154, -1.214308500289917, -1.0097978115081787, 0.42385539412498474, -0.34753209352493286, -0.42284488677978516, 0.07266784459352493, 0.08723462373018265, -0.24149379134178162, 0.9733186960220337, -0.17142091691493988, -0.2401580661535263, -1.2843836545944214, -0.9031549096107483, -0.9153516292572021, -0.2455308586359024, -1.9853440523147583, 0.21364857256412506, -0.08441145718097687, -0.518179714679718, 1.9031466245651245, -0.12356329709291458, -1.6360688209533691, 0.7750474810600281, 0.19218742847442627, -0.5053431987762451, -0.5074941515922546, 1.5203818082809448, -1.8180526494979858, -0.3982902467250824, 0.8820626139640808, -0.5284919738769531, -0.771747887134552, 0.5191934108734131, -0.271884948015213, 0.39722663164138794, 0.17654499411582947, -0.1156025230884552, 1.1274462938308716, 0.002924802480265498, -0.06377144157886505, 0.6381075382232666, -0.8931490778923035, 0.5851443409919739, -0.609673798084259, -0.7396414875984192, -1.3514204025268555, 0.5131138563156128, -0.6527143120765686, 0.966693103313446, -0.2606486678123474, 0.34231066703796387, -1.8196492195129395, 0.30202487111091614, -1.7631850242614746, 0.5434522032737732, -0.8235442638397217, 1.244988203048706, -0.21473461389541626, 0.4190693199634552, 2.2835052013397217, -1.3838214874267578, -0.1835206151008606, -0.983660101890564, 0.9609683156013489, -0.6903768181800842, -0.3355218470096588, 0.6652508974075317, -0.1655832976102829, -0.7851109504699707, -1.326136589050293, -0.0670115128159523, 0.3809269368648529, 0.30636486411094666, -0.34767574071884155, 1.280518651008606, -0.21104493737220764, -2.4219141006469727, 1.433599829673767, -0.23393657803535461, -1.0677043199539185, 0.3399549424648285, 0.8629801869392395, 0.4047275185585022, 0.05907673388719559, 2.428832530975342, -0.5085029006004333, 1.6236003637313843, -1.23130202293396, -0.24963460862636566, -1.2632337808609009, -0.20768456161022186, 0.5445747375488281, 0.453883558511734, -0.8159344792366028, -0.4510394334793091, -0.008912947960197926, 0.8915981650352478, 1.2823275327682495, 0.8350726366043091, -0.23046787083148956, 0.13440483808517456, -0.09246641397476196, 1.2061318159103394, 0.5179900527000427, 0.45032551884651184, -0.6573429107666016, -0.8081185817718506, 1.652712106704712, -1.262985348701477, -1.4367400407791138, 0.16844043135643005, 1.5897271633148193, 1.5158588886260986, -0.007681731134653091, 0.4713500738143921, -1.282744288444519, 0.5397932529449463, -0.34416553378105164, 0.3101566433906555, -0.4324473738670349, 0.7932400107383728, 0.3230484127998352, -0.6468099355697632, 2.3222556114196777, 2.230116605758667, 0.29700860381126404, -0.12371906638145447, -0.5646997690200806, 0.8598904609680176, 0.868431568145752, -0.20737086236476898, -0.10779739171266556, 0.06934788823127747, -0.46251773834228516, 0.1894795447587967, 1.2609180212020874, -0.3707740902900696, -0.18310365080833435, -0.25136885046958923, -1.1854416131973267, 0.9955522418022156, -0.023568784818053246, -0.9086524248123169, -0.42453160881996155, 1.228451132774353, 0.22945542633533478, 0.2232065349817276, -0.6343193650245667, 0.49133893847465515, 0.9841569662094116, -1.6659836769104004, 0.7164921164512634, 1.838423728942871, -1.7696853876113892, 1.611203908920288, -1.2353392839431763, -0.4907793402671814, -0.4919902980327606, -1.148404598236084, -0.03996533900499344, -0.9065629839897156, -0.014876915141940117, -1.3467985391616821, 1.5248490571975708, -0.39326211810112, -0.17694619297981262, -1.1411960124969482, -0.5076355338096619, 0.7671293020248413, -0.6985551118850708, 0.11847446858882904, 0.6462210416793823, 1.672709584236145, -0.6053872108459473, 0.6421993374824524, -0.2382345348596573, 0.20843353867530823, 0.23944173753261566, -0.5034253001213074, 1.416641354560852, 0.2031642645597458, 1.7340223789215088, 0.24847416579723358, -0.30183252692222595, 1.1622742414474487, 0.8777286410331726, -0.2568017244338989, -0.2829362750053406, 0.522710382938385, 0.27639129757881165, -0.75881028175354, 1.3216845989227295, 2.8631107807159424, 1.3900482654571533, 0.241219624876976, 0.5383657813072205, -0.09835521876811981, -0.5015391111373901, -0.229423388838768, 0.8853813409805298, -1.514758586883545, -0.8393839001655579, 0.276713490486145, -0.42275261878967285, -0.5876739621162415, -0.6903136968612671, -0.24925190210342407, 0.08046550303697586, -0.7076709270477295, -1.8613958358764648, -0.4801463186740875, 0.5693007707595825, -0.22839561104774475, 0.5292395949363708, -2.087873935699463, -1.4336448907852173, -0.6390920281410217, 0.6577020287513733, -0.7067866921424866, -2.3124172687530518, -0.023855015635490417, -0.7524508833885193, -0.5788511037826538, -0.5225896835327148, 0.3310769200325012, 0.6910415291786194, 0.6302158236503601, -0.448915958404541, -0.824195921421051, 0.29669833183288574, -0.8665972948074341, 0.15823449194431305, 1.8824371099472046, 0.5863696336746216, -1.3501213788986206, -0.7807411551475525, 2.024724006652832, 0.6533763408660889, 0.15808416903018951, -0.2562021315097809, -1.0203055143356323, 0.4134024977684021, -0.5310468077659607, -0.030254017561674118, -1.0444194078445435, 1.6175198554992676, 0.8897354602813721, -0.2465866357088089, -0.042804356664419174, 0.6585993766784668, -0.008898078463971615, -1.1760607957839966, -0.17227968573570251, 1.271715760231018, -0.8547961711883545, -0.7033680081367493, -0.30987101793289185, 0.5773982405662537, 0.6427584886550903, -0.47401922941207886, -0.6943168640136719, 0.7534263730049133, 0.19871288537979126, 0.8115544319152832, 1.3886536359786987, -0.7714877128601074, 1.8457884788513184, -0.44188210368156433, 0.7268243432044983, -1.1414417028427124, 1.083560585975647, 0.07447785884141922, 2.00378680229187, -0.23276662826538086, 0.21558250486850739, 0.4749319553375244, 0.38241055607795715, -0.7028957009315491, 0.1826901137828827, 0.46163490414619446, 0.3959208130836487, -1.5784345865249634, 0.6016729474067688, -0.23828266561031342, -0.4233337342739105, -0.2969518005847931, -0.8657987117767334, -0.4422678053379059, -2.047088384628296, 1.0588881969451904, -0.8472130298614502, -1.019211769104004, -1.3602007627487183, -0.23451994359493256, 0.013734621927142143, -0.43254294991493225, -0.8711475729942322, 1.5223554372787476, -1.9098519086837769, 0.5735207200050354, 0.322995662689209, -0.6364787220954895, -0.08089598268270493, 1.170815110206604, -0.20302699506282806, -1.7648648023605347, -1.020935297012329, -3.1540541648864746, 0.9741203188896179, -0.4475584626197815, -0.3955541253089905, 2.1260902881622314, 0.6596947908401489, -0.6363154649734497, 1.85396146774292, 1.0502930879592896, 0.634168803691864, 1.5189380645751953, 0.8458857536315918, 0.7240087985992432, -0.1731702834367752, -0.5942229628562927, 1.6525940895080566, -0.7298223376274109, 0.7525390386581421, 0.9372560977935791, -1.167286992073059, -1.110593318939209, -0.12378136068582535, -1.818550705909729, 1.5168473720550537, 0.842993974685669, 1.2419325113296509, 0.5279088020324707, 0.5872975587844849, 0.21259063482284546, 0.10871244966983795, -0.005968460813164711, 0.23161821067333221, 1.1600332260131836, 0.5017881989479065, 0.8306217193603516, 0.13997429609298706, -1.0061922073364258, 0.33356592059135437, -0.09280896186828613, -1.4300837516784668, -0.11043120175600052, 0.3501006066799164, 1.0943931341171265, -1.2339659929275513, -1.6705114841461182, -0.3106932044029236, -0.23822954297065735, -0.7687777876853943, -0.46120893955230713, 0.4299837648868561, 0.2571967542171478, -2.5218687057495117, -1.753268837928772, 0.8655115962028503, 0.33676451444625854, 1.0003886222839355, 1.238226056098938, 0.9125460386276245, 0.15205146372318268, 1.0254522562026978, 0.25613901019096375, 0.4058467447757721, -0.8289394378662109, -0.2174106240272522, -0.5134778022766113, 1.5150690078735352, 0.5422889590263367, -0.7876943349838257, 0.4117695689201355, 0.5931316018104553, -0.5486061573028564, 0.11371231824159622, 0.2790086567401886, -0.0069029321894049644, 0.6696082949638367, -0.06804915517568588, -1.0944479703903198, -0.24979835748672485, 0.49565058946609497, 0.8843240141868591, -0.7166030406951904, 0.7634130120277405, 0.1805049479007721, -0.3828410804271698, 0.04993174225091934, 0.3860800862312317, 1.2914358377456665, -1.424889087677002, 0.8424753546714783, 0.05642632395029068, -0.2731720209121704, 0.7114309668540955, -0.4235333204269409, -0.7544879913330078, 0.9467280507087708, 0.5016811490058899, 0.5741146206855774, -1.8343265056610107, -0.9904181957244873, 1.4796584844589233, 1.205428123474121, -0.8924543261528015, 0.9261534214019775, 0.5724263191223145, 1.610745906829834, -0.41282525658607483, -0.18229030072689056, 0.3989194929599762, -1.625313639640808, 1.612216830253601, 1.5291392803192139, -1.0551425218582153, 1.0851922035217285, -1.0539568662643433, 0.17435361444950104, -1.2802727222442627, -0.521490752696991, 1.1152814626693726, -0.12608440220355988, 0.6407822966575623, 1.0388256311416626, -0.6163834929466248, 0.04853564128279686, -0.4136868715286255, 0.5878329277038574, -1.1896097660064697, 1.4182978868484497, -0.565498411655426, 0.44803887605667114, -1.2319411039352417, -0.13616549968719482, -0.47264549136161804, -0.12966378033161163, -0.09345844388008118, -0.10991980880498886, 1.130288004875183, 0.2273252010345459, 0.002407368738204241, -0.9045761823654175, 0.18871378898620605, 0.2966594398021698, 0.025606803596019745, 0.9066112637519836, 0.4256826341152191, 1.304330587387085, 1.4624090194702148, 1.179894208908081, -1.3715837001800537, -0.8991438150405884, 0.2872646749019623, -1.134179711341858, 0.64827960729599, -0.14721350371837616, 0.4787790775299072, 0.5949388146400452, 1.3706501722335815, 0.4911820590496063, -0.4344128966331482, 0.9660990238189697, 0.636043906211853, -0.3611375689506531, 0.10407011955976486, 0.3770574629306793, 0.4340772032737732, 0.6991193890571594, -0.6605750918388367, -0.9701743125915527, 0.7590489983558655, -0.6689321398735046, 0.07844013720750809, -2.156221628189087, -1.1681774854660034, 1.1725088357925415, -1.7007192373275757, -1.8482967615127563, 1.1198514699935913, 2.3882360458374023, -0.2737979292869568, -0.3265838027000427, -0.07809873670339584, -0.13325294852256775, 0.01788824424147606, -0.4809691309928894, 0.6032553911209106, 0.04937399923801422, -0.1750452220439911, -0.7769331336021423, 1.8218328952789307, -0.5938766598701477, -0.1803048700094223, -0.1301230490207672, 1.076937198638916, 0.0567326620221138, -1.173717975616455, 1.8463119268417358, 1.8012953996658325, 1.1936291456222534, -0.947531521320343, 0.3840937912464142, 0.20036189258098602, 1.0641121864318848, -0.7724988460540771, -0.7290351390838623, -0.5549488663673401, 1.275887370109558, -0.6348855495452881, -0.3811050355434418, -1.4842729568481445, 0.4433296322822571, 0.5143446922302246, -0.11899149417877197, 0.48150262236595154, -0.7282582521438599, 0.23088037967681885, -1.1867965459823608, 1.1726144552230835, -0.5558991432189941, 1.6444376707077026, 1.813446283340454, 0.019144337624311447, 0.13053958117961884, -0.5960967540740967, 1.1394680738449097, -1.2024849653244019, 0.5538825988769531, -0.7576653957366943, -0.9103195071220398, -0.35801175236701965, 0.3223222494125366, 0.17117084562778473, -0.0975191593170166, 0.2689945101737976, 0.7628869414329529, -0.17967315018177032, -0.45900076627731323, -0.5474802851676941, -0.9474450349807739, -1.0688612461090088, -0.5857110023498535, -0.19710120558738708, -0.46776828169822693, -0.061693061143159866, 0.9095639586448669, -0.6075285077095032, -0.3261461853981018, -1.388018012046814, 0.626539409160614, -0.7154273390769958, 0.0368180125951767, 0.061286356300115585, -0.23501251637935638, 0.057546790689229965, -0.5926794409751892, 0.09040789306163788, 0.425263375043869, 0.5783573985099792, -1.3407999277114868, -1.5670173168182373, 0.7307857871055603, 0.8211098909378052, 0.965429961681366, -1.707277774810791, 0.4642806947231293, 1.6589823961257935, 0.03860696777701378, -0.23843520879745483, 1.6515733003616333, -0.5627779364585876, -0.04347961023449898, -0.3362785577774048, -0.4005224406719208, 0.7017894983291626, -0.2022421509027481, -1.0964680910110474, 0.5408880710601807, -0.042306724935770035, -1.685219645500183, 1.6206058263778687, 0.7998324036598206, 0.9133027195930481, 1.5442267656326294, 0.6234477758407593, 0.5723457336425781, 1.187821626663208, -1.3055671453475952, -0.6111853718757629, 0.7139545679092407, 0.6892866492271423, -0.6590649485588074, -0.5101990103721619, -0.7647536993026733, 1.6383353471755981, -0.9119325876235962, -0.5486868023872375, 1.5303956270217896, -0.04819955304265022, -0.003660881891846657, 0.34717419743537903, 0.17599137127399445, 0.8185557126998901, -1.591871976852417, -0.9099759459495544, -1.2771662473678589, 0.7493373155593872, -0.23543129861354828, 1.156261920928955, -0.3371560275554657, 1.2879120111465454, 0.48910367488861084, -1.676729679107666, 2.1489903926849365, -0.3136374354362488, 0.1893448382616043, -0.6249315142631531, -1.3112399578094482, 0.7727407217025757]], "dtype": "float32"}, {"__nd__": [-0.5722377896308899, -0.17042067646980286, -0.8672198057174683, -1.1684606075286865, 0.9646062850952148, 0.573298990726471, 0.15396609902381897, 0.8992977738380432, -1.042039394378662, 0.5164653658866882, 0.171287402510643, 0.09697610139846802, 0.8644300699234009, -0.489889919757843, 1.5149787664413452, 0.35340675711631775, -2.222109794616699, -1.3763691186904907, 1.0678695440292358, -0.4955761432647705, 0.24040935933589935, 0.6791120767593384, -0.6961582899093628, 0.994544506072998, 0.6571705341339111, -1.3425291776657104, -1.1905816793441772, 0.3510945439338684, 0.7565508484840393, 2.2143309116363525, 1.1508443355560303, 0.2465992420911789, 0.29446274042129517, 0.4472803771495819, 1.013895869255066, -2.0563249588012695, -0.30949822068214417, 1.5753251314163208, 0.15977662801742554, -0.057410165667533875, -0.47867339849472046, -0.12278920412063599, 1.3605796098709106, 0.008566852658987045, -0.27720919251441956, -1.2707971334457397, 1.3781319856643677, -0.19743089377880096, -1.4700257778167725, -0.35181769728660583, 0.37508872151374817, -0.807625949382782, 0.3562791645526886, 0.1615193635225296, -2.892259359359741, 0.38324975967407227, 0.6695321798324585, 0.8127633333206177, 0.5913410782814026, -1.570310354232788, -0.015787538141012192, -1.033121943473816, 0.43420812487602234, 0.004470256622880697, -0.03305499255657196, 1.306054949760437, -0.34382164478302, -0.659095287322998, 0.21969793736934662, -1.1612604856491089, -1.435689091682434, -0.3831598460674286, -0.2059459388256073, 0.5813158750534058, 0.038410164415836334, 0.656586766242981, 0.6855311393737793, 0.40162956714630127, 1.849833369255066, -0.014914625324308872, 1.490241527557373, 1.4429103136062622, -0.0863998681306839, -0.2605367600917816, 0.336622953414917, 1.191375494003296, 0.011804617010056973, 0.9087028503417969, -0.06725723296403885, -0.46407565474510193, 3.124751567840576, -0.08905494958162308, 1.4029138088226318, 1.3117644786834717, 0.8547403812408447, -0.14494779706001282, -0.26427310705184937, -0.29003259539604187, 0.6378635168075562, 1.2953624725341797, 0.10577783733606339, -1.1717028617858887, -0.22338970005512238, 0.17390811443328857, 0.43848201632499695, 0.9190492630004883, -1.3913683891296387, 3.4314489364624023, 0.5078748464584351, -0.26654234528541565, -1.2013213634490967, 0.2977979779243469, 0.4162524342536926, 0.436559796333313, 0.6281450986862183, 1.6669518947601318, -1.0804798603057861, 0.8596826791763306, -1.249987244606018, -0.34743013978004456, 1.285058856010437, -0.3781665861606598, -0.7536143660545349, -0.4930872619152069, -0.5579047203063965, -1.0052435398101807, 0.04868249595165253, -0.1798771470785141, 0.8519222736358643, -0.538055419921875, 0.8401301503181458, -0.27960601449012756, -1.1462818384170532, -0.7790490984916687, -0.6360455751419067, -1.192267894744873, 0.46322086453437805, -0.870159387588501, 1.164827585220337, -1.2610220909118652, -1.2472373247146606, 1.9205126762390137, 0.532428503036499, 0.9274364709854126, 0.16526515781879425, -2.4595284461975098, 0.07773589342832565, 0.7685965299606323, 0.24993856251239777, -0.10003863275051117, 0.280133992433548, 0.6682236790657043, 0.4674864709377289, -0.9562853574752808, 1.6389561891555786, 2.4228153228759766, -1.6333295106887817, -0.37351059913635254, -0.9797397255897522, -0.463321715593338, -0.6216166019439697, -2.17195725440979, -0.9501702785491943, 0.06945439428091049, 0.39575791358947754, -1.2927944660186768, -0.6178364157676697, -0.9487476944923401, 0.6351253986358643, -0.8239951133728027, -2.2808258533477783, -1.2678983211517334, 0.5997166037559509, 0.9165006875991821, -0.8280436396598816, -0.8023244738578796, 0.6605389714241028, 1.5437546968460083, -0.3591077923774719, -0.9511765837669373, 0.5059990286827087, 0.18361888825893402, 0.9579702019691467, 0.8321908116340637, -0.22680452466011047, 0.5069083571434021, -1.592803955078125, -1.783150553703308, 0.47193822264671326, -0.5419175028800964, 1.7940399646759033, -0.13131354749202728, -1.7754520177841187, 0.3899097144603729, 0.5779486894607544, 2.24188232421875, 1.8318932056427002, -0.037031471729278564, 0.8508110642433167, -0.28808045387268066, 0.05336952209472656, -0.514178991317749, 0.8056657314300537, 0.4360714554786682, -2.2150754928588867, 3.297954559326172, -0.7390208840370178, -0.2806629240512848, 0.8360843062400818, 0.047498807311058044, -0.5365877151489258, 0.198395773768425, 1.1539891958236694, 0.7852771282196045, -0.8349644541740417, 0.14763955771923065, 0.4943388104438782, -0.23591271042823792, 1.7381600141525269, 1.0259755849838257, 0.052137427031993866, 0.409607857465744, -0.1623220145702362, -1.0046334266662598, -0.9734079837799072, 1.9051713943481445, -0.5625976324081421, -0.6817207932472229, -0.707777202129364, 1.4521490335464478, -0.7653237581253052, -0.041676707565784454, -2.4622504711151123, 0.6353932023048401, 0.49808168411254883, -0.3340544104576111, -1.5781221389770508, 0.06910145282745361, -0.5499892830848694, 0.6606658101081848, -1.3130549192428589, -1.0167995691299438, -0.10274549573659897, -1.609473705291748, -0.09665782749652863, 0.8152102828025818, -1.0060317516326904, -0.6791133284568787, 1.6070642471313477, 0.16950224339962006, -0.892702579498291, 0.9678371548652649, 0.2787744104862213, 0.3805885314941406, -0.7818112969398499, 0.7135570049285889, 0.43845146894454956, -1.0187559127807617, -0.7540004253387451, -0.22345398366451263, -0.49849027395248413, -0.12008576095104218, 0.5113300085067749, -0.13649976253509521, -0.0979093462228775, 0.34822651743888855, -0.04910029098391533, -0.1359523981809616, -0.6135936379432678, 0.5479174852371216, 1.682438611984253, 0.814519464969635, -2.24615478515625, -1.0212438106536865, 0.5722967982292175, -2.2572684288024902, -0.06208690628409386, 1.8588478565216064, 2.657059907913208, 0.7797561883926392, -1.1971083879470825, 0.8155195116996765, 2.1846530437469482, 1.4371625185012817, -0.7759956121444702, 1.1823230981826782, -2.340578556060791, 1.3300726413726807, -0.9796238541603088, 0.6506437659263611, -0.9203543663024902, -0.06284692883491516, 1.5734156370162964, 0.3183320164680481, 1.2747180461883545, -0.42113015055656433, 0.2621288001537323, -0.07596656680107117, 0.14897283911705017, -0.12745577096939087, 0.748166024684906, 1.194305658340454, -1.1276757717132568, -1.2417439222335815, -0.4227617383003235, -0.9329286813735962, 0.8709990382194519, -0.6235566735267639, 0.5657048225402832, 1.2691267728805542, 0.7478510141372681, -0.3263545036315918, 0.5695291757583618, -0.142717182636261, -1.2642018795013428, 0.5777758359909058, -0.3368613123893738, -1.5224573612213135, 1.6985780000686646, 0.45009347796440125, -0.7522777915000916, 0.01578926481306553, -0.4502415955066681, -2.0226669311523438, 0.5046596527099609, 0.5014861822128296, 0.13670875132083893, -0.3898608386516571, 1.755789875984192, 0.4191737174987793, 0.3268766403198242, -1.12477707862854, -1.7590932846069336, -0.945087194442749, 0.1709071546792984, -0.43508613109588623, 0.7293322086334229, 0.6594918370246887, 0.05353621765971184, 1.109625220298767, 0.2578478157520294, 1.4411416053771973, 0.15833449363708496, 0.9005935192108154, 0.41699934005737305, 1.3333860635757446, 0.6635259985923767, -0.6386439800262451, -0.5045449137687683, -1.0673497915267944, 0.9556404948234558, -0.07414174824953079, -0.4201691150665283, -0.2935318350791931, 0.4269571900367737, -1.8620768785476685, 1.1317777633666992, -1.178490161895752, 1.6350523233413696, 0.8699440360069275, 0.11641408503055573, -0.5204850435256958, -0.6713038682937622, 0.7405484318733215, 1.2260794639587402, -0.8603487014770508, 0.5628595352172852, -0.5146474242210388, -1.0421457290649414, 1.2921600341796875, -1.1408188343048096, 0.8975529074668884, 0.8195090293884277, 1.0606155395507812, 1.7451844215393066, -1.3540669679641724, -1.2952618598937988, -0.7962702512741089, 0.04079202935099602, 0.9061514735221863, -0.6898931264877319, -0.17286598682403564, 0.6123193502426147, 0.7579978704452515, 1.4160555601119995, -0.11440347135066986, -1.413757085800171, -0.5335823893547058, -0.6035767793655396, 0.35038167238235474, -1.8449457883834839, 0.672051191329956, -0.9189780950546265, -0.5955294966697693, 0.5752995014190674, 0.03473757207393646, -0.4092819094657898, 1.0239852666854858, 0.5662643313407898, -0.47921499609947205, 0.41682082414627075, 0.5799723267555237, -0.6823391914367676, 1.6249594688415527, -0.8617061972618103, 1.3784282207489014, -0.20861972868442535, 0.35097384452819824, -0.3006991744041443, -0.8522723317146301, 0.43607935309410095, -1.0868505239486694, 0.8625079393386841, 0.4853271245956421, 0.26511797308921814, 0.7504274249076843, 2.3109540939331055, 0.9060793519020081, -1.5508586168289185, -0.9902036190032959, 0.5304788947105408, -1.4863917827606201, -1.0618287324905396, 0.4609408974647522, 0.33631736040115356, 0.8095853924751282, 0.9668818712234497, 1.163429617881775, 2.4527196884155273, -0.018246591091156006, 0.427757203578949, 1.27739417552948, 0.8101273775100708, -1.4141522645950317, 1.6920748949050903, 1.6154170036315918, -0.9651138186454773, 1.143497347831726, -1.371131181716919, 1.2454681396484375, -0.8752917051315308, -0.09152881056070328, 0.4853608012199402, -1.3779017925262451, -0.6334577202796936, -1.0192952156066895, -1.4783339500427246, 0.9290039539337158, 2.014235496520996, -1.2000290155410767, 0.44553089141845703, -0.9137377738952637, 1.559751033782959, 1.4011204242706299, 0.7780581116676331, 1.1022244691848755, -0.18838892877101898, -0.6332046389579773, 2.191862106323242, -0.7537364959716797, -0.8755252361297607, 1.0079433917999268, -1.1259193420410156, 2.3250176906585693, 0.3800932466983795, -0.29187795519828796, -0.3306029736995697, -0.4022473096847534, -0.06784424185752869, -0.38974547386169434, -0.750770092010498, -0.5941449403762817, -1.114578366279602, 0.4358265995979309, 2.492285966873169, -1.0139998197555542, 0.28537991642951965, -1.2490804195404053, -0.7377040982246399, 0.029516596347093582, 0.9951100945472717, 2.284085988998413, -1.4506747722625732, -0.5598697662353516, 1.5157432556152344, 1.2101649045944214, 0.7510774731636047, 0.26457056403160095, -0.3258410394191742, 1.8828814029693604, -0.1588028073310852, -0.1819787621498108, 1.0676807165145874, -0.19023053348064423, -0.9358072876930237, -0.6289411187171936, -0.36101019382476807, 0.6274131536483765, 0.3384895324707031, -0.5821158289909363, 0.24297043681144714, -0.7815033793449402, -0.554189145565033, -0.40855178236961365, 0.34168094396591187, 2.3864705562591553, -0.5534822940826416, 0.22398409247398376, 0.5415970087051392, 1.6365786790847778, -0.6062881946563721, -0.9527314305305481, 1.0787553787231445, 0.8203896284103394, -0.3982122540473938, 1.1563769578933716, 1.7809488773345947, -1.7126398086547852, -1.1462070941925049, 0.15678641200065613, -2.1482012271881104, 1.0756072998046875, 0.3042222261428833, 0.796005368232727, -0.2632538676261902, 0.28329038619995117, -0.2581138610839844, -0.4873163104057312, 0.06891871243715286, 0.7413849830627441, -0.5304303169250488, 0.3024621903896332, 0.2628783881664276, 0.6553066968917847, 0.13174572587013245, -1.297375202178955, 0.8359912633895874, -0.7955182790756226, 0.08729932457208633, -0.5197157859802246, 0.15798994898796082, -0.05239023640751839, 0.1212320327758789, 0.4291418194770813, 1.2648606300354004, -1.071079969406128, -1.2014826536178589, 0.8110689520835876, -1.053039312362671, 0.3893220126628876, 1.0794672966003418, -0.6363314986228943, 0.3761349320411682, -0.46433600783348083, 0.5685380697250366, 0.1892932802438736, -1.0602056980133057, 1.4500257968902588, -1.3389371633529663, 1.0838627815246582, 0.10240539908409119, -0.07512455433607101, -0.06117216497659683, -0.09161414206027985, 0.4822462797164917, 1.880704402923584, 0.9788774847984314, -1.9520487785339355, -0.665472149848938, -0.34983304142951965, -0.20979464054107666, 2.77087664604187, 0.2152625322341919, -0.5453711152076721, 0.827826976776123, -0.8845109343528748, -0.9998400211334229, 0.3187579810619354, -0.49302220344543457, 0.5191086530685425, 0.09469981491565704, -0.23154357075691223, -0.16972336173057556, 0.8633673191070557, 1.6653680801391602, 1.0499612092971802, -0.8036734461784363, -0.13465547561645508, 0.2482997328042984, 0.12419808655977249, -0.7723351716995239, 1.6440171003341675, 2.093766689300537, -0.4275563657283783, 0.401091068983078, 0.9308553338050842, -0.716785192489624, -0.5329918265342712, 0.9256814122200012, 0.2825561761856079, 0.06178842857480049, -0.5411686897277832, -1.728630781173706, -0.79563969373703, 0.9432751536369324, 0.2159053534269333, 0.31918907165527344, -0.29340505599975586, -1.4874063730239868, 0.8767593502998352, -0.3633166551589966, -0.552143394947052, 0.3728417158126831, 1.8684924840927124, -0.35733845829963684, 0.8329920172691345, -0.36829379200935364, -1.3527789115905762, -0.77869713306427, -0.10551908612251282, 0.04257142171263695, 1.0801995992660522, 0.5573449730873108, 0.49428102374076843, 0.6285951137542725, -1.3890681266784668, 0.4894316792488098, 0.5620132088661194, -0.7076791524887085, 0.05843760073184967, -0.659706711769104, -0.14800402522087097, 3.3382763862609863, -0.6383892893791199, 0.12063375115394592, -0.5224952101707458, 0.2741214334964752, 0.8072028160095215, -0.16184477508068085, -2.480842351913452, 1.400732398033142, -1.3048943281173706, -2.1116671562194824, 2.125692844390869, 0.1010032445192337, -0.793178141117096, 0.24119716882705688, -0.09914316236972809, -0.21827514469623566, 0.6621220111846924, 0.7537348866462708, 1.0324974060058594, 1.9964160919189453, 0.6666774749755859, -1.7949775457382202, 1.7699302434921265, -0.6637043356895447, 1.622925043106079, 0.5904019474983215, 0.09858115762472153, 1.501724123954773, 1.8461393117904663, 0.7285727262496948, 0.3516863286495209, -3.56527042388916, 1.120926022529602, 1.0500632524490356, -0.4548857510089874, 1.5419560670852661, 0.7711868286132812, -0.005420129746198654, -0.5820633769035339, 1.5740748643875122, 0.4159228503704071, -0.6774352788925171, -0.3016248643398285, 1.1334283351898193, -0.05070099979639053, -0.10712678730487823, -0.03503831848502159, -1.2122573852539062, 0.8750631213188171, -0.12096907198429108, -0.12004662305116653, -3.2348504066467285, -0.1331213265657425, -2.188692569732666, -0.3000847101211548, 0.8186631202697754, 1.5812554359436035, -0.03626635670661926, 0.335427463054657, 0.10397650301456451, -1.535082459449768, -0.35959145426750183, 0.4315136969089508, 0.2496640384197235, -1.2461928129196167, -0.7194323539733887, -0.944229781627655, 0.2565079927444458, -0.03739846497774124, -0.45890745520591736, 0.27177894115448, 1.7158851623535156, 0.25542205572128296, -0.13591571152210236, -0.8304136991500854, 0.7369197010993958, 0.5613272190093994, 0.8786686062812805, -0.42990657687187195, 0.6232561469078064, -2.3408737182617188, -1.0112075805664062, 0.841297447681427, 1.1048104763031006, -2.312379837036133, -0.4149172008037567, 1.324651837348938, -0.30901098251342773, 0.5994669795036316, 0.4656464755535126, -0.5954878926277161, -0.3360832929611206, -1.4934133291244507, 0.6313855648040771, -1.8536018133163452, -0.5806686282157898, 0.9461912512779236, 0.7436854243278503, 0.5531955361366272, 0.8662802577018738, -0.07081270962953568, -0.6924906373023987, 0.770234227180481, 1.927628517150879, 0.18491189181804657, -0.9915845394134521, 0.9139683246612549, -1.3259689807891846, 0.8477519154548645, -0.6656473278999329, 0.6250678300857544, 0.6810464859008789, -0.9769943952560425, -0.6116594076156616, -0.3943263590335846, 1.603503704071045, 0.14244040846824646, 1.6254216432571411, -1.3910212516784668, -0.5133427381515503, 1.4483939409255981, -0.863808274269104, -0.42786335945129395, 0.18402090668678284, 0.2037954479455948, 1.367419958114624, 0.6512753367424011, 1.4706002473831177, -0.12102778255939484, -0.2734978497028351, 0.7539410591125488, -1.2296875715255737, 0.3573129177093506, -0.15448087453842163, -0.08541349321603775, 0.42930227518081665, -1.3362253904342651, -0.7642577886581421, 0.48961004614830017, 0.5714606046676636, 0.7472819685935974, -1.0234460830688477, 1.1146875619888306, -0.8666326403617859, -0.3304761052131653, -1.0707768201828003, 1.0374009609222412, -0.48679280281066895, 0.6018571853637695, 0.7154349088668823, -1.5680241584777832, 0.3378697633743286, -1.098132610321045, -0.9770482182502747, -0.7079113125801086, -1.2124500274658203, 0.9645592570304871, -0.8220296502113342, -1.094373345375061, -0.3248055577278137, -1.0222814083099365, -0.6366453766822815, 0.7349664568901062, -0.6899040937423706, -0.8656908273696899, -0.3384169936180115, -0.5679410099983215, 0.3949678838253021, 0.3018275201320648, -0.6975030303001404, -0.08089731633663177, 0.7276769876480103, 0.12172749638557434, -0.326460599899292, -0.8569895029067993, -2.163071870803833, -0.9407405257225037, 0.5191400051116943, 0.7309988737106323, -0.7899107933044434, -0.12476254254579544, -1.3447743654251099, -0.5200275182723999, 0.23928752541542053, -2.09462571144104, -0.42418181896209717, 0.414518803358078, 0.6091452836990356, 2.247096538543701, 0.8118618130683899, 1.5533210039138794, 1.0527125597000122, 0.8242627382278442, -1.5871516466140747, 1.6968191862106323, -1.7667934894561768, -1.0701483488082886, -0.06551650911569595, -1.6595220565795898, 0.25573185086250305, -0.4188530445098877, -0.3269914388656616, -0.40503188967704773, 0.37871408462524414, 1.0369902849197388, 0.45756852626800537, 0.5916641354560852, 1.9616402387619019, 1.5803327560424805, -0.022587958723306656, -0.4852176606655121, 0.5596452355384827, 0.31214526295661926, -0.8355971574783325, 0.6656754612922668, 0.7689986228942871, -0.29238712787628174, 1.119040846824646, -0.9262492656707764, -0.03963835909962654, -1.5279512405395508, -0.33531463146209717, 0.43400880694389343, 1.2878912687301636, 0.847787618637085, -0.06586529314517975, -1.093300461769104, 1.3533512353897095, 1.2821698188781738, -1.592918872833252, 0.9552730321884155, 0.2225019782781601, 0.6339807510375977, 0.533138632774353, 0.6680068373680115, -0.9060253500938416, 0.12897998094558716, 0.5390203595161438, -0.14352229237556458, -0.08333937078714371, -1.2039968967437744, -0.7032639980316162, -1.21235191822052, 0.449633926153183, -0.752289354801178, -1.191519856452942, -1.0262819528579712, -0.5989760756492615, 0.17727059125900269, 1.6253105401992798, 0.5576093792915344, 0.31901848316192627, -0.025543296709656715, 0.05390765517950058, 2.3671131134033203, 0.35085511207580566, 0.2713530659675598, 0.652686595916748, 1.084686040878296, -0.1470494270324707, -0.5478454232215881, 0.22059953212738037, -0.22728127241134644, 0.8713359236717224, -0.5974459052085876, 0.12231646478176117, 0.6504456996917725, -1.1817487478256226, 1.4297846555709839, 0.31570717692375183, -0.6422982811927795, 0.7961577773094177, -0.24277369678020477, 0.2546459138393402, -0.6168500185012817, 0.5453940033912659, 1.3775992393493652, 0.11637727916240692, 0.5866098403930664, -0.11766397207975388, -1.0396636724472046, 2.352276086807251, -0.4724279046058655, 0.560606837272644, 0.5690557956695557, 0.3892005383968353, -0.941572368144989, -0.16649626195430756, -0.26593032479286194, -0.027437569573521614, -0.5810875296592712, -0.5826082825660706, -2.1697440147399902, -1.299753189086914, -0.8609469532966614, 0.27937567234039307, 1.5479792356491089, -2.0974605083465576, -0.01975543238222599, 0.28864622116088867, -1.0779120922088623, -0.4965403974056244, -0.7664154171943665, 0.5239346623420715, -0.15143601596355438, -0.14191988110542297, -0.5223646759986877, 0.10131652653217316, 1.6397775411605835, -0.1957472562789917, 2.823836088180542, 1.3188064098358154, 2.9842283725738525, -0.9839287996292114, -0.32102155685424805, -0.15361681580543518, 0.7289466261863708, -1.5639792680740356, 1.294697880744934, 0.39368245005607605, 0.24265243113040924, -1.4096148014068604, -0.7941402792930603, 1.0004440546035767, -0.33633899688720703, -0.16977465152740479, 0.17751385271549225, -0.7045373916625977, -1.620387315750122, 0.29283761978149414, 0.569053590297699, -0.7980943918228149, -0.5164411067962646, 2.434842348098755, -1.8999557495117188, -0.8347856998443604, -0.8811313509941101, -0.9507675766944885, 0.5424479842185974, 0.11144419014453888, 1.2471041679382324, 1.430457592010498, -0.09070094674825668, 0.00982669461518526, -0.025673799216747284, 1.0779824256896973, 0.2902504801750183, 0.6901256442070007, -0.325183629989624, 0.8562983870506287, 1.2146522998809814, -0.7139858603477478, 1.3410474061965942, -0.35493049025535583, -0.8130170106887817, 1.098198413848877, 0.6474953889846802, -1.8037036657333374, 0.44660982489585876, 0.5677729845046997, 1.5152148008346558, 1.296769142150879, 0.6320816278457642, -0.3714546859264374, -1.3281185626983643, -0.8225196003913879, -0.42559781670570374, -0.29071399569511414, -0.5177933573722839, -1.3616210222244263, 0.10504132509231567, 0.3990229070186615, 0.45996588468551636, 0.3480730354785919, -0.6818933486938477, -0.2974948585033417, -0.7208477258682251, -1.4241284132003784, -0.5882744789123535, 0.11821392923593521, -0.3220285177230835, 0.36469539999961853, 0.6602913737297058, -1.4829927682876587, 1.4993728399276733, -0.12364724278450012, 0.26967984437942505, 2.1642799377441406, -0.44891801476478577, 0.8893184065818787, 0.9670587778091431, -0.08288343250751495, 0.05808182433247566, -0.575772762298584, 0.9464213848114014, -0.2573966383934021, -1.2542362213134766, -0.12154367566108704, -0.17953427135944366, -3.0694310665130615, 0.9148382544517517, 0.2906286418437958, -1.6138782501220703, 1.5283359289169312, -0.4755117893218994, -0.48525649309158325, 0.696312427520752, 2.278028726577759, 1.3301278352737427, 0.8965405225753784, 1.3245984315872192, 0.24213649332523346, -0.338955283164978, 0.6786993145942688, -1.3416661024093628, 0.7500404715538025, -0.3763177692890167, 0.543254017829895, -0.7286465764045715, -0.34706905484199524, 0.42563286423683167, -1.0344886779785156, -0.30750370025634766, -0.9700995087623596, -3.0061421394348145, 0.6014788746833801, -1.6946412324905396, 0.5325405597686768, 2.2188730239868164, -0.41109806299209595, -0.42456871271133423, -1.2302578687667847, 2.4238507747650146, 1.9748966693878174, 0.7375171184539795, 1.1360735893249512, -0.7332554459571838, 0.6331756711006165, -0.2707478404045105, -1.7888320684432983, 0.6213217377662659, -0.3550890386104584, -0.3053900897502899, -0.001157268532551825, 1.456707239151001, 0.18592295050621033, -2.0487818717956543, -1.6323977708816528, 0.4733353853225708, -0.7516319155693054, 0.6271104216575623, 0.40949246287345886, -1.364140272140503, 1.0411434173583984, -0.40783822536468506, -1.4462107419967651, 0.4259496033191681, 0.07250337302684784, -0.589914858341217, -0.5476022362709045, -1.700355887413025, -1.1075267791748047, -0.3519800305366516, -0.8322141766548157, 0.26360809803009033, -0.29199686646461487, 1.569382667541504, 1.9311126470565796, -0.19322289526462555, 1.4182473421096802, -0.11726648360490799, -0.43655794858932495, 0.981487512588501, -0.9320257306098938, -1.4759337902069092, 2.7268118858337402, -0.8173710107803345, 0.7180641889572144, -0.45230308175086975, -0.5500902533531189, -0.764846682548523, -0.15129739046096802, -0.34058043360710144, -0.16342008113861084, -0.5707690119743347, -0.48593655228614807, 2.003816604614258, -0.44421476125717163, -0.11588756740093231, -1.1865215301513672, -2.048109292984009, -3.2082650661468506, 1.6236729621887207, -0.8137003779411316, -0.934855580329895, 1.7084625959396362, 1.198926329612732, -0.9547901749610901, 1.3513426780700684, 0.9294124841690063, -0.41795116662979126, -0.3007495701313019, -0.7852007150650024, 1.329209804534912, 0.21724723279476166, -1.4746469259262085, 0.46545353531837463, -1.110159993171692, -1.072983741760254, 1.8583561182022095, -0.9314252138137817, 0.16885656118392944, -0.474692702293396, 0.8259192109107971, -1.0409592390060425, 0.8317088484764099, 0.9204391837120056, 0.12251153588294983, 1.990835428237915, 0.40713346004486084, -1.6509062051773071, -1.0318474769592285, -0.20945024490356445, -0.5603871941566467, -1.1827290058135986, 1.0102932453155518, 0.4667009711265564, -0.7468623518943787, -0.15846221148967743, 1.1816132068634033, -0.765217125415802, -0.0753057599067688, 0.17606675624847412, 1.1967501640319824, -0.8515061736106873, -1.2913309335708618, 0.41970568895339966, -1.2583049535751343, 0.4105188548564911, 0.6206650137901306, -1.2017773389816284, 1.286534309387207, -0.8951027393341064, 0.37184959650039673, -0.732450008392334, 0.5107781887054443, -1.3379855155944824, 1.1570979356765747, 0.4782065153121948, 0.518828272819519, -1.250354528427124, 1.6192781925201416, -1.4664959907531738, -0.2746361494064331, 0.7886611223220825, -0.19759559631347656, -0.25833800435066223, 0.9193965196609497, 1.7695679664611816, -0.13442213833332062, -0.30504998564720154, -0.0997106283903122, 0.9575223922729492, 1.7389318943023682, -2.1208107471466064, 0.5064651370048523, 0.5158804059028625, 0.508412778377533, -0.08938255906105042, -0.8385754823684692, -0.2604810297489166, -0.2509034276008606, -1.4646284580230713, 0.9113888144493103, -0.8780169486999512, 0.758633017539978, -0.25406163930892944, 0.23700359463691711, -0.9583438038825989, 0.3523874580860138, -0.040718499571084976, -0.07625939697027206, 0.5534511804580688, -1.1086773872375488, 1.764901876449585, -1.4296478033065796, -0.3672025203704834, 0.9414357542991638, 0.8760160207748413, -1.2129307985305786, 0.8696956634521484, 1.0479239225387573, 0.5811567902565002, 0.28722989559173584, 1.3393453359603882, -0.2492058426141739, -0.9376797080039978, 0.06040160357952118, 1.4221227169036865, 0.06789321452379227, -0.312404066324234, 0.31502771377563477, 0.734633207321167, 0.3942056894302368, 0.7982240319252014, 0.7995837330818176, 0.17090252041816711, 0.37600451707839966, -0.6763102412223816, 2.4730923175811768, 0.12311004102230072, -0.9008038640022278, 0.10008177906274796, -1.1690282821655273, -1.1809308528900146, -2.3003828525543213, -0.9572360515594482, 1.0425387620925903, -0.6198620796203613, -0.19282390177249908, -0.4252239465713501, -0.003915505018085241, 0.5366833209991455, 2.1343913078308105, 0.12785375118255615, -0.06893305480480194, -0.36287492513656616, -0.34332340955734253, 0.10868864506483078, 1.3356008529663086, -0.24679197371006012, 0.6136767268180847, -1.1933223009109497, -0.3471752405166626, 0.03691926598548889, 0.5951422452926636, -0.18699046969413757, -0.649131715297699, -0.901091456413269, 1.2290953397750854, 0.12262532860040665, -0.3249686062335968, 0.09885112196207047, -0.8359420299530029, -1.1135928630828857, 0.6581467986106873, 0.36601191759109497, 0.46380624175071716, 0.2950231730937958, -2.4493649005889893, 1.268082857131958, -1.7305413484573364, -1.0147091150283813, 0.13891129195690155, -0.6102059483528137, 0.36770057678222656, -0.2521009147167206, -2.247408151626587, -0.15900544822216034, 0.021496320143342018, 0.36979183554649353, 0.5365695953369141, -0.8716577887535095, 0.19287462532520294, 2.6505069732666016, 0.349483460187912, -0.034328896552324295, -1.294304370880127, -0.2907868027687073, 0.600010871887207, -0.7742525935173035, -0.669983983039856, 0.019311202690005302, -0.17141026258468628, 0.5691441893577576, -0.5512348413467407, -1.597367763519287, -0.4490646719932556, -0.0025547780096530914, 0.5539937019348145, -1.1413843631744385, -0.3616158962249756, 0.04128054156899452, 0.2788967490196228, 1.5328198671340942, -0.16278724372386932, 1.011966347694397, -1.7307636737823486, -0.20993199944496155, 1.2813602685928345, 1.0983383655548096, 1.8740873336791992, 0.3593466877937317, 0.33141544461250305, -0.18045450747013092, 0.3172646462917328, 0.7365871071815491, 0.9023221731185913, 0.47679129242897034, -0.15030992031097412, 1.3341064453125, 0.20430335402488708, -0.751833438873291, -0.09569596499204636, 1.7191845178604126, -0.3298834264278412, -1.8258678913116455, 0.005389842204749584, 0.880620539188385, -0.24152040481567383, -0.2828313708305359, -1.3956056833267212, -1.1845170259475708, -0.9116234183311462, -0.2931332290172577, -1.548867106437683, -0.6489040851593018, 1.5870513916015625, 2.6640336513519287, 1.659387230873108, -0.15450964868068695, 0.7322357296943665, -1.037781000137329, -0.439350426197052, -1.458783745765686, -0.5915058851242065, -1.0158764123916626, -0.5739539861679077, 0.5280655026435852, 1.3595826625823975, 0.5901786684989929, 0.7038242220878601, 0.8021565675735474, 1.1419490575790405, 0.7242786288261414, -2.3619537353515625, 0.3858163356781006, 0.5526060461997986, -0.1793547421693802, -0.9699618220329285, 0.27334555983543396, -1.6922869682312012, -1.4275821447372437, 1.443220615386963, -2.1411170959472656, 0.9756190180778503, -0.09531665593385696, 0.16657516360282898, 0.184517040848732, 0.020241988822817802, 2.012650489807129, 0.019787145778536797, -1.8745869398117065, -1.4175093173980713, 0.45442473888397217, 0.36927101016044617, 0.7797926664352417, -0.9498797059059143, -0.010918366722762585, 0.7380689978599548, 0.2875349521636963, 0.4538697600364685, 0.5914328694343567, -0.5476407408714294, 0.9924778342247009, -0.4497010111808777, -0.29378587007522583, 1.1901288032531738, -0.4360062777996063, 0.143659770488739, -0.7119063138961792, -0.11578179895877838, -0.05278156325221062, -0.4382867217063904, 0.45510563254356384, 0.03815390169620514, -0.6420042514801025, 1.4913357496261597, 0.6354647874832153, -1.125124216079712, -0.42094266414642334, -0.4857846200466156, -0.11297160387039185, 0.23447683453559875, 0.2903086245059967, 0.5086237192153931, 1.3739285469055176, 0.3359428942203522, 0.8982791900634766, 0.2177526205778122, -1.8585729598999023, 1.3968172073364258, -1.0750676393508911, 0.47763746976852417, -0.36358967423439026, -0.8186532258987427, -0.4958783686161041, 0.8897425532341003, -1.3383358716964722, 1.4953863620758057, -1.185648798942566, 0.2875511646270752, 0.5594233274459839, -0.6075726747512817, -1.5254623889923096, -1.234446406364441, 1.0435645580291748, -1.1550275087356567, 1.1246620416641235, -1.2952651977539062, 0.5922574996948242, 1.3988430500030518, -1.3171908855438232, -0.5708678960800171, 0.05919978767633438, 0.22355306148529053, 0.04289727658033371, -0.09442342072725296, 0.07506198436021805, 0.6231008768081665, -0.39258119463920593, 0.9875308871269226, 1.0554234981536865, 1.7205067873001099, 1.4146277904510498, -2.5616140365600586, 1.0682525634765625, -0.05830622464418411, 2.1779184341430664, 0.36779263615608215, -0.840178370475769, -0.11135608702898026, 1.1630228757858276, -0.609959065914154, 1.1407222747802734, 1.3476179838180542, 0.9247373342514038, -0.3494744896888733, -0.05960696190595627, -1.4972096681594849, 0.9792746901512146, -0.3939184248447418, 1.737011194229126, 1.724943995475769, -1.925923228263855, 0.9887033104896545, -1.0955313444137573, 0.8115910291671753, 0.1754731386899948, 0.6839026808738708, 0.40254485607147217, -0.4504019021987915, -2.005506992340088, 0.22457747161388397, -0.5056154727935791, 0.34698134660720825, -0.35198819637298584, -0.37617841362953186, 0.08035565167665482, 0.29175204038619995, -1.2577766180038452, -0.15349164605140686, -0.3342640697956085, -2.016502618789673, 0.7144536375999451, -1.975008487701416, -0.33558619022369385, 0.6387120485305786, -1.611583948135376, 2.34836483001709, -0.7122359871864319, 1.1665229797363281, 1.5345059633255005, 0.4133692681789398, -0.08179810643196106, 0.08286133408546448, 0.12817002832889557, 0.11476471275091171, 1.1001898050308228, -1.5560275316238403, -0.07372677326202393, 1.1688847541809082, 0.7207039594650269, 1.0821928977966309, -0.9526445865631104, -1.6176085472106934, -1.1953394412994385, 2.078325033187866, 1.309517502784729, 1.2204066514968872, 0.9337165355682373, 1.433540940284729, -0.12053335458040237, -0.0030901941936463118, -0.38425755500793457, 0.6528867483139038, 0.3628106713294983, 0.514468789100647, 0.9362248778343201, -0.30387961864471436, -0.2746100425720215, -0.8446120619773865, 0.6194115877151489, 0.47286978363990784, 1.4907374382019043, -1.0068907737731934, 0.3840232789516449, 0.2546873092651367, 1.6461572647094727, 0.06315254420042038, 0.22425001859664917, -0.849317193031311, 0.9290397763252258, 0.43337738513946533, -0.46596604585647583, 0.20626075565814972, -0.666996955871582, -1.0808050632476807, 1.827998399734497, -0.5822516083717346, -0.08181317895650864, 0.1383214294910431, 2.2001218795776367, 0.1923077255487442, 0.10066385567188263, -1.4296997785568237, 1.5764092206954956, 0.48622408509254456, -2.072101354598999, 0.44883355498313904, 1.659156084060669, 0.03523058444261551, 1.95376455783844, -0.5633043646812439, -2.431873321533203, -0.318751722574234, 0.02704816870391369, -0.8112557530403137, -1.135985255241394, -1.3062877655029297, 0.5803109407424927, -1.1776154041290283, 0.1643662452697754, -1.6027625799179077, 0.10135586559772491, 0.419291615486145, -0.45397132635116577, -1.1257365942001343, 1.883004903793335, -1.430988073348999, -1.349502444267273, 0.6390460729598999, -1.3312695026397705, 0.37132546305656433, 0.24360692501068115, -0.7610772848129272, -1.3679563999176025, 0.12912608683109283, 0.16566012799739838, -0.46456098556518555, 1.3740485906600952, -1.2189383506774902, 1.0182448625564575, -0.43498608469963074, 0.39602720737457275, -0.8054024577140808, -1.5977097749710083, 0.33457639813423157, -0.6878852248191833, -1.7462741136550903, -0.13758255541324615, 0.7052507996559143, 0.5201415419578552, 0.770240843296051, -2.6809744834899902, -1.5244057178497314, 0.6980931162834167, -1.230560541152954, -0.25467249751091003, -0.6877606511116028, 0.26164939999580383, 1.477399230003357, -0.15496312081813812, -0.7442339658737183, 0.1678762286901474, -0.42498767375946045, -0.506058394908905, 1.089797854423523, 0.7896958589553833, -1.3819876909255981, -1.326070785522461, 0.08101371675729752, 0.2850296199321747, 1.1109960079193115, -0.23355449736118317, -0.812046468257904, -0.05637850612401962, 0.6420579552650452, -0.4016065001487732, 1.3583539724349976, 1.2309885025024414, -1.5250104665756226, -0.4379086494445801, -0.9603299498558044, -0.8211553692817688, 1.466127634048462, 0.827585756778717, 0.6229906678199768, 0.6849326491355896, 1.876219391822815, 1.781394600868225, -1.0866276025772095, 0.3141942024230957, -0.6570805907249451, -0.07901564240455627, 1.4078668355941772, 0.337064653635025, 0.5033040642738342, -0.7438399791717529, -0.8529603481292725, 1.6645631790161133, -2.1049981117248535, -0.9712194800376892, -1.390936017036438, 2.3368799686431885, 0.442402720451355, -1.5958585739135742, -0.762454628944397, -0.3172153830528259, -1.1804591417312622, 0.021618351340293884, -1.1615811586380005, -1.0826854705810547, 0.08971943706274033, 0.786421000957489, -0.32426148653030396, -1.6960153579711914, -0.09057900309562683, -0.5267337560653687, -0.08747344464063644, 1.0433428287506104, -1.42533540725708, -0.9883049130439758, 0.7960520386695862, 0.5531272888183594, -0.645902156829834, -1.6485916376113892, 2.325704574584961, -0.674763560295105, 0.8130041360855103, 1.908474087715149, 0.3679196536540985, -0.6313862204551697, -2.1137664318084717, 0.044014956802129745, 0.2807313799858093, 1.8530287742614746, 0.769556999206543, 0.2311587929725647, 1.6187973022460938, 1.7352488040924072, -2.0599589347839355, -0.6142733097076416, -1.5131645202636719, -0.4542490839958191, 0.7404825091362, -0.6592825651168823, -0.4861186742782593, 0.769315779209137, -0.4694879353046417, 0.043172240257263184, 0.9280909299850464, -0.3518511652946472, 0.8041343688964844, -0.45675989985466003, 0.3123573362827301, -0.7528611421585083, -0.9648626446723938, -0.6202649474143982, -0.9808571934700012, 0.06171834468841553, -0.8617843985557556, 0.6941277980804443, -0.5708616971969604, -0.14191128313541412, -0.24112153053283691, -0.6818436980247498, -2.261773109436035, -1.353088617324829, 1.5148588418960571, -0.46998104453086853, -0.38710707426071167, -1.212679147720337, -0.18748518824577332, 1.974863052368164, 0.2570062279701233, -0.942664623260498, 0.7446783781051636, -0.4324606657028198, 0.2864808440208435, 1.1735929250717163, 0.04735775291919708, -0.9431897401809692, -1.619326114654541, 0.3754751682281494, 0.9586033225059509, 1.8497909307479858, -0.2771851420402527, -0.5776085257530212, -0.25526905059814453, -1.6865391731262207, -2.5412633419036865, 0.27708208560943604, 0.14526546001434326, 0.45038920640945435, -1.4660545587539673, 2.1176366806030273, 0.5457906723022461, -0.4694875478744507, -0.9348583817481995, 0.04775983467698097, -0.593159019947052, -0.08982374519109726, 0.9248517751693726, -1.433144211769104, -0.2161288559436798, 1.1593660116195679, -0.6518628001213074, -0.9485368728637695, 1.2686749696731567, -0.6602894067764282, -0.4705020487308502, -0.8619367480278015, -0.3578798472881317, 0.6836729049682617, -0.9340742826461792, 0.11868532001972198, 0.8079667091369629, -1.35919988155365, 0.1631811261177063, -0.1626012772321701, -0.45045459270477295, -1.2457051277160645, -1.8034411668777466, 0.2892131507396698, -0.495795875787735, 0.04350971058011055, -1.2479850053787231, -0.07137416303157806, -0.5838286876678467, 2.107227325439453, 0.1525043547153473, 0.8266268968582153, -0.5548952221870422, 0.5292529463768005, 1.2955653667449951, -2.238985776901245, -0.9529584646224976, -0.8980182409286499, 1.0551649332046509, 0.8599798083305359, 1.466675877571106, 0.45027923583984375, -1.940895438194275, 0.18234173953533173, 1.721580982208252, -0.05048312246799469, 0.563556432723999, 0.3780517876148224, -1.2512147426605225, 1.7193505764007568, -0.7515129446983337, -0.6228132843971252, 0.03738086298108101, -0.9214171171188354, 1.1095584630966187, 1.4028501510620117, -0.8695800304412842, 2.3974313735961914, 0.22891445457935333, -0.7561083436012268, -0.7578994035720825, 1.358418583869934, -0.45657601952552795, 1.725103735923767, -1.2278809547424316, 0.803021252155304, -0.0023422425147145987, 0.6215001344680786, 1.7163184881210327, 0.5746933221817017, 1.092207670211792, -0.11430339515209198, 0.20782053470611572, -0.15349286794662476, -0.5604497194290161, -1.3610703945159912, -0.45519527792930603, -0.7261714935302734, 0.9364054799079895, 1.1948331594467163, -0.5251549482345581, 1.6657607555389404, 1.122107744216919, 0.5567797422409058, -0.6569682359695435, -0.829363226890564, -0.06454385071992874, 0.8841198086738586, -0.3965301811695099, -0.10739405453205109, -0.24491150677204132, -0.34390586614608765, -0.9464359879493713, 1.3076223134994507, 0.9985294342041016, -0.38618746399879456, -1.4739406108856201, 0.5385057926177979, 0.47853603959083557, -0.39774855971336365, -1.0694167613983154, -0.6113346815109253, -0.6248766183853149, -1.1156383752822876, 0.36359769105911255, 1.6553099155426025, 0.8451124429702759, -0.4699016809463501, -0.16976846754550934, 0.9154714941978455, 0.9868638515472412, -0.07563226670026779, 0.9086140990257263, -0.4908430278301239, 2.336620330810547, 0.9313428997993469, 1.3318333625793457, 1.263831377029419, 0.7401323318481445, -1.1704957485198975, -0.7337178587913513, 0.95122891664505, 1.2242475748062134, 0.8687021136283875, -1.7054665088653564, -1.5732163190841675, 0.19273468852043152, -0.6934249401092529, 0.7290894985198975, 0.07825577259063721, -1.4586923122406006, 0.4384814202785492, 0.3366953432559967, 1.1596174240112305, 1.4424431324005127, -0.2457277774810791, -1.651186466217041, -0.5844133496284485, -0.28120070695877075, -0.7442417144775391, 0.27182939648628235, -0.22922928631305695, 1.1558977365493774, 0.8790388107299805, 1.2454478740692139, 2.3730247020721436, 0.30050942301750183, -1.2301181554794312, 2.8777999877929688, 0.180922731757164, 1.3739351034164429, -0.5004878044128418, -0.10579884052276611, -0.23382340371608734, -0.5110233426094055, -0.28569066524505615, -0.4125027060508728, -0.5559443831443787, 1.0525046586990356, 0.21679607033729553, 1.5737345218658447, 0.8315290212631226, -1.5650907754898071, -1.5242955684661865, -1.7893202304840088, 0.9991921186447144, -1.6165220737457275, -0.5904897451400757, 0.563586950302124, -1.3244322538375854, -0.6742908358573914, -0.36006826162338257, 0.8885963559150696, -0.11295302957296371, -2.0142838954925537, -0.38526517152786255, -0.2776769697666168, 0.3993541896343231, 1.1876606941223145, -0.9290674328804016, 0.4530651569366455, -1.1292731761932373, 0.8708134889602661, -0.7900328636169434, -0.5380614995956421, 1.5184941291809082, -0.5968793034553528, 0.6301385164260864, 2.0073022842407227, 0.8292663097381592, -0.9793534278869629, -0.026881007477641106, -0.24332158267498016, -2.867305278778076, -0.2705347239971161, 1.2504907846450806, 0.9490726590156555, -0.8193154335021973, -0.9341521263122559, 0.9790186285972595, 1.0198036432266235, -0.4542805254459381, 1.7780237197875977, -0.9045847654342651, -0.2848449647426605, -2.249453544616699, 0.41498708724975586, 0.93340665102005, 0.133344367146492, 0.7566165328025818, -0.5270880460739136, 0.4050987660884857, 0.13152986764907837, 2.174732208251953, 1.6829736232757568, -0.16812632977962494, -1.1019539833068848, 1.5316812992095947, -0.06422590464353561, -1.4593292474746704, 0.6984423398971558, -0.04284743592143059, 1.0173228979110718, 1.6802209615707397, 2.046828031539917, -1.1841236352920532, 2.235666275024414, 0.36130502820014954, -0.5551095604896545, 1.4669725894927979, 0.24507370591163635, 0.4027748703956604, 0.24253158271312714, 1.3924864530563354, -0.12773774564266205, -0.6817183494567871, -0.9291352033615112, -0.2564793825149536, -2.0895280838012695, -0.7814710140228271, -1.2759267091751099, -1.7486523389816284, -0.20007924735546112, -1.1963287591934204, 0.9962998032569885, -0.23806902766227722, 0.1092580109834671, 0.41506174206733704, -0.1590556800365448, -2.7579987049102783, -0.5716575384140015, 0.6763877868652344, 0.6644616723060608, 0.037127431482076645, 0.9732295274734497, -0.995851993560791, -0.12825915217399597, 0.7198702692985535, -1.0126419067382812, -0.35498082637786865, 3.2391514778137207, 0.5714003443717957, -0.7297978401184082, -0.616188108921051, -0.09381243586540222, -0.5908746719360352, -0.5112580060958862, -0.3552686870098114, 1.0203373432159424, -0.7298353314399719, -1.122083067893982, 0.15464109182357788, -1.8781180381774902, -0.3917516767978668, 0.6833159923553467, -0.1566954404115677, -0.006934743840247393, -0.5259150862693787, 0.565065860748291, 1.3328962326049805, 0.24060176312923431, 0.41890189051628113, -0.08378702402114868, -0.6645696759223938, -1.3539985418319702, -0.24806183576583862, -0.4402581453323364, 1.8342933654785156, -0.40849828720092773, -0.6448135375976562, 0.4890331029891968, 1.5104947090148926, -1.7905192375183105, -0.5906493067741394, 0.14095333218574524, 0.07841718196868896, -0.239820659160614, 0.672284722328186, -0.09994391351938248, 0.9143838286399841, -0.8836039900779724, -0.48041340708732605, -0.21010123193264008, -0.8870187997817993, 0.5174175500869751, -0.4574296474456787, 0.37020936608314514, 0.060301702469587326, -0.21254512667655945, 0.029221458360552788, -0.5929732918739319, 1.3279536962509155, 0.44952911138534546, -0.6058595180511475, -0.25064361095428467, -0.05965214967727661, -0.2951980233192444, 0.7186802625656128, -0.19050762057304382, 0.3457571268081665, -0.954901933670044, 0.41368693113327026, -0.4672025144100189, -0.9240301847457886, 0.01571362093091011, -0.13538870215415955, -0.7432875633239746, 0.6416847705841064, -0.5326437950134277, -0.8581942915916443, -0.03745623305439949, -0.1400769054889679, 1.8329746723175049, -0.24715815484523773, -0.291657954454422, 1.7126235961914062, -1.8571076393127441, -0.04514651000499725, 0.6920984387397766, 1.2850130796432495, 0.6843956112861633, -0.7669976353645325, -0.1970735341310501, 0.31313371658325195, -2.5974786281585693, -0.9578237533569336, -0.4671715199947357, -0.15288202464580536, -0.05629631504416466, -1.5955241918563843, 0.9302436113357544, -1.283380389213562, 0.08241371810436249, 0.44541800022125244, 1.027074933052063, 1.7417209148406982, -1.180508017539978, -0.4589337110519409, -0.002748503116890788, -1.3825323581695557, 0.6301324367523193, 0.31305429339408875, 1.3903570175170898, -1.6774362325668335, 1.842213749885559, -0.7888297438621521, 1.3064771890640259, 0.6867408156394958, 1.2670114040374756, 0.05270179733633995, 0.11474980413913727, -0.9366470575332642, -0.7512353658676147, 2.1430253982543945, 1.2671899795532227, 1.3021366596221924, -0.8343408107757568, 0.8204423785209656, -2.335047960281372, -0.9387392401695251, -0.9012348055839539, 0.7024669647216797, -1.3547518253326416, 1.1276861429214478, 0.2394799292087555, 0.30686214566230774, 0.054239559918642044, -0.485246866941452, 0.8581308722496033, 0.03503179922699928, 1.1930476427078247, 2.521430492401123, -0.4184058606624603, -0.5335078239440918, -1.1493035554885864, 0.591257631778717, -0.8538969159126282, -1.0960779190063477, 0.18145455420017242, 0.15008403360843658, -0.13307692110538483, 1.3706047534942627, -0.09175112843513489, -0.00451447069644928, -1.2361968755722046, 0.1827230155467987, 0.7602312564849854, -1.7507562637329102, -0.2500363290309906, -0.7104718685150146, -1.2758381366729736, -0.10394122451543808, -0.7930495142936707, -0.4271174669265747, 0.6444895267486572, 0.029826637357473373, 0.34355059266090393, 0.3460603654384613, 0.6131433844566345, 1.1538933515548706, 1.1056658029556274, -0.006011785939335823, -1.2024004459381104, -3.1076884269714355, 1.0836758613586426, -1.4600510597229004, 0.4512227177619934, -0.6442723870277405, 0.591752827167511, -1.7423396110534668, 1.5457764863967896, -0.1830175518989563, 0.5132040977478027, -0.28864333033561707, -1.6200932264328003, 0.659947395324707, 0.6194988489151001, 0.14374306797981262, -0.29686781764030457, -0.384964257478714, -0.769425094127655, -0.08871781826019287, 0.9307785630226135, -0.11596866697072983, 0.41978219151496887, -0.3071664571762085, 2.6877379417419434, -0.3218814432621002, -0.564393937587738, -0.17069979012012482, -0.5916076898574829, 1.2810797691345215, 0.3016372621059418, 1.429989218711853, -0.520516037940979, -0.5293111801147461, 1.5508462190628052, 0.3809974193572998, -1.1325559616088867, -0.7819841504096985, -0.33901289105415344, 0.024041280150413513, -1.4274028539657593, -1.177231788635254, -0.12621363997459412, 0.10808399319648743, -0.5876011252403259, 1.0786950588226318, 1.330989122390747, 0.40278100967407227, 2.149698495864868, 0.15162377059459686, -0.26921382546424866, 0.2469441294670105, 1.768330454826355, 0.2951137125492096, 0.8273469805717468, 0.28784671425819397, -0.144795760512352, -0.6020107865333557, 0.3736259639263153, 0.5007010698318481, -0.6342291235923767, -0.8985752463340759, 0.4679039716720581, 1.340469479560852, 0.8179271817207336, -0.5903931856155396, 0.1948157697916031, -0.010602869093418121, 0.5005497336387634, 1.44121515750885, 0.38112738728523254, -1.1954861879348755, 0.9785640239715576, 0.2949424982070923, -1.2104378938674927, 0.5028231143951416, 0.845194399356842, 2.0613796710968018, 0.05245792865753174, 2.149509906768799, 0.27232444286346436, 1.7041140794754028, 0.44028690457344055, -1.2879291772842407, -0.5967419147491455, 1.6689205169677734, -1.0860919952392578, -0.2858896851539612, -0.5208622217178345, -1.9421615600585938, -0.33460792899131775, 0.5332837700843811, 0.27126258611679077, -0.43648600578308105, 0.7566787004470825, 1.0734668970108032, 0.6601344347000122, 0.45458510518074036, 1.0552769899368286, 0.6428532600402832, -0.14038686454296112, 0.6351301074028015, -0.7014364004135132, -0.4954765737056732, 2.1420204639434814, -0.6579523682594299, -0.5113110542297363, -0.8381251692771912, -0.7385687828063965, -0.10838165134191513, -0.39708951115608215, -1.3711274862289429, 0.4158518612384796, 1.4940286874771118, -2.1447694301605225, 1.3124289512634277, 1.1090096235275269, 1.557524561882019, -0.04633740708231926, 1.319291353225708, 0.26517629623413086, 1.0637975931167603, -1.5567973852157593, 0.4635789394378662, 0.3193344473838806, 1.4077966213226318, -0.8454786539077759, 0.5148099660873413, -0.4711996614933014, -0.3686891496181488, 0.4840598702430725, 0.1284724324941635, -1.5299075841903687, 0.4442722201347351, 0.011369092389941216, 2.153212785720825, 0.39201414585113525, 1.1967883110046387, 1.011759638786316, 0.26144832372665405, 1.417320728302002, 0.5248706340789795, 0.9280304908752441, -0.7813296914100647, 0.7829415202140808, -0.4589982032775879, -0.38800281286239624, 0.1572113037109375, -0.3489009141921997, -0.5551112294197083, -0.1781352162361145, 0.8205578923225403, 0.8902816772460938, 0.6238454580307007, 0.19249717891216278, -1.9169032573699951, 0.9370253086090088, -0.40949776768684387, -0.14142408967018127, 1.9712393283843994, 0.19480688869953156, 1.684563398361206, 0.5635180473327637, 0.9374810457229614, -2.0484039783477783, -0.16590331494808197, -2.0433034896850586, 0.8530838489532471, 1.566914439201355, -0.04388990253210068, -1.1883071660995483, -0.8059871792793274, -1.2348724603652954, 0.08066997677087784, 0.043761517852544785, 0.6943191885948181, -0.16717477142810822, -0.6078947186470032, 1.2583516836166382, -0.9834551811218262, -0.2710393965244293, 0.39838701486587524, -0.8588717579841614, -1.613025426864624, 1.5818012952804565, 1.3948644399642944, 0.6992645859718323, -1.4533237218856812, -0.817764937877655, 0.15686023235321045, -0.24401278793811798, -0.562743067741394, -0.5017608404159546, 0.8978107571601868, -0.24139107763767242, 0.3224351406097412, 1.0101110935211182, -0.24806591868400574, -0.13419991731643677, -0.15856753289699554, -0.8362321257591248, -1.6813112497329712, -1.2971943616867065, 0.03092707134783268, 1.3409086465835571, -0.7255610823631287, -0.5690189599990845, 0.06540476530790329, -2.3718249797821045, 1.965453863143921, 0.44076061248779297, -0.026491763070225716, 1.2073317766189575, -0.6085094809532166, 1.9051923751831055, 0.2353089302778244, 0.3445337414741516, -2.2412526607513428, -1.5845892429351807, -2.116413116455078, 3.5705227851867676, 0.24588704109191895, -0.8214146494865417, 0.8723594546318054, 1.3380510807037354, 1.3305720090866089, -0.6355627179145813, 1.2317450046539307, 0.04342987760901451, 1.7546683549880981, 0.6143984198570251, -0.9915037155151367, -0.5186232328414917, -1.196065902709961, -2.3546855449676514, -0.13959302008152008, 0.7921515107154846, 0.06490395218133926, 1.8943860530853271, -0.757989227771759, -1.1930838823318481, -0.8347406983375549, 1.248179316520691, 1.8421258926391602, 1.7889634370803833, -1.2342615127563477, -0.8157297968864441, 1.4375591278076172, 0.6273996829986572, 0.13312537968158722, -0.3107270300388336, 1.4335191249847412, 0.277305006980896, -0.21801446378231049, 1.4311946630477905, -1.2944681644439697, 0.864543616771698, 0.3670963644981384, -0.7530734539031982, -0.828039824962616, -0.6641520857810974, -0.4596814215183258, -1.353196620941162, -0.7620536088943481, -1.3841878175735474, -1.3714886903762817, -0.4731244146823883, -0.22178612649440765, -0.3288981318473816, -1.1384896039962769, 0.6602324843406677, 0.37127819657325745, -0.32499685883522034, -0.2864540219306946, -0.2815881669521332, -0.06495506316423416, 1.715341329574585, 0.5088805556297302, -0.05560019612312317, -0.6830959916114807, 0.017166702076792717, 3.07134747505188, -0.3235503137111664, -0.1315292865037918, 0.018038015812635422, 0.1427832394838333, -0.13058076798915863, 1.822497844696045, 1.1781259775161743, 0.055446647107601166, -1.2012265920639038, -1.5829795598983765, -1.3831275701522827, -1.3070355653762817, -0.8436573147773743, 1.4726511240005493, -1.9333646297454834, -0.7377656102180481, 2.2593636512756348, 0.4231989085674286, -0.7590815424919128, 0.08742713183164597, 0.7206388711929321, -3.0275821685791016, -0.02517639473080635, 0.232696533203125, 0.2933661937713623, 0.640408456325531, 3.007396697998047, -1.1170320510864258, 0.3263307809829712, -0.886568546295166, 1.5473673343658447, -1.1649507284164429, -0.0310344398021698, 1.8644837141036987, 0.9213123917579651, -0.784442663192749, 2.00506854057312, 0.924392580986023, 1.4603826999664307, 0.64640212059021, 0.4890991747379303, -0.5613101720809937, -0.3205871284008026, -0.9100334644317627, -0.29763302206993103, -0.6969446539878845, -0.09035109728574753, 0.33117175102233887, 0.6576225161552429, -0.5584225654602051, 0.4369222819805145, 0.05177019163966179, -0.7437981963157654, -0.9775553941726685, 0.7676750421524048, 0.15324001014232635, 0.05184305086731911, -0.8010916113853455, -0.5624780058860779, 0.022109711542725563, 0.9523270726203918, -1.0792734622955322, -1.1803635358810425, -0.8347675204277039, 0.27724742889404297, 0.07034365832805634, 0.028571369126439095, -0.2765059471130371, 0.559961199760437, 0.3343595862388611, 1.2557200193405151, -0.5595533847808838, 0.32565662264823914, 0.2260514795780182, -1.66863214969635, 1.0543185472488403, 0.8049440979957581, 1.7481473684310913, 1.0822410583496094, 1.1198545694351196, -1.1068120002746582, 1.6522632837295532, -0.22258691489696503, 0.3096240758895874, 0.3708798885345459, -0.9238574504852295, -1.493036150932312, -1.9816615581512451, -0.2448083460330963, 0.3570767343044281, -0.31515955924987793, -0.43532195687294006, 0.7844482660293579, 1.9959897994995117, 0.9339327812194824, 1.6265289783477783, -0.8997432589530945, -0.09294399619102478, 1.1386057138442993, -2.4259860515594482, 0.06648727506399155, 1.01146399974823, -0.33185097575187683, -2.079634428024292, -0.026739783585071564, -0.965444803237915, 1.2716745138168335, -1.185855507850647, -0.17303651571273804, 1.156590223312378, -0.1752963364124298, -0.1561281830072403, 0.3679161071777344, -0.15577425062656403, -0.8552541136741638, 0.36350297927856445, -0.856273889541626, -0.3039711117744446, -1.2562991380691528, 0.3352946639060974, 0.034431084990501404, -0.1012900322675705, -0.9021162390708923, 0.5482597351074219, 0.21178153157234192, -1.1576005220413208, -0.8231194615364075, -0.4508014917373657, 0.5466187596321106, -0.6607133150100708, 1.350104808807373, 0.4232773780822754, 0.6782099604606628, 0.3939162790775299, 0.8458594679832458, -0.6959283351898193, -3.2010843753814697, 0.24413825571537018, 0.3201540410518646, -0.2567403018474579, 0.6318114399909973, -1.1024988889694214, 0.38187336921691895, -1.3041601181030273, 0.695789635181427, 2.470705509185791, -2.695322275161743, -0.9452394247055054, 0.7245679497718811, 0.5592758059501648, 0.9374874830245972, -1.0613559484481812, -0.5883492827415466, 0.12286563962697983, 0.7057347893714905, 1.2845523357391357, 0.23311971127986908, 0.665513813495636, 0.44432535767555237, -1.0343525409698486, 0.5396679043769836, -0.22508054971694946, 1.5201369524002075, -0.23761270940303802, 0.1940680891275406, 0.2407587468624115, -0.9574389457702637, -1.9006162881851196, 2.3446121215820312, 0.7066845893859863, 2.2597885131835938, 0.19178548455238342, 0.23921096324920654, -1.1673835515975952, 0.8701203465461731, -0.16121509671211243, 0.6484191417694092, 0.0007310205255635083, -1.4164669513702393, -0.7839095592498779, -0.5036500096321106, -0.12267174571752548, -0.09133362770080566, -0.5566020607948303, 0.5723909139633179, 0.8069887757301331, 1.9647549390792847, 1.646066427230835, 2.1846187114715576, 0.32426103949546814, -0.24640525877475739, 0.28909924626350403, -0.6679136753082275, -1.3016316890716553, 1.4213436841964722, -1.2292125225067139, 1.5780575275421143, -1.4084320068359375, 0.9067286849021912, 0.6130558252334595, -0.818745493888855, -0.6300426721572876, 0.059022050350904465, -0.9120805263519287, 1.229903221130371, -0.3933616876602173, -1.506044864654541, 0.6535877585411072, -0.4979788362979889, -0.038460418581962585, -0.4771215617656708, 0.3513072431087494, 0.6859995722770691, 0.7199677228927612, 0.34847161173820496, -0.5370708107948303, -0.5760862231254578, 0.886746346950531, -1.8225038051605225, -1.3886849880218506, 0.9454799890518188, -0.30016636848449707, 0.4957520663738251, 0.6857839822769165, -1.2395520210266113, -0.33196893334388733, -0.7669446468353271, 1.2656010389328003, 1.2130411863327026, -0.723452627658844, 0.46095985174179077, -0.5736923813819885, -0.8582645058631897, -1.324215054512024, -0.685670018196106, 0.319465309381485, 0.8924190998077393, -0.10037274658679962, 0.9214152097702026, -1.0532777309417725, 0.22837014496326447, 0.8451884984970093, -0.7474625706672668, -0.0822233185172081, -0.7340170741081238, 0.7732852101325989, 1.3819332122802734, -0.3405521512031555, 0.8104047775268555, 0.7175484299659729, 1.1441242694854736, 0.5484912395477295, -0.6978936791419983, 0.6915886998176575, -1.6099220514297485, -0.9141170978546143, -0.269808828830719, -1.6520627737045288, 0.3385545015335083, 0.444099098443985, 0.4724651575088501, 0.40939122438430786, -2.0321226119995117, 0.9061484336853027, -1.6984835863113403, -0.8664076328277588, 0.3508296310901642, -0.7719057202339172, -0.8839297294616699, 0.4741448760032654, -1.2466834783554077, 1.3161892890930176, 0.06455083936452866, -1.3513904809951782, -1.6490671634674072, -0.5458526611328125, -0.2029981017112732, 0.1099553182721138, 0.6060992479324341, -0.928862452507019, -1.0706980228424072, -2.5391154289245605, 0.5795071721076965, -0.18085318803787231, -1.794354796409607, -0.8554061055183411, 1.1132128238677979, -0.09574857354164124, -2.1734750270843506, -0.2881314158439636, -1.5714669227600098, 1.1042507886886597, 0.8487585186958313, -0.13206009566783905, 0.44929853081703186, 1.1825100183486938, 0.6008800268173218, 0.38855862617492676, 0.3703990578651428, -0.5815524458885193, -1.4053759574890137, -0.5670247077941895, 0.5754393339157104, 0.22185084223747253, -0.7563543319702148, -1.4037915468215942, 0.29505160450935364, -1.8128080368041992, 0.4555581510066986, 0.9937686920166016, 0.32272031903266907, -1.7941803932189941, -2.065624237060547, 0.8570897579193115, -0.9571558237075806, 0.5423848032951355, -0.3292504549026489, -2.052061080932617, -2.3638498783111572, 0.43443062901496887, 1.2133203744888306, -0.571183443069458, -0.2472873032093048, 0.33872276544570923, -1.7997452020645142, -0.026717625558376312, -1.916333556175232, 1.0499937534332275, 0.1023448258638382, -0.670185923576355, -1.3026005029678345, -0.5560635328292847, 0.6285610795021057, -0.4404323101043701, -0.18677225708961487, -0.3400844931602478, -0.34939658641815186, -0.13892798125743866, 0.07159125804901123, -0.19732432067394257, -0.033446844667196274, -0.7096765041351318, 0.5998866558074951, 1.5904028415679932, 1.5547130107879639, 1.0891871452331543, -0.7611936926841736, 0.07360469549894333, 0.8466376066207886, -0.3095814287662506, 0.23143382370471954, -1.0830817222595215, 0.045393966138362885, -1.1195069551467896, -0.37505781650543213, 0.9977752566337585, 0.22550109028816223, -0.06546497344970703, -2.6048731803894043, -1.146221399307251, 0.7955296635627747, 1.8468296527862549, -0.03854948654770851, -0.31162014603614807, 0.4149186909198761, 0.816658616065979, -2.01369047164917, -1.0967825651168823, 1.0726845264434814, 1.4923514127731323, 2.2444026470184326, -0.698451817035675, 1.6106654405593872, -0.396380215883255, 1.2257707118988037, -1.0963099002838135, -0.07018531858921051, -1.6655173301696777, 0.7251659631729126, -0.505509614944458, -1.9229633808135986, 1.25843346118927, -0.2550295889377594, 0.6164153814315796, -0.3239710032939911, -0.13880443572998047, -1.0897852182388306, -0.044631339609622955, -0.6495392918586731, 0.21273425221443176, 2.018925666809082, 0.2481713891029358, 0.09806269407272339, 0.321900874376297, 0.7006253004074097, 1.2037551403045654, 0.48496291041374207, -0.5175527930259705, 1.6753575801849365, -0.42529553174972534, 0.007719325367361307, 0.4416039288043976, 1.3184194564819336, -1.1835907697677612, -1.7749861478805542, 0.23846915364265442, -0.7809562683105469, 0.04731234908103943, -1.262081265449524, 0.24846525490283966, -0.09590597450733185, 0.2660828232765198, -0.2960463762283325, 0.7426877617835999, -1.7532925605773926, 1.9605400562286377, -0.9178289771080017, -0.6520133018493652, -1.6323350667953491, -1.023345708847046, -0.2872259020805359, 0.0027137980796396732, -0.47441530227661133, 2.2335455417633057, 1.8693188428878784, 1.1129064559936523, 0.46999889612197876, -0.5397517681121826, 0.5800701975822449, 0.0198926143348217, 2.309323310852051, 0.6186379194259644, 0.3280588984489441, -0.27670231461524963, -0.036328528076410294, -0.2835868299007416, -0.6597167253494263, 0.756746232509613, 0.7760816812515259, 2.491757392883301, -0.33688807487487793, -1.2448234558105469, -0.6706481575965881, 0.5565692186355591, -1.383988380432129, -1.3456926345825195, 0.23849894106388092, 0.8933041095733643, 1.9979945421218872, 0.73499995470047, -1.6594651937484741, 0.8928592205047607, 1.1749622821807861, -0.6397414803504944, 0.32306188344955444, -1.1997675895690918, 0.45635828375816345, 0.0001356053544441238, -1.9769989252090454, -1.184483528137207, -0.3614160418510437, -1.5650153160095215, -0.8722493648529053, -1.608426570892334, -0.8936643004417419, -0.6053223013877869, -0.41031405329704285, -0.5772759318351746, -0.8473992943763733, -0.6034106016159058, 0.3696552813053131, 2.3641345500946045, 0.44745540618896484, -0.7528588175773621, 0.4444197118282318, -0.7138481140136719, -0.7058995366096497, -0.22944696247577667, 0.022421149536967278, 0.6015704274177551, -0.8324287533760071, -0.6918202042579651, -1.683518886566162, -1.6032966375350952, 1.0408238172531128, -0.5219410061836243, -1.849326252937317, 0.021891459822654724, -0.3132023215293884, 0.9080946445465088, -0.15724623203277588, 1.7661935091018677, -1.0562132596969604, -1.3814374208450317, 0.9394625425338745, 1.1190444231033325, 0.29754987359046936, 0.49052610993385315, -1.6138954162597656, 0.3204289972782135, -1.4487361907958984, -0.2553376853466034, -0.5675512552261353, -0.9876134395599365, -0.5393096208572388, 1.0027121305465698, -0.8561215400695801, -1.1920199394226074, -1.4295096397399902, -0.5234570503234863, 0.6868159770965576, -0.39461997151374817, 0.7282931804656982, -1.8254446983337402, -0.7676609754562378, -2.4088993072509766, 0.843601644039154, 1.1561453342437744, -0.14330898225307465, 0.6904067397117615, 0.8014616966247559, -0.2542532682418823, 0.7750835418701172, 1.2987635135650635, -0.9423432350158691, 1.089219331741333, -0.8432062268257141, -0.6818602085113525, -1.0598801374435425, -1.1217647790908813, 1.253143548965454, -0.9923875331878662, -0.03677889704704285, -0.42904752492904663, -0.8189002275466919, 1.2803529500961304, 1.1703886985778809, -0.07837270945310593, -0.57686448097229, -1.2043763399124146, -0.16429387032985687, 0.45459291338920593, 0.26378920674324036, 1.1062662601470947, 0.27412450313568115, 0.0628514513373375, -1.3310606479644775, 0.9306734800338745, 1.0371613502502441, 0.06446865200996399, 0.2714294195175171, -0.21043315529823303, 1.0222822427749634, -0.40311986207962036, 1.2209314107894897, 0.5316150784492493, -1.7824703454971313, -0.15963540971279144, -0.4062819480895996, -0.37126168608665466, 1.875084638595581, 1.7540631294250488, 0.4466840326786041, -0.26591619849205017, 0.3943297564983368, -0.3515329658985138, 1.2548326253890991, 0.510093629360199, -0.0888078436255455, 0.2202558070421219, 1.5524795055389404, -0.6529617309570312, 1.9573878049850464, 2.0955259799957275, 0.2629908621311188, -0.153622567653656, 0.5458351373672485, 0.7329429984092712, -0.9786975979804993, 0.0022107150871306658, 0.583565354347229, -0.830612301826477, -0.23648422956466675, 0.1799384206533432, 0.01924215629696846, 0.24816690385341644, 0.8021169900894165, -0.1471364051103592, -0.12399058043956757, 0.25627562403678894, -0.37911519408226013, 1.872752070426941, -0.8924791216850281, 0.6502799987792969, 0.5104594826698303, 0.4664386808872223, -0.015922503545880318, 0.7703215479850769, 1.3501096963882446, 0.5245826244354248, -1.2355135679244995, -0.1301237791776657, -1.9840641021728516, -3.0682380199432373, 0.6837825775146484, 0.04056277871131897, -0.48527175188064575, 2.8096415996551514, 1.173750400543213, -0.01502344012260437, 0.015324178151786327, -1.0164573192596436, 0.4325925409793854, -2.733255624771118, -2.643667459487915, -0.5566866993904114, -0.39032378792762756, -0.8790459632873535, -2.255600929260254, 0.7279271483421326, -0.07964113354682922, -0.7110387086868286, 0.7631353139877319, -0.13786308467388153, -1.9012125730514526, 1.4770623445510864, -0.10230529308319092, 1.25486159324646, -1.3121800422668457, -0.8501982092857361, 0.9312407374382019, -1.5745840072631836, 2.3841540813446045, 0.35529354214668274, -0.4500688910484314, -0.9225654602050781, -0.9508329033851624, -2.4119760990142822, 0.7666352987289429, 1.6422221660614014, 1.2659175395965576, -0.8571795225143433, 0.7430221438407898, -0.09039881080389023, 0.8384019732475281, -0.1454741209745407, -1.2513586282730103, -1.11293625831604, 0.2304055243730545, 0.05622315779328346, 0.22214064002037048, -1.6958383321762085, -0.7675034999847412, -0.247990220785141, 1.0910261869430542, -0.7486793994903564, -0.4534279406070709, -0.1559019684791565, 0.6874068975448608, 0.9705964922904968, 0.8573017120361328, -0.38660886883735657, -2.4909205436706543, -0.8348526954650879, 0.5268590450286865, 1.75339937210083, -0.8996121287345886, -0.280769944190979, 0.37566184997558594, 0.18710441887378693, -1.4856911897659302, 0.40700429677963257, 0.7909821271896362, 0.8495987057685852, 1.1718008518218994, -0.27771738171577454, -0.2270263284444809, 0.020717723295092583, -0.18684083223342896, 0.09159217774868011, -1.061603307723999, 0.43643903732299805, -0.7566249966621399, -0.05598035082221031, 0.2731328010559082, 0.4781939685344696, -0.7395739555358887, 1.6635783910751343, 0.4564208686351776, -1.412044882774353, 1.0967686176300049, 0.38748934864997864, -1.1665383577346802, 0.7023042440414429, 0.37757766246795654, -1.8140358924865723, 1.7463005781173706, 1.2410087585449219, -0.25791412591934204, -1.8277883529663086, -0.6936488747596741, -1.1682442426681519, -0.7732633352279663, -1.654427409172058, -0.3778638541698456, 0.8386522531509399, -0.6884869337081909, -0.26556065678596497, 1.2087254524230957, 1.5158473253250122, 0.13015618920326233, -0.6692354083061218, -0.09056831151247025, -1.6450610160827637, 0.0938233807682991, -0.3939264714717865, 0.8157133460044861, -1.1145573854446411, -0.2439783215522766, 1.3301125764846802, 0.6850215792655945, -0.4680024981498718, 0.09458190202713013, -0.06758984178304672, -0.0817694142460823, 0.58689945936203, -0.8876526355743408, -0.8370242714881897, -2.511945962905884, 0.5242605805397034, 0.3370167016983032, -0.4618760347366333, -0.06859232485294342, 1.9425493478775024, -0.6298187971115112, -0.31691616773605347, 1.37336003780365, 0.8037813901901245, 1.2220027446746826, -1.9075722694396973, -0.6558403372764587, -1.1312426328659058, 0.34172940254211426, 0.44173118472099304, -0.5917139053344727, 0.9343949556350708, 0.14158640801906586, -0.5694971680641174, 1.1036521196365356, -1.8958303928375244, 0.6058385968208313, -2.042895793914795, 0.4669438600540161, -0.44837430119514465, 0.995975911617279, 1.8191837072372437, -0.31699249148368835, 0.44076257944107056, -0.592109203338623, 0.7193500399589539, 0.7976769208908081, -1.2304872274398804, 0.0063630626536905766, 0.24146154522895813, -0.2775929868221283, 1.559300184249878, 0.0222467090934515, 0.026863442733883858, -0.6101998686790466, -0.31827157735824585, 1.0573183298110962, -1.7175195217132568, 0.033658716827631, 1.703198790550232, 0.24562203884124756, 0.7063864469528198, -1.592004418373108, 0.12638919055461884, 0.8981300592422485, 2.199753999710083, -0.3893184959888458, -0.4809478223323822, -1.4586091041564941, -1.3269211053848267, 0.9021188616752625, 0.7371260523796082, -1.1226890087127686, 0.39085131883621216, -1.3884587287902832, -0.41154882311820984, 0.6719458103179932, -0.8732215166091919, 2.116710662841797, -1.978376030921936, -0.17677956819534302, 0.20777970552444458, -0.8268148303031921, 0.6038362979888916, -0.4536250829696655, -0.4934180974960327, -1.7649441957473755, 1.4908705949783325, -1.0425570011138916, -0.395965039730072, -2.3288540840148926, -0.4607428312301636, 1.0754607915878296, 0.40128469467163086, -0.4629318416118622, -1.6730791330337524, -0.11704613268375397, 1.1383544206619263, -0.5217916965484619, -1.8832534551620483, -1.5702488422393799, -0.6908354163169861, -1.0813193321228027, -2.197946548461914, -0.13768407702445984, -0.3696891963481903, -0.38938817381858826, 0.09660456329584122, -0.10279470682144165, -1.116491436958313, 1.2712180614471436, -1.196457028388977, 1.101575493812561, 0.09881433844566345, 0.9202223420143127, -0.9216544032096863, 0.1349724978208542, -0.18121537566184998, 1.40478515625, 0.5861736536026001, -0.5355757474899292, -1.3245927095413208, 0.4355975389480591, -0.9906193017959595, 0.5858945250511169, -1.270362377166748, -0.08583279699087143, -0.328893780708313, 0.9324939250946045, 0.5348328351974487, -3.440925359725952, 0.05318301171064377, 0.3158148229122162, 0.30044418573379517, -0.6398382782936096, -0.5412179827690125, -0.3504047393798828, 0.7405369877815247, -1.285335898399353, 0.009438610635697842, 0.431766152381897, -1.3382856845855713, 0.25759372115135193, 0.10683959722518921, 1.7830095291137695, -0.4186137914657593, -1.0020396709442139, -1.988822340965271, 2.231030225753784, 0.3583294749259949, 0.7254546284675598, -2.127594232559204, 0.44013601541519165, -0.10881473124027252, -1.161948323249817, 1.6180492639541626, -0.7159983515739441, 0.5257129073143005, 0.964697003364563, 0.9089418649673462, 0.4419367015361786, -1.4092446565628052, 0.6961591839790344, 1.9754886627197266, 1.36063814163208, -0.23671162128448486, -0.34323665499687195, 0.22254490852355957, -1.1071747541427612, -1.3527129888534546, 0.44879069924354553, 0.3976171612739563, 0.32427531480789185, 0.1319759339094162, -1.8065106868743896, 0.14367815852165222, -1.5406824350357056, 0.7466557621955872, 0.5034164786338806, 0.6409531235694885, 0.6189391016960144, -0.5899802446365356, 0.7073566317558289, -1.8487035036087036, -2.6222283840179443, 0.3116706311702728, 0.697956383228302, 1.258806824684143, 0.08434761315584183, 1.5760483741760254, 0.9225205779075623, -0.6013611555099487, -2.3769795894622803, -0.84431391954422, 0.6923434734344482, -1.5554163455963135, 0.8106151819229126, 1.38882315158844, -0.208067387342453, -0.44583654403686523, 1.1859021186828613, -0.48296135663986206, -0.7525973916053772, 0.43287989497184753, -1.003990650177002, 0.5244731307029724, -0.005211625713855028, -1.276043176651001, 1.233999490737915, 0.8615535497665405, 0.8397588133811951, -0.5289208292961121, -0.0985705554485321, -1.9394192695617676, -0.1530887335538864, 0.584568202495575, 1.3930772542953491, -0.6005735993385315, -1.8928366899490356, 0.4063341021537781, 1.2549391984939575, -0.6678998470306396, -0.9259809851646423, 0.2711174190044403, 0.67967689037323, -0.11833242326974869, 0.5109414458274841, -0.6942476034164429, -0.8897651433944702, -1.468930959701538, -0.9769428968429565, 0.3392569124698639, -0.3338780403137207, -0.5114288926124573, -0.9741758108139038, 0.22739027440547943, 0.21723464131355286, 1.029740333557129, 0.3586670458316803, 0.510899543762207, 1.8889282941818237, 1.381668210029602, 0.5636287927627563, 0.06887039542198181, 0.5063462257385254, -0.631854772567749, 0.3198217451572418, 0.5562055706977844, -0.47963035106658936, -0.5713713765144348, 0.4742411673069, 0.6252214908599854, 0.8219577074050903, -0.4202911853790283, -0.7911074757575989, -0.16457298398017883, 0.021238023415207863, 0.37868279218673706, 0.41572219133377075, 0.8360058069229126, 0.5539011359214783, 0.29894325137138367, 0.27439573407173157, -0.295341432094574, -1.3056474924087524, 0.24455271661281586, 0.18572600185871124, 0.6538925766944885, 0.6213803291320801, -0.5919443964958191, -0.08386678248643875, 0.8635668754577637, -0.7315011620521545, 0.5254948735237122, -0.48526230454444885, 0.1588764637708664, -0.7971195578575134, 0.4533369839191437, 0.493098646402359, -1.122313380241394, 0.16699333488941193, 1.0885121822357178, -0.245906800031662, -0.3549305200576782, 1.1940540075302124, -0.914394736289978, 0.01724889874458313, -0.8966155648231506, 0.7821879982948303, -0.39936909079551697, 0.6954096555709839, 1.4434876441955566, 0.06136081740260124, -2.149456262588501, -0.2905995845794678, 1.44882071018219, -0.6311783790588379, -0.08920887112617493, 0.3794383108615875, 0.2968345582485199, -0.7672368288040161, -0.7291939854621887, 0.555752694606781, 0.08888964354991913, -0.6009389162063599, 0.37106820940971375, 0.8898663520812988, -0.5243703722953796, -0.007314146496355534, 1.1964138746261597, 0.18687298893928528, -2.101046562194824, -0.23518116772174835, 0.32540154457092285, -1.1302835941314697, 1.0894360542297363, -1.923706293106079, -0.7587499022483826, -1.3362376689910889, -0.45969638228416443, 0.9296600222587585, -0.8217669725418091, -0.06681083142757416, -0.04529441520571709, -1.622210144996643, -1.5330305099487305, 2.3449578285217285, 2.0194902420043945, 0.4877661168575287, -0.16085344552993774, -1.3633209466934204, -0.23827652633190155, 0.1689605712890625, -0.5938441753387451, -0.37686413526535034, 0.4551633894443512, -0.08050639182329178, -0.8512547612190247, 1.6089420318603516, 0.17743228375911713, -0.20469500124454498, -0.7887366414070129, -1.6962425708770752, 1.0651648044586182, -2.240478515625, -0.1233186349272728, -0.3399679958820343, 0.553011953830719, -0.43799445033073425, -0.447488009929657, -0.7277397513389587, 1.2672103643417358, 0.45210859179496765, -0.08066686242818832, 0.9486004710197449, 0.030163820832967758, -1.2334907054901123, 0.21320100128650665, -0.1550140231847763, -0.43646231293678284, -0.038747720420360565, 1.4833101034164429, 0.29668837785720825, 0.5535925626754761, -0.5547348260879517, -0.5213201642036438, 0.7252765893936157, 0.7231065630912781, 0.35667669773101807, 1.6252410411834717, -0.1764579564332962, -0.08823580294847488, -0.6318564414978027, 0.40824538469314575, 0.9048925042152405, 0.5501802563667297, -0.5640209317207336, 0.6527968645095825, 1.8660342693328857, -1.5930650234222412, -0.01884360983967781, 0.870185136795044, 0.42494821548461914, 0.3868585228919983, 0.014488659799098969, -0.4486139714717865, 0.6144846677780151, -0.18374991416931152, -1.0924041271209717, -1.703590989112854, -1.0861105918884277, 0.2625281512737274, -0.35965123772621155, 0.07590485364198685, 1.5158109664916992, -0.3940059244632721, 1.2323497533798218, 0.3865604102611542, -2.446352481842041, 1.472918152809143, -1.1526727676391602, 0.7440840601921082, 1.2986842393875122, 0.9110007882118225, 0.8053630590438843, -0.5794904232025146, -0.8895931243896484, -0.5003432631492615, 1.078978180885315, 1.1197876930236816, 1.0099493265151978, 1.000335454940796, 1.1902698278427124, 0.8951342701911926, -0.03488732874393463, -0.5399783849716187, -1.1097747087478638, -1.3508244752883911, 0.9692322611808777, 0.4273083209991455, 0.22383686900138855, -0.5697377920150757, -0.15231473743915558, -0.2301701009273529, 0.05050872266292572, 0.585508406162262, 0.5430821776390076, -1.3099980354309082, -0.6499797701835632, 0.21424627304077148, -0.5010210275650024, -1.8854480981826782, -1.011733889579773, -0.4218501150608063, 0.5282567143440247, 1.2821327447891235, -0.810551106929779, 0.8518205881118774, -0.5046836137771606, 0.874239981174469, 0.783981442451477, -1.0349030494689941, -0.6689186692237854, -1.2565399408340454, -0.5533402562141418, 0.36551719903945923, -0.8166377544403076, 1.1690007448196411, 1.3118245601654053, 0.2720467150211334, 0.14745159447193146, 0.623807430267334, 0.023684661835432053, -1.2875761985778809, -0.5125824213027954, 0.27004873752593994, -0.05198356509208679, -0.7520107626914978, -0.8167654871940613, -0.4840015470981598, -0.3845106065273285, 0.2616063356399536, -1.2589553594589233, 0.2783251702785492, 0.06596265733242035, 0.36924871802330017, -0.2321370542049408, 0.6642770767211914, -1.6501052379608154, 0.3004685342311859, 2.393742322921753, 0.2803663909435272, 1.4580556154251099, -0.6521773934364319, -0.5513685941696167, -0.528526246547699, 0.3993636667728424, 0.3290500342845917, -0.5543874502182007, 0.23377303779125214, 0.5798271894454956, -1.4161499738693237, -0.884101152420044, 0.00898076593875885, 0.7276179790496826, -1.259517788887024, -0.5084285736083984, 0.16598756611347198, 0.3916548788547516, 0.22291497886180878, 1.7814713716506958, 0.7419278025627136, 0.684929370880127, 0.8248820304870605, 0.08524216711521149, -0.08987176418304443, 0.30349719524383545, -0.06679973751306534, 0.3242461681365967, 0.3235037922859192, -0.9287378191947937, 0.9108754396438599, 0.011747399345040321, -0.8220732808113098, -0.0854475200176239, 1.3418066501617432, -0.07562702149152756, -0.43885213136672974, -1.1569721698760986, -0.3204398453235626, 0.2936006188392639, 1.1944246292114258, 0.6809202432632446, 0.2998611330986023, 0.4586339294910431, 1.263076663017273, -0.786982536315918, 0.9088447690010071, -0.8486815094947815, -0.4177486002445221, -1.1826425790786743, 0.03250163421034813, -1.6744115352630615, 1.3890107870101929, -0.8923148512840271, 0.8522191047668457, 0.3794642984867096, -1.1322022676467896, 1.525275468826294, -0.9185753464698792, 0.5848683714866638, 0.5248143076896667, -0.1235242635011673, 2.570781707763672, -0.8281323313713074, 0.5607665777206421, 1.783551812171936, -0.06507924944162369, 0.394122838973999, 0.8525249361991882, -1.3079373836517334, 0.25413739681243896, 0.36063066124916077, 1.8607654571533203, 0.9554345011711121, -0.6956338882446289, -0.1667412966489792, 0.5049188137054443, 0.15843802690505981, 0.002746798563748598, 0.7132605910301208, -2.0919241905212402, -1.167445182800293, -0.8709563612937927, 2.905155897140503, 1.525995135307312, 1.8422644138336182, 1.0589499473571777, 0.7776635885238647, 1.0991592407226562, -0.9666544795036316, -0.49977219104766846, -0.7465465664863586, 1.017564058303833, 0.11542381346225739, 0.36317625641822815, 0.347332239151001, 0.10177282243967056, 1.7018685340881348, 0.38493210077285767, 0.6137755513191223, -2.269528865814209, 0.09657619893550873, -0.49748119711875916, -1.2078874111175537, 0.9317529201507568, 0.4952180087566376, 1.369093656539917, -1.0032520294189453, 1.439725637435913, -0.48404422402381897, -0.42989885807037354, -0.5545446872711182, 0.17094887793064117, 0.5477710366249084, -0.028660854324698448, -0.4250389635562897, 1.2438573837280273, 2.05545711517334, -0.5248972773551941, -1.3253785371780396, 1.3841838836669922, 2.2872743606567383, -1.0098955631256104, 0.477508544921875, 2.1840429306030273, 1.2529946565628052, 1.2606884241104126, -0.040000513195991516, 0.14807628095149994, -0.7480947971343994, -0.43134769797325134, -2.685875654220581, 1.8941352367401123, 0.7036188840866089, 0.5719639658927917, -0.10968542098999023, -0.0845756083726883, 0.2948778569698334, -0.2236744463443756, 0.052136581391096115, 0.16135092079639435, 0.04787638783454895, -0.9899335503578186, -0.27139079570770264, 0.23206347227096558, -0.6731562614440918, 1.5408892631530762, -0.328143447637558, -1.0924769639968872, -0.00865100510418415, -1.6683825254440308, 0.24985066056251526, -0.5923712253570557, -0.7121457457542419, 1.21408212184906, -1.0067577362060547, -0.9258230924606323, 1.7441295385360718, 0.6438711881637573, -0.3554110527038574, -1.371431827545166, -0.4178907871246338, 0.45358583331108093, -1.295472502708435, -0.1652185171842575, -1.2657454013824463, -1.2245079278945923, 0.8619414567947388, -0.05413150042295456, 0.12811771035194397, 0.0967424288392067, -0.2725089192390442, -0.660611629486084, 0.17827683687210083, 0.6482557058334351, 0.06882119923830032, 0.6800366044044495, 0.0027006464079022408, 0.10455130785703659, 0.19281305372714996, -0.6100344657897949, -0.6095423698425293, -0.20645169913768768, -0.7998371720314026, 0.36222079396247864, -0.9290422797203064, 1.3590456247329712, -1.7402266263961792, 0.26694023609161377, -0.4479774236679077, -1.3020553588867188, 0.4714222550392151, 1.233771562576294, -1.215880274772644, -1.6804120540618896, -0.4502093195915222, -0.609398365020752, 0.5578647255897522, 0.3543523848056793, -0.29296907782554626, 0.5845323204994202, -0.30978700518608093, -0.15648414194583893, -0.7756634950637817, 0.03128884360194206, -0.682725727558136, -0.5343844294548035, -1.3071894645690918, -0.4676779806613922, -0.04765244945883751, -0.016513913869857788, 0.6599498391151428, 1.491152286529541, 0.532748281955719, -1.103515625, 0.01376999169588089, 0.8505629301071167, 1.5424137115478516, -0.43709689378738403, -1.0063427686691284, 1.049084186553955, 0.5516738891601562, -1.5225584506988525, 0.41236990690231323, -1.8447445631027222, 0.2905786633491516, 0.5806186199188232, -0.43398651480674744, 0.6789438128471375, -0.20498116314411163, 0.19114618003368378, -1.2321527004241943, -1.606810450553894, 1.379705786705017, -0.8322291374206543, 0.2526717483997345, 1.4046404361724854, -0.43981003761291504, 0.18848441541194916, 0.07219811528921127, -0.10203386843204498, 0.1910463571548462, 0.3069600462913513, 0.38360652327537537, -0.869964063167572, -0.375497967004776, 0.9383687376976013, 1.3637897968292236, 2.4024996757507324, -1.648341417312622, 0.8538360595703125, 1.1171314716339111, -0.3370417058467865, 0.7282645106315613, -0.3418906629085541, 0.42650076746940613, -1.073019027709961, -1.0028671026229858, -0.2906959652900696, -1.7611021995544434, 0.3769259750843048, -1.1357665061950684, 0.6652823686599731, 0.5574131011962891, 0.7282679080963135, -0.055241357535123825, -0.03248906508088112, -1.727528691291809, 0.31035009026527405, -0.4590916037559509, -0.8195599913597107, -0.39873406291007996, -1.383326530456543, -0.5428655743598938, 0.273655503988266, 0.5077471733093262, -0.559790849685669, -1.1077126264572144, -0.22658629715442657, -0.9499101638793945, 1.188571810722351, 0.5252007246017456, -0.4610474407672882, -1.1260799169540405, -0.11812801659107208, 0.07195795327425003, -0.2655259966850281, 1.0476857423782349, 0.7300600409507751, 1.1921159029006958, -1.7076107263565063, -0.3205195963382721, -0.7204451560974121, -0.12772919237613678, -1.2841753959655762, 0.5067670345306396, 1.932258129119873, -0.08773411065340042, 0.2627650201320648, 0.4940469563007355, -0.4777986705303192, -1.8915830850601196, -0.12264155596494675, -1.2266719341278076, 0.2747771739959717, -0.8374006748199463, 0.5029081702232361, -0.392738401889801, -0.7981140613555908, -0.44211578369140625, -1.0859124660491943, 1.6340922117233276, -1.4194272756576538, 2.0915822982788086, -0.690811038017273, 0.8723409175872803, 2.085973024368286, 0.008310849778354168, -2.0679221153259277, 0.5273200869560242, 1.4779412746429443, 1.4207451343536377, -1.868606448173523, 0.8528529405593872, -0.768611490726471, 0.39894813299179077, -0.16757112741470337, -1.4645997285842896, -1.721847414970398, -0.03264901787042618, 0.9118742346763611, -1.218230128288269, 0.13827821612358093, 1.6904773712158203, -0.13453304767608643, 1.024254560470581, 0.682800829410553, 0.5169075131416321, -0.23141248524188995, -0.36808276176452637, 0.33028191328048706, -1.0123900175094604, -0.2548293173313141, -0.8011121153831482, 0.4601095914840698, -0.7857651710510254, -1.401404619216919, -0.03192019462585449, -1.1560382843017578, -0.6466230154037476, 0.42643195390701294, 1.0752806663513184, 0.46909478306770325, 0.4173792600631714, 0.059831228107213974, -1.6258296966552734, -0.8980093002319336, -0.2097354382276535, -0.9169133901596069, 1.2409651279449463, -0.3814323842525482, 1.2238861322402954, -0.30060485005378723, 0.20904767513275146, -0.16943414509296417, 0.19987422227859497, -1.492479920387268, -0.3655882775783539, 1.575480341911316, -0.4716086983680725, -1.1916818618774414, 0.36722931265830994, 0.43235090374946594, -0.21800057590007782, -0.7765234112739563, 0.46672430634498596, 1.0806046724319458, 0.23460757732391357, 0.29229170083999634, 0.9033519625663757, -0.23422643542289734, -0.19358383119106293, 1.537718415260315, 0.39796021580696106, 1.1601274013519287, 0.8010803461074829, 0.18693695962429047, 0.9461050629615784, -0.2569488286972046, -0.9438074827194214, -0.13604865968227386, -1.0373886823654175, 0.267337828874588, -0.3281693458557129, 1.9726781845092773, 0.6261140704154968, -0.43163615465164185, 1.3248530626296997, -1.416404128074646, 0.04769293591380119, 0.5388235449790955, 1.1275265216827393, -0.14296041429042816, -0.19635097682476044, -1.481038212776184, 1.8722015619277954, -0.5840389132499695, -0.7999320030212402, 2.3244407176971436, 1.1556549072265625, -0.8181912899017334, 0.03900868818163872, 1.6076688766479492, -1.6794511079788208, 0.6732349395751953, 1.9140273332595825, 0.17861546576023102, 1.1405240297317505, 0.833977997303009, -0.08124127238988876, -0.008105447515845299, -0.08237013220787048, -0.5569275617599487, 1.7102863788604736, -0.36560043692588806, -0.9334716796875, 1.1908499002456665, -0.8582904934883118, 0.005288244225084782, -0.4505510628223419, 0.9736696481704712, -0.9526703953742981, 0.8960978984832764, 0.692554235458374, -0.22708526253700256, -1.6073417663574219, -1.0557268857955933, -0.8559952974319458, 1.7330989837646484, -1.3623073101043701, 0.23223963379859924, 1.1322288513183594, 1.4262073040008545, -0.5352729558944702, 0.3954007029533386, -1.140841007232666, 1.1120226383209229, 2.5921785831451416, -1.2500901222229004, -0.16334058344364166, 1.2642133235931396, -0.9040010571479797, 1.2856016159057617, 0.8255152702331543], "dtype": "float32"}, {"__nd__": [-2.7810139656066895, 2.100860118865967, 1.695637822151184, 1.638644814491272, 0.044319335371255875, -1.0222065448760986, 1.2189223766326904, 0.43410876393318176, 0.5919312238693237, -1.4977082014083862, 0.4930128753185272, -1.2422966957092285, 0.11032047867774963, 0.36538365483283997, -0.6804856061935425, 0.36873552203178406, -0.2836592495441437, 2.1602840423583984, 0.3070712089538574, 1.8721364736557007, -0.08695054054260254, -1.9706388711929321, 0.6230099201202393, -0.16468748450279236, 0.9569445252418518, -1.8784655332565308, -0.4680320918560028, 0.6533125638961792, -1.924710988998413, 1.5278890132904053, -0.16756273806095123, 0.6680141091346741, 0.2446395307779312, -0.5418195128440857, 0.8630309104919434, -1.119038701057434, -0.24656249582767487, 0.6295543313026428, -0.6608416438102722, -1.6706467866897583, -1.1781901121139526, -0.12318923324346542, 0.46687257289886475, 0.04017356038093567, 0.839740514755249, 1.1201465129852295, -0.19769808650016785, -0.9180278778076172, 0.5314282774925232, -0.10166303813457489, -0.05484258010983467, 0.22949686646461487, -0.37105879187583923, -2.09981107711792, 0.44001534581184387, -0.5166200399398804, 1.7712409496307373, -0.47917068004608154, 1.0059046745300293, 0.9368239045143127, -0.05154982954263687, 1.1881879568099976, 0.46732333302497864, 0.3524315357208252, -0.4373374581336975, -1.3322253227233887, -1.3756251335144043, -0.5556758046150208, 0.8200973868370056, 2.1587095260620117, -0.14385585486888885, 1.1583417654037476, 0.5658899545669556, -0.7593710422515869, 0.07697892189025879, -1.684248924255371, 0.8118426203727722, 0.4019313156604767, 0.9040481448173523, -0.13641157746315002, -0.8475263714790344, -0.42813241481781006, -0.28805240988731384, 0.006003157701343298, -0.4374904930591583, 0.9020804166793823, -0.5065577030181885, -1.2896490097045898, -0.8986168503761292, -0.6202517151832581, -0.32585659623146057, 0.768970251083374, -1.2219226360321045, -0.4315401017665863, -0.7315716743469238, -2.188072681427002, -0.574967622756958, -0.6028668284416199, 2.1113765239715576, -1.2339835166931152, 1.0902812480926514, -1.4789961576461792, -2.0367116928100586, -0.9651100039482117, -0.6799358129501343, 0.2562941014766693, 0.8886638283729553, -0.8347854614257812, 0.3335210680961609, 1.1862883567810059, -1.4568299055099487, 0.46935099363327026, -1.7520296573638916, 1.7513313293457031, -2.2164578437805176, 1.0078057050704956, 0.5684115886688232, 0.6022675037384033, -0.20198394358158112, -2.0890352725982666, -1.340590238571167, 1.186295986175537, -0.5214023590087891, 2.4947662353515625, 0.28719398379325867, -0.25298580527305603, 0.36211779713630676, 0.17876683175563812, -0.7223595380783081, -0.6339210867881775, -1.1914750337600708, 0.35021549463272095, -0.5034229755401611, -0.00020582570869009942, -0.3758072853088379, 1.428218126296997, 1.1397202014923096, -0.19173790514469147, -0.24906165897846222, 0.8624836802482605, -0.8939031362533569, -0.12560679018497467, 0.2613251209259033, -0.07139123976230621, -0.4365192651748657, -1.1607248783111572, 0.5298395156860352, 1.253407597541809, 0.15976864099502563, -2.4514951705932617, 0.35566046833992004, -1.5467848777770996, 0.20571525394916534, 0.24032963812351227, 1.2168023586273193, -1.3696403503417969, 0.10470981895923615, 1.7336530685424805, 1.0196166038513184, -0.9914944767951965, -0.4534059166908264, -1.0540738105773926, 0.37784600257873535, 1.575796365737915, -0.39778539538383484, 0.8876293897628784, 0.5309446454048157, -1.315782904624939, 0.6601672768592834, -0.5382260084152222, 0.48109564185142517, -1.5689884424209595, 0.7217846512794495, 0.23888446390628815, -1.8612147569656372, -0.797627866268158, 2.5973312854766846, -0.8336045742034912, -0.051213011145591736, 0.06672466546297073, -1.096326470375061, 0.1985519528388977, -0.983857274055481, -0.06037689745426178, 0.786865770816803, 0.3706772029399872, 0.03565073758363724, 1.2512431144714355, -0.5063554644584656, 0.3462298512458801, -0.5954300761222839, 0.37447023391723633, -0.6695901155471802, 0.5545896887779236, 0.7535513639450073, -0.9160977005958557, 1.3228601217269897, 0.2776888906955719, -0.9398419260978699, -0.7194358110427856, -0.5950241088867188, -0.5777406096458435, -1.0006496906280518, -0.0163373164832592, -0.2375013828277588, -0.5956145524978638, -0.6921810507774353, 0.2738717794418335, -0.6170908212661743, 1.6274006366729736, -0.024955153465270996, -0.8333690166473389, -0.8246597051620483, 0.5798742771148682, -1.3656922578811646, -0.40249380469322205, -0.826897382736206, -0.4293234050273895, -1.5196298360824585, -0.9481180310249329, -0.3481050729751587, 0.2758055627346039, 0.8416725397109985, -1.6640104055404663, -1.4077061414718628, -0.1842474341392517, -0.48877108097076416, -0.6472131609916687, -0.8678721785545349, 0.6641104221343994, 0.3814566731452942, 1.0767123699188232, 0.10325852036476135, 0.20469897985458374, 0.1407865732908249, -0.09333109110593796, -0.2661566436290741, -0.058437447994947433, -0.634046196937561, -0.7678694725036621, 1.684941053390503, 0.2343882918357849, -1.4599794149398804, 0.6275821924209595, -1.154751181602478, 0.899971067905426, -0.3164254128932953, 0.23850058019161224, 1.4362399578094482, -1.166699767112732, 0.7414166927337646, -0.12279634177684784, -1.2729718685150146, 2.9812891483306885, -1.1533907651901245, -0.850098729133606, -0.025757385417819023, 0.5829474329948425, -3.7944226264953613, 1.7633877992630005, 0.8489871025085449, 0.43996429443359375, 0.39498162269592285, -0.5486114621162415, -0.6899850964546204, 0.7688783407211304, -0.45123356580734253, -0.4544181227684021, 0.5766834020614624, -0.4416700303554535, 0.8861541152000427, 2.03983211517334, 0.4370651841163635, -0.5031558871269226, -0.49133509397506714, -1.2765086889266968, -0.36933350563049316, -1.7094186544418335, 0.8579885363578796, 0.7169568538665771, -0.2330167144536972, 0.13838045299053192, 0.931460976600647, 1.3830316066741943, -0.11436000466346741, 0.8465486168861389, -0.3584084212779999, 1.2854175567626953, 0.9659548401832581, -1.6090682744979858, 1.337801218032837, -1.6336779594421387, -0.488503634929657, -1.4845094680786133, -0.10311847180128098, -0.4121167063713074, -0.04835507273674011, -0.7323719263076782, -1.003056287765503, -1.5837929248809814, 0.31862252950668335, -2.06197190284729, 0.17648063600063324, 0.0096451910212636, 0.2795375883579254, -1.7985535860061646, -1.267021894454956, 0.08267363160848618, 1.1945114135742188, 1.0670206546783447, 0.6036592125892639, -1.1999940872192383, -0.10214128345251083, -0.39726993441581726, -0.3328896760940552, 0.11079048365354538, 0.047614362090826035, -1.7307953834533691, -0.8424930572509766, 0.9351773858070374, -1.0173883438110352, -1.37283456325531, -1.7473363876342773, -0.03340495005249977, 0.6642440557479858, 0.8758642673492432, 0.8476409316062927, 0.674557089805603, 0.443302720785141, -0.4956234097480774, 0.2212200164794922, 0.33048316836357117, -1.170280933380127, 0.8445426225662231, -0.14482636749744415, -0.007166530471295118, -0.7956534028053284, 1.2447365522384644, -1.3841577768325806, -1.5303858518600464, -0.5488144159317017, 0.30220234394073486, -0.16883429884910583, -0.16895151138305664, 0.5836535096168518, 0.16858258843421936, 0.2951347231864929, 0.777283251285553, -0.886930525302887, -2.4748764038085938, 1.8040724992752075, 1.4135630130767822, 1.3317033052444458, -0.7409866452217102, 0.8686277866363525, 0.7084227800369263, -1.008342981338501, -1.8067915439605713, 0.04752612113952637, -0.1497296243906021, -0.5679606199264526, -0.998943567276001, 1.3781346082687378, -0.7807781100273132, 1.7830132246017456, 0.5026658773422241, 0.8054084777832031, -0.6226465106010437, 1.0510321855545044, -0.6272526979446411, -0.8070787191390991, 0.4234859347343445, 0.08409427851438522, 0.12730105221271515, 0.49696293473243713, -1.2186981439590454, 0.2822941839694977, -0.021107234060764313, -0.2701326310634613, 0.9888033866882324, 0.1645050346851349, 0.18860742449760437, -1.2368357181549072, 0.3775927722454071, -1.656311273574829, -1.126488447189331, 0.361240416765213, -0.2810293436050415, 0.6254405975341797, 1.5552583932876587, -0.20345667004585266, 1.6305769681930542, -1.2208210229873657, -0.3208998441696167, -0.7528799772262573, -0.2020578384399414, -0.698992133140564, -1.3066552877426147, 0.6685124039649963, -0.9177635908126831, -1.2961962223052979, 1.0555827617645264, 0.6850987672805786, -0.21597178280353546, 0.0786205604672432, 0.5687029957771301, -1.0714010000228882, -1.4928350448608398, 0.8760150074958801, 0.3245026171207428, -0.23584529757499695, 0.9026139974594116, -0.6099522113800049, -0.3188681900501251, -1.1031475067138672, 0.5721952319145203, -1.1233993768692017, -0.6255623698234558, 0.40859130024909973, -0.05895410105586052, 1.2896970510482788, 2.2560884952545166, -0.4493771195411682, -2.126695394515991, 0.3321118950843811, 0.6420170664787292, -0.3708750605583191, 0.5979681015014648, 0.8968435525894165, -0.22877861559391022, -1.5704119205474854, 0.0871078297495842, 1.9278241395950317, -0.6718313694000244, -0.38021403551101685, -0.9076168537139893, -1.0664150714874268, 1.3247601985931396, -0.48972249031066895, -0.34135133028030396, 2.1709089279174805, 0.4557734429836273, 0.5087750554084778, 0.4284771978855133, -0.469820499420166, 1.7784948348999023, -0.035142675042152405, 0.33407947421073914, -0.30001676082611084, 0.025367897003889084, -1.1760759353637695, 0.792000412940979, -0.19089028239250183, -0.8349871039390564, 0.4189898371696472, 1.2498505115509033, -0.5290780067443848, 1.2367544174194336, -1.2632874250411987, -0.37413185834884644, 0.4123278558254242, -0.011754113249480724, 0.14742602407932281, -0.7858810424804688, 0.12144670635461807, 2.0724122524261475, 2.551476240158081, -0.1734338253736496, 0.42986729741096497, 0.43996676802635193, -1.8643203973770142, 0.30127155780792236, 1.1082595586776733, 3.197904348373413, -0.9009379148483276, -0.325775682926178, 0.7214711308479309, -0.4463963508605957, -1.6001569032669067, 0.37911897897720337, 1.1640545129776, 1.6199086904525757, 2.687828302383423, -1.1439334154129028, 1.6551861763000488, 0.08695321530103683, -0.9571842551231384, -2.2296459674835205, -0.9692727327346802, 0.29110684990882874, -2.097425699234009, -0.07890518009662628, 0.44954803586006165, 1.516210675239563, 0.3248196840286255, -0.7145950794219971, 0.868979275226593, 0.2031235247850418, 1.016727089881897, 1.8507837057113647, 0.002334489254280925, -0.3232501149177551, -0.43945643305778503, 1.168867826461792, 0.5185495018959045, -0.9533538818359375, -0.02606034465134144, 0.07438316941261292, 1.4171109199523926, -0.08743153512477875, 1.4128530025482178, -1.4897150993347168, 0.5791471600532532, 1.8998560905456543, -0.054480232298374176, -0.9211041927337646, -0.46664851903915405, 1.3413050174713135, -1.3022935390472412, 0.34380003809928894, 1.1228675842285156, -1.1451020240783691, 0.19797232747077942, -0.15911681950092316, -1.0728528499603271, 0.3113774061203003, -0.28476470708847046, 1.8190357685089111, 1.2273274660110474, -0.5268229246139526, 0.4305163621902466, -0.7459537386894226, -2.6222636699676514, -0.8660075068473816, -0.30475154519081116, -1.1732544898986816, 1.0158854722976685, -1.0750941038131714, -0.17274659872055054, 0.03044130653142929, 1.105339527130127, 0.09632968157529831, 1.1221035718917847, -0.06695768982172012, -0.25026172399520874, -0.765031099319458, 1.3831294775009155, -0.23012661933898926, 1.044235110282898, 0.18569964170455933, 0.9796206951141357, -1.9182963371276855, -0.10487812012434006, 0.7862566709518433, 0.7240534424781799, -0.06049642711877823, -0.5070152878761292, -1.4225062131881714, -0.7959156036376953, 0.22809265553951263, 0.17948435246944427, 0.15459978580474854, -0.7385360598564148, 1.2117908000946045, -0.2836053669452667, -0.5607593655586243, 0.10994230210781097, -0.5571310520172119, 0.5480229258537292, -0.2714548707008362, 1.3738741874694824, 0.03846165910363197, 0.5540378093719482, -0.551072359085083, -3.0866165161132812, 1.161429524421692, 0.7703853845596313, -0.04211035743355751, 0.6725979447364807, -0.17364384233951569, -0.8096845746040344, -1.9552464485168457, -0.9887098670005798, -1.0977957248687744, -0.7313250303268433, 1.718595027923584, 0.34239134192466736, -0.2560789883136749, -0.6106307506561279, 0.1521303653717041, 0.24390801787376404, 1.0544770956039429, 1.3010987043380737, -0.3632178008556366, -1.4923094511032104, -0.27435165643692017, -1.2407560348510742, 0.28243276476860046, -0.40488123893737793, 1.4833861589431763, -0.6142852306365967, 0.33265724778175354, -0.860852062702179, 0.7307981848716736, -0.8022670149803162, -0.776266872882843, 0.21771976351737976, -0.5256346464157104, -0.06177828460931778, -0.9255936741828918, 1.1261957883834839, 0.2389049232006073, -0.3781604468822479, -0.15967732667922974, -1.3766546249389648, -0.047353655099868774, -0.803642988204956, -0.18718378245830536, 0.7513271570205688, -0.43283626437187195, -2.0348048210144043, -0.23369009792804718, 2.1316285133361816, -1.2464501857757568, -0.8294625878334045, 0.033124372363090515, -2.3606183528900146, -0.0879860445857048, 0.22882087528705597, -1.1732814311981201, -2.1457643508911133, 0.8414684534072876, 1.6807053089141846, 1.6628583669662476, -1.115893840789795, -0.5622471570968628, -0.9620209336280823, -0.39578837156295776, -0.7960571646690369, -0.06991729885339737, -0.26965048909187317, 0.8732641339302063, -0.1575460582971573, -1.2500395774841309, 1.619228720664978, -0.4132068157196045, 0.17499220371246338, 0.25584834814071655, -0.1325601041316986, 0.9556158781051636, 1.4873168468475342, 0.16240938007831573, 0.5720955729484558, -0.46996206045150757, 0.07450272142887115, 1.4197676181793213, 1.505565881729126, -0.6471993327140808, 1.1592463254928589, -1.188189148902893, 0.4667620360851288, -0.04941230267286301, 0.9814906716346741, -0.9139475226402283, -0.49833616614341736, -1.7351322174072266, -0.2226780652999878, -2.3373706340789795, 0.3705458343029022, -0.5554243922233582, -1.023760437965393, 0.8546825647354126, -1.8805391788482666, 1.7905975580215454, 0.40143629908561707, -1.4923396110534668, 0.25275281071662903, 1.0891661643981934, -0.5793376564979553, -0.10601259022951126, -0.8764762282371521, -1.2543463706970215, 1.4058961868286133, 0.11174185574054718, 0.2725007236003876, -0.49280229210853577, -1.2480120658874512, 0.46377235651016235, -2.1783125400543213, 0.7793110013008118, 0.9222081899642944, -0.4752791225910187, -0.30585330724716187, -0.9902015924453735, 0.4054487943649292, 0.8317305445671082, -0.8299938440322876, 1.8940736055374146, 0.10090448707342148, 1.1764936447143555, 0.8043181896209717, -0.3240196108818054, -0.5394975543022156, 0.16856831312179565, 0.14222201704978943, 0.6544206142425537, -0.22765882313251495, 0.5420756936073303, -2.1446621417999268, 0.9596090912818909, 0.6066141724586487, -2.0236353874206543, 0.9594310522079468, -0.06798344850540161, 0.11340395361185074, -0.6226363182067871, 0.36395323276519775, 0.06459680944681168, -0.7277864813804626, -0.21716810762882233, -1.5727490186691284, 1.1215486526489258, 0.8412861227989197, 0.3222615122795105, 0.08033942431211472, 0.9955060482025146, -0.9597347378730774, -1.1458240747451782, 0.20670802891254425, 0.03039480559527874, -0.05953873693943024, -0.7446531653404236, -0.8720179200172424, 0.24185200035572052, 0.2048492729663849, -0.3330646753311157, 0.5038803815841675, 0.7503108978271484, 0.9903122782707214, 0.32491207122802734, 0.32314708828926086, 0.9842003583908081, -0.523414671421051, -1.2891740798950195, 0.028806934133172035, 1.6589430570602417, -0.906388521194458, 0.41553863883018494, 1.73654043674469, 0.3356914222240448, 1.1618882417678833, -0.005540719721466303, 0.9624332189559937, -1.2036144733428955, -1.4228594303131104, -1.2851296663284302, -0.1595379114151001, 1.0400817394256592, 0.11018065363168716, 1.3292372226715088, -0.17157669365406036, 0.38374659419059753, 0.8303514719009399, -1.2727622985839844, -0.539675772190094, -1.7019017934799194, 0.68903648853302, -1.5140730142593384, -1.63364577293396, 0.9688317179679871, 0.45790359377861023, 1.7595223188400269, -0.6811848282814026, 0.5668485760688782, -0.6143385767936707, 0.16224338114261627, 1.153275728225708, 1.000313639640808, -1.141226053237915, 1.6761581897735596, 0.404023140668869, -0.6288505792617798, 0.14429491758346558, -1.9001586437225342, 0.9675353765487671, 0.22404757142066956, -2.399742364883423, -1.353386402130127, 0.875572919845581, -0.5032622814178467, -0.8994720578193665, -1.252524733543396, 0.14769324660301208, -1.8828437328338623, 1.595081090927124, -0.95643550157547, 0.4631859362125397, 0.7612426280975342, 0.3500112295150757, 1.6282799243927002, -0.042676977813243866, -1.0243884325027466, 0.46954071521759033, 0.7956840395927429, -0.6021789312362671, 1.0618417263031006, 0.47285959124565125, -1.0910412073135376, -1.104131817817688, -0.3969564735889435, 1.8137314319610596, 0.853053629398346, -1.765403151512146, 0.32107844948768616, -0.6492933034896851, -1.0516860485076904, -0.273281455039978, 0.21795673668384552, -1.400160551071167, 1.0420717000961304, 0.3542834222316742, -0.8213739395141602, -0.2679580748081207, -3.1389193534851074, 0.10305869579315186, -0.7889825105667114, 1.697312355041504, 0.7018701434135437, -1.9436112642288208, -1.0608233213424683, 0.1463668793439865, -0.046492159366607666, -0.6026636362075806, -1.1474477052688599, 0.2729632258415222, -1.2387579679489136, 1.8432879447937012, -0.669000506401062, 1.9230982065200806, 0.989920437335968, -0.21344208717346191, 1.8585081100463867, -2.423548460006714, -2.654188394546509, 1.31077241897583, -0.48588883876800537, 0.6221032738685608, -1.096259593963623, 0.18387602269649506, -2.4788899421691895, -1.8322077989578247, -1.0381871461868286, 0.05383856222033501, -0.6713812351226807, 0.2667926251888275, -0.48005133867263794, -0.014625942334532738, -0.06719912588596344, 0.6316872239112854, 2.1195285320281982, 0.5258735418319702, -0.5495198369026184, -1.5668816566467285, 1.5717518329620361, -0.49851885437965393, -1.026928186416626, 0.12993231415748596, -0.37138932943344116, -1.6985071897506714, 0.6334469318389893, -0.7400617599487305, 0.039844974875450134, -0.6445806622505188, -1.9013996124267578, -0.9179443717002869, -1.0846307277679443, -0.11117078363895416, -2.4389357566833496, -0.25413748621940613, -0.007204891182482243, 0.7387577891349792, 1.0380380153656006, -0.7667905688285828, 1.8599259853363037, 1.1844881772994995, 0.45877885818481445, -0.4847393333911896, 0.07775646448135376, -0.6702151298522949, -0.643042802810669, -0.10727307945489883, 0.4677404761314392, -0.4281383752822876, -0.33966484665870667, -0.196663498878479, -0.5634183883666992, -1.6569067239761353, 0.5504682660102844, 1.146843671798706, 0.10374507308006287, -2.343613862991333, -0.559658944606781, 0.21632365882396698, 0.13025875389575958, 0.0035224270541220903, -1.3474950790405273, -0.43048208951950073, 1.3828887939453125, 0.5869218707084656, -0.5576593279838562, 1.1358941793441772, -0.5546392798423767, 0.7641734480857849, 0.21411488950252533, 1.2982257604599, 0.2552018463611603, -0.2894296944141388, 0.48048102855682373, -1.495998501777649, 0.35509881377220154, 0.5901119709014893, -0.34825146198272705, -1.8680169582366943, -0.3371405601501465, -0.8612637519836426, 1.3219064474105835, -1.1388319730758667, -1.041885495185852, 0.7943206429481506, -0.9420984387397766, 0.16997963190078735, -0.15329095721244812, -0.14876961708068848, -0.19712859392166138, 2.4560863971710205, -2.3085920810699463, -0.46156078577041626, -1.3421944379806519, 1.6559226512908936, 1.3255914449691772, -0.22958678007125854, 0.7815673351287842, -1.5434460639953613, 0.33887821435928345, 0.40293586254119873, 0.7000869512557983, -2.3697845935821533, -1.5770295858383179, 1.6932315826416016, 0.030316337943077087, -0.22510220110416412, -1.6768836975097656, 0.45399755239486694, -1.7993896007537842, -0.03476728871464729, 0.2619670331478119, 0.87024986743927, -1.577562689781189, 1.181525468826294, -0.762251079082489, 2.1033456325531006, 1.8825278282165527, 0.9997063279151917, 0.21787774562835693, 0.4457626938819885, -1.362680435180664, 0.004830622114241123, -0.47534677386283875, -0.03094196878373623, 0.9451786279678345, -0.5357053279876709, -0.6400630474090576, 0.6642297506332397, 1.5472080707550049, 0.8754765391349792, 0.9854310750961304, -1.099224328994751, -0.22089430689811707, 0.07898270338773727, 0.767237663269043, -0.3535042107105255, -0.5549876093864441, 0.6726216077804565, 0.6429962515830994, -1.7059308290481567, 1.5165722370147705, -0.43051236867904663, -0.7524937391281128, -1.469383716583252, -0.43738555908203125, 0.23162519931793213, 2.2595338821411133, -1.0452773571014404, -1.0372587442398071, -0.0838252380490303, -0.13528907299041748, 0.2540318965911865, 1.3146412372589111, 1.0379780530929565, 1.2410221099853516, 0.4215065836906433, -0.609724760055542, 0.14866212010383606, 1.236698865890503, 0.2505209445953369, 2.242295026779175, 0.16763542592525482, 1.5595173835754395, -1.470260739326477, -1.5341708660125732, 0.20129796862602234, -0.6010292768478394, 1.1127995252609253, 0.5500422120094299, -0.8016679883003235, -2.0018298625946045, -0.08290205150842667, -0.19971320033073425, 1.1111719608306885, -1.4307222366333008, 1.0191066265106201, -1.5830644369125366, 0.38932541012763977, -1.1950160264968872, 0.7455930113792419, 0.316665381193161, 1.0774592161178589, 2.305178642272949, 0.4398992359638214, -1.0337234735488892, -1.2021174430847168, 2.831045627593994, 0.6308148503303528, 0.043931297957897186, 0.02970900945365429, -0.7269602417945862, 0.7251588106155396, -0.05924816429615021, -0.062075454741716385, 0.06279642134904861, -1.3236713409423828, -1.3787299394607544, -0.09621556103229523, 0.5302308201789856, 1.1257143020629883, 0.6766878962516785, 0.20106174051761627, -0.021910669282078743, -1.3954987525939941, -0.4086059331893921, 1.1674833297729492, 0.8164723515510559, -0.1628345400094986, 0.6593962907791138, 1.5768502950668335, 1.0229114294052124, 0.8562052249908447, 1.3143446445465088, -1.7205610275268555, -1.0271352529525757, 1.7525110244750977, 0.24962246417999268, -0.560359001159668, -1.2299175262451172, -0.3284405469894409, -1.2102233171463013, -1.5388082265853882, -0.8406538963317871, -0.2428816258907318, 0.8680256605148315, -0.6947315335273743, 0.6523247957229614, -0.39340364933013916, 1.4735751152038574, -1.0420196056365967, -1.9338550567626953, 0.17760048806667328, 0.648885190486908, -1.7532414197921753, -0.7832475900650024, 0.14464154839515686, -0.7752061486244202, -0.33633953332901, -0.2936093509197235, -0.4717857539653778, -0.2048490345478058, -0.0944657102227211, 0.5061926245689392, 0.7632317543029785, -0.2619304060935974, 0.6135410070419312, -0.9180755615234375, -1.1835991144180298, 0.9821645021438599, -0.5959476828575134, 0.18383531272411346, -0.3971395194530487, 0.03210902959108353, 0.38216885924339294, -1.3029173612594604, -0.4421115517616272, 0.22592651844024658, 0.21438054740428925, -0.028446178883314133, -1.2490167617797852, 2.7364110946655273, 0.19805705547332764, -1.2659863233566284, -1.562360167503357, 1.250580906867981, 0.34967857599258423, 0.9334769248962402, -0.8159840703010559, -1.6416163444519043, -0.4549385905265808, 0.547522246837616, -0.22728592157363892, -2.043900966644287, 0.4222063720226288, -2.220477819442749, 0.3283819258213043, -1.0776346921920776, -0.5205918550491333, -0.6494491100311279, 0.17636996507644653, -0.0022164920810610056, 1.7100461721420288, -0.9457860589027405, -1.29752516746521, -0.5925851464271545, 0.8694393038749695, 0.13331827521324158, -0.6476046442985535, -1.279760718345642, 1.1610548496246338, 0.41520965099334717, -0.5881569385528564, 0.22524835169315338, 0.5122007131576538, -1.8150006532669067, -0.2364436686038971, -0.29447460174560547, 1.5823744535446167, -1.7969828844070435, 0.8429349660873413, -0.22913917899131775, 0.5979018807411194, -0.8718799948692322, -0.6921690702438354, 0.1391909122467041, -0.3775968551635742, 0.564134955406189, -0.40803369879722595, -0.4674675464630127, -0.22915273904800415, -0.3990382254123688, 1.0069332122802734, -0.09360328316688538, -0.27490749955177307, 0.6982577443122864, -0.6888195872306824, 0.5929893851280212, -1.5284086465835571, 1.3578163385391235, -0.05459217354655266, -0.2903674840927124, -1.4454957246780396, -0.7209067940711975, 0.4834951162338257, -0.5577667951583862, -1.4292378425598145, -0.6183249950408936, 1.7571734189987183, 0.49785661697387695, -0.2558410167694092, 0.01082731131464243, -0.39416804909706116, -0.863278329372406, -1.269208550453186, 0.992713212966919, -0.5831929445266724, -2.8662543296813965, 0.47180628776550293, 1.233803153038025, -1.2692519426345825, -0.6635940670967102, 1.2760189771652222, -0.2910844087600708, -1.3515985012054443, 0.5539276599884033, 0.05924778804183006, -0.9884824752807617, 0.7289031744003296, 0.0516231507062912, 0.7386929392814636, -1.3217025995254517, -0.23916593194007874, 0.18451808393001556, -0.2304418534040451, -0.013613715767860413, -0.34170398116111755, -2.136178970336914, -1.1239444017410278, 0.7177063822746277, 0.7053427696228027, 0.4236621856689453, 0.16963855922222137, 0.1145491749048233, 1.0516390800476074, -0.3829672932624817, -0.47131314873695374, -1.3970662355422974, 0.43467795848846436, 0.19318802654743195, -1.681217908859253, -2.6944291591644287, -0.597962498664856, 0.20269057154655457, -0.4440463185310364, 0.06542913615703583, 1.886602520942688, -1.3796212673187256, 0.16930098831653595, 0.5425567626953125, 1.6115690469741821, 1.1724711656570435, -1.363215684890747, 0.3282381594181061, 0.09987986832857132, -0.8030944466590881, 1.3238128423690796, -0.3120640218257904, 1.423499584197998, 0.730949878692627, 0.7520285248756409, 0.05134545639157295, -1.566848635673523, -2.191152811050415, -0.7301643490791321, -1.2005032300949097, -1.0925400257110596, -0.08199921995401382, -0.29960429668426514, -0.3729719817638397, -0.020225072279572487, -1.1346147060394287, -0.7799244523048401, 1.8082911968231201, 0.19059857726097107, -0.4779113531112671, -2.244601011276245, -1.6435515880584717, 0.1603870391845703, -0.3365895748138428, -1.093243956565857, -1.5371956825256348, 0.8526278734207153, 1.4034950733184814, -1.861631989479065, 1.704241156578064, -0.6602895259857178, 0.6270322203636169, 0.5978399515151978, 1.2445355653762817, -0.11464060097932816, 0.3690667152404785, -0.1737947314977646, -0.20688585937023163, -1.6664832830429077, 0.2408204972743988, 0.41765207052230835, -1.243653655052185, 0.30168941617012024, 0.6072987914085388, -1.4552655220031738, -0.07847652584314346, -0.5885134935379028, -1.308223843574524, -0.18949931859970093, -0.6233811378479004, 1.1867868900299072, -0.27396708726882935, -0.22134318947792053, 0.5975726842880249, -0.5432886481285095, 1.2907830476760864, -1.3015440702438354, -0.2485620379447937, -0.4068771302700043, -0.06104392930865288, 0.4195120930671692, -0.7307115197181702, -1.9677274227142334, -1.380341649055481, 0.6432017087936401, 1.323744773864746, -1.6502583026885986, 0.44442981481552124, 0.33453789353370667, -0.2591371238231659, 0.8899090886116028, 0.8898811936378479, -2.1749629974365234, -0.7463939189910889, -0.45093443989753723, -0.16800612211227417, -0.547187089920044, 1.0403622388839722, 0.5374826192855835, 0.7185066938400269, -0.625697672367096, -1.2232205867767334, -0.4307800531387329, 0.48842287063598633, 0.5856834650039673, -1.2550283670425415, -0.17330728471279144, -1.4554920196533203, 0.30321478843688965, 1.308882713317871, -0.10849756747484207, 0.5501885414123535, -1.3842161893844604, 0.23352085053920746, -0.814225435256958, -0.4403402805328369, -0.6052705645561218, -0.40945976972579956, 0.9085705280303955, 0.932036280632019, -0.7292715311050415, -0.8588553667068481, 0.7174672484397888, 0.49449217319488525, -0.3976857364177704, 1.5557140111923218, -0.7982410192489624, 0.6502204537391663, -1.3855777978897095, 0.7818261981010437, -0.1744219809770584, 2.0548102855682373, 2.100559711456299, 0.5201764702796936, -0.5522904396057129, 0.8122435212135315, 0.8598788976669312, 0.789975106716156, -0.05424630269408226, -0.7504487037658691, -0.41483983397483826, 0.14694428443908691, -1.0010160207748413, 0.10465912520885468, 1.3123611211776733, 1.3416253328323364, 0.3044762909412384, 0.17215514183044434, -0.2861449122428894, -0.05426204577088356, 0.48649320006370544, -0.20456938445568085, 2.449427604675293, 0.7203653454780579, 1.368497371673584, 0.27052006125450134, -0.4489458501338959, 1.3799363374710083, 0.7270431518554688, -0.007452195044606924, -0.37529516220092773, 0.45911359786987305, 1.0238136053085327, -1.5699872970581055, -1.0183824300765991, 1.7961591482162476, 0.268297404050827, 0.8205337524414062, -0.8790519833564758, -0.07982601970434189, 0.6979485154151917, -0.7743081450462341, 0.546502947807312, -1.2027398347854614, 1.3821195363998413, -0.2816942036151886, -1.2012842893600464, -1.513382077217102, 0.262727826833725, -0.5944459438323975, -0.3206643760204315, 0.18603862822055817, 0.805890679359436, -0.05330537259578705, -1.279148817062378, -1.471376657485962, 0.7883526086807251, 0.1874501258134842, 0.845310628414154, -0.1599012017250061, 0.5196004509925842, -0.8091482520103455, -0.3556033968925476, 0.24206896126270294, -0.5997979640960693, 0.47462621331214905, -0.021725138649344444, -0.6012288928031921, -0.41723936796188354, -0.23014137148857117, 1.1688129901885986, -1.620337724685669, -0.3764127194881439, 2.6072938442230225, -1.0823605060577393, 0.49172699451446533, -0.5467795133590698, -0.8604573011398315, -0.4545491337776184, 1.6376532316207886, 0.3576779067516327, 1.0111194849014282, -1.9331183433532715, -0.7256413698196411, 1.124050259590149, 0.9538652896881104, 3.376432418823242, 0.9331558346748352, 0.1595238894224167, 0.14845983684062958, 1.5052363872528076, -0.3625411093235016, -0.8068609237670898, -0.6251687407493591, -0.5562476515769958, -0.02540922723710537, -0.5980737805366516, 1.385957956314087, -0.7916578650474548, 0.6455883383750916, -1.52910315990448, 0.2051655650138855, 0.33041343092918396, 0.12742920219898224, 1.2663335800170898, -0.592810332775116, 2.4026503562927246, 0.6840559244155884, -0.5809808373451233, -0.16160021722316742, 0.6050001382827759, 1.0267517566680908, -0.29531753063201904, 0.4263863265514374, -0.9231647849082947, -0.3666803538799286, 0.2149129956960678, 0.8097002506256104, 0.40347689390182495, 0.5408739447593689, 0.21854470670223236, 0.5544580221176147, 0.8917113542556763, 0.7702645659446716, 2.4420769214630127, 0.6015951633453369, 0.11663239449262619, -0.19395127892494202, 0.4507851004600525, -0.2783085107803345, 0.7775139212608337, -1.6373167037963867, -0.4039947986602783, 0.9181326031684875, -0.8913072347640991, -0.12034940719604492, -0.33747541904449463, -0.1558723896741867, 1.4596632719039917, -0.46855154633522034, -0.8633149862289429, -0.5215455293655396, 0.8936676383018494, 1.0507365465164185, -0.01928073726594448, 0.8492571115493774, 0.21178695559501648, 0.13492201268672943, -1.2885338068008423, -0.8048798441886902, 0.009537232108414173, 1.0338860750198364, 3.1083571910858154, 1.2877507209777832, 0.6269570589065552, -0.1678454428911209, 1.8117172718048096, -0.5106620192527771, 0.9229071140289307, 2.129190444946289, -0.5881057977676392, -0.20633608102798462, -0.8122332692146301, 1.4140068292617798, 1.5077745914459229, -1.2040444612503052, 1.8158576488494873, 0.28038352727890015, 0.8102355599403381, 1.1530826091766357, -0.5195867419242859, 0.24283288419246674, 0.5449921488761902, -1.4175506830215454, -0.9825106859207153, 1.2644964456558228, 0.08724909275770187, -1.379477620124817, 0.45611822605133057, -0.05837836116552353, -0.8335133194923401, -0.3260134756565094, -0.2894270122051239, 0.8191049695014954, -1.3513315916061401, 1.0908033847808838, 0.7162824273109436, -0.4113309681415558, 1.5194709300994873, -0.13795919716358185, -0.1488492488861084, -1.2928487062454224, 0.27904096245765686, -0.49659591913223267, -0.5075215697288513, 0.840959370136261, -1.215452790260315, 1.5822772979736328, -2.063713550567627, 0.8647934198379517, -0.8772646188735962, -0.9153450727462769, 0.6404505372047424, -0.1915026754140854, -0.31815171241760254, 1.8572481870651245, 2.1846988201141357, -1.6062506437301636, -0.11404140293598175, -0.5449056625366211, -0.0013532837620005012, 0.9666310548782349, 0.024948062375187874, -0.42294442653656006, 1.903124213218689, -1.508540391921997, 0.05493839830160141, 0.4836999475955963, -0.9432599544525146, 0.3967190384864807, 0.20518046617507935, -0.19119587540626526, -0.15101982653141022, 0.7205966711044312, -0.2199651449918747, -0.12525148689746857, 1.1772146224975586, -2.167976140975952, 0.6233932971954346, 0.11809016764163971, 0.41106563806533813, 1.5424408912658691, 0.7820284962654114, 1.0611729621887207, 0.29553547501564026, -1.0717883110046387, 0.41457122564315796, -1.2977343797683716, -0.4570452868938446, -0.5102881789207458, -0.013546240516006947, -0.6968741416931152, -1.6706790924072266, 3.3569514751434326, -1.1970561742782593, -1.2733813524246216, -0.1353970170021057, -1.6197376251220703, 1.5176211595535278, 0.49791285395622253, -1.758705735206604, 2.4905989170074463, -0.3368113934993744, 0.19626881182193756, -1.1725918054580688, -1.155391812324524, -0.13751041889190674, 0.026553096249699593, -0.09254160523414612, -2.1679880619049072, -0.19643813371658325, -2.047572374343872, 1.3712372779846191, 0.07769545912742615, 0.8421067595481873, 0.09037923812866211, -0.7553800940513611, -0.21821144223213196, 0.1533316671848297, 0.9019049406051636, -2.8118951320648193, -1.2819160223007202, -1.275933861732483, -0.5460562705993652, 0.2110040783882141, -0.9532679915428162, -0.7993214726448059, -1.9249026775360107, -1.1193045377731323, -0.028145281597971916, -0.13531392812728882, -1.9115257263183594, 1.014007329940796, 0.3883984684944153, -0.08428196609020233, 0.39971745014190674, -1.142259120941162, 0.6136811971664429, 1.3047760725021362, -0.21051374077796936, -2.1620452404022217, 0.38800010085105896, 0.6162688732147217, -1.1942189931869507, 1.0853222608566284, 0.5956934690475464, -1.141098141670227, -1.5213639736175537, 0.38300618529319763, -1.3962619304656982, -1.1828999519348145, -0.022318650037050247, -1.613997220993042, -0.08841254562139511, -0.5239253640174866, -0.527198851108551, 0.2134934514760971, -0.6808930039405823, 1.0660831928253174, 1.0423468351364136, -0.7250446081161499, 2.3673527240753174, -1.774661660194397, 1.3719040155410767, -0.21351024508476257, 0.9610350131988525, -0.6997717022895813, 0.12205430120229721, 0.3057348430156708, -0.7421322464942932, -0.2692473232746124, 2.7570998668670654, -0.11154425144195557, -0.864495575428009, 1.2400496006011963, 0.48087358474731445, 1.2371883392333984, -0.7242165803909302, 3.194159507751465, 1.4539004564285278, -0.15519516170024872, -0.36276373267173767, -0.7669894099235535, 0.2674734890460968, 0.7617855668067932, 1.0310218334197998, -1.42822265625, -1.5972850322723389, 0.9290131330490112, -1.193498969078064, 0.2735033631324768, -1.8744268417358398, 2.159066915512085, 0.5373088121414185, 1.2540041208267212, 1.5725241899490356, -0.36688748002052307, 1.1562104225158691, 1.1843172311782837, 0.8004951477050781, -0.5874251127243042, -1.5251305103302002, -1.2902997732162476, 0.40938425064086914, 0.8738314509391785, 1.1763596534729004, 0.8972508311271667, -0.968364417552948, -1.2909929752349854, 0.07626702636480331, -0.2331438958644867, -0.3614538609981537, -0.955992579460144, -0.6810117959976196, 2.063041925430298, -0.5229454040527344, 0.4830608069896698, 1.7825146913528442, 0.6468439102172852, -0.40576112270355225, 0.9079814553260803, 0.022548293694853783, 0.8386284112930298, 0.18746228516101837, 0.4803345501422882, 2.2842726707458496, -0.6995819807052612, -0.15018680691719055, -0.3834538459777832, 0.808616042137146, 0.6458708047866821, -0.6521511077880859, 0.9640724062919617, 1.1007908582687378, -0.0037751735653728247, 1.4894461631774902, 1.4194012880325317, -1.7974663972854614, 0.42647069692611694, 0.8804332613945007, 0.06095525994896889, -0.16263365745544434, 0.7607181072235107, -0.15306425094604492, 1.1121476888656616, 0.9774795174598694, 1.8383386135101318, -0.6767627000808716, -0.498939573764801, -0.7598001956939697, 0.48018407821655273, -0.6203843355178833, -0.6582464575767517, 0.26965606212615967, -0.12533943355083466, -0.05876980349421501, -0.6939001083374023, -1.0668549537658691, -0.2932763397693634, 0.2941167950630188, 0.21586523950099945, 0.28589147329330444, 1.31560480594635, 1.4504812955856323, -1.3646430969238281, -0.2554827928543091, -0.5892376899719238, 0.1949959695339203, 1.4525514841079712, 0.17578501999378204, 0.6544548273086548, -0.15923789143562317, -1.4354093074798584, -1.4104005098342896, 1.7366653680801392, 0.08173253387212753, 1.957592487335205, -0.5004917979240417, 0.41997188329696655, -1.213819980621338, -0.03155052661895752, -0.05493278428912163, -1.5430560111999512, -1.7274962663650513, -1.2388949394226074, -1.5732754468917847, 0.49126923084259033, -2.0960869789123535, 1.6225135326385498, -0.3299266993999481, -1.0811702013015747, 0.28582462668418884, 0.2382708340883255, -1.2607067823410034, -0.5328903794288635, -0.9386651515960693, 0.40891748666763306, -0.05259363353252411, 0.6027124524116516, -0.7199362516403198, 1.5934056043624878, -2.1565351486206055, -1.161582112312317, -2.132992744445801, 1.044419765472412, -0.8929501175880432, -0.3836515545845032, -0.8633022308349609, -0.8396087288856506, -1.08748197555542, 0.03807051107287407, -0.9298962354660034, 0.9027237892150879, -0.3747835159301758, 0.36755919456481934, 0.07633085548877716, 1.0197944641113281, 0.97752445936203, -0.5517393946647644, 0.5213600993156433, 1.0918465852737427, 0.5113954544067383, 0.33823642134666443, 0.3552277386188507, -1.610019326210022, -0.6518890261650085, -1.5088152885437012, 0.7943194508552551, 0.061756525188684464, -1.1528042554855347, 0.8022552132606506, 0.8694336414337158, 0.4946078658103943, -1.2675741910934448, -0.38301682472229004, -1.303725004196167, 0.8074898719787598, -0.2502700090408325, -0.27576199173927307, -1.073332667350769, 1.1732321977615356, -0.8269981741905212, -0.45914459228515625, -0.8657863736152649, -1.3097163438796997, -0.3358364701271057, 0.39748767018318176, -1.0387614965438843, 1.6602598428726196, -0.4099014103412628, -0.19733326137065887, -1.1997700929641724, 0.6070305109024048, 0.873149573802948, -0.5645008683204651, 2.2782320976257324, -1.0495058298110962, -1.0369750261306763, -1.3614636659622192, -0.7032806873321533, 0.01650257222354412, -1.658797025680542, -1.3988841772079468, 0.5455384850502014, -0.6881005167961121, -0.6207169890403748, 0.3802935779094696, 2.241656541824341, -1.6661255359649658, -2.091381788253784, 0.18331211805343628, 0.5371713042259216, 0.39908578991889954, -0.16818474233150482, 2.4464972019195557, -0.38835155963897705, 0.6095229387283325, -0.5240916013717651, -2.6129567623138428, -0.5537886023521423, 0.08933676034212112, -1.141000747680664, 1.105680227279663, 0.7243408560752869, -1.489255428314209, -1.2027508020401, -0.0821545273065567, 1.1359494924545288, -0.1915341466665268, -0.8092155456542969, 1.1507387161254883, 1.7650501728057861, 0.3975355327129364, -0.5110238790512085, -1.1749517917633057, -0.5026739835739136, -0.5959554314613342, 0.031842488795518875, 1.3992886543273926, 0.501284658908844, 0.8229036331176758, -0.7154940962791443, -0.5159530639648438, -1.1742995977401733, -0.956105649471283, 1.0100910663604736, 0.06876453012228012, -0.15412873029708862, -0.49954766035079956, 0.31434640288352966, -0.28664451837539673, 1.340874433517456, 1.4931979179382324, 1.6754316091537476, -0.8858068585395813, 0.6875597834587097, -0.6740232706069946, 0.5948317646980286, 0.22936341166496277, 0.3950519859790802, 0.3276404142379761, 1.965248465538025, 1.0568339824676514, 0.6719807386398315, 0.5425736308097839, 0.2974267601966858, -1.770216941833496, -1.0744764804840088, -0.4245595932006836, -0.01889418065547943, -0.7505145072937012, 2.171992540359497, 0.07201378047466278, -0.12258534133434296, -0.6061251163482666, 0.5727983117103577, -0.41563376784324646, 0.2643890082836151, -0.6847941875457764, 1.2832293510437012, -0.5664012432098389, -1.368471622467041, -0.7523853778839111, -0.9523431062698364, -0.5046674013137817, -0.1516198068857193, -1.8695563077926636, -0.2543439567089081, -0.5791131854057312, 0.025157760828733444, -0.8699747920036316, -0.13936638832092285, -1.630661129951477, -0.17945048213005066, 0.5898317098617554, 0.28897419571876526, -0.19015845656394958, 1.295250654220581, 1.489617943763733, -0.6360417008399963, 0.43426036834716797, 1.5374191999435425, 0.70708829164505, -0.35834717750549316, -0.28657954931259155, 0.789916455745697, 0.824999988079071, 1.3103798627853394, -0.10728678852319717, 1.5166324377059937, 1.4571571350097656, -2.31125807762146, 0.41595956683158875, -1.659554362297058, 1.6751195192337036, 0.060974955558776855, 0.7931135892868042, -0.35105058550834656, 0.6612608432769775, 2.2412166595458984, -1.6061820983886719, -0.835183322429657, -0.5387880206108093, -0.6734509468078613, 1.115464210510254, -0.11235317587852478, 0.6614170670509338, 0.7240951657295227, -0.12723016738891602, -1.220156192779541, 0.031966276466846466, 0.8036409020423889, -1.7932065725326538, 0.8814250230789185, -0.9032943248748779, 0.5699069499969482, 1.080725908279419, 0.055678997188806534, -1.3242324590682983, -1.2359882593154907, -0.40320879220962524, 0.0738682895898819, 0.4335245192050934, 0.4778122305870056, -0.5810698866844177, -0.24741005897521973, 1.6381953954696655, -0.760821521282196, 0.37546396255493164, 0.03628809005022049, -0.10806531459093094, -0.17109376192092896, 0.8420746326446533, 0.3262845575809479, 0.9092658758163452, 0.23270869255065918, -0.23200823366641998, -0.4298999607563019, 0.6889379620552063, -0.32130166888237, -0.6222106218338013, 0.3609611690044403, 0.7812981009483337, -0.95127272605896, -0.015563364140689373, -0.3483598232269287, -0.3817059099674225, 0.40063512325286865, 0.2637830376625061, -3.0349957942962646, 0.20479816198349, 1.6268128156661987, -0.13047048449516296, 0.15186798572540283, -0.525765597820282, -1.3858647346496582, -0.3180970251560211, -1.8287556171417236, -0.3009336292743683, -1.4478703737258911, 1.3795236349105835, -0.7404286861419678, 1.5367802381515503, 0.21893228590488434, -0.36224502325057983, -0.047270890325307846, -1.58365797996521, -0.8282821178436279, 0.23007124662399292, 0.3876848518848419, -2.660888910293579, -0.16397108137607574, 0.18476273119449615, 0.8317694664001465, 0.7638944387435913, 1.0464048385620117, 0.8198708891868591, 0.4845734238624573, 0.5918264389038086, -0.8937861919403076, -0.1792282611131668, -0.10124773532152176, -1.275429368019104, -1.1362903118133545, -1.0856679677963257, 0.3862308859825134, -0.19360513985157013, 0.9390483498573303, -0.8854631781578064, 0.9286327362060547, -0.7661555409431458, 1.3281677961349487, -1.068691611289978, 0.8805009126663208, -0.6161872744560242, -0.04686947539448738, -0.1984092891216278, 0.4184730648994446, 0.4268918037414551, 0.039631858468055725, 1.0077543258666992, -1.4553934335708618, -0.6825631856918335, -0.5993112921714783, 1.280372142791748, -1.0774496793746948, 0.8013932108879089, 0.6445780992507935, 0.3628881573677063, -1.0233951807022095, -0.8923548460006714, 0.06050059199333191, 0.25015902519226074, 0.46398696303367615, -0.6465744376182556, 0.2889488935470581, -1.341261863708496, -0.4928338825702667, 1.5622142553329468, 0.30290600657463074, 0.10597576946020126, 0.6399537324905396, 0.5046454071998596, -0.18613925576210022, 0.6465880870819092, 0.6340550780296326, 1.6204828023910522, 0.7196032404899597, -0.5835357308387756, 0.6172528266906738, -1.2004534006118774, -0.43011221289634705, -1.094497799873352, 0.5523496866226196, -0.6906018853187561, 0.9780378937721252, 1.7997010946273804, 1.0853487253189087, -2.28134822845459, -0.19212567806243896, -0.023966193199157715, -0.9272307753562927, 0.5704543590545654, 0.21808378398418427, 0.037449486553668976, -1.5124881267547607, 0.6874603629112244, -1.6100714206695557, -1.5358668565750122, 0.802805483341217, -1.157815933227539, -0.6260530948638916, -0.46392685174942017, -0.7871782183647156, -0.4182298481464386, -1.0850900411605835, -0.15018422901630402, 1.3448312282562256, -0.0406678169965744, -0.058179568499326706, -0.13731040060520172, -0.47362756729125977, -0.11946975439786911, 1.250617265701294, -0.5634517073631287, -0.44098013639450073, -0.5064013600349426, -1.125793218612671, 0.49255216121673584, -0.2135971635580063, -0.6820522546768188, -0.14691472053527832, -0.19896294176578522, -0.08689852803945541, 0.5975855588912964, -0.8250001072883606, 1.9831430912017822, -0.45003822445869446, -1.5573084354400635, -0.022291621193289757, 1.0645081996917725, 0.732659101486206, -1.010266900062561, -0.31864655017852783, -1.3704744577407837, -0.4095410406589508, 1.2502638101577759, 0.6578769087791443, -2.3888227939605713, 0.19550730288028717, -1.0132026672363281, 1.0147743225097656, 0.8183567523956299, 0.529498815536499, -0.005836694501340389, 0.6483774781227112, -2.5429818630218506, -0.5449513792991638, 1.005012035369873, 0.33876070380210876, 0.9919848442077637, 2.7286014556884766, 0.04398522153496742, 0.7580811381340027, -0.15361963212490082, -0.21040084958076477, 1.8839788436889648, -1.496436595916748, 0.38251298666000366, -0.37456411123275757, -0.07441750168800354, -1.4033918380737305, 0.4029094874858856, 0.7629579901695251, 1.026742935180664, -1.4761431217193604, -0.09156736731529236, -0.6926248073577881, 0.7392739057540894, -0.02816767431795597, 0.5413195490837097, -0.9077687859535217, 0.4598292410373688, 1.1098201274871826, -0.22934304177761078, -1.082839846611023, 0.6194394826889038, -0.13029320538043976, -1.1528539657592773, -1.708519697189331, 0.47970476746559143, -1.0469779968261719, 0.791626513004303, -1.5102452039718628, 0.754615068435669, -0.2134993076324463, 0.41695624589920044, 0.2606295943260193, -1.3741158246994019, -1.0599313974380493, -0.9092453122138977, 0.9914371967315674, -0.7561872005462646, -0.483693391084671, 0.4427250027656555, -0.4417685866355896, -0.5229254961013794, -1.8440501689910889, -0.3480645418167114, -0.590198814868927, -0.3995131552219391, 1.2324497699737549, -0.8577349781990051, -0.4334568381309509, 0.9597612619400024, -0.9888939261436462, -0.6497705578804016, 1.4942359924316406, 1.2983298301696777, -1.4819278717041016, 0.3621009290218353, 1.3219897747039795, 1.0131162405014038, -0.1823679804801941, -0.5698339939117432, 0.00033325402182526886, -0.35916638374328613, 0.8518674373626709, 0.3043932318687439, 1.646954894065857, -0.2931894361972809, 1.8576737642288208, -0.3960424065589905, 0.1024731695652008, -1.003676176071167, -1.2874234914779663, 0.5189891457557678, 0.04843117296695709, 1.7947179079055786, 0.4037252962589264, 0.7306252717971802, -0.3015778660774231, -0.8971090316772461, -0.13638417422771454, 0.4141433537006378, -1.0300692319869995, -1.3549495935440063, -0.6846342086791992, -0.46267127990722656, 0.26064226031303406, -0.8521313667297363, -0.4976547956466675, 0.38118496537208557, 0.5034199953079224, 1.59109365940094, -1.665482759475708, -1.343590259552002, 1.0641616582870483, -0.6206679344177246, 1.9133905172348022, 0.18051211535930634, 1.943466305732727, 0.435465008020401, 0.15751391649246216, -2.4115655422210693, -0.0795283392071724, -0.2284567505121231, -0.4168068766593933, -0.02146463841199875, 1.5602267980575562, 0.5237557291984558, -1.464791178703308, 0.7984850406646729, 0.47317641973495483, -0.4637235701084137, 0.7461180090904236, 0.11590845137834549, -0.24594132602214813, 0.32642364501953125, -1.0492702722549438, -0.6806479692459106, -2.356602191925049, -1.3575799465179443, 0.39988332986831665, 1.7154474258422852, -0.013630670495331287, -0.5808828473091125, -1.5311185121536255, -1.2959877252578735, 0.4986092150211334, -1.9440242052078247, 0.1710595190525055, 0.21434108912944794, -1.9998457431793213, 1.2900300025939941, 0.7532097101211548, 0.08903516083955765, -0.6615660786628723, 1.5865297317504883, 0.4663466513156891, 1.4692308902740479, 1.3077102899551392, -1.121737003326416, 0.1569468230009079, -0.2883146405220032, -0.07797732204198837, -0.6234614253044128, 0.6316189169883728, -1.2123501300811768, 1.337734341621399, -0.016972854733467102, 0.6459431052207947, 0.9708716869354248, -1.1399884223937988, -0.565278947353363, 0.9029055833816528, -2.037149429321289, -0.7227455377578735, -0.22222022712230682, -0.8769478797912598, -0.44339680671691895, -1.0126644372940063, -2.088616371154785, -0.8601655960083008, 0.060104504227638245, -0.12505370378494263, -0.2554524838924408, 0.03333597257733345, -0.5021488666534424, -1.2793049812316895, 0.10846388339996338, -0.23832467198371887, -0.6917272806167603, 0.7870482206344604, -0.236871600151062, 0.2937885522842407, 1.7129918336868286, 0.759721577167511, 1.086269736289978, 0.005103789735585451, 2.2688095569610596, -1.560178279876709, 0.3792973458766937, 1.962354063987732, 0.9784989356994629, 0.5080271363258362, -0.30937910079956055, -0.683921217918396, -0.01875370927155018, 0.2450045943260193, 0.6901014447212219, -0.7809412479400635, 0.6427801251411438, -0.6551452875137329, 1.1436400413513184, -0.38093551993370056, -0.10881704837083817, -0.8314030766487122, -0.7396383881568909, 0.17393505573272705, -1.112518072128296, 0.5755844712257385, -0.209319069981575, -0.26761892437934875, 1.412495493888855, -0.1790355145931244, 0.2753359377384186, 2.464012622833252, 2.106436014175415, -0.3558128774166107, -0.12075001746416092, -2.5246732234954834, -0.09596087783575058, 0.7078719139099121, -1.5624340772628784, -0.11536478996276855, -1.8418138027191162, 0.40069863200187683, 1.2902987003326416, 1.7253265380859375, -0.20977187156677246, 0.6776403188705444, 1.375758409500122, 0.8740149736404419, 0.7268297672271729, 1.1859382390975952, 0.6875087022781372, -0.6305275559425354, -0.3735351264476776, 0.0305260568857193, -0.35222020745277405, 0.35030004382133484, 1.2887065410614014, -0.9606273174285889, -0.6531983017921448, -0.7244880199432373, -1.3874244689941406, -0.7169473767280579, 0.22551889717578888, -0.5265952944755554, 1.3368135690689087, -0.3735228478908539, 0.3085266053676605, 0.6962906718254089, 0.3976575434207916, -0.04452112317085266, 0.15910272300243378, 0.1079762652516365, 0.07149398326873779, -0.6137727499008179, 0.26266124844551086, -0.7712422609329224, 1.0327142477035522, 0.9672099351882935, -0.3911251723766327, 1.5448081493377686, 0.26901164650917053, 0.358175665140152, -0.5418310761451721, -1.3070523738861084, 0.5280380249023438, 0.4486194849014282, -0.32701748609542847, 0.8486812114715576, -0.21585246920585632, -0.13458338379859924, -0.537487804889679, -1.0002238750457764, -1.7512071132659912, -0.8408383727073669, -0.734915018081665, 1.081739902496338, -0.27312877774238586, 0.09787900745868683, -0.44013452529907227, -0.07314776629209518, 0.7785926461219788, 1.005383849143982, 0.44932591915130615, 0.24113775789737701, 0.22328819334506989, 0.5828995108604431, 1.0988484621047974, 0.9338158369064331, -1.4593510627746582, 1.6907602548599243, -0.6432645916938782, 1.7783467769622803, 0.444288432598114, -0.5857313871383667, -1.8408946990966797, -0.26764023303985596, -0.00925468746572733, -1.003124713897705, 1.0020813941955566, -0.8155077695846558, 0.665492057800293, -0.5322265028953552, -2.080181837081909, -0.7953000068664551, -0.9543588757514954, -0.36604782938957214, -0.9676830768585205, -1.647613286972046, -0.700552761554718, 0.8420071005821228, -2.242431640625, -0.5087082386016846, -0.5658661723136902, -1.286845088005066, 0.07022033631801605, -0.22753547132015228, 0.349555641412735, -0.15809586644172668, 0.13206373155117035, -1.4562979936599731, -1.80831778049469, 1.0247082710266113, 0.0748090147972107, 0.8265454769134521, -0.7094154953956604, -0.05757506936788559, -0.03429107367992401, 0.04926207661628723, 1.0934137105941772, -1.8475838899612427, 2.21799373626709, 0.44472911953926086, -1.1755985021591187, -1.4782778024673462, -0.5778074264526367, -0.15575699508190155, 0.23214004933834076, 1.181915521621704, -1.3468834161758423, -1.1335506439208984, -0.25539106130599976, -0.33407270908355713, 1.584673285484314, 0.6178598999977112, 0.5824097394943237, -0.6277490258216858, -0.7731444239616394, 1.0643541812896729, -0.10791873931884766, 1.5718376636505127, -0.7743626236915588, -0.6863822937011719, 0.685366153717041, 1.6992897987365723, -0.5345776677131653, 1.7527973651885986, -1.2213654518127441, 0.9979190230369568, 0.41591811180114746, 0.20202553272247314, -1.0704487562179565, -1.7792398929595947, 2.1657917499542236, -0.9767752289772034, 0.07905957102775574, 0.06789583712816238, -0.1918572634458542, -0.3932221531867981, 0.24015597999095917, -0.9719690084457397, 0.26976466178894043, -0.8209297060966492, 0.7751988768577576, 0.44166114926338196, -1.622846245765686, 0.9642298221588135, 0.122517891228199, 1.5870217084884644, -0.5938431620597839, -1.5477005243301392, -0.6628967523574829, -0.12959034740924835, 2.0540103912353516, -1.88387930393219, 0.9199936985969543, 0.6578559875488281, -1.083962082862854, 1.17598557472229, -1.4382652044296265, 0.6807644367218018, -0.05681459978222847, -0.3212146461009979, 1.6936793327331543, 0.006360059604048729, 0.8544937372207642, -1.0648205280303955, -0.8461116552352905, 1.3450696468353271, -0.0818401426076889, 0.28705137968063354, -0.3813150227069855, 0.00443692272529006, 0.41327741742134094, 1.226342797279358, 1.0962719917297363, -1.386916160583496, -0.1409171223640442, 1.1570407152175903, 1.7885777950286865, 0.49539050459861755, -1.2365127801895142, 0.9932109117507935, -0.17396610975265503, -0.7485345602035522, -0.12314444035291672, 0.20739248394966125, 1.4480199813842773, 1.1942075490951538, -0.011542467400431633, -0.47214454412460327, -1.4398322105407715, -0.18638113141059875, 0.886793315410614, 1.0408397912979126, 0.3632226586341858, -0.5072993040084839, -0.10044017434120178, 0.2392822802066803, 0.33171525597572327, -1.2687956094741821, 1.459123134613037, 0.9518921971321106, 0.37753626704216003, -0.8586546778678894, 0.5080152750015259, -1.8795301914215088, -0.630192220211029, 2.5839920043945312, 3.1440634727478027, 0.48697105050086975, -0.38156044483184814, -1.153965711593628, 0.6635822653770447, -0.4306109547615051, -0.9457457661628723, -0.7764759659767151, 0.8789895176887512, -0.32715997099876404, 0.30472877621650696, -1.2259660959243774, -1.0123664140701294, 0.2607901692390442, 0.5996405482292175, -0.7364648580551147, 0.8249557614326477, 1.151340126991272, 1.9149329662322998, 1.4969239234924316, 1.211543083190918, -0.2668638527393341, 1.6499534845352173, 1.667908787727356, -1.403551697731018, 0.04980286583304405, -1.054157018661499, 0.32268986105918884, 0.40474140644073486, 0.6854190230369568, 1.9355199337005615, 0.3413485288619995, -0.7241256237030029, -0.4603724181652069, 1.1596492528915405, 2.479708671569824, -1.2140754461288452, -1.127295970916748, -1.2497179508209229, 0.21072353422641754, 0.6165980100631714, 1.8745825290679932, -0.43975958228111267, 1.7380911111831665, 0.5632613301277161, 1.5532498359680176, -0.151467964053154, -0.8187665939331055, 0.26893994212150574, -1.0331120491027832, 0.7841396927833557, -2.4944205284118652, -1.0763592720031738, -0.23417086899280548, -0.11557652056217194, 0.052293118089437485, -0.08771318942308426, 0.3114297091960907, 0.14099901914596558, -0.29706647992134094, -1.8537508249282837, -1.4453363418579102, 1.329354166984558, 0.028227152302861214, 0.666283369064331, -0.30991044640541077, 0.24438229203224182, -0.7016482949256897, 0.6888951063156128, 0.6250838041305542, 0.5554743409156799, -0.4390201270580292, 0.6559475660324097, 1.244648814201355, -0.3167458176612854, 0.8897568583488464, 0.5120835304260254, 0.02322390489280224, 0.6361449360847473, -0.9341797828674316, 0.290348619222641, 1.6485669612884521, -0.4544224739074707, -0.43739286065101624, -1.6473596096038818, 1.6697769165039062, -1.6048696041107178, -0.10303284227848053, 0.8403738737106323, -0.46054333448410034, 1.4470609426498413, 1.0454270839691162, 1.0434619188308716, 0.1850813925266266, 0.8602023720741272, -0.9552357196807861, -0.7899925708770752, 0.4602237343788147, -1.4962912797927856, -0.7216107845306396, 1.3151541948318481, -0.9248979091644287, 0.7101210951805115, -0.34942519664764404, -2.219449758529663, -0.5162752270698547, -0.34375983476638794, 1.4278956651687622, -0.5584993958473206, -0.05080372840166092, -0.3974866569042206, -0.16485600173473358, 0.704708456993103, -0.11370069533586502, -1.6563491821289062, 1.1090891361236572, -1.78755521774292, -0.3109890818595886, -0.07318145036697388, -1.3004611730575562, -0.553693950176239, -0.5067957639694214, -0.2668037712574005, 1.3380123376846313, -0.010150217451155186, -1.2187557220458984, -0.4805290997028351, -1.0157139301300049, -0.8830710053443909, 1.3247852325439453, -2.1262309551239014, -1.114923119544983, -1.1221901178359985, 0.2033580243587494, -0.8631305694580078, -0.4936181306838989, -0.5181326270103455, 0.25617560744285583, -1.2710798978805542, -0.571308970451355, -0.9283610582351685, -0.45453235507011414, -0.8097752332687378, -0.4087553918361664, -0.5646858811378479, 2.081237554550171, -0.12383702397346497, -0.6890815496444702, -0.6541978120803833, 1.4677209854125977, 1.8231745958328247, -0.5719126462936401, 0.9607082605361938, -1.444751262664795, 0.4302176237106323, 0.1994839310646057, -0.750950813293457, -0.7244699001312256, 0.12730105221271515, -0.4958236813545227, 0.7140159010887146, 1.9656155109405518, -1.145786166191101, -1.2726247310638428, 1.4434419870376587, -0.06182747334241867, 0.04414902254939079, 0.4058339595794678, -1.0936392545700073, -2.236476182937622, -1.2234135866165161, -1.0883749723434448, 0.27793771028518677, 1.0620448589324951, -0.820684552192688, -0.681187093257904, 1.5446680784225464, -0.19912868738174438, 1.1813658475875854, 0.20031553506851196, -0.10339359939098358, 0.5471461415290833, -0.8097801208496094, 1.181060791015625, 0.4537409543991089, 2.6408274173736572, 0.4157187044620514, 0.8223865032196045, 0.016811346635222435, 0.4462336599826813, 1.688961148262024, 0.757860541343689, -0.577322781085968, 0.06796887516975403, 1.0446783304214478, -0.43549999594688416, 0.5589074492454529, 1.6046050786972046, -1.4019256830215454, 0.5155993103981018, 0.33558887243270874, -0.5489871501922607, 2.361079454421997, 1.8959505558013916, 0.8810044527053833, -1.280829668045044, -0.3400144577026367, 1.3083500862121582, -1.5005539655685425, 0.5821201205253601, -1.2146745920181274, 0.786032497882843, -0.9831712245941162, 1.1740890741348267, -0.5626160502433777, -1.4194961786270142, -0.3016156554222107, -0.02087414637207985, 0.7479183077812195, -2.035217761993408, 1.5379207134246826, 0.3053969442844391, 0.5449559688568115, -0.34008365869522095, -0.29961684346199036, -0.7092756032943726, -1.3803354501724243, -0.49386516213417053, -0.14053179323673248, 0.6450568437576294, 0.4342888295650482, 0.45411598682403564, 0.21123503148555756, 1.5882736444473267, 0.3939306139945984, -0.40166744589805603, 0.5243294835090637, -2.095355987548828, -0.5146605968475342, 0.4683774411678314, -0.08238031715154648, -0.5066419839859009, -1.0289230346679688, 0.2507482171058655, -2.254211187362671, -0.10128356516361237, 1.7414485216140747, -0.7339209914207458, -0.503488302230835, 0.012006723321974277, 0.09799706935882568, -0.5913441181182861, -0.04468229040503502, 0.3974319398403168, 1.3280601501464844, -0.6278055310249329, 0.00653119245544076, 1.212893009185791, 0.861239492893219, -2.233917236328125, 0.7350120544433594, -0.37158462405204773, -0.05117101967334747, 1.2390880584716797, -0.22327736020088196, -0.07287314534187317, 0.6766669750213623, 0.42103999853134155, -0.4655766785144806, -0.09510049223899841, -1.5566338300704956, -0.6287132501602173, -0.6805964112281799, 0.320090651512146, 0.5599613189697266, -1.1532405614852905, -1.6778866052627563, -0.04599440097808838, 0.7977424263954163, 0.969496488571167, -0.9174214601516724, -0.01347994152456522, -1.4253878593444824, -1.2442386150360107, 0.2669030427932739, -0.971605658531189, 2.025578022003174, 2.694831132888794, -0.3509518504142761, 1.3103649616241455, 0.5626327991485596, -1.2820450067520142, -0.6459198594093323, 0.7931479215621948, -0.5023857951164246, 0.32868462800979614, -0.2873465418815613, 0.3653150796890259, -0.1127048060297966, -0.5692803263664246, -0.972609281539917, 0.38070347905158997, 0.5138672590255737, 0.626931369304657, 0.5697535276412964, 0.7353489995002747, 0.3891035318374634, 0.5226984620094299, -0.47827810049057007, 0.6505322456359863, -1.2479963302612305, -1.6392016410827637, -1.0101515054702759, 0.63201504945755, 0.6899315118789673, 0.8945974707603455, -0.5817973017692566, -0.5578877329826355, -0.5796008706092834, 0.38801783323287964, 0.7624946236610413, -3.5176491737365723, -0.1545681208372116, 0.19887858629226685, 0.5310168862342834, 0.447001188993454, 1.0900893211364746, -0.769228994846344, 1.2377688884735107, 0.44235220551490784, 0.823440432548523, 1.0823817253112793, -0.8440287709236145, 0.6950106024742126, -1.0755577087402344, 0.7066723108291626, 1.0773974657058716, 1.3959306478500366, -0.12299197167158127, 0.7779765129089355, -1.2796534299850464, 1.0742944478988647, 2.0245954990386963, 0.9202807545661926, 0.2845865488052368, 0.02163681946694851, -1.671956181526184, -0.17644146084785461, -1.3603520393371582, -0.6483216285705566, 0.2676308751106262, 1.1653083562850952, -1.4033880233764648, -1.0234792232513428, -0.5670983791351318, 0.36876776814460754, 0.3307028114795685, -0.4038165509700775, -0.600731372833252, -2.2975196838378906, 0.9069280624389648, -0.6072055101394653, 1.7971069812774658, -0.07009568810462952, 0.030661828815937042, 0.6506079435348511, 0.16846616566181183, 0.4772302508354187, -0.17533525824546814, 0.11122725903987885, 1.1945809125900269, -0.446538507938385, -0.04750329256057739, -0.9668720960617065, -0.3878439664840698, -0.09714244306087494, 1.2926478385925293, 0.6272354125976562, -0.7699090838432312, 0.8426511883735657, -2.1062490940093994, -0.3636176884174347, -1.3041001558303833, 0.2758963108062744, 0.9243724346160889, -1.1040668487548828, -0.21596717834472656, 1.1538360118865967, -1.5170162916183472, -1.0220530033111572, 1.3991833925247192, -1.4993025064468384, 1.0223618745803833, 1.2755403518676758, -0.4829704463481903, -0.7650485634803772, 0.7845840454101562, 0.47129037976264954, 0.4030013084411621, 0.1339302957057953, 0.05341315269470215, -0.41301536560058594, 0.7645775675773621, 2.117488384246826, -0.4798385202884674, 0.7766635417938232, -0.703203022480011, 0.5493507981300354, 0.2751876711845398, -1.0920289754867554, 0.45841479301452637, 1.6200412511825562, 0.4388808310031891, 0.6467287540435791, -0.8797554969787598, -0.4905552566051483, 0.5689839124679565, 1.420699954032898, 0.029864788055419922, -0.7319110631942749, 0.4038826823234558, -0.6129356622695923, 0.7380164265632629, 0.6750265955924988, -1.391453504562378, 2.302530288696289, 0.8063800930976868, -0.4239095151424408, -1.2095515727996826, 0.6892256736755371, -2.47213077545166, 0.8015601634979248, 2.3162527084350586, 1.1559525728225708, 0.32571297883987427, 0.6896609663963318, -0.38278988003730774, 0.3645186126232147, -0.6652781963348389, -2.861562728881836, -0.6152801513671875, 4.318342325859703e-05, -0.6367068886756897, 0.9187600612640381, -1.7895796298980713, 1.630820393562317, 0.19907034933567047, -0.3308313190937042, 0.7760980725288391, -0.16165703535079956, -1.8856905698776245, -0.6496497392654419, -1.8786790370941162, -0.06161501258611679, -0.009525113739073277, -0.9291301369667053, -1.4173563718795776, 0.18264460563659668, 0.7303194403648376, -0.8114899396896362, 0.47724977135658264, 1.6480882167816162, -0.8653488755226135, 0.5711252093315125, -1.8065053224563599, 0.18451938033103943, -0.4115030765533447, -1.3180434703826904, 0.2534923851490021, -0.6473166346549988, 0.7819253206253052, 0.685206949710846, 0.8777327537536621, -0.17706795036792755, -1.037436842918396, -0.9910584092140198, 0.05258866026997566, -1.7513872385025024, 0.37765589356422424, 0.9491636753082275, -0.4045824110507965, 0.3035788834095001, 0.7579183578491211, -0.7274258136749268, 1.7591477632522583, 0.8273292779922485, -1.5572654008865356, 1.2939479351043701, 0.8138229250907898, 0.7716988921165466, 0.6633061766624451, 2.836639404296875, 1.9411884546279907, -0.8123840689659119, -0.6693094372749329, 0.043057989329099655, -0.9980775713920593, -1.7537058591842651, -0.7835413813591003, 1.3443095684051514, -0.2346385419368744, 1.025213360786438, -0.49617254734039307, 0.7217755913734436, -1.046691656112671, 1.6510555744171143, 0.7884782552719116, -0.01644855923950672, 1.031785011291504, 0.3576009273529053, -0.987157940864563, -0.5133383274078369, 0.0014146865578368306, -0.3494678735733032, 0.7184815406799316, -1.2747668027877808, 0.4025057256221771, -0.6306734681129456, 0.7345560789108276, 0.4844684600830078, -0.5693512558937073, -0.5218466520309448, 0.6788063049316406, 1.564998745918274, 0.8658354878425598, -0.3293684124946594, 1.187814474105835, 0.5094946026802063, 0.5575590133666992, -0.31154903769493103, 0.41528812050819397, 0.709365963935852, 1.424508810043335, -0.5015406012535095, -0.4195531904697418, -0.48406368494033813, -0.752711296081543, -0.2728913426399231, -2.1503031253814697, -0.6770943999290466, -0.9925376176834106, -0.46535685658454895, -1.853074073791504, -1.3110560178756714, -0.18414440751075745, 3.0852863788604736, 0.36248165369033813, 0.829196572303772, -1.0414776802062988, -0.5848132371902466, 0.29657745361328125, -0.26023000478744507, -1.1094458103179932, -0.3850996494293213, 1.3431284427642822, -1.8714498281478882, -1.0786393880844116, 0.416777640581131, -0.23485420644283295, 1.2856675386428833, 1.5140882730484009, -1.1813640594482422, 2.1918845176696777, -1.7380409240722656, 0.8309144973754883, 1.0479166507720947, -1.245762825012207, -1.2808551788330078, 0.47693002223968506, 0.3899240791797638, 0.5573241710662842, -0.8378863334655762, -0.7015883326530457, -1.6935865879058838, 1.274522304534912, 1.4631620645523071, 0.368409663438797, -1.3430700302124023, 0.6149176359176636, 0.937126636505127, 2.2199819087982178, -1.3460205793380737, 1.0811913013458252, -0.31243789196014404, -0.026111552491784096, 0.2536645829677582, 0.1609940230846405, -0.5362452864646912, -0.9686586260795593, -0.9414915442466736, -1.2397761344909668, 0.1663201004266739, -0.47765791416168213, -1.6898341178894043, -0.5508787035942078, 0.41725173592567444, 1.5474458932876587, 1.0413154363632202, -0.723375678062439, -0.6717970371246338, -0.03676356002688408, -0.724151611328125, 0.9685302972793579, -2.116314172744751, -0.6133046746253967, -1.2103569507598877, -0.02820681594312191, -0.20734357833862305, 0.7286855578422546, -0.3426856994628906, 2.9593403339385986, 0.829971432685852, 0.2572033405303955, -0.04777495190501213, -0.3858957290649414, 0.2232448309659958, -0.8585493564605713, -0.5227232575416565, 0.8539717793464661, 0.529853880405426, -1.4577616453170776, -0.7621130347251892, -0.5460006594657898, -0.6281717419624329, 1.0068529844284058, -1.5631158351898193, -1.356825351715088, -0.6814952492713928, 1.6238739490509033, -0.3886532187461853, -0.3517541289329529, -0.8619617223739624, -0.08210236579179764, -0.42530474066734314, 1.3352853059768677, 1.6716221570968628, 0.8468091487884521, -1.1275190114974976, 0.32493141293525696, -0.4729308485984802, -0.16760358214378357, 0.6652147769927979, 0.5436320304870605, 0.03426114842295647, 0.7457266449928284, 0.8940967321395874, 1.558229684829712, -0.689443826675415, -1.2159686088562012, -0.9594826102256775, -1.2804312705993652, -0.7850663661956787, -1.5376185178756714, -0.10414940118789673, 0.5227644443511963, 0.9257277846336365, 1.9193564653396606, -0.9813307523727417, -0.38849908113479614, 1.651924729347229, 0.47355180978775024, 2.693037986755371, -0.7800138592720032, -0.3207806944847107, -0.34162795543670654, 0.8374078869819641, 1.6201350688934326, 0.5050309896469116, -0.948733389377594, 1.1599005460739136, 0.8359416127204895, 0.2505142390727997, -1.2050811052322388, -0.26518622040748596, 0.10931327193975449, 1.0630606412887573, -2.319916248321533, -0.02987104468047619, 1.2056708335876465, -1.59592604637146, -1.4509445428848267, 0.7235453128814697, 0.6479758620262146, 0.1008935496211052, 1.628340482711792, 0.7754068374633789, 0.21859732270240784, -0.6406999230384827, 2.642284631729126, 1.0230497121810913, 1.4894044399261475, 1.1641403436660767, -0.9147112369537354, 0.4906942844390869, -2.0778274536132812, -1.2668960094451904, -1.9306716918945312, -2.091986894607544, 0.5861623287200928, 1.8812469244003296, -2.0472252368927, -1.2052714824676514, -0.5236852169036865, 1.6392581462860107, -0.20025157928466797, 1.8232728242874146, 0.21897085011005402, 1.242186188697815, -0.26313289999961853, 0.9487766027450562, -1.1354961395263672, -0.24654227495193481, -0.14087191224098206, -0.025258278474211693, -0.6096380352973938, 0.9899702072143555, 0.6190319657325745, 1.052051067352295, 0.9186676740646362, 0.7962546348571777, 1.2866483926773071, -1.6469694375991821, 0.11471961438655853, 0.498418390750885, 0.03568682819604874, -0.4669438898563385, 0.25893208384513855, 1.0686125755310059, -0.19192537665367126, 0.8238022923469543, -0.9985068440437317, -0.5780485272407532, -0.6799783110618591, 0.6287599802017212, -1.2274521589279175, -0.018245823681354523, -0.6948904395103455, 1.3848552703857422, -0.15417256951332092, -2.329835891723633, -0.5330732464790344, -1.9934823513031006, -0.18842796981334686, -0.5057389140129089, -0.7130707502365112, -1.068608283996582, 1.4075000286102295, -0.814785361289978, 0.17172081768512726, -0.1006641834974289, -1.2060695886611938, -0.659426212310791, -0.11768513172864914, -0.40899163484573364, -0.19554422795772552, 0.18756753206253052, -1.9082857370376587, 0.07821246981620789, -0.6137076020240784, 1.3023985624313354, 0.8365145325660706, 0.7655004262924194, -0.29776692390441895, -0.9582781195640564, -0.023569149896502495, 1.4579756259918213, 0.601260781288147, -0.2792096734046936, 0.7790826559066772, -2.0722765922546387, 0.06810170412063599, 0.8907743692398071, 0.34121114015579224, 1.2419753074645996, 2.2324342727661133, 0.4173000752925873, 0.19961827993392944, 0.632912278175354, -1.2029895782470703, 0.6337251663208008, -0.3713315427303314, 1.012830376625061, 1.1696528196334839, -1.0989327430725098, -0.6295337080955505, 1.2020736932754517, -0.2226284146308899, -0.037345968186855316, 0.9509748220443726, 0.9272131323814392, -1.2830078601837158, 0.9376401305198669, -0.6357126235961914, 0.7899182438850403, 1.6257621049880981, -1.2162669897079468, -1.2459686994552612, -0.18686099350452423, -1.1621912717819214, -0.09277059882879257, -0.7146096229553223, -0.29255107045173645, -0.5391973257064819, -1.380766749382019, 1.2577273845672607, 1.3748459815979004, 1.0765578746795654, 0.776906430721283, -0.43187415599823, 1.8385342359542847, 0.11098933964967728, 1.1525037288665771, -0.028260642662644386, -0.605591356754303, 2.882721424102783, -1.4406758546829224, -0.25305137038230896, -0.4295012354850769, -1.1114262342453003, -0.3576250374317169, -1.6338688135147095, 0.4735870957374573, 0.8140414357185364, -1.6076723337173462, -0.17647193372249603, -1.901502251625061, -0.8985990285873413, 0.2880289554595947, 2.3693947792053223, 0.20930756628513336, 0.7676131725311279, 1.0604873895645142, -2.575861692428589, 0.07805778086185455, 0.35528382658958435, -2.028291702270508, 1.8979566097259521, 0.0591103620827198, 0.31629669666290283, -0.41078564524650574, 0.02838386595249176, -1.4981460571289062, -1.0564357042312622, -0.277288019657135, -0.5382817983627319, 0.031239451840519905, 1.1309512853622437, 0.4784490168094635, 1.5816948413848877, 1.1238402128219604, 0.7468835711479187, -0.16034971177577972, 0.02178896963596344, 1.5714658498764038, -1.515754222869873, -0.5527712106704712, 2.0314202308654785, -1.4826897382736206, -0.49260154366493225, 0.6833113431930542, -1.0886485576629639, 1.7887781858444214, -0.9793081879615784, 1.0477160215377808, 2.132962942123413, -0.19536012411117554, -1.098125696182251, -0.21532434225082397, 0.8430729508399963, 1.3582041263580322, 0.9450218081474304, -1.2410274744033813, -0.14178241789340973, -1.3153034448623657, 0.046670880168676376, -0.6710476875305176, -1.0173698663711548, -1.0505456924438477, -0.7689774632453918, 0.32111528515815735, -0.5653389692306519, 0.3229231834411621, -0.3494656980037689, -0.09347337484359741, 0.0015741001116111875, 0.7244625687599182, -0.4616696536540985, -0.07560429722070694, -0.7960447669029236, -0.47691792249679565, -0.16683289408683777, 0.33039215207099915, -0.5577659606933594, 0.2518354058265686, -0.9846800565719604, -0.7370419502258301, -1.3155027627944946, 0.6980526447296143, -0.12392351776361465, -0.8057718276977539, -0.6168431639671326, -0.6371340751647949, 0.6834037899971008, -0.22708997130393982, -0.22869011759757996, 0.6876781582832336, -0.7595256567001343, 0.40205612778663635, -1.9779866933822632, -1.636064887046814, 0.6810420155525208, -0.588894784450531, 1.4932806491851807, -0.0076890732161700726, 0.20441417396068573, 0.7522691488265991, 0.3001086711883545, -2.576483964920044, 0.08287905156612396, 0.07316631078720093, -1.430736780166626, 1.7540414333343506, -0.4880800247192383, 0.0684690773487091, 1.7088849544525146, 0.41825762391090393, 1.0106836557388306, -0.9819083213806152, 0.6179654002189636, -0.6955899596214294, -0.3214077651500702, 0.8004294633865356, -0.7644282579421997, -0.8420187830924988, -0.6480504274368286, -1.780540108680725, 0.970960795879364, -1.1449464559555054, 1.2905865907669067, -0.5983090400695801, -1.2179110050201416, 1.3525398969650269, -2.763828754425049, -0.9496897459030151, -0.28328096866607666, 1.0214180946350098, 0.9808381199836731, -0.3110644519329071, -0.19873668253421783, -0.5701761245727539, 0.5920524597167969, -0.6707436442375183, -0.8042405247688293, -1.6181706190109253, -0.5807170867919922, 0.49303945899009705, 0.5225803256034851, 0.13908718526363373, 0.8921073079109192, -1.2597697973251343, 0.1798863261938095, -0.25551387667655945, -0.3406929671764374, -0.40503761172294617, 1.470815658569336, -0.5486162900924683, -0.1431151032447815, -0.5601808428764343, -0.8052514791488647, 1.32025146484375, 0.9347643852233887, 0.9733521342277527, 1.8379127979278564, -0.12551793456077576, 1.4369845390319824, -0.13537542521953583, 0.8183719515800476, 1.330579400062561, 0.5719934701919556, -0.15062539279460907, -0.20126724243164062, -0.791398286819458, 1.989053726196289, 0.3803819417953491, 0.4866662919521332, -0.7709499597549438, 0.3939059376716614, 0.5456106066703796, -0.48279789090156555, -0.3646111786365509, -0.6901983618736267, 0.11384060233831406, 0.6742240786552429, 1.7700997591018677, 0.24049684405326843, 0.04506433382630348, -0.7326793670654297, 2.4896886348724365, -0.8388861417770386, 0.932130753993988, 1.2872332334518433, 0.9259316325187683, 0.19146348536014557, 1.0523320436477661, 1.523291826248169, 0.865750789642334, 1.5762563943862915, 0.3495875895023346, 0.18022973835468292, -0.9911463856697083, 2.088437795639038, -1.881683588027954, -0.6166135668754578, -0.9677754640579224, 0.357027530670166, -1.5480530261993408, -1.4549616575241089, -1.3902263641357422, 1.1467788219451904, -0.20003002882003784, -0.6030464768409729, -1.9024252891540527, 0.22616001963615417, 1.4143874645233154, 0.29664912819862366, 0.47466176748275757, -0.6988134384155273, 3.5551562309265137, 0.6681897044181824, -0.3311658799648285, -1.7334012985229492, -0.10405851155519485, 0.2885933220386505, -0.1204790398478508, 0.17908000946044922, 1.238876461982727, -0.5598065853118896, 0.6638371348381042, 1.0418870449066162, -0.35312536358833313, 1.3012291193008423, 0.3626519441604614, -0.2615242004394531, -0.6218446493148804, -0.01686982996761799, 2.3388938903808594, -1.063740611076355, -1.962456226348877, -1.6464191675186157, 0.05737929046154022, 1.6782093048095703, 0.25648319721221924, -1.5374592542648315, -0.0011422568932175636, 0.9917313456535339, 0.8429687023162842, 0.15671396255493164, 0.25541946291923523, 1.3495253324508667, 0.6206342577934265, -0.5063996911048889, 0.1978037804365158, 0.7331335544586182, -0.8134592771530151, -0.2711515724658966, -1.0300389528274536, 0.1615600734949112, 0.4712231457233429, -0.02127315290272236, 1.2609823942184448, -0.2737608850002289, 0.7015925645828247, -2.221759080886841, 1.3801785707473755, -0.43735313415527344, -0.29485583305358887, -1.2191827297210693, 0.08723921328783035, 1.2194809913635254, 0.9228824973106384, 0.44892486929893494, 0.7801235914230347, 2.0897772312164307, 1.1367336511611938, -1.224516749382019, -0.030848240479826927, -0.6705154776573181, 0.7137317657470703, -0.565864622592926, -1.0590516328811646, -0.08194699883460999, -0.06444140523672104, -1.7784541845321655, -0.24406401813030243, -0.44728580117225647, 1.0609732866287231, -0.21389886736869812, -0.7699381709098816, -0.7249436974525452, 0.6870331168174744, 1.8055915832519531, 0.9223738312721252, -1.6157561540603638, -0.49628132581710815, 1.097296953201294, -0.33330240845680237, -0.24312284588813782, 0.7361007928848267, 1.3209233283996582, -1.1682339906692505, -0.625598132610321, -0.362260639667511, -0.038566704839468, -0.9264068603515625, -0.9387128949165344, 2.1598565578460693, 0.1688084453344345, -1.2431176900863647, -0.1289726346731186, 0.9455206990242004, -0.18519549071788788, 0.1088162437081337, -0.204302117228508, -0.7812364101409912, 1.3675144910812378, 0.39159277081489563, -1.2464790344238281, -0.6072617173194885, 1.1673333644866943, -1.6201218366622925, -1.6088414192199707, -0.9190046787261963, -0.4545261859893799, -2.3746137619018555, -1.2690349817276, -1.308489203453064, 0.20982567965984344, 2.0191409587860107, 0.0432392954826355, 0.31907111406326294, 1.0517325401306152, 0.06484594941139221, 0.6991556882858276, -0.22218239307403564, 1.547856330871582, 0.1543722152709961, 0.2995476722717285, 0.023755649104714394, 0.6994783282279968, -0.9258280992507935, -0.35783976316452026, -0.06656499207019806, 1.3702627420425415, -0.9817106127738953, -2.5021159648895264, 1.0314152240753174, 0.6433363556861877, -0.10644807666540146, -0.041567038744688034, -2.2288577556610107, 0.5343195199966431, -0.32356029748916626, -1.200552225112915, -0.7193877100944519, -0.44342517852783203, 1.073749303817749, 1.2479323148727417, -0.21125143766403198, 0.49477896094322205, 0.7952777743339539, 0.483233779668808, 0.8123018741607666, -1.0561895370483398, -1.3661322593688965, -1.6026051044464111, 0.006413924042135477, 0.25480183959007263, 1.2853773832321167, 0.24275235831737518, -0.016880720853805542, 0.5290752053260803, 0.3015219271183014, 0.865347683429718, 0.37015700340270996, -0.9125645160675049, -0.27254918217658997, -0.32316333055496216, -0.9892839193344116, 1.2819063663482666, 0.4552876651287079, 0.786244809627533, -0.22793816030025482, 0.4448886513710022, 0.5394402742385864, -0.9933028817176819, -0.044226739555597305, -1.360129714012146, 0.8428635001182556, 1.5581104755401611, -0.8202333450317383, -0.7158950567245483, 1.0846728086471558, 1.1042472124099731, -0.4771374762058258, 0.02368968538939953, -0.5811697840690613, 0.657788872718811, 0.23388679325580597, -0.19293516874313354, -0.9879198670387268, -2.260862112045288, -0.1932811290025711, -0.3029983639717102, -0.3014663755893707, 0.5584232211112976, -1.1640788316726685, 0.5680801868438721, -1.3304554224014282, 0.5828002691268921, 0.7466254830360413, 0.19968988001346588, 0.00868184119462967, -0.14081044495105743, 0.315783828496933, -1.611299753189087, -0.510027289390564, -0.10979940742254257, 0.25798559188842773, -0.4447072148323059, 1.065039038658142, -1.2582671642303467, -1.7709028720855713, 0.11780638247728348, -0.2884620428085327, -0.7417702078819275, -1.1802153587341309, 0.4135582745075226, 0.28904515504837036, -0.08422403037548065, -0.927370011806488, -1.0235682725906372, 0.08713475614786148, 0.6045745015144348, 1.2006829977035522, 0.9267843961715698, -0.03469821810722351, 0.019146306440234184, 1.7402122020721436, 0.2481098771095276, -1.1228866577148438, 0.08914317935705185, 0.1711108535528183, 0.09049761295318604, -0.44478410482406616, -0.5797703266143799, 0.9824027419090271, 0.4616730809211731, 0.33929359912872314, 0.2403235286474228, -0.8162374496459961, 0.6789693236351013, -1.2588443756103516, 0.5938458442687988, -0.6104620099067688, 1.4112223386764526, -0.7318254113197327, -0.18618127703666687, 1.4968359470367432, 0.021375659853219986, 0.5581121444702148, -0.3872547149658203, 1.3519216775894165, 0.0749528557062149, -1.0667225122451782, 0.4233909845352173, -1.2106422185897827, 0.7922908067703247, -0.02638884261250496, -0.22572773694992065, 0.15226510167121887, -1.70514714717865, 0.7980947494506836, 1.246336817741394, 0.8017610907554626, -0.4369145929813385, -0.945862889289856, 2.3326284885406494, -1.2404932975769043, 1.1560299396514893, 1.072556495666504, -0.2751297950744629, -0.6808547973632812, 0.7702386975288391, -1.4153915643692017, 0.9100316762924194, 0.4953819513320923, 0.8562706112861633, 2.2072505950927734, -0.9591748118400574, -1.9499555826187134, -1.035025715827942, 1.1004160642623901, 0.8395172357559204, 0.583488404750824, -0.3141951560974121, 0.026814617216587067, 1.2779393196105957, -0.0761801227927208, 1.8343899250030518, -0.9782934188842773, -0.7859014272689819, -1.0529075860977173, 0.2136770635843277, -0.2894238531589508, -0.37046581506729126, -0.9749643802642822, 0.5651758909225464, 0.4861490726470947, -0.787508487701416, -0.7564867734909058, -0.4783220589160919, 0.19216026365756989, -0.08169759064912796, 1.0901243686676025, -0.5897865295410156, 0.4351939558982849, -0.08794189989566803, -0.874578058719635, 2.192322254180908, 0.5073978304862976, 0.20380300283432007, 2.9416701793670654, -0.2755277454853058, 1.402031421661377, -1.181294322013855, 0.5605812668800354, -1.1157664060592651, 0.10729046911001205, 0.7250179648399353, 0.5555503964424133, -1.5491902828216553, 1.324844241142273, -0.943386971950531, 1.776432991027832, -0.4808826744556427, -1.5579079389572144, 0.4755863845348358, 0.6318156719207764, -0.44782015681266785, -0.23029382526874542, 0.8010953068733215, 0.22238561511039734, -1.3460650444030762, 0.48440223932266235, 0.2769228518009186, 0.7681297659873962, -0.05116556957364082, 0.7499017715454102, 0.642368495464325, 0.5023053288459778, 0.9911289215087891, 0.7645381689071655, -0.262434720993042, -0.2705593705177307, 1.5804048776626587, 0.7774280905723572, -0.5727453231811523, 1.2808163166046143, -0.2051568180322647, 0.5910183787345886, -0.8422350287437439, 0.8523265719413757, 0.41655656695365906, -0.5599759221076965, 0.1797063648700714, -1.5029765367507935, -1.054802656173706, 0.6960526704788208, -0.4711620807647705, -0.2605000138282776, -0.22027337551116943, 1.385199785232544, 0.055554959923028946, -0.9432227611541748, -0.42918145656585693, -1.181727647781372, -1.657999038696289, -1.2644093036651611, 1.4887416362762451, -0.5351101756095886, 0.47820934653282166, -0.17324519157409668, 1.4470009803771973, 1.218536138534546, 2.8300132751464844, 0.6916884183883667, 0.7841039299964905, -0.28488975763320923, 0.7961917519569397, 1.0269358158111572, 0.7578899264335632, -0.20655596256256104, -0.848339855670929, -0.6625112295150757, 0.546736478805542, -1.3653370141983032, 0.0023037181235849857, 0.567764401435852, -0.36799156665802, 0.976723313331604, 0.4615548849105835, -0.05779210478067398, 0.17599423229694366, -2.103907346725464, 0.6583002209663391, -1.431939721107483, -1.2173947095870972, 0.7288641333580017, -1.905984878540039, 1.165946125984192, -1.0600037574768066, 0.8732359409332275, 2.0373294353485107, 0.8803048729896545, 1.1059248447418213, 0.389254093170166, 1.8646819591522217, 1.1656702756881714, 1.4990233182907104, -0.5251290202140808, -0.7310348153114319, -1.2782644033432007, -0.2800315320491791, 0.7684335708618164, -0.7637399435043335, -0.32104507088661194, -1.2788411378860474, -0.9142100214958191, 1.054459810256958, 0.7355571985244751, 1.2858186960220337, 0.11865659058094025, 0.2640950381755829, -0.20098115503787994, -0.6647859215736389, 0.5846388339996338, 2.8097639083862305, 0.10100868344306946, -0.4844416379928589, 0.7129363417625427, -0.12025506049394608, -1.3566884994506836, -0.45001715421676636, -0.48645269870758057, -1.0935543775558472, -0.09002744406461716, 0.3774273693561554, -0.3154616057872772, -0.5985870361328125, 1.2205188274383545, 0.737740159034729, 0.15243738889694214, -1.003225564956665, 0.28776267170906067, 1.206626534461975, 0.3586723804473877, -1.0251888036727905, -0.5859949588775635, -0.17019128799438477, 3.3759567737579346, -0.10662918537855148, 0.28905975818634033, 0.08609835058450699, 0.03146865591406822, 1.5760153532028198, 0.8719319105148315, -0.34675461053848267, 1.428105115890503, 0.5365682244300842, -0.6704864501953125, 0.1212383434176445, 0.7660004496574402, 0.10077393800020218, 2.5297586917877197, 0.24144035577774048, -1.1915465593338013, 1.271096110343933, 0.23209023475646973, 0.5501394271850586, -1.203161358833313, 1.7220739126205444, 0.1493515968322754, 1.9025224447250366], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[-1.8377147912979126, 1.894152283668518, 0.9041316509246826, 3.4946916103363037, -0.8790346384048462, -0.11425592750310898, 1.2166590690612793, 0.16344688832759857, 1.6538680791854858, -1.2960401773452759, 0.6320498585700989, -1.2636274099349976, -0.05169284716248512, 0.396556556224823, 0.11332176625728607, 0.813080906867981, 1.499143362045288, 3.6264359951019287, 0.054450422525405884, 2.273460626602173, 0.3704608976840973, -3.2053189277648926, 0.5926124453544617, -0.30542996525764465, 1.2804943323135376, -2.2721939086914062, -1.1535203456878662, 0.7346075177192688, -1.6238150596618652, 0.9841337203979492, 1.304571270942688, 0.7904645800590515, 0.1602640599012375, -0.6088976263999939, 0.617881715297699, -1.8496284484863281, -0.38657668232917786, -0.6771032810211182, -0.8230544924736023, -1.7141894102096558, -0.6171764731407166, -0.04379111900925636, 1.02034330368042, 0.04408113285899162, 0.8929458856582642, 2.35272216796875, -0.8204612731933594, -0.6286084055900574, 1.8536936044692993, -0.38066649436950684, -0.2595909535884857, 0.12697888910770416, -0.3326081931591034, -2.058366298675537, 2.5517122745513916, -0.8627874255180359, 1.886103630065918, -0.6176773309707642, 0.8559700846672058, 2.6791605949401855, -0.052650973200798035, 1.6749372482299805, 0.8408483862876892, 0.34988003969192505, -0.4674409031867981, -2.4094936847686768, -1.0147653818130493, -1.1001073122024536, 0.7086698412895203, 3.5557310581207275, 0.1173829510807991, 1.4948192834854126, 0.701127290725708, -0.024311505258083344, 0.14461341500282288, -0.15751078724861145, 1.466985821723938, -0.3503729999065399, -1.8997677564620972, -0.12823611497879028, -0.8560269474983215, -0.2292780578136444, -0.21886424720287323, -0.08865948766469955, -0.02341518923640251, -1.0464186668395996, -0.4942530393600464, -0.37269967794418335, -0.8407859802246094, -0.6593048572540283, 0.3448655605316162, 0.8204407095909119, -1.5274766683578491, 0.025194549933075905, -0.8199520707130432, -2.012089967727661, -1.0312248468399048, -0.5774478912353516, 1.8724397420883179, -3.1839439868927, 1.0328104496002197, -1.0514447689056396, -2.2107632160186768, -0.925650954246521, -1.0378774404525757, 0.8966648578643799, 0.735230565071106, -4.885711669921875, 0.7425001263618469, 0.3624746799468994, -1.0500521659851074, 0.44807758927345276, -1.6085773706436157, 1.8221296072006226, -2.502833843231201, 1.2177834510803223, 0.6875041127204895, 0.09583602100610733, 0.672542154788971, -1.7899678945541382, 0.008426793850958347, 1.3162240982055664, 0.42265304923057556, 2.3437297344207764, 1.4870991706848145, -0.7934690713882446, 0.3526272177696228, -0.0687946081161499, -0.21002377569675446, -0.7683870792388916, -2.0673491954803467, 0.26197782158851624, -1.3422999382019043, 0.15527547895908356, -0.7259629368782043, -0.3816479742527008, 0.8196946382522583, -0.9819054007530212, -0.9342942237854004, 0.31441497802734375, -1.7294524908065796, 0.7207306027412415, 0.9161555171012878, 0.9252467155456543, -0.6655844449996948, 0.5837568640708923, 0.5334334373474121, 0.8840168118476868, 0.0522426962852478, -2.5641605854034424, 0.39993101358413696, -1.2642805576324463, 0.0249556303024292, 0.05389546602964401, 1.6230430603027344, -3.1862661838531494, 2.091399908065796, 1.6991665363311768, -1.036083459854126, -0.804551362991333, -0.4740125834941864, -1.8840445280075073, 1.4093180894851685, 1.5331244468688965, -0.7844019532203674, 2.2005715370178223, 1.3457436561584473, -3.6541993618011475, -0.013685914687812328, -0.0971483662724495, 1.7613099813461304, -2.0839452743530273, 0.8001161217689514, 0.05939660966396332, -0.6250905990600586, 0.9934365749359131, 1.7917556762695312, -4.383670330047607, 0.4674789011478424, 0.5824150443077087, -1.0341243743896484, 0.23189149796962738, -0.9285207986831665, 0.2875092923641205, 0.8854739665985107, 0.40096792578697205, -2.031777858734131, 0.32913151383399963, -0.20341573655605316, 0.5143632292747498, -0.6222782731056213, 0.43651390075683594, -2.577711343765259, 0.7516244649887085, 1.307340145111084, 0.47551777958869934, 5.100184440612793, 0.34294578433036804, -1.6747539043426514, -0.6134682893753052, -0.5714238286018372, -0.9129160046577454, -1.3191684484481812, -0.5373754501342773, -3.6050877571105957, 0.1359073370695114, -0.3358266353607178, -0.38142600655555725, 0.23584197461605072, 1.6820499897003174, -0.3071008622646332, -0.5342397689819336, -0.10756708681583405, -1.1026639938354492, 0.02940373308956623, -0.2915651202201843, -0.8873938322067261, -0.24312391877174377, -1.3922380208969116, -1.2874175310134888, -0.38086262345314026, 0.3618308901786804, 0.5035170912742615, -1.1819674968719482, -1.679767370223999, -0.7717618942260742, -1.2514622211456299, -0.3271608054637909, -2.183981418609619, 0.12980321049690247, -1.0170804262161255, 1.062469720840454, 0.46215906739234924, 1.1168854236602783, -0.08558908849954605, -0.30865946412086487, -0.7688751220703125, -0.1342465877532959, -0.289713591337204, -0.7299811244010925, 2.919571876525879, 0.9726643562316895, -1.3905797004699707, -1.5149071216583252, -1.0343488454818726, 1.2146697044372559, 0.9448953866958618, 0.23069679737091064, 0.3161791265010834, -1.0480645895004272, 0.4846985340118408, -0.6847784519195557, -1.4417736530303955, 2.564408302307129, -0.23411433398723602, -1.1646056175231934, 0.20095521211624146, 1.8106045722961426, -4.537757873535156, 1.9343907833099365, 0.7276967763900757, 0.33300477266311646, 0.14465653896331787, -0.5406111478805542, -0.6671642065048218, 1.1020060777664185, -0.4103046655654907, -0.7196272611618042, 0.645685613155365, -0.044582828879356384, 0.3018290400505066, 2.4930312633514404, -0.47676795721054077, -0.36400899291038513, -1.0636792182922363, -6.258723735809326, -0.40920665860176086, -1.6688650846481323, 3.6066174507141113, 1.2105358839035034, 0.4998050928115845, -0.6485132575035095, 1.4978848695755005, 1.6870225667953491, 0.7254225015640259, 0.556550920009613, 1.1476963758468628, 2.3162310123443604, 0.31423431634902954, -0.8800826668739319, 1.981188178062439, -1.6061975955963135, -0.25655895471572876, -1.666549563407898, 1.804986834526062, -0.4748883843421936, 0.08699741959571838, -0.8794640898704529, -0.9241401553153992, -1.541184902191162, 0.6310646533966064, -2.1401383876800537, -1.7480638027191162, 0.1231951117515564, 0.3540998101234436, -2.0798580646514893, -3.656707525253296, 0.05735747888684273, 1.5794475078582764, 0.28495800495147705, 0.22505249083042145, -0.21698853373527527, -1.7070680856704712, -0.5838767290115356, 1.9416639804840088, 0.10426256060600281, 0.0444829948246479, -1.4173299074172974, -1.0681058168411255, 1.1700917482376099, -0.9158442616462708, -1.3522175550460815, -1.3722479343414307, 0.7082204222679138, 0.9456827640533447, 1.060289978981018, 0.9472758173942566, 0.38051313161849976, 1.767075538635254, -0.14878037571907043, 0.3729086220264435, 0.8493779897689819, -1.5956345796585083, 0.1004335880279541, -0.07638847082853317, -0.6918624639511108, -1.9874025583267212, 0.944141149520874, -1.3114508390426636, -1.2082878351211548, -0.5178938508033752, 1.7497392892837524, -0.18600080907344818, -1.457849144935608, 0.8786112070083618, 0.06771012395620346, 0.6066285967826843, 0.6525992751121521, -1.4702805280685425, -2.7415661811828613, 1.7502806186676025, 1.3662142753601074, 0.8963122367858887, -0.7555766701698303, -0.3593416213989258, 2.091613292694092, -1.9416966438293457, -1.923495888710022, -0.5087745785713196, -0.7641854882240295, -0.47926056385040283, -1.3546391725540161, 2.2622785568237305, -0.9002551436424255, 2.144723653793335, 0.4961256980895996, 0.5618842244148254, -0.9574161767959595, 1.294473648071289, -0.5381589531898499, -1.0751125812530518, 1.3612525463104248, 0.201324000954628, 0.9970085024833679, 0.4272935092449188, -3.2098114490509033, -1.1207547187805176, 0.4808554947376251, -0.25033488869667053, 0.4766920804977417, 1.3349822759628296, 0.0747353732585907, -1.4664347171783447, 0.6458752751350403, -0.8758513331413269, -1.1915971040725708, -1.3052579164505005, -0.6083342432975769, 1.6958919763565063, 1.3995813131332397, -0.7661897540092468, 2.672085762023926, -1.1895694732666016, -0.1070319339632988, -1.1802171468734741, -0.16194294393062592, -1.604455828666687, -2.0346953868865967, -0.04052245616912842, -0.695067822933197, -1.33927321434021, 1.4015166759490967, 0.37557750940322876, 1.3900861740112305, -0.7030271887779236, 0.38342350721359253, -0.8855747580528259, -1.7817925214767456, 1.1790106296539307, -0.3446483314037323, -0.3099178969860077, 0.7573434710502625, -0.9852461814880371, -0.5742445588111877, -1.2874547243118286, -0.6271539926528931, -0.1084703654050827, -0.8229925632476807, 2.894758462905884, 0.4287106394767761, 1.8777087926864624, -1.0924383401870728, 0.4520815908908844, -1.8526430130004883, 0.15322822332382202, 1.5918855667114258, 0.407288521528244, -0.945554256439209, -2.2168772220611572, -0.22125940024852753, -2.1495182514190674, -0.07022411376237869, 2.3430447578430176, 1.9082159996032715, 1.5303372144699097, -4.792173385620117, -0.7181052565574646, 1.8928923606872559, -0.6502073407173157, 1.1709128618240356, 1.982992172241211, 0.4222085475921631, 1.1941254138946533, 1.3520399332046509, -0.6210141181945801, 2.4311065673828125, -0.024840861558914185, 0.4871026277542114, -1.0469640493392944, 0.1978389471769333, -1.2104641199111938, -0.7073589563369751, -0.0634133443236351, 0.7184113264083862, -0.7015740871429443, 2.223052740097046, -0.6327489018440247, 2.1002042293548584, -3.260420560836792, -1.2903934717178345, 0.8991373777389526, 0.25624069571495056, -0.1372028887271881, -0.11152273416519165, -0.6971381902694702, 1.8884460926055908, 2.6021053791046143, -0.2858999967575073, 0.42393651604652405, 0.448999285697937, -2.0491950511932373, 0.7064127922058105, -0.04308461397886276, 3.6371097564697266, -2.212536096572876, 0.08710374683141708, 0.7035027146339417, -2.4720258712768555, -2.4155638217926025, 0.38861313462257385, 1.2832598686218262, 3.1024997234344482, -0.6287385821342468, -0.43027082085609436, 3.516765594482422, 2.3345141410827637, -0.677463173866272, -2.2007699012756348, -0.7031958699226379, 1.7375085353851318, -2.208585500717163, -0.06243234872817993, 2.0853283405303955, 1.6667557954788208, 0.11203059554100037, -1.1698609590530396, 0.4763983190059662, 0.2618621289730072, 1.1090208292007446, 2.3882524967193604, -0.30864742398262024, -0.20708754658699036, -0.3007727563381195, 1.2220593690872192, 0.33531510829925537, -1.7234913110733032, 0.6519392132759094, 0.017084931954741478, 0.7378649711608887, -1.671812891960144, 1.6531916856765747, -0.5968071818351746, 1.4990525245666504, 1.0393986701965332, 0.5748713612556458, -0.5171751379966736, 1.4855983257293701, -0.7529631853103638, 0.23617112636566162, 0.4133130609989166, 2.158262014389038, 0.2353757619857788, -0.3213900327682495, -0.8544155359268188, -1.2096349000930786, 0.6582478284835815, -0.6032130122184753, 1.4065346717834473, 1.356358289718628, -0.3676532804965973, 0.454475075006485, -1.2117003202438354, -2.576018810272217, -2.230839967727661, -0.24865290522575378, -1.3062421083450317, 0.34997880458831787, -2.5710108280181885, -0.1062791645526886, 0.32483476400375366, 1.177033543586731, 0.05330163985490799, 1.0477232933044434, 0.27607211470603943, -0.4678104817867279, -0.8182294964790344, 0.8351043462753296, 1.4595954418182373, -0.47593769431114197, -0.1504324972629547, -0.4349329173564911, -1.8251440525054932, -0.20996911823749542, 0.7093837261199951, 1.8677533864974976, 0.08758240938186646, 0.29782024025917053, -0.502762496471405, 0.5961929559707642, -1.2101795673370361, 0.3030312955379486, 0.16161216795444489, -0.7302359938621521, 1.1337631940841675, 0.03858796879649162, -0.36006036400794983, -1.6582430601119995, -0.27376025915145874, 0.550899088382721, 0.1560351699590683, 1.4361586570739746, -1.3640413284301758, 0.17175345122814178, -0.29213812947273254, -2.854673385620117, 3.1836063861846924, 0.46069246530532837, 0.5646284818649292, 1.4017369747161865, -1.1839334964752197, -0.728470504283905, -2.171043634414673, -1.0763367414474487, 0.2657841145992279, 2.9434902667999268, 1.8748815059661865, 0.47559988498687744, -0.21049827337265015, -0.7465917468070984, 0.11075843125581741, 0.09827283769845963, 2.486438035964966, -1.546392560005188, -0.586783230304718, -0.8881890773773193, 1.1609631776809692, -0.6976835131645203, 0.456342875957489, 0.3520764112472534, 1.5747414827346802, -0.54688960313797, -0.2503366470336914, -1.4421731233596802, 1.7769830226898193, -0.11861427873373032, -0.9139378666877747, 0.5451220870018005, -0.10089293122291565, -0.7338091731071472, -1.9319380521774292, 1.4134645462036133, 0.6636186838150024, -0.17360210418701172, 1.1674944162368774, -1.782861351966858, 0.1677277535200119, -1.3484021425247192, 0.2629120945930481, 1.6789259910583496, -0.5078901052474976, -1.9797000885009766, -0.2083350121974945, 1.4610885381698608, -1.0482110977172852, -0.8694393038749695, -0.19046299159526825, -1.9977152347564697, 0.5610775947570801, -0.7403413653373718, -1.1927680969238281, -2.4484200477600098, 1.0505967140197754, 6.81354284286499, 1.2082396745681763, -1.2678462266921997, 0.01668085716664791, -0.9271987676620483, -0.4780532121658325, -0.5770480036735535, 0.026726115494966507, -1.94805908203125, -1.4864028692245483, -1.1764922142028809, -3.331547498703003, 1.831756353378296, 0.6217061281204224, 0.33212825655937195, 0.1334652155637741, -0.20159095525741577, 0.6970306038856506, 2.1811225414276123, -1.0377753973007202, 4.423603057861328, -1.4615428447723389, 1.4820139408111572, -0.3090883791446686, 1.9102574586868286, -0.19037795066833496, 1.858119010925293, -1.0863391160964966, 0.3883744180202484, -0.7563602328300476, 1.010931134223938, -1.0387288331985474, 1.234570860862732, -3.1109304428100586, -0.7719026803970337, -2.4140379428863525, -1.8335329294204712, -0.978593111038208, -1.028475284576416, 1.3712821006774902, 0.9394349455833435, 1.4651358127593994, 0.18895135819911957, -1.3549174070358276, -0.3158564269542694, 1.125003457069397, -0.5174867510795593, -0.15542349219322205, -1.6386255025863647, -1.9230024814605713, 1.5488810539245605, 0.13462252914905548, 2.3208508491516113, -0.451887309551239, 1.6626379489898682, 0.5249375700950623, -2.1147594451904297, 0.30528682470321655, 0.9231780171394348, -0.4595726728439331, -0.3112710416316986, -0.9566547274589539, 0.8729424476623535, 0.9494850635528564, -0.9235439300537109, 0.4847722053527832, 0.41435521841049194, 2.192246198654175, 0.7159801721572876, -0.3543417155742645, -1.6422032117843628, -0.029539676383137703, 1.280282735824585, 0.6821936368942261, -0.36553654074668884, 0.34231510758399963, -0.16596920788288116, 2.0340449810028076, -0.006910224445164204, -1.382315993309021, -0.9936032891273499, 0.7367122173309326, -0.1745259016752243, -2.655965805053711, -1.1991750001907349, -1.4622981548309326, -0.11036497354507446, -1.8604625463485718, -1.766536831855774, 1.0034408569335938, 1.4278501272201538, 0.7820429801940918, 0.24044081568717957, 0.8087799549102783, -1.5612348318099976, 0.6000291109085083, 1.0739305019378662, 1.1600134372711182, -0.8172118663787842, 0.8644567131996155, -1.0133414268493652, 0.33571141958236694, 1.1197799444198608, 0.04157223179936409, -0.5910477638244629, 0.8364391922950745, 0.38309594988822937, 0.4758652150630951, 0.46566399931907654, 0.0718274936079979, -0.40211907029151917, -1.5768423080444336, -0.4699974060058594, 1.1112300157546997, -1.6646859645843506, 0.37497884035110474, 4.122775077819824, 0.22170934081077576, 0.9978928565979004, 1.915305495262146, -0.25738102197647095, -0.6903964281082153, -1.1213144063949585, -0.6857514977455139, -0.1801067292690277, 1.3300740718841553, -1.764298439025879, 1.520564317703247, -0.575465738773346, 0.257076621055603, 0.9540513157844543, -1.8467668294906616, -1.1911287307739258, -2.1964638233184814, 0.5368106961250305, -1.5764780044555664, -1.133481502532959, -0.6212096214294434, 0.6166388988494873, 1.7453943490982056, -1.1967898607254028, 0.24376513063907623, 0.17623038589954376, 1.0597435235977173, 0.926287829875946, 1.5878313779830933, -0.8567195534706116, 3.4889461994171143, -0.027854347601532936, -1.0465383529663086, -0.10368205606937408, 0.4362850487232208, 1.2819302082061768, 0.9895380139350891, -2.4711976051330566, -2.3393425941467285, -0.1339346170425415, -1.8081543445587158, -0.2773798108100891, -2.540583610534668, -0.6907509565353394, -2.0776543617248535, 2.241010904312134, -0.7824538946151733, 1.1980195045471191, -0.9476312398910522, 0.6500192284584045, 1.556593894958496, -0.18892528116703033, -1.0494128465652466, 1.465305209159851, 0.8824062943458557, 1.138456106185913, 1.269979476928711, -0.011428632773458958, 0.6510837078094482, -3.348689556121826, 0.17096975445747375, 0.683541476726532, 0.39598995447158813, -1.2122831344604492, 0.4778721332550049, -1.3320256471633911, -1.323536992073059, -0.3851208984851837, 5.74074649810791, -2.195120334625244, 1.060359001159668, 0.030771246179938316, -3.307492733001709, -0.8411950469017029, -2.69404673576355, 0.08557485044002533, -2.640589475631714, 0.11069115996360779, 1.383643388748169, -5.769237041473389, -1.9051697254180908, 0.09799031168222427, -2.8331856727600098, -0.15727172791957855, -1.7809404134750366, 0.7900769114494324, -1.940893530845642, 2.211658477783203, -0.8666300177574158, 1.1874220371246338, 2.2127747535705566, 2.690058946609497, 0.7274671792984009, -2.4166219234466553, -2.893991231918335, 1.3706445693969727, -0.4676072299480438, -0.5802168846130371, -2.3359713554382324, 0.9526886343955994, -2.929699659347534, -1.5643398761749268, -1.6251804828643799, 0.044051218777894974, -1.078439474105835, 0.28618505597114563, -0.3092156648635864, -0.7513865828514099, -0.7562144994735718, 0.614490270614624, 1.3867051601409912, 1.4171459674835205, 0.39791685342788696, -1.843987226486206, 1.9887242317199707, -0.7609585523605347, -0.03978455066680908, -0.011811730451881886, -0.20210735499858856, -2.200333833694458, 0.5308390259742737, -0.642550528049469, -0.19217093288898468, -0.6808059811592102, -1.0872498750686646, 0.03263270854949951, -3.177720069885254, -0.046130336821079254, -2.5799882411956787, -0.9690312743186951, -1.861496090888977, 1.2448701858520508, 0.9598456025123596, -3.603839159011841, 1.1519842147827148, 1.6939690113067627, 0.44331973791122437, -0.4951247274875641, 1.6527953147888184, -0.9962077140808105, -0.527363657951355, -0.7218530178070068, 1.5344078540802002, -0.40326064825057983, -0.0780583918094635, -0.15279093384742737, -0.5373738408088684, -2.609462261199951, 0.9878115057945251, 1.0426318645477295, 0.7666628956794739, -1.1241791248321533, -2.113330602645874, 0.08899111300706863, 0.05320614576339722, 0.10220080614089966, -1.3248447179794312, -0.2938162386417389, 0.8530284762382507, 0.564969003200531, -1.5155868530273438, 1.1618174314498901, -0.3579946756362915, 0.8855423927307129, 2.2340846061706543, 1.883805751800537, -0.3919999897480011, 0.0531938262283802, 1.4359333515167236, -1.2032780647277832, -0.3574332594871521, 0.22530339658260345, -0.3236178457736969, -1.8293335437774658, -1.3084198236465454, -1.0371171236038208, 2.1117002964019775, -3.3265910148620605, -0.2562997043132782, 0.4128264784812927, -0.47881707549095154, 2.146641731262207, -0.14001956582069397, -0.3139273524284363, 0.41875067353248596, 1.8942234516143799, -2.2915093898773193, -0.8312678933143616, -1.4676158428192139, 1.6809732913970947, 0.7483327388763428, -0.30455780029296875, 0.8343469500541687, -1.922945499420166, 0.9600710868835449, -2.2558443546295166, 5.386742115020752, -2.2355926036834717, -2.055133581161499, 1.7766801118850708, 0.4950527846813202, -0.11461786180734634, -1.8924293518066406, 0.3850729167461395, -1.8420605659484863, 1.6386593580245972, 0.2513691484928131, 0.8694533705711365, -1.968252420425415, 1.0769048929214478, -0.6327936053276062, 1.9109690189361572, 2.275643825531006, 1.176053524017334, -0.01629994437098503, 0.7114236950874329, -2.7015697956085205, 4.148996353149414, -0.810676634311676, 0.4351333975791931, 1.7756942510604858, 0.4924190044403076, -1.6980578899383545, 0.6941982507705688, 1.1765590906143188, 1.0739372968673706, 1.0964757204055786, -1.1165131330490112, -0.19644205272197723, -1.2623330354690552, 0.8351565003395081, -0.47961196303367615, -0.6874892711639404, 1.126191258430481, 0.27835968136787415, -1.519404649734497, 0.8628936409950256, -0.19743648171424866, -0.7081483006477356, -1.9068092107772827, -0.3863183259963989, 2.0990583896636963, 2.57942533493042, -0.6979668140411377, -0.4287179112434387, 0.2620876133441925, -0.3912741243839264, 0.28009384870529175, 3.205681800842285, 1.074602723121643, 1.7433804273605347, 0.49653008580207825, -0.910613477230072, 1.6401540040969849, 1.1754971742630005, 0.25509124994277954, 1.9073413610458374, 0.6271230578422546, 2.173163890838623, -2.234650135040283, -1.632684588432312, -0.4923388957977295, -0.976280152797699, 0.9775975346565247, 0.563896894454956, -0.5899737477302551, -3.4187159538269043, -1.6189552545547485, 0.5815087556838989, 1.172349452972412, -1.4954830408096313, 2.0412800312042236, -1.8349589109420776, -0.37398266792297363, -2.0860674381256104, 0.8205671310424805, 0.2985467314720154, 1.7439988851547241, 0.9239920377731323, 0.5420507788658142, -2.8248326778411865, -1.0836000442504883, 2.727717399597168, -4.618338584899902, 1.1695544719696045, 1.093542218208313, 0.35087594389915466, 1.1865954399108887, 0.10305032879114151, -0.6082460284233093, 0.569369912147522, -2.0183427333831787, -2.896536350250244, -0.3474315106868744, 0.7978708744049072, 1.2559970617294312, 0.4025338590145111, -0.024093560874462128, -2.822673797607422, -0.3336776793003082, -0.1089499443769455, 0.8978090882301331, 0.792479395866394, -0.11318140476942062, 1.1270781755447388, 0.5318979024887085, 0.48046889901161194, 1.126943826675415, -1.9485743045806885, -2.202538251876831, 2.1509180068969727, 2.1844584941864014, -1.63893461227417, -0.09742172062397003, -1.5529557466506958, -2.6208178997039795, -2.3515186309814453, -4.498832702636719, -1.5149344205856323, -1.5575594902038574, 0.20340745151042938, 0.1843542605638504, 1.0378693342208862, 0.32420217990875244, 1.6245143413543701, -1.910481333732605, -1.9439373016357422, 0.17810265719890594, 2.4960951805114746, -1.3097270727157593, -1.586303472518921, 0.10344646126031876, -1.094860553741455, 0.9449059367179871, -0.043255604803562164, -0.3177908658981323, -3.5252790451049805, 1.413813591003418, -0.06507724523544312, 0.7917532324790955, -0.45306381583213806, 0.672069251537323, -0.2791544497013092, -2.397812843322754, 1.2757443189620972, -1.346206545829773, -0.4093305468559265, -0.4593515694141388, -0.17648953199386597, 0.6013186573982239, -2.290285348892212, -3.764472723007202, 0.34697622060775757, 0.4981226623058319, -0.15887907147407532, -1.5398809909820557, 1.439985990524292, -0.5355167388916016, -0.7921586632728577, -4.466874599456787, 0.98858243227005, 0.07193204015493393, 0.19830474257469177, -1.2120800018310547, -1.7971959114074707, -0.42841750383377075, 0.48050060868263245, -0.31071028113365173, -2.745095729827881, 0.19073566794395447, -2.934800386428833, 0.13086523115634918, -0.9035980105400085, -1.9251716136932373, -0.08831305801868439, 1.1451765298843384, 1.0026742219924927, 1.327599048614502, 0.06421135365962982, -2.3118655681610107, 0.0397428534924984, 2.19659161567688, 2.059567928314209, -1.1962045431137085, -0.9910450577735901, 0.530508279800415, -0.26765942573547363, 2.1116409301757812, 0.48035499453544617, 1.62740957736969, -1.3792541027069092, -1.1518381834030151, -0.15978066623210907, -1.0030410289764404, -3.0881712436676025, 1.176055908203125, -0.49495184421539307, 1.0399253368377686, -2.23386287689209, -1.1228487491607666, -2.160871982574463, -0.10065587610006332, 1.2070372104644775, 0.09648291021585464, 0.7758586406707764, -0.17418168485164642, -0.2016797661781311, 0.9126113653182983, 0.2150133103132248, 1.09858238697052, 0.9046988487243652, -1.1315289735794067, 0.5173757076263428, -1.0232627391815186, 0.1754661500453949, -0.12359778583049774, -0.27433010935783386, -0.7595593929290771, 0.774010956287384, -0.4492858648300171, -1.0754450559616089, -2.823687791824341, -0.18332478404045105, 2.00718092918396, -0.41454172134399414, 2.5207319259643555, -0.029624653980135918, 0.10601063072681427, -2.0324409008026123, -0.7896440029144287, 1.0753734111785889, -0.895304262638092, -3.5552542209625244, 0.1287371963262558, -0.7540743947029114, -0.5561732053756714, 0.342353492975235, 1.3281364440917969, 0.9329395890235901, -1.3475394248962402, 1.0186192989349365, -0.744767963886261, -1.573606014251709, 0.6919187903404236, 0.503757894039154, 0.883233368396759, -2.273569345474243, -1.4778196811676025, -0.2374420315027237, -0.3190188407897949, -0.8947197198867798, -0.578632116317749, -2.2902157306671143, -2.515101194381714, 0.37499719858169556, 1.1407222747802734, 1.3739991188049316, -1.2783223390579224, -0.2715717554092407, 0.12962020933628082, -0.28126728534698486, -0.743565022945404, -0.023290524259209633, 0.21349146962165833, 0.2929111421108246, -1.6751707792282104, -1.998619794845581, -1.4088159799575806, 1.629323124885559, -0.9246596693992615, -0.562087893486023, 2.121192216873169, -0.9002837538719177, -0.15409483015537262, 1.6134884357452393, 4.932762145996094, 2.0969889163970947, -1.0637433528900146, 2.215834379196167, 0.23426678776741028, -1.119402527809143, 1.3319121599197388, -3.9325942993164062, 1.4159119129180908, 0.46847087144851685, 1.3224962949752808, -0.8154159784317017, -1.4486225843429565, -2.1655380725860596, -2.0294029712677, -1.3498029708862305, -0.6097283363342285, -0.19017353653907776, -2.3870458602905273, -0.2793991267681122, 1.3884981870651245, -0.9189158082008362, 1.3079878091812134, 3.7411916255950928, -2.048977851867676, -1.6920950412750244, -2.9497745037078857, -2.4217402935028076, -0.12291877716779709, -0.1451370120048523, -1.0927609205245972, -0.5513765215873718, 3.216095447540283, 1.3877750635147095, -1.9086650609970093, 1.2832616567611694, -0.9865127205848694, 0.578245997428894, 1.4673835039138794, 1.9020376205444336, 0.16487185657024384, 2.9256033897399902, -0.2767427861690521, -0.2516203224658966, -0.8454033136367798, 0.36159324645996094, 0.6553645730018616, -1.126465916633606, 0.48180076479911804, 0.7342955470085144, -1.849912166595459, -0.14999522268772125, -2.2521297931671143, 1.538077712059021, -0.22215896844863892, -0.6317527294158936, 0.8490954041481018, -0.5877306461334229, 2.5030694007873535, 2.7684712409973145, 0.05439663678407669, 1.3142794370651245, -1.3312692642211914, 0.5813424587249756, -0.2794461250305176, 0.27716872096061707, 1.437174916267395, -0.6258065104484558, -1.9803372621536255, -0.5668445825576782, 0.5799649357795715, 0.44635531306266785, -1.797518014907837, -0.1129312589764595, 0.23175948858261108, -0.19538182020187378, -0.9805405735969543, 1.3025822639465332, -2.787804365158081, -1.4748337268829346, -0.2963518500328064, -0.16060219705104828, -0.7788530588150024, 0.567743182182312, 0.027154654264450073, -0.2241201102733612, -0.7425709366798401, -1.2234904766082764, -0.6012829542160034, 1.0585408210754395, 0.8237807154655457, -1.2305283546447754, 0.012112710624933243, 0.8020360469818115, 0.20882435142993927, 3.3778631687164307, 1.2990299463272095, 0.3882737457752228, -1.6395263671875, 0.372165322303772, -2.9425978660583496, -0.636874794960022, -0.5748588442802429, -0.24088314175605774, 0.5964683890342712, 1.2855027914047241, -1.7240805625915527, -0.28485605120658875, 0.7360989451408386, 0.8559780716896057, -0.08423633128404617, 0.3030352294445038, -0.7747436761856079, 0.8227292895317078, -1.1775954961776733, 2.0769717693328857, -0.1742052137851715, 2.800689458847046, 2.2804648876190186, 0.24441425502300262, -1.6911925077438354, 2.0158565044403076, 1.5773640871047974, 0.9232715368270874, 2.266350507736206, -1.8722158670425415, 0.532835066318512, 2.282980442047119, -1.3400260210037231, 0.013808922842144966, 1.4938308000564575, 0.7280750870704651, 0.4961192309856415, 2.5834808349609375, -0.03383735567331314, -0.8649796843528748, 0.06290602684020996, -0.34237635135650635, 1.006343960762024, 2.2564539909362793, 2.4374005794525146, -0.2646543085575104, 0.8786131739616394, 1.0795756578445435, 0.401634156703949, -0.34734049439430237, -0.5417150855064392, 0.7392928600311279, 1.8792492151260376, -1.3179789781570435, -1.8086503744125366, -0.22679907083511353, 3.754049062728882, 0.5139697790145874, -0.9814836382865906, -0.13235867023468018, 0.6980888247489929, -0.8342143297195435, 0.5151529908180237, -0.7600875496864319, 1.3618533611297607, -1.2128525972366333, 0.28758949041366577, -1.9548053741455078, 0.19587364792823792, -1.2528281211853027, -0.41552871465682983, 0.20603516697883606, 1.0922529697418213, -0.24824044108390808, -1.0874133110046387, -1.0942047834396362, 1.1997196674346924, 0.7785204648971558, 0.7071554064750671, -0.32716384530067444, 1.0393056869506836, -0.5581686496734619, -0.5117387175559998, 0.7766873240470886, -0.6459495425224304, 0.44652247428894043, -0.3779219090938568, -0.7004772424697876, -0.4057515859603882, -0.1388462483882904, 0.34635409712791443, -1.260031819343567, -0.47995713353157043, 3.169300079345703, -1.0894718170166016, 0.4376939535140991, -0.7360049486160278, -1.2448906898498535, -0.1923670768737793, 1.3867597579956055, 0.36434438824653625, 0.20509934425354004, -2.034548282623291, 0.8919761776924133, 0.9605787396430969, 2.9597020149230957, 3.349055051803589, 1.5753880739212036, 0.3287374973297119, 0.06402638554573059, 1.919254183769226, 0.476237952709198, -5.27182674407959, 0.1391739398241043, -0.6309365630149841, 0.2738487720489502, -0.9403905272483826, 3.4552319049835205, -3.8706367015838623, 1.0903598070144653, -0.7665525078773499, -0.23810447752475739, -0.5653484463691711, -0.5211567282676697, 3.9687271118164062, 2.554450273513794, 2.0150833129882812, 0.8452577590942383, -0.04265860840678215, -0.08319640904664993, 0.7040042877197266, 0.9401302337646484, 0.12643963098526, 0.2923712134361267, -1.103345513343811, 0.2026754766702652, -0.7976611852645874, -1.0369945764541626, 0.32082661986351013, 0.30788442492485046, 0.22974061965942383, 0.27548831701278687, 1.1608428955078125, 0.6887151598930359, 2.2891499996185303, -1.2217317819595337, -0.31138038635253906, -0.15229639410972595, 0.5200004577636719, -0.4561878442764282, 0.04946470260620117, -1.6427550315856934, 0.7752946615219116, 0.870578408241272, -0.7198547124862671, -2.883190631866455, -1.3498848676681519, 0.22789472341537476, 1.0334718227386475, -0.48625317215919495, -0.15900307893753052, -0.8979766964912415, 1.1167951822280884, 0.9406846165657043, 0.08456477522850037, -1.3292988538742065, 0.5439252257347107, -0.5563331246376038, -1.2332124710083008, -0.18836252391338348, -0.07510421425104141, 1.0144802331924438, 3.6915581226348877, 1.0558840036392212, 0.7679572105407715, 0.08937564492225647, 0.8721984028816223, 0.46538400650024414, -3.727980852127075, 2.3337907791137695, -0.9568066596984863, 2.5102317333221436, -1.3949075937271118, 0.5461720824241638, 1.7383133172988892, -0.35360243916511536, 1.570235252380371, 0.19911080598831177, 0.8414563536643982, 1.1235933303833008, -0.6152396202087402, 0.5212534666061401, 1.3195029497146606, -1.4360685348510742, -1.4525972604751587, 1.473027229309082, -1.5803132057189941, -1.0697859525680542, 0.8270453810691833, -0.501815915107727, 1.348042368888855, 0.7534149289131165, 1.0268938541412354, 1.065908432006836, -1.9122021198272705, 1.132578730583191, 0.7160094976425171, -0.18896375596523285, 2.2763493061065674, -0.28440919518470764, 0.8571293950080872, -1.1644879579544067, 0.09086862206459045, -0.3304036259651184, -0.3041442632675171, 0.0707358568906784, -1.2985153198242188, 3.002321720123291, -1.8229583501815796, 1.8966411352157593, -0.8030727505683899, -2.2948732376098633, 0.6178837418556213, 0.2717360854148865, -0.18030989170074463, 0.8019880056381226, 2.6332311630249023, -0.9297723174095154, -0.058308348059654236, -2.1183247566223145, -0.6180787086486816, 1.270978569984436, 0.3745045065879822, -0.42555367946624756, 1.759924292564392, -0.5916110873222351, -0.32814812660217285, 0.4890853464603424, 1.1533819437026978, -2.3471171855926514, -0.15429231524467468, 0.7544345259666443, 0.09526417404413223, -0.6636002063751221, -0.20559397339820862, -4.828785419464111, 1.9063795804977417, -0.5493584871292114, 0.7344443202018738, 0.09760311990976334, 0.4142345190048218, 0.6896951794624329, -0.9003094434738159, 1.1625761985778809, -0.6352168917655945, -1.1343293190002441, -1.040887475013733, -1.5398257970809937, -0.46514710783958435, -0.8731693625450134, 2.680384874343872, 2.4097073078155518, -1.9748103618621826, 1.8902552127838135, -1.6126089096069336, -0.48723652958869934, -0.4207651913166046, -1.5962930917739868, 0.7346881031990051, 0.8160126805305481, -1.8699079751968384, 2.54252290725708, -0.6835299134254456, 0.7405409216880798, -0.8498477935791016, 0.35980355739593506, 0.8482919335365295, 0.11052405834197998, -0.29684701561927795, -0.1273033618927002, -0.2764062285423279, -0.40509265661239624, -2.0328452587127686, 0.08542437106370926, 0.653224527835846, -0.41720184683799744, -1.6973118782043457, -0.997620165348053, 0.7873693108558655, 1.5147801637649536, -4.496862411499023, -0.9410797953605652, -1.3386472463607788, -0.9341394305229187, 1.8748986721038818, -1.0231127738952637, 0.0036772312596440315, -1.72542405128479, -1.1529535055160522, -0.5464284420013428, 0.487270712852478, -1.201279878616333, 1.3201011419296265, 1.4865607023239136, -0.032383278012275696, 0.40128615498542786, -2.0202314853668213, 0.7246229648590088, 1.1588659286499023, -0.20789700746536255, -1.2266794443130493, 0.9102424383163452, 0.47836601734161377, -2.6994481086730957, -0.41102278232574463, 0.5158059000968933, -1.0847442150115967, -2.836073398590088, 2.7768752574920654, -1.2390477657318115, -0.6259971857070923, -0.66768479347229, -0.0357598178088665, 1.5647414922714233, -2.1530468463897705, -0.42712217569351196, 0.5286670327186584, -0.6945800185203552, 1.776528239250183, 0.9797153472900391, -0.13627202808856964, 1.1395950317382812, -2.3567488193511963, 1.6650866270065308, -1.682230830192566, 1.140920877456665, -1.1893974542617798, 3.5027811527252197, -0.5411884188652039, -0.4889393448829651, -0.671799898147583, 2.9211323261260986, -1.105920672416687, -0.8472869396209717, 1.1633641719818115, -1.2423996925354004, 1.0701221227645874, -0.7951045036315918, 3.8940236568450928, 4.871566295623779, -0.3777492642402649, -1.2867648601531982, -0.7744808197021484, 2.6722257137298584, 1.576332449913025, 4.054192066192627, -2.01983642578125, -1.6516894102096558, 0.824423611164093, -2.5779149532318115, 0.27690696716308594, -0.964543342590332, 2.375340223312378, 1.6938873529434204, 1.3693780899047852, 1.8075913190841675, 1.021756649017334, 1.110182523727417, 1.300004005432129, 0.8589032292366028, -1.8094654083251953, -1.4163691997528076, -1.4843062162399292, 1.767439842224121, -2.843824625015259, 2.037569522857666, 0.632199227809906, -1.0112732648849487, -1.0825060606002808, 0.024268751963973045, -0.07972200959920883, -0.5393312573432922, -1.70503568649292, -0.6195736527442932, 1.9954843521118164, -0.2622590959072113, -0.3105144500732422, 1.3034398555755615, 0.4134305417537689, -0.17249929904937744, 0.6163101196289062, 0.4675286114215851, 0.9570460915565491, 0.29618215560913086, 1.7546275854110718, 2.357339382171631, -0.6939273476600647, -0.045883290469646454, -0.3787282407283783, 1.3878451585769653, 0.4867837131023407, -1.015579104423523, 2.469620704650879, 0.7902507781982422, -0.04885571449995041, 2.6170494556427, 1.3425980806350708, -3.12306547164917, -0.017801489681005478, 1.416887640953064, -0.03978612646460533, -0.8308550715446472, 0.3599967956542969, -0.1447855532169342, 1.1870527267456055, 0.09234585613012314, 2.844365119934082, -0.46980589628219604, -0.3281850814819336, -5.950384616851807, 0.21394528448581696, -0.8293213844299316, -1.3197418451309204, 0.7524033784866333, -4.177106857299805, -0.18241547048091888, -0.8983393311500549, -0.7980321049690247, 0.47912898659706116, 1.5873456001281738, -0.09809229522943497, 0.2470056265592575, 0.9939866662025452, 1.4111350774765015, -1.260932445526123, -0.113582082092762, -1.1306521892547607, 1.853888750076294, 1.4682464599609375, 0.09412094950675964, 0.4925355017185211, -0.8681795597076416, -2.7271227836608887, -0.09774792194366455, 1.9311827421188354, -0.419605553150177, 2.297581672668457, -0.4538179337978363, -1.4177274703979492, -1.470054268836975, 0.6324338316917419, -0.3129742443561554, -1.2133090496063232, -2.056863784790039, -1.296718955039978, 0.462503045797348, 2.418519973754883, -2.601439952850342, 1.6919561624526978, -0.2950255572795868, 1.3778976202011108, 0.2691342830657959, 0.34507301449775696, -0.45047423243522644, -0.5279719233512878, -0.6998659372329712, -0.02062627300620079, 0.7510169148445129, -0.007210531737655401, -0.22050049901008606, 2.2811946868896484, -1.6000486612319946, -0.9864646792411804, -1.1740902662277222, -1.5555510520935059, 0.042158834636211395, -2.408714771270752, -0.6941146850585938, -2.8313488960266113, -0.9221859574317932, 0.005829441361129284, -0.8263654708862305, 0.6983582973480225, -0.6197982430458069, 0.7851762175559998, 0.1535203754901886, 0.9907598495483398, 1.377655267715454, -0.3761102557182312, 0.38601601123809814, 1.7290890216827393, 1.4853339195251465, 0.18871620297431946, 0.5217180848121643, -1.2025411128997803, -3.248415946960449, -1.5926644802093506, -0.3426254391670227, 1.2467026710510254, -0.37092074751853943, 0.8030555844306946, 0.39889436960220337, -0.3129826784133911, -1.6960393190383911, -0.1369665563106537, -1.5052851438522339, 0.7340884208679199, -0.3120349049568176, -0.46351543068885803, -1.5922632217407227, 1.1807879209518433, -1.0774508714675903, -0.2377718836069107, -1.081503987312317, -2.18544602394104, 0.32859566807746887, 2.507730484008789, -1.6333787441253662, 2.482884645462036, 1.209950566291809, -0.1723979413509369, -0.5372232794761658, 0.3434540927410126, 1.0044357776641846, -0.41582855582237244, 1.6420722007751465, -0.5816636085510254, -3.354506731033325, -1.334158182144165, -0.6842655539512634, 0.7808983325958252, -2.030313491821289, -2.169381618499756, 0.4264959692955017, -0.4346531927585602, -0.6239020228385925, -0.3228098154067993, 3.0233957767486572, -1.5548739433288574, -0.472006231546402, 0.5705416798591614, 0.5305062532424927, 0.37530502676963806, 0.7177689075469971, 2.1995818614959717, -0.34878048300743103, 1.3277169466018677, -0.1632276326417923, -5.052221298217773, 1.2671034336090088, -0.08699208498001099, -2.1197621822357178, 0.8684853315353394, 1.3727216720581055, -1.16233491897583, -1.081447720527649, -0.028541872277855873, 2.126293659210205, 0.665372371673584, -1.6808220148086548, 0.8784337043762207, 2.362180709838867, 0.06184384599328041, -0.46403104066848755, 1.349539875984192, -0.9519774913787842, -0.4401065409183502, -0.08477620035409927, 2.6293017864227295, 0.2500830590724945, 3.100205659866333, -1.0615874528884888, -0.715066134929657, -3.219494581222534, -0.7134870886802673, 1.3054770231246948, 1.8432732820510864, -1.3751287460327148, 0.44112423062324524, 1.389978289604187, -0.5627428293228149, 0.45588138699531555, -2.0581095218658447, 2.006941318511963, -2.4918854236602783, 0.28786763548851013, -0.5218412280082703, 0.7579090595245361, -0.20093663036823273, -0.08291622251272202, 0.6483465433120728, 2.39357852935791, 0.2506246268749237, 0.7151603698730469, -0.661274254322052, -0.5403603315353394, -2.9798085689544678, -1.5462424755096436, -1.003885269165039, 0.5732040405273438, -3.0632870197296143, 1.7135818004608154, -0.019121015444397926, -1.5923221111297607, -0.5021968483924866, 0.9710349440574646, -0.4165872633457184, 0.19051681458950043, -1.8535020351409912, 1.2386853694915771, -0.5755376815795898, -1.5792137384414673, 0.6780622005462646, -0.5556114912033081, -1.108810305595398, -0.6256058812141418, -1.933523178100586, 1.652814507484436, -0.2748667895793915, -4.002296447753906, -1.1515957117080688, -1.0862492322921753, -6.626784324645996, 0.6086037755012512, -0.8462257385253906, 0.2980665862560272, -0.18310394883155823, -0.5844948291778564, 1.3683273792266846, -0.15985192358493805, 0.23185449838638306, 0.7926441431045532, 0.3295615017414093, -0.12913267314434052, -0.7955554723739624, 0.7488349080085754, 3.0780234336853027, 3.3254311084747314, -0.27526283264160156, 2.3119089603424072, 1.011788249015808, -3.0770885944366455, 0.43215411901474, -1.5298042297363281, 1.5397312641143799, -0.334533154964447, 0.8093821406364441, 2.418381452560425, -1.3408948183059692, 2.3132407665252686, -0.14664971828460693, 0.946730375289917, -0.4639107584953308, 1.2165501117706299, 0.7473862767219543, -0.11031408607959747, 0.48689383268356323, -3.2766528129577637, 0.10311825573444366, -2.9748244285583496, -0.5673705339431763, 1.2803303003311157, -1.478808879852295, 1.8853479623794556, -0.4871442914009094, 1.0774974822998047, 1.5344984531402588, -0.5537533164024353, -1.2534834146499634, -2.0439581871032715, -0.9618557691574097, -0.021035632118582726, -3.331763982772827, -0.23835787177085876, 0.3544577360153198, 1.4012876749038696, 1.4826741218566895, -0.6672770977020264, 1.094675064086914, -0.662922739982605, -0.16820980608463287, 0.2542332112789154, 0.8738424181938171, -1.336378812789917, 1.1468850374221802, -0.6872780323028564, -0.5370528697967529, -0.42901596426963806, 1.5762089490890503, 0.4784090518951416, -0.7752570509910583, 0.29591697454452515, 0.006720831152051687, -0.5435410737991333, -2.954209327697754, -0.25349077582359314, -1.4901130199432373, 0.24821452796459198, 0.17796042561531067, -3.175562620162964, 0.447964608669281, 1.3927912712097168, 0.20107969641685486, 0.3253813683986664, -0.528462290763855, -1.6635078191757202, -3.975025177001953, -2.1916842460632324, -0.30225157737731934, -1.5089640617370605, 1.3786171674728394, -1.2658871412277222, 1.8360408544540405, -0.7192330956459045, -0.16269534826278687, -0.7602254748344421, -1.5967092514038086, -1.2805335521697998, 1.2935889959335327, 0.41641661524772644, -2.4160985946655273, -0.27748212218284607, -0.3286731243133545, 0.38474851846694946, 1.0351369380950928, 2.737016201019287, 1.5616647005081177, 1.101912260055542, 0.8204759955406189, -0.8669130206108093, -0.25257933139801025, -0.7459702491760254, -1.1241912841796875, -0.5032333731651306, -1.5759332180023193, 0.9067305326461792, 0.05107565224170685, 1.7768511772155762, -1.0412901639938354, 1.1936992406845093, -0.5826668739318848, 1.313141107559204, -1.033768653869629, 0.9037140607833862, 0.20741382241249084, -0.09629173576831818, 0.4649522006511688, 1.0839489698410034, 0.3108638525009155, 0.0787876546382904, 1.069826602935791, -1.577282190322876, -0.8304091095924377, -1.2875510454177856, 2.5873005390167236, -1.5344911813735962, 0.6043655276298523, 1.4310466051101685, 0.3524196147918701, -1.0262646675109863, -0.44187024235725403, 0.1790090948343277, -0.6252474784851074, 1.0994226932525635, -0.5672310590744019, 0.36998459696769714, -1.1312540769577026, -0.7201394438743591, 1.6399978399276733, -0.4725104570388794, 1.311776041984558, 0.6223388910293579, 0.9204384088516235, 0.4144947826862335, 1.4521976709365845, 0.47924575209617615, 1.7521690130233765, 1.0554144382476807, -0.5381777882575989, 0.8565018773078918, -1.3074531555175781, -0.4865134060382843, -1.1087960004806519, 0.6457785367965698, -0.09903955459594727, 0.2911181151866913, 1.852160930633545, 1.717469573020935, -1.2953625917434692, -1.5979220867156982, 1.1348216533660889, -0.6678415536880493, 0.5679556727409363, -0.15089687705039978, 0.40791693329811096, -1.580603837966919, -0.25893086194992065, -0.29393839836120605, 0.6854649782180786, 0.5715694427490234, -1.5705457925796509, -0.3190681040287018, -0.8428931832313538, -0.8543785810470581, -0.46126288175582886, -0.6328279972076416, -0.14214858412742615, 2.2511708736419678, 0.363608181476593, -1.4318732023239136, -0.49615535140037537, -1.040177822113037, -0.19287212193012238, 1.3962393999099731, -0.5479450821876526, -1.0904812812805176, -2.0625336170196533, -1.3220176696777344, 0.5629263520240784, -0.8919689059257507, -0.6872240900993347, -0.3779812455177307, -0.6077254414558411, -0.06989457458257675, -0.028389010578393936, -0.195027694106102, 2.2183494567871094, -0.50234055519104, -0.644702136516571, -0.6903842091560364, 1.5928529500961304, -0.009533378295600414, -1.4176005125045776, -0.3663230836391449, -1.401517391204834, 0.8731875419616699, 1.2370400428771973, 0.6563100218772888, -2.07753324508667, -0.03307383507490158, -0.8997711539268494, -0.761396050453186, 0.5897625088691711, 0.5892267227172852, 3.0485637187957764, 0.6844597458839417, -1.9164992570877075, -0.4895819127559662, 0.07178904861211777, 0.33241912722587585, 0.9746308326721191, 2.6695752143859863, 0.8690300583839417, 1.1500368118286133, -2.0351057052612305, -0.21262530982494354, 4.157638072967529, 1.3398867845535278, 0.8867835402488708, -1.4096759557724, -0.9588669538497925, -1.9728039503097534, 0.8776890635490417, 1.3443498611450195, 0.04559027776122093, -1.4282498359680176, -0.2676692605018616, -0.8349907398223877, -2.910397529602051, 0.3653097450733185, 0.5943341851234436, -1.1211612224578857, 0.9423377513885498, 0.8947256207466125, -1.0148258209228516, -1.0152153968811035, 1.390150785446167, -0.22872444987297058, -1.0031460523605347, -1.322939395904541, -3.355029582977295, -1.6976803541183472, 0.6812119483947754, -1.3726799488067627, 0.5415275692939758, 0.027210596948862076, 0.6347682476043701, 0.5347542762756348, -0.6648145318031311, -1.588760256767273, -0.2977839410305023, 0.9026051163673401, 0.31800177693367004, -0.8510128855705261, 0.327587753534317, -0.40695834159851074, -2.1813201904296875, -2.2751922607421875, -0.30396705865859985, -0.7626033425331116, -1.2353017330169678, 1.8642195463180542, 0.07417359948158264, -0.0173637717962265, -0.05015111342072487, -0.904309868812561, -0.45953235030174255, 1.7242923974990845, -0.06484194099903107, -1.2541041374206543, 1.3101508617401123, 1.4232864379882812, 0.665920615196228, -1.4062399864196777, -0.4696451425552368, -0.19572877883911133, 0.33942970633506775, 1.6721175909042358, 0.15372620522975922, 0.9039501547813416, -1.4848095178604126, 2.667156457901001, -0.4133961796760559, 0.1045842170715332, -1.1142592430114746, -3.144233226776123, 0.2594609558582306, -0.6799036264419556, 2.4003219604492188, 0.474190890789032, 2.3656678199768066, -0.36656591296195984, -1.830031156539917, 0.7851704359054565, 0.33962687849998474, -1.61056387424469, -1.4357770681381226, -1.0389187335968018, -0.42771825194358826, 0.38104960322380066, -1.5885714292526245, -0.3239399492740631, -0.3390958309173584, 1.0448861122131348, 1.5757839679718018, -3.7368571758270264, -1.0868041515350342, 1.2050888538360596, -0.7156079411506653, 1.0398286581039429, 0.30565017461776733, 0.23555698990821838, -0.047343719750642776, 0.684800922870636, -2.4213016033172607, 0.3885684013366699, -0.11194414645433426, -0.879065215587616, 0.3005960285663605, 2.0981698036193848, -0.6669703125953674, -1.6189994812011719, 1.0009617805480957, -0.7873053550720215, -2.0848746299743652, 0.7533942461013794, 0.1105712428689003, -1.4078670740127563, 0.17821677029132843, -4.34473180770874, 0.23953045904636383, -1.9633532762527466, -0.49964433908462524, -0.9682289958000183, 1.7613601684570312, 3.9683873653411865, -0.9471872448921204, -0.7712170481681824, -1.9418352842330933, -0.15712635219097137, -2.278945207595825, 0.12696515023708344, 0.3647421896457672, -1.4295114278793335, 1.812875747680664, 0.5421193242073059, -0.29102200269699097, -0.6794769763946533, 2.6227591037750244, -0.5683690309524536, 1.4688949584960938, 2.9299182891845703, -1.3152755498886108, -0.10876554995775223, -1.1878381967544556, 0.28037914633750916, -1.6840436458587646, 1.5920509099960327, -1.785548210144043, 2.122323989868164, -0.20916439592838287, 0.6183523535728455, 1.1733274459838867, -1.135839819908142, -0.6662973165512085, 1.4992259740829468, -2.136855363845825, -1.4337276220321655, 0.8068862557411194, -1.6710805892944336, -0.5820517539978027, 0.5036649107933044, -1.8783422708511353, -0.9467825889587402, 0.06564787775278091, -4.090994358062744, 0.07141965627670288, -1.1040812730789185, 0.2867971658706665, -2.025472640991211, 3.0968637466430664, -0.27845582365989685, 2.707759141921997, 2.1326448917388916, -1.011518120765686, 0.33701077103614807, -0.5200458765029907, -0.23466414213180542, 0.8568166494369507, 0.24715067446231842, 2.2211203575134277, -1.8966845273971558, 0.36309605836868286, 1.723582148551941, -0.19809047877788544, 1.5992982387542725, -0.09971208870410919, -0.10268784314393997, 0.8570815324783325, 1.1798605918884277, 0.4180988371372223, 0.7174562215805054, 0.12476194649934769, 0.7139365077018738, 2.074883460998535, -0.4214620292186737, -0.11358395218849182, -0.6285896301269531, -1.6769121885299683, 0.9720752239227295, -0.9566248655319214, 0.4538053870201111, -1.7062506675720215, -0.12301989644765854, 1.6413344144821167, -0.3218771815299988, 1.8460216522216797, 1.4265568256378174, 0.18805651366710663, -0.3190893530845642, 0.44847527146339417, -3.1929802894592285, 0.39223843812942505, 0.6556668281555176, -5.261651992797852, -5.056717395782471, -1.9386824369430542, 0.40808650851249695, -0.13828670978546143, 0.5618566870689392, 0.5032123923301697, 0.5192421078681946, 1.3293561935424805, 1.5265190601348877, 1.189692735671997, 1.636809229850769, 3.4920084476470947, -0.7348800897598267, 0.005612244829535484, -0.0260099358856678, 0.31573018431663513, -0.3301008343696594, 0.901342511177063, 0.22833405435085297, -0.669307291507721, 1.5664740800857544, -1.6847341060638428, -0.3642199635505676, 0.6245474219322205, -0.6217486262321472, 1.438660740852356, -0.502903401851654, -0.36501777172088623, 0.6142686605453491, 2.9966702461242676, -0.2727624177932739, 0.1269271820783615, 0.10203316807746887, 0.23165203630924225, -1.4102286100387573, 3.145986557006836, -1.109134554862976, 0.9606544375419617, 0.10696163028478622, -0.699259877204895, 1.5500444173812866, 0.3014090061187744, -1.240093469619751, -0.5157244801521301, -0.9974798560142517, -1.553040623664856, 1.4491021633148193, -0.2780904471874237, 0.8371670246124268, -1.4232834577560425, 0.6247760653495789, -0.10028280317783356, -0.9186424016952515, -1.285507082939148, -0.03137533366680145, 0.7026181817054749, 2.72700834274292, -0.6291082501411438, 0.15368123352527618, -0.7712393999099731, 0.5185622572898865, 1.1361005306243896, 1.0718549489974976, 0.5145735740661621, 0.401990681886673, -0.050030771642923355, 0.593379020690918, 3.4227612018585205, 0.7363699078559875, -1.5311353206634521, 2.104452610015869, -0.6583995223045349, 5.70465612411499, 0.9151784777641296, -0.8279266953468323, -1.8508338928222656, -0.2876723110675812, 0.09366663545370102, 1.1992329359054565, 0.3489542007446289, -0.8672332167625427, 2.8109731674194336, -1.694111943244934, -3.4929912090301514, -1.856885552406311, -1.435680627822876, -2.9822700023651123, -0.12083523720502853, -1.9716882705688477, -2.4590041637420654, 0.2693854868412018, -1.883736252784729, -0.46809977293014526, -0.38566598296165466, 3.9752047061920166, 0.07101455330848694, -0.3032928705215454, 0.22238992154598236, 0.594127357006073, -5.299376964569092, -1.4189943075180054, -1.4926460981369019, 1.6202304363250732, 1.0035027265548706, 0.9346531629562378, -0.6888949275016785, 0.0962422639131546, -0.17277443408966064, -0.3442299962043762, 4.269221305847168, -1.0977612733840942, 1.6515499353408813, 1.260558009147644, -1.2532035112380981, -2.6422860622406006, -0.4011397659778595, -0.8250022530555725, 0.5782237648963928, 1.3405765295028687, -1.2752676010131836, -0.847994327545166, -1.2777870893478394, -0.9051859974861145, 1.6795471906661987, 0.5759140253067017, 0.7178165912628174, -1.6204702854156494, -2.8342020511627197, 1.1703698635101318, -0.1794136017560959, 1.4951196908950806, -0.5498469471931458, -0.710621178150177, -0.9479326605796814, 0.8485627174377441, 0.08397524803876877, 1.1937634944915771, -1.54133939743042, 0.926578938961029, 0.4452250599861145, 0.2496267855167389, -1.0103002786636353, -1.668450951576233, 2.462268829345703, -0.046175576746463776, -0.49310219287872314, -0.010691341944038868, 0.7631698250770569, -0.7851384282112122, 0.0947628766298294, -2.279736280441284, 0.42941150069236755, -1.2691553831100464, 1.116640329360962, -0.9469106793403625, -1.855767011642456, 0.7444389462471008, 0.34603843092918396, 1.8419278860092163, -0.9541047811508179, -3.011533260345459, -0.5364052057266235, -0.27914008498191833, 2.193849802017212, -2.3573713302612305, -0.9931573867797852, -0.860893189907074, -1.1689220666885376, 3.481701612472534, 0.5237307548522949, 0.6899821162223816, -1.0035077333450317, -3.0653769969940186, 1.5917145013809204, 1.063729166984558, 1.1260783672332764, 1.963340163230896, -0.8125501275062561, 0.6826492547988892, -1.2830215692520142, -1.6733918190002441, -0.5100489854812622, 1.5180673599243164, 0.5144354701042175, 1.1478042602539062, 0.4719274342060089, -1.3743796348571777, 1.5424789190292358, 1.038347840309143, 2.175366163253784, 0.7270437479019165, -2.0979928970336914, 0.9171277284622192, -0.19160239398479462, -0.8979634642601013, 0.12677940726280212, -0.060513224452733994, 1.4383617639541626, 1.7199203968048096, -1.8969706296920776, -0.16538771986961365, -1.2203588485717773, 0.038409274071455, 0.7823143601417542, 1.1209229230880737, -0.04129885137081146, -0.6631506085395813, -0.2477857768535614, -0.2987973093986511, 6.1075873374938965, -1.5450177192687988, 0.9094001054763794, 1.0033951997756958, 0.25628936290740967, 1.3830534219741821, 0.6043553352355957, 3.12666392326355, -0.5216071009635925, 1.5371407270431519, 2.6206750869750977, 0.8253676295280457, 0.7532376646995544, -0.07784271240234375, -0.30758047103881836, -1.3555103540420532, -1.1100316047668457, -0.9563891887664795, -0.0775194987654686, 0.31338852643966675, 0.5582906007766724, -0.7437366843223572, -0.9761170744895935, 0.9242588877677917, 0.7829161882400513, -0.8239083886146545, -1.1751625537872314, 1.0539424419403076, 2.14689302444458, 1.6796156167984009, 1.0995391607284546, 1.932396411895752, -2.2789173126220703, 1.2813133001327515, 1.6900662183761597, -0.09148669242858887, -0.6337677240371704, 2.2963809967041016, 1.4427415132522583, 0.6639236211776733, 2.209735870361328, 0.3396603465080261, -2.7555248737335205, -0.8254802823066711, 1.3026623725891113, 2.4662647247314453, -1.3202089071273804, -1.8923239707946777, -0.7541792988777161, -0.41634657979011536, 0.1289214938879013, 2.0296871662139893, 0.2637037932872772, 1.3059656620025635, 0.4713347852230072, 1.2997840642929077, -0.258512407541275, -0.5883370637893677, 0.1764003485441208, -0.47263023257255554, -0.3574920892715454, -4.181034088134766, -1.1557157039642334, -0.29228609800338745, 0.5401687622070312, 0.029941348358988762, -0.09092388302087784, 1.6703763008117676, 0.04171954467892647, -0.31054940819740295, -0.40093767642974854, -1.7006033658981323, 2.002779483795166, 0.05180187150835991, 0.4164482653141022, -0.6204590201377869, 0.8173771500587463, -2.3332161903381348, 0.4585385024547577, -0.020927032455801964, -0.06000426411628723, -0.05251690372824669, 2.378666639328003, 0.3578454852104187, -0.5482103228569031, 1.39051353931427, 0.9064372181892395, -0.5925901532173157, 0.7921409010887146, -1.2081022262573242, 0.08927454054355621, -0.23172198235988617, -1.9288206100463867, -0.42390888929367065, -2.1412220001220703, 1.6710883378982544, -1.1873303651809692, 1.5496695041656494, 1.330776572227478, -0.6039979457855225, 1.8379616737365723, 1.1870986223220825, 0.5579753518104553, 1.8688108921051025, 1.0249451398849487, -2.119354248046875, -1.20047926902771, 0.5590900778770447, -1.1784244775772095, -0.5006826519966125, 3.0309348106384277, -1.1433744430541992, 2.17618989944458, -0.9923620820045471, -2.0305278301239014, -1.4136234521865845, -0.47532474994659424, 1.4243245124816895, -0.34324324131011963, 0.10351068526506424, -0.439392626285553, -0.8188915252685547, 0.8698177337646484, -0.7848818302154541, -1.7884522676467896, 1.2363016605377197, 1.0625042915344238, 0.9984332323074341, 1.6983094215393066, -1.815029263496399, -0.8221663236618042, 0.05494439974427223, -0.15458373725414276, 1.3883227109909058, 2.9936556816101074, -1.915345549583435, -0.48943910002708435, -3.4730918407440186, -3.107445478439331, 1.5947011709213257, -2.298226833343506, -1.140825867652893, 0.9571939706802368, -0.15249128639698029, -1.8636231422424316, -0.3876555263996124, -0.5002975463867188, 0.08564253151416779, -2.3145883083343506, -0.0674334317445755, -0.848222017288208, -0.45607027411460876, -1.8376275300979614, 0.271095335483551, 2.081904888153076, -1.1150133609771729, 0.28747934103012085, -0.5100080966949463, -0.31571173667907715, 2.0234875679016113, 2.3653512001037598, -0.595399022102356, 0.6303644180297852, -1.1060336828231812, 3.604746103286743, -0.7575949430465698, -0.9141921997070312, -0.6875700354576111, -0.5032631158828735, -1.1674374341964722, 0.49782490730285645, 2.4618449211120605, -0.5100997686386108, -0.05444146320223808, 1.5750007629394531, -0.9847256541252136, 3.4533183574676514, -0.7243689894676208, -1.3849916458129883, -1.3038091659545898, -0.6240057349205017, 1.0539636611938477, -1.4076493978500366, 1.722617506980896, 0.8758316040039062, -1.3078569173812866, 1.4320323467254639, 0.02523243986070156, 1.0125563144683838, 0.1733589470386505, -0.9509767293930054, -0.10817122459411621, -0.9233194589614868, 1.6603871583938599, 1.9675686359405518, 3.5321640968322754, 0.04472770914435387, 0.8216757774353027, 0.024731678888201714, 0.5062718391418457, 1.3309839963912964, 0.47295206785202026, -0.5641800165176392, 0.10957389324903488, 1.0154136419296265, 0.6534199118614197, 1.6295626163482666, 3.2676644325256348, 0.2265714704990387, 0.8935802578926086, 0.9925765991210938, -0.5856419205665588, 3.317446708679199, 2.305643320083618, 0.8294320702552795, -0.017694484442472458, -0.3304680287837982, -0.039463527500629425, -1.4958490133285522, 1.4134448766708374, -1.3277559280395508, 0.875035285949707, -2.4330458641052246, 0.4168216586112976, 0.030303893610835075, -0.6298149228096008, -0.26183196902275085, -0.15142588317394257, 1.0584886074066162, -2.9447052478790283, -0.28959113359451294, -0.6136434078216553, -0.9578399062156677, -0.11951156705617905, -0.2700468897819519, -0.690568745136261, -0.2317095249891281, -0.2912701964378357, -0.08248625695705414, -0.7093273997306824, 0.40018728375434875, 0.17581963539123535, 0.07403799146413803, 1.7691941261291504, 3.0280098915100098, 1.3574835062026978, 0.059968508780002594, -0.8030521273612976, -0.029537063091993332, 0.6613892912864685, -1.541106104850769, -0.45385977625846863, -1.8316227197647095, 0.16329242289066315, -2.4645907878875732, 0.12880939245224, 1.7649197578430176, -0.56751549243927, 0.11116805672645569, 0.6268647313117981, 0.7703966498374939, -0.41861286759376526, 2.0327248573303223, 0.7652528882026672, 1.3230348825454712, -0.820112943649292, 1.727892518043518, -0.377399206161499, 3.218057870864868, -2.597996950149536, -0.5058363080024719, -0.42988821864128113, 0.8008670806884766, 1.4703803062438965, -0.11833228915929794, -0.20794153213500977, 0.41800668835639954, -0.16673383116722107, -0.5690270066261292, -1.2463867664337158, -0.6624801158905029, -0.3696221709251404, -1.2742408514022827, -0.21665623784065247, 0.9644657969474792, -1.152963638305664, -2.0284812450408936, 0.24948978424072266, -0.14581556618213654, 2.193406581878662, -0.48534008860588074, -0.3359670639038086, -2.4618988037109375, -1.261533260345459, 0.351305216550827, -0.7561586499214172, 1.6206737756729126, 2.2842299938201904, -0.3265606462955475, 1.6759560108184814, 1.0003405809402466, -2.0290634632110596, -0.5272178053855896, -3.335452079772949, -0.5472918748855591, 0.5302579998970032, -0.5220498442649841, 0.3978441655635834, -1.0923547744750977, -0.22065289318561554, -1.1086996793746948, 0.9885046482086182, 3.9966373443603516, 1.5629044771194458, -0.0767895057797432, 0.6132257580757141, -1.6965748071670532, 0.9465972781181335, 0.2328438013792038, 0.8587104678153992, -1.6723161935806274, -1.6394356489181519, -1.1001341342926025, 1.0661059617996216, 0.6617889404296875, 0.6580915451049805, 1.758730173110962, -0.02619428001344204, -1.2645963430404663, 0.09200805425643921, 1.0131726264953613, -3.9388153553009033, 0.5919673442840576, 0.3803706169128418, 0.14735622704029083, -5.092191696166992, 0.8636597394943237, -0.5998678207397461, 1.2465648651123047, -0.4681326150894165, -0.27768608927726746, 0.9836071729660034, -0.8560266494750977, 0.3867225646972656, -0.24844485521316528, 1.1123552322387695, -1.793046474456787, 2.5833215713500977, -1.3698276281356812, 1.5995806455612183, -1.929329514503479, 1.1047303676605225, 2.1648411750793457, -0.17132584750652313, 0.18956005573272705, -1.3451699018478394, -2.140852451324463, -0.7511403560638428, -0.8398230671882629, 0.1335236132144928, 0.2271203249692917, 0.30769556760787964, -1.3175113201141357, -1.284062385559082, -3.3443219661712646, 0.38118618726730347, 0.08092131465673447, 1.6218526363372803, -1.468845248222351, -3.6890602111816406, 1.6298359632492065, 1.1902624368667603, 2.114896535873413, -0.728062093257904, 0.23982083797454834, 0.6435340046882629, 0.34518682956695557, 1.1262612342834473, 0.5400581359863281, -0.4676329791545868, -0.6050928831100464, 0.2975991666316986, 0.12744443118572235, -0.6450710892677307, -1.2926613092422485, -0.0482032373547554, 1.9702988862991333, -0.5325635671615601, -0.6193856000900269, -0.12263593822717667, -2.5026204586029053, -1.1532355546951294, -0.2882719933986664, -0.014851951040327549, -0.9560694098472595, -2.2234814167022705, -0.25118106603622437, 1.25832200050354, -2.287426710128784, -0.3068464994430542, 0.9609581232070923, -1.510694980621338, 1.3895224332809448, 2.077185869216919, -0.2850577235221863, -0.09520500898361206, 1.1649768352508545, 0.7802404165267944, 0.2201300859451294, 0.11480318754911423, 2.868943691253662, -0.3895931839942932, 0.8601961135864258, 2.155585527420044, -0.7880848050117493, 0.5890827775001526, -1.292598009109497, 0.6859890818595886, -0.7015364170074463, -1.2684338092803955, 2.229485273361206, 1.438957691192627, 0.6813127398490906, 0.23780123889446259, 0.5385957360267639, -1.9333873987197876, 1.0371243953704834, 1.7844533920288086, 0.12485887855291367, -1.5558818578720093, 0.9519289135932922, -0.7347231507301331, 0.5746033191680908, 0.6324082612991333, -2.0148885250091553, 2.4598214626312256, -0.675131618976593, 3.6917545795440674, -1.1207631826400757, 0.7807437777519226, -1.951167345046997, 1.417878270149231, 2.965634822845459, 1.1556942462921143, -0.5773329138755798, 1.069119930267334, -0.18139463663101196, 0.4327522814273834, -0.6524130702018738, -2.6418986320495605, -1.9375278949737549, 0.15074515342712402, -0.9708108305931091, 0.9142857193946838, -2.0230486392974854, 3.899045467376709, -0.36628031730651855, -1.043022632598877, 0.6463730335235596, 0.4374816417694092, -1.901099443435669, -1.1635346412658691, -1.110581874847412, -0.12873134016990662, 1.1942404508590698, -0.8320794701576233, -4.5100226402282715, 0.3371661305427551, 1.4255578517913818, -0.8001108169555664, -0.032131560146808624, 3.8264682292938232, -0.5765700340270996, 0.5673713088035583, -1.776821255683899, -1.0750006437301636, 0.039890553802251816, -3.271397590637207, 6.189054012298584, -0.744626522064209, 0.380748987197876, 2.759134292602539, 0.05091957002878189, -1.0868016481399536, -1.1444975137710571, -0.3189045190811157, 0.6918163299560547, -1.728808879852295, -0.30930253863334656, 0.5382334589958191, -0.4268421530723572, 0.740963339805603, 0.6062629222869873, -2.9615495204925537, 1.6209697723388672, 1.067155122756958, -2.497704029083252, 1.2236027717590332, 1.4595801830291748, 1.6629894971847534, -0.24837277829647064, 3.7532601356506348, 2.402534008026123, -0.27702316641807556, -1.6668474674224854, -1.5196605920791626, -1.3513507843017578, -1.717695713043213, -0.27890124917030334, 1.4827901124954224, -1.1888126134872437, -0.4072982668876648, -0.2777511477470398, 0.6795817613601685, -0.7571399211883545, 3.380767345428467, 0.09683029353618622, 0.2528264820575714, 2.0274558067321777, -0.7856038212776184, -1.4631072282791138, -0.5582008957862854, 0.1480567306280136, 0.6452262997627258, 1.8143616914749146, -1.015027642250061, -2.0519425868988037, 0.04362212121486664, 1.4946258068084717, -0.2839522659778595, -0.647564709186554, -0.557767927646637, 0.1774912029504776, 1.462480902671814, -1.2484642267227173, -0.911720871925354, 1.273393988609314, -0.8370766043663025, 1.0382530689239502, -0.34284135699272156, 0.10764393210411072, 0.7312974333763123, 1.1802117824554443, -0.4923705756664276, 2.060831069946289, -1.0685514211654663, -0.9035617113113403, -0.25251272320747375, -2.007957696914673, -1.2853585481643677, -1.5034373998641968, 0.11544394493103027, -1.6111727952957153, -0.1133400946855545, 0.3838781416416168, 3.90090274810791, 0.779266357421875, 0.00693752896040678, -0.8340885043144226, -0.18485833704471588, 2.9877207279205322, -0.4965360164642334, -0.9494770169258118, -3.133784532546997, 1.022720217704773, -1.484872579574585, -2.593334436416626, 1.8963497877120972, 0.2066400945186615, 0.16485437750816345, 2.136793851852417, -1.4127120971679688, 0.9117878079414368, -1.998721957206726, 0.6992210149765015, 0.3787159025669098, -1.2396756410598755, 0.49617549777030945, 0.6750690340995789, 0.6648108959197998, -0.29459476470947266, 1.142527461051941, -0.9266757369041443, -3.2820816040039062, 1.5292377471923828, 1.7218736410140991, 0.4098595082759857, -1.2624398469924927, 0.5287312269210815, 1.9563980102539062, 2.7563815116882324, -1.6897691488265991, 3.654918670654297, -0.3462108075618744, 0.2095506191253662, 0.053329430520534515, 0.15436215698719025, 0.3423183858394623, -1.0397913455963135, -0.6457964181900024, -2.4534099102020264, 0.21036797761917114, -1.5697673559188843, -2.9955782890319824, -0.1626177728176117, -0.4118574559688568, 1.6892170906066895, 0.89940345287323, -0.9454144239425659, -1.3369885683059692, -0.02026965096592903, -0.4967345893383026, 0.8789601922035217, 0.617778480052948, 0.2041195183992386, -0.5651078820228577, 0.21543900668621063, -0.8319731950759888, 1.347630500793457, -6.415892601013184, 3.337533950805664, 0.49197933077812195, 0.8592319488525391, -0.8697947859764099, -2.0869381427764893, 3.0572586059570312, -0.8632948398590088, -0.5056756138801575, 0.7555558085441589, 0.6673120260238647, -1.4517968893051147, -0.701657235622406, -0.30043187737464905, -0.40870586037635803, -0.11934226751327515, -2.787231683731079, -1.3511626720428467, -2.712592124938965, 1.5401089191436768, -0.01384841836988926, -1.0532454252243042, -0.8640741109848022, -0.5049043297767639, 0.5688037872314453, 1.376906394958496, 2.468014717102051, 3.0420260429382324, -0.045120783150196075, 1.3480830192565918, 0.8896507620811462, 0.5893454551696777, 0.3174736499786377, -0.022604692727327347, -0.4583990275859833, 0.7105838656425476, 0.31724950671195984, -0.7603034973144531, -1.4107887744903564, -0.9337335824966431, -0.5966812968254089, -2.3582613468170166, -1.2388689517974854, -1.7017549276351929, -0.28927066922187805, 0.9912077188491821, -0.3200368285179138, 1.6482110023498535, -1.7816123962402344, -0.2052524983882904, 1.526960015296936, -0.6658144593238831, 1.7667553424835205, -0.3364953398704529, -2.8156960010528564, -0.3045891523361206, 1.3712012767791748, 1.7691892385482788, -0.5764877200126648, -0.5141450762748718, 0.8107253313064575, 1.8224440813064575, -3.0857224464416504, -0.9820992350578308, -0.0695406049489975, -0.47362715005874634, 0.9231453537940979, -2.2746682167053223, 0.30298930406570435, 2.8031556606292725, -1.344825029373169, -1.803469181060791, 0.8250601887702942, 1.627610683441162, 1.4530929327011108, 1.6940232515335083, 0.5018702149391174, -2.4092657566070557, -0.22793032228946686, 3.9878060817718506, 2.2782931327819824, 1.511225700378418, 0.14281003177165985, -0.8959864974021912, 0.9351397156715393, -2.050172805786133, -1.0338671207427979, -2.982884168624878, -1.9978561401367188, 6.1945977210998535, 1.920856237411499, -2.0209240913391113, -1.331756353378296, -1.1699296236038208, 2.1138007640838623, -0.09078232198953629, 2.1437795162200928, 1.5548917055130005, 1.240199327468872, -0.8897247910499573, 1.6907155513763428, -1.6416548490524292, -0.19418546557426453, 1.7784900665283203, -0.1814192235469818, 1.1016433238983154, 2.6229665279388428, 2.296520233154297, 1.1070404052734375, 0.45606398582458496, 1.1934067010879517, 0.8307900428771973, -1.6925232410430908, -1.8366152048110962, -0.8399804830551147, -0.357040137052536, 0.27658048272132874, -0.2892081141471863, 0.9518900513648987, -0.48650896549224854, 1.7402366399765015, -0.6293547749519348, -0.3963238596916199, -0.8722169399261475, 0.09539870172739029, -1.5312528610229492, 0.14878705143928528, -1.693534255027771, -1.199873685836792, 0.18486422300338745, -2.086568593978882, -0.8371956944465637, -1.92947518825531, -2.132598400115967, -0.3837335407733917, -0.18917134404182434, -1.8541568517684937, 0.6644979119300842, -0.856644332408905, 0.5650189518928528, 0.11841538548469543, -0.677823007106781, -1.5535715818405151, -3.4947197437286377, -0.164117693901062, 0.09717806428670883, 1.7128446102142334, -1.892349123954773, -0.935825765132904, -0.13878680765628815, 0.9524029493331909, 2.777491331100464, 1.3187339305877686, 0.6693404912948608, 1.9813114404678345, 0.9378648996353149, 1.551194429397583, 0.772253155708313, -1.1638822555541992, 1.8698972463607788, -2.5077664852142334, 2.0185229778289795, 1.1724013090133667, 0.45026928186416626, 1.577518105506897, 2.2280375957489014, 2.2283108234405518, -0.009927096776664257, 1.1863948106765747, -0.32508352398872375, 0.7868313789367676, -0.2423427253961563, 4.275184154510498, 1.1658694744110107, -1.2259271144866943, -2.2456257343292236, 1.5248860120773315, -0.4732491672039032, 0.205753892660141, -0.08183436095714569, 1.722003698348999, -1.9824329614639282, 1.0167063474655151, -1.2737019062042236, 0.8170700669288635, 1.9566965103149414, -1.0812574625015259, -0.8426579833030701, -0.5096819400787354, -0.7162660956382751, 0.667796790599823, -0.8864023089408875, -0.5092134475708008, -0.32587674260139465, -1.3355350494384766, 1.3576470613479614, 1.0649681091308594, 0.6340703964233398, 0.9373540878295898, -0.4381830394268036, 1.6345829963684082, 0.281259685754776, 1.1173992156982422, -0.404934823513031, -0.7477344274520874, 3.3479256629943848, -1.945115566253662, 0.20559294521808624, -0.14349345862865448, -1.5029302835464478, -0.6331200003623962, -2.464186906814575, -0.14728674292564392, -0.5532339811325073, -1.6046234369277954, -0.15184815227985382, -2.0425117015838623, -0.47459647059440613, -0.0933031365275383, 2.604994773864746, -0.04107262194156647, 0.6750248074531555, 1.0459635257720947, -1.340600848197937, -0.34228602051734924, 0.327525794506073, -1.883866310119629, 1.6531141996383667, -0.22126342356204987, 0.2963819205760956, -0.8264309167861938, -0.46075358986854553, -1.4225398302078247, -0.6266973614692688, -0.3155415952205658, 0.32014110684394836, -0.14926575124263763, 1.6180307865142822, 0.8990312814712524, 1.4879531860351562, 1.4572118520736694, 0.7522250413894653, -0.6046669483184814, -1.3997739553451538, 0.7878662943840027, -1.4989360570907593, -1.1017704010009766, 0.6055254936218262, -0.9923242330551147, -0.6093637943267822, 3.0684220790863037, -1.0460351705551147, 5.7728681564331055, -0.3448794484138489, 3.966775417327881, 2.101059913635254, -0.06731829047203064, -1.1012494564056396, -0.33558064699172974, -0.4159996807575226, 1.6443052291870117, 1.3845264911651611, -1.0708965063095093, -0.7283512949943542, -1.012491226196289, 0.37810152769088745, -1.779741644859314, -1.0291377305984497, -0.1415710598230362, -0.6279839873313904, 1.5048234462738037, -0.5191736817359924, 0.7351923584938049, 0.11812815070152283, 0.7110682725906372, 1.7340143918991089, 0.7136310935020447, -1.8019050359725952, 0.35286757349967957, -1.0316671133041382, -1.1673893928527832, -0.20351137220859528, 0.37973305583000183, -0.5646514296531677, -2.6928558349609375, -0.21374551951885223, -4.706504821777344, -0.9989141225814819, 0.7382999062538147, -1.0759987831115723, -0.6787937879562378, -0.5721335411071777, -1.5121616125106812, 0.7827112078666687, -0.016832418739795685, -0.2641347646713257, 0.6235513091087341, 0.8153610825538635, 0.18937501311302185, -2.046046257019043, -0.1963762789964676, -1.7806538343429565, -0.3867022693157196, 3.0970165729522705, 0.015269442461431026, 1.070925235748291, 0.7366203665733337, 1.1317367553710938, -2.512545108795166, 1.0869909524917603, 1.3521133661270142, -1.167555332183838, 1.6725168228149414, 0.6364412307739258, 0.08589719235897064, 2.3050220012664795, 0.5686280727386475, 1.1574726104736328, -0.8498120903968811, 0.6233046650886536, -1.309272289276123, -0.1934622824192047, 0.5418347716331482, -0.475033700466156, -1.3034791946411133, -1.382947564125061, -1.551189661026001, 1.0878688097000122, -1.013075351715088, 1.3304439783096313, -0.6174651980400085, -1.263991355895996, 0.8815600872039795, -3.4245834350585938, -1.1118927001953125, 0.04325902462005615, 1.4288712739944458, 0.3321026563644409, -1.387593388557434, -0.2319251149892807, -3.2556543350219727, 0.533240020275116, -0.42013341188430786, -0.8104181289672852, -1.4017987251281738, -0.26239147782325745, 0.4868161976337433, -0.38850298523902893, 1.2452224493026733, 0.18043343722820282, -0.9440664052963257, -0.5278728604316711, -0.301478773355484, 2.6090433597564697, -0.84136962890625, 1.9585877656936646, -0.9365788698196411, -1.5175505876541138, 3.5542290210723877, -2.855915069580078, 2.111241579055786, 1.3685436248779297, 0.8594136834144592, 2.905117988586426, -1.0697264671325684, 1.4466174840927124, -0.01662234030663967, 2.1741864681243896, 0.9161869287490845, 1.7944326400756836, -1.295198917388916, -1.7503745555877686, -0.2160971760749817, 1.9957168102264404, 0.07886423170566559, -1.2166541814804077, 0.7134193181991577, -0.14110712707042694, 0.005636171903461218, -0.4350471794605255, -0.1362585425376892, -0.7006964087486267, 0.48655200004577637, 0.6344117522239685, 1.573074221611023, -0.03344472497701645, 1.42559015750885, -2.363816022872925, 2.65684175491333, -0.5757071375846863, -0.2785501480102539, 1.724716067314148, 1.718153953552246, -0.18333104252815247, 0.04168281331658363, 2.3736681938171387, 0.4117938280105591, 1.413314700126648, -0.24700090289115906, 1.2963240146636963, -2.6907098293304443, 2.244144916534424, -2.938037872314453, -0.9126833081245422, -0.44319361448287964, 1.1252089738845825, 0.005342761054635048, -0.23784539103507996, -1.4233818054199219, 1.178695559501648, -0.6924985647201538, -0.61179518699646, -1.1616168022155762, 0.3298356235027313, 1.2743958234786987, 0.16430415213108063, 0.2862135171890259, -1.1335428953170776, 2.4669487476348877, 0.09449420869350433, -0.240299254655838, -2.978803873062134, -0.05947943404316902, 0.1872832030057907, -0.4338001608848572, 0.46160975098609924, -0.3002709150314331, -0.5158358812332153, 0.5889231562614441, 2.544823169708252, -0.22496455907821655, 3.1403419971466064, 0.8371893763542175, -1.038736343383789, -1.1383686065673828, -0.09396417438983917, 1.4465117454528809, -2.0346291065216064, -2.035714864730835, -1.712060809135437, 1.1434414386749268, 2.005418300628662, 0.25259724259376526, -1.790637493133545, 0.6709036231040955, 1.429002285003662, 0.90779709815979, 0.23937508463859558, 0.19365623593330383, 0.6738441586494446, 0.8722023367881775, -0.19729766249656677, -0.22220197319984436, 0.6103985905647278, -0.839682400226593, -0.29123905301094055, -1.081394076347351, 0.08513537049293518, 0.8117035031318665, 0.5368605852127075, 1.9373208284378052, -0.2755005955696106, 2.233499765396118, -2.1068828105926514, 0.24405933916568756, -0.3852543830871582, -0.9398829936981201, -1.3642512559890747, -0.040717028081417084, 1.0217968225479126, 2.095989465713501, 0.5933087468147278, 0.7526625990867615, 1.6371240615844727, 1.685599446296692, -1.0945552587509155, -1.036341905593872, -1.109961986541748, 0.9965566396713257, -1.698421597480774, -1.07206130027771, 0.5691885948181152, 1.1838338375091553, -1.9960490465164185, -1.5392853021621704, -0.03707659989595413, 0.1845826655626297, 3.2222447395324707, -0.5123529434204102, -1.428665041923523, 0.9963464736938477, 1.8461982011795044, 1.7654483318328857, -2.108962297439575, -0.14336897432804108, -1.9356530904769897, -0.2568678557872772, -0.3196631371974945, 0.5243155360221863, -0.6719242930412292, -0.6356703639030457, -0.3347419500350952, 0.3399193584918976, 0.322084903717041, -0.9493827819824219, -0.7879641056060791, 2.947866439819336, 0.18810097873210907, -1.2430837154388428, 0.20522107183933258, -1.0955570936203003, -1.4361263513565063, -0.09836854040622711, 5.412469387054443, -1.8129750490188599, 2.849832057952881, 0.8326276540756226, -1.2553821802139282, 1.7114222049713135, 1.6693580150604248, -1.8744255304336548, -2.079890012741089, -0.5905487537384033, -0.38357317447662354, -3.0975639820098877, -1.1697847843170166, -1.235705852508545, -0.10063132643699646, 2.5098018646240234, -0.5485010147094727, 2.399733304977417, 1.0477638244628906, 0.18811199069023132, 1.6373649835586548, 1.512695550918579, 0.7056918144226074, 0.9721383452415466, -0.46370986104011536, 0.769787609577179, 1.4446104764938354, -0.7258731126785278, 0.0705202966928482, -0.19678905606269836, 0.4552379250526428, -0.9440842866897583, -2.3152883052825928, 1.2847681045532227, 3.6463356018066406, -0.05378454923629761, -0.8791522979736328, -4.499214172363281, 0.3715285360813141, -0.6571766138076782, -1.6771212816238403, -4.381999492645264, -0.16473735868930817, 2.0732030868530273, 1.2767374515533447, -0.43608489632606506, 1.147077202796936, 0.8075709342956543, -4.2326507568359375, -0.437258780002594, -1.236769676208496, -0.4195466935634613, -1.610443353652954, 0.01983683928847313, 0.20186099410057068, 1.2633140087127686, 0.2483072578907013, -0.17870375514030457, 0.5155739188194275, -0.004534680861979723, 0.9801005721092224, 0.5385815501213074, -0.7740228772163391, 0.04978128895163536, -0.23856747150421143, -0.6422476172447205, 1.296167016029358, -1.0472419261932373, 0.9346479773521423, -0.004643641412258148, -0.5453738570213318, -0.9672858715057373, -0.7593546509742737, 0.34975454211235046, -1.6650798320770264, 1.0254379510879517, 2.1870951652526855, 0.41030552983283997, -0.506171703338623, 1.145792007446289, 2.335444450378418, -0.26185131072998047, -1.5618529319763184, 1.1515264511108398, -0.2724606692790985, 0.2690899074077606, -0.3666304647922516, -1.0245741605758667, -2.220126152038574, -1.1483781337738037, -0.5027710199356079, -0.5117045044898987, 0.6071869730949402, -0.23467989265918732, 0.5706071257591248, -1.2538371086120605, 0.8630226850509644, 1.2962899208068848, 0.03993741795420647, 0.028290867805480957, 0.30528515577316284, 0.7269195318222046, -0.4630299210548401, 0.3346112370491028, 0.4467201232910156, 0.3336803615093231, -0.6605372428894043, -0.13709519803524017, -0.6718617081642151, -2.915863513946533, -1.096015214920044, -0.9925172328948975, -0.9861053228378296, -1.0251654386520386, -0.2565816044807434, 0.15027020871639252, 0.05887286737561226, -1.6843934059143066, -1.3611247539520264, -0.137071892619133, 1.2243496179580688, 1.157602071762085, 1.5840893983840942, -1.7137565612792969, 1.7245149612426758, 1.3529019355773926, 0.1991676390171051, -1.127448558807373, -0.2832562029361725, 0.6431775093078613, 0.2719865143299103, -1.823011040687561, -0.5747594833374023, 0.9628626704216003, -0.22142933309078217, 0.4388905465602875, -0.0839683935046196, -0.8503283858299255, 0.17725087702274323, -1.6950461864471436, 1.3916500806808472, -2.2909412384033203, 1.1368902921676636, -0.8836752772331238, 0.04951255023479462, 2.6471195220947266, -0.013765635900199413, 0.2261628955602646, -2.7243428230285645, -0.2261379510164261, -0.36419397592544556, -1.2308670282363892, 0.5636751651763916, 0.39487770199775696, 0.6713594794273376, -0.07837431132793427, -0.213902547955513, 0.13097093999385834, -1.738394856452942, 0.5527600049972534, 1.3649977445602417, -0.6405871510505676, -0.4303164780139923, 0.18142645061016083, 2.595092296600342, 1.1913402080535889, 1.6931853294372559, 0.7637936472892761, 0.6521335244178772, -0.9690970778465271, 1.5188014507293701, -1.088822603225708, 1.4723076820373535, 1.2274501323699951, -0.5349786281585693, 1.810049057006836, -1.185004711151123, -2.141392230987549, -1.1520169973373413, 0.7446504831314087, 1.0454373359680176, -0.12503956258296967, -0.363922655582428, -0.020548798143863678, -3.6826224327087402, 0.038803741335868835, 2.0711629390716553, -0.5085027813911438, -1.6200625896453857, 0.49398183822631836, -0.36209437251091003, -0.26181092858314514, -0.4101332426071167, -1.2915838956832886, 0.4124435484409332, 0.5004212856292725, -1.3103697299957275, -1.770029902458191, -0.9164972901344299, 0.6052117347717285, -0.9954919219017029, 0.9769843816757202, -0.40321052074432373, 0.5052682757377625, 1.893373727798462, -1.39350163936615, 2.187086343765259, 1.8632251024246216, 0.30286386609077454, 2.824629068374634, -0.28529852628707886, 1.255516529083252, -1.5442626476287842, 2.206183433532715, -1.07411789894104, 0.002238558605313301, 1.0992991924285889, -0.028554260730743408, 0.3481363356113434, 1.0970326662063599, -2.1392276287078857, 1.3760865926742554, -2.336089849472046, -0.9120510220527649, 1.1621044874191284, -0.10059475898742676, -1.623201847076416, -0.11349742114543915, -0.7090825438499451, 1.634405255317688, 0.6957477927207947, 1.1299455165863037, 1.1396801471710205, 0.3925633132457733, -0.05486247316002846, 3.9327621459960938, 1.1898977756500244, 0.15685628354549408, 1.39373779296875, 0.23037217557430267, -0.11922411620616913, 0.14417503774166107, 0.9091958999633789, 0.6244110465049744, -2.270167112350464, 1.642086386680603, -0.1931247115135193, -0.17031291127204895, 0.0159776471555233, 0.8603450059890747, 2.0226356983184814, -0.5830772519111633, 0.6471587419509888, -1.4879546165466309, -0.7989405393600464, 0.6681789755821228, -0.9870601892471313, -0.7510910034179688, -0.49040356278419495, 1.3214728832244873, 0.4941052198410034, -1.1957260370254517, 0.5662062764167786, 0.22822222113609314, -1.6677708625793457, -1.3878434896469116, 0.673347532749176, -0.3667544424533844, -0.847619891166687, -0.7028186321258545, 2.2161178588867188, 1.1598488092422485, 3.836616039276123, 2.1410200595855713, 0.7164597511291504, -1.535177230834961, 3.744892120361328, 1.1868007183074951, 1.205260992050171, -0.3165084719657898, -0.9430076479911804, -0.9086549878120422, 0.7622615098953247, -2.3016457557678223, 0.03493819013237953, 1.3345575332641602, -0.4489878714084625, 0.34975844621658325, 0.9858599901199341, -0.5299013257026672, 0.08135636150836945, -0.7746081352233887, 0.15200135111808777, -1.3383994102478027, -1.346085548400879, 0.9606051445007324, -2.4085052013397217, 0.8813425898551941, -1.2433604001998901, -0.24859431385993958, 1.1367143392562866, 1.5403335094451904, 1.1641241312026978, 0.21723809838294983, 2.8117425441741943, 1.1101454496383667, 1.1454572677612305, -0.5057610869407654, -2.1807026863098145, -0.7311125993728638, -0.4974710941314697, -1.7631638050079346, -1.2553962469100952, -0.6794177889823914, -2.942143678665161, 0.47115856409072876, 1.008834719657898, 1.4548873901367188, 2.0936291217803955, 0.06546614319086075, 0.6388511657714844, 1.3026951551437378, 0.9096184372901917, 0.7610188722610474, 4.1070098876953125, -2.256807565689087, -0.7964474558830261, 2.1104989051818848, -0.1340097039937973, -0.34297239780426025, -1.379034161567688, -0.22923651337623596, -3.4104535579681396, -0.23411397635936737, -0.16035789251327515, -1.1748113632202148, -0.6443949341773987, 1.214417576789856, 0.6514840722084045, 0.024080896750092506, -3.737414598464966, 0.4798337519168854, 0.8909600377082825, -1.5099434852600098, 0.2571282386779785, -0.592139482498169, -0.3654725253582001, 2.8310749530792236, 0.9050961136817932, 1.5631418228149414, 1.6939611434936523, 0.3238440454006195, 2.7458603382110596, -0.675809919834137, -0.7630931735038757, -0.07958021759986877, 0.25947532057762146, -0.2049574851989746, 0.757434606552124, -0.07300667464733124, 0.5039239525794983, 2.6825084686279297, 1.0128207206726074, -2.146486759185791, -1.0270248651504517, 0.6347780227661133, 0.4488557279109955, -1.557979941368103, -0.6765495538711548, 0.3222469389438629, 2.4622583389282227], [-3.4430794715881348, 2.1198627948760986, 1.0859572887420654, 0.7410812377929688, -0.966915488243103, -1.6429119110107422, 0.9766378402709961, 0.2926276922225952, -0.20216183364391327, -1.0931366682052612, 0.6379547119140625, -1.3019039630889893, 0.614497184753418, 0.5436856150627136, 0.11792251467704773, 0.5900201797485352, 0.569797933101654, 1.9139530658721924, -0.06084173545241356, 1.9951444864273071, 0.14093171060085297, -2.957160711288452, -0.7708979249000549, 0.43660786747932434, 1.0327097177505493, -0.23248198628425598, 0.009102891199290752, 0.43773889541625977, -1.1396033763885498, 1.1003122329711914, -0.7246807813644409, 0.7803220748901367, -0.30710411071777344, -0.5890468955039978, 0.38538867235183716, -2.4676146507263184, -0.8714410066604614, 1.0993479490280151, -0.8281930088996887, -1.6761093139648438, -0.9591392278671265, -0.027322743088006973, 1.3863030672073364, 0.03262899070978165, 1.1055128574371338, -0.16317422688007355, -1.8415404558181763, -0.8643767237663269, 1.525992751121521, -0.33568528294563293, -0.20686055719852448, 0.08739998936653137, -0.43767353892326355, -2.1544363498687744, -2.3901915550231934, -0.32118359208106995, 0.012936118990182877, -0.08467752486467361, 2.3850154876708984, 1.0278230905532837, -0.07702198624610901, 1.597883939743042, 0.7627764344215393, 0.3581729531288147, -0.41767364740371704, -0.34483304619789124, -1.3646457195281982, -0.2544603943824768, 0.6442689895629883, 2.2492830753326416, -0.3362586200237274, 0.8107840418815613, 0.677505373954773, -0.43361932039260864, 0.07867752015590668, -1.7219418287277222, -0.23526930809020996, 0.11905991286039352, 0.7863175868988037, -0.12391461431980133, 0.10388810187578201, 0.8450890183448792, -0.31345516443252563, -0.36129772663116455, -0.33676454424858093, -0.3638119697570801, -0.50055992603302, 0.22236740589141846, -0.7451425790786743, -0.6689995527267456, 4.836834907531738, 0.8921465277671814, -2.658499002456665, -1.8751730918884277, -2.0475006103515625, -2.3504276275634766, -0.42977041006088257, -0.509425163269043, 2.1475210189819336, -3.1403322219848633, 0.8763832449913025, -2.4066126346588135, -2.312931776046753, -0.9459564089775085, -0.5059891939163208, 0.8980635404586792, 2.8770809173583984, -9.640411376953125, 0.9462104439735413, 1.2471122741699219, -1.8639267683029175, 1.3184823989868164, -1.71647310256958, 1.4000415802001953, -1.9309310913085938, 0.5180485248565674, -0.293587327003479, -0.10565730929374695, 2.2170498371124268, -2.301064968109131, -0.05883995443582535, 1.8023735284805298, 0.39945682883262634, 1.7266557216644287, 0.778959333896637, 0.7301349639892578, 0.42178261280059814, 0.04261447861790657, -1.1529710292816162, -0.678068995475769, -0.12942789494991302, 0.3974253535270691, 1.7250360250473022, -0.10529477149248123, -0.570164680480957, -0.2608523368835449, 1.053644061088562, -1.7937686443328857, 0.36771583557128906, 1.9423459768295288, -0.8812116384506226, 4.3859405517578125, 0.498943567276001, 0.8878302574157715, -0.372775137424469, 0.4569838345050812, 0.5183883309364319, 0.8544039726257324, 0.285313218832016, -2.3493587970733643, 0.5403763651847839, -1.0499423742294312, 0.24151311814785004, -0.42197033762931824, 1.5169367790222168, 0.14030905067920685, -0.36797550320625305, 1.9073063135147095, 0.19833612442016602, -0.04333211109042168, 0.09562404453754425, -2.2340612411499023, 0.3805522322654724, 1.540853500366211, -0.574053168296814, 1.6408087015151978, 1.2874029874801636, -1.8436917066574097, 1.2786089181900024, -1.1319591999053955, 0.08791784197092056, -1.7460126876831055, 0.41982969641685486, 1.0993129014968872, -0.49598178267478943, -2.516510486602783, 3.1088063716888428, 1.9793184995651245, -0.006661407649517059, 1.4312777519226074, -1.5656071901321411, 0.6293182373046875, -2.0177977085113525, 0.3268434703350067, 0.8351489305496216, 0.05234173312783241, -0.4265691637992859, 3.9367308616638184, 0.4474732577800751, 1.0469698905944824, 0.18350395560264587, 0.1462738960981369, 0.33593809604644775, 0.3114160895347595, 0.23441314697265625, -4.4169745445251465, 0.034353453665971756, 0.3096442222595215, -0.45717135071754456, -0.944235622882843, -0.5867736339569092, -0.7476158142089844, -0.6956252455711365, 0.5188706517219543, 1.5006362199783325, -5.566246509552002, -0.2927822768688202, 0.1970292329788208, -0.08389915525913239, 1.6193081140518188, -0.46862661838531494, -0.8900892734527588, -1.089475154876709, 0.8572121858596802, -1.9528058767318726, -0.5538285970687866, -0.6656989455223083, -0.5937904119491577, -1.9305306673049927, -0.6395113468170166, -0.3070705533027649, -0.4398921728134155, 0.8941842317581177, -0.792336642742157, -0.4203656017780304, -1.7457867860794067, -1.2951276302337646, -1.0610922574996948, -0.5277451872825623, 0.19500575959682465, 1.147383451461792, 1.121950387954712, -1.7016477584838867, 0.6065845489501953, 0.060293637216091156, -0.2515713572502136, -0.6719681024551392, -0.09169258177280426, -1.0521485805511475, -1.2786624431610107, 2.322453022003174, 2.116912841796875, -1.4395312070846558, 2.3144469261169434, -1.1775496006011963, 2.0399746894836426, -3.2450060844421387, 0.22671028971672058, 3.2753565311431885, -0.8621360063552856, 0.9553282260894775, -0.14660489559173584, -0.7346860766410828, 3.6503756046295166, -2.2564728260040283, -0.7757883667945862, -0.3673311769962311, 1.942387580871582, -2.9397506713867188, 1.9354923963546753, -0.1815449744462967, 0.3196078836917877, 0.6486403346061707, -0.6316912174224854, -0.6512880325317383, 0.23424850404262543, -0.4935745596885681, -0.26458650827407837, 1.17289400100708, 0.0265431459993124, 0.019243557006120682, 2.286518096923828, 2.274829864501953, 0.3715216815471649, -0.5464064478874207, 0.9405515193939209, -0.36312025785446167, -0.025605546310544014, 2.80143666267395, 0.9465919733047485, -0.5613970756530762, 0.48279932141304016, -1.9147316217422485, 2.3013415336608887, 0.8488517999649048, 1.3660521507263184, -0.4848250448703766, 0.77239990234375, 1.7668533325195312, -1.2182570695877075, 0.9444834589958191, -1.6395249366760254, 3.0368998050689697, -1.932747483253479, -0.7655398845672607, -0.7627980709075928, -0.09685271233320236, -0.769362211227417, -1.1021648645401, -1.8474321365356445, 1.1972846984863281, -3.975431203842163, -1.4425559043884277, 0.7071851491928101, 0.3686771094799042, -3.057492971420288, -0.561345100402832, -0.05415695905685425, 1.0825508832931519, 0.5105396509170532, 0.5705610513687134, -1.0961167812347412, -1.0835967063903809, -0.5925301909446716, 2.4785115718841553, 0.5062920451164246, 0.5361254811286926, -3.515956163406372, 1.6380977630615234, 1.2908250093460083, -0.05409087985754013, -1.3464044332504272, -1.5534019470214844, -0.9376437664031982, 1.5455479621887207, 1.9927669763565063, 0.8692833781242371, 0.19061298668384552, -1.9793552160263062, -0.31604641675949097, 0.3349270224571228, 0.022625694051384926, 1.273114562034607, -0.7940171957015991, -0.00119927863124758, -0.9508985280990601, 0.7148396968841553, 1.6821945905685425, -1.4046231508255005, -2.8074445724487305, -0.5688282251358032, -1.4346994161605835, 0.03888339176774025, 0.2533201277256012, 1.4231806993484497, 1.1471946239471436, -0.2637327015399933, 0.992810845375061, -0.49182549118995667, -2.6969525814056396, 4.298703193664551, 1.5097577571868896, 0.9815046787261963, -1.2152926921844482, 0.9217807650566101, -0.8814351558685303, -2.153127670288086, -2.5745160579681396, -0.04468923434615135, 0.3924911320209503, -0.4267438054084778, -0.9820435643196106, 1.3830897808074951, -0.26715928316116333, 1.759656310081482, 0.7386782169342041, 0.05538834631443024, -0.6355145573616028, 2.118258476257324, -1.6380507946014404, -0.02166260965168476, 0.5785358548164368, -0.7126859426498413, -0.45627641677856445, 1.6221247911453247, -2.023914098739624, -3.3289504051208496, -0.8284321427345276, -0.34029924869537354, 1.4266670942306519, -0.17999033629894257, 0.2419002503156662, -1.1570998430252075, 0.49858272075653076, -2.9038970470428467, -1.0100208520889282, -0.28953495621681213, -1.0074360370635986, 0.6718907356262207, 1.5342373847961426, 0.8477084040641785, 0.8871302008628845, -0.5056318044662476, -0.16670802235603333, -0.10224243998527527, -0.1810102015733719, -0.15848393738269806, 0.094314344227314, 0.7956551313400269, -1.1540778875350952, -1.3454833030700684, 1.635926604270935, 1.0196022987365723, -0.9180556535720825, 0.5222089886665344, -0.6208438873291016, -1.12201988697052, -1.5249978303909302, 1.566723108291626, 0.5794652104377747, -0.1747884303331375, 0.04749799892306328, -0.7929118275642395, -0.43144434690475464, -1.2246686220169067, 1.2724661827087402, -1.552433729171753, -0.8510362505912781, 2.0195960998535156, -1.9245901107788086, 0.4155317544937134, 4.007594585418701, -0.1627090573310852, -1.8519448041915894, 0.05941639468073845, 0.25869297981262207, -0.3547239899635315, 0.19015425443649292, 2.3287081718444824, -0.2411624938249588, -1.9727765321731567, -1.385549545288086, 2.303307056427002, -2.1272778511047363, -0.2500966489315033, -1.6153968572616577, -0.43000397086143494, 0.8468502759933472, -0.9312600493431091, 0.3956320285797119, 1.286695122718811, 0.3387213945388794, 0.07999999821186066, 1.2337720394134521, -0.9771891832351685, 3.1628472805023193, 2.243767499923706, 1.1956113576889038, 0.7281571626663208, 0.5848415493965149, -1.1466586589813232, -0.26570671796798706, 0.5258898138999939, 0.3959314823150635, 1.6931639909744263, 0.09925728291273117, -0.8524556756019592, 2.3288092613220215, 1.26167893409729, 0.8236193656921387, 0.7939582467079163, 0.48311498761177063, 0.31055015325546265, 2.408982992172241, -0.19411957263946533, 2.219242811203003, 2.409238815307617, -0.06109156832098961, 0.3800051808357239, -0.020551621913909912, -2.2601003646850586, -0.0768212229013443, 3.171884536743164, 2.960063934326172, 0.23743979632854462, -0.44352084398269653, 0.7087569832801819, -1.596156358718872, -0.9644240736961365, 0.38207483291625977, -0.2898165285587311, 1.4902762174606323, 2.744805335998535, -1.3279414176940918, -0.4691943824291229, -0.7119626998901367, -1.1799989938735962, -2.4698736667633057, -1.157211422920227, 0.6981372833251953, -1.9938117265701294, 0.02476285584270954, 0.6959682703018188, 1.4770219326019287, 1.5590758323669434, -1.7147961854934692, 0.5424796938896179, 0.08474283665418625, 0.9997697472572327, 2.021059274673462, 0.5279161334037781, 0.388765424489975, -0.03254345431923866, 0.492056667804718, 0.6455778479576111, -2.5953621864318848, 0.4435908794403076, 0.011989066377282143, 0.655163586139679, 2.2566235065460205, 1.4816795587539673, -1.5680837631225586, -0.9381657838821411, 1.4727405309677124, -0.20541700720787048, -0.3126577138900757, 2.002211809158325, 0.13354292511940002, -2.1558268070220947, 0.6545787453651428, 1.987828254699707, -0.03842853009700775, 0.6145132780075073, 0.20847299695014954, -0.7507069110870361, 0.3196254074573517, -0.8341324329376221, 2.3979077339172363, 1.3238011598587036, -0.04751056060194969, 0.5374876260757446, -0.8913389444351196, -2.441066026687622, -1.850913166999817, -0.3197236955165863, -0.36851173639297485, 0.1334637850522995, -0.965862512588501, -0.15732869505882263, -0.09841210395097733, 1.1209383010864258, 0.1315903514623642, 1.3778654336929321, -0.04106750339269638, 0.42701220512390137, -1.071128487586975, -0.7089053392410278, 0.8868770599365234, 2.8286783695220947, 0.14828269183635712, 0.0145698431879282, -2.3622589111328125, 0.4892871081829071, 0.7087762951850891, 1.1770355701446533, 0.3119681179523468, -1.7875323295593262, -0.19044287502765656, -3.51375675201416, 0.8560352921485901, 0.2528095543384552, 0.261715292930603, -0.6488504409790039, 1.1762089729309082, -0.4831923246383667, 0.8831912279129028, 1.6533432006835938, -2.299470901489258, 1.770132064819336, -0.10182662308216095, 1.711273431777954, -2.233921527862549, 0.4343072474002838, 0.12304616719484329, -3.989889621734619, 1.2446939945220947, -0.26903200149536133, -0.20675194263458252, -0.02518673799932003, -0.7696855068206787, -0.9048537015914917, -1.7193818092346191, -1.1896311044692993, -2.0725646018981934, -3.914231538772583, 2.76436710357666, 1.487195372581482, -0.1276860237121582, -0.7212239503860474, 0.16771520674228668, 0.15762877464294434, 0.7415998578071594, 6.512005805969238, -0.34766995906829834, -1.7758450508117676, -1.838817834854126, -2.9370453357696533, 0.34217318892478943, -1.2363828420639038, 1.8458585739135742, -0.5766139030456543, 0.8917821645736694, -1.048270583152771, 1.1016931533813477, -0.007876717485487461, -0.780843198299408, 0.7095398902893066, -0.7435848712921143, -0.42056727409362793, -0.35344210267066956, 1.1566553115844727, -0.022042352706193924, -0.15190859138965607, -1.7502026557922363, -2.0495991706848145, 1.8757730722427368, -0.8065047860145569, -3.4756929874420166, 2.1364951133728027, -0.47539022564888, -2.0349652767181396, 1.6449674367904663, 2.7929539680480957, -1.436731219291687, -0.3681427538394928, -0.18929235637187958, -2.3915536403656006, -0.9567420482635498, 0.7554416060447693, -1.2060400247573853, -2.0201964378356934, 0.7391278147697449, 1.5531405210494995, 1.558752179145813, -1.0369044542312622, -0.03830632194876671, -0.9626719951629639, -0.8405699729919434, -0.9576696157455444, -2.6889290809631348, -0.4175114035606384, 2.3277158737182617, -1.4847228527069092, -0.8511722683906555, 1.6580979824066162, 0.819768488407135, 0.37547168135643005, 0.24817515909671783, -0.12450828403234482, 1.2060086727142334, 2.449918270111084, 0.2288498431444168, 2.0238401889801025, -0.18357065320014954, 2.215175151824951, -0.5883952379226685, 2.2748053073883057, 2.1961889266967773, 0.5981475710868835, -1.1398628950119019, 0.5983412265777588, 2.461287021636963, 0.6035593748092651, -0.9007661938667297, -5.353812217712402, -2.301290512084961, -0.24991421401500702, -2.860790729522705, 0.16107849776744843, -0.2937048375606537, -1.0205243825912476, 0.7775496244430542, -1.679063081741333, 0.8658766746520996, 0.2699955403804779, -1.284601092338562, -0.2192557454109192, 1.1356630325317383, -0.5293594598770142, -0.08927217870950699, 0.5223305225372314, -3.160492181777954, 1.6240744590759277, -0.0946686640381813, 4.066064834594727, -0.4199117124080658, 3.0044360160827637, 0.17040948569774628, -0.5332098007202148, 2.9395205974578857, 0.9739084839820862, -0.489051878452301, -0.4785782992839813, -0.25722789764404297, 0.8471865057945251, 0.6111739873886108, -0.9413672089576721, 2.9025509357452393, -0.08474614471197128, 1.9525015354156494, 0.7778708934783936, -0.34830284118652344, -0.30574360489845276, 0.10437308996915817, 2.7944376468658447, 0.8545606732368469, -0.1945287138223648, -0.2620505690574646, -0.8151686191558838, 1.2587212324142456, -0.0032653326634317636, -1.4384326934814453, 0.9868544936180115, -2.0205488204956055, 2.5755562782287598, -0.8956269025802612, -0.3379497528076172, -0.5131398439407349, -0.7461941838264465, 2.383481979370117, -1.370943307876587, 0.7637847065925598, 1.4075685739517212, -0.022866088896989822, -0.4063638150691986, 0.10342954844236374, -2.1073992252349854, -0.294193834066391, -0.9852664470672607, 0.3306735157966614, 0.45281505584716797, -1.538250207901001, -2.1097137928009033, 0.3118545711040497, 0.10526143014431, 0.7703976631164551, -0.6012674570083618, 0.6661635637283325, -0.21651895344257355, 1.5540201663970947, 3.6888749599456787, 1.9482566118240356, -1.447090983390808, -0.5978044271469116, 0.12572260200977325, 1.5280969142913818, -0.254000723361969, 0.7315585017204285, -1.6300591230392456, 0.14404265582561493, 1.1108747720718384, -1.8189030885696411, 0.762986958026886, 1.6662917137145996, -0.4407190680503845, -0.8017099499702454, -0.0479934997856617, 0.8297894597053528, -1.2185766696929932, 0.8707870244979858, 0.7937901020050049, 0.3237471580505371, 0.9702708125114441, -1.1353405714035034, -1.4011396169662476, -1.3847160339355469, 0.397558331489563, -1.4854975938796997, -1.1534645557403564, 1.392795443534851, 1.4445266723632812, 1.4539966583251953, -0.6793068647384644, 1.2691305875778198, -0.971694827079773, -0.40592899918556213, 0.17634862661361694, 1.3944365978240967, -1.066853404045105, 1.0933678150177002, 0.5170599818229675, -1.3115220069885254, -0.5608407855033875, 0.2948313057422638, 1.181166410446167, 2.5862786769866943, -2.5344839096069336, -1.6638747453689575, -0.14356940984725952, -1.238264560699463, -1.043558955192566, -1.0471198558807373, 0.26673761010169983, -3.0750811100006104, 1.0395007133483887, -1.6455090045928955, -0.24278712272644043, 1.233029842376709, 0.08267801254987717, 1.6284174919128418, -0.3422005772590637, -0.7956250309944153, 1.1034724712371826, 0.7819483876228333, -0.0831127017736435, 1.1161019802093506, 0.7596942782402039, -1.3647856712341309, -2.0675365924835205, 0.24450083076953888, 0.9446178078651428, 0.6859245300292969, -1.53964102268219, 0.3333691954612732, -1.5108428001403809, -0.2490886151790619, -0.10411670804023743, 0.6501703858375549, -1.534399151802063, 1.6380574703216553, 1.1551934480667114, -0.5692646503448486, 0.09587384760379791, -4.036098957061768, 0.3945539891719818, -1.6548877954483032, 0.6159714460372925, -0.46258267760276794, -2.6553046703338623, -2.2432405948638916, 0.15748554468154907, 0.36832892894744873, -1.0377339124679565, -1.151292085647583, 0.2918233275413513, -0.6589821577072144, 1.7530927658081055, -1.3229058980941772, 1.4698514938354492, 0.5304243564605713, 0.0682249367237091, 0.8029981255531311, -2.434934616088867, -2.6063990592956543, 1.68716299533844, -0.2375452071428299, 0.4365001618862152, -1.0602128505706787, -1.011016845703125, -2.370311975479126, -1.8886018991470337, -1.713381290435791, 0.054675936698913574, -2.8217568397521973, 0.7395303249359131, -0.8537314534187317, 1.0705236196517944, -0.09051797538995743, 0.6942612528800964, 1.8319045305252075, -0.7831606864929199, -0.3570781648159027, -2.665008306503296, 1.0783824920654297, -0.6572132706642151, -1.958324670791626, 0.5244151949882507, 0.2982275187969208, -1.4680439233779907, 0.6708645820617676, -0.8689442873001099, 0.07247500121593475, -0.785049557685852, -4.1617536544799805, -0.6449035406112671, -0.9058288335800171, -0.08735740929841995, -2.056924343109131, 1.273872971534729, -0.23122411966323853, 0.9389925003051758, 1.44252610206604, -0.6765537261962891, 1.7830607891082764, 1.2231818437576294, 0.4854673743247986, -0.5107451677322388, -0.03483046963810921, -0.7389476299285889, -0.4505445659160614, -0.03477073460817337, 0.6916860342025757, -0.5911777019500732, -0.5798909068107605, 0.09816130250692368, -0.7647206783294678, -1.2166998386383057, -1.2166988849639893, 1.2717264890670776, 1.3983408212661743, -0.35521408915519714, -2.684602737426758, -0.17468810081481934, -0.20916911959648132, 0.5872834920883179, -1.0038598775863647, -0.49564918875694275, 1.6047524213790894, 0.3096234202384949, -0.5055968761444092, 1.102037787437439, -1.184138298034668, 0.5780187845230103, 1.6250965595245361, -1.5485824346542358, -0.01768532581627369, -0.32464399933815, -0.5783914923667908, -0.2710020840167999, 1.1275033950805664, 0.7494789361953735, -0.7676370143890381, -1.854948878288269, 0.004691999405622482, -0.7108494639396667, 0.9996862411499023, 0.3774040639400482, -2.701677083969116, 0.9234473705291748, 1.5189918279647827, -0.12078976631164551, -0.1344670206308365, -0.16002102196216583, -1.0412588119506836, 2.933267116546631, -2.5358188152313232, -1.1327674388885498, -1.5041757822036743, 1.7824252843856812, 1.2124632596969604, -0.21745875477790833, -2.6690077781677246, -1.7173731327056885, -1.65556800365448, 2.054210901260376, -3.799238920211792, -3.151557683944702, -1.1062511205673218, 1.7453627586364746, 0.2943947911262512, -0.8090940117835999, -1.0537536144256592, 0.9080386757850647, -1.9590274095535278, 0.7988064885139465, -1.1171759366989136, 0.6627565622329712, -1.5241963863372803, 1.4932712316513062, -0.740123450756073, 2.1241350173950195, -0.13961267471313477, 0.9156554937362671, 0.26482459902763367, 0.692169189453125, -1.6136224269866943, 2.0754480361938477, 0.7475162148475647, -0.1422104835510254, 1.7440719604492188, -1.463918924331665, -1.1201893091201782, 0.5505567193031311, 0.7431588172912598, -0.4341946244239807, 0.9211336970329285, -1.1121665239334106, -0.2109777182340622, -0.3440597355365753, 0.8304661512374878, -0.9416214823722839, -0.6153625845909119, -0.5168922543525696, -1.6108710765838623, -1.21677565574646, 0.593166172504425, -0.1323399394750595, -0.6067757606506348, -0.9101048111915588, 0.3464983403682709, 0.9528734683990479, 2.054609775543213, 0.34636610746383667, -0.8650012016296387, -1.7570780515670776, 0.025968031957745552, 0.8128952383995056, -0.9271456599235535, 0.5080969333648682, 0.9274111986160278, 0.3475329279899597, -0.4989691376686096, 0.9347565174102783, 1.3254271745681763, 0.5196523070335388, 2.119537591934204, -0.432242214679718, 0.9250133633613586, -1.0777313709259033, -2.085930347442627, -2.1314730644226074, 0.5073971152305603, 1.1371208429336548, 0.4571644961833954, -0.2156534492969513, -0.9139285087585449, 0.2027955949306488, 0.29926562309265137, 0.9572282433509827, -1.628159761428833, 5.642531871795654, -1.4744073152542114, 1.6481908559799194, -0.6283987760543823, 0.6617398262023926, 0.3234887421131134, 0.7412019371986389, 2.2754616737365723, 0.022104939445853233, -3.3281314373016357, -1.1502655744552612, 2.800891160964966, 4.628150463104248, -0.7361970543861389, 0.0011532489443197846, -3.389873743057251, -0.34241074323654175, -0.14587101340293884, 0.6323639750480652, 0.9504513740539551, -2.597795009613037, -1.8224855661392212, 0.8002437949180603, 0.8705406188964844, 1.2507593631744385, 0.8975971341133118, 0.7555782794952393, 1.7853935956954956, -0.693905770778656, 0.2273661494255066, 0.7615816593170166, 0.8690162897109985, 0.31054452061653137, 1.2020450830459595, 2.407367467880249, 1.056420922279358, 0.7714003324508667, -0.0821075439453125, -1.3283114433288574, -2.201026201248169, 1.853865385055542, -0.6651082038879395, -0.11491737514734268, -0.4858117401599884, -0.6439070701599121, -5.566864490509033, -5.7469162940979, -0.5983311533927917, -0.7320777177810669, 0.6383653879165649, -0.42669060826301575, 0.6916475296020508, -1.3965626955032349, 0.7718241810798645, -1.0867154598236084, -2.5225555896759033, 0.1760958880186081, 1.8172962665557861, -1.4577325582504272, -0.825796365737915, -1.3030022382736206, -0.9351367354393005, -1.816189169883728, 0.0368502102792263, 0.23777298629283905, 0.3679242730140686, 0.16013982892036438, 0.7114264369010925, 3.5971739292144775, -0.4339982867240906, 0.49971067905426025, -1.171013355255127, -0.5364991426467896, 1.2241771221160889, -1.0280568599700928, -0.19652609527111053, -0.33706143498420715, 0.19882720708847046, 0.3711414933204651, 1.4085665941238403, -2.080803394317627, 0.3100292384624481, -0.6664036512374878, -0.12229761481285095, -1.3032262325286865, 3.321225643157959, 0.6991015076637268, 1.0831818580627441, -3.0692601203918457, 1.8121682405471802, -0.07346538454294205, 1.0945252180099487, -0.23039083182811737, -1.523889422416687, -0.5326472520828247, 0.958767294883728, -0.14707247912883759, -2.7737152576446533, -0.5625419020652771, -0.5514731407165527, 0.6134181618690491, -0.8082513213157654, 1.5229380130767822, -0.943132758140564, 7.283539295196533, 2.158188819885254, 1.2422927618026733, -1.2625583410263062, -1.3152555227279663, 1.5887466669082642, 0.4302244782447815, 0.35702550411224365, -0.597455620765686, -0.8123302459716797, 0.8940500020980835, -0.2432851791381836, 1.5757681131362915, 0.2059740126132965, 0.7154702544212341, -1.6631972789764404, 0.14550553262233734, -0.7606555223464966, 0.8869543671607971, -2.4008970260620117, 0.8031802177429199, -0.4239363372325897, 1.1088122129440308, -2.462806463241577, -0.29360097646713257, 1.4527764320373535, -0.3487265706062317, 2.9555084705352783, 0.04804067313671112, 0.516434907913208, -0.33084720373153687, -0.5238731503486633, 0.36619678139686584, -0.7166110277175903, -1.5611506700515747, 1.3375998735427856, 1.1631428003311157, 0.7268369793891907, -2.7435052394866943, 0.6603440642356873, -0.04752378165721893, -0.3590225577354431, -1.1077710390090942, -0.470340758562088, 0.18988323211669922, -0.12913209199905396, -2.660583019256592, -0.9610295295715332, 2.2049076557159424, -0.5189448595046997, 3.1153311729431152, -0.796396017074585, -0.2592798173427582, -1.659057855606079, -1.074310064315796, 1.0358638763427734, -0.5418344140052795, -3.474400281906128, 0.2867664694786072, -0.8515160083770752, -1.6385607719421387, 1.1612094640731812, 1.1617265939712524, 0.706426739692688, -1.2146495580673218, 0.8205053210258484, -0.6192240715026855, 1.8340197801589966, 0.5273775458335876, 0.46429532766342163, 0.7574339509010315, -1.7761491537094116, -0.2569054961204529, -2.2092392444610596, 0.2302122563123703, 0.019530408084392548, 0.3237251341342926, -2.2978835105895996, 0.1633584052324295, 0.8441833853721619, 0.7708881497383118, -0.913815975189209, 0.16754242777824402, 0.62037193775177, 1.1370173692703247, -0.6457558870315552, -0.491604745388031, -2.4535279273986816, -0.13165831565856934, 0.17534644901752472, -1.7671451568603516, -2.077282190322876, -0.4152563214302063, 1.3640161752700806, -1.1787463426589966, 0.1315450668334961, 2.111825704574585, -0.5313581824302673, 0.9993457198143005, -0.005731718614697456, 2.1501693725585938, 1.0322977304458618, -2.0436270236968994, 1.2941887378692627, 0.01484583504498005, -0.28718578815460205, 1.3711495399475098, 0.22883708775043488, 1.5089536905288696, 0.5946325659751892, 0.03493737801909447, -0.12877272069454193, -1.5672603845596313, -2.2471861839294434, -0.33458441495895386, -1.5154443979263306, -0.9694970846176147, -0.33636143803596497, 2.537661552429199, -0.40527698397636414, -0.49825170636177063, -1.3542596101760864, -2.4792470932006836, 0.7403553128242493, 3.584390640258789, -0.08972878009080887, -1.949131727218628, -1.1622051000595093, 0.5679131150245667, -0.7321301102638245, -1.0865416526794434, -1.2609955072402954, 2.729719638824463, 1.4024091958999634, -1.8821439743041992, 1.2451094388961792, -0.2946176826953888, 0.5902376174926758, 0.09204389899969101, 1.0882389545440674, 0.8658747673034668, 1.2969977855682373, 0.2419506311416626, -0.2228042036294937, -2.0824480056762695, 0.38647496700286865, 0.4031481444835663, -0.513002872467041, 2.280258893966675, 0.5706244707107544, -0.9954259991645813, 0.20486705005168915, -1.2019122838974, -1.260455846786499, -0.17335213720798492, -0.5104473233222961, 1.1531562805175781, -0.5450731515884399, -1.5120035409927368, 2.154315948486328, -1.8128445148468018, -0.21865038573741913, -1.4356064796447754, -1.153937816619873, -0.19031839072704315, 0.4261990487575531, 1.5178098678588867, -0.5898765921592712, -1.9192103147506714, -1.5578871965408325, 0.8471609950065613, 1.3111246824264526, -1.600932002067566, 0.7722490429878235, 0.13022999465465546, -0.24309177696704865, 0.2278004139661789, 0.28411221504211426, -1.9113696813583374, -0.958906888961792, -0.7949967384338379, -0.1599370688199997, -0.3489227592945099, 1.8706541061401367, -0.3878571391105652, -0.18112768232822418, -0.01947433315217495, -1.2222557067871094, -0.6473049521446228, 1.2570449113845825, -0.1073007807135582, -1.264769434928894, -0.22873568534851074, -0.02478957176208496, 0.4659399092197418, 0.7548241019248962, -2.6291472911834717, 0.6219356060028076, -3.5334203243255615, 2.7016186714172363, 1.2073246240615845, -0.8325102925300598, -0.6560943722724915, -0.6651396155357361, 0.9564223885536194, -0.579994797706604, -0.05599796026945114, -1.0789861679077148, 0.7063242793083191, -2.1687188148498535, 0.11530447751283646, 2.0486717224121094, -0.7350794076919556, -0.35564011335372925, -0.9733533263206482, -2.1588475704193115, -0.18321499228477478, 0.6276135444641113, 2.1531527042388916, 0.34716513752937317, 0.11606273800134659, 1.8786466121673584, 1.5112264156341553, 0.8288272619247437, 1.623419165611267, -0.3928527534008026, -0.039934851229190826, -1.7102160453796387, -0.936803936958313, 0.35520216822624207, -0.010678495280444622, -0.7583792209625244, 0.5448789000511169, 2.880579710006714, -1.0094836950302124, 0.0804307833313942, 0.5218799114227295, -0.5270808339118958, 3.132333517074585, 0.391446977853775, 0.7600081562995911, -0.4317978620529175, 0.6587705016136169, 1.1911534070968628, 5.3783698081970215, 0.4572578966617584, -0.4226078987121582, 0.4386049509048462, 0.5810209512710571, -1.895918607711792, -2.707716703414917, -0.3310375213623047, 0.8525335788726807, 3.8178601264953613, -1.2554054260253906, -0.09270483255386353, 0.6391847133636475, -0.8313096165657043, 0.5443874597549438, 0.8181279897689819, 1.3765767812728882, 2.9814982414245605, -2.1817140579223633, -1.4475914239883423, 0.3267558813095093, -0.24970239400863647, -0.6529110074043274, 0.17682874202728271, 0.944712221622467, 0.04090658575296402, -1.3764543533325195, -1.3793647289276123, -0.4838336110115051, 0.6889132857322693, 0.7589982151985168, -0.17200586199760437, 0.5134603977203369, -1.0198838710784912, -0.41619741916656494, -1.2393406629562378, -0.666403591632843, 0.5411673784255981, -0.607463002204895, -0.5778355598449707, -0.4907558560371399, -1.0504612922668457, -1.0729373693466187, -2.1042354106903076, -0.5004979968070984, 2.4748003482818604, -1.415814757347107, 0.6925677061080933, -0.8352248072624207, -1.1644819974899292, -1.1913058757781982, 3.7630903720855713, 0.06612125039100647, 1.1499526500701904, -1.917950987815857, -0.06797031313180923, -0.7567049860954285, 0.8302509188652039, 3.350294351577759, 1.108803629875183, 0.3916451036930084, 0.012181408703327179, 0.8682342171669006, 1.460347294807434, -3.7539234161376953, -1.4066368341445923, -0.9235445857048035, -0.6453117728233337, -0.7851644158363342, 1.609014868736267, -0.07497790455818176, 0.7399604320526123, -2.0736911296844482, -0.7665980458259583, 0.20221339166164398, -0.5849360227584839, -0.657086968421936, 0.1504949927330017, 2.676711320877075, 0.6250255107879639, -0.7493143081665039, -0.17265461385250092, 0.5927832722663879, 1.0208994150161743, -1.4312092065811157, -0.760852575302124, 0.10109718143939972, -0.25027450919151306, -3.081244707107544, -0.7186131477355957, 0.6579546332359314, 1.7774162292480469, 0.1576332002878189, 1.1936606168746948, 1.6076639890670776, -0.9201757907867432, 2.409933567047119, 0.03734353557229042, -0.36720284819602966, -0.7948381900787354, 0.9971559047698975, 0.16111540794372559, 0.8729419112205505, -1.522065281867981, -0.3244657516479492, 0.9899106621742249, -0.7538427710533142, -2.5519285202026367, -3.412647008895874, 0.879084050655365, 1.0440118312835693, -2.29435133934021, -1.430558681488037, -0.3457464277744293, 1.2279013395309448, 0.8493461012840271, 0.19443346560001373, 4.4191365242004395, 0.2773139178752899, 0.5006003975868225, -1.5717272758483887, -0.7786019444465637, 0.7962526082992554, 0.9057230949401855, 3.7398927211761475, 0.8706154823303223, 0.709550678730011, -0.18376801908016205, 0.4894477427005768, -1.5117448568344116, -0.7316983342170715, 2.2124524116516113, -0.4285021126270294, 2.105132818222046, 3.708131790161133, 1.8417567014694214, 0.40026068687438965, -0.7463926672935486, 1.8620420694351196, 0.20727643370628357, 0.9087292551994324, 1.2480370998382568, -0.5014078617095947, 1.1140538454055786, 0.05293092131614685, -1.456758737564087, -0.7622159719467163, 1.1216431856155396, -1.5013775825500488, -1.4131722450256348, -2.486074686050415, 0.4626959264278412, -4.5194783210754395, -0.9964022636413574, 0.7920807600021362, 0.4817264676094055, -1.5031719207763672, 1.2186880111694336, 0.711750864982605, 0.0989602655172348, 1.6789345741271973, -0.5160681009292603, -0.3672279417514801, -0.300608366727829, 0.6317082047462463, -0.7905427813529968, -0.15431763231754303, 0.9722444415092468, -1.4222623109817505, 0.049466706812381744, -1.2596724033355713, 0.9398896098136902, -0.7417859435081482, 0.22106009721755981, 0.6791577935218811, 0.1367034614086151, -1.277672290802002, 3.0290634632110596, 1.6250618696212769, -2.1012730598449707, -0.35078302025794983, 0.20797821879386902, 0.4097503125667572, -1.7893110513687134, -0.10905463248491287, -0.4416264295578003, 1.8479878902435303, 1.2676732540130615, 0.18448244035243988, 0.3767831325531006, -1.2874335050582886, 0.43858906626701355, 0.46944302320480347, 0.9687261581420898, -0.1929468810558319, 2.0805954933166504, -0.17758743464946747, -3.0570082664489746, -0.10519371181726456, -2.0617282390594482, 1.1002283096313477, 0.06458009034395218, 0.7113940715789795, 3.5504677295684814, 0.4093460142612457, 1.760063886642456, -0.6248143315315247, -0.830436110496521, 1.1541805267333984, -1.3918979167938232, -0.5961967706680298, -0.4994659423828125, 0.1167839989066124, 1.9584344625473022, -0.20782271027565002, 5.690956115722656, -1.6082161664962769, -1.2907606363296509, 0.3411005735397339, -1.6172432899475098, 1.3136130571365356, -0.4771225154399872, -1.6839579343795776, 2.294745445251465, 0.14324043691158295, -1.7838518619537354, -1.343330979347229, -0.6719820499420166, -0.46803033351898193, 0.008231816813349724, 0.006197501439601183, -0.9393936395645142, 0.029850637540221214, -2.0455844402313232, -2.5206587314605713, 0.2391386181116104, 0.6518855690956116, 0.6154683828353882, -1.0313788652420044, 2.773297071456909, -3.632927656173706, 1.0339627265930176, -3.986971855163574, -1.5000784397125244, -2.742727041244507, -0.5014210343360901, 0.43285664916038513, -1.0232651233673096, -0.8039010763168335, -2.2844953536987305, -1.3965204954147339, -0.022694718092679977, -2.122201442718506, -2.9053194522857666, -0.22986046969890594, 1.9542351961135864, 0.00554779963567853, 0.3115977942943573, 0.29771673679351807, 0.4412577152252197, 1.3520106077194214, -0.20874227583408356, -2.166330099105835, 0.4942237138748169, 1.5152535438537598, -3.3774940967559814, -0.13093696534633636, 0.07031390815973282, 1.9989287853240967, -1.0181727409362793, -0.5017480850219727, -2.603376865386963, -0.9649248123168945, 0.809349536895752, 3.459577798843384, -2.680539608001709, -0.16963855922222137, -0.6266960501670837, 0.24929150938987732, -0.6530429124832153, -1.7954784631729126, 1.0744248628616333, -1.158722162246704, 2.8857550621032715, -3.0646331310272217, 0.6304091215133667, -1.197388768196106, 0.9243699908256531, -1.6161861419677734, 4.938028335571289, 0.3632224202156067, -0.35129427909851074, -1.8445327281951904, 2.8440418243408203, 0.9998072385787964, -0.8351209759712219, 2.2981042861938477, 1.4155120849609375, 1.1288810968399048, 0.3579469323158264, 3.563077211380005, 2.0652947425842285, -0.061555586755275726, -0.8864349126815796, -0.819736123085022, 0.3283393979072571, 2.765747308731079, -0.5879751443862915, -2.3735268115997314, -1.1815370321273804, 1.0051110982894897, -0.39091694355010986, -0.15933886170387268, -2.34901762008667, 0.9426736831665039, 0.00579337403178215, 1.9115593433380127, 0.8092034459114075, 0.30597415566444397, 1.0672272443771362, 1.1264081001281738, 3.9652099609375, -2.2843785285949707, -1.924514651298523, -2.0979552268981934, -1.2082990407943726, 0.5866677165031433, 1.9816116094589233, 0.7768721580505371, -0.8168419599533081, -0.9052150249481201, -1.2517517805099487, 0.14076733589172363, -0.3486504852771759, -0.8053731322288513, -0.6966601014137268, 2.0772669315338135, -0.1858646720647812, 1.3732516765594482, 1.42255699634552, 0.1698080599308014, -0.3917617201805115, 0.2761398255825043, 0.9924356341362, 1.8721263408660889, 0.19016847014427185, 1.5762437582015991, 2.411574363708496, -1.2818827629089355, -0.15468177199363708, -0.20907345414161682, 0.606858491897583, 0.5248362421989441, -0.5247501730918884, -0.0013476551976054907, 1.3330347537994385, -0.18649713695049286, 0.8947738409042358, 1.5176608562469482, -2.259946584701538, 0.9380944967269897, 0.2888103425502777, -0.03784792125225067, 0.03237991780042648, 1.0313152074813843, -0.629148006439209, 1.061047077178955, -0.058284904807806015, -0.22390487790107727, -0.7800444960594177, -0.9700725078582764, 1.7461413145065308, 0.16654996573925018, -0.9773890376091003, -0.3579080402851105, 1.2281697988510132, 3.189384937286377, 0.18452149629592896, -0.6039459109306335, -0.7034481167793274, 1.768937587738037, 1.0796427726745605, -0.19162628054618835, 0.7433475852012634, 0.7355779409408569, 1.4546897411346436, -0.7141016125679016, -0.274452269077301, 0.6904667019844055, -1.0071725845336914, 1.2078584432601929, -0.798295259475708, 2.7785611152648926, -1.1059882640838623, -1.3833848237991333, -1.4141749143600464, 2.198244571685791, -0.8201366662979126, 1.9302691221237183, -0.21960505843162537, 2.3909857273101807, -1.2202447652816772, -0.9584997296333313, -0.06595421582460403, -1.5225334167480469, -1.6673178672790527, -1.2242971658706665, -0.7973159551620483, -0.8387657403945923, -1.9820092916488647, 1.4685144424438477, -0.3259030878543854, 0.6334269046783447, 0.3903726637363434, -0.08425037562847137, -1.7663801908493042, -0.741248369216919, 0.13920673727989197, 0.32389986515045166, 0.18104775249958038, 2.1768293380737305, 1.1850414276123047, 2.272897243499756, -2.45125150680542, -1.3007080554962158, -2.2426085472106934, 1.8408838510513306, -1.7382190227508545, 1.4060437679290771, -0.8314869403839111, -2.087958574295044, -1.0981321334838867, 0.14573770761489868, -1.0603588819503784, 2.60703706741333, -0.2741638720035553, 1.0415083169937134, 0.5986620783805847, 1.0050607919692993, -0.0564054511487484, -2.2685093879699707, -0.54054856300354, 0.7361820340156555, -2.3553123474121094, 0.12840193510055542, 1.0933897495269775, -0.8244653940200806, -2.7885048389434814, -1.5258076190948486, -1.122471570968628, 0.5706955194473267, -2.0268802642822266, 0.7982548475265503, 1.1062923669815063, -0.7826570868492126, -1.496922492980957, -0.6230069994926453, -1.463407278060913, 0.561799168586731, -0.4443097412586212, -0.9326977133750916, 2.137174367904663, 0.8022274374961853, -0.9841102957725525, -0.6265422105789185, -1.0635532140731812, -1.0018548965454102, -0.35792043805122375, 0.9939714670181274, -0.6509678363800049, 1.500943660736084, -0.35068055987358093, -0.07342693209648132, -0.9723330140113831, 1.1468805074691772, 0.914495587348938, -0.6696900129318237, 2.0646533966064453, 0.18311814963817596, -1.6916635036468506, -1.0763428211212158, -0.7184515595436096, 0.9292337894439697, -1.3544756174087524, -0.8141908645629883, 0.8989271521568298, -1.4529542922973633, -0.4885985851287842, -0.3471895456314087, 3.2861130237579346, -1.5864160060882568, -1.3134292364120483, 1.1500722169876099, -0.23449496924877167, 0.5941064953804016, -1.0975472927093506, 3.3604068756103516, -0.312624454498291, 1.2124788761138916, -0.9845036864280701, -2.3217787742614746, -2.5787711143493652, -0.9272183775901794, -2.570530414581299, 0.998244047164917, 3.22891902923584, -3.078993320465088, -2.841251850128174, -1.0844359397888184, 0.12273646891117096, 1.496307373046875, 0.9007576107978821, 1.0949876308441162, 1.570551872253418, 0.8985856771469116, -0.3487076759338379, -0.4071364402770996, -1.2817866802215576, -0.7906551361083984, 2.239459991455078, 2.1302332878112793, 0.603160560131073, -1.6103249788284302, 0.5402072668075562, -1.0517890453338623, -2.294724941253662, -0.8195046782493591, 1.3617326021194458, -0.5980315208435059, 0.35910212993621826, 0.1263173669576645, -1.9250454902648926, -0.42364373803138733, 1.9786884784698486, 1.6048904657363892, 1.7318401336669922, 0.8211075663566589, 1.7965266704559326, -0.667294979095459, 0.7439004778862, -0.33743026852607727, 0.6825048923492432, 1.0836621522903442, 1.796448826789856, 1.2951916456222534, 0.5609420537948608, 1.286167860031128, 0.20401936769485474, -1.6524252891540527, -1.8994017839431763, -2.238234519958496, 1.0883150100708008, -1.6799259185791016, 2.4276058673858643, 0.5439278483390808, 2.4570424556732178, -0.33923447132110596, 1.0272111892700195, -1.0217243432998657, 0.2969658076763153, -1.2680617570877075, 1.324377417564392, -0.4623655378818512, -1.2048447132110596, -0.42791250348091125, -2.57494854927063, -0.8656403422355652, -0.00858287326991558, -2.8120546340942383, -1.9260821342468262, -0.8257871866226196, -2.502194881439209, -1.708260416984558, -0.0555715411901474, -4.859560966491699, 0.13398019969463348, 0.4949760437011719, 0.27907392382621765, -0.7762647271156311, 1.8875948190689087, 1.3547120094299316, -1.7081992626190186, -0.6944502592086792, 1.7200523614883423, 0.48144668340682983, 0.12390081584453583, -1.4083229303359985, 1.1794694662094116, 0.009996579959988594, 2.2291183471679688, -0.09467123448848724, 0.6409650444984436, 2.478196144104004, -1.3498951196670532, 0.2722894847393036, -1.4276024103164673, 1.1895970106124878, 0.6104307770729065, 0.8132159113883972, -0.7735401391983032, 1.8990026712417603, 2.0973458290100098, -0.36948537826538086, 0.4427347183227539, -0.48050200939178467, 0.7822349667549133, 1.5475424528121948, -0.07708024233579636, 0.6976090669631958, 0.02268938720226288, 2.397326707839966, -0.11072202771902084, 0.7242120504379272, 0.8542158007621765, -1.8524301052093506, 1.5429575443267822, -1.7017475366592407, 1.163802146911621, 0.9014375805854797, -0.11920760571956635, -1.4347320795059204, -1.1540379524230957, 0.08294054865837097, 0.5108330845832825, -0.7750869989395142, 0.9514564871788025, 0.728148341178894, -1.2281997203826904, 1.5377888679504395, -1.106964349746704, -0.1290934532880783, 0.0037660961970686913, -0.13175752758979797, 0.29428529739379883, 0.8011637330055237, 3.5240535736083984, 0.6144182085990906, -0.5673944354057312, 0.005324218887835741, -0.46358317136764526, 1.3151863813400269, -1.0303175449371338, -0.7545379400253296, 0.3024938702583313, 0.8542673587799072, -0.27191877365112305, 3.136061429977417, -0.037955913692712784, 0.32371848821640015, 0.4976879954338074, 0.4038538932800293, -1.888736367225647, -0.39703568816185, 2.554337501525879, -0.36299553513526917, 0.969649612903595, -0.9499703049659729, -1.3891292810440063, -2.5002286434173584, -1.765385389328003, -1.7008440494537354, -1.6722791194915771, 1.3747196197509766, -0.8005964159965515, 0.2146267294883728, 2.3490793704986572, -0.5300049185752869, -0.21563906967639923, -1.5071783065795898, -0.07179290801286697, 1.5877262353897095, 0.4033094644546509, -2.8217670917510986, -1.3508208990097046, 0.6871868968009949, 1.3242262601852417, 0.2092909812927246, -0.1683339774608612, 1.3673781156539917, 0.06559823453426361, 0.5277499556541443, -0.9237900972366333, -0.24363169074058533, 1.0736626386642456, -1.1044763326644897, -0.2561070919036865, -2.1899025440216064, 0.707436203956604, -0.16014179587364197, 1.146695852279663, -1.477235198020935, 0.98321932554245, -0.8450097441673279, 1.3089983463287354, -0.9670903086662292, 0.8692307472229004, -0.41792961955070496, 0.6801140308380127, -0.7402598261833191, 0.7860509157180786, 0.8320165872573853, 0.18855559825897217, 0.7105780243873596, -1.0027039051055908, -0.717185378074646, -0.6449856162071228, 0.3702728748321533, -1.0836515426635742, 0.08499051630496979, 1.4863659143447876, 0.3611917197704315, -1.135571837425232, -0.7053036093711853, 0.16646870970726013, -0.24137064814567566, 0.34179824590682983, -0.6553636193275452, 0.48260122537612915, -1.433098316192627, -0.15212136507034302, 1.5976885557174683, 0.002930300310254097, -1.0236164331436157, 0.6614588499069214, 0.6439504027366638, 0.5195061564445496, -0.1633499413728714, -0.9149999022483826, 1.7345319986343384, 0.7294982671737671, -1.6768914461135864, 2.0914833545684814, -1.0994030237197876, -0.23621398210525513, -1.1923637390136719, 1.7618207931518555, 0.4090176224708557, -1.324891448020935, 1.7307302951812744, 0.8265132904052734, -0.9642053842544556, -1.2845964431762695, -0.0018197810277342796, -1.0162352323532104, 0.5672705769538879, 3.1988935470581055, -1.0674824714660645, -1.5902507305145264, 2.1311440467834473, -2.0787835121154785, -6.036557197570801, -0.31027212738990784, -1.7382948398590088, -0.6256991624832153, -0.42010432481765747, -0.8214159607887268, -0.44109681248664856, 0.16677825152873993, 0.5146287083625793, -3.2151191234588623, -0.011791955679655075, 1.699825406074524, 1.4989672899246216, 0.24144473671913147, -1.2285627126693726, 1.1286178827285767, -0.0877135843038559, 0.3770465552806854, -0.4978257417678833, 1.7359774112701416, 0.33894824981689453, -0.3475322127342224, -0.7267988920211792, -0.425193190574646, -1.4726216793060303, -0.10553991049528122, 1.102554202079773, 1.1987711191177368, 2.004472494125366, 0.047405630350112915, -2.7844114303588867, 0.80967116355896, 0.7889155745506287, 0.682083785533905, -1.309322714805603, -0.2758185565471649, -1.4947923421859741, -1.2706258296966553, 1.3116545677185059, 0.6541933417320251, -3.206108570098877, 0.1388896405696869, -0.17897219955921173, -0.25128456950187683, 0.30244359374046326, 1.3100041151046753, -0.3155497610569, 0.8214190602302551, -1.8872509002685547, -0.16825133562088013, 0.2665928304195404, 0.3649007976055145, 0.43936070799827576, 2.826362371444702, 0.4940187335014343, 1.5269030332565308, -1.894003987312317, -0.21184124052524567, 0.14401885867118835, -0.45683160424232483, 2.336568593978882, -0.35006263852119446, 0.3228760361671448, -1.594067931175232, 1.8727202415466309, -2.2117562294006348, 2.3984038829803467, -1.3520954847335815, -0.008014855906367302, -0.4670858085155487, -0.3253896236419678, 0.4451111853122711, 0.3062581717967987, -1.054358959197998, 0.3514772355556488, 0.837767481803894, -0.45116209983825684, -1.2491977214813232, 1.5438462495803833, 0.16038639843463898, -0.7203395962715149, -1.740753412246704, -0.9693071246147156, -0.8333804607391357, 0.3071739375591278, -1.6431403160095215, 0.7362512350082397, -0.632594108581543, 0.5934466123580933, 0.19805017113685608, -2.65297269821167, -1.3353997468948364, -3.1670615673065186, 1.2338249683380127, -0.26164665818214417, -1.286983609199524, 0.36231139302253723, -0.439480185508728, -1.466354250907898, -2.5592243671417236, -0.49178364872932434, -0.4938371181488037, -0.7163918614387512, 1.5975006818771362, -1.7745450735092163, -0.7642607688903809, 2.6428961753845215, -0.9375842213630676, -0.6156688332557678, 1.1403452157974243, -0.8673099875450134, -1.907387137413025, 1.0922144651412964, 1.4832926988601685, 1.0642367601394653, -0.76874178647995, -0.6241103410720825, 0.05992614105343819, -0.3722975552082062, 1.4050040245056152, 0.4230738580226898, 4.598537445068359, -0.412797212600708, 2.185485601425171, -0.7448219060897827, 0.10057197511196136, -0.9840850830078125, -2.299088954925537, -0.12776708602905273, -2.8644864559173584, 1.9880194664001465, 0.0014090972254052758, 2.2541890144348145, -0.06399579346179962, -0.6122785806655884, -1.4154950380325317, 0.49674513936042786, -1.9463664293289185, -1.1114894151687622, -1.381483554840088, 0.089737169444561, 0.24204008281230927, -0.9084499478340149, -3.6407012939453125, -0.5666659474372864, 0.8006972074508667, 1.1846904754638672, -3.500612258911133, -2.219489336013794, 0.8085981607437134, -0.8349750638008118, 1.9103587865829468, 0.6214152574539185, 3.211223840713501, 1.8136157989501953, 0.31930628418922424, -2.4177043437957764, 0.18284237384796143, -0.43795183300971985, -0.5537947416305542, 0.4688810408115387, 1.5828948020935059, 1.480293869972229, -2.2506606578826904, 1.4371455907821655, 1.0084762573242188, -0.72101891040802, 0.5998547673225403, 0.44275107979774475, 0.3088005781173706, -0.2816317081451416, -0.4155798554420471, -3.453951597213745, -2.520159959793091, -0.09481102973222733, 1.8077608346939087, 1.7817610502243042, -0.5637961030006409, -0.28251367807388306, -0.5239002108573914, -0.061835650354623795, 0.06861086934804916, -1.8059895038604736, -1.4229395389556885, 0.9136325120925903, -1.929059624671936, 0.6165658235549927, 0.5202038288116455, 0.7148362994194031, -0.9954578876495361, 1.5958328247070312, 0.45398765802383423, 1.480424404144287, -0.48265400528907776, -0.5464682579040527, -0.11174878478050232, 0.057813599705696106, -0.5001898407936096, -1.744697093963623, 1.1868761777877808, -2.183290719985962, 1.2214514017105103, 0.2975652813911438, 0.8344708681106567, 1.1063648462295532, -1.1246823072433472, -1.2374796867370605, -0.983794629573822, -1.9013621807098389, 1.141790747642517, 0.9891915917396545, 0.25400301814079285, -0.5729109644889832, -1.9585598707199097, -0.8290572166442871, -0.8079900741577148, 0.1963217854499817, -2.5956203937530518, -0.39793989062309265, -0.9192891716957092, -0.3688007891178131, -1.1370489597320557, -0.6178416609764099, -0.44533050060272217, 0.13617844879627228, 0.5383332371711731, -0.7202569246292114, 0.22526432573795319, 1.5788286924362183, -0.4123755693435669, -0.5168803930282593, -0.1494872123003006, 2.2207775115966797, -2.1399521827697754, 0.44033050537109375, 1.7103508710861206, 3.068972110748291, 1.5194517374038696, -0.6030375957489014, -0.9402901530265808, -1.2856292724609375, 1.7453460693359375, 2.214212656021118, -1.353421926498413, 0.1878192275762558, -0.08514882624149323, 1.2724618911743164, -0.30015993118286133, -0.7938680648803711, -1.2962626218795776, -1.4251456260681152, -0.5343155264854431, -1.0353477001190186, 0.8292175531387329, -0.9543471336364746, -0.5338671803474426, 1.5857747793197632, -0.15243378281593323, 1.6091811656951904, 2.6760799884796143, 1.9463802576065063, -0.3565346598625183, 1.1984463930130005, -3.2605860233306885, 0.892745316028595, 0.7470957040786743, -4.113540172576904, 1.3238366842269897, -2.108112096786499, 0.40267273783683777, 1.1196578741073608, 1.103971004486084, 0.4044686257839203, 0.4213279187679291, 0.9473115801811218, 1.3397489786148071, 1.2523982524871826, 2.5056655406951904, 2.8815701007843018, -0.852561891078949, -1.758561611175537, 0.11737620085477829, -1.0726783275604248, -1.1615643501281738, 1.7801893949508667, -1.6144930124282837, -0.6239121556282043, 1.3601316213607788, -1.7119810581207275, -0.9778249263763428, -0.3404994606971741, -1.8077380657196045, 0.8967077136039734, -0.4263843297958374, 2.1483867168426514, 0.7004290819168091, 1.6202560663223267, -1.1601779460906982, 0.5764609575271606, 0.29063400626182556, -0.6800843477249146, 0.3079226016998291, 1.6742501258850098, -0.4437609314918518, 0.12398017197847366, 1.581174612045288, -0.5628708600997925, 1.534839391708374, 0.5630911588668823, 1.4118059873580933, -0.6498380899429321, -0.8709017038345337, 1.0924042463302612, -1.1902931928634644, -0.4635327160358429, 0.9463899731636047, 0.4331150949001312, -0.3110811412334442, 0.720037579536438, -0.7429701685905457, -1.8122477531433105, -2.4508931636810303, 1.5019549131393433, 1.538555383682251, -0.32281213998794556, -0.07411889731884003, -0.1257016956806183, 0.5823302268981934, 0.37638625502586365, 1.3933143615722656, 0.6528573036193848, 0.22430568933486938, 0.08845789730548859, 0.6361641883850098, 3.5011346340179443, 1.1466439962387085, -1.4505923986434937, 2.0226495265960693, -0.6586155295372009, 0.2604002356529236, 0.5992956161499023, -0.6113350987434387, -1.8473457098007202, -0.5227187871932983, -0.16188345849514008, -2.7850234508514404, 0.2538623809814453, -0.8150741457939148, -0.11356718838214874, -0.7510589361190796, -2.7564847469329834, -0.9849435687065125, -0.8564091920852661, -2.588731527328491, 0.7742300033569336, -0.786507785320282, -4.503560543060303, 0.9247043132781982, -2.593416213989258, -0.5999324917793274, -1.5673308372497559, 2.686347246170044, 0.07515589147806168, -0.15447157621383667, 0.4111345708370209, -1.2444900274276733, -0.1680908501148224, -1.7119882106781006, -1.489396095275879, 0.3285443186759949, 1.6002693176269531, -1.7856084108352661, -0.7353814840316772, -0.17111437022686005, 0.21520684659481049, 0.9456592202186584, 0.8757373094558716, -2.4532110691070557, 2.603569507598877, 0.9593363404273987, -0.18291498720645905, -1.947838306427002, -0.20163659751415253, -0.12741747498512268, 0.11616487801074982, 1.1909831762313843, -1.2190874814987183, -1.2481050491333008, -1.2011934518814087, -0.03246716409921646, 2.29636549949646, 0.6771693825721741, 1.696204423904419, 0.01691838540136814, -0.514218270778656, 1.1778554916381836, -0.11365286260843277, -0.023393802344799042, -0.7666255235671997, -0.704626202583313, 0.9620723128318787, 2.855712890625, -0.402263343334198, 1.4786388874053955, -1.3842318058013916, 0.9121511578559875, 0.402445524930954, -0.11069338768720627, -1.188612699508667, -1.427718162536621, 2.6751279830932617, -0.9461679458618164, 0.13510368764400482, 0.03531722351908684, -2.6758267879486084, -1.123334288597107, 1.7896935939788818, 0.7314605712890625, -1.46759831905365, -1.4124188423156738, 0.4140437841415405, 1.4883774518966675, -1.9211598634719849, 1.2054126262664795, -0.006510096602141857, 2.3493270874023438, -1.9611780643463135, -0.8169988989830017, -1.091063141822815, 0.4486596882343292, 1.9846906661987305, -2.948132276535034, -0.3261510729789734, 1.0379728078842163, -0.8048698306083679, 1.755195140838623, -1.1892980337142944, 0.6534942984580994, -1.298850655555725, -3.5657155513763428, 1.7086163759231567, -0.48771828413009644, 1.2814700603485107, 1.2725862264633179, -0.8409294486045837, 1.3701244592666626, -0.43079647421836853, -0.8408907055854797, -0.4382924437522888, 0.8460770845413208, 0.2849491834640503, 1.0791056156158447, 1.0974386930465698, -1.20528244972229, -0.11624038964509964, 0.6805376410484314, 2.421700954437256, 0.24339817464351654, -1.0243738889694214, 0.9042354822158813, -0.078001007437706, -0.9147171974182129, -0.739590048789978, 0.8287802934646606, 1.6814141273498535, 1.6254552602767944, -1.156320571899414, 0.04013623669743538, -1.218326449394226, -0.48788270354270935, 1.2497845888137817, 1.14817214012146, -0.13361284136772156, -0.7823688983917236, 1.2716145515441895, -0.3454415202140808, 1.8655036687850952, -1.4405288696289062, 1.4631965160369873, 1.0220547914505005, 0.6748994588851929, 1.814862608909607, 0.4081803560256958, -0.9896498322486877, -0.29284271597862244, 1.7472918033599854, -1.9092046022415161, 1.586612343788147, 0.16584669053554535, -1.7337831258773804, 1.1942003965377808, -1.182913899421692, -1.1351237297058105, -0.8629637360572815, 0.4592888355255127, -0.7820253968238831, -0.032387129962444305, -1.115511178970337, -0.2724802792072296, 0.14213860034942627, 0.023265670984983444, -0.5446562170982361, 3.1159307956695557, 1.352993130683899, 2.2764692306518555, 1.4420442581176758, 3.604557991027832, -1.565609097480774, 0.7216667532920837, 1.576900601387024, 2.561363458633423, 0.1660887449979782, -1.003146767616272, 0.8610142469406128, 1.4750549793243408, 0.8467360734939575, 1.6277718544006348, 0.3418816328048706, -0.6353695392608643, -0.5288224816322327, 0.44834721088409424, 2.3050382137298584, -1.4558486938476562, -1.4974883794784546, -1.1104289293289185, 0.4877060651779175, 3.648242473602295, 1.269361972808838, -2.024796485900879, 2.0673165321350098, 0.27392590045928955, 1.6416877508163452, 0.2696015238761902, -1.6936691999435425, -1.4595637321472168, 0.8823278546333313, 0.07836051285266876, -3.9952917098999023, -1.3193755149841309, 0.17515568435192108, 0.21490433812141418, -0.5488362312316895, -0.014960023574531078, 1.6644736528396606, -0.05035661160945892, -0.5187680721282959, -5.167938709259033, -1.6884795427322388, 0.24310848116874695, 0.01570224016904831, 0.6628363132476807, -1.319915771484375, -0.5808057188987732, 0.2447780966758728, 0.3033565282821655, 0.16966456174850464, 0.6301578879356384, -1.5206011533737183, 4.544835567474365, 1.4271410703659058, -0.19781498610973358, 1.0041935443878174, 0.5038758516311646, 0.8188139796257019, 2.9325692653656006, -1.0029042959213257, 0.010965113528072834, 0.6607744097709656, 1.941730260848999, -1.5147145986557007, -1.8869432210922241, 2.4425618648529053, -1.9423186779022217, 3.476590871810913, 1.7413856983184814, -0.46470317244529724, 3.702819585800171, 1.0716053247451782, 2.2998881340026855, -1.6528488397598267, 0.960671067237854, -0.8360794186592102, 0.011560922488570213, 0.5901530385017395, -1.5867897272109985, -0.016339831054210663, 1.00555419921875, -0.8036572933197021, 2.0292012691497803, -0.7864097952842712, -3.91827130317688, -0.7249788641929626, 0.9161232709884644, 1.1991403102874756, -1.7428910732269287, -0.15106408298015594, 0.12131195515394211, -0.8693843483924866, 0.6002297401428223, 0.3203560411930084, -1.3801898956298828, 1.795498251914978, -4.390141487121582, -1.4756314754486084, 0.2512497901916504, -2.552433490753174, 0.2596738338470459, -0.5753939151763916, 0.4036365747451782, 1.3049465417861938, -2.497332811355591, -1.3080264329910278, -0.45953214168548584, 0.6449247002601624, -0.6712872982025146, 1.1717543601989746, -1.6621652841567993, -1.1465383768081665, -2.380357503890991, 0.09745868295431137, 1.1525415182113647, -2.6157662868499756, 0.7903360724449158, 0.32517868280410767, -1.18500816822052, 0.45667895674705505, -1.3364388942718506, -0.4380360245704651, -0.3269288241863251, 0.08781837671995163, 0.19729706645011902, 1.9509949684143066, -0.5614216923713684, -0.8480153679847717, -0.3996191620826721, 1.370084285736084, 0.9502554535865784, -0.38014504313468933, 1.1140085458755493, -1.179044485092163, 1.9556622505187988, 0.16336897015571594, -0.4434323012828827, -0.7444786429405212, 0.22389651834964752, 0.674502432346344, 0.2585837244987488, 1.1264753341674805, -0.9499885439872742, -0.9670524597167969, 1.6165794134140015, 0.549068033695221, 1.8330551385879517, -0.7221790552139282, -1.9165759086608887, -2.2419540882110596, -1.3454538583755493, -2.635305404663086, 2.1433849334716797, 0.8978533744812012, -2.457597255706787, -1.6844279766082764, 1.1728354692459106, 0.042570602148771286, -1.083509087562561, 0.18614919483661652, 2.3961102962493896, 0.6634255647659302, -0.6876893639564514, 1.98957097530365, 1.7436710596084595, 2.461857557296753, -0.1165066808462143, 1.2196238040924072, 0.07460634410381317, 0.34987062215805054, 2.0441431999206543, 0.6631553173065186, -0.6288254857063293, 0.17808286845684052, 1.0437027215957642, -1.6272082328796387, 0.20355689525604248, 3.1701931953430176, -2.208306074142456, 1.052220106124878, 0.6675450205802917, -0.42945346236228943, 1.0584205389022827, 1.7133833169937134, 0.740497887134552, -0.764344334602356, -0.32195141911506653, 3.0276832580566406, -1.5708779096603394, 0.7191731929779053, -1.015160083770752, 0.8616870641708374, -1.016610026359558, -0.1171727329492569, -2.0266058444976807, -2.227421283721924, -0.27356407046318054, 0.35619431734085083, 0.46942147612571716, -3.0704121589660645, 0.5417649149894714, 0.6264943480491638, -0.3862292766571045, -0.6442101001739502, 2.1592931747436523, -0.048229366540908813, 0.507297694683075, -0.7863857746124268, 0.015180972404778004, 1.3912492990493774, 0.44752442836761475, 0.6237828135490417, -0.8930344581604004, 1.4023284912109375, -2.2546143531799316, -0.3285577893257141, 0.6217747926712036, -2.6980085372924805, -0.5184460878372192, 0.3615093529224396, 0.041515205055475235, -0.5109767913818359, -2.2474148273468018, 0.5027643442153931, -1.5059218406677246, -0.23591259121894836, 1.5152100324630737, -0.633699357509613, 0.09509796649217606, -1.5930109024047852, 0.18728019297122955, -0.22023731470108032, 1.9493712186813354, 0.3598776161670685, 1.3306350708007812, 0.024107562378048897, 0.6273857951164246, 1.9614171981811523, -1.126873254776001, -2.323235273361206, -0.5963140726089478, -0.3827759325504303, -0.5572777390480042, 1.5919989347457886, -0.1590167135000229, -0.39646124839782715, 0.657993495464325, -0.0007419674657285213, -0.5743798613548279, 2.5083370208740234, -0.9599279761314392, -1.2996892929077148, 0.24787402153015137, -1.2985906600952148, 0.8127508163452148, -1.1520057916641235, -1.7099980115890503, -0.058234602212905884, 0.6245136857032776, 0.5635896921157837, -1.4880928993225098, -0.17418105900287628, -1.7775951623916626, -1.2557766437530518, 2.212395668029785, -0.3307288885116577, 1.9135769605636597, 2.985966444015503, -0.36260268092155457, 1.3494162559509277, 1.442873239517212, -1.2063978910446167, -1.147714376449585, 1.7125509977340698, -0.26872989535331726, 0.9517890214920044, 0.0646570548415184, -0.23101449012756348, -0.8667338490486145, -1.907293438911438, -1.020695447921753, 0.8526821136474609, -0.7855668067932129, -0.5352590084075928, -1.6409004926681519, -0.5637184977531433, 0.804735541343689, 0.6657521724700928, -0.7052567005157471, 1.0411375761032104, -1.098771095275879, -1.639372706413269, 0.334181547164917, 2.080545425415039, 1.3665741682052612, 0.5774039626121521, 0.23315639793872833, -0.34777796268463135, -0.907000720500946, 0.5261006355285645, 0.5869383811950684, -3.0370380878448486, 0.26000335812568665, 0.4774796962738037, -0.12511785328388214, 6.785863876342773, 1.6039786338806152, 0.14160150289535522, 1.6208864450454712, 0.5698415040969849, 0.732137143611908, 1.2979907989501953, -0.8884183168411255, 0.25595811009407043, -0.4473833441734314, -0.7527009844779968, 1.5225768089294434, 0.5596230626106262, -0.587709903717041, 0.9580895304679871, -2.5011370182037354, 1.0561457872390747, 1.9730474948883057, 0.3536221385002136, 0.26772835850715637, 0.3317265510559082, -0.795741081237793, -0.6659353375434875, -0.9297480583190918, 0.4058946967124939, -0.07991097867488861, 0.5717004537582397, -0.518160879611969, -0.8147538304328918, -2.4361572265625, 0.4274863600730896, 1.3395754098892212, -0.9671371579170227, -0.5104125142097473, -3.4858007431030273, 0.36198991537094116, -1.7438063621520996, 1.8383376598358154, 0.18777616322040558, -0.7524462938308716, 0.8312280178070068, 0.5489312410354614, 1.9840075969696045, 0.5410764217376709, 0.8401798009872437, 0.8210803866386414, -0.22824521362781525, -0.158558651804924, -2.0765979290008545, 0.7446324229240417, 0.16505151987075806, 0.448648601770401, -2.0322177410125732, -0.6957312226295471, 0.8522766828536987, -1.8759194612503052, 0.1876896172761917, -1.9333348274230957, 0.5146033763885498, -0.5694084763526917, -0.6880389451980591, -0.17172877490520477, 1.243265986442566, -1.8559023141860962, -1.7381303310394287, 2.3815348148345947, -1.4770736694335938, 0.8379589915275574, 0.8836640119552612, -0.3775744140148163, -0.22705380618572235, 0.41758960485458374, -0.07131151109933853, 0.5046799778938293, 0.1555585116147995, -1.3176690340042114, -0.8511137366294861, 1.1400368213653564, 2.1386423110961914, -0.17044800519943237, 0.3311150074005127, -1.1869492530822754, 0.5606744289398193, 0.44767460227012634, -0.9221903681755066, -0.6400781869888306, 1.943491816520691, -0.08295510709285736, 0.5086764097213745, 1.8716316223144531, 1.395896077156067, 1.0486308336257935, 1.8190159797668457, -0.1225007101893425, -0.2967893183231354, 0.5465574860572815, -0.8589644432067871, 0.8391749262809753, 0.6578320860862732, -0.7726062536239624, 1.950160264968872, -1.9904779195785522, -1.0062546730041504, -1.2649576663970947, 0.7505268454551697, -2.104565143585205, 0.2355528324842453, 2.352717638015747, 1.1561830043792725, 0.4577331244945526, 0.37169212102890015, -0.7608911395072937, 0.4428713321685791, -0.6483386754989624, -2.7976038455963135, -0.1293056607246399, -0.11059457808732986, -0.5527085065841675, 1.1048392057418823, -1.4789801836013794, 1.1071863174438477, -1.5955605506896973, -0.2272021323442459, 0.4379386007785797, 0.12362554669380188, -1.9009631872177124, -2.1790482997894287, -1.1340726613998413, 0.4135090112686157, 0.4711058735847473, -0.9750366806983948, 1.1037181615829468, 0.8674590587615967, 0.7724766731262207, -0.7438806891441345, 0.6563028693199158, 0.6440725326538086, -0.8418954610824585, 0.5649713277816772, -1.8173106908798218, -0.6949411034584045, -1.1755897998809814, -6.724612712860107, 3.986614465713501, -1.138580560684204, 1.0795655250549316, 1.126401424407959, 0.6083047986030579, 0.6192187666893005, -1.0891104936599731, -1.251463532447815, -0.6714229583740234, -1.9130940437316895, 0.6971413493156433, 0.1947333663702011, -0.262332022190094, 0.7708614468574524, 1.30122709274292, -1.8188050985336304, 3.5541954040527344, 0.7263264060020447, -6.773098468780518, 1.3443185091018677, 0.29767081141471863, 0.2634890377521515, 2.7312724590301514, 1.463283896446228, 2.2861721515655518, 0.45945286750793457, -0.9813649654388428, -0.11603124439716339, -1.2989938259124756, -1.7937872409820557, 1.0369044542312622, 1.1911447048187256, -0.40264853835105896, 2.1151697635650635, -0.11529850959777832, 0.7941409349441528, -1.3045001029968262, 0.42239224910736084, 1.6221171617507935, -0.47648048400878906, 1.7492517232894897, -0.33955687284469604, -0.5167213082313538, -0.305497944355011, 0.8610619902610779, 1.2339510917663574, 0.8498432040214539, -1.4283655881881714, 0.3767222762107849, 0.10604825615882874, -0.18234053254127502, 0.46845099329948425, -0.01191283855587244, -0.3390159606933594, 0.992466390132904, 1.5243234634399414, -0.2961757779121399, 0.15111230313777924, 0.7970126867294312, 0.18811078369617462, 0.8383228778839111, -0.2653961479663849, 0.6686797142028809, 0.7197821736335754, 1.4829328060150146, -0.37163791060447693, -1.033679485321045, -0.8753306865692139, -2.3962583541870117, -0.2328043282032013, -2.1962409019470215, -1.1549311876296997, -0.6615462899208069, 4.904879570007324, -1.9365822076797485, -2.0697531700134277, 0.721695601940155, 3.630183219909668, -0.840753436088562, -0.5076683163642883, -0.8578788042068481, 1.989048719406128, 0.4400392174720764, -2.4735107421875, -1.9335665702819824, -3.1501128673553467, 0.9653467535972595, -1.012481689453125, -0.4147815406322479, -1.0934828519821167, 0.3161608576774597, 1.670560598373413, 1.2994641065597534, -0.5628204941749573, 2.372720956802368, -0.28715038299560547, 0.6663547158241272, 0.5968678593635559, -1.3457943201065063, -1.8978655338287354, 0.38699138164520264, 0.5892148613929749, 1.443251132965088, -1.2636427879333496, -1.0039616823196411, -4.237420082092285, 0.8871976137161255, 1.3346396684646606, 0.46446362137794495, -1.2881134748458862, 0.7186280488967896, 1.1025699377059937, 1.7926058769226074, -0.771270751953125, 1.942747950553894, -0.3136802911758423, -0.16127972304821014, -0.39192384481430054, 0.09011249989271164, -1.4287617206573486, -1.6429370641708374, -1.210012674331665, -2.840322256088257, 0.16894659399986267, -2.5686092376708984, -6.192514419555664, -0.41611382365226746, -0.543191134929657, 1.8500699996948242, 1.6311103105545044, 0.16020479798316956, -1.417709469795227, -0.25109487771987915, -0.7322096824645996, -0.668713390827179, -4.053224563598633, -0.30525365471839905, -0.14353817701339722, -1.1192138195037842, -0.3830812871456146, 1.036221981048584, -0.18742327392101288, 3.300410747528076, 0.8970440626144409, 0.5559336543083191, -0.5434016585350037, -0.7741524577140808, 2.0019960403442383, -0.864649772644043, -0.19960427284240723, 1.2380772829055786, 0.9575036764144897, -1.4665026664733887, -0.7572122812271118, -0.765982985496521, -0.9947806596755981, 2.5879833698272705, -1.8223271369934082, -1.3570507764816284, -3.569859743118286, 1.9204494953155518, -1.7736947536468506, -1.654137372970581, -0.9214857816696167, 0.6462987065315247, -2.636725425720215, 1.0872092247009277, 2.1238372325897217, 0.9714000225067139, -1.26046884059906, -0.2460588663816452, 0.6878527998924255, 0.9479364156723022, 1.1231893301010132, 0.24205879867076874, -0.09408528357744217, 0.3062264323234558, 1.1258471012115479, 4.032803535461426, 1.566915512084961, -1.4895764589309692, -0.9035497307777405, -0.8496958017349243, -0.5803050398826599, -2.5589561462402344, -0.6961812973022461, -2.7259297370910645, -0.627410352230072, 1.8107521533966064, -0.22427166998386383, -6.6229400634765625, 1.702345609664917, -1.050168752670288, 2.7090554237365723, -0.5121744275093079, 2.089043140411377, -0.5550621747970581, 0.006886804010719061, 1.9067341089248657, 1.0380054712295532, -0.5037725567817688, 2.694427490234375, 0.1939176619052887, -2.997403144836426, -1.222232699394226, -0.4216098487377167, -0.736272931098938, 0.8439767360687256, -2.1166956424713135, -0.024979406967759132, 0.600098192691803, -0.2750210165977478, -0.7746725678443909, 0.7277528643608093, 1.2855265140533447, -1.2435606718063354, 1.6051256656646729, 0.9378986358642578, 0.2668100893497467, -0.349883109331131, 2.8108983039855957, 1.923187494277954, 1.0255275964736938, 2.1737828254699707, -1.474789023399353, -0.04648760333657265, -1.916569471359253, -1.2851557731628418, -2.1846892833709717, -2.108288049697876, -3.7498819828033447, 1.9057128429412842, -2.203768253326416, -1.0592581033706665, -1.0190633535385132, 1.7350558042526245, 0.469620943069458, 1.149990439414978, -1.2045254707336426, 1.2529919147491455, 0.000831071229185909, 0.8038144111633301, -1.1864144802093506, -0.3563211262226105, 1.9076679944992065, 0.2383897304534912, 0.09232961386442184, 2.127033233642578, 3.8566484451293945, 0.9622998833656311, 0.11895637959241867, 0.7480604648590088, 1.9541525840759277, -1.8447353839874268, 0.6387852430343628, 1.3148984909057617, 1.0632104873657227, -0.8781256079673767, 0.7350313067436218, 1.237442135810852, -0.15937897562980652, 0.4431511163711548, -1.2325526475906372, 1.8156764507293701, -0.011977586895227432, 0.8331078290939331, -0.9154114723205566, 0.21563799679279327, 1.9170063734054565, 2.1378984451293945, -0.5570346713066101, -2.4332709312438965, 0.021685605868697166, -1.9089878797531128, 0.3624219596385956, -0.569510281085968, -1.6890097856521606, -1.63730788230896, 0.2504078149795532, -1.6355942487716675, -0.1584089696407318, 0.6252949237823486, -1.1880919933319092, -3.467578172683716, 4.27054500579834, -0.3897170126438141, 0.5410295724868774, -0.4041089415550232, -1.8695143461227417, -0.7250897884368896, -0.3103964924812317, 0.5853132605552673, -0.8831163644790649, 1.1954686641693115, -1.0782328844070435, -2.02354097366333, -1.0559189319610596, 0.21475036442279816, 0.09062279760837555, 0.22220930457115173, 0.6083219051361084, -1.7873845100402832, 0.4064582586288452, 1.271443486213684, 2.7082266807556152, 1.7997783422470093, 2.235062599182129, 1.5045748949050903, 0.3647286891937256, -0.23003609478473663, -1.4331544637680054, 1.0491969585418701, -0.5277504920959473, 2.1554360389709473, 1.3454796075820923, -0.8721336722373962, -0.587962806224823, 0.5512209534645081, -3.0546703338623047, -0.09298986196517944, -2.9233264923095703, 0.318495512008667, -1.6169910430908203, 0.9649398326873779, -1.6789753437042236, 0.6713908314704895, 1.6915525197982788, -1.8198448419570923, -1.9822056293487549, -1.4349709749221802, -1.9102284908294678, 0.05195947363972664, -0.8384549021720886, 0.1503034383058548, -1.084418773651123, -1.4821332693099976, 1.4198696613311768, 1.8528897762298584, 0.6757269501686096, 0.9246504306793213, 1.1045637130737305, 1.4238537549972534, -0.5541037321090698, 1.166067361831665, 0.5966582894325256, 0.07460399717092514, 2.846421957015991, -0.810404360294342, 0.08581563085317612, -2.0685558319091797, -0.6935916543006897, -1.5262658596038818, -0.765204131603241, -0.0021735248155891895, 0.8825194835662842, -1.5753470659255981, -0.20144033432006836, -1.2777153253555298, -0.9677507281303406, 1.8967885971069336, 2.2667596340179443, 0.2654784917831421, 0.8107854127883911, 1.1426959037780762, -2.279963493347168, 0.24311012029647827, 0.0005612624227069318, -3.2886545658111572, 1.4044674634933472, -0.6377057433128357, 0.3988271951675415, -0.01969018206000328, 0.7255074977874756, -1.7212635278701782, -1.5316516160964966, -0.5992944836616516, -2.1091840267181396, -0.015649691224098206, 0.48582667112350464, -0.5999301671981812, 1.591659665107727, 1.4176392555236816, 0.9033389687538147, 0.1700606346130371, -0.531139075756073, 1.393269419670105, -1.5616520643234253, 1.4338512420654297, 0.8800063729286194, -1.522148847579956, -0.09535512328147888, -0.6049453020095825, -1.07809579372406, -2.028547525405884, -0.6661855578422546, 1.3949542045593262, 3.1476082801818848, -0.20788346230983734, -1.2408052682876587, -0.1608729511499405, 1.0877230167388916, 1.7754559516906738, 0.2580818831920624, -1.332417607307434, -0.39985281229019165, -1.4469635486602783, -0.3372024893760681, -0.709010899066925, -1.018007755279541, -1.3460670709609985, -0.5843325853347778, 0.690403401851654, -0.5075671672821045, -0.10247351229190826, 0.6900648474693298, -1.108945369720459, 0.4846368730068207, 2.2571027278900146, -0.7499374151229858, -0.035471610724925995, -1.2871923446655273, -2.0657386779785156, -0.158340722322464, 0.40580061078071594, -1.833630084991455, -0.04543709754943848, -2.1929385662078857, -1.7820168733596802, -0.5622609257698059, 0.9956108927726746, 0.43015575408935547, -1.0191004276275635, -0.7078779339790344, -0.17030839622020721, 0.48503226041793823, -0.35358184576034546, -0.2610836625099182, 0.5361478328704834, -0.9510124921798706, 0.6051764488220215, -1.9782918691635132, -1.5837572813034058, -0.41688910126686096, -1.55769944190979, 0.16372351348400116, 0.0689324364066124, 0.46056845784187317, -0.008372828364372253, 0.07226696610450745, -2.278864860534668, -0.6312817335128784, -0.2645205557346344, -1.2740631103515625, 1.903395652770996, -0.1982049196958542, 0.01435251533985138, 1.0291850566864014, 0.23943130671977997, 0.814702570438385, -0.8859737515449524, 0.6015139222145081, 2.745875597000122, -0.7392621040344238, 0.6963177919387817, -0.20883013308048248, -1.3505773544311523, -1.158194899559021, -2.0282785892486572, 1.2116707563400269, -1.4209593534469604, 1.4316989183425903, -0.47921299934387207, -1.1739259958267212, 1.5100411176681519, -2.4833171367645264, -1.1449816226959229, -1.0167354345321655, 0.8803819417953491, -3.6167142391204834, -2.6306283473968506, -0.19422698020935059, -1.5160698890686035, 0.7382962107658386, -0.33189529180526733, -0.7983002066612244, -1.644450306892395, 0.9357559680938721, 0.5883083939552307, -1.2790228128433228, 2.27424955368042, 1.16937255859375, -1.5973271131515503, 0.25635993480682373, -0.21360081434249878, 0.3565414249897003, -0.07760611176490784, 0.9038754105567932, -0.5526922941207886, -2.357046604156494, 1.3579150438308716, -1.7821694612503052, 1.1447993516921997, 1.110300898551941, 0.886358916759491, 2.824301242828369, -0.42983773350715637, 1.0310124158859253, 0.19975152611732483, -0.06971945613622665, 3.210486650466919, -0.7263391613960266, -1.6133122444152832, 0.0008029913296923041, 0.6540385484695435, 1.9353387355804443, 0.3853996992111206, -0.043509114533662796, 0.9927283525466919, 0.9243838787078857, 0.39545831084251404, -0.412546843290329, -0.11327369511127472, -0.8128107190132141, 0.038585055619478226, 0.6409345269203186, 3.1516199111938477, 1.47103750705719, -0.3485663831233978, -0.6499589681625366, 2.3663673400878906, -1.2761553525924683, -0.7297781109809875, 1.5020382404327393, 0.9727902412414551, 0.22792503237724304, 0.4475269317626953, 1.3683480024337769, 1.9564859867095947, 1.767250657081604, 0.18554382026195526, -0.02128944732248783, 0.25767600536346436, 1.4123740196228027, -1.849711298942566, -0.10460950434207916, -1.1260802745819092, -0.6617008447647095, -1.2770768404006958, -1.1592129468917847, -1.5660736560821533, 1.2202178239822388, 0.42321357131004333, -0.6431982517242432, -2.8384480476379395, -0.7311477661132812, 0.9281114339828491, 0.21157251298427582, 1.4202663898468018, -0.29006218910217285, 3.797971248626709, 1.117701768875122, -0.342183381319046, -0.5711681246757507, -0.10868087410926819, 0.19817474484443665, 0.4514171779155731, 0.27223819494247437, 1.1183890104293823, 1.3571549654006958, 0.5083180069923401, 2.9052889347076416, -0.5526604056358337, 1.4746921062469482, -0.06486426293849945, -1.1983755826950073, -0.29575905203819275, 0.24328866600990295, 2.25870680809021, -1.180399775505066, -1.9058938026428223, -1.9440492391586304, -1.980173110961914, 1.4969052076339722, 0.2723025679588318, -1.354727029800415, 0.3872320353984833, 0.39169132709503174, 0.9908456802368164, 0.05387737974524498, 0.19096581637859344, 2.2936155796051025, 0.8280183672904968, -1.0358189344406128, 1.305006504058838, 0.9811373353004456, -0.9403366446495056, -0.19717048108577728, -1.0665026903152466, 0.12865589559078217, 0.3057863712310791, 0.19673338532447815, 2.0795609951019287, -0.2918696999549866, 1.4043605327606201, -2.245671510696411, 0.8014716506004333, -0.39205536246299744, 0.01379545871168375, -0.9242779612541199, 0.061499711126089096, 1.0078061819076538, -1.3392837047576904, 0.11551856994628906, 0.9532387256622314, 1.9825993776321411, 1.8144853115081787, 0.44718775153160095, -1.3568978309631348, -0.11782127618789673, 0.43501242995262146, 0.2857123911380768, -1.1355112791061401, -0.03885485604405403, -1.1291029453277588, -1.2519950866699219, -0.6981198191642761, -0.3201202154159546, 0.2671979069709778, 0.7611320614814758, -0.34933236241340637, -1.2159004211425781, 0.8445647954940796, 1.9146066904067993, 1.3321363925933838, -3.1994476318359375, -0.1628095656633377, -1.353539228439331, -0.28154805302619934, 0.5675981640815735, 1.301150918006897, 1.1126482486724854, -1.2348077297210693, -1.0002260208129883, 0.41706135869026184, -0.5558299422264099, -0.9039785861968994, -0.7614119648933411, 2.9894521236419678, 0.31189367175102234, -1.2438104152679443, -0.16107024252414703, -0.45210689306259155, -0.17290323972702026, 1.1514967679977417, -0.7174601554870605, 1.1895148754119873, -0.23625417053699493, -0.3672371506690979, -1.4926283359527588, 0.03635116294026375, 0.5370736718177795, -1.378526210784912, -1.0807455778121948, -0.14106114208698273, -0.43137839436531067, -2.0754969120025635, -0.7791776061058044, -1.388466715812683, 3.401007890701294, 1.8456374406814575, 0.495880126953125, 2.9561970233917236, 1.1579816341400146, 0.027913950383663177, -1.7597886323928833, -0.4440637230873108, 1.6556638479232788, 0.813418447971344, -0.0890207588672638, -1.0072418451309204, 0.6102903485298157, -1.1269584894180298, -0.5802369117736816, -0.34115251898765564, 1.704529047012329, -0.9747247695922852, -2.3185501098632812, 0.6540420651435852, -1.1690181493759155, 0.13034893572330475, 2.718837261199951, -0.740757167339325, -1.439214825630188, 0.7244094610214233, -1.8616054058074951, -1.2433741092681885, -0.42778635025024414, 0.5174793004989624, 1.2834193706512451, 0.017715224996209145, 1.9484786987304688, 0.544402539730072, -0.39478594064712524, -0.416199654340744, -1.1151024103164673, -0.686142086982727, -1.5798017978668213, 0.15829448401927948, -0.05170995742082596, 2.002962112426758, 0.2943097949028015, -0.090538889169693, 0.5097505450248718, -1.8368176221847534, 0.6837275624275208, 0.21968339383602142, -2.180396556854248, 1.3705562353134155, -0.5342475175857544, -2.674758195877075, 1.2744792699813843, -0.7703179717063904, 0.7418857216835022, 0.13081422448158264, -0.7505810856819153, -0.3632090091705322, -1.7620766162872314, -0.9250665903091431, -3.432586669921875, 0.11489791423082352, 1.6033647060394287, 1.7174339294433594, -1.3597289323806763, 1.4727506637573242, -0.5295676589012146, -0.565568208694458, -0.7302068471908569, -0.8440169095993042, 0.751778244972229, 0.23429551720619202, -0.16295556724071503, -0.8739662766456604, -2.3994815349578857, -0.7501760721206665, -0.2778921127319336, -0.9655812382698059, 0.5816606283187866, -1.2292536497116089, 0.5641495585441589, -1.3423488140106201, 0.6511447429656982, 0.06877243518829346, 0.9652987122535706, 0.359622597694397, 0.1130259707570076, 0.22751522064208984, -0.8837775588035583, -1.1493052244186401, -0.8679742217063904, 0.3273915946483612, 0.7045482397079468, 3.3879146575927734, -0.8441363573074341, -1.3503086566925049, -1.1170399188995361, -2.4014198780059814, -1.1587947607040405, -1.2735178470611572, 0.9943403601646423, 0.38079774379730225, -0.20467419922351837, -1.4208637475967407, -0.9546341300010681, 0.1690586507320404, -0.5890684723854065, 1.217887282371521, 1.4745492935180664, -0.25762084126472473, -0.7671883702278137, 2.001757860183716, 0.2426714450120926, -1.1275465488433838, 0.08353311568498611, 1.1840488910675049, 0.052849750965833664, 0.7847449779510498, -0.5832878351211548, 1.4097583293914795, 1.8461395502090454, 0.658393144607544, -0.5392588376998901, -0.6252682209014893, 0.46342378854751587, -1.3338794708251953, 0.755098819732666, -3.02984881401062, 0.9898403882980347, -0.23536722362041473, -0.2104347199201584, 1.307266116142273, -0.12658371031284332, 0.47552111744880676, 0.559728741645813, -0.1922917366027832, 0.7766293287277222, -1.5512560606002808, -0.048199862241744995, -2.627119779586792, 0.13131539523601532, 0.2043272852897644, -0.29134345054626465, 0.0563405379652977, -1.5942456722259521, 1.3003244400024414, 1.0847631692886353, 0.9642831087112427, -0.5886508226394653, -2.4978463649749756, 4.566019535064697, 2.491002082824707, 2.926730155944824, 2.0133278369903564, -1.4738409519195557, -0.740100085735321, -0.17877602577209473, -1.233616828918457, 1.3930047750473022, 0.6345213055610657, 0.20441515743732452, 1.9006602764129639, 0.14706364274024963, -1.9319148063659668, -0.5556500554084778, 1.4970301389694214, 0.16452206671237946, 1.6325446367263794, -0.28235310316085815, 0.012063532136380672, 3.444226026535034, -0.11960437148809433, 2.0556788444519043, -0.8690378665924072, -0.7474233508110046, -0.8962644338607788, -0.40935105085372925, -0.22658276557922363, -0.36997589468955994, -0.45931151509284973, 0.3542894721031189, 0.41814443469047546, -0.8108301758766174, 0.33730414509773254, -0.25180140137672424, -0.4185488224029541, -1.754296064376831, 0.9485952258110046, -0.6902364492416382, 0.6783174276351929, 0.2164897918701172, -1.7174961566925049, 2.9762747287750244, 0.7655075788497925, 0.04825951159000397, 2.506965398788452, -0.45333167910575867, 0.7626464366912842, -1.40586519241333, 2.455618143081665, -1.172370195388794, 0.010423143394291401, 0.7765589356422424, 0.3730950951576233, -2.3811798095703125, 1.237854242324829, -0.11773840337991714, 1.504995584487915, -1.125875473022461, -1.9006783962249756, 0.4448481798171997, 2.382625102996826, 0.07792294025421143, -1.523171305656433, -2.0268325805664062, 2.891789436340332, 1.0321787595748901, -1.1919225454330444, 0.032798949629068375, 0.07242467254400253, -0.05183778703212738, 1.0331224203109741, 0.651176929473877, -0.22259129583835602, 1.860395908355713, 0.6735085844993591, -0.41547805070877075, 0.33770424127578735, 2.318758010864258, 0.8788536787033081, -0.3020929992198944, 1.5111597776412964, -0.24085605144500732, 0.6422626972198486, 0.6132885813713074, 1.1116863489151, 3.5099093914031982, -0.7230385541915894, -0.8085198402404785, -1.2373532056808472, -0.9502723217010498, 0.44603797793388367, -0.181528240442276, -0.5057946443557739, 0.35242700576782227, 1.0550189018249512, 0.5738440155982971, -1.1221809387207031, 0.7577344179153442, -1.8113008737564087, -1.674643635749817, -1.122841238975525, 1.1731544733047485, -0.8514769077301025, 0.7290188670158386, -0.7399502992630005, 1.9439737796783447, 1.1846323013305664, 0.11423647403717041, -0.9626450538635254, 0.780332624912262, -0.40522101521492004, 0.04227323457598686, 0.5856190323829651, -0.7401863932609558, -0.37538832426071167, -1.0096808671951294, -0.5054471492767334, 0.4736888110637665, -1.8522107601165771, -0.0607764907181263, 0.40922537446022034, -0.4962693154811859, 0.05420191213488579, 0.393928200006485, -0.2601938843727112, 0.2976599633693695, -1.354766607284546, 0.15042060613632202, -2.0770249366760254, -1.2647556066513062, 0.5894245505332947, -1.963984727859497, 0.9496945738792419, -0.9401466250419617, 0.361066609621048, 1.4751464128494263, 1.6175789833068848, 0.5220592617988586, 0.39597442746162415, 1.9222301244735718, 1.2274450063705444, 1.4452033042907715, -0.4429486095905304, -0.8248498439788818, -1.1630747318267822, -0.4725111722946167, -1.9235923290252686, -1.7621711492538452, -0.6411020755767822, -0.17480342090129852, -2.302366256713867, 0.9716053605079651, 0.9890980124473572, 3.185905933380127, 0.11325718462467194, 0.3119843304157257, -2.6856436729431152, -1.7387765645980835, 0.611322283744812, 3.0844385623931885, -0.8489629030227661, 0.3384043574333191, 0.8823832869529724, -0.16379837691783905, -0.47499340772628784, -0.3752894997596741, -1.6409308910369873, 2.0572502613067627, 0.05495399609208107, 1.4347611665725708, 0.992642343044281, -0.6499614119529724, 1.2158143520355225, 0.6383886933326721, 0.8925020694732666, -2.06850528717041, 0.022927163168787956, 0.5538445711135864, -0.44103673100471497, -0.5787104368209839, -0.5901144742965698, -0.9199939966201782, 2.471776008605957, 0.42622309923171997, 1.681998610496521, 0.05113374814391136, 0.03265054523944855, 1.0110595226287842, 0.6845980882644653, -1.0578570365905762, -1.3793710470199585, 1.7989401817321777, -0.9723886847496033, 0.9821378588676453, 0.4225093126296997, -0.5276755094528198, 2.3936362266540527, -1.250688076019287, -0.6402254104614258, -3.1516542434692383, -2.497314214706421, 0.6024661660194397, -0.9616692662239075, 2.2977795600891113, -1.5664150714874268, 2.549852132797241]], "dtype": "float32"}, {"__nd__": [-0.002058063866570592, 0.0014582969015464187], "dtype": "float32"}, {"__nd__": [1.008293867111206, 1.016686201095581], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(layer_norm_fwd, json.loads(_TESTS), 'triton', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")